In [ ]:
# # 논문 1 사후 교정 분석 지침 (S0~S8 결과 기준, 2026-10-19)

# ## 0. 먼저 지켜야 할 원칙

# 사후분석은 **확증이 아니라 보완**입니다. 아래 규칙을 어기면 지금까지 쌓은 동결 체계의 신뢰가 무너집니다.

# 1. **동결 판정은 바꾸지 않습니다.** 사후분석 결과는 "PA-" 번호(Post-hoc Analysis)로 따로 표기하고, 본문에서 사전 명시 분석과 시각적으로 구분합니다.
# 2. **폴더와 동결을 분리합니다.** `paper1/09_PA/`를 만들고 `PA_0_PLAN.json`(분석 목록, 정의, 임계값)을 **실행 전에** 해시와 태그로 남깁니다. 사후분석이어도 "사후분석 안에서의 사전등록"은 가능합니다.
# 3. **보고 의무:** 모든 PA 결과는 방향이 기대와 반대여도 보고합니다. 기각된 PA 목록을 부록에 둡니다.
# 4. **다중성:** PA 항목 수를 먼저 세고(아래 약 14개), 본문에 "탐색적, 다중성 보정 없음"이라고 씁니다. 핵심 주장에 쓰는 PA는 3개 이하로 제한합니다(PA-A1, A2, B1 권장).
# 5. **진단 먼저, 코드 변경은 나중.** 각 PA는 ① 진단 출력 ② 해석 ③ (필요할 때만) 코드 수정 순서입니다.
# 6. **재현성 선행:** S4 코드 복구, 잘못된 태그(`S4-main`, `S4-final`) 정리, 미푸시 태그 푸시, `oa_raw` 외부 백업을 PA 시작 전에 끝냅니다(PA-0).

# ---

# ## PA-0. 재현성 정리 (분석 아님, 1~2일)

# | 작업 | 확인 방법 | 완료 기준 |
# |---|---|---|
# | S4 코드 3종 저장 | `ls paper1/s4_*.py` | `s4_main.py`, `s4_6_rotation.py`, `s4_6b_floor_check.py` 존재 |
# | 잘못된 태그 정리 | `git show --stat S4-main S4-final` 후 로컬 삭제, CHANGELOG에 정정 기록 | 태그가 실제 커밋을 가리킴 |
# | 미추적 파일 | `git status --short paper1/` | 04_S4, 07_S7, 08_S8 포함 전부 추적 |
# | 푸시와 백업 | `git push --follow-tags`, `archive/oa_raw_*.tar.gz` 복사 | 원격에 태그 존재 |
# | 전체 manifest | 각 단계 해시를 `paper1/MASTER_MANIFEST.csv` 한 장으로 통합 | 논문 부록용 |

# ---

# ## 파트 A. 강점 강화

# ### PA-A1. N2(완만 추세) 귀무 재설계로 "연속 변화" 결론을 정량 강화

# **왜:** 현재 "체제 전환 없음"은 N2 통과 실패라는 **부정적 증거**에 기대고 있습니다. N2의 검정력이 낮아서 실패했을 가능성(점프가 있어도 못 잡음)과 점프가 정말 없는 경우를 구분하지 못합니다. 이 구분이 논문의 핵심 주장의 방어선입니다.

# **진단 먼저**
# - S5 N2 시뮬레이션에서 **관측 기울기와 같은 선형 추세**가 얼마나 큰 R(비용 감소율)을 만드는지 분포를 저장해 두었는지 확인합니다. 저장돼 있으면 재사용합니다.

# **교정 분석 (검정력 곡선)**
# 1. 관측 자료의 토픽 비중에 **인위적 점프**를 심습니다. 점프 크기 δ ∈ {0.5, 1, 2, 3}× 관측 2020 점프(JS 기준)를 연도 t* ∈ {2008, 2012, 2015, 2020}에 삽입합니다.
# 2. 각 조합에서 S5 탐지를 B=200회 반복하고, N2 귀무 대비 p<0.05가 나오는 비율(=검정력)을 구합니다.
# 3. 출력: 점프 크기별 검정력 곡선.

# **판독 규칙(미리 정함)**
# - 관측 점프 크기의 1배에서 검정력 ≥ 0.8이면: "이 크기의 점프가 있었다면 탐지했을 것이므로 N2 비통과는 연속 변화의 증거"로 강화됩니다.
# - 검정력 < 0.5이면: "체제 전환 부재"가 아니라 **"체제 전환 여부 판정 불가(검정력 부족)"**로 리프레이밍합니다. 이 경우도 유효한 결과입니다.

# **추가:** 구간별 상수(piecewise constant) 대 구간별 선형(piecewise linear) 적합의 BIC·교차검증 오차 비교를 `all`, `fixed14`, `within` 각각에서 보고합니다. 선형 연속 모형이 우세하면 직접 증거가 됩니다.

# ### PA-A2. S6 신규 진입 주도 결과의 "구성 방어" 강화

# **왜:** 이 결과가 논문의 새 중심 기여가 됩니다. 현재 약점은 (a) 이름 기반 식별, (b) 지속 저자의 선택성, (c) 부트스트랩 구간이 점추정을 포함하지 못한 점입니다.

# **교정 분석 3종**

# 1. **ID 기반 대조 (2020년대 한정):** F7의 OpenAlex ID로 마지막 쌍(2015–19→2020–25)의 신규·지속 판별을 이름 대신 ID로 다시 하고 진입 몫을 비교합니다. 차이가 ±0.03 이내면 "이름 기반 결과가 ID 기반과 일치"라고 쓸 수 있습니다. **2020+ 한정임을 명시**합니다.
# 2. **진입 정의 민감도:** 현재 신규는 "첫 출판 연도 ≥ t1 시작"입니다. 대안을 두 가지 더 돌립니다.
#    - 신규 = 첫 출판이 t1 시작 후 **2년 이내**
#    - 신규 = 이전 5년간 코퍼스에 없던 저자(복귀를 별도 칸에서 흡수)
#    진입 몫이 모든 정의에서 적응 몫보다 큰지 확인합니다.
# 3. **부트스트랩 보고 방식 교정:** 구간이 점추정을 포함하지 않는 문제는 절대값 합 통계의 상향 편향입니다. 편향 보정 값 `2×관측 − 부트스트랩 중앙값`과 **저자 단위 클러스터 부트스트랩**(저자 복원추출만)을 병기합니다. 곱셈 가중 부트스트랩은 민감도로 내립니다.

# **판독:** 세 교정에서 "진입 > 적응" 방향이 유지되면 강화입니다.

# ### PA-A3. S7 선두 국가 교체 서술의 정밀화

# **왜:** 미국↓ 중국↑는 견고하지만, "알려진 국가 기준 상위 5개국 몫은 변하지 않음"이 가장 독창적인 서술입니다.

# **교정 분석**
# 1. **국가 미상 질량의 시간 변화 교정:** 미상(11%→4%)이 상위 5개국 몫의 겉보기 증가를 만든다는 해석을 **복구판(variant=rec)**에서도 재현하는지 표로 제시합니다(이미 격자 안에 있으므로 재집계만 필요).
# 2. **기간 길이·부분연도 교정:** P0709(3년), P2025(6년, 2025 부분연도)의 길이 불균등 영향을 보려고 기간을 **2007–11, 2012–16, 2017–21, 2022–25**의 균등 4구간(4~5년)으로 바꿔 상위 5개국 몫, HHI, 중국·미국 몫을 재계산합니다. 방향이 같으면 강화입니다.
# 3. **HHI의 비단조성:** P1519 최저, P2025 반등이 중국 몫 증가에 의한 것인지 확인하려고 **중국 제외 HHI**를 보고합니다.

# ### PA-A4. S2·S3의 투명성을 "방법론적 기여"로 정리 (분석 아님, 서술 작업)

# - 동결·CHANGELOG·사후 결정 기록 체계를 **부록 표 한 장**으로 만듭니다(단계 / 동결 해시 / 결과 열람 여부 / 사후 결정 사항).
# - 이는 약점(완전 비열람이 아님)을 강점(모든 예외가 기록됨)으로 바꾸는 장치입니다.

# ---

# ## 파트 B. 약점 리프레이밍

# 각 약점마다 **현재 문구 → 교정 문구(before/after)**와 **뒷받침 분석**을 짝지었습니다.

# ### B1. 체제 서사 철회

# | | 문구 |
# |---|---|
# | Before | "체제 전환 서사를 철회했다" (실패처럼 읽힘) |
# | After | "교통 지식 생산의 주제 구조 변화는 점프보다 연속적 재배치로 서술하는 편이 자료에 부합한다. 2020 부근은 세 표본에서 유지되는 유일한 후보이나 측정 아티팩트와 구분되지 않는다" |

# **뒷받침:** PA-A1(검정력 곡선). 검정력이 충분하면 "연속 변화 발견", 부족하면 "판정 불가"로 쓰되, 어느 쪽이든 제목에서 **"체제 전환"을 빼는 것**이 안전합니다. 제목 가안은 "교통 지식 생산의 연속적 재배치와 갱신 메커니즘" 계열로 수정을 검토하세요.

# ### B2. 2020 변화점이 아티팩트 후보로 남은 문제

# **진단 먼저:** 2020 점프가 (a) 논문 수 급증(3,866→4,794), (b) T-ITS·T-TE 대형 저널 비중 변화, (c) 임베딩 보유율 변화 중 무엇과 함께 움직이는지 분리합니다.

# **PA-B2 교정 분석**
# 1. **표본 크기 정규화:** 2020 전후 연도를 **연도당 동일 수(예: 3,800편)로 하향 무작위 추출**(B=200)해 JS 점프를 재계산합니다.
# 2. **저널 가중 고정:** `within` 표본이 이미 이 역할을 합니다. 2020이 `within`에서도 유지되고 하향 추출 후에도 유지되면 "표본 크기 아티팩트가 아님"으로 말할 수 있습니다.
# 3. **COVID 인접 사건 분리:** 2020 이후 주제(자율주행, 전기, 팬데믹 이동성)의 점유 변화를 S3 카드 기준으로 나눠 **어느 토픽이 2020 점프를 구성하는지** 상위 10개를 표로 냅니다.

# **리프레이밍:** "아티팩트 후보"를 "표본 크기·저널 구성 통제 후에도 유지되는 2020년 부근의 가속"으로 바꿀 수 있는지가 이 PA의 목표입니다. 유지되지 않으면 "아티팩트"로 확정합니다.

# ### B3. 신생 토픽 정의 4종 불일치 (Jaccard 0.04~0.47)

# | | 문구 |
# |---|---|
# | Before | "단일 신생 목록으로 주장할 수 없다" |
# | After | "신생은 정의 의존적 개념이며, 어느 정의에서도 반복적으로 나타나는 **핵심 신생 집합**과 정의 특이적 토픽을 구분해 보고한다" |

# **PA-B3 교정 분석**
# 1. D1~D4에서 **n_defs ≥ 3**인 토픽(핵심 집합)과 n_defs = 1인 토픽을 나눠 S4 4분면·저널 구성 분해를 각각 재요약합니다.
# 2. D1이 이질적인 이유를 **D1을 2005–09 대 2015–19로 재구성한 D1'**(사후 정의, 표기)와 비교해 검증합니다. F6 생성 코드를 먼저 확인하세요.
# 3. 핵심 집합 토픽의 신규 저자 진입 몫(S6 연계)을 보고합니다. 일관되게 신규 진입으로 설명되면 PA-A2와 연결되는 강화 문장이 됩니다.

# ### B4. S3 등급의 낮은 변별력

# | | 문구 |
# |---|---|
# | Before | "통과했으나 약한 증거" |
# | After | "등급은 변별 기준이 아니라 **분석 대상 토픽의 이질성을 점검하는 감사 도구**로 사용했다. 실질적으로 변별하는 기준은 F1(안정성)과 F3(비연구·복합)이다" |

# **PA-B4 교정 분석 (등급 대체안)**
# 1. F2·F4를 빼고 **F1+F3만 쓴 이분 등급**(안정, 불안정)을 사후 정의로 만들고, S4·S5·S6 핵심 결과를 "F1·F3 모두 통과 토픽(145개)"으로 재추정해 병기합니다.
# 2. 더 변별적인 지표를 추가합니다: 토픽 안 논문 간 평균 코사인(임베딩 응집도), 부트스트랩 Jaccard의 10퍼센타일. 사후 표기로만 사용합니다.
# 3. 결과가 전체와 거의 같으면 "등급 선택이 결과를 좌우하지 않음"이라고 씁니다.

# ### B5. S4 안정성 0.700(경계)

# **진단 먼저:** big 50개 중 35개 유지의 **이항 구간**을 계산합니다(Wilson 95% 약 [0.56, 0.81]). 구간이 0.70을 포함하므로 "경계 통과"가 아니라 **"0.70 전후에서 불확정"**이라고 쓰는 것이 정직합니다.

# **PA-B5 교정 분석**
# 1. 상위 3분위 컷을 2/3 대신 **0.60, 0.70, 0.75 분위**로 바꿔 안정성이 어떻게 변하는지 곡선으로 제시합니다(사전 규칙 변경 아님, 민감도 곡선).
# 2. 라벨 6종을 **"성장/쇠퇴 방향"만** 남겨 부호 안정성을 재계산합니다(앞서 부호 유지율 0.98). "4분면의 질 축이 불안정하고 방향 축은 견고하다"로 리프레이밍하면 정확한 서술입니다.

# ### B6. 위험군 검증의 순환성

# | | 문구 |
# |---|---|
# | Before | "검증이 순환적이다" |
# | After | "동명이인 위험은 독립 정답 없이 **상한·하한 범위**로 다뤘고, 결과는 변형 7종과 위험군 미제외(none) 변형에서 일관했다" |

# **PA-B6 교정 분석 (독립 정답 확보)**
# 1. **ORCID 기반 소표본:** OpenAlex에서 ORCID가 있는 저자만 추려 이름→ORCID 분할 여부를 정답으로 하는 위험군 정밀도·재현율을 계산합니다(ID 분할 정의와 독립). 표본이 작으면 구간을 넓게 씁니다.
# 2. **수작업 표본:** 이름 빈도 상위 구간(행 10편 이상)에서 50쌍을 무작위로 뽑아 소속·분야로 동일인 여부를 판정합니다. 연구자 1인 판정이므로 한계로 적습니다.
# 3. S6 homonym 결과에 이 범위를 덧붙여 "변형 7종 범위 + 독립 정밀도"를 함께 보고합니다.

# ### B7. 분석 집합이 임베딩 보유 논문(83.9%)

# **PA-B7 교정 분석**
# 1. **경계(bounds) 분석:** 토픽 결측 14,810편이 전부 특정 토픽(예: 증가 토픽 vs 쇠퇴 토픽)이라는 **최악·최선 대입**으로 S4 핵심 결과(big_change 방향)의 유지 여부를 확인합니다(S2.6(c)에서 이월된 항목).
# 2. **제목 분류기 예측 토픽(민감도 전용)**을 tau 0.5, F1 하한 0.30에서 붙여 S4·S5 핵심 결과를 재현합니다. 분류기 정확도가 0.508이므로 "방향이 유지되는가"만 봅니다.
# 3. **저널×연도 상호작용 IPW:** 2007년 하락을 보정하는 모형을 민감도로 추가하고 변화점이 이동하는지 봅니다. 동결 후 추가이므로 CHANGELOG에 "탐색적"으로 남깁니다.

# ### B8. S8: 6개 도메인 중 2개 통과

# | | 문구 |
# |---|---|
# | Before | "2/6만 통과" |
# | After | "사전 기준으로 정규식 사전의 정밀도를 검증한 결과, 교통안전·대중교통에서만 정렬 분석을 수행했고 나머지 4개는 부록에 precision을 공개했다. RQ5는 **사례 2개의 정렬 간극 지도**로 제한한다" |

# **PA-B8 교정 분석**
# 1. **미달 4개 도메인의 부록 정렬 결과**를 "참고용(precision 미달)" 표지로 같은 코드로 돌려 부록에 둡니다. 방향이 통과 도메인과 같으면 일반화 가능성의 약한 단서, 다르면 "도메인 의존"으로 보고합니다.
# 2. **사전 개선 후 재검증(별도 사후 표지):** precision 0.76~0.84 도메인에서 오탐 패턴을 보고(예: active_urban의 "accessibility"), 개정 사전 v2를 만들어 **새로운 무작위 표본 25편**으로 다시 검증합니다. 이 경우 반드시 "사후 개정 사전"으로 표기하고 본 결과와 섞지 않습니다.
# 3. **소득 통제 문제 정리(PA-B9)로 연결.**

# ### B9. SBR 간극이 소득 수준과 겹침 (국가 수준 편 ρ가 약함)

# **왜:** 소득군 SBR은 뚜렷하지만 GDP 통제 후 약하다는 점이 가장 큰 해석 위험입니다.

# **PA-B9 교정 분석**
# 1. **소득군 내 분석:** HIC 안에서(58~86개국), UMC 안에서 각각 공급 강도와 부담 강도의 순위상관을 구합니다. 소득군 안의 상관이 양이면 "소득 효과만이 아님"의 단서가 됩니다.
# 2. **로그 GDP 대신 소득군 더미 통제**와 **GDP·인구 이중 통제**를 병행합니다.
# 3. **공급 강도의 정의 민감도:** 공급 강도를 질량 기준에서 **논문 수 기준**으로 바꾸고, 총 질량이 작은 국가(min_country_mass 10)의 영향을 20과 5에서 확인합니다.
# 4. **결론 문구:** "소득 수준과 분리되지 않는 간극"이라고 쓰며 필요 충족 증거로 쓰지 않습니다.

# ---

# ## 파트 C. 정합성·결과 간 교차 점검

# ### PA-C1. S6와 S7을 한 문장으로 잇는 교차 분석

# **목표:** "신규 진입 저자가 중국 몫 증가를 이끌었다"가 S6의 토픽 갱신과 같은 현상인지 확인합니다.
# 1. 신규 저자 중 중국 소속 비중이 높은 토픽(S7 7-3)과 S4 성장 big 토픽의 겹침을 계산합니다.
# 2. **중국 소속 신규 저자를 제외한 S6 분해**(진입·적응 몫)를 다시 돌려 진입 우세가 유지되는지 봅니다. 유지되면 "신규 진입 주도는 중국 유입에 의한 것만이 아니다"라는 강한 문장이 가능합니다. 유지되지 않으면 두 결과는 같은 현상의 두 측면입니다. 어느 쪽이든 보고합니다.

# ### PA-C2. 모집단 불일치 정리 (S6 저자 층 vs S7 국가 층)

# - S6은 임베딩 보유 논문의 저자(76,984편), S7 국가 층은 창 안 전체(81,645편)입니다. 두 결과를 한 문장으로 엮으려면 **S7 국가 몫을 임베딩 보유 논문만으로 재계산한 값**을 부록에 병기해 차이가 ±0.01 이내인지 보여 줍니다.

# ---

# ## 실행 순서와 일정(제안)

# | 순서 | 항목 | 이유 | 난이도 |
# |---|---|---|---|
# | 1 | PA-0 재현성 정리 | 모든 PA의 전제 | 낮음 |
# | 2 | PA-0.5: `PA_0_PLAN.json` 동결 + 태그 | 사후분석의 신뢰 | 낮음 |
# | 3 | PA-A1 검정력 곡선, B1 문구 확정 | 핵심 주장의 방어선, 제목 결정에 영향 | 중간 |
# | 4 | PA-B2 (2020 하향 추출) | A1과 연결 | 중간 |
# | 5 | PA-A2 (ID 대조, 진입 정의 민감도, 부트스트랩 교정) | 중심 기여 강화 | 중간 |
# | 6 | PA-C1, C2 | 결과 간 정합성 | 중간 |
# | 7 | PA-B3, B4, B5, B7 | S3·S4 약점 리프레이밍 | 중간 |
# | 8 | PA-A3, B9, B8 | S7·S8 정리 | 중간 |
# | 9 | PA-B6 (ORCID·수작업) | 시간 소요, 병렬 가능 | 높음(인력) |
# | 10 | PA-A4 부록 표, 논문 문구 확정 | 마무리 | 낮음 |

# **우선순위 3개만 고른다면:** PA-A1(검정력), PA-A2(진입 주도 강화), PA-B2(2020 아티팩트 분리). 논문 서사의 중심이 이 세 곳에 걸려 있습니다.

# ---

# ## 사후분석 결과 보고 형식

# - **표지:** 모든 PA 표와 그림 제목에 `[Post-hoc]`을 붙입니다.
# - **CHANGELOG:** `PA_CHANGELOG.md`에 PA별로 "목적 / 정의 / 결과 / 본 결과 변경 여부(없음)"를 한 줄씩 기록합니다.
# - **해석 한계 문구(영문 원고용 예시):** "Post-hoc analyses were specified after inspecting the primary results and are reported as exploratory; no pre-registered decision rule was altered."
# - **금지:** PA 결과를 근거로 S0~S8의 통과·실패 판정을 바꾸는 것, PA에서 유리한 정의만 골라 본문에 올리는 것.

# ---

# ## 지금 필요한 입력

# 1. S5의 N2 시뮬레이션 출력이 파일로 저장돼 있는지(`regime_nulls.csv`의 분포 열), 그리고 `s5_lib.py`의 비용·벌점 함수 위치.
# 2. F6 생성 코드 위치(D1 재구성용)와 `(2) 데이터셋정의.ipynb`의 `BASE` 정의.
# 3. 연구자 시간 여건: PA-B6(수작업)과 PA-B8-2(새 표본 25편 코딩)를 할 수 있는지.
# 4. 위 우선순위(A1, A2, B2)로 먼저 가도 되는지, 또는 다른 약점을 우선하고 싶은지.

# 확인해 주시면 `PA_0_PLAN.json` 동결 스크립트와 PA-A1 코드부터 단일 셀로 드리겠습니다. 진단 출력(기존 N2 분포, 기간별 JS 점프 크기)을 먼저 붙여 주시면, 점프 크기 격자(δ)를 실측에 맞춰 정하겠습니다.

In [5]:
# -*- coding: utf-8 -*-
# =====================================================================================
# 논문 1 사후 교정분석 (PA) 단일 셀. 미실행 코드: QUICK=True로 먼저 시험.
# MODE: "freeze"(PA_0_PLAN.json 동결, 결과 미열람) -> "tag"(커밋·태그) -> "check"(PA-0 재현성 점검, 읽기 전용) -> "a1"(PA-A1 검정력 곡선)
# 원칙: S0~S8 판정 불변. 모든 산출물·표 제목에 [Post-hoc]. 해석은 '함께 움직인다'까지.
# 주의: S5 탐지기(s5_lib)를 첨부에서 못 봤으므로, a1은 자체 대체 탐지기(Hellinger 좌표 + 동적계획 분할)를 씀.
#       S5와 정확히 같은 통계가 아님. USE_S5_LIB 훅에 s5_lib 함수를 연결하면 그쪽을 씀.
# =====================================================================================
import json, hashlib, subprocess, warnings
from pathlib import Path
import numpy as np, pandas as pd
from scipy.spatial.distance import jensenshannon
warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)

MODE = "check"       # "freeze" | "tag" | "check" | "a1"
QUICK = True          # True: 귀무 B=200, 검정력 B=40 (파일명 _QUICK)

ROOT = Path.home() / "Research/Minnesota_Transport"
DATA, S0, S1, S3, PA = ROOT/"analysis_data/final", ROOT/"paper1/00_S0", ROOT/"paper1/01_S1", ROOT/"paper1/03_S3", ROOT/"paper1/09_PA"
now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
sha_b = lambda b: hashlib.sha256(b).hexdigest(); sha_f = lambda p: sha_b(Path(p).read_bytes())
tf = lambda s: s.astype(str).str.strip().str.lower().isin(["1", "1.0", "true", "t", "yes"])
git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
class Halt(Exception): pass

# ------------------------------ PA_0_PLAN (실행 전 동결) ------------------------------
PLAN = dict(
    version=1, seed=2026,
    status="사후분석. S0~S8 결과 열람 후 설계. 탐색적, 다중성 보정 없음. 사전 명시 판정(통과/실패)은 변경하지 않음.",
    n_pa=14, core_claim_pa=["PA-A1", "PA-A2", "PA-B2"],
    core_note="지침서 0절은 A1·A2·B1(문구 작업)을, 우선순위 절은 A1·A2·B2를 권장해 불일치. 분석 3개는 A1·A2·B2로 둠(B1은 A1 결과로 정하는 문구).",
    items={
        "PA-A1": "N2 검정력 곡선 + 구간별 상수 대 선형 BIC", "PA-A2": "S6 진입 우세의 구성 방어(ID 대조, 진입 정의 2종, 부트스트랩 교정)",
        "PA-A3": "S7 국가 미상 교정, 균등 4구간, 중국 제외 HHI", "PA-B1": "체제 서사 철회 문구 확정(PA-A1 연동, 분석 아님)",
        "PA-B2": "2020 변화점 아티팩트 분리(하향 추출, 구성 토픽)", "PA-B3": "신생 핵심 집합(n_defs>=3)과 D1'", "PA-B4": "F1+F3 이분 등급 재추정",
        "PA-B5": "S4 안정성 Wilson 구간, 분위 컷 곡선, 방향 축만", "PA-B6": "위험군 독립 정답(ORCID, 수작업 50쌍)", "PA-B7": "결측 경계 분석, 분류기 대입, 저널x연도 IPW",
        "PA-B8": "미달 4도메인 부록 정렬, 사전 v2(새 25편)", "PA-B9": "소득군 내 상관, 더미 통제, 공급 정의 민감도", "PA-C1": "S6x S7 교차(중국 소속 신규 제외 분해)", "PA-C2": "S7 국가 몫을 임베딩 보유 논문만으로 재계산"},
    pa_a1=dict(window=[2007, 2025], samples=["all", "fixed14", "within"], delta_x=[0.5, 1, 2, 3], t_star=[2008, 2012, 2015, 2020],
               B_null=1000, B_power=200, null="N2 = 표본별 선형 추세 적합 + 다항 표본 잡음", alpha=0.05,
               baseline="선형 추세 적합 확률(점프 없음)에 점프를 삽입. 점프 크기 = delta x 관측 2020 점프(JS 거리, 전3년 대 후3년 평균 비중), 삽입 방향 = 관측 2020 이후 평균 - 이전 평균",
               statistic="R = 1 - cost(k*)/cost(1). Hellinger 좌표 제약 분할(min_len 3, k<=6), k* = argmin cost_k + lam(k-1), lam = pen_c*cost_1/T, pen_c=0.5(대체 탐지기 값)",
               reading="power(1x)>=0.8: 연속 변화 증거로 강화 / power(1x)<0.5: '체제 전환 여부 판정 불가'로 리프레이밍 / 사이: 둘 다 쓰지 않고 곡선으로만 보고",
               within_rule="저널 내: 전 연도 연도당 5편 이상인 저널만, 고정 저널 비중 고정(wbar)",
               bic_compare="Hellinger 좌표 상위 5 PC에서 구간별 상수 대 구간별 선형, 최소 구간 3, k<=6, BIC"),
    forbidden="PA 결과로 S0~S8 판정 변경, PA에서 유리한 정의만 본문 게재",
    report_marker="[Post-hoc]", changelog="PA_CHANGELOG.md")
spec_hash = lambda s: sha_b(json.dumps({k: v for k, v in s.items() if k not in ("sha256", "created", "refs")}, sort_keys=True, ensure_ascii=False).encode())
PP = PA/"PA_0_PLAN.json"

def freeze():
    PA.mkdir(parents=True, exist_ok=True); h = spec_hash(PLAN); print("PA-0 해시(예정)", h[:12])
    if PP.exists(): old = json.loads(PP.read_text(encoding="utf-8")); print("이미 동결:", "동일" if old["sha256"] == h else "⛔ 다름(새 버전으로 분리하고 CHANGELOG에 사유)"); return
    refs = {f.name: sha_f(f) for f in [S0/"PREREG.json", S3/"L7_topic_grade.csv", ROOT/"paper1/07_S7/S7_0_PREREG.json", ROOT/"paper1/08_S8/S8_0_PREREG_v2.json"] if f.exists()}
    PP.write_text(json.dumps(dict(PLAN, created=now(), sha256=h, refs=refs), ensure_ascii=False, indent=1), encoding="utf-8")
    (PA/"PA_CHANGELOG.md").write_text(f"# PA_CHANGELOG\n\n- {now()} | PA-0 계획 동결(해시 {h[:12]}) | 목적: 사후분석 14개 항목·A1 정의·판독 규칙을 실행 전에 고정 | 본 결과 변경 여부: 없음\n", encoding="utf-8")
    print("저장: PA_0_PLAN.json, PA_CHANGELOG.md | refs", list(refs))

def tag():
    if not PP.exists(): raise Halt("먼저 freeze")
    t = f"PA-0-plan-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
    if git("tag", "-l", t).stdout.strip(): print("이미 있음:", t); return
    r = git("add", "--", "paper1/09_PA"); print("add", r.returncode, r.stderr.strip())
    r = git("commit", "-m", "PA-0 plan freeze (post-hoc, before execution)"); print("commit", r.returncode, (r.stdout + r.stderr)[:200])
    r = git("tag", "-a", t, "-m", "PA-0 plan"); print("tag", r.returncode, r.stderr.strip())

def check():   # PA-0 재현성 (읽기 전용: 삭제·수정 없음)
    print("[S4 코드]", {f: (ROOT/"paper1"/f).exists() for f in ["s4_main.py", "s4_6_rotation.py", "s4_6b_floor_check.py"]})
    for t in ["S4-main-2026-10-16", "S4-final-2026-10-16"]:
        print(f"\n[git show --stat {t}]\n", git("show", "--stat", "--format=%h %s (%ad)", t).stdout[:600])
    print("[git status paper1/]\n", git("status", "--short", "--", "paper1/").stdout[:1500] or "(깨끗함)")
    print("[원격 태그 미푸시 점검: 로컬 태그]", git("tag", "--sort=-creatordate").stdout.split()[:12])
    print("[origin 태그]", git("ls-remote", "--tags", "origin").stdout[:800] or "(원격 태그 없음 또는 origin 미설정)")
    print("[oa_raw 백업 존재]", [str(p) for p in (ROOT/"archive").glob("oa_raw_*.tar.gz")] if (ROOT/"archive").exists() else "archive/ 없음")
    print("※ 태그 정정은 직접: 가리키는 커밋 확인 -> git tag -d <이름> -> CHANGELOG에 정정 기록 -> 재태그 -> git push --follow-tags")

# ------------------------------ PA-A1 ------------------------------
USE_S5_LIB = None   # 예: from s5_lib import detect_R; USE_S5_LIB = detect_R  (시그니처 f(shares[T,K], years) -> float)
def load_F1X():
    F1 = pd.read_csv(DATA/"F1_papers.csv", low_memory=False); F1["pid"] = F1.pid.astype(str)
    exf = S0/"EXCLUDE_PIDS_FROZEN.csv"; log = pd.read_csv(S0/"FREEZE_LOG.csv"); last = log[log.file == exf.name].iloc[-1]
    if sha_f(exf) != last.sha256: raise Halt("EXCLUDE_PIDS_FROZEN.csv 해시 불일치")
    return F1[~F1.pid.isin(set(pd.read_csv(exf, dtype=str, encoding="utf-8-sig").pid))].copy()

def seg_costs(X, minlen=3, kmax=6):
    T = len(X); cs = np.vstack([np.zeros(X.shape[1]), np.cumsum(X, 0)]); sq = np.r_[0, np.cumsum((X**2).sum(1))]
    S = cs[None, :, :] - cs[:, None, :]; a = np.arange(T+1); L = (a[None, :] - a[:, None]).astype(float)
    C = (sq[None, :] - sq[:, None]) - (S**2).sum(2)/np.where(L > 0, L, 1); C[L < minlen] = np.inf
    best = np.full((kmax+1, T+1), np.inf); best[0, 0] = 0
    for k in range(1, kmax+1): best[k] = np.min(best[k-1][:, None] + C, axis=0)
    return best[:, T]
def detect_R(shares, years, pen_c=0.5, kmax=6):
    if USE_S5_LIB: return USE_S5_LIB(shares, years)
    c = seg_costs(np.sqrt(shares), 3, kmax); ks = np.arange(1, kmax+1); ok = np.isfinite(c[1:]); lam = pen_c*c[1]/len(shares)
    crit = np.where(ok, c[1:] + lam*(ks-1), np.inf); k = int(ks[np.argmin(crit)]); return 1 - c[k]/c[1]

def make_panel(sample, tops):
    tix = {t: i for i, t in enumerate(tops)}; K = len(tops); yrs = np.arange(2007, 2026)
    F = load_F1X(); F = F[F.year.between(2007, 2025) & tf(F.has_topic) & F.topic150.notna()]; F = F.assign(k=F.topic150.astype(int).map(tix)).dropna(subset=["k"])
    if sample == "fixed14":
        FIXED = set(json.loads((S0/"PREREG.json").read_text(encoding="utf-8"))["event"]["fixed_venues"]); F = F[F.venue_short.isin(FIXED)]
    if sample == "within":
        n = F.groupby(["venue_short", "year"]).size().unstack(fill_value=0).reindex(columns=yrs, fill_value=0); vs = list(n.index[(n >= 5).all(1)]); F = F[F.venue_short.isin(vs)]
    else: vs = ["ALL"]; F = F.assign(venue_short="ALL")
    C = np.zeros((len(yrs), len(vs), K)); np.add.at(C, (F.year.values-2007, F.venue_short.map({v: i for i, v in enumerate(vs)}).values, F.k.astype(int).values), 1)
    n = C.sum(2); W = (n/n.sum(1, keepdims=True)).mean(0); W = W/W.sum()    # 고정 저널 비중(within) 또는 [1]
    return yrs, C, n.astype(int), W, len(F)
def agg(P, W): return np.tensordot(P, W, axes=([1], [0])) if False else (P*W[None, :, None]).sum(1)
def draw(P, n, W, rng):
    S = np.zeros((P.shape[0], P.shape[2]))
    for y in range(P.shape[0]):
        for j in range(P.shape[1]):
            if n[y, j] > 0: S[y] += W[j]*rng.multinomial(n[y, j], P[y, j]/P[y, j].sum())/n[y, j]
    return S/W.sum()
def jump(S, i): pre, post = S[max(0, i-3):i].mean(0), S[i:i+3].mean(0); return float(jensenshannon(pre, post))

def a1():
    if not PP.exists(): raise Halt("먼저 freeze")
    sp = json.loads(PP.read_text(encoding="utf-8"))
    if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(PLAN): raise Halt("동결 파일과 현재 PLAN이 다릅니다(사후 변경 금지)")
    cfg = PLAN["pa_a1"]; sfx = "_QUICK" if QUICK else ""; Bn = 200 if QUICK else cfg["B_null"]; Bp = 40 if QUICK else cfg["B_power"]
    rng = np.random.default_rng(PLAN["seed"]); tops = np.array(sorted(pd.read_csv(S3/"L7_topic_grade.csv").topic.astype(int))); rows, info, bic_rows = [], [], []
    for sample in cfg["samples"]:
        yrs, C, n, W, npap = make_panel(sample, tops); T, J, K = C.shape; Pobs = np.divide(C, n[:, :, None], out=np.full(C.shape, 1.0/K), where=n[:, :, None] > 0)
        coef = np.polyfit(yrs, Pobs.reshape(T, -1), 1); Ptr = np.clip((coef[0]*yrs[:, None] + coef[1]).reshape(Pobs.shape), 1e-9, None); Ptr /= Ptr.sum(2, keepdims=True)
        Sobs, Str = agg(Pobs, W), agg(Ptr, W); i20 = list(yrs).index(2020); Jobs = jump(Sobs, i20); d = Sobs[i20:].mean(0) - Sobs[:i20].mean(0)
        Robs = detect_R(Sobs, yrs); Rnull = np.array([detect_R(draw(Ptr, n, W, rng), yrs) for _ in range(Bn)]); q95 = float(np.quantile(Rnull, 1-cfg["alpha"])); pobs = float((Rnull >= Robs).mean())
        info.append(dict(sample=sample, n_papers=npap, n_venues=J, J_obs_2020=Jobs, R_obs=Robs, null_q95=q95, p_N2_this_detector=pobs)); print(f"\n[{sample}] 논문 {npap:,}, 저널 {J}, 관측 2020 점프(JS) {Jobs:.4f}, R_obs {Robs:.3f}, 귀무 95% {q95:.3f}, p(이 탐지기) {pobs:.3f}")
        for ts in cfg["t_star"]:
            it = list(yrs).index(ts); base_j = jump(Str, it)
            for dx in cfg["delta_x"]:
                target = base_j + dx*Jobs; lo, hi = 0.0, 2.0/np.abs(d).max()
                def inj(a):
                    P = Ptr.copy(); P[it:] = np.clip(P[it:] + a*d[None, None, :], 1e-9, None); return P/P.sum(2, keepdims=True)
                if jump(agg(inj(hi), W), it) < target: rows.append(dict(sample=sample, t_star=ts, delta_x=dx, power=np.nan, note="목표 점프 도달 불가")); continue
                for _ in range(40):
                    mid = (lo+hi)/2; lo, hi = (mid, hi) if jump(agg(inj(mid), W), it) < target else (lo, mid)
                Pi = inj(hi); Rs = np.array([detect_R(draw(Pi, n, W, rng), yrs) for _ in range(Bp)])
                rows.append(dict(sample=sample, t_star=ts, delta_x=dx, power=float((Rs > q95).mean()), R_mean=float(Rs.mean()), jump_target=target, jump_base=base_j, note=""))
        # 구간별 상수 대 선형 BIC (상위 5 PC)
        Z = np.sqrt(Sobs); Z = Z - Z.mean(0); U, s, Vt = np.linalg.svd(Z, full_matrices=False); Y = (U*s)[:, :5]; N = Y.size
        def seg_rss(lin):
            R = np.full((T+1, T+1), np.inf)
            for a_ in range(T+1):
                for b_ in range(a_+3, T+1):
                    y = Y[a_:b_]; x = np.arange(b_-a_); A = np.c_[np.ones_like(x), x] if lin else np.ones((len(x), 1)); R[a_, b_] = ((y - A @ np.linalg.lstsq(A, y, rcond=None)[0])**2).sum()
            best = np.full((7, T+1), np.inf); best[0, 0] = 0
            for k in range(1, 7): best[k] = np.min(best[k-1][:, None] + R, axis=0)
            return best[:, T]
        for name, lin in [("constant", False), ("linear", True)]:
            rs = seg_rss(lin)
            for k in range(1, 7):
                if np.isfinite(rs[k]): bic_rows.append(dict(sample=sample, model=name, k=k, rss=rs[k], bic=N*np.log(rs[k]/N + 1e-12) + (k*5*(2 if lin else 1) + (k-1))*np.log(N)))
    PW, IN, BC = pd.DataFrame(rows), pd.DataFrame(info), pd.DataFrame(bic_rows)
    PW.to_csv(PA/f"PA_A1_power_curve{sfx}.csv", index=False, encoding="utf-8-sig"); IN.to_csv(PA/f"PA_A1_observed{sfx}.csv", index=False, encoding="utf-8-sig"); BC.to_csv(PA/f"PA_A1_bic{sfx}.csv", index=False, encoding="utf-8-sig")
    print("\n[Post-hoc] PA-A1 검정력(행: 표본 x t*, 열: delta x 관측 점프)\n", PW.pivot_table(index=["sample", "t_star"], columns="delta_x", values="power").round(2).to_string())
    one = PW[PW.delta_x == 1].groupby("sample").power.agg(["min", "median", "max"]).round(2); print("\n[1배 점프 검정력 요약]\n", one.to_string())
    for sname, r in one.iterrows():
        v = "강화(연속 변화 증거)" if r["min"] >= 0.8 else ("리프레이밍(판정 불가)" if r["median"] < 0.5 else "곡선으로만 보고(0.5~0.8 구간 또는 t*별 혼재)"); print(f"  {sname}: 사전 판독 규칙 → {v}")
    print("\n[Post-hoc] BIC 최솟값(표본별 상수 대 선형)\n", BC.loc[BC.groupby(["sample", "model"]).bic.idxmin()].round(1).to_string(index=False))
    print("\n※ 대체 탐지기·5 PC·within 정의는 S5와 같지 않음. 규칙 판독(min/median)은 PLAN의 reading을 코드로 옮긴 것이며, 경계 사례는 사람이 곡선을 보고 판단. S0~S8 판정은 바꾸지 않음.")
    if not QUICK:
        with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f: f.write(f"- {now()} | PA-A1 | 목적: N2 검정력 곡선 | 정의: PA_0_PLAN {sp['sha256'][:12]} | 결과: PA_A1_*.csv | 본 결과 변경 여부: 없음\n")

try: {"freeze": freeze, "tag": tag, "check": check, "a1": a1}[MODE]()
except Halt as e: print(f"⛔ 중단: {e}")

[S4 코드] {'s4_main.py': False, 's4_6_rotation.py': False, 's4_6b_floor_check.py': False}

[git show --stat S4-main-2026-10-16]
 tag S4-main-2026-10-16
Tagger: LEEYJ1021 <yongjaelee1021@gmail.com>

S4 main
33b3912 CHANGELOG_S3: S3-final 생성 순서 기록 (Fri Oct 16 11:07:38 2026 +0900)

 paper1/03_S3/CHANGELOG_S3.md | 1 +
 1 file changed, 1 insertion(+)


[git show --stat S4-final-2026-10-16]
 tag S4-final-2026-10-16
Tagger: LEEYJ1021 <yongjaelee1021@gmail.com>

S4 final (4.7은 S6 직전으로 이월)
33b3912 CHANGELOG_S3: S3-final 생성 순서 기록 (Fri Oct 16 11:07:38 2026 +0900)

 paper1/03_S3/CHANGELOG_S3.md | 1 +
 1 file changed, 1 insertion(+)

[git status paper1/]
  M paper1/01_S1/CHANGELOG_S1.md
 M paper1/03_S3/CHANGELOG_S3.md
 M paper1/04_S4/CHANGELOG_S4.md
 M paper1/06_S6/CHANGELOG_S6.md
 M paper1/06_S6/S6_MANIFEST.json
 M paper1/06_S6/S6_final_summary.md
 M paper1/07_S7/CHANGELOG_S7.md
 M "paper1/S0 \354\202\254\354\240\204\353\223\261\353\241\235\302\267\353\217\231\352\262\260.ipynb"
?? paper1/01_S1/S1_F

In [6]:
#

In [7]:
# -*- coding: utf-8 -*-
# PA-0 재현성 정리 단일 셀. MODE: "find"(S4 코드 탐색, 읽기 전용) -> "verify"(manifest 해시 검증, 읽기 전용) -> "commit"(정리 커밋·태그)
# commit은 DRY=True면 계획만 출력. 결과·판정은 바꾸지 않는 정리 작업.
import json, re, hashlib, sqlite3, tarfile, subprocess
from pathlib import Path
import pandas as pd

MODE = "find"
DRY = True
ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"
now = lambda: pd.Timestamp.now().isoformat(timespec="seconds"); d_ = pd.Timestamp.now().strftime("%Y-%m-%d")
sha_f = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
class Halt(Exception): pass
HEX = re.compile(r"^[0-9a-f]{64}$")

def find():
    pat = re.compile(r"s4_main|s4_6_rotation|s4_6b|floor_check|journal_decomposition|topic_quadrants")
    print("[paper1 직하 .py]", sorted(p.name for p in P1.glob("*.py")))
    hits = list(ROOT.rglob("s4*.py")) + list(ROOT.rglob("*s4*.ipynb"))
    print("[s4 이름 파일]", [str(p.relative_to(ROOT)) for p in hits if ".git" not in p.parts][:20])
    for nb in ROOT.rglob("*.ipynb"):
        if ".git" in nb.parts or "oa_raw" in nb.parts: continue
        try: cells = json.loads(nb.read_text(encoding="utf-8")).get("cells", [])
        except Exception: continue
        n = [i for i, c in enumerate(cells) if c.get("cell_type") == "code" and pat.search("".join(c.get("source", [])))]
        if n: print(f"[노트북 코드 셀 일치] {nb.relative_to(ROOT)} | 셀 {len(n)}개 (예: {n[:6]}) | 수정 {pd.Timestamp(nb.stat().st_mtime, unit='s'):%Y-%m-%d %H:%M}")
    hist = Path.home()/".ipython/profile_default/history.sqlite"
    if hist.exists():
        con = sqlite3.connect(hist); r = con.execute("select session, line, source from history").fetchall(); m = [(s, l) for s, l, src in r if pat.search(src or "")]
        print(f"[IPython 히스토리 일치] {len(m)}줄, 세션 {sorted({s for s, _ in m})[:8]} (sqlite 직접 조회로 복구 가능)")
    print("※ 코드가 나오면 paper1/s4_main.py 등으로 저장하고, 저장본 실행 여부와 무관하게 '산출물은 manifest 해시로 검증됨, 코드는 복구본'이라고 CHANGELOG에 적으세요.")

def collect_hashes(o, out, key=None):
    if isinstance(o, dict):
        for k, v in o.items():
            if isinstance(v, str) and HEX.match(v): out.append((k if "/" in str(k) or "." in str(k) else key, v))
            else: collect_hashes(v, out, k)
    elif isinstance(o, list):
        for v in o:
            if isinstance(v, dict) and "sha256" in v and any(f in v for f in ("file", "name", "path")): out.append((v.get("file") or v.get("name") or v.get("path"), v["sha256"]))
            else: collect_hashes(v, out, key)

def verify():
    tot = {"일치": 0, "불일치": [], "파일없음": [], "이름 해석 불가": 0}
    for mf in sorted(P1.glob("0[1-8]_*/*MANIFEST*.json")) + sorted(P1.glob("0[1-8]_*/*_meta*.json")):
        try: J = json.loads(mf.read_text(encoding="utf-8"))
        except Exception as e: print("읽기 실패", mf.name, e); continue
        pairs = []; collect_hashes(J, pairs)
        for name, h in pairs:
            if not name: tot["이름 해석 불가"] += 1; continue
            cand = [mf.parent/name, ROOT/name, P1/name]; f = next((c for c in cand if c.exists() and c.is_file()), None)
            if f is None: tot["파일없음"].append(f"{mf.name}:{name}"); continue
            if sha_f(f) == h: tot["일치"] += 1
            else: tot["불일치"].append(f"{mf.name}:{name}")
    print("[manifest 검증]", {k: (len(v) if isinstance(v, list) else v) for k, v in tot.items()})
    for k in ("불일치", "파일없음"):
        if tot[k]: print(f" {k} 예:", tot[k][:15])
    return tot

def commit():
    tot = verify()
    if tot["불일치"]: raise Halt("해시 불일치가 있습니다: 원인(동결 이후 파일 변경, 줄바꿈 등)을 확인한 뒤 진행하세요")
    gi = ROOT/".gitignore"; cur = gi.read_text(encoding="utf-8") if gi.exists() else ""
    add_ign = [x for x in ["paper1/01_S1/oa_raw/", "paper1/01_S1/_backup_before_income_tv/", "archive/*.tar.gz"] if x not in cur]
    sz = []
    for f in git("ls-files", "-o", "--exclude-standard", "paper1").stdout.splitlines():
        if any(f.startswith(x.rstrip("/*")) for x in add_ign if x.endswith("/")): continue
        s = (ROOT/f).stat().st_size
        if s > 30e6: sz.append((f, round(s/1e6, 1)))
    if sz: raise Halt(f"30MB 초과 미추적 파일(무시 목록 또는 LFS 검토): {sz}")
    tg = {t: git("rev-list", "-n", "1", t).stdout.strip() for t in ["S4-main-2026-10-16", "S4-final-2026-10-16"]}
    on_origin = git("ls-remote", "--tags", "origin").stdout
    del_t = [t for t, c in tg.items() if c.startswith("33b3912") and t not in on_origin]
    print("삭제할 잘못된 태그(로컬 전용, 33b3912 가리킴):", del_t); print(".gitignore 추가:", add_ign)
    print("커밋 1: paper1/02_S2 03_S3 04_S4 | 커밋 2: 나머지 paper1 변경(CHANGELOG, S1 파일, S0 노트북, 09_PA) + .gitignore")
    note = (f"- {now()} | 재현성 정리(PA-0): 잘못된 태그 S4-main-2026-10-16, S4-final-2026-10-16(실제로는 S3 CHANGELOG 커밋 33b3912을 가리킴, S4 산출물 미포함)을 로컬 삭제. "
            f"S2·S3·S4 산출물은 2026-10-16 이전에 생성됐으나 미추적이었고, 이번에 manifest 해시 {tot['일치']}건 일치를 확인한 뒤 {d_}에 처음 커밋. 커밋 날짜는 생성일이 아님 | 본 결과 변경: 없음\n")
    if DRY: print("\nDRY=True: 변경 없음. 위 계획이 맞으면 DRY=False로 다시 실행."); return
    for t in del_t: print("tag -d", t, git("tag", "-d", t).stdout.strip())
    if add_ign: gi.write_text(cur.rstrip("\n") + "\n" + "\n".join(add_ign) + "\n", encoding="utf-8")
    for f in [P1/"04_S4/CHANGELOG_S4.md", P1/"09_PA/PA_CHANGELOG.md"]:
        with f.open("a", encoding="utf-8") as h: h.write(note)
    r = git("add", "-A", "--", "paper1/02_S2", "paper1/03_S3", "paper1/04_S4"); print("add1", r.returncode, r.stderr.strip())
    r = git("commit", "-m", "Record S2/S3/S4 outputs (created before S5; manifest-verified, no result change)"); print("commit1", r.returncode, (r.stdout + r.stderr)[:160])
    r = git("tag", "-a", f"S4-results-recommit-{d_}", "-m", "S4(S2,S3) outputs first committed; verified against manifests"); print("tag", r.returncode, r.stderr.strip())
    r = git("add", "-A", "--", "paper1", ".gitignore"); print("add2", r.returncode, r.stderr.strip())
    r = git("commit", "-m", "PA-0 cleanup: changelogs, S1 files, gitignore for raw data"); print("commit2", r.returncode, (r.stdout + r.stderr)[:160])
    print("[tags]", git("tag", "--sort=-creatordate").stdout.split()[:8], "\n[미추적 잔여]", git("status", "--short", "--", "paper1").stdout[:300] or "없음")

try: {"find": find, "verify": verify, "commit": commit}[MODE]()
except Halt as e: print(f"⛔ 중단: {e}")

[paper1 직하 .py] ['s2_measurement.py', 's4_1_keywords.py', 's4_close.py', 's5_0_freeze.py', 's5_close.py', 's5_detect.py', 's5_lib.py', 's5_phi.py', 's6_0_freeze.py', 's6_close.py', 's6_lib.py', 's6_run.py', 's7_run.py']
[s4 이름 파일] ['paper1/s4_1_keywords.py', 'paper1/s4_close.py', 'paper1/.ipynb_checkpoints/s4_close-checkpoint.py']
[노트북 코드 셀 일치] paper1/S3. 토픽 증거 카드와 신뢰 등급.ipynb | 셀 5개 (예: [29, 30, 32, 33, 34]) | 수정 2026-10-16 02:21
[노트북 코드 셀 일치] paper1/S4. 신생·쇠퇴의 재정의.ipynb | 셀 3개 (예: [1, 3, 6]) | 수정 2026-10-16 02:44
[노트북 코드 셀 일치] paper1/S5. 체제 식별.ipynb | 셀 3개 (예: [0, 2, 7]) | 수정 2026-10-16 03:38
[노트북 코드 셀 일치] paper1/S6. 체제별 갱신 분해.ipynb | 셀 3개 (예: [0, 1, 2]) | 수정 2026-10-19 06:22
[노트북 코드 셀 일치] paper1/S7. 지역 집중과 신규 유입, S8. 공급-부담 정렬.ipynb | 셀 1개 (예: [0]) | 수정 2026-10-19 07:52
[노트북 코드 셀 일치] paper1/교정분석.ipynb | 셀 2개 (예: [0, 1]) | 수정 2026-10-19 08:11
[노트북 코드 셀 일치] paper1/.ipynb_checkpoints/S3. 토픽 증거 카드와 신뢰 등급-checkpoint.ipynb | 셀 5개 (예: [29, 30, 32, 33, 34]) | 수정 2026-10-16 02:21
[노트북 코드 셀 일치

In [9]:
# -*- coding: utf-8 -*-
# S4 코드 복구 + S5 연결 정보 점검. 새 폴더 paper1/_recovered_s4/ 에만 쓰기. 기존 파일 수정 없음.
import json, re, sqlite3, ast
from pathlib import Path
import pandas as pd
MODE = "recover"      # "recover" | "s5info"
ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; OUT = P1/"_recovered_s4"

def recover():
    OUT.mkdir(exist_ok=True); pat = re.compile(r"s4_main|s4_6_rotation|s4_6b|floor_check|journal_decomposition|topic_quadrants|rotation_index")
    nb = P1/"S4. 신생·쇠퇴의 재정의.ipynb"; cells = json.loads(nb.read_text(encoding="utf-8"))["cells"]
    print(f"[노트북] {nb.name} | 전체 셀 {len(cells)} | 수정 {pd.Timestamp(nb.stat().st_mtime, unit='s'):%Y-%m-%d %H:%M}")
    for i, c in enumerate(cells):
        if c.get("cell_type") != "code": continue
        src = "".join(c.get("source", []))
        (OUT/f"nb_cell_{i:02d}.py").write_text(src, encoding="utf-8")
        out = "".join("".join(o.get("text", [])) for o in c.get("outputs", []) if o.get("output_type") == "stream")
        first = src.strip().splitlines()[0][:110] if src.strip() else ""
        flag = "◀ 일치" if pat.search(src) else ""
        try: ast.parse(src); ok = "구문 OK"
        except SyntaxError: ok = "구문 오류(매직 명령 포함 가능)"
        print(f"  셀 {i:02d} | {len(src):6,}자 | 출력 {len(out):6,}자 | {ok} | {first} {flag}")
    h = Path.home()/".ipython/profile_default/history.sqlite"
    if h.exists():
        con = sqlite3.connect(h); rows = [(s, l, src) for s, l, src in con.execute("select session, line, source from history order by session, line") if pat.search(src or "")]
        for s, l, src in rows: (OUT/f"hist_{s}_{l:04d}.py").write_text(src, encoding="utf-8")
        print(f"\n[IPython 히스토리] {len(rows)}줄 저장: " + ", ".join(f"{s}:{l}({len(src):,}자)" for s, l, src in rows))
    print(f"\n저장 위치: {OUT}\n※ 셀·히스토리는 수정본이 여러 번 섞여 있을 수 있음: 어느 것이 '산출물을 만든 최종 실행본'인지는 파일의 s4_meta.json·MANIFEST 시각과 비교해 사람이 확정.")
    for f in ["S4_meta.json", "S4_MANIFEST.json"]:
        p = P1/"04_S4"/f
        if p.exists(): print(f"  {f} 수정 {pd.Timestamp(p.stat().st_mtime, unit='s'):%Y-%m-%d %H:%M:%S}")

def s5info():
    for fn in ["s5_lib.py", "s5_detect.py"]:
        t = (P1/fn).read_text(encoding="utf-8"); tree = ast.parse(t); print(f"\n[{fn}] {len(t.splitlines())}줄")
        for n in tree.body:
            if isinstance(n, ast.FunctionDef): print(f"  def {n.name}({', '.join(a.arg for a in n.args.args)})  # {(ast.get_docstring(n) or '').splitlines()[0][:70] if ast.get_docstring(n) else ''}")
            elif isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id.isupper(): print(f"  {n.targets[0].id} = {ast.unparse(n.value)[:90]}")
    for p in list((P1/"05_S5").glob("regime_nulls*.csv")) + list((P1/"05_S5").glob("regime_breaks*.csv")):
        d = pd.read_csv(p, nrows=3); print(f"\n[{p.name}] 열: {list(d.columns)}")
    print("\n※ 위 함수 목록(특히 비용·벌점·귀무 시뮬레이션 관련)을 붙여 주세요.")

{"recover": recover, "s5info": s5info}[MODE]()

[노트북] S4. 신생·쇠퇴의 재정의.ipynb | 전체 셀 7 | 수정 2026-10-16 02:44
  셀 00 |  4,141자 | 출력      0자 | 구문 OK | # # S4 신생·쇠퇴의 재정의: 스텝별 지침 
  셀 01 |  6,628자 | 출력  1,681자 | 구문 OK | # -*- coding: utf-8 -*- ◀ 일치
  셀 02 |      1자 | 출력      0자 | 구문 OK | # 
  셀 03 |  4,278자 | 출력  1,835자 | 구문 OK | # -*- coding: utf-8 -*- ◀ 일치
  셀 04 |      1자 | 출력      0자 | 구문 OK | # 
  셀 05 |  2,267자 | 출력      0자 | 구문 OK | # # 4.6b 해석: 이전 "연 단위 신호 없음"은 바닥 정의 탓이었습니다 
  셀 06 |  3,606자 | 출력      0자 | 구문 OK | # # S4 진행 현황 업데이트 (2026-10-16 기준) ◀ 일치

[IPython 히스토리] 13줄 저장: 1079:23(12,689자), 1079:25(6,310자), 1079:26(4,399자), 1080:1(6,295자), 1080:3(5,307자), 1080:4(6,628자), 1080:6(4,278자), 1083:1(4,720자), 1084:1(13,597자), 1084:3(13,594자), 1084:5(13,596자), 1084:7(6,278자), 1084:9(2,948자)

저장 위치: /home/yjlee/Research/Minnesota_Transport/paper1/_recovered_s4
※ 셀·히스토리는 수정본이 여러 번 섞여 있을 수 있음: 어느 것이 '산출물을 만든 최종 실행본'인지는 파일의 s4_meta.json·MANIFEST 시각과 비교해 사람이 확정.
  S4_meta.json 수정 2026-10-16 02:10:54
  S4_MANIFEST.json 수정 2026-10-16 02:42:57


In [11]:
# -*- coding: utf-8 -*-
# S4 코드 후보 판정표. 기본은 읽기 전용(SAVE=False). 모든 시각은 로컬(KST) 기준으로 통일.
import re, sqlite3, json, datetime as dt
from pathlib import Path
import pandas as pd
SAVE = False
ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; S4 = P1/"04_S4"; REC = P1/"_recovered_s4"
loc = lambda p: dt.datetime.fromtimestamp(Path(p).stat().st_mtime)
SIG = {"s4_main.py": ["journal_decomposition.csv", "topic_quadrants.csv", "S4_meta.json"],
       "s4_6_rotation.py": ["rotation_index_periods.csv"],
       "s4_6b_floor_check.py": ["yearly_js_poolfloor_check.csv"]}
# 1) 산출물 수정 시각(로컬)
OUT_T = {f: loc(S4/f) for fs in SIG.values() for f in fs if (S4/f).exists()}
print("[산출물 수정 시각(로컬)]"); [print(f"  {f:34s} {t:%Y-%m-%d %H:%M:%S}") for f, t in sorted(OUT_T.items(), key=lambda x: x[1])]
# 2) 히스토리 세션 시각
con = sqlite3.connect(Path.home()/".ipython/profile_default/history.sqlite"); cols = [r[1] for r in con.execute("pragma table_info(sessions)")]
sess = {r[0]: (pd.Timestamp(r[1]).to_pydatetime() if r[1] else None, pd.Timestamp(r[2]).to_pydatetime() if r[2] else None) for r in con.execute("select session, start, end from sessions")}
rows = []
for p in sorted(REC.glob("hist_*.py")):
    s, l = map(int, re.findall(r"hist_(\d+)_(\d+)", p.name)[0]); src = p.read_text(encoding="utf-8"); st, en = sess.get(s, (None, None))
    rows.append(dict(src=p.name, session_start=st, session_end=en, chars=len(src), **{k: [f for f in v if f in src] for k, v in SIG.items()}))
for i in (1, 3):
    p = REC/f"nb_cell_{i:02d}.py"; src = p.read_text(encoding="utf-8")
    rows.append(dict(src=p.name, session_start=None, session_end=loc(P1/"S4. 신생·쇠퇴의 재정의.ipynb"), chars=len(src), **{k: [f for f in v if f in src] for k, v in SIG.items()}))
T = pd.DataFrame(rows); pd.set_option("display.width", 250, "display.max_colwidth", 60)
print("\n[후보 표: 열 s4_*.py 는 그 코드 안에 들어 있는 해당 산출물 파일명]\n", T.to_string(index=False))
# 3) 규칙: 서명 파일명을 포함하고, 산출물 최초 수정 시각 이전에 시작한 후보 중 가장 늦은 것
for tgt, fs in SIG.items():
    t_out = min(OUT_T[f] for f in fs if f in OUT_T) if any(f in OUT_T for f in fs) else None
    c = T[T[tgt].map(len) > 0].copy()
    if t_out is not None: c = c[c.session_start.isna() | (c.session_start <= t_out)]
    print(f"\n[{tgt}] 산출물 시각 {t_out} | 후보 {len(c)}개:", list(c.src))
    if SAVE and len(c):
        pick = c.sort_values(["session_start", "src"], na_position="first").iloc[-1].src
        body = (REC/pick).read_text(encoding="utf-8")
        (P1/tgt).write_text(f"# 복구본: {pick} (S4 산출물 시각 이전의 후보 중 규칙 선택; 2026-10-19 복구, 최초 실행은 2026-10-16)\n# 최종 실행본임을 증명하지는 못함: 산출물은 MANIFEST 해시로 검증\n{body}", encoding="utf-8"); print("  저장:", tgt, "←", pick)
print("\n※ 규칙은 추정입니다. 후보가 2개 이상이면 diff로 차이를 보고 사람이 확정하세요.")

[산출물 수정 시각(로컬)]
  topic_quadrants.csv                2026-10-16 11:10:54
  journal_decomposition.csv          2026-10-16 11:10:54
  S4_meta.json                       2026-10-16 11:10:54
  rotation_index_periods.csv         2026-10-16 11:26:56
  yearly_js_poolfloor_check.csv      2026-10-16 11:29:15

[후보 표: 열 s4_*.py 는 그 코드 안에 들어 있는 해당 산출물 파일명]
               src              session_start                session_end  chars                                                     s4_main.py             s4_6_rotation.py            s4_6b_floor_check.py
hist_1079_0023.py 2026-10-16 10:41:56.134582                        NaT  12689 [journal_decomposition.csv, topic_quadrants.csv, S4_meta.json]                           []                              []
hist_1079_0025.py 2026-10-16 10:41:56.134582                        NaT   6310                                                             [] [rotation_index_periods.csv]                              []
hist_1079_0026.py 2026-10-16 10:41:56.13458

In [12]:
#

In [13]:
# -*- coding: utf-8 -*-
# S4 복구본 검증 + 저장. 기본 SAVE=False(읽기 전용). 이미 있는 파일은 덮어쓰지 않음.
import json, difflib
from pathlib import Path
SAVE = False
ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; REC = P1/"_recovered_s4"; S4 = P1/"04_S4"
rd = lambda n: (REC/n).read_text(encoding="utf-8")
PICK = {"s4_main.py": "hist_1079_0023.py", "s4_6_rotation.py": "hist_1080_0004.py", "s4_6b_floor_check.py": "hist_1080_0006.py"}
NBC = {"s4_6_rotation.py": "nb_cell_01.py", "s4_6b_floor_check.py": "nb_cell_03.py"}

# 1) 선택본 = 노트북 셀(같은 코드)인가
for tgt, nb in NBC.items():
    a, b = rd(PICK[tgt]).strip(), rd(nb).strip(); r = difflib.SequenceMatcher(None, a, b).ratio()
    print(f"[{tgt}] {PICK[tgt]} vs {nb}: {'동일' if a == b else f'차이 있음(유사도 {r:.4f})'}")
    if a != b:
        d = [l for l in difflib.unified_diff(a.splitlines(), b.splitlines(), lineterm="", n=0) if not l.startswith(("---", "+++", "@@"))]; print("   차이 줄 수", len(d), "| 예:", d[:6])

# 2) 후보 간 차이(rotation): 앞선 수정본과 무엇이 다른가
base = rd("hist_1080_0004.py").splitlines()
for other in ["hist_1079_0025.py", "hist_1080_0001.py"]:
    d = [l for l in difflib.unified_diff(rd(other).splitlines(), base, lineterm="", n=0) if not l.startswith(("---", "+++", "@@"))]
    print(f"\n[rotation] {other} -> 선택본: 변경 줄 {len(d)}개 (예)"); [print("  ", l[:150]) for l in d[:12]]

# 3) 노트북 셀의 저장된 출력과 04_S4/s4_6_output.txt 일치 여부
nb = json.loads((P1/"S4. 신생·쇠퇴의 재정의.ipynb").read_text(encoding="utf-8"))["cells"]
for i, f in [(1, "s4_6_output.txt")]:
    o = "".join("".join(x.get("text", [])) for x in nb[i].get("outputs", []) if x.get("output_type") == "stream").strip(); t = (S4/f).read_text(encoding="utf-8", errors="replace").strip()
    print(f"\n[셀 {i:02d} 출력 vs {f}] {'동일' if o == t else '다름'} | 길이 {len(o)} / {len(t)}")
    if o != t:
        k = next((j for j, (x, y) in enumerate(zip(o, t)) if x != y), min(len(o), len(t))); print("   첫 차이 위치", k, "| 노트북:", repr(o[k:k+60]), "| 파일:", repr(t[k:k+60]))

# 4) 저장(기존 파일 보호)
if SAVE:
    for tgt, src in PICK.items():
        p = P1/tgt
        if p.exists(): print("건너뜀(이미 있음):", tgt); continue
        hdr = (f"# 복구본: {src} (+ 노트북 셀 교차 확인). 2026-10-19 복구. 최초 실행은 2026-10-16 (산출물 시각 이전 세션)\n"
               f"# 최종 실행본임을 증명하지는 못함: 산출물은 S4_MANIFEST.json 해시로 검증. 재실행하지 않음\n")
        p.write_text(hdr + rd(src), encoding="utf-8"); print("저장:", tgt, "←", src)

[s4_6_rotation.py] hist_1080_0004.py vs nb_cell_01.py: 동일
[s4_6b_floor_check.py] hist_1080_0006.py vs nb_cell_03.py: 동일

[rotation] hist_1079_0025.py -> 선택본: 변경 줄 38개 (예)
   -W["topic"] = W.topic150.astype(int); W["fixed"] = tf(W.fixed_venue)
   +PRE = json.loads((ROOT / "paper1/00_S0/PREREG.json").read_text(encoding="utf-8"))
   +def _find(o):
   +    if isinstance(o, dict):
   +        for k, v in o.items():
   +            if k == "fixed_venues": return v
   +            r = _find(v)
   +            if r is not None: return r
   +FIXED = set(_find(PRE)); assert len(FIXED) == 14, len(FIXED)
   +W["topic"] = W.topic150.astype(int); W["fixed"] = W.venue_short.isin(FIXED)
   -# 연도 x 저널 x 토픽 개수, 저널 가중(전 기간 평균, 저널 내 성분 시계열용)
   +print(f"저널 {len(venues)}개 | 고정 저널로 판정된 수 {int(fixed_v.sum())} (14여야 함)")

[rotation] hist_1080_0001.py -> 선택본: 변경 줄 10개 (예)
   -W["topic"] = W.topic150.astype(int); W["fixed"] = tf(W.fixed_venue)
   +PRE = json.loads((ROOT / "paper1/00_S0/PREREG.json").read_text(e

In [15]:
# -*- coding: utf-8 -*-
# S4 복구본 저장 + CHANGELOG 기록 + 출력 파일 점검 + manifest 해시 검증. DRY=True면 쓰기 없음.
import json, re, hashlib
from pathlib import Path
import pandas as pd

DRY = True
ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; REC = P1/"_recovered_s4"; S4 = P1/"04_S4"
now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
sha_f = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
rd = lambda n: (REC/n).read_text(encoding="utf-8")
PICK = {"s4_main.py": "hist_1079_0023.py", "s4_6_rotation.py": "hist_1080_0004.py", "s4_6b_floor_check.py": "hist_1080_0006.py"}
MARK = "S4 코드 복구:"
LOG_RECOVER = (f"- {now()} | S4 코드 복구: s4_main.py(←IPython 히스토리 1079:23), s4_6_rotation.py(←1080:4, 노트북 셀 01과 동일), s4_6b_floor_check.py(←1080:6, 노트북 셀 03과 동일). "
               "2026-10-16 최초 실행 시 .py로 저장되지 않았음. 선택 근거: 산출물 생성 시각 이전 세션, 산출 파일명 포함, 노트북 셀 01 출력(저널 34개, 고정 저널 14개)이 선택본과 일치. "
               "최종 실행본임은 증명하지 못함. 재실행하지 않음. 산출물 무결성은 S4_MANIFEST.json 해시로 검증 | 본 결과 변경: 없음\n")
LOG_NOTES = (f"- {now()} | S4 정정 이력: 4.6 초기 수정본(1079:25, 1080:1)은 고정 저널을 fixed_venue 열로 판정했고 선택본은 PREREG.json의 fixed_venues 14개로 판정함(산출물은 후자). "
             "04_S4/s4_6_output.txt는 스크립트 경로 오류 메시지(128자)이며 실제 4.6 출력이 아님: 실제 출력은 S4 노트북 셀 01. "
             "태그 S4-main-2026-10-16, S4-final-2026-10-16(33b3912, 11:07:38)은 S4 산출물(11:10:54 이후 생성) 이전 커밋을 가리켜 삭제 예정 | 본 결과 변경: 없음\n")

# 1) 복구본 저장
print("[1. 복구본 저장]")
for tgt, src in PICK.items():
    p = P1/tgt
    if p.exists(): print(f"  건너뜀(이미 있음): {tgt}"); continue
    if not (REC/src).exists(): print(f"  ⛔ 원본 없음: {src}"); continue
    hdr = (f"# 복구본: {src} (+ 노트북 셀 교차 확인). 2026-10-19 복구. 최초 실행은 2026-10-16 (산출물 시각 이전 세션)\n"
           f"# 최종 실행본임을 증명하지는 못함: 산출물은 S4_MANIFEST.json 해시로 검증. 재실행하지 않음\n")
    print(f"  {'저장 예정' if DRY else '저장'}: {tgt} ← {src} ({len(rd(src)):,}자)")
    if not DRY: p.write_text(hdr + rd(src), encoding="utf-8")

# 2) CHANGELOG 기록(중복 방지)
print("\n[2. CHANGELOG]")
for f in [P1/"04_S4/CHANGELOG_S4.md", P1/"09_PA/PA_CHANGELOG.md"]:
    txt = f.read_text(encoding="utf-8") if f.exists() else ""
    if MARK in txt: print(f"  이미 기록됨: {f.name}"); continue
    print(f"  {'추가 예정' if DRY else '추가'}: {f.name}")
    if not DRY:
        with f.open("a", encoding="utf-8") as h: h.write(("" if txt.endswith("\n") or not txt else "\n") + LOG_RECOVER + LOG_NOTES)

# 3) 출력 파일 점검: 에러 로그인데 증거처럼 보이는 파일 찾기
print("\n[3. 04_S4/*_output.txt 점검]")
for p in sorted(S4.glob("*_output.txt")):
    t = p.read_text(encoding="utf-8", errors="replace"); err = bool(re.search(r"can't open file|Traceback|No such file|Error", t))
    print(f"  {p.name}: {len(t):,}자 | {'⛔ 오류 메시지로 보임' if err else '정상 출력으로 보임'} | 첫 줄: {t.strip().splitlines()[0][:90] if t.strip() else '(빈 파일)'}")

# 4) manifest 해시 검증(읽기 전용)
HEX = re.compile(r"^[0-9a-f]{64}$")
def collect(o, out, key=None):
    if isinstance(o, dict):
        for k, v in o.items():
            if isinstance(v, str) and HEX.match(v): out.append((k if ("/" in str(k) or "." in str(k)) else key, v))
            else: collect(v, out, k)
    elif isinstance(o, list):
        for v in o:
            if isinstance(v, dict) and "sha256" in v and any(f in v for f in ("file", "name", "path")): out.append((v.get("file") or v.get("name") or v.get("path"), v["sha256"]))
            else: collect(v, out, key)
tot = {"일치": 0, "불일치": [], "파일없음": [], "이름 해석 불가": 0}
for mf in sorted(P1.glob("0[1-8]_*/*MANIFEST*.json")) + sorted(P1.glob("0[1-8]_*/*_meta*.json")):
    try: J = json.loads(mf.read_text(encoding="utf-8"))
    except Exception as e: print("  읽기 실패", mf.name, e); continue
    pairs = []; collect(J, pairs)
    for name, h in pairs:
        if not name: tot["이름 해석 불가"] += 1; continue
        f = next((c for c in [mf.parent/name, ROOT/name, P1/name] if c.exists() and c.is_file()), None)
        if f is None: tot["파일없음"].append(f"{mf.name}:{name}"); continue
        (tot.__setitem__("일치", tot["일치"] + 1) if sha_f(f) == h else tot["불일치"].append(f"{mf.name}:{name}"))
print("\n[4. manifest 검증]", {k: (len(v) if isinstance(v, list) else v) for k, v in tot.items()})
for k in ("불일치", "파일없음"):
    if tot[k]: print(f"  {k}({len(tot[k])}건):", tot[k][:20])
print("\nDRY=True: 쓰기 없음. 계획이 맞으면 DRY=False로 다시 실행." if DRY else "\n완료. 다음: PA-0 정리 셀의 MODE='commit'(DRY=True 먼저).")

[1. 복구본 저장]
  저장 예정: s4_main.py ← hist_1079_0023.py (12,689자)
  저장 예정: s4_6_rotation.py ← hist_1080_0004.py (6,628자)
  저장 예정: s4_6b_floor_check.py ← hist_1080_0006.py (4,278자)

[2. CHANGELOG]
  추가 예정: CHANGELOG_S4.md
  추가 예정: PA_CHANGELOG.md

[3. 04_S4/*_output.txt 점검]
  s4_1_output.txt: 649자 | 정상 출력으로 보임 | 첫 줄: [인접 기간 top-10 Jaccard: 쌍별 중앙값 (전체 / A등급)]
  s4_6_output.txt: 129자 | ⛔ 오류 메시지로 보임 | 첫 줄: python3: can't open file '/home/yjlee/Research/Minnesota_Transport/paper1/s4_6_rotation.py

[4. manifest 검증] {'일치': 144, '불일치': 0, '파일없음': 0, '이름 해석 불가': 22}

DRY=True: 쓰기 없음. 계획이 맞으면 DRY=False로 다시 실행.


In [16]:
import json, re
from pathlib import Path
P1 = Path.home() / "Research/Minnesota_Transport/paper1"; HEX = re.compile(r"^[0-9a-f]{64}$")
def walk(o, path, out):
    if isinstance(o, dict):
        for k, v in o.items():
            if isinstance(v, str) and HEX.match(v) and not ("/" in str(k) or "." in str(k)): out.append((path + "/" + str(k)))
            else: walk(v, path + "/" + str(k), out)
    elif isinstance(o, list):
        for i, v in enumerate(o): walk(v, f"{path}[{i}]", out)
for mf in sorted(P1.glob("0[1-8]_*/*MANIFEST*.json")) + sorted(P1.glob("0[1-8]_*/*_meta*.json")):
    out = []; walk(json.loads(mf.read_text(encoding="utf-8")), "", out)
    if out: print(mf.name, out[:6])

S3_MANIFEST.json ['/spec_sha256']
S4_MANIFEST.json ['/spec_sha256']
L1_meta.json ['/spec_sha256']
L2_meta.json ['/spec_sha256', '/output_sha256']
L3_meta.json ['/spec_sha256', '/output_sha256']
L4_meta.json ['/spec_sha256', '/output_sha256']
L5_meta.json ['/spec_sha256', '/manual_meta_sha256', '/manual_decision_sha256', '/output_sha256']
L5_topic38_49_meta.json ['/label_sha256', '/spec_sha256']
L7_meta.json ['/spec_sha256', '/output_sha256']
S4_meta.json ['/spec_sha256']
S5_meta.json ['/spec_sha256']
S7_meta.json ['/spec_sha256']
S7_meta_QUICK.json ['/spec_sha256']
S8_meta.json ['/spec_sha256']


In [17]:
import json, re, hashlib
from pathlib import Path
P1 = Path.home() / "Research/Minnesota_Transport/paper1"; HEX = re.compile(r"^[0-9a-f]{64}$")
sha_f = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
TARGET = ("output_sha256", "manual_meta_sha256", "manual_decision_sha256", "label_sha256")
# 1) 대상 해시 수집
want = []
for mf in sorted(P1.glob("0[1-8]_*/*_meta*.json")):
    J = json.loads(mf.read_text(encoding="utf-8"))
    for k in TARGET:
        if isinstance(J.get(k), str) and HEX.match(J[k]): want.append((mf, k, J[k]))
# 2) 해당 폴더(01~08)의 파일 전체 해시 인덱스(체크포인트·oa_raw 제외, 200MB 초과 제외)
idx = {}
for f in P1.glob("0[1-8]_*/**/*"):
    if f.is_file() and ".ipynb_checkpoints" not in f.parts and "oa_raw" not in f.parts and f.stat().st_size < 200e6:
        idx.setdefault(sha_f(f), []).append(f.relative_to(P1).as_posix())
# 3) 대조
ok = miss = 0
for mf, k, h in want:
    hit = idx.get(h)
    print(f"{mf.name:26s} {k:24s} {'일치 → ' + ', '.join(hit[:2]) if hit else '⛔ 같은 해시의 파일 없음'}")
    ok += bool(hit); miss += not hit
print(f"\n파일 해시 일치 {ok}건 / 미일치 {miss}건 (대상 {len(want)}건)")

L2_meta.json               output_sha256            일치 → 03_S3/L2_representative_papers.csv
L3_meta.json               output_sha256            일치 → 03_S3/L3_topic_metrics.csv
L4_meta.json               output_sha256            일치 → 03_S3/L4_structure_fingerprint.csv
L5_meta.json               output_sha256            일치 → 03_S3/L5_topic_flags.csv
L5_meta.json               manual_meta_sha256       일치 → 03_S3/L5_topic38_49_meta.json
L5_meta.json               manual_decision_sha256   일치 → 03_S3/L5_topic38_49_decision.csv
L5_topic38_49_meta.json    label_sha256             일치 → 02_S2/s2_open_topic38_49_manual.csv
L7_meta.json               output_sha256            일치 → 03_S3/L7_topic_grade.csv

파일 해시 일치 8건 / 미일치 0건 (대상 8건)


In [18]:
from pathlib import Path
import pandas as pd
f = Path.home() / "Research/Minnesota_Transport/paper1/09_PA/PA_CHANGELOG.md"
line = (f"- {pd.Timestamp.now().isoformat(timespec='seconds')} | PA-0 해시 검증: S1~S8 manifest·meta의 파일 해시 152건이 현재 파일과 일치(불일치 0, 파일없음 0). "
        "규칙 해시(spec_sha256) 14건은 단계별 spec_hash 재계산으로 검증. 코드 복구 3종과 무관하게 산출물 무결성은 해시로 확인됨 | 본 결과 변경: 없음\n")
t = f.read_text(encoding="utf-8")
if "PA-0 해시 검증:" not in t:
    f.write_text(t + ("" if t.endswith("\n") else "\n") + line, encoding="utf-8"); print("추가")
else: print("이미 있음")

추가


In [19]:
# -*- coding: utf-8 -*-
# S5 연결 정보 추출 (읽기 전용). s5_lib/s5_detect에서 PA-A1에 필요한 함수(비용, 벌점, N2 귀무, 변화점)의 시그니처와 소스를 출력.
import ast, re
from pathlib import Path
import pandas as pd
ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"
KEY = re.compile(r"null|trend|n2|cost|penal|pen|segment|dp|detect|break|change|bic|simul|perm|js|hellinger|phi|share", re.I)
MAXLINES = 45

def show(fn):
    t = (P1/fn).read_text(encoding="utf-8"); lines = t.splitlines(); tree = ast.parse(t)
    print(f"\n{'='*20} {fn} ({len(lines)}줄) {'='*20}")
    print("[최상위 상수]")
    for n in tree.body:
        if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id.isupper(): print(f"  {n.targets[0].id} = {ast.unparse(n.value)[:120]}")
    print("[함수 목록]")
    funcs = [n for n in tree.body if isinstance(n, ast.FunctionDef)]
    for n in funcs: print(f"  L{n.lineno:<4} def {n.name}({', '.join(a.arg for a in n.args.args)})")
    print(f"\n[핵심 후보 함수 소스: 이름이 /{KEY.pattern[:40]}.../ 에 걸리는 것, 각 {MAXLINES}줄까지]")
    for n in funcs:
        if KEY.search(n.name):
            seg = lines[n.lineno-1: n.end_lineno]; print(f"\n--- {n.name} (L{n.lineno}-{n.end_lineno}) ---"); print("\n".join(seg[:MAXLINES]) + ("\n   ...(생략)" if len(seg) > MAXLINES else ""))

for fn in ["s5_lib.py"]: show(fn)
print(f"\n{'='*20} s5_detect.py: main 흐름 요약 {'='*20}")
t = (P1/"s5_detect.py").read_text(encoding="utf-8"); L = t.splitlines(); print(f"{len(L)}줄")
for i, l in enumerate(L, 1):
    if re.search(r"^\s*(def |class )|N2|null|trend|from s5_lib|import s5_lib|PEN|KMAX|MIN_LEN|min_len|k_max|penalty|\.to_csv", l): print(f"  L{i:<4} {l.strip()[:150]}")

print(f"\n{'='*20} S5 산출 파일 {'='*20}")
for p in sorted((P1).glob("05_S5/*.csv")):
    try: d = pd.read_csv(p, nrows=2); print(f"  {p.name}: 열 {list(d.columns)[:14]}")
    except Exception as e: print("  읽기 실패", p.name, e)
for p in sorted(P1.glob("05_S5/*_meta*.json")) + sorted(P1.glob("05_S5/*PREREG*.json")): print("  json:", p.name)
print("\n※ 출력 전체를 붙여 주세요. 길면 '핵심 후보 함수 소스' 부분과 '산출 파일' 부분만이라도 주세요.")


==================== s5_lib.py (164줄) ====================
[최상위 상수]
  ROOT = Path.home() / 'Research/Minnesota_Transport'
  YEARS = np.arange(2000, 2026)
  SAMPLES = ['all', 'fixed14', 'within']
[함수 목록]
  L17   def find_key(o, key)
  L24   def load_counts()
  L36   def sample_P(cnt, fixed_v, wbar, kind, tmask)
  L47   def detrend_P(P, n)
  L55   def seg_costs(X, w)
  L66   def segment(X, w, kmax, minlen)
  L82   def seg_means(X, w, br)
  L86   def jump(X, w, br)
  L90   def cost_given(X, w, br)
  L97   def detect(P, n, pen, kmax, minlen, keep)
  L105  def uni_bic(v, kmax, minlen)
  L111  def near(y, ys, tol)
  L114  def gen_boot(cnt, rng)
  L119  def gen_N0(cnt, rng)
  L126  def gen_N1(cnt, rng)
  L136  def trend_probs(cnt)
  L149  def gen_N2(cnt, P2, rng)
  L156  def decomp_break(cnt, a0, a1, b1)

[핵심 후보 함수 소스: 이름이 /null|trend|n2|cost|penal|pen|segment|dp|.../ 에 걸리는 것, 각 45줄까지]

--- detrend_P (L47-52) ---
def detrend_P(P, n):
    w = n.astype(float); x = YEARS - np.average(YEARS, wei

In [20]:
# -*- coding: utf-8 -*-
# S5 연결용 추가 정보 추출 (읽기 전용, 파일 수정 없음)
import ast, json
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; S5 = P1/"05_S5"
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

# 1) s5_lib 함수 소스 (핵심 3종 + 보조)
t = (P1/"s5_lib.py").read_text(encoding="utf-8"); L = t.splitlines()
for n in ast.parse(t).body:
    if isinstance(n, ast.FunctionDef) and n.name in ("find_key", "load_counts", "sample_P", "near", "gen_N0", "gen_N1"):
        print(f"\n--- s5_lib.{n.name} (L{n.lineno}-{n.end_lineno}) ---"); print("\n".join(L[n.lineno-1:n.end_lineno]))

# 2) s5_detect 앞부분: 설정, pen, run_P, run, 관측 탐지
D_ = (P1/"s5_detect.py").read_text(encoding="utf-8").splitlines()
print("\n--- s5_detect.py L1-64 ---"); print("\n".join(f"L{i+1:<3} {l[:200]}" for i, l in enumerate(D_[:64])))
print("\n--- s5_detect.py L64-80 (귀무 실행, p 계산) ---"); print("\n".join(f"L{i+65:<3} {l[:240]}" for i, l in enumerate(D_[64:80])))

# 3) 동결 규칙의 탐지·귀무 항목
pre = json.loads((S5/"S5_0_PREREG.json").read_text(encoding="utf-8"))
print("\n--- S5_0_PREREG.json 최상위 키 ---", list(pre.keys()))
for k in ("window", "detect", "detection", "nulls", "bootstrap", "penalty", "samples"):
    if k in pre: print(f"[{k}]", json.dumps(pre[k], ensure_ascii=False)[:700])

# 4) 산출 파일: 관측 변화점, 귀무 요약, N2 반복 분포
print("\n--- regime_breaks.csv ---\n", pd.read_csv(S5/"regime_breaks.csv").drop(columns=["curve"], errors="ignore").to_string(index=False))
print("\n--- regime_nulls.csv ---\n", pd.read_csv(S5/"regime_nulls.csv").to_string(index=False))
R = pd.read_csv(S5/"regime_nulls_reps.csv"); n2 = R[R["null"] == "N2"]
print("\n--- N2 반복 분포 (regime_nulls_reps.csv) ---")
print(n2.groupby("sample").R.describe(percentiles=[.5, .95]).round(4).to_string())
print("N2 반복 수:", n2.groupby("sample").size().to_dict(), "| 반복당 k 분포:", n2.groupby("sample").k.value_counts().unstack(fill_value=0).to_dict())
meta = json.loads((S5/"S5_meta.json").read_text(encoding="utf-8")); print("\n--- S5_meta.json ---", json.dumps({k: meta[k] for k in meta if k in ("B", "null_pass", "obs_breaks", "quick")}, ensure_ascii=False)[:600])
print("\n※ 전체 출력을 붙여 주세요(길면 1~3번 소스와 4번 표는 꼭).")


--- s5_lib.find_key (L17-22) ---
def find_key(o, key):
    if isinstance(o, dict):
        for k, v in o.items():
            if k == key: return v
            r = find_key(v, key)
            if r is not None: return r

--- s5_lib.load_counts (L24-33) ---
def load_counts():
    cnt = np.rint(np.load(S4 / "yearly_venue_topic_counts.npy")).astype(np.int64)      # (년, 저널, 토픽)
    venues = pd.read_csv(S4 / "venues_order.csv").venue.astype(str).values
    topics = pd.read_csv(S4 / "topics_order.csv").topic.astype(int).values
    PRE = json.loads((S0 / "PREREG.json").read_text(encoding="utf-8"))
    fixed = set(find_key(PRE, "fixed_venues")); fixed_v = np.array([v in fixed for v in venues])
    assert cnt.shape == (len(YEARS), len(venues), len(topics)), cnt.shape
    assert int(fixed_v.sum()) == 14, int(fixed_v.sum())
    assert int(cnt.sum()) == 76984, int(cnt.sum())
    return cnt, venues, topics, fixed_v

--- s5_lib.sample_P (L36-45) ---
def sample_P(cnt, fixed_v, wbar, kind, tmask=None

In [21]:
# -*- coding: utf-8 -*-
# PA-A1 v2: S5 원본 탐지기(s5_lib.detect)·N2 귀무 반복분포(regime_nulls_reps.csv)에 연결. 미실행 코드.
# MODE: "freeze2"(v2 계획 동결, 결과 미실행) -> "tag2"(커밋·태그) -> "a1"(검정력 곡선, QUICK=True 먼저)
# v1(PA_0_PLAN.json)은 건드리지 않고 PA_0_PLAN_v2.json을 새로 만든다. S0~S8 판정 불변. 모든 산출물 [Post-hoc].
import sys, json, hashlib, subprocess, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)

MODE = "freeze2"      # "freeze2" | "tag2" | "a1"
QUICK = True          # a1: True면 검정력 B=40(파일명 _QUICK), False면 동결값 B=200

ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; PA = P1/"09_PA"; S5 = P1/"05_S5"; S4 = P1/"04_S4"
now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
sha_b = lambda b: hashlib.sha256(b).hexdigest(); sha_f = lambda p: sha_b(Path(p).read_bytes())
git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
class Halt(Exception): pass
spec_hash = lambda s: sha_b(json.dumps({k: v for k, v in s.items() if k not in ("sha256", "created", "refs")}, sort_keys=True, ensure_ascii=False).encode())
V1P, V2P = PA/"PA_0_PLAN.json", PA/"PA_0_PLAN_v2.json"

PA_A1_V2 = dict(
    window=[2000, 2025], samples=["all", "fixed14", "within"], delta_x=[0.5, 1, 2, 3], size_check_delta=0, t_star=[2008, 2012, 2015, 2020],
    B_power=200, alpha=0.05,
    detector="s5_lib.detect(P, n, pen, kmax=6, minlen=3) 그대로. pen = c*phi_s*(K-1)*log(T), c=1.0, phi_s는 S5_0_PREREG.json의 detection.phi. 표본은 s5_lib.sample_P(all, fixed14, within)",
    null="S5 N2 반복 분포(regime_nulls_reps.csv, null=N2, 표본별 B=500)를 그대로 사용. 판정은 S5와 같은 p=(1+#{null_R>=R})/(1+500) < 0.05 (재시뮬레이션 없음)",
    baseline="s5_lib.trend_probs(cnt)의 N2 추세 확률(저널x토픽 선형 추세)에 점프를 삽입. 관측 셀 크기, gen_N2로 다항 표집",
    injection="t* 이후(t>=t*) 모든 저널의 토픽 확률에 a*d를 더하고 0 절단 후 재정규화. d = 'all' 표본 관측 비중의 (2020 이후 평균 - 이전 평균). a는 'all' 표본에서 이등분법으로 정해 세 표본에 같은 계수를 씀",
    jump_def="J(t)=||가중평균(sqrt P)[t,t+3) - 가중평균(sqrt P)[t-3,t)||/sqrt(2) (가중치 4n, Hellinger 거리). 목표: J(주입)-J(N2 추세 기저)=delta*J_obs(2020, 'all', 관측). 표본별 실현 비율을 함께 보고",
    size_check="delta=0(점프 없음)에서 N2 반복 R의 기각률이 약 0.05인지 확인(검정 크기 점검)",
    secondary="변화점 위치 회수: 선택 변화점 중 t*±1년 이내가 있는 비율",
    reading="delta=1에서 t*별 검정력의 최솟값>=0.8: '이 크기의 점프가 있었다면 탐지했을 것이므로 N2 비통과는 연속 변화 증거'로 강화 / 중앙값<0.5: '체제 전환 여부 판정 불가(검정력 부족)'로 리프레이밍 / 그 사이: 어느 쪽도 쓰지 않고 곡선만 보고",
    reproduce_check="관측 R이 regime_nulls.csv의 obs_R(0.7595, 0.7755, 0.7033)과 0.001 이내로 재현되지 않으면 중단",
    bic_compare="관측 비중의 Hellinger 좌표 상위 5 PC에서 구간별 상수 대 구간별 선형, min_len 3, k<=6, BIC")

def build_v2():
    if not V1P.exists(): raise Halt("v1 계획(PA_0_PLAN.json)이 없습니다")
    v1 = json.loads(V1P.read_text(encoding="utf-8"))
    if v1["sha256"] != spec_hash(v1): raise Halt("v1 해시 불일치")
    p = {k: v for k, v in v1.items() if k not in ("sha256", "created", "refs")}
    p.update(version=2, supersedes_sha256=v1["sha256"], pa_a1=PA_A1_V2,
             decision_status="v1 동결 후 PA-A1 실행 전에, S5 원본 탐지기·구간(2000~2025)·N2 반복분포로 교체. PA-A1 검정력·BIC 결과는 미열람. 단, S5의 N2 반복 분포(요약 통계)와 s5_lib 소스는 열람한 상태. 나머지 PA 항목·규칙은 v1과 동일.",
             change_reason="v1의 PA-A1은 자체 대체 탐지기(2007~2025)였고 S5와 통계가 달라 '같은 CCTV로 시험'이 되지 않음.")
    return p, v1

def freeze2():
    p, v1 = build_v2(); h = spec_hash(p); print("PA-0 v2 해시(예정)", h[:12], "| v1", v1["sha256"][:12])
    if V2P.exists(): print("이미 동결:", "동일" if json.loads(V2P.read_text(encoding="utf-8"))["sha256"] == h else "⛔ 다름"); return
    refs = {f.name: sha_f(f) for f in [V1P, S5/"S5_0_PREREG.json", S5/"regime_nulls_reps.csv", S5/"regime_nulls.csv", P1/"s5_lib.py", P1/"s5_detect.py", S4/"yearly_venue_topic_counts.npy"] if f.exists()}
    V2P.write_text(json.dumps(dict(p, created=now(), sha256=h, refs=refs), ensure_ascii=False, indent=1), encoding="utf-8")
    with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f:
        f.write(f"- {now()} | PA-0 v2 동결(해시 {h[:12]}, v1 {v1['sha256'][:12]} 대체 대상은 PA-A1만): 탐지기를 S5 원본(s5_lib.detect), 구간을 2000~2025, 귀무 기준을 S5 N2 반복분포로 교체, 검정 크기 점검(delta=0) 추가. 사유: v1 대체 탐지기가 S5와 달랐음. PA-A1 결과 미열람 | 본 결과 변경 여부: 없음\n")
    print("저장: PA_0_PLAN_v2.json | refs", list(refs))

def tag2():
    if not V2P.exists(): raise Halt("먼저 freeze2")
    t = f"PA-0-plan-v2-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
    if git("tag", "-l", t).stdout.strip(): print("이미 있음:", t); return
    r = git("add", "--", "paper1/09_PA"); print("add", r.returncode, r.stderr.strip())
    r = git("commit", "-m", "PA-0 plan v2 (PA-A1 detector -> S5 original, before execution)"); print("commit", r.returncode, (r.stdout + r.stderr)[:200])
    r = git("tag", "-a", t, "-m", "PA-0 plan v2"); print("tag", r.returncode, r.stderr.strip())

# ------------------------------ PA-A1 본 계산 ------------------------------
def a1():
    if not V2P.exists(): raise Halt("먼저 freeze2와 tag2")
    p, _ = build_v2(); sp = json.loads(V2P.read_text(encoding="utf-8"))
    if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(p): raise Halt("동결 파일과 현재 계획이 다릅니다(사후 변경 금지)")
    if not git("tag", "-l", "PA-0-plan-v2-*").stdout.strip(): raise Halt("tag2를 먼저 실행하세요(동결 태그 없음)")
    sys.path.insert(0, str(P1)); import s5_lib as L
    cfg = PA_A1_V2; sfx = "_QUICK" if QUICK else ""; Bp = 40 if QUICK else cfg["B_power"]; rng = np.random.default_rng(int(p["seed"]))
    s5 = json.loads((S5/"S5_0_PREREG.json").read_text(encoding="utf-8")); D = s5["detection"]; PHI = {k: v["phi"] for k, v in D["phi"].items()}; C0, KMAX, MINLEN = D["c_primary"], D["k_max"], D["min_len"]
    cnt, venues, topics, fixed_v = L.load_counts(); wbar = cnt.sum((0, 2))/cnt.sum(); T, K = len(L.YEARS), len(topics); YEARS = np.asarray(L.YEARS); SAM = cfg["samples"]
    pen = lambda s: C0*PHI[s]*(K-1)*np.log(T)
    def detect_s(c_arr, s):
        P, n = L.sample_P(c_arr, fixed_v, wbar, s); d = L.detect(P, n, pen(s), KMAX, MINLEN); return d, P, n
    # 1) 관측 재현 점검 + N2 기준선
    NR = pd.read_csv(S5/"regime_nulls_reps.csv"); N2 = {s: NR[(NR["null"] == "N2") & (NR["sample"] == s)].R.values for s in SAM}; OBSR = pd.read_csv(S5/"regime_nulls.csv"); info = []
    p_of = lambda s, R: (1 + float((N2[s] >= R).sum()))/(1 + len(N2[s]))
    for s in SAM:
        d, P, n = detect_s(cnt, s); ref = float(OBSR[(OBSR["null"] == "N2") & (OBSR["sample"] == s)].obs_R.iloc[0])
        if abs(d["R"] - ref) > 1e-3: raise Halt(f"S5 관측 R 재현 실패({s}): {d['R']:.4f} vs {ref:.4f}. 탐지기 연결이 S5와 다름")
        info.append(dict(sample=s, obs_R=d["R"], obs_k=d["k"], obs_breaks=[int(YEARS[b]) for b in d["br"]], N2_B=len(N2[s]), N2_q95=float(np.quantile(N2[s], .95)), p_N2=p_of(s, d["R"])))
    print("[관측 재현 OK: S5와 R 일치]\n", pd.DataFrame(info).round(4).to_string(index=False))
    # 2) 점프 정의, 방향, 기저
    def J(P, n, it):
        X, w = np.sqrt(P), 4.0*n; a, b = slice(max(0, it-3), it), slice(it, it+3)
        m = lambda sl: np.average(X[sl], axis=0, weights=w[sl]) if w[sl].sum() > 0 else X[sl].mean(0)
        return float(np.linalg.norm(m(b) - m(a))/np.sqrt(2))
    i20 = int(np.where(YEARS == 2020)[0][0]); Pall, nall = L.sample_P(cnt, fixed_v, wbar, "all")
    Jobs = {s: J(*L.sample_P(cnt, fixed_v, wbar, s), i20) for s in SAM}; d_vec = Pall[i20:].mean(0) - Pall[:i20].mean(0)
    P2 = L.trend_probs(cnt); n_cell = cnt.sum(2).astype(float); print("\n관측 2020 점프(Hellinger, 3년 창):", {k: round(v, 4) for k, v in Jobs.items()})
    def inject(a, it):
        Q = P2.copy(); Q[it:] = np.clip(Q[it:] + a*d_vec[None, None, :], 1e-9, None); return Q/Q.sum(2, keepdims=True)
    def jump_exp(Q, s, it): return J(*L.sample_P(Q*n_cell[:, :, None], fixed_v, wbar, s), it)
    # 3) 검정 크기 점검(delta=0)
    rows = []
    def run_reps(Q, tag, ts, dx, a, jt):
        hits = {s: [] for s in SAM}; rej = {s: [] for s in SAM}; Rs = {s: [] for s in SAM}
        for _ in range(Bp):
            cc = L.gen_N2(cnt, Q, rng)
            for s in SAM:
                d, _, _ = detect_s(cc, s); R = d["R"]; Rs[s].append(R); rej[s].append(p_of(s, R) < cfg["alpha"]); hits[s].append(any(abs(int(YEARS[b]) - ts) <= 1 for b in d["br"]) if ts else np.nan)
        for s in SAM: rows.append(dict(kind=tag, sample=s, t_star=ts, delta_x=dx, a=a, power=float(np.mean(rej[s])), R_mean=float(np.mean(Rs[s])), hit_near_tstar=float(np.nanmean(hits[s])) if ts else np.nan,
                                       J_extra_ratio=jt[s] if jt else np.nan))
    run_reps(P2, "size_check", 0, 0, 0.0, None); print("\n[검정 크기 점검 delta=0: 기각률(약 0.05여야 함)]", {r["sample"]: round(r["power"], 3) for r in rows})
    # 4) 검정력 곡선
    for ts in cfg["t_star"]:
        it = int(np.where(YEARS == ts)[0][0]); base = {s: jump_exp(P2, s, it) for s in SAM}
        for dx in cfg["delta_x"]:
            target = dx*Jobs["all"]; lo, hi = 0.0, 1.0/np.abs(d_vec).max()
            for _ in range(6):
                if jump_exp(inject(hi, it), "all", it) - base["all"] >= target: break
                hi *= 2
            else:
                for s in SAM: rows.append(dict(kind="power", sample=s, t_star=ts, delta_x=dx, power=np.nan)); continue
            for _ in range(40):
                mid = (lo + hi)/2; lo, hi = (mid, hi) if jump_exp(inject(mid, it), "all", it) - base["all"] < target else (lo, mid)
            Q = inject(hi, it); jt = {s: (jump_exp(Q, s, it) - base[s])/Jobs[s] for s in SAM}; run_reps(Q, "power", ts, dx, hi, jt)
        print(f"  t*={ts} 완료")
    PW = pd.DataFrame(rows); pw = PW[PW.kind == "power"]
    print("\n[Post-hoc] PA-A1 검정력(S5 N2 기준, 행: 표본 x t*, 열: delta x 관측 2020 점프)\n", pw.pivot_table(index=["sample", "t_star"], columns="delta_x", values="power").round(2).to_string())
    print("\n[표본별 실현 점프 비율(delta=1, 'all'에서 맞춘 계수)]\n", pw[pw.delta_x == 1].pivot_table(index="sample", columns="t_star", values="J_extra_ratio").round(2).to_string())
    one = pw[pw.delta_x == 1].groupby("sample").power.agg(["min", "median"]).round(2); print("\n[delta=1 요약]\n", one.to_string())
    for s, r in one.iterrows(): print(f"  {s}: 사전 판독 → " + ("강화(연속 변화 증거)" if r["min"] >= 0.8 else ("리프레이밍(판정 불가)" if r["median"] < 0.5 else "곡선으로만 보고")))
    # 5) 구간별 상수 대 선형 BIC (관측, 상위 5 PC)
    bic = []
    for s in SAM:
        P, n = L.sample_P(cnt, fixed_v, wbar, s); Z = np.sqrt(P); Z = Z - Z.mean(0); U, sv, Vt = np.linalg.svd(Z, full_matrices=False); Y = (U*sv)[:, :5]; N = Y.size
        def seg_rss(lin):
            R = np.full((T+1, T+1), np.inf)
            for a_ in range(T+1):
                for b_ in range(a_ + MINLEN, T+1):
                    y = Y[a_:b_]; x = np.arange(b_ - a_); A = np.c_[np.ones_like(x), x] if lin else np.ones((len(x), 1)); R[a_, b_] = ((y - A @ np.linalg.lstsq(A, y, rcond=None)[0])**2).sum()
            best = np.full((KMAX+1, T+1), np.inf); best[0, 0] = 0
            for k in range(1, KMAX+1): best[k] = np.min(best[k-1][:, None] + R, axis=0)
            return best[:, T]
        for name, lin in [("constant", False), ("linear", True)]:
            rs = seg_rss(lin)
            for k in range(1, KMAX+1):
                if np.isfinite(rs[k]): bic.append(dict(sample=s, model=name, k=k, rss=rs[k], bic=N*np.log(rs[k]/N + 1e-12) + (k*5*(2 if lin else 1) + (k-1))*np.log(N)))
    BC = pd.DataFrame(bic); print("\n[Post-hoc] BIC 최솟값(표본별 상수 대 선형)\n", BC.loc[BC.groupby(["sample", "model"]).bic.idxmin()].round(1).to_string(index=False))
    PW.to_csv(PA/f"PA_A1_v2_power_curve{sfx}.csv", index=False, encoding="utf-8-sig"); pd.DataFrame(info).to_csv(PA/f"PA_A1_v2_observed{sfx}.csv", index=False, encoding="utf-8-sig"); BC.to_csv(PA/f"PA_A1_v2_bic{sfx}.csv", index=False, encoding="utf-8-sig")
    print("\n※ 표본별 실현 점프 비율이 1에서 크게 벗어나면 'delta 배'의 의미가 표본마다 다름. 사전 판독은 코드로 옮긴 것이고 경계 사례는 곡선을 보고 판단. S0~S8 판정은 바꾸지 않음.")
    if not QUICK:
        with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f: f.write(f"- {now()} | PA-A1(v2) | 목적: N2 검정력 곡선 | 정의: PA_0_PLAN_v2 {sp['sha256'][:12]} | 결과: PA_A1_v2_*.csv | 본 결과 변경 여부: 없음\n")

try: {"freeze2": freeze2, "tag2": tag2, "a1": a1}[MODE]()
except Halt as e: print(f"⛔ 중단: {e}")

PA-0 v2 해시(예정) a98f9697fa67 | v1 4178cadadec9
저장: PA_0_PLAN_v2.json | refs ['PA_0_PLAN.json', 'S5_0_PREREG.json', 'regime_nulls_reps.csv', 'regime_nulls.csv', 's5_lib.py', 's5_detect.py', 'yearly_venue_topic_counts.npy']


In [22]:
# # PA(사후 교정분석) 진행 정리: 2026-10-19 기준

# ## 1. 한눈에 보기

# | 구분 | 상태 |
# |---|---|
# | 지침서 PA-0 (재현성 정리) | **부분 완료** (검증·계획은 끝, 파일 변경은 대부분 남음) |
# | PA-0.5 (`PA_0_PLAN` 동결) | **v1 완료·태그됨, v2 동결 완료·태그 전** |
# | PA-A1 (N2 검정력 곡선) | 코드 준비됨, **미실행** |
# | 나머지 PA 12개 | 미시작 |

# 지금 가장 가까운 다음 한 걸음은 `MODE="tag2"` 실행입니다.

# ## 2. 지금까지 한 일

# ### 2-1. 사후분석 계획 동결 (PA-0.5)
# - **v1** (`PA_0_PLAN.json`): 해시 `4178cadadec9`, 커밋 `738a423`, 태그 `PA-0-plan-2026-10-19`. 사후분석 14개 항목, 핵심 주장용 PA(A1, A2, B2), 판독 규칙을 실행 전에 고정했습니다.
# - **v2** (`PA_0_PLAN_v2.json`): 해시 **`a98f9697fa67`**, v1 해시를 `supersedes_sha256`에 기록했습니다. 바뀐 것은 PA-A1뿐입니다.
#   - 탐지기를 제 대체 구현에서 **S5 원본**(`s5_lib.detect`, 벌점 `c·φ·(K−1)·log T`)으로 교체했습니다.
#   - 분석 구간을 2007~2025에서 S5와 같은 **2000~2025**로 맞췄습니다.
#   - 귀무 기준선을 `regime_nulls_reps.csv`의 **N2 반복 500회**로 바꿨습니다.
#   - 검정 크기 점검(δ=0)과 관측 R 재현 점검(0.001 이내)을 추가했습니다.
# - **재동결 사유:** v1 탐지기가 S5와 달라 "같은 CCTV로 시험"이 되지 않았기 때문입니다. 이 사유와 "PA-A1 결과 미열람"이 `PA_CHANGELOG.md`에 기록됐습니다. 다만 `s5_lib` 소스와 N2 반복분포 요약은 열람한 뒤였다는 점도 계획서에 적어 두었습니다.

# ### 2-2. 재현성 점검(PA-0)에서 확인된 것
# - **해시 검증:** S1~S8 manifest의 파일 해시 **152건 모두 일치**, 불일치·파일없음 0건입니다. 규칙 해시(`spec_sha256`) 14건은 각 단계의 `spec_hash`로 이미 검증됐습니다. 검증 기록은 `PA_CHANGELOG.md`에 한 줄 추가했습니다.
# - **S4 코드:** 별도 `.py`로 저장된 적이 없어 IPython 히스토리와 노트북 셀에서 복구 후보를 확정했습니다(`s4_main` ← 히스토리 1079:23, `s4_6_rotation` ← 1080:4, `s4_6b_floor_check` ← 1080:6). 노트북 셀과 코드가 동일하고, 노트북 출력("저널 34개, 고정 저널 14개")이 선택본과 맞았습니다. 최종 실행본임은 증명하지 못한다고 CHANGELOG에 적도록 했습니다.
# - **잘못된 태그:** `S4-main`, `S4-final`은 S3 CHANGELOG 커밋(`33b3912`, 11:07:38)을 가리켰고, S4 산출물은 11:10:54 이후 생성됐습니다. 즉 그 태그에는 S4 산출물이 담길 수 없었습니다.
# - **오류 로그 발견:** `04_S4/s4_6_output.txt`(129자)는 실제 출력이 아니라 경로 오류 메시지입니다. 실제 4.6 출력은 노트북 셀 01에만 있습니다.
# - **S5 연결 정보 확보:** `s5_lib`, `s5_detect`, S5 동결 규칙, N2 반복분포를 확인했습니다. N2 반복 R은 매우 좁게 분포합니다(all 표준편차 0.004, 95% 0.7723). 관측 R 0.7595는 N2 중앙값(0.7657)보다 낮아 p=0.94가 나왔습니다.

# ## 3. 아직 안 한 일 (PA-0 마무리)

# 지침서 PA-0 표 5개 항목 기준입니다.

# | 항목 | 상태 | 해야 할 일 |
# |---|---|---|
# | S4 코드 3종 저장 | **미확인** | 복구·기록 셀을 `DRY=False`로 실행했는지 확인. `ls paper1/s4_*.py`에 3개가 있으면 완료 |
# | 잘못된 태그 정리 | **미실행** | PA-0 정리 셀 `MODE="commit"`을 `DRY=True`로 계획 확인 후 `DRY=False` |
# | 미추적 파일 추적 | **미실행** | 위 `commit`에 포함. 끝나면 `git status --short paper1/`에 `??`가 없어야 함 |
# | 푸시와 백업 | **미실행** | `git push --follow-tags` 직접 실행(원격 비공개 확인), `oa_raw_20261013.tar.gz`를 저장소 밖으로 복사 |
# | 전체 manifest 통합 | **코드도 아직 없음** | `paper1/MASTER_MANIFEST.csv` 한 장 생성 코드 필요 |

# 원격 `origin`에는 `s0-frozen` 태그 2개뿐이고, S1~S8과 PA-0 동결 태그는 모두 로컬에만 있습니다. 푸시가 가장 급합니다. `.gitignore`에는 `oa_raw/`, `_backup_before_income_tv/`, `archive/*.tar.gz`가 들어 있고, `.ipynb_checkpoints/`와 `_recovered_s4/`는 아직 추가가 필요합니다.

# ## 4. 앞으로 할 일 (순서)

# ### 단계 A. 지금 바로
# 1. **`MODE="tag2"` 실행:** v2 동결을 커밋하고 `PA-0-plan-v2-2026-10-19` 태그를 찍습니다. `a1`은 이 태그가 없으면 중단되도록 막아 두었습니다.
# 2. PA-0 마무리: 위 3번의 미완 항목을 순서대로 처리합니다.

# ### 단계 B. PA-A1 (핵심 주장의 방어선)
# 1. **`a1`을 `QUICK=True`로 시험 실행.** 확인할 것은 다음 둘입니다.
#    - "관측 재현 OK" (S5 관측 R과 0.001 이내 일치)
#    - δ=0 기각률 (약 0.05여야 함, B=40이면 0~0.15 정도). 크게 어긋나면 본 실행을 멈추고 출력을 붙여 주세요.
# 2. `QUICK=False`로 본 실행(B=200).
# 3. **결과 판독 (동결된 규칙):**
#    - δ=1에서 t*별 검정력 최솟값이 0.8 이상이면 "이 크기의 점프가 있었다면 탐지했을 것이므로 N2 비통과는 연속 변화의 증거"로 **강화**.
#    - 중앙값이 0.5 미만이면 "체제 전환 부재"가 아니라 **"체제 전환 여부 판정 불가(검정력 부족)"** 로 리프레이밍.
#    - 그 사이면 어느 쪽도 쓰지 않고 곡선만 보고.
# 4. 구간별 상수 대 선형 BIC 비교 결과도 함께 확인합니다.
# 5. PA-B1 문구를 확정하고 제목에서 **"체제 전환"을 뺄지** 결정합니다.

# ### 단계 C. 핵심 주장용 나머지 두 PA
# | PA | 내용 | 필요한 것 |
# |---|---|---|
# | **PA-B2** | 2020 변화점이 표본 크기·저널 구성의 아티팩트인지 분리(연도당 3,800편 하향 추출 B=200, 2020 점프 구성 토픽 상위 10) | A1 결과, `s5_lib` 재사용 |
# | **PA-A2** | S6 신규 진입 우세의 구성 방어(ID 기반 대조, 진입 정의 2종, 부트스트랩 편향 보정) | `s6_lib.py`, `s6_run.py` 함수 정보(**아직 못 봄**) |

# ### 단계 D. 정합성·약점 리프레이밍
# - **PA-C1, C2:** S6×S7 교차(중국 소속 신규 저자 제외 분해), S7 국가 몫을 임베딩 보유 논문만으로 재계산.
# - **PA-B3, B4, B5, B7:** 신생 핵심 집합, F1+F3 이분 등급, S4 안정성 Wilson 구간(약 [0.56, 0.81]), 결측 경계 분석. B3의 D1' 재구성은 **F6 생성 코드 위치**가 필요합니다.
# - **PA-A3, B9, B8:** S7 국가 미상 교정·균등 4구간·중국 제외 HHI, 소득군 내 상관, 미달 4도메인 부록 정렬.
# - **PA-B6:** ORCID 기반 소표본과 수작업 50쌍. 인력이 필요해서 병렬로 진행하세요.
# - **PA-A4:** 동결·CHANGELOG 체계를 부록 표 한 장으로 정리(분석 아님).

# ### 단계 E. 마무리
# - 모든 PA 표와 그림 제목에 `[Post-hoc]` 표지.
# - 기각된 PA 목록을 부록에 수록.
# - "Post-hoc analyses were specified after inspecting the primary results and are reported as exploratory; no pre-registered decision rule was altered." 문구를 영문 원고에 포함.
# - S0~S8 통과·실패 판정은 PA 결과로 바꾸지 않습니다.

# ## 5. 지금 확인이 필요한 질문

# 1. **PA-B6, B8-2의 수작업 인력:** 수작업 50쌍과 새 표본 25편 코딩을 할 시간이 있는지 알려 주세요.
# 2. **우선순위:** A1 → A2 → B2로 가도 되는지, 다른 약점을 먼저 다루고 싶은지 알려 주세요.
# 3. **다음 코드용 정보:** PA-A2에는 `s6_lib.py`·`s6_run.py`의 함수 목록과 신규·지속 판별 코드, B3에는 F6 생성 코드 위치가 필요합니다.

# ## 6. 주의할 점
# - **이중 동결을 기억하세요.** PA-A1의 결과를 본 뒤에는 정의나 임계값을 바꿀 수 없습니다. 바꿔야 한다면 v3로 분리하고 사유를 남겨야 하는데, 이 경우 "결과를 보기 전"이라는 주장은 약해집니다.
# - **검정력이 높게 나와도 단정하지 마세요.** 삽입한 점프의 모양(2020 이후 평균 − 이전 평균 방향)에 대한 검정력입니다. 다른 모양의 변화에 대한 검정력은 아닙니다.
# - **해석은 "함께 움직인다"까지**입니다. 연속 변화와 체제 전환 모두 인과 주장이 아닙니다.

# `tag2` 출력과 `a1` 시험 실행 출력을 붙여 주시면 다음 단계(PA-B2, PA-A2 코드)로 넘어가겠습니다.

In [ ]:
# # 평가: 논문 1 사후 교정분석(PA) 체계

# 전반적으로 방법론적 규율이 매우 높은 작업입니다. 다만 몇 가지는 "투명성"이라는 강점이 실제로는 약해지는 지점이니, 투고 전에 짚어 두는 편이 좋습니다.

# ## 1. 강점

# **동결·기록 체계가 실제로 작동합니다.** 사후분석을 "PA-" 번호로 분리하고, 실행 전에 계획을 해시·태그로 고정하며, 기각된 PA도 보고하고 S0~S8 판정을 바꾸지 않는다는 원칙은 사후분석 설계의 모범에 가깝습니다. 특히 다음이 좋습니다.

# - v1 → v2 재동결 시 사유("같은 CCTV로 시험이 아님")와 "s5_lib 소스와 N2 요약은 열람한 상태"라는 불리한 사실까지 계획서에 적은 점
# - 관측 R 재현 점검(0.001 이내, 불일치 시 중단)과 δ=0 검정 크기 점검. 검정력 분석에서 가장 흔히 빠지는 두 장치입니다.
# - 판독 규칙(≥0.8 강화 / <0.5 판정 불가 / 사이는 곡선만)을 실행 전에 못 박은 점
# - manifest 해시 152건 일치 검증, 복구 코드를 "최종 실행본임은 증명 못 함"이라고 솔직히 기록한 점
# - 오류 로그(`s4_6_output.txt` 129자)를 증거로 오인하지 않고 찾아낸 점

# **PA-A1의 문제 설정이 정확합니다.** "체제 전환 없음"이 N2 비통과라는 부정적 증거에만 기대고 있고, 검정력 부족과 점프 부재를 구분하지 못한다는 진단은 핵심을 짚습니다.

# ## 2. 주요 우려

# **(1) N2 귀무가 이미 "잘 맞는" 귀무일 가능성: 검정력 곡선 해석의 핵심 위험**
# 첨부된 N2 반복 분포를 보면 `all` 표본에서 N2의 R 표준편차가 0.004, 95% 분위수가 0.7723으로 매우 좁고, 관측 R(0.7595)은 N2 중앙값(0.7657)보다 낮습니다. 저널×토픽별 선형 추세 150×34개를 적합한 귀무는 자유도가 매우 커서, 관측 구조를 거의 그대로 재현하도록 과적합된 귀무일 수 있습니다. 두 가지 귀결이 있습니다.
# - N2 비통과는 "점프가 없다"보다 "귀무가 관측 자료를 이미 흡수한다"에 가깝고, 이는 검정력 곡선이 높게 나와도 "연속 변화의 증거"로 읽기 어렵다는 뜻입니다. 높게 나오면 오히려 N2가 점프에 민감하다는 확인일 뿐입니다.
# - 반대로 `fixed14`(N2 R 최대 0.80, 표준편차 0.027)와 `within`은 분포 형태가 달라 표본별로 검정력이 크게 다를 것입니다. 사전 판독 규칙이 표본별로 따로 적용되므로 결과가 갈릴 가능성이 큽니다.

# **(2) 주입 점프가 단일 모양(방향 d)에 한정**
# `d`는 'all' 표본의 2020 전후 평균 차이이고, 그 계수를 세 표본에 동일하게 씁니다. 표본별 실현 점프 비율이 1에서 벗어나면 "δ배"의 의미가 표본마다 달라집니다(계획서가 이를 인정하고 비율을 보고하도록 한 점은 좋음). 더 근본적으로, 이 방향은 "관측된 2020 점프 모양"이라서 *다른 모양의 점프*(특정 저널 한정, 소수 토픽 급변 등)에 대한 검정력은 알 수 없습니다. 이미 주의사항에 적으셨으니, 논문 본문에서는 "관측 형태의 점프에 대한 검정력"으로 한정해 쓰세요.

# **(3) δ=1이 "작은" 점프가 아닐 수 있음**
# 관측 2020 점프는 JS 0.29(all)로 큽니다. 1배에서 검정력이 높은 것은 당연하고, 체제 전환 부재 주장에 의미 있는 것은 더 작은 점프(0.25~0.5배)에서의 검정력입니다. 격자가 0.5부터 시작하므로 0.25 정도를 추가하는 것을 고려하되, 이미 동결했으므로 v3 대신 "탐색적 확장"으로 따로 표기하는 방식이 낫습니다.

# **(4) 판독 규칙의 min/median 혼용**
# "t*별 최솟값 ≥0.8" 대 "중앙값 <0.5"는 비대칭입니다. t*=2008처럼 연초·말단에 가까운 지점은 구조적으로 검정력이 낮아(최소 구간 3년, 점프 후 3년 창) 최솟값 기준이 거의 항상 실패하고, 중앙값 기준은 쉽게 걸립니다. 결과가 "곡선만 보고"로 수렴할 가능성이 높으며, 그 자체는 정직한 결론입니다. 다만 B1 문구를 미리 그렇게 준비해 두세요.

# **(5) 사후 동결의 정직성 한계**
# v2 재동결은 s5_lib 소스와 N2 분포를 본 뒤 이뤄졌습니다. 기록은 충분하지만, 본문에서는 "사전등록"이 아니라 "결과 미열람 상태에서의 사후 명세"라고 표현하는 편이 안전합니다. 또 PA-A1 코드 안에 `QUICK` 시험 실행이 있는데, QUICK 결과를 본 뒤 설정을 바꾸면 v3 사유가 됩니다. QUICK은 오류 점검용으로만 쓰고 결과 해석은 하지 않겠다는 규칙을 PA_CHANGELOG에 한 줄 적어 두세요.

# **(6) 다중성과 핵심 PA 수 불일치**
# 지침서 0절(A1, A2, B1)과 우선순위 절(A1, A2, B2)이 달랐고, 계획서에서 이를 인지해 A1·A2·B2로 정리한 것은 좋은 처리입니다. 다만 B1이 A1 결과로 정해지는 문구 작업이라 실질적으로 핵심 주장은 4개에 걸칩니다. "3개 이하" 원칙은 형식상 유지되지만 본문에서 B1을 숨기지 마세요.

# ## 3. 재현성 정리(PA-0)에 대해

# - **태그 문제 진단은 정확합니다.** S4 태그가 11:07 커밋을 가리키고 산출물은 11:10 이후 생성이므로, 태그를 삭제·재태그하되 CHANGELOG에 정정을 남기는 계획이 옳습니다. 다만 *산출물 시각이 커밋보다 늦다*는 사실 자체가 "S4 결과가 동결 전에 열람·변경되지 않았다"는 증명이 아니므로, 부록 표에서 "산출물 최초 커밋일 ≠ 생성일"이라고 명시하신 점이 중요합니다.
# - **S4 코드 복구**는 후보 선별 규칙("산출물 시각 이전 세션 중 가장 늦은 것")이 추정임을 계속 밝히세요. 특히 `s4_main.py`는 후보가 1개(1079:23)뿐이지만, 히스토리 1080:3이 `topic_quadrants.csv`·`S4_meta.json`을 포함한 수정본이고 산출물 시각(11:10)보다 *늦은* 세션(11:22)이라 배제된 것입니다. 이후 수정 이력이 산출물에 반영됐을 가능성은 낮지만, 두 코드의 diff를 한 번 부록에 남기면 더 안전합니다.
# - 해시 검증에서 "이름 해석 불가 22건"이 `spec_sha256`, `output_sha256` 등 키 이름이었고 이후 파일 해시 8건으로 별도 대조해 모두 일치한 점은 잘 마무리됐습니다. 다만 `spec_sha256` 14건은 "각 단계 spec_hash로 이미 검증"이라고만 적혀 있으니, 실제 재계산 출력이 로그에 남아 있는지 확인하세요.
# - 원격에는 `s0-frozen` 태그 2개뿐이고 이후 모든 동결 태그가 로컬에만 있습니다. **푸시가 가장 급합니다.** 외부 타임스탬프(원격 push 시각)가 없으면 동결의 시점 증명력이 약합니다. 가능하면 Zenodo나 OSF 같은 외부 보관도 고려하세요.

# ## 4. 코드 수준 지적

# - `a1()`의 `for ... else` 구문: 목표 점프에 도달하지 못했을 때 `continue`가 안쪽 `for s in SAM` 루프에 걸려 바깥 루프 흐름을 의도대로 건너뛰지 않습니다. 이후 `Q = inject(hi, it)`가 그대로 실행되어 도달 불가 조건에서도 계산을 돌릴 수 있습니다. 실행 전 수정 필요합니다(오류 수정이므로 동결 사양 변경은 아님, CHANGELOG에는 기록).
# - 이분법 상한 확장(`hi *= 2`, 최대 6회)은 합리적이나, 점프는 0 절단과 재정규화 때문에 `a`에 대해 비단조일 수 있습니다. 40회 이분법이 수렴했는지(주입 점프 vs 목표) 사후 확인을 출력에 포함하세요.
# - BIC 비교의 자유도(`k*5*(2 if lin else 1) + (k-1)`)와 `N = Y.size`는 PC 좌표를 독립 관측으로 취급합니다. 연도 시계열은 자기상관이 강해 BIC가 과분할 쪽으로 기울 수 있습니다. 결과를 "직접 증거"로 쓰기보다 보조 지표로 두세요. 또 상수 모형과 선형 모형은 구간 경계에서 연속성을 강제하지 않아 선형 구간 모형이 불연속 점프를 흡수할 수 있으니, 연속 piecewise-linear(꺾은선) 모형을 별도로 두는 것이 "연속 변화"라는 주장에 더 직접적입니다.
# - N2 기준선과 주입 후 표본추출(`gen_N2`)은 관측 셀 크기를 유지하므로 표본 크기 아티팩트(B2)와는 별개입니다. B2의 하향 추출은 셀 크기 자체를 바꿔야 해서 코드 재사용이 제한적입니다.

# ## 5. 서사·프레이밍

# - 제목에서 "체제 전환"을 빼는 방향은 타당합니다. 다만 A1 결과가 "판정 불가"로 나오면 "연속적 재배치"라는 긍정 주장도 약해지므로, "점프보다 연속 재배치로 서술하는 편이 자료에 부합"이라는 완화된 문구가 유지돼야 합니다. 현재 B1의 After 문구는 이 점에서 적절합니다.
# - "신규 진입 주도"를 새 중심 기여로 삼으려면 PA-A2가 중요하지만, 그 결과가 사후분석이라는 점이 약점입니다. S6 자체가 사전등록 분석이므로, "사전 명시 결과 + PA로 강건성 확인" 구조를 본문에서 분명히 하세요.
# - 부록 표(A4)는 "완전 비열람이 아님"을 강점으로 바꾸는 장치이므로, 정직함이 곧 신뢰입니다. 열람한 항목은 빠짐없이 적으세요.

# ## 6. 다음 단계 제안

# 1. `tag2` 실행 후 즉시 원격 푸시(`--follow-tags`). 이후 `a1` 실행.
# 2. `a1` 실행 전 위 `for/else` 버그 수정을 CHANGELOG에 기록.
# 3. QUICK 실행은 오류 확인과 "관측 재현 OK", δ=0 기각률만 확인하고 곡선은 해석하지 않기.
# 4. 본 실행 후 결과가 "곡선만 보고"로 나와도 실패가 아닙니다. 그 경우 B1을 "판정 불가와 연속 재배치 서술의 병기"로 정리하면 됩니다.
# 5. PA-A2 진행 전 `s6_lib.py`, `s6_run.py`의 신규·지속 판별 함수 목록을 확보하세요.

# 총평: 절차적 엄밀성은 상위권입니다. 가장 큰 실질 위험은 절차가 아니라 **N2 귀무의 성격(과적합 가능성)과 검정력 해석의 범위**이므로, 본문에서 "관측 형태 점프에 대한 검정력"으로 한정해 쓰는 것을 권합니다.

In [24]:
# -*- coding: utf-8 -*-
# PA-A1 v2 실행본 (패치판). 미실행 코드.
# MODE: "freeze2" -> "tag2" -> "a1"(동결 사양 그대로) -> "explore"(동결 밖 탐색적 확장, 별도 표기)
# 패치 내용 (동결 사양 PA_A1_V2 딕셔너리는 원본과 동일 -> spec 해시 불변):
#   P1. for/else + continue 버그 수정: 목표 점프 도달 불가 시 해당 (t*, delta) 전체를 NaN 처리하고 건너뜀
#   P2. 이분법 수렴 점검: 실현 점프 / 목표 비율을 기록하고 2% 초과 이탈이면 flag
#   P3. QUICK 모드는 오류 점검 전용: 재현 OK와 delta=0 기각률만 출력, 곡선·판독 규칙 출력 안 함
#   P4. explore 모드(동결 밖, [Post-hoc][Exploratory]): delta=0.25 추가, 연속 꺾은선(piecewise-linear) BIC
# 이 파일의 변경은 CHANGELOG에 '코드 오류 수정(사양 불변)'으로 기록할 것.
import sys, json, hashlib, subprocess, warnings, itertools
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)

MODE = "tag2"      # "freeze2" | "tag2" | "a1" | "explore"
QUICK = True          # True: 검정력 B=40, 오류 점검 전용(곡선 해석 금지)

ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; PA = P1/"09_PA"; S5 = P1/"05_S5"; S4 = P1/"04_S4"
now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
sha_b = lambda b: hashlib.sha256(b).hexdigest(); sha_f = lambda p: sha_b(Path(p).read_bytes())
git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
class Halt(Exception): pass
spec_hash = lambda s: sha_b(json.dumps({k: v for k, v in s.items() if k not in ("sha256", "created", "refs")}, sort_keys=True, ensure_ascii=False).encode())
V1P, V2P = PA/"PA_0_PLAN.json", PA/"PA_0_PLAN_v2.json"

# ---- 동결 사양: 원본과 글자 단위로 동일해야 함 (수정 금지) ----
PA_A1_V2 = dict(
    window=[2000, 2025], samples=["all", "fixed14", "within"], delta_x=[0.5, 1, 2, 3], size_check_delta=0, t_star=[2008, 2012, 2015, 2020],
    B_power=200, alpha=0.05,
    detector="s5_lib.detect(P, n, pen, kmax=6, minlen=3) 그대로. pen = c*phi_s*(K-1)*log(T), c=1.0, phi_s는 S5_0_PREREG.json의 detection.phi. 표본은 s5_lib.sample_P(all, fixed14, within)",
    null="S5 N2 반복 분포(regime_nulls_reps.csv, null=N2, 표본별 B=500)를 그대로 사용. 판정은 S5와 같은 p=(1+#{null_R>=R})/(1+500) < 0.05 (재시뮬레이션 없음)",
    baseline="s5_lib.trend_probs(cnt)의 N2 추세 확률(저널x토픽 선형 추세)에 점프를 삽입. 관측 셀 크기, gen_N2로 다항 표집",
    injection="t* 이후(t>=t*) 모든 저널의 토픽 확률에 a*d를 더하고 0 절단 후 재정규화. d = 'all' 표본 관측 비중의 (2020 이후 평균 - 이전 평균). a는 'all' 표본에서 이등분법으로 정해 세 표본에 같은 계수를 씀",
    jump_def="J(t)=||가중평균(sqrt P)[t,t+3) - 가중평균(sqrt P)[t-3,t)||/sqrt(2) (가중치 4n, Hellinger 거리). 목표: J(주입)-J(N2 추세 기저)=delta*J_obs(2020, 'all', 관측). 표본별 실현 비율을 함께 보고",
    size_check="delta=0(점프 없음)에서 N2 반복 R의 기각률이 약 0.05인지 확인(검정 크기 점검)",
    secondary="변화점 위치 회수: 선택 변화점 중 t*±1년 이내가 있는 비율",
    reading="delta=1에서 t*별 검정력의 최솟값>=0.8: '이 크기의 점프가 있었다면 탐지했을 것이므로 N2 비통과는 연속 변화 증거'로 강화 / 중앙값<0.5: '체제 전환 여부 판정 불가(검정력 부족)'로 리프레이밍 / 그 사이: 어느 쪽도 쓰지 않고 곡선만 보고",
    reproduce_check="관측 R이 regime_nulls.csv의 obs_R(0.7595, 0.7755, 0.7033)과 0.001 이내로 재현되지 않으면 중단",
    bic_compare="관측 비중의 Hellinger 좌표 상위 5 PC에서 구간별 상수 대 구간별 선형, min_len 3, k<=6, BIC")

def build_v2():
    if not V1P.exists(): raise Halt("v1 계획(PA_0_PLAN.json)이 없습니다")
    v1 = json.loads(V1P.read_text(encoding="utf-8"))
    if v1["sha256"] != spec_hash(v1): raise Halt("v1 해시 불일치")
    p = {k: v for k, v in v1.items() if k not in ("sha256", "created", "refs")}
    p.update(version=2, supersedes_sha256=v1["sha256"], pa_a1=PA_A1_V2,
             decision_status="v1 동결 후 PA-A1 실행 전에, S5 원본 탐지기·구간(2000~2025)·N2 반복분포로 교체. PA-A1 검정력·BIC 결과는 미열람. 단, S5의 N2 반복 분포(요약 통계)와 s5_lib 소스는 열람한 상태. 나머지 PA 항목·규칙은 v1과 동일.",
             change_reason="v1의 PA-A1은 자체 대체 탐지기(2007~2025)였고 S5와 통계가 달라 '같은 CCTV로 시험'이 되지 않음.")
    return p, v1

def freeze2():
    p, v1 = build_v2(); h = spec_hash(p); print("PA-0 v2 해시(예정)", h[:12], "| v1", v1["sha256"][:12])
    if V2P.exists(): print("이미 동결:", "동일" if json.loads(V2P.read_text(encoding="utf-8"))["sha256"] == h else "⛔ 다름"); return
    refs = {f.name: sha_f(f) for f in [V1P, S5/"S5_0_PREREG.json", S5/"regime_nulls_reps.csv", S5/"regime_nulls.csv", P1/"s5_lib.py", P1/"s5_detect.py", S4/"yearly_venue_topic_counts.npy"] if f.exists()}
    V2P.write_text(json.dumps(dict(p, created=now(), sha256=h, refs=refs), ensure_ascii=False, indent=1), encoding="utf-8")
    with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f:
        f.write(f"- {now()} | PA-0 v2 동결(해시 {h[:12]}) | 사유: v1 대체 탐지기가 S5와 달랐음. PA-A1 결과 미열람 | 본 결과 변경 여부: 없음\n")
    print("저장: PA_0_PLAN_v2.json")

def tag2():
    if not V2P.exists(): raise Halt("먼저 freeze2")
    t = f"PA-0-plan-v2-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
    if git("tag", "-l", t).stdout.strip(): print("이미 있음:", t); return
    r = git("add", "--", "paper1/09_PA"); print("add", r.returncode, r.stderr.strip())
    r = git("commit", "-m", "PA-0 plan v2 (PA-A1 detector -> S5 original, before execution)"); print("commit", r.returncode, (r.stdout + r.stderr)[:200])
    r = git("tag", "-a", t, "-m", "PA-0 plan v2"); print("tag", r.returncode, r.stderr.strip())
    print("※ 이어서 git push --follow-tags 로 원격 타임스탬프를 확보하세요.")

# ------------------------------ 공통 컨텍스트 ------------------------------
def make_ctx():
    if not V2P.exists(): raise Halt("먼저 freeze2와 tag2")
    p, _ = build_v2(); sp = json.loads(V2P.read_text(encoding="utf-8"))
    if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(p): raise Halt("동결 파일과 현재 계획이 다릅니다(사후 변경 금지)")
    if not git("tag", "-l", "PA-0-plan-v2-*").stdout.strip(): raise Halt("tag2를 먼저 실행하세요(동결 태그 없음)")
    sys.path.insert(0, str(P1)); import s5_lib as L
    cfg = PA_A1_V2; rng = np.random.default_rng(int(p["seed"]))
    s5 = json.loads((S5/"S5_0_PREREG.json").read_text(encoding="utf-8")); D = s5["detection"]; PHI = {k: v["phi"] for k, v in D["phi"].items()}
    C0, KMAX, MINLEN = D["c_primary"], D["k_max"], D["min_len"]
    cnt, venues, topics, fixed_v = L.load_counts(); wbar = cnt.sum((0, 2))/cnt.sum(); T, K = len(L.YEARS), len(topics); YEARS = np.asarray(L.YEARS); SAM = cfg["samples"]
    pen = lambda s: C0*PHI[s]*(K-1)*np.log(T)
    def detect_s(c_arr, s):
        P, n = L.sample_P(c_arr, fixed_v, wbar, s); return L.detect(P, n, pen(s), KMAX, MINLEN), P, n
    NR = pd.read_csv(S5/"regime_nulls_reps.csv"); N2 = {s: NR[(NR["null"] == "N2") & (NR["sample"] == s)].R.values for s in SAM}; OBSR = pd.read_csv(S5/"regime_nulls.csv")
    p_of = lambda s, R: (1 + float((N2[s] >= R).sum()))/(1 + len(N2[s]))
    info = []
    for s in SAM:
        d, P, n = detect_s(cnt, s); ref = float(OBSR[(OBSR["null"] == "N2") & (OBSR["sample"] == s)].obs_R.iloc[0])
        if abs(d["R"] - ref) > 1e-3: raise Halt(f"S5 관측 R 재현 실패({s}): {d['R']:.4f} vs {ref:.4f}")
        info.append(dict(sample=s, obs_R=d["R"], obs_k=d["k"], obs_breaks=[int(YEARS[b]) for b in d["br"]], N2_B=len(N2[s]), N2_q95=float(np.quantile(N2[s], .95)), p_N2=p_of(s, d["R"])))
    print("[관측 재현 OK: S5와 R 일치]\n", pd.DataFrame(info).round(4).to_string(index=False))
    def J(P, n, it):
        X, w = np.sqrt(P), 4.0*n; a, b = slice(max(0, it-3), it), slice(it, it+3)
        m = lambda sl: np.average(X[sl], axis=0, weights=w[sl]) if w[sl].sum() > 0 else X[sl].mean(0)
        return float(np.linalg.norm(m(b) - m(a))/np.sqrt(2))
    i20 = int(np.where(YEARS == 2020)[0][0]); Pall, _ = L.sample_P(cnt, fixed_v, wbar, "all")
    Jobs = {s: J(*L.sample_P(cnt, fixed_v, wbar, s), i20) for s in SAM}; d_vec = Pall[i20:].mean(0) - Pall[:i20].mean(0)
    P2 = L.trend_probs(cnt); n_cell = cnt.sum(2).astype(float); print("\n관측 2020 점프(Hellinger, 3년 창):", {k: round(v, 4) for k, v in Jobs.items()})
    def inject(a, it):
        Q = P2.copy(); Q[it:] = np.clip(Q[it:] + a*d_vec[None, None, :], 1e-9, None); return Q/Q.sum(2, keepdims=True)
    jump_exp = lambda Q, s, it: J(*L.sample_P(Q*n_cell[:, :, None], fixed_v, wbar, s), it)
    return dict(p=p, sp=sp, cfg=cfg, rng=rng, L=L, cnt=cnt, fixed_v=fixed_v, wbar=wbar, T=T, YEARS=YEARS, SAM=SAM, detect_s=detect_s, p_of=p_of, info=info,
                Jobs=Jobs, P2=P2, inject=inject, jump_exp=jump_exp, d_vec=d_vec, MINLEN=MINLEN, KMAX=KMAX)

def run_reps(c, Q, kind, ts, dx, a, ratio, Bp, rows, conv_flag=""):
    hits = {s: [] for s in c["SAM"]}; rej = {s: [] for s in c["SAM"]}; Rs = {s: [] for s in c["SAM"]}
    for _ in range(Bp):
        cc = c["L"].gen_N2(c["cnt"], Q, c["rng"])
        for s in c["SAM"]:
            d, _, _ = c["detect_s"](cc, s); Rs[s].append(d["R"]); rej[s].append(c["p_of"](s, d["R"]) < c["cfg"]["alpha"])
            hits[s].append(any(abs(int(c["YEARS"][b]) - ts) <= 1 for b in d["br"]) if ts else np.nan)
    for s in c["SAM"]:
        rows.append(dict(kind=kind, sample=s, t_star=ts, delta_x=dx, a=a, power=float(np.mean(rej[s])), R_mean=float(np.mean(Rs[s])),
                         hit_near_tstar=float(np.nanmean(hits[s])) if ts else np.nan, J_extra_ratio=(ratio or {}).get(s, np.nan), conv_flag=conv_flag, B=Bp))

def power_grid(c, deltas, Bp, rows):
    """P1, P2 반영: 도달 불가면 NaN 행 후 건너뜀. 수렴 오차 flag 기록."""
    for ts in c["cfg"]["t_star"]:
        it = int(np.where(c["YEARS"] == ts)[0][0]); base = {s: c["jump_exp"](c["P2"], s, it) for s in c["SAM"]}
        for dx in deltas:
            target = dx*c["Jobs"]["all"]; lo, hi = 0.0, 1.0/np.abs(c["d_vec"]).max(); reached = False
            for _ in range(6):
                if c["jump_exp"](c["inject"](hi, it), "all", it) - base["all"] >= target: reached = True; break
                hi *= 2
            if not reached:
                for s in c["SAM"]: rows.append(dict(kind="power", sample=s, t_star=ts, delta_x=dx, power=np.nan, conv_flag="unreachable"))
                continue
            for _ in range(40):
                mid = (lo + hi)/2; lo, hi = (mid, hi) if c["jump_exp"](c["inject"](mid, it), "all", it) - base["all"] < target else (lo, mid)
            Q = c["inject"](hi, it); ratio = {s: (c["jump_exp"](Q, s, it) - base[s])/c["Jobs"][s] for s in c["SAM"]}
            achieved = (c["jump_exp"](Q, "all", it) - base["all"])/target
            flag = "" if abs(achieved - 1) <= 0.02 else f"conv_off({achieved:.3f})"
            run_reps(c, Q, "power", ts, dx, hi, ratio, Bp, rows, flag)
        print(f"  t*={ts} 완료")

def size_check(c, Bp, rows):
    run_reps(c, c["P2"], "size_check", 0, 0, 0.0, None, Bp, rows)
    print("[검정 크기 점검 delta=0: 기각률(약 0.05여야 함)]", {r["sample"]: round(r["power"], 3) for r in rows if r["kind"] == "size_check"})

# ------------------------------ BIC ------------------------------
def pcs(c, s, npc=5):
    P, _ = c["L"].sample_P(c["cnt"], c["fixed_v"], c["wbar"], s); Z = np.sqrt(P); Z = Z - Z.mean(0)
    U, sv, _ = np.linalg.svd(Z, full_matrices=False); return (U*sv)[:, :npc]
def bic_step(c, Y, lin):   # 동결 사양: 구간별 상수 대 구간별 선형(불연속 허용)
    T, N, M = c["T"], Y.size, c["MINLEN"]; R = np.full((T+1, T+1), np.inf)
    for a_ in range(T+1):
        for b_ in range(a_+M, T+1):
            y = Y[a_:b_]; x = np.arange(b_-a_); A = np.c_[np.ones_like(x), x] if lin else np.ones((len(x), 1)); R[a_, b_] = ((y - A @ np.linalg.lstsq(A, y, rcond=None)[0])**2).sum()
    best = np.full((c["KMAX"]+1, T+1), np.inf); best[0, 0] = 0
    for k in range(1, c["KMAX"]+1): best[k] = np.min(best[k-1][:, None] + R, axis=0)
    return [dict(model="linear" if lin else "constant", k=k, rss=best[k, T], bic=N*np.log(best[k, T]/N + 1e-12) + (k*5*(2 if lin else 1) + (k-1))*np.log(N)) for k in range(1, c["KMAX"]+1) if np.isfinite(best[k, T])]
def bic_continuous(c, Y, kmax_bp=4):   # 탐색적: 연속 꺾은선(힌지) 모형, 꺾임 수 0..kmax_bp, 꺾임 간격>=MINLEN
    T, N, M = c["T"], Y.size, c["MINLEN"]; t = np.arange(T, dtype=float); out = []
    for kb in range(0, kmax_bp+1):
        best = (np.inf, None)
        for bps in itertools.combinations(range(M, T-M+1), kb):
            if any(b2-b1 < M for b1, b2 in zip(bps[:-1], bps[1:])): continue
            A = np.c_[np.ones(T), t, *[np.maximum(t-b, 0) for b in bps]]; rss = ((Y - A @ np.linalg.lstsq(A, Y, rcond=None)[0])**2).sum()
            if rss < best[0]: best = (rss, bps)
        out.append(dict(model="piecewise_linear_continuous", k=kb+1, rss=best[0], bic=N*np.log(best[0]/N + 1e-12) + 5*(2+kb)*np.log(N), breakpoints=[int(c["YEARS"][b]) for b in best[1]]))
    return out

# ------------------------------ 모드 ------------------------------
def a1():
    c = make_ctx(); sfx = "_QUICK" if QUICK else ""; Bp = 40 if QUICK else c["cfg"]["B_power"]; rows = []
    size_check(c, Bp, rows)
    if QUICK:
        print("\n[QUICK] 오류 점검 전용: 재현 OK와 delta=0 기각률만 확인. 곡선·판독 규칙은 본 실행(QUICK=False)에서만 해석(v3 방지).")
        power_grid(c, [1], 8, rows)   # 코드 경로 점검용 최소 실행, 결과 출력 안 함
        return
    power_grid(c, c["cfg"]["delta_x"], Bp, rows)
    PW = pd.DataFrame(rows); pw = PW[PW.kind == "power"]
    print("\n[Post-hoc] PA-A1 검정력(S5 N2 기준)\n", pw.pivot_table(index=["sample", "t_star"], columns="delta_x", values="power").round(2).to_string())
    print("\n[수렴 flag]\n", pw[pw.conv_flag.fillna("") != ""][["sample", "t_star", "delta_x", "conv_flag"]].drop_duplicates().to_string(index=False) or "없음")
    print("\n[표본별 실현 점프 비율(delta=1)]\n", pw[pw.delta_x == 1].pivot_table(index="sample", columns="t_star", values="J_extra_ratio").round(2).to_string())
    one = pw[pw.delta_x == 1].groupby("sample").power.agg(["min", "median"]).round(2); print("\n[delta=1 요약]\n", one.to_string())
    for s, r in one.iterrows(): print(f"  {s}: 사전 판독 → " + ("강화(연속 변화 증거)" if r["min"] >= 0.8 else ("리프레이밍(판정 불가)" if r["median"] < 0.5 else "곡선으로만 보고")))
    bic = []
    for s in c["SAM"]:
        Y = pcs(c, s); bic += [dict(sample=s, **r) for lin in (False, True) for r in bic_step(c, Y, lin)]
    BC = pd.DataFrame(bic); print("\n[Post-hoc] BIC 최솟값(상수 대 선형; 자기상관으로 과분할 가능, 보조 지표)\n", BC.loc[BC.groupby(["sample", "model"]).bic.idxmin()].round(1).to_string(index=False))
    PW.to_csv(PA/"PA_A1_v2_power_curve.csv", index=False, encoding="utf-8-sig"); pd.DataFrame(c["info"]).to_csv(PA/"PA_A1_v2_observed.csv", index=False, encoding="utf-8-sig"); BC.to_csv(PA/"PA_A1_v2_bic.csv", index=False, encoding="utf-8-sig")
    print("\n※ 검정력은 '관측 2020 점프 방향의 점프'에 한정. S0~S8 판정 불변.")
    with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f:
        f.write(f"- {now()} | PA-A1(v2) | 정의: PA_0_PLAN_v2 {c['sp']['sha256'][:12]} | 코드 패치(사양 불변): for/else 버그 수정, 수렴 flag, QUICK 해석 금지 | 결과: PA_A1_v2_*.csv | 본 결과 변경 여부: 없음\n")

def explore():
    """동결 밖 탐색적 확장. 모든 산출물·표 제목에 [Post-hoc][Exploratory]. 본 판독 규칙에 쓰지 않음."""
    c = make_ctx(); Bp = 40 if QUICK else c["cfg"]["B_power"]; sfx = "_QUICK" if QUICK else ""; rows = []
    print("[Post-hoc][Exploratory] 작은 점프(0.25x) 검정력")
    power_grid(c, [0.25], Bp, rows); PW = pd.DataFrame(rows)
    print(PW[PW.kind == "power"].pivot_table(index=["sample", "t_star"], columns="delta_x", values="power").round(2).to_string())
    bic = []
    for s in c["SAM"]:
        Y = pcs(c, s); bic += [dict(sample=s, **r) for r in bic_continuous(c, Y)]
    BC = pd.DataFrame(bic); print("\n[Post-hoc][Exploratory] 연속 꺾은선 BIC(꺾임 수 0~4)\n", BC.drop(columns="rss").round(1).to_string(index=False))
    PW.to_csv(PA/f"PA_A1_v2_EXPLORATORY_power025{sfx}.csv", index=False, encoding="utf-8-sig"); BC.to_csv(PA/f"PA_A1_v2_EXPLORATORY_bic_continuous{sfx}.csv", index=False, encoding="utf-8-sig")
    if not QUICK:
        with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f:
            f.write(f"- {now()} | PA-A1(v2) 탐색적 확장 | 정의: 동결 밖(delta=0.25, 연속 꺾은선 BIC), 판독 규칙에 미사용 | 결과: PA_A1_v2_EXPLORATORY_*.csv | 본 결과 변경 여부: 없음\n")

try: {"freeze2": freeze2, "tag2": tag2, "a1": a1, "explore": explore}[MODE]()
except Halt as e: print(f"⛔ 중단: {e}")

add 0 
commit 0 [main ae98ccb] PA-0 plan v2 (PA-A1 detector -> S5 original, before execution)
 2 files changed, 83 insertions(+)
 create mode 100644 paper1/09_PA/PA_0_PLAN_v2.json

tag 0 
※ 이어서 git push --follow-tags 로 원격 타임스탬프를 확보하세요.


In [ ]:
# # 지금까지 한 일과 앞으로 할 일 (2026-10-19 17:39 기준)

# ## 1. 지금까지 한 일

# ### 1-1. 사후분석(PA) 체계 정리
# | 항목 | 내용 | 상태 |
# |---|---|---|
# | 평가 | 지침서·코드·로그 전체 검토. 큰 위험은 N2 귀무의 과적합 가능성, 검정력 해석 범위, min/median 판독 규칙의 비대칭 | 완료 |
# | 코드 패치 | `for/else` 버그 수정, 수렴 flag, QUICK 해석 금지, `explore` 모드(δ=0.25, 연속 꺾은선 BIC) 추가 | 완료 |
# | 동결 사양 | `PA_A1_V2` 딕셔너리는 원본과 글자 단위로 동일. `freeze2` 재실행에서 "이미 동결: 동일" 확인 | 완료 |

# ### 1-2. 동결·태그·푸시
# | 시점 | 커밋 | 태그 | 내용 |
# |---|---|---|---|
# | v1 계획 동결 | `738a423` | `PA-0-plan-2026-10-19` | 사후분석 14개 항목, 판독 규칙 |
# | v2 계획 동결 | `ae98ccb` | `PA-0-plan-v2-2026-10-19` | PA-A1 탐지기를 S5 원본으로 교체, 해시 `a98f9697fa67` |
# | 실행 코드 | `26e7e64` | `PA-A1-v2-code-2026-10-19` | 패치된 `pa_a1_v2_run.py`, 시험 실행 전 상태 |
# | 본 실행 설정 | `15f545a` | `PA-A1-v2-fullrun-2026-10-19` | `QUICK=False`만 변경 |

# - GitHub(`LEEYJ1021/transport-knowledge-paper1`)에 `main`과 위 태그, S0~S8 동결 태그를 푸시했습니다. 이 푸시가 외부 타임스탬프입니다.
# - `S5-results-2026-10-16`과 `S6-0-freeze-2026-10-16`은 lightweight 태그라 `--follow-tags`에서 빠졌고, 따로 푸시했습니다.

# ### 1-3. 태그 정정
# - 잘못된 태그 3개를 로컬에서 삭제했고 푸시 전이라 원격에는 올라간 적이 없습니다.
#   - `S4-main-2026-10-16`, `S4-final-2026-10-16`: 실제로는 S4 산출물이 없는 `33b3912`를 가리켰습니다.
#   - `PA-A1-v2-code-2026-10-19`(첫 시도): 코드 파일 없이 `&&` 없는 명령으로 만들어진 것이었고, 이후 올바른 커밋에 다시 찍었습니다.

# ### 1-4. 재현성 점검(PA-0)에서 확인된 것
# - S1~S8 manifest의 파일 해시 152건이 현재 파일과 모두 일치합니다(불일치 0, 파일없음 0). 이 기록은 `PA_CHANGELOG.md`에 한 줄 추가됐습니다.
# - S4 코드 복구 후보를 판정했습니다(`s4_main` ← 히스토리 1079:23, `s4_6_rotation` ← 1080:4, `s4_6b_floor_check` ← 1080:6).
# - `s4_6_output.txt`(129자)는 오류 메시지이고 실제 출력이 아님을 발견했습니다.

# ### 1-5. PA-A1 시험 실행(QUICK=True) 결과
# - 관측 R 재현 OK: `all` 0.7595, `fixed14` 0.7755, `within` 0.7033으로 S5와 일치
# - δ=0 기각률: 0.0, 0.025, 0.10 (B=40에서 정상 범위)
# - 4개 t* 모두 오류 없이 완료
# - 관측 2020 점프(3년 창 Hellinger): `all` 0.190, `fixed14` 0.179, `within` 0.150. S5의 `jump` 통계(0.29)와는 정의가 다릅니다.

# ### 1-6. 지금 진행 중
# - **PA-A1 본 실행**(`QUICK=False`, B=200)이 17:37에 시작해 백그라운드로 돌고 있습니다(PID 2316587). 끝나야 CSV 3개와 로그가 한꺼번에 나옵니다.

# ## 2. 아직 안 한 일 (PA-0 마무리)

# | 항목 | 상태 | 할 일 |
# |---|---|---|
# | S4 코드 3종 저장 | **미저장** (`s4_main.py` 등이 untracked 목록에도 없음) | 복구·기록 셀을 `DRY=False`로 실행 |
# | 태그 정정 CHANGELOG | **미기록** | 삭제한 태그와 사유를 `CHANGELOG_S4.md`·`PA_CHANGELOG.md`에 기록(위 복구 셀이 같이 기록) |
# | 미추적 산출물 커밋 | **미실행** | `02_S2`, `03_S3`, `04_S4` 산출물과 노트북은 아직 커밋되지 않았음. 정리 셀 `MODE="commit"`을 `DRY=True`로 계획 확인 후 `DRY=False` |
# | `.gitignore` | 부분 | `oa_raw/`, `_backup_before_income_tv/`, `archive/*.tar.gz` 외에 `.ipynb_checkpoints/`, `_recovered_s4/` 추가 필요 |
# | `oa_raw` 외부 백업 | **미확인** | `archive/oa_raw_20261013.tar.gz`를 저장소 밖(외장 디스크 등)에 복사 |
# | `MASTER_MANIFEST.csv` | **코드 없음** | 단계별 해시를 한 장으로 통합하는 코드 작성 |
# | 원격 비공개 확인 | **미확인** | GitHub에서 저장소가 Private인지 확인 |
# | lightweight 태그 메모 | 미기록 | 부록 표에 "`S5-results`, `S6-0-freeze`는 lightweight, 외부 시각은 10-19 푸시 시각" |

# 참고로 정리 셀의 "잘못된 태그 삭제" 단계는 이미 수동으로 처리했으므로 비어 있을 것입니다. 커밋 날짜가 생성일이 아니라는 문구도 CHANGELOG에 넣어야 합니다.

# ## 3. 앞으로 할 일 (순서)

# ### 단계 A. PA-A1 본 실행 후 (지금 대기 중)
# 1. 완료 확인: `ls -l paper1/09_PA/PA_A1_v2_*.csv`
# 2. 로그 전체를 붙여 주시면 **동결된 판독 규칙**대로 읽습니다.
#    - δ=0 기각률이 약 0.02~0.09인지. 벗어나면 검정력 해석 전에 먼저 문제를 봅니다.
#    - 수렴 flag와 표본별 점프 실현 비율(`within`은 1보다 작을 가능성)
#    - δ=1에서 t*별 검정력의 최솟값·중앙값 → 강화 / 판정 불가 / 곡선만 보고
# 3. 판독이 어떻게 나와도 B1 문구와 제목을 정리합니다. 제목에서 "체제 전환"을 빼는 방향이 유력합니다.
# 4. 결과 열람 후에는 정의·임계값을 바꾸지 않습니다. 바꾸면 v3로 분리해야 하고 "결과 열람 전" 주장이 약해집니다.
# 5. `MODE="explore"`로 δ=0.25 검정력과 연속 꺾은선 BIC를 돌립니다. 결과는 `[Post-hoc][Exploratory]`로만 표기하고 판독 규칙에는 쓰지 않습니다.

# ### 단계 B. 핵심 PA 나머지 둘
# | PA | 내용 | 필요한 것 |
# |---|---|---|
# | **PA-B2** | 2020 점프가 표본 크기·저널 구성 아티팩트인지 분리(연도당 3,800편 하향 추출 B=200, 2020 점프를 구성하는 토픽 상위 10) | A1 결과. 하향 추출은 셀 크기를 바꿔야 해서 `gen_N2` 재사용에 제한이 있음 |
# | **PA-A2** | S6 신규 진입 우세의 구성 방어(ID 기반 대조 2020+, 진입 정의 2종, 부트스트랩 편향 보정과 저자 단위 클러스터 부트스트랩) | **`s6_lib.py`, `s6_run.py`의 함수 목록과 신규·지속 판별 코드** (아직 못 봄) |

# ### 단계 C. 정합성 점검
# - **PA-C1:** 중국 소속 신규 저자를 제외한 S6 분해로 진입 우세가 유지되는지(S6×S7 연결)
# - **PA-C2:** S7 국가 몫을 임베딩 보유 논문만으로 재계산해 ±0.01 이내인지

# ### 단계 D. 약점 리프레이밍
# - **PA-B3:** 신생 핵심 집합(n_defs≥3)과 D1' 재구성. **F6 생성 코드 위치**와 `(2) 데이터셋정의.ipynb`의 `BASE` 정의가 필요합니다.
# - **PA-B4:** F1+F3 이분 등급 재추정(145개 토픽)
# - **PA-B5:** S4 안정성 Wilson 구간, 분위 컷 곡선, 방향 축만 재계산
# - **PA-B7:** 결측 경계(최악·최선 대입) 분석, 분류기 대입, 저널×연도 IPW
# - **PA-A3, B9, B8:** S7 국가 미상 교정·균등 4구간·중국 제외 HHI, 소득군 내 상관, 미달 4개 도메인 부록 정렬
# - **PA-B6:** ORCID 소표본과 수작업 50쌍. 인력이 필요하니 병렬로 진행하세요.

# ### 단계 E. 마무리
# - **PA-A4:** 동결·CHANGELOG 체계를 부록 표 한 장으로 정리. 열람한 항목은 빠짐없이 기재(s5_lib 소스와 N2 요약은 v2 동결 전에 열람했음).
# - 모든 PA 표와 그림 제목에 `[Post-hoc]` 표지, 기각된 PA 목록을 부록에 수록
# - 영문 원고에 "Post-hoc analyses were specified after inspecting the primary results and are reported as exploratory; no pre-registered decision rule was altered." 포함
# - S0~S8 통과·실패 판정은 PA 결과로 바꾸지 않습니다.

# ## 4. 본문에 쓸 때의 주의
# - 검정력은 "관측된 2020 점프 방향의 점프"에 한정됩니다. 다른 모양의 변화에 대한 검정력은 아닙니다.
# - v2는 "결과 미열람 상태에서의 사후 명세"입니다. "사전등록"이라고 쓰지 마세요.
# - 3년 창 점프(0.19)와 S5의 `jump` 통계(0.29)를 섞어 쓰지 마세요.
# - BIC 비교는 자기상관 때문에 과분할 쪽으로 기울 수 있으니 보조 지표로만 쓰세요.
# - 해석은 "함께 움직인다"까지입니다. 연속 변화와 체제 전환 모두 인과 주장이 아닙니다.

# ## 5. 지금 필요한 입력
# 1. **PA-A1 본 실행 로그**(완료 후)
# 2. **`s6_lib.py`, `s6_run.py` 함수 목록**(PA-A2용). 원하시면 읽기 전용 추출 스크립트를 드립니다.
# 3. **F6 생성 코드 위치와 `BASE` 정의**(PA-B3용)
# 4. **수작업 인력**: PA-B6(50쌍)과 PA-B8-2(새 표본 25편 코딩)를 할 시간이 있는지
# 5. 우선순위를 A1 → A2 → B2로 유지할지

# 가장 가까운 한 걸음은 본 실행이 끝나기를 기다리는 것이고, 그 사이에 PA-0 마무리 중 코드 복구 저장과 정리 커밋을 `DRY=True`로 계획 확인해 둘 수 있습니다. 본 실행이 같은 CPU를 쓰므로 무거운 작업은 피하세요.

In [ ]:
# yjlee@u24:~/Research/Minnesota_Transport/paper1$ ps -p 2316587 -o pid,etime,cmd
#     PID     ELAPSED CMD
# [1]+  Done                    nohup python3 pa_a1_v2_run.py > 09_PA/PA_A1_v2_full_log.txt 2>&1
# yjlee@u24:~/Research/Minnesota_Transport/paper1$ ls -l 09_PA/PA_A1_v2_*.csv
# cat 09_PA/PA_A1_v2_full_log.txt
# -rw-rw-r-- 1 yjlee yjlee 1995 Oct 19 17:38 09_PA/PA_A1_v2_bic.csv
# -rw-rw-r-- 1 yjlee yjlee  312 Oct 19 17:38 09_PA/PA_A1_v2_observed.csv
# -rw-rw-r-- 1 yjlee yjlee 5637 Oct 19 17:38 09_PA/PA_A1_v2_power_curve.csv
# nohup: ignoring input
# [관측 재현 OK: S5와 R 일치]
#   sample  obs_R  obs_k         obs_breaks  N2_B  N2_q95   p_N2
#     all 0.7595      3       [2012, 2020]   500  0.7723 0.9441
# fixed14 0.7755      4 [2008, 2015, 2020]   500  0.7965 0.3992
#  within 0.7033      4 [2009, 2015, 2020]   500  0.7181 0.6966

# 관측 2020 점프(Hellinger, 3년 창): {'all': 0.1901, 'fixed14': 0.1788, 'within': 0.1495}
# [검정 크기 점검 delta=0: 기각률(약 0.05여야 함)] {'all': 0.035, 'fixed14': 0.075, 'within': 0.065}
#   t*=2008 완료
#   t*=2012 완료
#   t*=2015 완료
#   t*=2020 완료

# [Post-hoc] PA-A1 검정력(S5 N2 기준)
#  delta_x          0.5   1.0   2.0   3.0
# sample  t_star
# all     2008    0.03  0.07  0.03  0.04
#         2012    0.06  0.05  0.04  0.06
#         2015    0.06  0.05  0.03  0.06
#         2020    0.02  0.06  0.06  0.04
# fixed14 2008    0.07  0.06  0.08  0.06
#         2012    0.06  0.04  0.06  0.06
#         2015    0.04  0.08  0.03  0.08
#         2020    0.06  0.06  0.06  0.06
# within  2008    0.06  0.08  0.08  0.07
#         2012    0.06  0.06  0.07  0.06
#         2015    0.08  0.08  0.07  0.08
#         2020    0.07  0.08  0.08  0.06

# [수렴 flag]
#   sample  t_star  delta_x       conv_flag
#     all    2008      0.5 conv_off(6.189)
# fixed14    2008      0.5 conv_off(6.189)
#  within    2008      0.5 conv_off(6.189)
#     all    2008      1.0 conv_off(3.095)
# fixed14    2008      1.0 conv_off(3.095)
#  within    2008      1.0 conv_off(3.095)
#     all    2008      2.0 conv_off(1.547)
# fixed14    2008      2.0 conv_off(1.547)
#  within    2008      2.0 conv_off(1.547)
#     all    2008      3.0 conv_off(1.032)
# fixed14    2008      3.0 conv_off(1.032)
#  within    2008      3.0 conv_off(1.032)
#     all    2012      0.5 conv_off(6.517)
# fixed14    2012      0.5 conv_off(6.517)
#  within    2012      0.5 conv_off(6.517)
#     all    2012      1.0 conv_off(3.258)
# fixed14    2012      1.0 conv_off(3.258)
#  within    2012      1.0 conv_off(3.258)
#     all    2012      2.0 conv_off(1.629)
# fixed14    2012      2.0 conv_off(1.629)
#  within    2012      2.0 conv_off(1.629)
#     all    2012      3.0 conv_off(1.086)
# fixed14    2012      3.0 conv_off(1.086)
#  within    2012      3.0 conv_off(1.086)
#     all    2015      0.5 conv_off(6.318)
# fixed14    2015      0.5 conv_off(6.318)
#  within    2015      0.5 conv_off(6.318)
#     all    2015      1.0 conv_off(3.159)
# fixed14    2015      1.0 conv_off(3.159)
#  within    2015      1.0 conv_off(3.159)
#     all    2015      2.0 conv_off(1.579)
# fixed14    2015      2.0 conv_off(1.579)
#  within    2015      2.0 conv_off(1.579)
#     all    2015      3.0 conv_off(1.053)
# fixed14    2015      3.0 conv_off(1.053)
#  within    2015      3.0 conv_off(1.053)
#     all    2020      0.5 conv_off(6.063)
# fixed14    2020      0.5 conv_off(6.063)
#  within    2020      0.5 conv_off(6.063)
#     all    2020      1.0 conv_off(3.032)
# fixed14    2020      1.0 conv_off(3.032)
#  within    2020      1.0 conv_off(3.032)
#     all    2020      2.0 conv_off(1.516)
# fixed14    2020      2.0 conv_off(1.516)
#  within    2020      2.0 conv_off(1.516)

# [표본별 실현 점프 비율(delta=1)]
#  t_star   2008  2012  2015  2020
# sample
# all      3.09  3.26  3.16  3.03
# fixed14  0.00  0.00  0.00 -0.00
# within  -0.01  0.01  0.01  0.01

# [delta=1 요약]
#            min  median
# sample
# all      0.05    0.05
# fixed14  0.04    0.06
# within   0.06    0.08
#   all: 사전 판독 → 리프레이밍(판정 불가)
#   fixed14: 사전 판독 → 리프레이밍(판정 불가)
#   within: 사전 판독 → 리프레이밍(판정 불가)

# [Post-hoc] BIC 최솟값(상수 대 선형; 자기상관으로 과분할 가능, 보조 지표)
#   sample    model  k  rss    bic
#     all constant  5  0.2 -672.2
#     all   linear  3  0.1 -745.3
# fixed14 constant  5  0.2 -681.0
# fixed14   linear  3  0.1 -748.9
#  within constant  4  0.5 -620.4
#  within   linear  3  0.2 -659.3

# ※ 검정력은 '관측 2020 점프 방향의 점프'에 한정. S0~S8 판정 불변.
# yjlee@u24:~/Research/Minnesota_Transport/paper1$

In [ ]:
# # PA-A1 본 실행 결과 판독

# **결론부터 말씀드리면, 이 결과는 "판정 불가"를 뒷받침하는 증거로 쓸 수 없습니다.** 사전 판독 규칙을 기계적으로 적용하면 세 표본 모두 "리프레이밍(판정 불가)"이 나옵니다. 하지만 아래 두 가지 때문에 이 실행은 검정력 분석으로서 성립하지 않는 것으로 보입니다.

# ## 1. 정상으로 확인된 것
# - 관측 R 재현 OK
# - δ=0 기각률: all 0.035, fixed14 0.075, within 0.065. 목표 범위(0.02~0.09) 안이라 검정 크기는 정상입니다.
# - BIC: 세 표본 모두 구간별 선형(k=3)이 구간별 상수(k=4~5)보다 BIC가 낮습니다(all −745 대 −672). 다만 선형 모형은 구간 경계의 불연속을 허용하고 자기상관 때문에 BIC가 편향되므로 보조 지표로만 쓰세요.

# ## 2. 문제 1: 검정력이 점프 크기와 무관하게 약 0.05
# δ가 0.5에서 3까지 변해도, t*가 어디여도 검정력은 0.02~0.08에 머뭅니다. 이는 δ=0일 때의 기각률(검정 크기)과 같은 수준입니다. 점프를 심어도 검정이 반응하지 않는다는 뜻입니다. 다음 두 가지 중 하나입니다.
# - 점프 주입 코드에 문제가 있다.
# - 탐지 통계(R = 1 − cost_k/cost_1)가 이런 점프에는 원래 둔감하다.

# ## 3. 문제 2: δ 보정 실패
# 수렴 flag가 거의 전 조합에서 떴습니다. 특히 realized ratio × δ가 모든 칸에서 약 3.09로 일정합니다(6.19×0.5, 3.095×1, 1.547×2, 1.032×3). 즉 δ를 바꿔도 실제로 주입된 점프는 항상 같은 값(관측 점프의 약 3.1배, 약 0.59)입니다. δ 축이 실제로는 작동하지 않았다는 뜻입니다.

# 또 δ=1에서 fixed14와 within의 실현 점프 비율은 0.00 근처입니다. 같은 계수를 모든 저널에 주입했는데도 `all` 표본에서만 큰 점프가 나타나고 나머지 두 표본에서는 점프가 거의 없습니다. 점프가 일부 저널(고정 14개 밖)에 몰렸거나 주입 구현에 문제가 있다는 신호입니다.

# 이 두 가지로 "δ=1에서 검정력 0.05"는 "관측 크기의 점프를 넣고도 못 잡았다"가 아닙니다. 실제로는 관측 크기가 아닌 점프를 넣었고, 그마저도 두 표본에는 거의 들어가지 않았습니다. 따라서 이 결과를 "검정력 부족"이나 "연속 변화"의 증거로 쓰면 안 됩니다.

# 원인이 코드 오류인지 탐지 통계의 둔감성인지는 아직 모릅니다. 제가 읽은 코드에서는 이분법 자체의 오류를 찾지 못했습니다. 짚이는 가설은 있지만 확인 없이 단정하지 않겠습니다.

# ## 4. 진단 (읽기 전용)
# 결과 파일과 주입 함수를 직접 들여다봅니다. CSV는 수정하지 않습니다.

# **(a) CSV에서 확인**
# ```
# cd ~/Research/Minnesota_Transport/paper1
# python3 - <<'EOF'
# import pandas as pd
# d = pd.read_csv("09_PA/PA_A1_v2_power_curve.csv")
# print(d[d.kind=="size_check"][["sample","power","R_mean"]])
# p = d[(d.kind=="power")&(d.delta_x==3)]
# print(p[["sample","t_star","a","power","R_mean","hit_near_tstar","J_extra_ratio"]].round(4).to_string(index=False))
# EOF
# ```
# 주입 크기(`a`)가 δ에 따라 변했는지, `R_mean`이 δ=0일 때보다 올랐는지, 변화점이 t*에서 잡혔는지(`hit_near_tstar`)를 봅니다.

# **(b) 주입 함수 점검.** 이 스크립트는 `try:` 이후 부분을 잘라내므로 `a1()`이 실행되어 CSV를 덮어쓰는 일이 없습니다.
# ```
# cd ~/Research/Minnesota_Transport/paper1
# python3 - <<'EOF'
# import numpy as np
# src = open("pa_a1_v2_run.py").read().split('try: {"freeze2"')[0]
# exec(src, globals())
# c = make_ctx(); it = int(np.where(c["YEARS"]==2020)[0][0]); a0 = 1.0/np.abs(c["d_vec"]).max()
# print("\na0 =", round(a0,4), "| max|d| =", round(np.abs(c["d_vec"]).max(),5))
# for s in c["SAM"]:
#     base = c["jump_exp"](c["P2"], s, it)
#     print(f"\n[{s}] base jump {base:.4f}")
#     for m in [0, 0.001, 0.01, 0.03, 0.1, 0.3, 1, 3]:
#         Q = c["inject"](m*a0, it); P, n = c["L"].sample_P(Q*c["cnt"].sum(2).astype(float)[:,:,None], c["fixed_v"], c["wbar"], s)
#         r = c["L"].detect(P, n, 0.0, 6, 3)  # 노이즈 없는 기대값 자료에서 R, k 확인(벌점 0: 진단 목적)
#         print(f"  a={m:>6}*a0 | jump_extra {c['jump_exp'](Q, s, it)-base:+.4f} | R(no-noise) {r['R']:.4f} k={r['k']}")
# EOF
# ```
# 이 출력으로 다음을 판별합니다.
# - 점프가 `a`에 대해 단조 증가하는지, 아니면 거의 계단처럼 뛰는지
# - `fixed14`와 `within`에서 점프가 정말 생기는지
# - 노이즈 없는 자료에서 R이 점프에 반응하는지(탐지 통계 둔감성 여부)

# 진단 코드는 제가 짠 것이고 실행해 본 적이 없습니다. `detect`의 `pen=0` 호출은 벌점을 0으로 두는 진단용이며 S5의 판정 방식이 아니니, 해석할 때 이 점을 감안하세요.

# ## 5. 기록과 절차

# **지금 할 것**
# 1. 이 실행 결과를 지우거나 덮어쓰지 마세요. CSV와 로그를 그대로 커밋하고 태그를 남깁니다. 이 시점부터 "PA-A1 결과 미열람"은 더 이상 사실이 아닙니다.
#    ```
#    cd ~/Research/Minnesota_Transport
#    git add paper1/09_PA/PA_A1_v2_*.csv paper1/09_PA/PA_A1_v2_full_log.txt paper1/09_PA/PA_CHANGELOG.md
#    git commit -m "PA-A1 v2 run 1 results (calibration failed; power ~ size; not interpretable)"
#    git tag -a PA-A1-v2-run1-2026-10-19 -m "run 1 outputs, seen"
#    git push --follow-tags
#    ```
# 2. `PA_CHANGELOG.md`에는 "run 1: 보정 실패(δ 무관 점프, fixed14·within 점프 0), 검정력 ≈ 크기. 판독 규칙의 기계적 결과는 판정 불가이나 해석 불가로 기록"이라고 한 줄 적어 두세요(자동 기록 줄에는 이 판단이 없습니다).

# **진단 이후 선택지**
# - **코드 오류가 확인되면:** 주입 구현의 오류 수정은 결과가 아니라 구현 결함에 대한 수정입니다. 다만 결과를 본 뒤의 수정이므로 v3 계획으로 분리하고, 사유를 "run 1의 보정 실패(실현 점프 비율이 δ와 무관, 두 표본에서 0)"로 기록하세요. run 1과 run 2를 둘 다 보고합니다.
# - **통계 둔감성이 확인되면:** 이는 S5의 N2 검정이 점프 유형의 변화에 둔감하다는 발견입니다. 이 경우 "판정 불가"는 오히려 S5의 R 통계 자체의 한계라는 논문 서술이 됩니다. 이건 부정적 결과이자 보고할 가치가 있는 결과입니다.
# - **둘 다 아니거나 불분명하면:** 점프 주입 방식(전 저널에 같은 방향 가산)이 현실적이지 않을 수 있으니, 저널 단위 주입 같은 대안을 v3로 검토합니다.

# ## 6. 논문 서술에 대한 주의
# - 이 결과로 B1 문구를 확정하지 마세요. "연속적 재배치" 서술은 지금 상태에서는 검정력 분석의 지지를 받지 못합니다. 다만 "판정 불가"로 쓰는 것도 아직 이릅니다.
# - 제목에서 "체제 전환"을 빼는 방향은 유지해도 되지만, 그 근거는 이 PA가 아니라 S5의 N2 비통과(부정적 증거)임을 분명히 하세요.

# (a), (b) 두 출력을 붙여 주시면 원인을 가르고 v3 필요 여부를 정리하겠습니다.

In [ ]:
# yjlee@u24:~/Research/Minnesota_Transport/paper1$ cd ~/Research/Minnesota_Transport/paper1
# python3 - <<'EOF'
# import pandas as pd
# d = pd.read_csv("09_PA/PA_A1_v2_power_curve.csv")
# print(d[d.kind=="size_check"][["sample","power","R_mean"]])
# p = d[(d.kind=="power")&(d.delta_x==3)]
# print(p[["sample","t_star","a","power","R_mean","hit_near_tstar","J_extra_ratio"]].round(4).to_string(index=False))
# EOF
#     sample  power    R_mean
# 0      all  0.035  0.765748
# 1  fixed14  0.075  0.758430
# 2   within  0.065  0.707200
#  sample  t_star   a  power  R_mean  hit_near_tstar  J_extra_ratio
#     all    2008 0.0  0.040  0.7654           0.000         3.0947
# fixed14    2008 0.0  0.065  0.7630           0.470         0.0000
#  within    2008 0.0  0.070  0.7078           0.945        -0.0069
#     all    2012 0.0  0.055  0.7659           0.995         3.2585
# fixed14    2012 0.0  0.060  0.7587           0.590         0.0000
#  within    2012 0.0  0.055  0.7067           0.000         0.0147
#     all    2015 0.0  0.055  0.7657           0.000         3.1588
# fixed14    2015 0.0  0.075  0.7627           0.450         0.0000
#  within    2015 0.0  0.080  0.7079           1.000         0.0076
#     all    2020 0.0  0.040  0.7653           1.000         3.0317
# fixed14    2020 0.0  0.060  0.7606           1.000        -0.0000
#  within    2020 0.0  0.065  0.7072           0.935         0.0067
# yjlee@u24:~/Research/Minnesota_Transport/paper1$ cd ~/Research/Minnesota_Transport/paper1
# python3 - <<'EOF'
# import numpy as np
# src = open("pa_a1_v2_run.py").read().split('try: {"freeze2"')[0]
# exec(src, globals())
# c = make_ctx(); it = int(np.where(c["YEARS"]==2020)[0][0]); a0 = 1.0/np.abs(c["d_vec"]).max()
# print("\na0 =", round(a0,4), "| max|d| =", round(np.abs(c["d_vec"]).max(),5))
# for s in c["SAM"]:
#     base = c["jump_exp"](c["P2"], s, it)
#     print(f"\n[{s}] base jump {base:.4f}")
#     for m in [0, 0.001, 0.01, 0.03, 0.1, 0.3, 1, 3]:
#         Q = c["inject"](m*a0, it); P, n = c["L"].sample_P(Q*c["cnt"].sum(2).astype(float)[:,:,None], c["fixed_v"], c["wbar"], s)
#         r = c["L"].detect(P, n, 0.0, 6, 3)  # 노이즈 없는 기대값 자료에서 R, k 확인(벌점 0: 진단 목적)
#         print(f"  a={m:>6}*a0 | jump_extra {c['jump_exp'](Q, s, it)-base:+.4f} | R(no-noise) {r['R']:.4f} k={r['k']}")
# EOF
# [관측 재현 OK: S5와 R 일치]
#   sample  obs_R  obs_k         obs_breaks  N2_B  N2_q95   p_N2
#     all 0.7595      3       [2012, 2020]   500  0.7723 0.9441
# fixed14 0.7755      4 [2008, 2015, 2020]   500  0.7965 0.3992
#  within 0.7033      4 [2009, 2015, 2020]   500  0.7181 0.6966

# 관측 2020 점프(Hellinger, 3년 창): {'all': 0.1901, 'fixed14': 0.1788, 'within': 0.1495}

# a0 = 57.5431 | max|d| = 0.01738

# [all] base jump 0.1299
#   a=     0*a0 | jump_extra +0.5762 | R(no-noise) 0.0000 k=1
#   a= 0.001*a0 | jump_extra +0.5762 | R(no-noise) 0.0000 k=1
#   a=  0.01*a0 | jump_extra +0.5766 | R(no-noise) 0.0000 k=1
#   a=  0.03*a0 | jump_extra +0.5769 | R(no-noise) 0.0000 k=1
#   a=   0.1*a0 | jump_extra +0.5771 | R(no-noise) 0.0000 k=1
#   a=   0.3*a0 | jump_extra +0.5772 | R(no-noise) 0.0000 k=1
#   a=     1*a0 | jump_extra +0.5772 | R(no-noise) 0.0000 k=1
#   a=     3*a0 | jump_extra +0.5772 | R(no-noise) 0.0000 k=1

# [fixed14] base jump 0.1093
#   a=     0*a0 | jump_extra -0.0000 | R(no-noise) 0.9618 k=6
#   a= 0.001*a0 | jump_extra +0.0182 | R(no-noise) 0.9644 k=6
#   a=  0.01*a0 | jump_extra +0.1374 | R(no-noise) 0.9775 k=6
#   a=  0.03*a0 | jump_extra +0.2929 | R(no-noise) 0.9875 k=6
#   a=   0.1*a0 | jump_extra +0.4483 | R(no-noise) 0.9939 k=6
#   a=   0.3*a0 | jump_extra +0.5180 | R(no-noise) 0.9953 k=6
#   a=     1*a0 | jump_extra +0.5489 | R(no-noise) 0.9957 k=6
#   a=     3*a0 | jump_extra +0.5556 | R(no-noise) 0.9958 k=6

# [within] base jump 0.0667
#   a=     0*a0 | jump_extra +0.0010 | R(no-noise) 0.0000 k=1
#   a= 0.001*a0 | jump_extra +0.0158 | R(no-noise) 0.0000 k=1
#   a=  0.01*a0 | jump_extra +0.1243 | R(no-noise) 0.0000 k=1
#   a=  0.03*a0 | jump_extra +0.2827 | R(no-noise) 0.0000 k=1
#   a=   0.1*a0 | jump_extra +0.4363 | R(no-noise) 0.0000 k=1
#   a=   0.3*a0 | jump_extra +0.5045 | R(no-noise) 0.0000 k=1
#   a=     1*a0 | jump_extra +0.5357 | R(no-noise) 0.0000 k=1
#   a=     3*a0 | jump_extra +0.5425 | R(no-noise) 0.0000 k=1
# yjlee@u24:~/Research/Minnesota_Transport/paper1$    cd ~/Research/Minnesota_Transport
#    git add paper1/09_PA/PA_A1_v2_*.csv paper1/09_PA/PA_A1_v2_full_log.txt paper1/09_PA/PA_CHANGELOG.md
#    git commit -m "PA-A1 v2 run 1 results (calibration failed; power ~ size; not interpretable)"
#    git tag -a PA-A1-v2-run1-2026-10-19 -m "run 1 outputs, seen"
#    git push --follow-tags
# [main 0007573] PA-A1 v2 run 1 results (calibration failed; power ~ size; not interpretable)
#  5 files changed, 199 insertions(+)
#  create mode 100644 paper1/09_PA/PA_A1_v2_bic.csv
#  create mode 100644 paper1/09_PA/PA_A1_v2_full_log.txt
#  create mode 100644 paper1/09_PA/PA_A1_v2_observed.csv
#  create mode 100644 paper1/09_PA/PA_A1_v2_power_curve.csv
# Enumerating objects: 14, done.
# Counting objects: 100% (14/14), done.
# Delta compression using up to 24 threads
# Compressing objects: 100% (10/10), done.
# Writing objects: 100% (10/10), 4.51 KiB | 4.51 MiB/s, done.
# Total 10 (delta 2), reused 0 (delta 0), pack-reused 0
# remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
# To https://github.com/LEEYJ1021/transport-knowledge-paper1.git
#    15f545a..0007573  main -> main
#  * [new tag]         PA-A1-v2-run1-2026-10-19 -> PA-A1-v2-run1-2026-10-19
# yjlee@u24:~/Research/Minnesota_Transport$

In [ ]:
# # 점검 결과: 수정이 통과했습니다

# run 1의 원인이던 이상 현상이 사라졌습니다.

# - **a=0에서 세 표본 모두 점프가 0**입니다. 이전에는 `all`이 +0.576이었습니다.
# - **`all`도 a에 대해 단조 증가한 뒤 포화**합니다(0.039 → 0.111 → 0.249 → 0.393 → 0.461 → 0.492 → 0.4995). `fixed14`와 `within`은 이전과 같고, 세 표본의 곡선이 비슷한 모양입니다.
# - 패치 줄과 파일명 치환(`run2`, 4곳)도 확인됐습니다.

# **run 2에서 예상되는 동작 (사양대로의 동작이라 문제가 아닙니다)**
# - `all`의 포화값이 약 0.50이므로 δ=3(목표 0.57)은 `unreachable`로 NaN 행이 생깁니다. `fixed14`와 `within`은 포화값이 0.54이지만 보정이 `all`에서 이뤄져서 같은 a를 쓰므로, 그 칸은 어차피 NaN입니다.
# - δ=2(목표 0.38)는 도달 가능하지만 포화 구간에 가까워 이분법 수렴 flag가 뜰 수 있습니다. flag가 뜨면 그 칸은 해석에서 따로 표시하세요.
# - 판독 규칙은 δ=1(목표 0.19, a≈0.015×a0 부근)에서만 적용하므로 영향이 없습니다.
# - 같은 a를 세 표본에 쓰기 때문에 표본별 실현 점프 비율은 1이 아닐 수 있습니다. 출력의 `J_extra_ratio` 표로 확인하세요. `within`은 관측 점프(0.1495)가 `all`(0.1901)보다 작아서 비율이 1보다 크게 나올 가능성이 큽니다.

# **이제 커밋·태그·푸시한 뒤 실행하세요.** 푸시를 먼저 하는 이유는 결과를 보기 전에 코드 상태의 외부 시각을 남기기 위해서입니다.

# ```
# cd ~/Research/Minnesota_Transport
# git add paper1/pa_a1_v2_run.py && git commit -m "PA-A1 v2 run2 code: fix inject() NaN normalization (n=0 cells); spec unchanged" && git tag -a PA-A1-v2-run2-code-2026-10-19 -m "run2 code, pre-run; run1 uninterpretable due to inject NaN bug" && git push --follow-tags
# cd paper1
# nohup python3 -u pa_a1_v2_run.py > 09_PA/PA_A1_v2_run2_full_log.txt 2>&1 &
# sleep 5; tail -f 09_PA/PA_A1_v2_run2_full_log.txt
# ```

# run 1은 약 1분에 끝났으니 이번에도 비슷할 것입니다(점프가 실제로 들어가면 탐지가 더 오래 걸릴 수 있습니다). `-u` 옵션 덕분에 로그가 실시간으로 쌓입니다.

# **기록**
# run 1 원인 확정 내용을 `PA_CHANGELOG.md`에 아직 적지 않으셨다면, 실행 전에 지난번 안내한 줄을 추가하세요. 실행 시작 전에 적어 두면 순서가 깔끔합니다.

# **본 실행 결과를 읽을 때 지켜야 할 것**
# - 판독 규칙(δ=1에서 t*별 검정력의 최솟값 ≥0.8 / 중앙값 <0.5)과 δ·t* 격자는 바꾸지 않습니다.
# - run 1과 run 2를 모두 부록에 싣고, run 2가 "run 1의 구현 결함만 고친 재실행"임을 명시합니다.
# - run 2 결과를 본 뒤 설정을 바꾸면 v3이며, 이번에는 그럴 사유가 없도록 로그 전체를 먼저 보고 판단하세요.

# 로그가 끝나면 전체를 붙여 주세요. δ=0 기각률(0.02~0.09), 수렴 flag, 실현 점프 비율, δ=1 검정력 표 순서로 읽겠습니다.

In [ ]:
# yjlee@u24:~/Research/Minnesota_Transport/paper1$ cd ~/Research/Minnesota_Transport
# git add paper1/pa_a1_v2_run.py && git commit -m "PA-A1 v2 run2 code: fix inject() NaN normalization (n=0 cells); spec unchanged" && git tag -a PA-A1-v2-run2-code-2026-10-19 -m "run2 code, pre-run; run1 uninterpretable due to inject NaN bug" && git push --follow-tags
# cd paper1
# nohup python3 -u pa_a1_v2_run.py > 09_PA/PA_A1_v2_run2_full_log.txt 2>&1 &
# sleep 5; tail -f 09_PA/PA_A1_v2_run2_full_log.txt
# [main d6eb03f] PA-A1 v2 run2 code: fix inject() NaN normalization (n=0 cells); spec unchanged
#  1 file changed, 5 insertions(+), 5 deletions(-)
# Enumerating objects: 8, done.
# Counting objects: 100% (8/8), done.
# Delta compression using up to 24 threads
# Compressing objects: 100% (5/5), done.
# Writing objects: 100% (5/5), 862 bytes | 862.00 KiB/s, done.
# Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
# remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
# To https://github.com/LEEYJ1021/transport-knowledge-paper1.git
#    0007573..d6eb03f  main -> main
#  * [new tag]         PA-A1-v2-run2-code-2026-10-19 -> PA-A1-v2-run2-code-2026-10-19
# [1] 2319625
# nohup: ignoring input
# [관측 재현 OK: S5와 R 일치]
#   sample  obs_R  obs_k         obs_breaks  N2_B  N2_q95   p_N2
#     all 0.7595      3       [2012, 2020]   500  0.7723 0.9441
# fixed14 0.7755      4 [2008, 2015, 2020]   500  0.7965 0.3992
#  within 0.7033      4 [2009, 2015, 2020]   500  0.7181 0.6966

# 관측 2020 점프(Hellinger, 3년 창): {'all': 0.1901, 'fixed14': 0.1788, 'within': 0.1495}
# [검정 크기 점검 delta=0: 기각률(약 0.05여야 함)] {'all': 0.035, 'fixed14': 0.075, 'within': 0.065}
#   t*=2008 완료
#   t*=2012 완료
#   t*=2015 완료
#   t*=2020 완료

# [Post-hoc] PA-A1 검정력(S5 N2 기준)
#  delta_x          0.5   1.0  2.0  3.0
# sample  t_star
# all     2008    0.90  1.00  1.0  1.0
#         2012    1.00  1.00  1.0  1.0
#         2015    1.00  1.00  1.0  1.0
#         2020    1.00  1.00  1.0  NaN
# fixed14 2008    0.56  0.44  1.0  1.0
#         2012    0.00  1.00  1.0  1.0
#         2015    1.00  1.00  1.0  1.0
#         2020    1.00  1.00  1.0  NaN
# within  2008    0.26  0.90  1.0  1.0
#         2012    0.10  1.00  1.0  1.0
#         2015    1.00  1.00  1.0  1.0
#         2020    1.00  1.00  1.0  NaN

# [수렴 flag]
#   sample  t_star  delta_x   conv_flag
#     all    2020      3.0 unreachable
# fixed14    2020      3.0 unreachable
#  within    2020      3.0 unreachable

# [표본별 실현 점프 비율(delta=1)]
#  t_star   2008  2012  2015  2020
# sample
# all      1.00  1.00  1.00  1.00
# fixed14  1.09  1.09  1.16  1.27
# within   1.34  1.27  1.09  1.43

# [delta=1 요약]
#            min  median
# sample
# all      1.00     1.0
# fixed14  0.44     1.0
# within   0.90     1.0
#   all: 사전 판독 → 강화(연속 변화 증거)
#   fixed14: 사전 판독 → 곡선으로만 보고
#   within: 사전 판독 → 강화(연속 변화 증거)

# [Post-hoc] BIC 최솟값(상수 대 선형; 자기상관으로 과분할 가능, 보조 지표)
#   sample    model  k  rss    bic
#     all constant  5  0.2 -672.2
#     all   linear  3  0.1 -745.3
# fixed14 constant  5  0.2 -681.0
# fixed14   linear  3  0.1 -748.9
#  within constant  4  0.5 -620.4
#  within   linear  3  0.2 -659.3

# ※ 검정력은 '관측 2020 점프 방향의 점프'에 한정. S0~S8 판정 불변.

In [ ]:
# # PA-A1 run 2 결과 판독

# run 2는 정상적으로 작동했고, 사전 판독 규칙의 기계적 결과는 세 표본이 서로 다르게 나왔습니다. 다만 규칙 문구가 말하는 "이 크기의 점프"는 실제로 훨씬 큰 점프여서, 본문에 쓸 때는 범위를 좁혀야 합니다.

# ## 1. 정상 작동 확인
# - **δ=0 기각률**은 0.035, 0.075, 0.065로 run 1과 같고 목표 범위 안입니다. 점프 주입 이전 단계가 수정과 무관하다는 확인이기도 합니다.
# - **수렴:** `unreachable`은 t*=2020, δ=3뿐입니다. 나머지는 2% 이탈 flag가 없어 보정이 수렴했습니다. 앞에서 "δ=2, 3은 대체로 도달 불가"라고 예상했는데 틀렸고, 포화값이 t*마다 달랐습니다.
# - **실현 점프 비율**은 `all`이 1.00으로 맞고, `fixed14`는 1.09~1.27, `within`은 1.09~1.43으로 목표보다 큽니다. 같은 a를 세 표본에 쓴 결과이고 사양대로입니다.

# ## 2. 동결 판독 규칙의 기계적 결과 (δ=1)
# | 표본 | 최솟값 | 중앙값 | 사전 판독 |
# |---|---|---|---|
# | all | 1.00 | 1.0 | 강화 |
# | fixed14 | 0.44 (t*=2008) | 1.0 | 곡선으로만 보고 |
# | within | 0.90 | 1.0 | 강화 |

# 세 표본의 판독이 갈렸으므로 "강화"를 일괄해서 쓸 수 없습니다. fixed14는 규칙에 따라 곡선만 보고합니다.

# ## 3. 본문에 쓰기 전에 짚어야 할 점

# **(1) "δ=1"은 관측 점프 크기가 아닙니다.** 앞서 제가 "관측 크기의 점프"라고 쓴 표현은 부정확했습니다. 동결된 `jump_def`는 "추세 기저 위에 더해지는 추가 점프 = δ × 관측 2020 점프 전체"입니다. 진단 출력(`base jump`)을 쓰면 관측 점프에서 추세 기저가 이미 만드는 점프를 뺀 초과분은 다음과 같습니다.

# | 표본 | 관측 점프 | 추세 기저 점프 | 관측 초과분 |
# |---|---|---|---|
# | all | 0.190 | 0.130 | 약 0.060 |
# | fixed14 | 0.179 | 0.109 | 약 0.070 |
# | within | 0.150 | 0.067 | 약 0.083 |

# δ=1에서 주입한 추가 점프는 `all` 기준 0.19이므로 관측 초과분의 약 3배입니다. 따라서 규칙에 적힌 "이 크기의 점프가 있었다면 탐지했을 것"은 과장입니다. 본문에서는 "관측 2020 점프와 같은 크기의 추가 점프(관측 초과분의 약 3배)"라고 정확히 쓰세요. 이는 동결 문구의 정의 문제이므로 부록에 투명하게 적는 것이 맞습니다.

# **(2) 가장 관련 있는 칸은 t*=2020입니다.** 관측된 후보 변화점이 2020이기 때문입니다. 이 줄에서는 δ=0.5에서도 검정력이 세 표본 모두 1.00입니다. δ=0.5의 추가 점프(약 0.095)는 `all` 기준 관측 초과분(0.060)의 약 1.6배입니다. 이 한 줄은 "2020에 초과분의 1.6배쯤 되는 점프가 있었다면 잡혔을 것"이라는 비교적 강한 근거입니다. 다만 초과분과 같은 크기(δ≈0.3~0.45)에서의 검정력은 아직 측정하지 않았습니다.

# **(3) 비단조 칸이 해석되지 않았습니다.**
# - fixed14, t*=2012: δ=0.5에서 0.00, δ=1에서 1.00
# - fixed14, t*=2008: δ=0.5에서 0.56, δ=1에서 0.44
# - within, t*=2012: δ=0.5에서 0.10

# B=200이면 표준오차가 약 0.035이므로 0.00 대 1.00은 잡음이 아닙니다. 짚이는 가설은 `fixed14`의 N2 R 분포가 k=3과 k=4 사이에서 둘로 갈라진 형태(앞서 나온 `N2 반복 분포` 표)여서 임계점 근처에서 반응이 계단처럼 뛴다는 것인데, 확인하지 않은 가설입니다. 아래 진단으로 확인하세요.

# **(4) N2 과적합 우려는 부분적으로 해소됐습니다.** 이전 평가에서 N2 귀무가 관측 구조를 너무 잘 흡수해 점프에 둔감할 수 있다고 우려했는데, 큰 점프(δ≥1)에서는 N2가 흡수하지 못하고 잡아냈습니다. 다만 작은 점프에서는 `fixed14`와 `within`의 민감도가 낮고 불안정합니다.

# **(5) BIC는 run 1과 같은 숫자입니다.** 같은 계산이고 새 증거가 아닙니다. 구간별 선형(k=3)이 구간별 상수보다 낮지만, 불연속을 허용하는 선형 모형이고 자기상관 편향이 있어 보조 지표로만 쓰세요.

# ## 4. B1 문구에 대한 영향
# - 동결 규칙의 결과가 "강화 2, 곡선만 1"이므로 "연속 변화의 증거"를 일괄 주장하지 마세요.
# - 근거 있는 서술은 다음과 같습니다. "N2 비통과는, 2020 부근에서 관측 초과분의 약 1.6배 이상의 추가 점프가 있었다면 탐지되었을 것이라는 점에서, 큰 점프의 부재와 부합한다. 더 작은 점프나 다른 t*에서는 표본에 따라 검정력이 낮거나 불안정하다."
# - 즉 "큰 점프는 배제하지만 관측 초과분 크기의 점프는 판정 불가"입니다. 제목에서 "체제 전환"을 빼는 방향은 유지해도 되지만, "연속 재배치가 확인됐다"는 문구는 이 결과로 지지되지 않습니다.
# - 검정력은 관측 2020 점프 방향의 점프에 한정됩니다.

# ## 5. 지금 할 일 (순서)

# **① 결과를 커밋하고 푸시합니다.** 코드가 끝난 뒤의 열람 기록이므로 태그를 남기세요.
# ```
# cd ~/Research/Minnesota_Transport
# git add paper1/09_PA/PA_A1_v2_run2_*.csv paper1/09_PA/PA_A1_v2_run2_full_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-A1 v2 run 2 results (inject NaN fixed; calibrated)" && git tag -a PA-A1-v2-run2-results-2026-10-19 -m "run2 outputs, seen" && git push --follow-tags
# ```

# **② `PA_CHANGELOG.md`에 직접 한 줄을 추가합니다.**
# ```
# - 2026-10-19 | PA-A1 v2 run 2 결과: δ=0 기각률 0.035/0.075/0.065. δ=1 판독 all 강화(min 1.00), fixed14 곡선만 보고(min 0.44, t*=2008), within 강화(min 0.90). 주의: δ=1 추가 점프는 관측 초과분(점프−추세 기저점프)의 약 3배. t*=2020은 δ=0.5에서 세 표본 모두 1.00. fixed14·within에서 비단조 칸 존재(미해석). 판독 규칙·격자 변경 없음 | 본 결과 변경: 없음
# ```

# **③ 비단조 칸 진단(읽기 전용)**
# ```
# cd ~/Research/Minnesota_Transport/paper1
# python3 - <<'EOF'
# import pandas as pd
# d = pd.read_csv("09_PA/PA_A1_v2_run2_power_curve.csv"); p = d[(d.kind=="power")&(d.sample.isin(["fixed14","within"]))&(d.delta_x.isin([0.5,1]))&(d.t_star.isin([2008,2012]))]
# print(p[["sample","t_star","delta_x","a","power","R_mean","hit_near_tstar","J_extra_ratio"]].round(3).to_string(index=False))
# EOF
# ```
# `R_mean`이 N2 q95(`fixed14` 0.7965, `within` 0.7181)에 비해 어디에 있는지, `hit_near_tstar`가 낮은지를 보세요. 낮다면 점프는 있었는데 탐지기가 다른 위치에 변화점을 찍은 것이고, 이는 R 통계의 한계와 위치 회수 결과를 구분해서 보고할 근거가 됩니다.

# **④ 탐색적 확장(선택).** 초과분 크기의 점프에서 검정력을 보려면 `explore`를 δ=0.25, 0.35, 0.45로 돌립니다. 초과분을 δ로 환산하면 대략 0.32(all), 0.37(fixed14), 0.44(within)입니다. 이 확장은 동결 밖이라 판독 규칙에 쓰지 않고 `[Post-hoc][Exploratory]`로만 보고합니다. 패치는 ①~③이 끝난 뒤 드리겠습니다. 이 확장 결과도 열람 후 분석이므로 본 결과를 바꾸는 근거로 쓰지 마세요.

# ③의 출력을 붙여 주시면 비단조 원인을 정리하겠습니다.

In [ ]:
# # 비단조 칸 진단 해석

# CHANGELOG 커밋(`d4313aa`)도 정상적으로 올라갔습니다. 이 출력으로 비단조 칸의 성격은 어느 정도 설명되지만, 아직 확인하지 못한 부분도 있습니다.

# ## 1. 읽히는 것

# **(1) 검정 통계 R이 N2 임계값 바로 아래에서 움직입니다.**

# `fixed14`의 N2 95% 임계값은 0.7965입니다. 진단 표의 `R_mean`은 다음과 같습니다.

# | 칸 | R_mean | 검정력 |
# |---|---|---|
# | 2008, δ=0.5 | 0.789 | 0.565 |
# | 2008, δ=1 | 0.796 | 0.445 |
# | 2012, δ=0.5 | 0.784 | 0.005 |
# | 2012, δ=1 | 0.826 | 1.000 |

# R 평균이 임계값 0.01 안에 있는 칸에서는 검정력이 0에서 1까지 거의 계단처럼 움직입니다. 2012년 δ=0.5(R≈0.784, 검정력 0.005)와 δ=1(R≈0.826, 검정력 1.0)의 차이는 R로 0.04에 불과합니다. 2008년 δ=0.5(0.789)에서 검정력이 0.565나 되는 것은 평균 0.789 아래에 분포가 둘로 갈라져 있다는 뜻입니다. 앞서 본 `fixed14` N2의 k 분포(k=3 301회, k=4 199회)처럼 k가 3과 4 사이를 오가는 구조와 맞지만, 이번 실행에서 k를 직접 저장하지 않아 이 설명은 가설입니다.

# **(2) 2008년의 미세한 하락(0.565 → 0.445)은 해석하지 마세요.** B=200에서 표준오차는 약 0.035라 두 값의 차이(0.12)는 약 2.4배 수준입니다. 한 칸만으로는 "점프가 커졌는데 검정력이 떨어졌다"고 읽기 어렵습니다. 이 한 칸이 `fixed14`의 최솟값(0.44)을 만들어 판독을 "곡선으로만 보고"로 바꿨다는 점은 주의할 사실이지만, 규칙은 그대로 적용합니다.

# **(3) 변화점 위치는 모든 칸에서 회수됩니다(`hit_near_tstar` = 1.0). 이것이 좋은 소식입니다.** 다만 이 수치는 기준선이 필요합니다. 점프가 없어도 t*±1에 변화점이 이미 잡히는 비율이 높을 수 있기 때문입니다. 마침 run 1의 δ=3 행이 보정 실패로 사실상 점프 없는 N2 자료였으므로, 그 값이 기준선이 됩니다.

# | 칸 | 기준선(run 1, 점프 없음) | run 2(점프 주입) |
# |---|---|---|
# | fixed14, 2008 | 0.47 | 1.0 |
# | fixed14, 2012 | 0.59 | 1.0 |
# | within, 2008 | 0.945 | 1.0 |
# | within, 2012 | 0.00 | 1.0 |

# `fixed14`와 `within`의 2012에서는 기준선에서 크게 오릅니다. 즉 점프를 넣으면 탐지기는 그 위치에 변화점을 찍는데, R이 N2 임계값을 넘지 못해서 검정력이 낮게 나온 것으로 보입니다. 이는 위치 회수(변화점이 어디인가)와 R 검정(N2보다 변화가 더 큰가)의 민감도가 서로 다르다는 뜻입니다. `within` 2008은 기준선이 이미 0.945라 구분력이 없습니다.

# 기준선으로 쓴 값이 run 1의 δ=3 행이라는 점은 본문에 밝혀야 합니다. 같은 `Q=P2`에서 뽑았으므로 사실상 점프 없는 N2 자료지만, 정식으로 설계된 기준선이 아닙니다.

# **(4) 실현 점프 비율을 함께 읽으세요.** δ=0.5에서도 `fixed14`는 0.55, `within`은 0.65~0.71이 실현됐습니다. `within`이 `fixed14`보다 같은 δ에서 더 큰 점프를 받습니다.

# ## 2. 보고할 때의 문장 틀

# 사전 판독 규칙은 그대로 적용해 "all·within 강화, fixed14 곡선만 보고"로 쓰되, 다음을 함께 적으세요.
# - δ=1의 주입 점프는 관측 초과분의 약 3배라는 점
# - t*=2020은 δ=0.5에서 세 표본 모두 검정력 1.00이라는 점
# - `fixed14`와 `within`에서 R의 반응이 임계값 근처에서 급격해 검정력이 칸마다 크게 달라진다는 점
# - 변화점 위치는 점프 주입 시 기준선보다 높은 비율로 회수된다는 점(기준선 한계 명시)

# "연속 변화의 증거"라는 일반화 문구는 여전히 피하세요. 근거 있는 서술은 "큰 점프의 부재와 부합하며, 관측 초과분 크기의 점프에 대한 판정은 표본과 t*에 따라 불안정하다"입니다.

# ## 3. 다음 단계: 초과분 크기의 점프 (탐색적)

# 관측 초과분과 같은 크기의 점프(δ 환산 약 0.32~0.44)에서의 검정력이 아직 없습니다. 이 한 줄이 "관측된 정도의 변화를 이 검정이 구분할 수 있는가"에 가장 직접 답합니다. `explore` 모드의 δ를 {0.25, 0.35, 0.45}로 늘려 돌리고, 동결 밖의 `[Post-hoc][Exploratory]`로만 보고합니다.

# ```
# cd ~/Research/Minnesota_Transport/paper1
# python3 - <<'EOF'
# p = "pa_a1_v2_run.py"; s = open(p, encoding="utf-8").read()
# assert s.count("power_grid(c, [0.25], Bp, rows)") == 1
# s = s.replace("power_grid(c, [0.25], Bp, rows)", "power_grid(c, [0.25, 0.35, 0.45], Bp, rows)")
# s = s.replace("power025", "power_small")
# s = s.replace("delta=0.25, 연속 꺾은선 BIC", "delta=0.25/0.35/0.45, 연속 꺾은선 BIC")
# s = s.replace('MODE = "a1"', 'MODE = "explore"', 1)
# open(p, "w", encoding="utf-8").write(s); print("패치 완료")
# EOF
# grep -n '^MODE\|^QUICK\|power_small\|0.35, 0.45' pa_a1_v2_run.py
# ```

# `MODE = "explore"`, `QUICK = False`가 보이고 `power_small`과 `[0.25, 0.35, 0.45]`가 나오면 커밋·푸시 후 실행합니다.

# ```
# cd ~/Research/Minnesota_Transport
# git add paper1/pa_a1_v2_run.py && git commit -m "PA-A1 v2 explore: small-jump grid (exploratory, outside frozen spec)" && git tag -a PA-A1-v2-explore-code-2026-10-19 -m "explore code, pre-run" && git push --follow-tags
# cd paper1
# nohup python3 -u pa_a1_v2_run.py > 09_PA/PA_A1_v2_explore_log.txt 2>&1 &
# sleep 5; tail -f 09_PA/PA_A1_v2_explore_log.txt
# ```

# 주의할 점은 다음과 같습니다.
# - `explore`는 판독 규칙에 쓰지 않습니다. 결과가 좋게 나와도 사전 판독(강화 2, 곡선만 1)을 바꾸지 마세요.
# - 이 코드는 동결 해시와 무관한 `explore` 부분만 바뀝니다. 그래도 사양 점검(`make_ctx`)은 그대로 통과해야 합니다.
# - 연속 꺾은선 BIC도 이번에 함께 계산됩니다.
# - 끝나면 `PA_A1_v2_run2_EXPLORATORY_power_small.csv`와 `..._bic_continuous.csv`가 생깁니다(앞선 치환으로 `run2`가 파일명에 들어갑니다).

# 로그를 붙여 주시면 δ=0.25~0.45 검정력과 연속 꺾은선 BIC를 읽겠습니다.

In [ ]:
# yjlee@u24:~/Research/Minnesota_Transport$ cd ~/Research/Minnesota_Transport/paper1
# python3 - <<'EOF'
# p = "pa_a1_v2_run.py"; s = open(p, encoding="utf-8").read()
# assert s.count("power_grid(c, [0.25], Bp, rows)") == 1
# s = s.replace("power_grid(c, [0.25], Bp, rows)", "power_grid(c, [0.25, 0.35, 0.45], Bp, rows)")
# s = s.replace("power025", "power_small")
# s = s.replace("delta=0.25, 연속 꺾은선 BIC", "delta=0.25/0.35/0.45, 연속 꺾은선 BIC")
# s = s.replace('MODE = "a1"', 'MODE = "explore"', 1)
# open(p, "w", encoding="utf-8").write(s); print("패치 완료")
# EOF
# grep -n '^MODE\|^QUICK\|power_small\|0.35, 0.45' pa_a1_v2_run.py
# 패치 완료
# 15:MODE = "explore"      # "freeze2" | "tag2" | "a1" | "explore"
# 16:QUICK = False          # True: 검정력 B=40, 오류 점검 전용(곡선 해석 금지)
# 190:    power_grid(c, [0.25, 0.35, 0.45], Bp, rows); PW = pd.DataFrame(rows)
# 196:    PW.to_csv(PA/f"PA_A1_v2_run2_EXPLORATORY_power_small{sfx}.csv", index=False, encoding="utf-8-sig"); BC.to_csv(PA/f"PA_A1_v2_run2_EXPLORATORY_bic_continuous{sfx}.csv", index=False, encoding="utf-8-sig")
# yjlee@u24:~/Research/Minnesota_Transport/paper1$ cd ~/Research/Minnesota_Transport
# git add paper1/pa_a1_v2_run.py && git commit -m "PA-A1 v2 explore: small-jump grid (exploratory, outside frozen spec)" && git tag -a PA-A1-v2-explore-code-2026-10-19 -m "explore code, pre-run" && git push --follow-tags
# cd paper1
# nohup python3 -u pa_a1_v2_run.py > 09_PA/PA_A1_v2_explore_log.txt 2>&1 &
# sleep 5; tail -f 09_PA/PA_A1_v2_explore_log.txt
# [main ce23737] PA-A1 v2 explore: small-jump grid (exploratory, outside frozen spec)
#  1 file changed, 4 insertions(+), 4 deletions(-)
# Enumerating objects: 8, done.
# Counting objects: 100% (8/8), done.
# Delta compression using up to 24 threads
# Compressing objects: 100% (5/5), done.
# Writing objects: 100% (5/5), 678 bytes | 678.00 KiB/s, done.
# Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
# remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
# To https://github.com/LEEYJ1021/transport-knowledge-paper1.git
#    d4313aa..ce23737  main -> main
#  * [new tag]         PA-A1-v2-explore-code-2026-10-19 -> PA-A1-v2-explore-code-2026-10-19
# [1] 2321783
# nohup: ignoring input
# [관측 재현 OK: S5와 R 일치]
#   sample  obs_R  obs_k         obs_breaks  N2_B  N2_q95   p_N2
#     all 0.7595      3       [2012, 2020]   500  0.7723 0.9441
# fixed14 0.7755      4 [2008, 2015, 2020]   500  0.7965 0.3992
#  within 0.7033      4 [2009, 2015, 2020]   500  0.7181 0.6966

# 관측 2020 점프(Hellinger, 3년 창): {'all': 0.1901, 'fixed14': 0.1788, 'within': 0.1495}
# [Post-hoc][Exploratory] 작은 점프(0.25x) 검정력
#   t*=2008 완료
#   t*=2012 완료
#   t*=2015 완료
#   t*=2020 완료
# delta_x         0.25  0.35  0.45
# sample  t_star
# all     2008    0.00  0.21  0.69
#         2012    1.00  1.00  1.00
#         2015    0.99  1.00  1.00
#         2020    1.00  1.00  1.00
# fixed14 2008    0.66  0.80  0.72
#         2012    0.00  0.00  0.00
#         2015    1.00  1.00  1.00
#         2020    0.41  0.68  1.00
# within  2008    0.18  0.40  0.40
#         2012    0.04  0.00  0.04
#         2015    1.00  1.00  1.00
#         2020    1.00  1.00  1.00

# [Post-hoc][Exploratory] 연속 꺾은선 BIC(꺾임 수 0~4)
#   sample                       model  k    bic              breakpoints
#     all piecewise_linear_continuous  1 -656.8                       []
#     all piecewise_linear_continuous  2 -746.6                   [2014]
#     all piecewise_linear_continuous  3 -760.1             [2007, 2016]
#     all piecewise_linear_continuous  4 -755.4       [2006, 2011, 2017]
#     all piecewise_linear_continuous  5 -744.4 [2003, 2006, 2011, 2017]
# fixed14 piecewise_linear_continuous  1 -675.5                       []
# fixed14 piecewise_linear_continuous  2 -749.0                   [2014]
# fixed14 piecewise_linear_continuous  3 -763.1             [2007, 2016]
# fixed14 piecewise_linear_continuous  4 -760.0       [2006, 2011, 2017]
# fixed14 piecewise_linear_continuous  5 -748.7 [2003, 2006, 2011, 2017]
#  within piecewise_linear_continuous  1 -646.1                       []
#  within piecewise_linear_continuous  2 -671.3                   [2012]
#  within piecewise_linear_continuous  3 -676.5             [2003, 2012]
#  within piecewise_linear_continuous  4 -664.2       [2003, 2007, 2016]
#  within piecewise_linear_continuous  5 -647.4 [2003, 2006, 2013, 2016]

In [ ]:
# # 탐색적 확장 결과 해석

# 이 결과는 동결 밖의 `[Post-hoc][Exploratory]`이므로 사전 판독(강화 2, 곡선만 보고 1)을 바꾸지 않습니다. 그 점을 전제로, 관측된 2020 부근 변화와 같은 크기의 점프에서 검정이 어떻게 반응하는지는 다음과 같이 읽힙니다.

# ## 1. 관측 초과분 크기(δ 환산 약 0.32 / 0.37 / 0.44)에서의 검정력

# 초과분은 관측 점프에서 추세 기저 점프를 뺀 값입니다. 점프는 거리라서 뺄셈은 근사치입니다.

# | 표본 | t*=2008 | t*=2012 | t*=2015 | t*=2020 |
# |---|---|---|---|---|
# | all (δ≈0.32) | 0.00~0.21 (약함) | 1.00 | 1.00 | 1.00 |
# | fixed14 (δ≈0.37) | 0.66~0.80 (불안정) | **0.00** | 1.00 | 0.41~0.68 (δ=0.45에서 1.00) |
# | within (δ≈0.44) | 0.18~0.40 (약함) | **0.00~0.04** | 1.00 | 1.00 |

# **읽히는 것**
# - 가장 관련 있는 위치인 t*=2020에서는 `all`과 `within`이 δ=0.25부터 이미 1.00이고, `fixed14`는 초과분 크기 근처(δ≈0.35~0.45)에서 0.68~1.00입니다. 2020에 초과분 정도의 점프가 있었다면 세 표본에서 대체로 잡혔을 것이라는 뜻입니다. 다만 `fixed14`는 확실하지 않습니다.
# - `all`의 N2 기준 R 분포가 매우 좁아서(표준편차 약 0.004) 작은 구조 변화에도 R이 기준을 넘는 것으로 보입니다. 이전 평가에서 걱정한 "N2가 관측 구조를 흡수해 둔감할 가능성"의 반대 방향이고, 과민할 가능성은 이 실험으로 판단할 수 없습니다. 검정 크기(δ=0 기각률)가 정상이므로 위양성 쪽 문제는 크지 않아 보입니다.

# **사각지대**
# - `fixed14`와 `within`의 t*=2012는 δ=0.45까지 검정력이 0에 가깝고, run 2에서 δ=0.5도 0.005~0.10이었다가 δ=1에서 1.00으로 뜁니다. 어딘가 벽이 있다는 뜻입니다. 선택된 k가 바뀌는 지점일 가능성이 있지만 이번 실행은 k를 저장하지 않아 확인하지 못했습니다(가설).
# - t*=2008은 세 표본에서 약하거나(all, within) 비단조적입니다(fixed14: 0.66, 0.80, 0.72).
# - 관측 변화점 후보 중 `all`의 2012, `within`의 2009가 사각지대에 걸립니다. 이 위치들에 대해서는 "탐지력이 낮아 판정할 수 없다"고 써야 합니다.

# ## 2. 연속 꺾은선 BIC

# `k`는 구간 수(꺾임 수+1)입니다.

# | 표본 | 최적 모형 | 꺾임 위치 | BIC | 꺾임 없음(직선)과의 차이 |
# |---|---|---|---|---|
# | all | k=3 | 2007, 2016 | −760.1 | 약 103 개선 |
# | fixed14 | k=3 | 2007, 2016 | −763.1 | 약 88 개선 |
# | within | k=3 | 2003, 2012 | −676.5 | 약 30 개선 |

# - 꺾임이 2개인 연속 모형이 직선보다 BIC로 크게 좋아서, 변화가 한 직선이 아니라 속도가 바뀌는 연속 변화로 기술하는 편이 자료에 잘 맞습니다.
# - 같은 PC 자료에서 불연속을 허용한 구간별 선형 모형의 최저 BIC(all −745, fixed14 −749, within −659)보다 연속 모형이 `all`·`fixed14`에서 약 15, `within`에서 약 18 낮습니다. 매개변수는 더 적으니, 이 자료에서는 점프 없는 연속 꺾임이 BIC로 더 낫다는 뜻입니다. 매개변수 수를 세는 방식이 두 계산에서 같지 않아서 숫자를 그대로 비교하기는 어렵고, 방향만 참고하세요.
# - 선택된 꺾임에 2020이 없습니다. 최대 5구간(꺾임 4개)까지 봐도 2020은 선택되지 않았습니다.
# - 연도 자료의 자기상관 때문에 BIC가 과분할 쪽으로 기울 수 있고, `within`은 꺾임 위치가 다른 두 표본과 달라 표본 간 일관성이 약합니다. 보조 지표로만 쓰세요.

# ## 3. PA-A1 전체 정리와 B1 문구

# 동결 규칙의 결과는 그대로입니다. 근거 있는 서술은 다음과 같습니다.

# - **쓸 수 있는 것:** "전체와 저널 내 표본에서 2015년 이후(특히 2020) 관측 초과분 크기의 점프가 있었다면 N2 검정이 탐지했을 가능성이 높다. N2 비통과는 이 위치들에서 그런 점프가 없었다는 점과 부합한다. 연속 꺾은선이 점프 모형보다 BIC로 낫다(탐색적)."
# - **쓰면 안 되는 것:** "체제 전환이 없었음이 확인됐다", "연속 변화의 증거다"(일반 주장). 2008~2012 부근은 `fixed14`와 `within`에서 검정력이 낮거나 불안정해서 판정 불가입니다.
# - 제목에서 "체제 전환"을 빼는 방향은 유지해도 됩니다. 문구는 "점프보다 연속적 재배치로 서술하는 편이 자료에 부합하며, 2020 부근 후보는 N2 비통과로 유지되지 않는다" 정도가 안전합니다. 검정력은 관측 2020 점프 방향의 점프에 한정된다는 점을 부록에 적으세요.
# - `fixed14` 곡선이 거칠어서, 본문에는 표 한 장으로 투명하게 보이는 편이 낫습니다.

# ## 4. 지금 할 일

# **① 파일 확인과 커밋·푸시.** 실행이 끝났는지부터 보세요.
# ```
# cd ~/Research/Minnesota_Transport/paper1
# ls -l 09_PA/PA_A1_v2_run2_EXPLORATORY_*.csv
# tail -2 09_PA/PA_CHANGELOG.md
# ```
# CSV 두 개가 있으면 끝난 것입니다. 이어서 CHANGELOG에 수기 줄을 추가하고 커밋하세요.
# ```
# cd ~/Research/Minnesota_Transport
# cat >> paper1/09_PA/PA_CHANGELOG.md <<'EOF'
# - 2026-10-19 | PA-A1 v2 탐색적 확장 해석 메모(동결 밖, 판독 규칙에 미사용): δ=0.25/0.35/0.45 검정력은 t*=2020에서 all·within 1.00, fixed14 0.41~1.00. t*=2012의 fixed14·within은 δ=0.45까지 ≈0(사각지대), t*=2008은 약하거나 비단조. 연속 꺾은선 BIC는 세 표본 모두 꺾임 2개가 최적(all·fixed14 2007/2016, within 2003/2012), 2020 꺾임은 선택되지 않음. k와 변화점 위치는 미저장이라 사각지대 원인은 가설 | 본 결과 변경: 없음
# EOF
# git add paper1/09_PA/PA_A1_v2_run2_EXPLORATORY_*.csv paper1/09_PA/PA_A1_v2_explore_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-A1 v2 exploratory results (small jumps, continuous BIC)" && git tag -a PA-A1-v2-explore-results-2026-10-19 -m "explore outputs, seen" && git push --follow-tags
# ```
# `pa_a1_v2_run.py`는 `MODE="explore"` 상태로 커밋됐습니다. 이후에 이 파일로 `a1`을 다시 돌릴 일은 없으니 그대로 두어도 됩니다.

# **② PA-A1은 여기서 닫는 것을 권합니다.** 사각지대 원인(k 전환)을 더 파는 것은 v3 계획이 되고, 지금 목적에는 필요하지 않습니다. 부록에는 run 1(보정 실패와 원인), run 2, 탐색적 확장을 모두 싣습니다.

# **③ 다음 PA 선택.** 우선순위는 B2(2020 아티팩트 분리)와 A2(신규 진입 방어)입니다. B2는 PA-A1 결과와 `s5_lib`로 바로 진행할 수 있습니다. A2에는 S6 코드 정보가 필요하니, 아래 읽기 전용 스크립트의 출력을 붙여 주세요.

# ```
# cd ~/Research/Minnesota_Transport/paper1
# python3 - <<'EOF'
# import ast, re
# from pathlib import Path
# P1 = Path.home()/"Research/Minnesota_Transport/paper1"
# KEY = re.compile(r"entr|persist|new|adapt|realloc|exit|decomp|boot|cluster|homonym|risk|name|oa_id|first_year|cohort", re.I)
# for fn in ["s6_lib.py", "s6_run.py"]:
#     t = (P1/fn).read_text(encoding="utf-8"); L = t.splitlines(); tree = ast.parse(t)
#     print(f"\n==== {fn} ({len(L)}줄) ====")
#     for n in tree.body:
#         if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id.isupper(): print("  ", n.targets[0].id, "=", ast.unparse(n.value)[:100])
#     for n in tree.body:
#         if isinstance(n, ast.FunctionDef): print(f"  L{n.lineno:<4} def {n.name}({', '.join(a.arg for a in n.args.args)})")
#     print("[핵심 후보 함수 소스, 각 40줄까지]")
#     for n in tree.body:
#         if isinstance(n, ast.FunctionDef) and KEY.search(n.name):
#             seg = L[n.lineno-1:n.end_lineno]; print(f"\n--- {n.name} (L{n.lineno}-{n.end_lineno}) ---"); print("\n".join(seg[:40]))
# t = (P1/"s6_run.py").read_text(encoding="utf-8").splitlines()
# print("\n==== s6_run.py 키워드 줄 ====")
# for i, l in enumerate(t, 1):
#     if re.search(r"entrant|persist|first_year|oa_id|homonym|risk|B_BOOT|bootstrap", l): print(f"L{i:<4} {l.strip()[:160]}")
# EOF
# ```
# 출력이 길면 함수 목록과 `entrant`·`persist` 관련 줄만 붙여 주셔도 됩니다.

# 푸시 결과와 S6 정보를 주시면 이어서 B2와 A2 중 어느 쪽을 먼저 할지 정하겠습니다.

In [ ]:
# yjlee@u24:~/Research/Minnesota_Transport/paper1$ cd ~/Research/Minnesota_Transport/paper1
# ls -l 09_PA/PA_A1_v2_run2_EXPLORATORY_*.csv
# tail -2 09_PA/PA_CHANGELOG.md
# -rw-rw-r-- 1 yjlee yjlee 1379 Oct 19 18:00 09_PA/PA_A1_v2_run2_EXPLORATORY_bic_continuous.csv
# -rw-rw-r-- 1 yjlee yjlee 3459 Oct 19 18:00 09_PA/PA_A1_v2_run2_EXPLORATORY_power_small.csv
# - 2026-10-19 | PA-A1 v2 run 2 결과: δ=0 기각률 0.035/0.075/0.065. δ=1 판독 all 강화(min 1.00), fixed14 곡선만 보고(min 0.44, t*=2008), within 강화(min 0.90). 주의: δ=1 추가 점프는 관측 초과분(점프−추세 기저점프)의 약 3배. t*=2020은 δ=0.5에서 세 표본 모두 1.00. fixed14·within에서 비단조 칸 존재(미해석). 판독 규칙·격자 변경 없음 | 본 결과 변경: 없음- 2026-10-19 | PA-A1 v2 run 2 결과 해석 메모: δ=0 기각률 0.035/0.075/0.065. δ=1 판독 all 강화(min 1.00), fixed14 곡선만 보고(min 0.44, t*=2008), within 강화(min 0.90). 주의: δ=1 추가 점프는 관측 초과분(점프−추세 기저점프)의 약 3배. t*=2020은 δ=0.5에서 세 표본 모두 1.00. fixed14·within에 비단조 칸 존재(미해석). 판독 규칙·격자 변경 없음 | 본 결과 변경: 없음
# - 2026-10-19T18:00:13 | PA-A1(v2) 탐색적 확장 | 정의: 동결 밖(delta=0.25/0.35/0.45, 연속 꺾은선 BIC), 판독 규칙에 미사용 | 결과: PA_A1_v2_run2_EXPLORATORY_*.csv | 본 결과 변경 여부: 없음
# yjlee@u24:~/Research/Minnesota_Transport/paper1$ cd ~/Research/Minnesota_Transport
# cat >> paper1/09_PA/PA_CHANGELOG.md <<'EOF'
# - 2026-10-19 | PA-A1 v2 탐색적 확장 해석 메모(동결 밖, 판독 규칙에 미사용): δ=0.25/0.35/0.45 검정력은 t*=2020에서 all·within 1.00, fixed14 0.41~1.00. t*=2012의 fixed14·within은 δ=0.45까지 ≈0(사각지대), t*=2008은 약하거나 비단조. 연속 꺾은선 BIC는 세 표본 모두 꺾임 2개가 최적(all·fixed14 2007/2016, within 2003/2012), 2020 꺾임은 선택되지 않음. k와 변화점 위치는 미저장이라 사각지대 원인은 가설 | 본 결과 변경: 없음
# EOF
# git add paper1/09_PA/PA_A1_v2_run2_EXPLORATORY_*.csv paper1/09_PA/PA_A1_v2_explore_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-A1 v2 exploratory results (small jumps, continuous BIC)" && git tag -a PA-A1-v2-explore-results-2026-10-19 -m "explore outputs, seen" && git push --follow-tags
# [main 3ad6eaa] PA-A1 v2 exploratory results (small jumps, continuous BIC)
#  4 files changed, 100 insertions(+)
#  create mode 100644 paper1/09_PA/PA_A1_v2_explore_log.txt
#  create mode 100644 paper1/09_PA/PA_A1_v2_run2_EXPLORATORY_bic_continuous.csv
#  create mode 100644 paper1/09_PA/PA_A1_v2_run2_EXPLORATORY_power_small.csv
# Enumerating objects: 13, done.
# Counting objects: 100% (13/13), done.
# Delta compression using up to 24 threads
# Compressing objects: 100% (9/9), done.
# Writing objects: 100% (9/9), 3.57 KiB | 3.57 MiB/s, done.
# Total 9 (delta 3), reused 0 (delta 0), pack-reused 0
# remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
# To https://github.com/LEEYJ1021/transport-knowledge-paper1.git
#    ce23737..3ad6eaa  main -> main
#  * [new tag]         PA-A1-v2-explore-results-2026-10-19 -> PA-A1-v2-explore-results-2026-10-19
# yjlee@u24:~/Research/Minnesota_Transport$ cd ~/Research/Minnesota_Transport/paper1
# python3 - <<'EOF'
# import ast, re
# from pathlib import Path
# P1 = Path.home()/"Research/Minnesota_Transport/paper1"
# KEY = re.compile(r"entr|persist|new|adapt|realloc|exit|decomp|boot|cluster|homonym|risk|name|oa_id|first_year|cohort", re.I)
# for fn in ["s6_lib.py", "s6_run.py"]:
#     t = (P1/fn).read_text(encoding="utf-8"); L = t.splitlines(); tree = ast.parse(t)
#     print(f"\n==== {fn} ({len(L)}줄) ====")
#     for n in tree.body:
#         if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id.isupper(): print("  ", n.targets[0].id, "=", ast.unparse(n.value)[:100])
#     for n in tree.body:
#         if isinstance(n, ast.FunctionDef): print(f"  L{n.lineno:<4} def {n.name}({', '.join(a.arg for a in n.args.args)})")
#     print("[핵심 후보 함수 소스, 각 40줄까지]")
#     for n in tree.body:
#         if isinstance(n, ast.FunctionDef) and KEY.search(n.name):
#             seg = L[n.lineno-1:n.end_lineno]; print(f"\n--- {n.name} (L{n.lineno}-{n.end_lineno}) ---"); print("\n".join(seg[:40]))
# t = (P1/"s6_run.py").read_text(encoding="utf-8").splitlines()
# print("\n==== s6_run.py 키워드 줄 ====")
# for i, l in enumerate(t, 1):
#     if re.search(r"entrant|persist|first_year|oa_id|homonym|risk|B_BOOT|bootstrap", l): print(f"L{i:<4} {l.strip()[:160]}")
# EOF

# ==== s6_lib.py (146줄) ====
#    ROOT = Path.home() / 'Research/Minnesota_Transport'
#    S6 = ROOT / 'paper1/06_S6'
#   L12   def pick(df, cands, what)
#   L16   def _find(o, key)
#   L22   def load_spec()
#   L28   def ipw_weights(W, spec)
#   L37   def load_all(spec)
#   L140  def make_pair(L, p0, p1, scheme, venue, topic, hom)
#   L146  def pairs_of(spec)
# [핵심 후보 함수 소스, 각 40줄까지]

# ==== s6_run.py (119줄) ====
#    QUICK = False
#    L = load_all(spec)
#    K = L['K']
#    PAIRS = pairs_of(spec)
#    MAIN = spec['main']
#    TERMS = ['adapt', 'realloc', 'entry_new', 'entry_ret', 'exit_final', 'exit_gap']
#    FR = pd.DataFrame(fr)
#    G = pd.DataFrame(grid)
#    RS = pd.DataFrame(res)
#   L17   def summarize(dec)
# [핵심 후보 함수 소스, 각 40줄까지]

# ==== s6_run.py 키워드 줄 ====
# L31   ex, drop = P.excess(); exc_rows.append(dict(pair=f"{p0}->{p1}", scheme=sc, excess_half=half(ex), dropped_entrant_mass=drop,
# L66   pd.DataFrame(boot_rows).to_csv(S6 / f"renewal_bootstrap{sfx}.csv", index=False, encoding="utf-8-sig")
# L86   fr.append(dict(pair=f"{p0}->{p1}", defn=d, subset=sub, n_persist=int(mk.sum()), n_zero_t0=int(z.sum()),
# L95   axes = list(itertools.product(PAIRS, [s for s in spec["schemes"] if "m_" + s in L["R"].columns], spec["venue_sets"], list(L["risk"]), spec["topic_sets"]))
# L108  G.groupby(["pair", "hom"])[["excess_half", "within_share", "entry_share"]].agg(["min", "max"]).to_csv(S6 / f"homonym_range{sfx}.csv", encoding="utf-8-sig")
# yjlee@u24:~/Research/Minnesota_Transport/paper1$

In [25]:
# -*- coding: utf-8 -*-
# paper1/MASTER_MANIFEST.csv + MASTER_TAGS.csv 생성. 파일 수정 없음, 결과·판정과 무관한 기록 작업.
import hashlib, subprocess
from pathlib import Path
import pandas as pd
ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"
git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
sha_f = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
SKIP = {"oa_raw", ".ipynb_checkpoints", "_backup_before_income_tv", "_recovered_s4", "__pycache__"}
tracked = set(git("ls-files", "paper1").stdout.splitlines())
rows = []
for d in sorted(P1.glob("[0-9][0-9]_*")):
    for f in sorted(d.rglob("*")):
        if not f.is_file() or any(s in f.parts for s in SKIP): continue
        rel = f.relative_to(ROOT).as_posix()
        rows.append(dict(stage=d.name, file=rel, bytes=f.stat().st_size, mtime=pd.Timestamp(f.stat().st_mtime, unit="s").isoformat(timespec="seconds"),
                         sha256=sha_f(f), git_tracked=rel in tracked))
for f in sorted(P1.glob("*.py")):
    rel = f.relative_to(ROOT).as_posix()
    rows.append(dict(stage="code", file=rel, bytes=f.stat().st_size, mtime=pd.Timestamp(f.stat().st_mtime, unit="s").isoformat(timespec="seconds"), sha256=sha_f(f), git_tracked=rel in tracked))
M = pd.DataFrame(rows); M.to_csv(P1/"MASTER_MANIFEST.csv", index=False, encoding="utf-8-sig")
tg = git("for-each-ref", "--format=%(refname:short)|%(objecttype)|%(objectname:short)|%(*objectname:short)|%(creatordate:iso)", "refs/tags").stdout.strip().splitlines()
T = pd.DataFrame([x.split("|") for x in tg], columns=["tag", "type", "obj", "commit_if_annotated", "created"]); T.to_csv(P1/"MASTER_TAGS.csv", index=False, encoding="utf-8-sig")
remote = set(l.split("refs/tags/")[1].replace("^{}", "") for l in git("ls-remote", "--tags", "origin").stdout.splitlines() if "refs/tags/" in l)
T["on_origin"] = T.tag.isin(remote); T.to_csv(P1/"MASTER_TAGS.csv", index=False, encoding="utf-8-sig")
print(M.groupby("stage").agg(files=("file", "size"), MB=("bytes", lambda x: round(x.sum()/1e6, 1)), untracked=("git_tracked", lambda x: int((~x).sum()))).to_string())
print(f"\n태그 {len(T)}개 | 원격 미푸시 {int((~T.on_origin).sum())}개:", list(T[~T.on_origin].tag)[:12])
print("※ git_tracked=False 파일이 남아 있으면 PA-0 정리 커밋을 먼저 끝내세요. 이 표는 기록용이며 결과를 바꾸지 않습니다.")

       files    MB  untracked
stage                        
00_S0     32   0.2          0
01_S1     50  31.1         15
02_S2     36  14.8         36
03_S3     26   0.9          6
04_S4     24   1.6         22
05_S5     19   0.3          0
06_S6     29   0.1          0
07_S7     27   1.0          1
08_S8     32   0.6          0
09_PA     14   0.0          0
code      14   0.2          3

태그 27개 | 원격 미푸시 0개: []
※ git_tracked=False 파일이 남아 있으면 PA-0 정리 커밋을 먼저 끝내세요. 이 표는 기록용이며 결과를 바꾸지 않습니다.


In [26]:
# -*- coding: utf-8 -*-
# PA-B2: 2020 변화의 표본 크기·저널 구성 아티팩트 분리. 미실행 코드.
# MODE: "freeze" -> "tag" -> (git push --follow-tags) -> "run"(QUICK=True: 오류 점검 전용) -> "run"(QUICK=False)
# S0~S8 판정 불변. 모든 산출물 [Post-hoc]. 판독 규칙은 PLAN_B2에 동결.
import sys, json, hashlib, subprocess, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
MODE = "freeze"
QUICK = True

ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; PA = P1/"09_PA"; S5 = P1/"05_S5"; S3 = P1/"03_S3"
now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
sha_b = lambda b: hashlib.sha256(b).hexdigest(); sha_f = lambda p: sha_b(Path(p).read_bytes())
git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
class Halt(Exception): pass
spec_hash = lambda s: sha_b(json.dumps({k: v for k, v in s.items() if k not in ("sha256", "created", "refs")}, sort_keys=True, ensure_ascii=False).encode())
V2P, BP = PA/"PA_0_PLAN_v2.json", PA/"PA_B2_PLAN.json"

PLAN_B2 = dict(
    version=1, seed=2026, parent="PA_0_PLAN_v2.json",
    status="사후 명세. PA-A1 run 1·2·탐색 결과를 본 뒤 작성. PA-B2 결과는 미열람. S0~S8 판정 불변.",
    caps=[3800, 3000], primary_cap=3800, B=200, focus_year=2020, window_pre=[2017, 2019], window_post=[2020, 2022], samples=["all", "fixed14", "within"],
    downsample="연도별 (저널x토픽) 셀 카운트에서 총 cap편을 비복원 추출(다변량 초기하). 연도 논문 수가 cap 이하이면 그대로. 저널 구성은 기대값상 유지",
    detector="s5_lib.detect(P, n, pen, kmax, minlen) 그대로. pen = c_primary*phi_s*(K-1)*log(T), S5_0_PREREG.json 값. 추출된 카운트로 s5_lib.sample_P 재계산",
    stats="(a) 2020 점프 J = ||가중평균(sqrt P)[2020,2023) - 가중평균(sqrt P)[2017,2020)||/sqrt(2), 가중치 4n (PA-A1과 동일), (b) 2020±1년 변화점 회수율, (c) R, (d) 변화점 연도 분포",
    reading="primary_cap에서 표본별 '유지' = (회수율>=0.80) AND (J 중앙값 >= 0.80*J_obs). 세 표본 모두 유지: '표본 크기 아티팩트 아님' / 일부: '혼재(표본 의존)' / 없음: '아티팩트 후보 확정'. cap=3000은 민감도로만 보고",
    descriptive="관측 자료 기술 1회: decomp_break(2017-19 대 2020-22)의 토픽 상위 10(저널 내/간), 2019->2020 저널별 논문 수 증가 상위 10",
    caveat="하향 추출은 표집 잡음을 늘려 J를 위로 편향시킬 수 있어 J만으로 판정하지 않음. 벌점은 S5 값 고정이라 추출 후 비용/벌점 비율이 달라짐. 검정은 논문 독립 가정",
    forbidden="결과를 본 뒤 cap·임계값·창 변경(변경 시 v2 계획으로 분리하고 사유 기록)")

def freeze():
    if not V2P.exists(): raise Halt("PA_0_PLAN_v2.json이 없습니다")
    h = spec_hash(PLAN_B2); print("PA-B2 계획 해시(예정)", h[:12])
    if BP.exists(): print("이미 동결:", "동일" if json.loads(BP.read_text(encoding="utf-8"))["sha256"] == h else "⛔ 다름"); return
    refs = {f.name: sha_f(f) for f in [V2P, S5/"S5_0_PREREG.json", P1/"s5_lib.py", PA/"PA_A1_v2_run2_power_curve.csv"] if f.exists()}
    BP.write_text(json.dumps(dict(PLAN_B2, created=now(), sha256=h, refs=refs), ensure_ascii=False, indent=1), encoding="utf-8")
    with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f:
        f.write(f"\n- {now()} | PA-B2 계획 동결(해시 {h[:12]}) | 사유: PA-A1 결과 열람 뒤의 사후 명세, B2 결과 미열람. QUICK 실행은 오류 점검 전용 | 본 결과 변경 여부: 없음\n")
    print("저장: PA_B2_PLAN.json")

def tag():
    if not BP.exists(): raise Halt("먼저 freeze")
    t = f"PA-B2-plan-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
    if git("tag", "-l", t).stdout.strip(): print("이미 있음:", t); return
    r = git("add", "--", "paper1/09_PA/PA_B2_PLAN.json", "paper1/09_PA/PA_CHANGELOG.md"); print("add", r.returncode, r.stderr.strip())
    r = git("commit", "-m", "PA-B2 plan freeze (before execution)"); print("commit", r.returncode, (r.stdout + r.stderr)[:200])
    r = git("tag", "-a", t, "-m", "PA-B2 plan"); print("tag", r.returncode, r.stderr.strip()); print("※ 이어서 git push --follow-tags (실행 전 외부 타임스탬프)")

def run():
    if not BP.exists(): raise Halt("먼저 freeze")
    sp = json.loads(BP.read_text(encoding="utf-8"))
    if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(PLAN_B2): raise Halt("동결 파일과 현재 계획이 다릅니다(사후 변경 금지)")
    if not git("tag", "-l", "PA-B2-plan-*").stdout.strip(): raise Halt("tag 단계를 먼저 실행하세요")
    sys.path.insert(0, str(P1)); import s5_lib as L
    sfx = "_QUICK" if QUICK else ""; B = 20 if QUICK else PLAN_B2["B"]; rng = np.random.default_rng(int(PLAN_B2["seed"]))
    s5 = json.loads((S5/"S5_0_PREREG.json").read_text(encoding="utf-8")); D = s5["detection"]; PHI = {k: v["phi"] for k, v in D["phi"].items()}
    C0, KMAX, MINLEN = D["c_primary"], D["k_max"], D["min_len"]
    cnt, venues, topics, fixed_v = L.load_counts(); wbar = cnt.sum((0, 2))/cnt.sum(); T, V, K = cnt.shape; YEARS = np.asarray(L.YEARS); SAM = PLAN_B2["samples"]; f = PLAN_B2["focus_year"]
    pen = lambda s: C0*PHI[s]*(K-1)*np.log(T); i20 = int(np.where(YEARS == f)[0][0]); pre, post = PLAN_B2["window_pre"], PLAN_B2["window_post"]
    def J(P, n, it, ):   # PA-A1과 동일 정의(3년 창)
        X, w = np.sqrt(P), 4.0*n; a, b = slice(max(0, it-3), it), slice(it, it+3)
        m = lambda sl: np.average(X[sl], axis=0, weights=w[sl]) if w[sl].sum() > 0 else X[sl].mean(0)
        return float(np.linalg.norm(m(b) - m(a))/np.sqrt(2))
    def detect_s(c_arr, s):
        P, n = L.sample_P(c_arr, fixed_v, wbar, s); return L.detect(P, n, pen(s), KMAX, MINLEN), P, n
    # 관측 재현 점검(S5와 R 일치)
    OBS = {}; ref = pd.read_csv(S5/"regime_nulls.csv")
    for s in SAM:
        d, P, n = detect_s(cnt, s); r0 = float(ref[(ref["null"] == "N2") & (ref["sample"] == s)].obs_R.iloc[0])
        if abs(d["R"] - r0) > 1e-3: raise Halt(f"S5 관측 R 재현 실패({s}): {d['R']:.4f} vs {r0:.4f}")
        OBS[s] = dict(J=J(P, n, i20), R=d["R"], breaks=[int(YEARS[b]) for b in d["br"]])
    print("[관측 재현 OK]", {s: (round(v["J"], 4), v["breaks"]) for s, v in OBS.items()})
    # ---- 관측 기술 통계 1회 ----
    a0, a1, b1 = pre[0]-int(YEARS[0]), post[0]-int(YEARS[0]), post[1]+1-int(YEARS[0])
    wi, bt, tot = L.decomp_break(cnt, a0, a1, b1)
    names = {}
    try:
        Cd = pd.read_csv(S3/"topic_evidence_cards.csv"); col = [c for c in Cd.columns if c.lower() in ("name", "label", "topic_name", "top_terms", "top10", "keywords")]
        if col: names = dict(zip(Cd.topic.astype(int), Cd[col[0]].astype(str).str.slice(0, 60)))
    except Exception as e: print("※ 카드 이름 생략:", str(e)[:80])
    gA = set(pd.read_csv(S3/"L7_topic_grade.csv").pipe(lambda g: g[g[[c for c in g.columns if "grade" in c.lower()][0]].astype(str).str.upper().str.strip() == "A"]).topic.astype(int))
    top = np.argsort(-np.abs(tot))[:10]
    TP = pd.DataFrame([dict(rank=r+1, topic=int(topics[i]), name=names.get(int(topics[i]), ""), delta=tot[i], within_journal=wi[i], between_journal=bt[i], grade_A=int(topics[i]) in gA) for r, i in enumerate(top)])
    print(f"\n[Post-hoc] {pre[0]}-{pre[1]} 대 {post[0]}-{post[1]} 토픽 변화 상위 10\n", TP.round(4).to_string(index=False))
    print(f"저널 간 성분 비중(전 토픽 절대합): {np.abs(bt).sum()/(np.abs(wi).sum()+np.abs(bt).sum()):.3f}")
    nv = cnt.sum(2); ip = int(np.where(YEARS == f-1)[0][0]); dv = nv[i20] - nv[ip]
    VG = pd.DataFrame(dict(venue=venues, fixed14=fixed_v, n_prev=nv[ip], n_focus=nv[i20], delta=dv, share_of_total_delta=dv/dv.sum())).sort_values("delta", ascending=False)
    print(f"\n[Post-hoc] {f-1}->{f} 저널별 논문 수 증가 상위 10 (전체 증가 {int(dv.sum())})\n", VG.head(10).round(3).to_string(index=False))
    TP.to_csv(PA/f"PA_B2_observed_topics{sfx}.csv", index=False, encoding="utf-8-sig"); VG.to_csv(PA/f"PA_B2_venue_growth{sfx}.csv", index=False, encoding="utf-8-sig")
    # ---- 하향 추출 복제 ----
    def cap_counts(c, cap):
        out = c.copy()
        for y in range(T):
            flat = c[y].ravel().astype(np.int64)
            if flat.sum() > cap: out[y] = rng.multivariate_hypergeometric(flat, cap, method="marginals").reshape(V, K)
        return out
    R_ = []
    for cap in PLAN_B2["caps"]:
        for b in range(B):
            cc = cap_counts(cnt, cap)
            for s in SAM:
                d, P, n = detect_s(cc, s); yrs = [int(YEARS[x]) for x in d["br"]]
                R_.append(dict(cap=cap, rep=b, sample=s, J=J(P, n, i20), R=d["R"], k=d["k"], hit=any(abs(y-f) <= 1 for y in yrs), breaks=str(yrs)))
        print(f"  cap={cap} 완료")
    RP = pd.DataFrame(R_); RP.to_csv(PA/f"PA_B2_cap_replicates{sfx}.csv", index=False, encoding="utf-8-sig")
    SM = RP.groupby(["cap", "sample"]).agg(J_med=("J", "median"), J_q05=("J", lambda x: x.quantile(.05)), J_q95=("J", lambda x: x.quantile(.95)), R_med=("R", "median"), hit_rate=("hit", "mean"), k_mode=("k", lambda x: x.mode().iloc[0])).reset_index()
    SM["J_obs"] = SM["sample"].map(lambda s: OBS[s]["J"]); SM["J_ratio"] = SM.J_med/SM.J_obs; SM.to_csv(PA/f"PA_B2_summary{sfx}.csv", index=False, encoding="utf-8-sig")
    print("\n[Post-hoc] PA-B2 하향 추출 요약\n", SM.round(3).to_string(index=False))
    if QUICK:
        print("\n[QUICK] 오류 점검 전용: 위 수치를 해석하거나 판독 규칙을 적용하지 마세요(B=20, v2 계획 사유 방지)."); return
    pr = SM[SM.cap == PLAN_B2["primary_cap"]].set_index("sample"); keep = {s: bool(pr.loc[s, "hit_rate"] >= 0.80 and pr.loc[s, "J_ratio"] >= 0.80) for s in SAM}
    print("\n[동결 판독, cap=%d]" % PLAN_B2["primary_cap"], {s: ("유지" if v else "실패") for s, v in keep.items()})
    print("→", "표본 크기 아티팩트 아님" if all(keep.values()) else ("혼재(표본 의존)" if any(keep.values()) else "아티팩트 후보 확정"))
    print("※ J는 표집 잡음으로 위로 편향될 수 있고, 판독은 PLAN_B2의 규칙을 코드로 옮긴 것입니다. S5 판정(체제 서사 철회)은 바꾸지 않습니다.")
    with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as fh:
        fh.write(f"- {now()} | PA-B2 실행 | 정의: PA_B2_PLAN {sp['sha256'][:12]} | 결과: PA_B2_*.csv | 본 결과 변경 여부: 없음\n")

try: {"freeze": freeze, "tag": tag, "run": run}[MODE]()
except Halt as e: print(f"⛔ 중단: {e}")

PA-B2 계획 해시(예정) 47c93a7c5739
저장: PA_B2_PLAN.json


In [ ]:
# yjlee@u24:~/Research/Minnesota_Transport/paper1$ cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-0 MASTER_MANIFEST·MASTER_TAGS 생성(기록 전용). 생성 시각 기준 해시이며 각 단계 동결 시점의 해시는 단계별 manifest가 증명 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/MASTER_MANIFEST.csv paper1/MASTER_TAGS.csv paper1/09_PA/PA_CHANGELOG.md && git commit -m "PA-0: master manifest and tag list" && git push --follow-tags
# [main c915932] PA-0: master manifest and tag list
#  3 files changed, 333 insertions(+)
#  create mode 100644 paper1/MASTER_MANIFEST.csv
#  create mode 100644 paper1/MASTER_TAGS.csv
# Enumerating objects: 11, done.
# Counting objects: 100% (11/11), done.
# Delta compression using up to 24 threads
# Compressing objects: 100% (7/7), done.
# Writing objects: 100% (7/7), 17.41 KiB | 8.70 MiB/s, done.
# Total 7 (delta 3), reused 0 (delta 0), pack-reused 0
# remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
# To https://github.com/LEEYJ1021/transport-knowledge-paper1.git
#    3ad6eaa..c915932  main -> main
# yjlee@u24:~/Research/Minnesota_Transport$ cd ~/Research/Minnesota_Transport/paper1
# # 1) MODE="freeze" 로 저장 후 실행 → 해시 확인
# # 2) MODE="tag" 로 바꿔 실행 → 이어서 푸시
# cd .. && git push --follow-tags && cd paper1
# # 3) MODE="run", QUICK=True → 오류 점검만 (수치 해석 금지)
# # 4) QUICK=False 로 바꿔 본 실행 (백그라운드 권장)
# nohup python3 -u pa_b2.py > 09_PA/PA_B2_full_log.txt 2>&1 &
# Everything up-to-date
# [1] 2326614
# yjlee@u24:~/Research/Minnesota_Transport/paper1$

In [ ]:
# yjlee@u24:~/Research/Minnesota_Transport$ cd ~/Research/Minnesota_Transport/paper1
# sed -i 's/^MODE = "tag"/MODE = "run"/' pa_b2.py && grep -n '^MODE\|^QUICK' pa_b2.py
# ( time python3 -u pa_b2.py ) 2>&1 | tee 09_PA/PA_B2_quick_log.txt
# 9:MODE = "run"
# 10:QUICK = True
# [관측 재현 OK] {'all': (0.1901, [2012, 2020]), 'fixed14': (0.1788, [2008, 2015, 2020]), 'within': (0.1495, [2009, 2015, 2020])}

# [Post-hoc] 2017-2019 대 2020-2022 토픽 변화 상위 10
#   rank  topic                                                         name   delta  within_journal  between_journal  grade_A
#     1     71 covid, 19, pandemic, travel, post, mobility, public, spread,  0.0175          0.0172           0.0003     True
#     2     69 prediction, graph, traffic, temporal, forecasting, flow, con  0.0085          0.0055           0.0030     True
#     3     45 phase, current, control, drives, fault, pmsm, torque, predic  0.0084          0.0029           0.0054     True
#     4    131 reinforcement, learning, deep, autonomous, control, signal,   0.0076          0.0060           0.0016     True
#     5     88 permanent, magnet, motor, flux, machines, machine, synchrono  0.0075          0.0009           0.0066     True
#     6     99 vehicular, edge, computing, offloading, internet, resource,   0.0069          0.0040           0.0029     True
#     7    138 driving, drivers, young, risky, self, anger, older, reported -0.0063         -0.0027          -0.0036     True
#     8    123 mobility, shared, maas, service, public, scooter, micromobil  0.0060          0.0069          -0.0009     True
#     9     20 privacy, secure, security, authentication, preserving, schem  0.0058          0.0038           0.0020     True
#    10    133 prediction, trajectory, intention, autonomous, driving, lear  0.0058          0.0040           0.0018     True
# 저널 간 성분 비중(전 토픽 절대합): 0.404

# [Post-hoc] 2019->2020 저널별 논문 수 증가 상위 10 (전체 증가 928)
#      venue  fixed14  n_prev  n_focus  delta  share_of_total_delta
#     T-ITS     True     425      706    281                 0.303
#      T-TE    False      94      219    125                 0.135
#       AAP     True     272      373    101                 0.109
#      TR-D     True     212      302     90                 0.097
#      TRIP    False      61      145     84                 0.091
#      TR-E     True     162      220     58                 0.062
#   ITS Mag    False      45       92     47                 0.051
#      RTBM    False      44       90     46                 0.050
# TranspPol     True     133      170     37                 0.040
#       TBS    False      55       90     35                 0.038
#   cap=3800 완료
#   cap=3000 완료

# [Post-hoc] PA-B2 하향 추출 요약
#   cap  sample  J_med  J_q05  J_q95  R_med  hit_rate  k_mode  J_obs  J_ratio
# 3000     all  0.194  0.191  0.199  0.716       1.0       3  0.190    1.021
# 3000 fixed14  0.184  0.180  0.191  0.671       1.0       3  0.179    1.028
# 3000  within  0.154  0.150  0.161  0.653       1.0       4  0.150    1.030
# 3800     all  0.190  0.187  0.192  0.728       1.0       3  0.190    0.997
# 3800 fixed14  0.180  0.176  0.183  0.746       1.0       4  0.179    1.004
# 3800  within  0.150  0.147  0.154  0.670       1.0       4  0.150    1.006

# [QUICK] 오류 점검 전용: 위 수치를 해석하거나 판독 규칙을 적용하지 마세요(B=20, v2 계획 사유 방지).

# real    0m1.313s
# user    0m4.155s
# sys     0m0.114s
# yjlee@u24:~/Research/Minnesota_Transport/paper1$

In [ ]:
# QUICK 점검은 통과했습니다. 관측 재현이 OK이고(all 0.1901/[2012, 2020], fixed14 0.1788, within 0.1495는 PA-A1과 같은 값), 하향 추출 두 cap과 요약 표가 끝까지 나왔으며, 소요 시간은 1.3초였습니다. 속도 문제는 없으니 본 실행도 B=200이라도 몇십 초면 끝날 것으로 보입니다. 이 수치는 B=20이므로 약속대로 해석하지 않고, 판독 규칙도 적용하지 않습니다.

# ## 먼저 기록할 것 (열람 사실)

# QUICK 출력에는 본 실행과 같은 값이 나오는 관측 기술 표(토픽 변화 상위 10, 저널별 증가 상위 10)와 하향 추출 요약이 포함됐고, 우리는 이를 이미 보았습니다. 그래서 "PA-B2 결과 미열람"은 더 이상 정확하지 않으므로, 아래처럼 실제 상태를 CHANGELOG에 남기세요. 본 실행 코드와 규칙은 바뀌지 않았습니다.

# ## 본 실행 절차

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B2 QUICK 점검(B=20, 수치 미해석): 오류 없이 완료(1.3초). QUICK 출력에 관측 기술 표(토픽 상위 10, 저널별 증가)와 하향 추출 요약이 포함되어 열람함. 기술 표는 결정적이라 본 실행과 같음. 판독 규칙·cap·창 변경 없음 | 본 결과 변경: 없음" >> 09_PA/PA_CHANGELOG.md
# sed -i 's/^QUICK = True/QUICK = False/' pa_b2.py && grep -n '^MODE\|^QUICK' pa_b2.py
# cd .. && git add paper1/pa_b2.py paper1/09_PA/PA_CHANGELOG.md paper1/09_PA/PA_B2_quick_log.txt paper1/09_PA/PA_B2_*_QUICK.csv
# git commit -m "PA-B2: QUICK check done; QUICK=False for full run (spec unchanged)" && git tag -a PA-B2-fullrun-2026-10-19 -m "full run config, pre-run" && git push --follow-tags
# cd paper1 && ( time python3 -u pa_b2.py ) 2>&1 | tee 09_PA/PA_B2_full_log.txt
# ```

# `grep`에서 `MODE = "run"`과 `QUICK = False`가 모두 보여야 합니다. 시간이 짧아 포그라운드로 돌려도 됩니다. 끝나면 `[동결 판독, cap=3800]` 줄과 `PA_B2_*.csv`(접미사 `_QUICK` 없음)가 생깁니다.

# ## 본 실행 후

# 이전과 같이 결과 CSV와 로그를 커밋·태그·푸시하고 로그 전체를 붙여 주세요.

# ```bash
# cd ~/Research/Minnesota_Transport
# git add paper1/09_PA/PA_B2_*.csv paper1/09_PA/PA_B2_full_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-B2 results" && git tag -a PA-B2-results-2026-10-19 -m "PA-B2 outputs, seen" && git push --follow-tags
# ```

# ## 해석 전에 알아 둘 설계 한계

# 이것은 결과가 아니라 동결된 설계에 대한 점검이므로 지금 적어 둡니다.

# - **cap이 약하게 작동합니다.** 분석 집합의 2020년 논문은 4,794편이고, cap 3,800은 이를 약 20% 줄이는 정도입니다. 표본 크기가 점프를 만들었다면 이 정도 하향 추출은 그 효과를 크게 지우지 못할 수 있습니다. 즉 이 검정은 "표본 크기 아티팩트"를 반증하는 힘이 제한적입니다. cap 3,000은 민감도로만 두었으므로 판독에는 쓰지 않지만, 본문에서는 두 cap의 결과를 함께 표로 보이세요.
# - **동결 판독은 점프와 회수율을 쓰는데, 둘 다 하향 추출에 둔감할 수 있습니다.** 그래서 "유지"가 나와도 "아티팩트가 아니다"의 강한 증거가 아니라 "이 정도 하향 추출로는 사라지지 않는다"로 쓰세요.
# - 저널 구성 효과는 `within` 표본이 이미 통제하므로, 본문에서는 "표본 크기" 판독과 분리해 서술하세요.
# - 보조로 보여 준 관측 표에서 2019→2020 저널별 증가는 T-ITS 한 곳이 30%를 차지한다는 점이 눈에 띕니다. 이는 기술 통계로 본문에 쓸 수 있지만, 판독 규칙과는 별개입니다.

# 실행 결과를 붙여 주시면 동결 규칙대로 읽겠습니다.

In [ ]:
# yjlee@u24:~/Research/Minnesota_Transport/paper1$ cd ~/Research/Minnesota_Transport/paper1
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B2 QUICK 점검(B=20, 수치 미해석): 오류 없이 완료(1.3초). QUICK 출력에 관측 기술 표(토픽 상위 10, 저널별 증가)와 하향 추출 요약이 포함되어 열람함. 기술 표는 결정적이라 본 실행과 같음. 판독 규칙·cap·창 변경 없음 | 본 결과 변경: 없음" >> 09_PA/PA_CHANGELOG.md
# sed -i 's/^QUICK = True/QUICK = False/' pa_b2.py && grep -n '^MODE\|^QUICK' pa_b2.py
# cd .. && git add paper1/pa_b2.py paper1/09_PA/PA_CHANGELOG.md paper1/09_PA/PA_B2_quick_log.txt paper1/09_PA/PA_B2_*_QUICK.csv
# git commit -m "PA-B2: QUICK check done; QUICK=False for full run (spec unchanged)" && git tag -a PA-B2-fullrun-2026-10-19 -m "full run config, pre-run" && git push --follow-tags
# cd paper1 && ( time python3 -u pa_b2.py ) 2>&1 | tee 09_PA/PA_B2_full_log.txt
# 9:MODE = "run"
# 10:QUICK = False
# [main 45cc113] PA-B2: QUICK check done; QUICK=False for full run (spec unchanged)
#  7 files changed, 222 insertions(+), 2 deletions(-)
#  create mode 100644 paper1/09_PA/PA_B2_cap_replicates_QUICK.csv
#  create mode 100644 paper1/09_PA/PA_B2_observed_topics_QUICK.csv
#  create mode 100644 paper1/09_PA/PA_B2_quick_log.txt
#  create mode 100644 paper1/09_PA/PA_B2_summary_QUICK.csv
#  create mode 100644 paper1/09_PA/PA_B2_venue_growth_QUICK.csv
# Enumerating objects: 17, done.
# Counting objects: 100% (17/17), done.
# Delta compression using up to 24 threads
# Compressing objects: 100% (12/12), done.
# Writing objects: 100% (12/12), 7.24 KiB | 3.62 MiB/s, done.
# Total 12 (delta 4), reused 0 (delta 0), pack-reused 0
# remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
# To https://github.com/LEEYJ1021/transport-knowledge-paper1.git
#    ca8a949..45cc113  main -> main
#  * [new tag]         PA-B2-fullrun-2026-10-19 -> PA-B2-fullrun-2026-10-19
# [관측 재현 OK] {'all': (0.1901, [2012, 2020]), 'fixed14': (0.1788, [2008, 2015, 2020]), 'within': (0.1495, [2009, 2015, 2020])}

# [Post-hoc] 2017-2019 대 2020-2022 토픽 변화 상위 10
#   rank  topic                                                         name   delta  within_journal  between_journal  grade_A
#     1     71 covid, 19, pandemic, travel, post, mobility, public, spread,  0.0175          0.0172           0.0003     True
#     2     69 prediction, graph, traffic, temporal, forecasting, flow, con  0.0085          0.0055           0.0030     True
#     3     45 phase, current, control, drives, fault, pmsm, torque, predic  0.0084          0.0029           0.0054     True
#     4    131 reinforcement, learning, deep, autonomous, control, signal,   0.0076          0.0060           0.0016     True
#     5     88 permanent, magnet, motor, flux, machines, machine, synchrono  0.0075          0.0009           0.0066     True
#     6     99 vehicular, edge, computing, offloading, internet, resource,   0.0069          0.0040           0.0029     True
#     7    138 driving, drivers, young, risky, self, anger, older, reported -0.0063         -0.0027          -0.0036     True
#     8    123 mobility, shared, maas, service, public, scooter, micromobil  0.0060          0.0069          -0.0009     True
#     9     20 privacy, secure, security, authentication, preserving, schem  0.0058          0.0038           0.0020     True
#    10    133 prediction, trajectory, intention, autonomous, driving, lear  0.0058          0.0040           0.0018     True
# 저널 간 성분 비중(전 토픽 절대합): 0.404

# [Post-hoc] 2019->2020 저널별 논문 수 증가 상위 10 (전체 증가 928)
#      venue  fixed14  n_prev  n_focus  delta  share_of_total_delta
#     T-ITS     True     425      706    281                 0.303
#      T-TE    False      94      219    125                 0.135
#       AAP     True     272      373    101                 0.109
#      TR-D     True     212      302     90                 0.097
#      TRIP    False      61      145     84                 0.091
#      TR-E     True     162      220     58                 0.062
#   ITS Mag    False      45       92     47                 0.051
#      RTBM    False      44       90     46                 0.050
# TranspPol     True     133      170     37                 0.040
#       TBS    False      55       90     35                 0.038
#   cap=3800 완료
#   cap=3000 완료

# [Post-hoc] PA-B2 하향 추출 요약
#   cap  sample  J_med  J_q05  J_q95  R_med  hit_rate  k_mode  J_obs  J_ratio
# 3000     all  0.194  0.189  0.198  0.715       1.0       3  0.190    1.020
# 3000 fixed14  0.184  0.179  0.189  0.670       1.0       3  0.179    1.030
# 3000  within  0.155  0.150  0.161  0.652       1.0       4  0.150    1.036
# 3800     all  0.190  0.186  0.193  0.728       1.0       3  0.190    0.998
# 3800 fixed14  0.179  0.175  0.183  0.746       1.0       4  0.179    1.000
# 3800  within  0.150  0.147  0.154  0.670       1.0       4  0.150    1.006

# [동결 판독, cap=3800] {'all': '유지', 'fixed14': '유지', 'within': '유지'}
# → 표본 크기 아티팩트 아님
# ※ J는 표집 잡음으로 위로 편향될 수 있고, 판독은 PLAN_B2의 규칙을 코드로 옮긴 것입니다. S5 판정(체제 서사 철회)은 바꾸지 않습니다.

# real    0m7.349s
# user    0m9.908s
# sys     0m0.394s
# yjlee@u24:~/Research/Minnesota_Transport/paper1$ cd ~/Research/Minnesota_Transport
# git add paper1/09_PA/PA_B2_*.csv paper1/09_PA/PA_B2_full_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-B2 results" && git tag -a PA-B2-results-2026-10-19 -m "PA-B2 outputs, seen" && git push --follow-tags
# [main 8bed2df] PA-B2 results
#  6 files changed, 1302 insertions(+)
#  create mode 100644 paper1/09_PA/PA_B2_cap_replicates.csv
#  create mode 100644 paper1/09_PA/PA_B2_full_log.txt
#  create mode 100644 paper1/09_PA/PA_B2_observed_topics.csv
#  create mode 100644 paper1/09_PA/PA_B2_summary.csv
#  create mode 100644 paper1/09_PA/PA_B2_venue_growth.csv
# Enumerating objects: 13, done.
# Counting objects: 100% (13/13), done.
# Delta compression using up to 24 threads
# Compressing objects: 100% (9/9), done.
# Writing objects: 100% (9/9), 28.74 KiB | 4.79 MiB/s, done.
# Total 9 (delta 3), reused 0 (delta 0), pack-reused 0
# remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
# To https://github.com/LEEYJ1021/transport-knowledge-paper1.git
#    45cc113..8bed2df  main -> main
#  * [new tag]         PA-B2-results-2026-10-19 -> PA-B2-results-2026-10-19
# yjlee@u24:~/Research/Minnesota_Transport$

In [ ]:
###아래를 1021부터 실행함##

In [ ]:
# # 논문 1 종합 정리 (2026-10-19, PA-B2 완료 시점)

# ## 1. 한눈에 보기

# | 구분 | 상태 | 핵심 |
# |---|---|---|
# | S0~S8 사전 명시 분석 | 완료·동결·푸시 | 체제 서사 철회, 신규 진입 주도 갱신, 선두 국가 교체, 정렬 분석 2/6 도메인 |
# | PA-0 재현성 | 대부분 완료 | 해시 152건 일치, MASTER_MANIFEST·TAGS 생성·푸시. **S4 코드 저장, 미추적 산출물 커밋은 확인 필요** |
# | PA-A1 (N2 검정력) | 완료 | run 1 무효(원인 확정), run 2 + 탐색 확장 |
# | **PA-B2 (2020 분리)** | **완료 (이번)** | 동결 판독 세 표본 모두 "유지" → 하지만 **판별력이 약한 검정** (3절) |
# | 나머지 PA 11개 | 미시작 | A2가 가장 중요 |
# | S9·S10 | 미시작 | |

# PA-B2 태그: `PA-B2-plan` (`b04eacd`), `PA-B2-code` (`ca8a949`), `PA-B2-fullrun` (`45cc113`), `PA-B2-results` (`8bed2df`). 모두 원격에 있습니다.

# ## 2. 지금까지 한 분석과 결과

# ### 2-1. 사전 명시 분석(S0~S8) 요약

# | 단계 | 핵심 결과 | 주의 |
# |---|---|---|
# | S0~S2 | F1X 111,416편, 고정 저널 14개. 국가 층 81,645편(보유율 94.7%, 수작업 200/200). 위험군 제외가 국가별로 불균등(중국 23.4%, 미국 8.4%) | 위험군 검증은 순환적, 소속 ≠ 국적 |
# | S3 | A 112 / B 33 / C 4 | 변별력 낮음 |
# | S4 | 크게 변한 토픽 50개(쇠퇴 25·성장 25), 쇠퇴의 저널 간 성분 중앙값 0.50, 성장 0.25. 안정성 0.700 | 경계 통과, 신생 정의 4종 불일치 |
# | S5 | 변화점 all 2012·2020, fixed14 2008·2015·2020, within 2009·2015·2020. N0·N1 통과, **N2 실패**(p 0.94/0.40/0.70) → 체제 서사 철회 | 2020은 세 표본 공통, 아티팩트 후보 |
# | S6 | 신규 진입 몫 0.24→0.38, 적응 0.25~0.28. 초과 진입분은 4쌍 모두 순열 최댓값 밖(순초과 0.026~0.039). 동명이인 변형 7종에서 방향 불변 | 순초과의 쌍 간 차이는 추세 근거 아님. K3는 첫 쌍 정의 민감 |
# | S7 | 미국 0.458→0.191, 중국 0.029→0.348(알려진 국가 기준). 상위 5개국 몫은 0과 구분 안 됨. 미국 감소는 초기 저널 구성, 중국 증가는 저널 내. 지역 변화도 신규 진입 항이 최대 | 집중도는 통계별 방향이 다름. 토픽별 중국 기여는 총량 효과와 미분리 |
# | S8 | 도메인 2/6 통과(safety, transit). 소속 기준 공급은 HIC 과잉·LIC/LMC 과소, 지표 일관 0.94, 두 렌즈 일치 85.9%, 중국·인도 제외 후 91~98% 유지 | GDP 통제 편 ρ는 약함(유의한 것 1개, 하한 0.002) |

# ### 2-2. 사후분석 계획과 동결 구조
# - 14개 항목, 핵심 3개(A1·A2·B2), 모든 산출물은 `[Post-hoc]` 표기, S0~S8 판정은 불변.
# - 계획 v1 → v2(탐지기를 S5 원본으로 교체) → PA-B2 계획(`47c93a7c5739`)을 실행 전에 해시·태그·푸시.
# - 결과를 본 뒤의 변경은 모두 CHANGELOG에 사유와 함께 기록. QUICK 실행은 오류 점검 전용이라고 못 박았으나, B2의 QUICK 출력에 기술 표와 요약이 포함되어 **열람했다는 사실을 기록**했습니다.

# ### 2-3. PA-A1 (N2 검정력) 결과

# | 실행 | 결과 |
# |---|---|
# | run 1 | 검정력이 점프 크기와 무관하게 약 0.05 → 주입 함수의 NaN 정규화 오류로 **무효**. 삭제하지 않고 커밋 |
# | run 2 (동결 사양) | δ=0 기각률 0.035/0.075/0.065(정상). 동결 판독: all 강화, within 강화, fixed14 곡선만 보고 |
# | 탐색 확장 | t*=2020에서 all·within은 δ=0.25부터 1.00, fixed14는 0.41~1.00. **t*=2012(fixed14·within)와 2008은 검정력 ≈0 또는 불안정(사각지대)**. 연속 꺾은선 BIC는 세 표본 모두 꺾임 2개(all·fixed14 2007/2016, within 2003/2012), **2020은 선택되지 않음** |

# 주의: δ=1의 추가 점프는 관측 초과분의 약 3배입니다. 근거 있는 서술은 "2020 부근에서 초과분 크기의 점프가 있었다면 탐지되었을 가능성이 높다"까지입니다.

# ### 2-4. PA-B2 (2020 변화의 표본 크기 아티팩트 분리) 결과

# **동결 판독(cap=3800, B=200)**

# | 표본 | 2020±1 회수율 | J 중앙값 / 관측 J | 판독 |
# |---|---|---|---|
# | all | 1.0 | 0.190 / 0.190 (0.998) | 유지 |
# | fixed14 | 1.0 | 0.179 / 0.179 (1.000) | 유지 |
# | within | 1.0 | 0.150 / 0.150 (1.006) | 유지 |

# cap=3000 민감도도 같습니다(J 비율 1.02~1.04, 회수율 1.0). 규칙상 결론은 **"표본 크기 아티팩트 아님"**입니다.

# **관측 기술 통계 (결정적, 판독과 무관)**
# - 2017~19 대 2020~22 토픽 변화 상위 10은 모두 A등급입니다. 1위는 COVID(토픽 71, +0.0175)이고 **거의 전부가 저널 내 변화**(0.0172 대 0.0003)입니다.
# - 이어서 교통 예측·그래프(69), 전기모터(45·88), 강화학습·자율주행(131·133), 차량 엣지 컴퓨팅(99), 공유 모빌리티(123), 보안(20)입니다. 운전자 행동(138)은 감소입니다.
# - 전기모터 토픽 45·88은 저널 간 성분이 큽니다(0.0054, 0.0066). 즉 **특정 저널의 몫 증가와 함께 움직입니다**. 전체 토픽의 저널 간 성분 비중은 0.404입니다.
# - 2019→2020 논문 증가분 928편 중 T-ITS가 281편(30.3%)을 차지하고, T-TE(13.5%), AAP(10.9%), TR-D(9.7%), TRIP(9.1%)가 뒤따릅니다.

# ## 3. PA-B2 결과의 올바른 해석 (과해석 방지)

# 동결 규칙의 결과는 분명하지만, 이 검정이 얼마나 센 증거인지는 따로 말해야 합니다.

# 1. **검정이 거의 통과하도록 설계된 구조입니다.** 2020년 논문은 4,794편이라 cap 3,800은 약 20%만 줄입니다. 그 정도 표집으로는 기대 비중이 바뀌지 않으므로 J 비율이 1.00 근처인 것은 거의 필연입니다. 회수율 1.0도 S5 부트스트랩에서 이미 2020이 1.0이었으므로 새로운 정보가 아닙니다.
# 2. **따라서 말할 수 있는 것은 좁습니다.** "연도 간 논문 수 차이를 20% 정도 줄여도 2020 부근 변화점과 점프 크기가 유지된다." "표본 크기 아티팩트가 아니라고 확정했다"까지는 가지 마세요.
# 3. **다루지 못한 것:** (a) 저널 구성 효과(T-ITS 30%, 저널 간 성분 0.404, 전기모터 토픽의 저널 간 성분), (b) 임베딩 보유율 변화 같은 측정 아티팩트, (c) 앞서 S5에서 N1 귀무가 2020 변화점을 약 99% 재현한 점. N1은 저널 규모를 유지하는 귀무이므로, 2020 변화점이 저널 규모 구조와 함께 움직인다는 단서입니다.
# 4. **내용으로 읽으면 2020은 한 가지 현상이 아닙니다.** COVID 토픽의 저널 내 급증과, 머신러닝·전기모터·자율주행 토픽의 지속 성장(일부는 특정 저널 몫과 함께)이 섞여 있습니다. 앞서 PA-A1에서 연속 꺾은선 BIC가 2020을 꺾임으로 선택하지 않았다는 점과 합치면, "2020은 단일 체제 전환이 아니라 COVID 충격과 연속 성장이 겹친 구간"이라는 서술이 자료와 가장 잘 맞습니다. 이 해석은 인과 주장이 아니고, 토픽 내용은 S3 카드와 대표 논문을 직접 읽어 확인해야 합니다(아직 읽지 않음).
# 5. **S5 판정은 바뀌지 않습니다.** 2020은 "아티팩트 후보"에서 "표본 크기 아티팩트만으로는 설명되지 않는 후보"로 문구만 정확해집니다. 체제 서사 철회(N2 실패)는 그대로입니다.

# ## 4. 현재 논문이 주장할 수 있는 것

# | 구분 | 문장 |
# |---|---|
# | **쓸 수 있음** | 주제 구조 변화는 점프보다 연속적 재배치로 서술하는 편이 자료에 부합한다 / 2020 부근에는 COVID 토픽의 저널 내 증가와 ML·전기 토픽 성장이 겹친 가속이 있으며 연도 간 표본 크기 차이만으로는 사라지지 않는다 / 갱신은 신규 진입이 가장 큰 항이며 같은 저널·연도 지속 저자 기대치를 넘는다 / 미국 몫 감소와 중국 몫 증가가 선두 교체의 핵심이며 미국 초기 감소는 저널 구성, 중국 증가는 저널 내 / 소속 위치 기준 공급은 소득군별로 안전·대중교통에서 HIC 과잉 |
# | **쓰면 안 됨** | 체제 전환이 없음이 확인됐다 / 2020이 아티팩트가 아니다(확정) / 연속 변화의 증거 일반 / 지식 생산이 소수 국가에 집중됐다 / 중국이 전기모터·텐서를 주도했다 / 저소득국의 필요가 충족되지 않았다 / 인과 해석 일체 |

# ## 5. 앞으로 해야 할 분석 (우선순위와 의존성)

# ### 단계 A. 즉시 (재현성 마무리, 1일)
# 1. **CHANGELOG에 B2 해석 메모 한 줄**(3절의 한계를 명시). 아래 명령으로 기록하세요.
#    ```bash
#    cd ~/Research/Minnesota_Transport
#    echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B2 본 실행 해석 메모: 동결 판독 3개 표본 모두 유지(cap 3800, 회수율 1.0, J 비율 0.998~1.006). 단, cap이 2020년 표본을 약 20%만 줄여 판별력이 약하고 저널 구성(저널 간 성분 0.404, T-ITS 증가 30%)과 측정 아티팩트는 다루지 못함. 문구는 '표본 크기 차이만으로는 사라지지 않는다'로 제한 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
#    git add paper1/09_PA/PA_CHANGELOG.md && git commit -m "PA-B2 interpretation memo" && git push
#    ```
# 2. **PA-0 확인:** S4 복구 코드 3종이 `paper1/`에 저장됐는지(`ls paper1/s4_*.py`), 미추적(`??`) 산출물이 남았는지(`git status --short paper1/`), `.gitignore`(`.ipynb_checkpoints/`, `_recovered_s4/`), `oa_raw` 외부 백업, GitHub 비공개 여부. 제가 이 상태를 직접 확인하지 못했습니다.

# ### 단계 B. 핵심 PA (가장 큰 효과)

# | PA | 내용 | 선행 필요 |
# |---|---|---|
# | **PA-A2** | 신규 진입 우세의 구성 방어: ID 기반 대조(2020+ 한정), 진입 정의 2종(첫 출판 후 2년 이내, 이전 5년 부재), 부트스트랩 편향 보정과 저자 단위 클러스터 부트스트랩 | **`s6_lib.make_pair`·`load_all` 소스** (이전 출력에는 함수 목록만 있어 신규·지속 판별 로직을 못 봄) |
# | **PA-C1** | 중국 소속 신규 저자를 제외해도 진입 우세가 유지되는가(S6×S7 연결) | A2와 같은 코드 기반 |
# | **PA-C2** | S7 국가 몫을 임베딩 보유 논문만으로 재계산, ±0.01 이내인지 | `country_mass_long` |

# A2와 C1이 논문의 새 중심 기여("신규 진입 주도")를 지탱하므로 이 둘을 먼저 하세요. 결과가 약해지면 서사와 저널 전략이 바뀝니다.

# ### 단계 C. 약점 리프레이밍 (새 자료 불필요한 것부터)

# | PA | 내용 | 난이도·메모 |
# |---|---|---|
# | **B5** | S4 안정성 Wilson 구간(약 [0.56, 0.81]), 분위 컷 곡선, 방향 축만 재계산 | 낮음, 기존 S4 파일만 사용 |
# | **B4** | F1+F3 이분 등급으로 S4~S6 재추정 | 중간 |
# | **B7** | 결측 최악·최선 경계, 분류기 대입, 저널×연도 IPW | 중간 |
# | **B3** | 신생 핵심 집합(n_defs≥3), D1' | F6 생성 코드와 `BASE` 정의 확인 필요 |
# | **A3** | S7 국가 미상 교정(rec 재집계), 균등 4구간, 중국 제외 HHI | 중간 |
# | **B9, B8** | 소득군 내 상관·더미 통제, 미달 4개 도메인 부록 정렬 | 중간 |
# | **B6** | ORCID 소표본 + 수작업 50쌍 | 인력 필요, 병렬 |
# | **A4** | 동결·열람 이력 부록 표 | 서술 작업 |

# ### 단계 D. B2를 보완할 선택 분석 (권장)
# PA-B2가 약한 이유를 직접 보완하는 분석입니다. 사후 명세이므로 `PA-B2b` 계획을 먼저 동결하고, 결과는 `[Post-hoc]`로만 보고하세요.
# - **저널 구성 분리:** 2020 점프에서 T-ITS(또는 상위 증가 저널 5개)를 제외한 재탐지와 JS 점프 재계산
# - **내용 분리:** COVID 토픽(71)과 관련 토픽을 제외한 재탐지로, 2020 점프가 COVID 토픽만으로 만들어지는지 확인
# - **토픽 카드 직접 읽기:** 71, 69, 45, 88, 131, 133, 99, 123, 20 (수작업, 해석의 안전장치)
# - **측정 아티팩트:** 2020 전후 임베딩 보유율 시계열(저널별)과 점프의 관계

# ### 단계 E. 마무리와 투고
# - S9: 후보 풀·추출 규칙 동결(명단은 만들지 않음)
# - S10: TRA 정규 선례 3편 이상 확인, 심사 공격-대응표, 규정 준수, 생성형 AI 사용 선언(실제 범위대로)
# - 본문 구성: "체제 전환"을 제목에서 빼고 "신규 진입 주도 갱신 + 선두 국가 교체"를 중심에 두는 안을 권합니다(저널 전략은 이전 정리 참조). B2 결과는 이 구성을 바꾸지 않습니다.
# - 영문 원고에 "Post-hoc analyses were specified after inspecting the primary results and are reported as exploratory; no pre-registered decision rule was altered." 포함, 기각된 PA 목록 부록

# ## 6. 지금 필요한 입력
# 1. **`s6_lib.py`의 `make_pair`·`load_all` 소스** (PA-A2·C1 코드를 쓰기 위해 가장 급함). 아래 명령의 출력을 붙여 주세요.
#    ```bash
#    cd ~/Research/Minnesota_Transport/paper1
#    python3 - <<'EOF'
#    import ast
#    t = open("s6_lib.py", encoding="utf-8").read(); L = t.splitlines()
#    for n in ast.parse(t).body:
#        if isinstance(n, ast.FunctionDef) and n.name in ("load_all", "make_pair", "pairs_of", "ipw_weights"):
#            print(f"\n--- {n.name} (L{n.lineno}-{n.end_lineno}) ---"); print("\n".join(L[n.lineno-1:n.end_lineno]))
#    EOF
#    ```
# 2. PA-0 확인 출력(`ls paper1/s4_*.py`, `git status --short paper1/`)
# 3. F6 생성 코드 위치와 `BASE` 정의(B3용)
# 4. 수작업 인력 여부(B6, B8-2)와 단계 D(B2 보완 분석)를 할지 여부

# 가장 가까운 한 걸음은 1번 출력입니다. 그것을 받으면 PA-A2의 계획 동결 코드부터 드리겠습니다.

In [ ]:
# PA-A2의 세 교정 중 지금 근거가 충분한 것은 3번(부트스트랩 교정)뿐입니다. 출력에는 `Pair` 클래스 본문이 없어서 진입 정의 민감도(2번)를 쓸 수 없고, ID 대조(1번)는 F7 파일 구조를 아직 못 봤습니다. 그래서 3번만 `PA-A2c`로 분리해 동결하고, 1·2번은 아래 출력을 받은 뒤 `PA-A2a/b`로 따로 동결하겠습니다. 한 계획서에 합쳐 놓고 나중에 수정하면 v2가 되기 때문입니다.

# ## PA-A2c: 부트스트랩 보고 방식 교정

# S6의 곱셈 가중 부트스트랩(저자×논문 Exp(1))은 그대로 두고, 저자 단위 클러스터 부트스트랩(저자 가중만)을 추가합니다. 편향 보정값 `2×관측 − 부트스트랩 중앙값`과 basic 구간 `[2×관측 − q_hi, 2×관측 − q_lo]`도 함께 냅니다. basic 구간은 정의상 점추정 쪽으로 이동하므로 "구간이 점추정을 포함하지 못한다"는 문제를 다룹니다. 핵심 통계는 진입 몫과 적응 몫의 차이입니다.

# `paper1/pa_a2c.py`로 저장합니다.

# ```python
# # -*- coding: utf-8 -*-
# # PA-A2c: S6 부트스트랩 보고 방식 교정(저자 단위 클러스터 부트스트랩 + 편향 보정). 미실행 코드.
# # MODE: "freeze" -> "tag" -> (git push --follow-tags) -> "run"(QUICK=True: 오류 점검 전용, 수치 미출력) -> "run"(QUICK=False)
# # S0~S8 판정 불변. 모든 산출물 [Post-hoc]. 신규·지속 판별과 분해식은 S6 코드(s6_lib.Pair) 그대로 사용.
# import sys, json, hashlib, subprocess, warnings
# from pathlib import Path
# import numpy as np, pandas as pd
# warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
# MODE = "freeze"
# QUICK = True

# ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; PA = P1/"09_PA"; S6D = P1/"06_S6"
# now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
# sha_b = lambda b: hashlib.sha256(b).hexdigest(); sha_f = lambda p: sha_b(Path(p).read_bytes())
# git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
# class Halt(Exception): pass
# spec_hash = lambda s: sha_b(json.dumps({k: v for k, v in s.items() if k not in ("sha256", "created", "refs")}, sort_keys=True, ensure_ascii=False).encode())
# V2P, BP = PA/"PA_0_PLAN_v2.json", PA/"PA_A2c_PLAN.json"

# PLAN = dict(
#     version=1, seed=2026, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(PA-A2의 3번 교정만). S6 결과는 열람한 상태. PA-A2c 결과는 미열람. ID 대조·진입 정의 민감도는 별도 계획(PA-A2a/b)으로 분리. S0~S8 판정 불변.",
#     setting="S6 주 설정: spec['main'] (scheme, venue, topic, hom)과 s6_lib.make_pair 그대로. 쌍은 s6_lib.pairs_of(spec) 전부",
#     B="spec['B_boot'] (S6와 같은 값, 실행 시 기록)", ci="spec['ci'] (S6와 같은 백분위)",
#     boot_modes={"author": "저자 가중 wa~Exp(1)만 적용, 논문 가중 1 (저자 단위 클러스터 복원추출의 연속형 근사)",
#                 "mult": "S6 원본: 저자 wa x 논문 wp 곱셈 Exp(1) (민감도로 병기)"},
#     stats={"entry_share": "half(entry)/sum(half(adapt,realloc,entry,exit))", "adapt_share": "half(adapt)/같은 분모",
#            "entry_minus_adapt": "entry_share - adapt_share", "excess_half": "half(P.excess(m1)[0])"},
#     report="통계별: 관측, 부트스트랩 중앙값, 편향 보정값(2*obs-median), 백분위 구간, basic 구간(2*obs-q_hi, 2*obs-q_lo)",
#     reading="저자 모드에서 entry_minus_adapt의 basic 구간 하한>0인 쌍 수를 보고. 4쌍 모두 >0이면 '진입>적응 방향이 저자 단위 재표집에서 유지'. 일부면 쌍별로 표기. mult 모드는 병기만 하며 판독에 쓰지 않음",
#     reproduce_check="관측 total_half가 renewal_decomposition.csv(scheme=main, variant=all_venues)와 1e-6 이내 일치하지 않으면 중단",
#     caveat="저자 가중만 흔드는 방식은 공저로 묶인 저자들의 상관과 논문 단위 변동을 반영하지 않음. 지속/진입 역할은 재표집에서 고정(S6와 동일). 동명이인·이름 기반 식별 문제는 이 분석이 다루지 않음(PA-A2a)",
#     forbidden="결과를 본 뒤 B·모드·판독 기준 변경(변경 시 v2 계획으로 분리하고 사유 기록)")

# def freeze():
#     if not V2P.exists(): raise Halt("PA_0_PLAN_v2.json이 없습니다")
#     h = spec_hash(PLAN); print("PA-A2c 계획 해시(예정)", h[:12])
#     if BP.exists(): print("이미 동결:", "동일" if json.loads(BP.read_text(encoding="utf-8"))["sha256"] == h else "⛔ 다름"); return
#     refs = {f.name: sha_f(f) for f in [V2P, S6D/"S6_0_PREREG.json", P1/"s6_lib.py", P1/"s6_run.py", S6D/"renewal_decomposition.csv", S6D/"renewal_bootstrap.csv"] if f.exists()}
#     BP.write_text(json.dumps(dict(PLAN, created=now(), sha256=h, refs=refs), ensure_ascii=False, indent=1), encoding="utf-8")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f:
#         f.write(f"\n- {now()} | PA-A2c 계획 동결(해시 {h[:12]}) | 사유: S6 부트스트랩 구간이 점추정을 포함하지 못한 문제의 보고 방식 교정. S6 결과·코드 열람 상태, PA-A2c 결과 미열람. QUICK은 수치 미출력 | 본 결과 변경 여부: 없음\n")
#     print("저장: PA_A2c_PLAN.json")

# def tag():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     t = f"PA-A2c-plan-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
#     if git("tag", "-l", t).stdout.strip(): print("이미 있음:", t); return
#     r = git("add", "--", "paper1/09_PA/PA_A2c_PLAN.json", "paper1/09_PA/PA_CHANGELOG.md", "paper1/pa_a2c.py"); print("add", r.returncode, r.stderr.strip())
#     r = git("commit", "-m", "PA-A2c plan freeze (before execution)"); print("commit", r.returncode, (r.stdout + r.stderr)[:200])
#     r = git("tag", "-a", t, "-m", "PA-A2c plan"); print("tag", r.returncode, r.stderr.strip()); print("※ 이어서 git push --follow-tags")

# def run():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     sp = json.loads(BP.read_text(encoding="utf-8"))
#     if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(PLAN): raise Halt("동결 파일과 현재 계획이 다릅니다")
#     if not git("tag", "-l", "PA-A2c-plan-*").stdout.strip(): raise Halt("tag 단계를 먼저 실행하세요")
#     sys.path.insert(0, str(P1)); import s6_lib as S
#     spec = S.load_spec(); L = S.load_all(spec); M = spec["main"]; PAIRS = S.pairs_of(spec); half = S.half
#     sfx = "_QUICK" if QUICK else ""; B = 20 if QUICK else int(spec["B_boot"]); lo_q, hi_q = spec["ci"]; rng = np.random.default_rng(int(PLAN["seed"]))
#     ref = pd.read_csv(S6D/"renewal_decomposition.csv"); ref = ref[(ref.scheme == M["scheme"]) & (ref.variant == "all_venues")].set_index("pair")
#     def stats(d, ex):
#         h = {t: half(d[t]) for t in ["adapt", "realloc", "entry", "exit"]}; tot = sum(h.values())
#         return dict(entry_share=h["entry"]/tot, adapt_share=h["adapt"]/tot, entry_minus_adapt=(h["entry"]-h["adapt"])/tot, excess_half=half(ex))
#     rows = []
#     for p0, p1 in PAIRS:
#         P = S.make_pair(L, p0, p1, M["scheme"], M["venue"], M["topic"], M["hom"]); d0 = P.decompose(); name = f"{p0}->{p1}"
#         r0 = float(ref.loc[name, "total_half"])
#         if abs(half(d0["total"]) - r0) > 1e-6: raise Halt(f"S6 관측 재현 실패({name}): {half(d0['total']):.8f} vs {r0:.8f}")
#         obs = stats(d0, P.excess()[0]); nAuth, nPap = P.nA, int(max(P.pc0.max(), P.pc1.max())) + 1
#         for mode in PLAN["boot_modes"]:
#             bs = []
#             for _ in range(B):
#                 wa = rng.exponential(size=nAuth); wp = rng.exponential(size=nPap) if mode == "mult" else np.ones(nPap)
#                 m0, m1 = P.m0 * wa[P.a0] * wp[P.pc0], P.m1 * wa[P.a1] * wp[P.pc1]
#                 bs.append(stats(P.decompose(m0, m1), P.excess(m1)[0]))
#             bs = pd.DataFrame(bs)
#             for c in bs.columns:
#                 med = float(bs[c].median()); ql, qh = np.percentile(bs[c], [lo_q, hi_q]); o = obs[c]
#                 rows.append(dict(pair=name, mode=mode, stat=c, obs=o, boot_median=med, bias_corrected=2*o - med, pct_lo=ql, pct_hi=qh, basic_lo=2*o - qh, basic_hi=2*o - ql, B=B))
#         print(f"  {name} 완료")
#     RS = pd.DataFrame(rows); RS.to_csv(PA/f"PA_A2c_results{sfx}.csv", index=False, encoding="utf-8-sig")
#     if QUICK: print("\n[QUICK] 오류 없이 완료. 수치는 출력하지 않음(_QUICK 파일을 열어 해석하지 마세요)."); return
#     print("\n[Post-hoc] PA-A2c 결과(저자 모드)\n", RS[RS["mode"] == "author"].round(4).to_string(index=False))
#     print("\n[Post-hoc] PA-A2c 결과(곱셈 모드, 병기)\n", RS[RS["mode"] == "mult"].round(4).to_string(index=False))
#     k = RS[(RS["mode"] == "author") & (RS.stat == "entry_minus_adapt")].set_index("pair"); kept = (k.basic_lo > 0)
#     print("\n[동결 판독] entry_minus_adapt basic 하한>0:", kept.to_dict(), "→", "4쌍 모두 유지" if kept.all() else ("일부 유지" if kept.any() else "유지 안 됨"))
#     print("※ 저자 가중만 흔든 근사이며 공저 연결성분·동명이인은 다루지 않음. S0~S8 판정 불변.")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as fh:
#         fh.write(f"- {now()} | PA-A2c 실행 | 정의: PA_A2c_PLAN {sp['sha256'][:12]} | 결과: PA_A2c_results.csv | 본 결과 변경 여부: 없음\n")

# try: {"freeze": freeze, "tag": tag, "run": run}[MODE]()
# except Halt as e: print(f"⛔ 중단: {e}")
# ```

# 실행 순서는 PA-B2와 같습니다.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 pa_a2c.py                                                           # 1) freeze: 해시 확인
# sed -i 's/^MODE = "freeze"/MODE = "tag"/' pa_a2c.py && python3 pa_a2c.py     # 2) tag
# cd .. && git push --follow-tags && cd paper1
# sed -i 's/^MODE = "tag"/MODE = "run"/' pa_a2c.py
# ( time python3 -u pa_a2c.py ) 2>&1 | tee 09_PA/PA_A2c_quick_log.txt         # 3) QUICK (수치 미출력)
# sed -i 's/^QUICK = True/QUICK = False/' pa_a2c.py
# cd .. && git add paper1/pa_a2c.py paper1/09_PA/PA_A2c_*QUICK* paper1/09_PA/PA_A2c_quick_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-A2c: QUICK done; QUICK=False" && git tag -a PA-A2c-fullrun-2026-10-21 -m "pre-run" && git push --follow-tags
# cd paper1 && ( time python3 -u pa_a2c.py ) 2>&1 | tee 09_PA/PA_A2c_full_log.txt   # 4) 본 실행
# ```

# ## 주의할 점

# 1. **미실행 코드이고 가정이 두 개 있습니다.** `S.half`가 `s6_lib`에 정의돼 있고(`s6_run.py`가 `import *`로 쓰고 있어서 그럴 것으로 봤습니다), `P.decompose()`의 반환값에 `entry`, `exit` 키가 있다고 가정했습니다. 후자는 `s6_run.py`의 multiverse 부분(`d["entry"]`)에서 확인한 것입니다. 오류가 나면 오류만 붙여 주세요.
# 2. **재현 점검이 들어 있습니다.** 관측 `total_half`가 `renewal_decomposition.csv`와 1e-6 이내로 맞지 않으면 중단합니다. 이 점검을 통과하면 `Pair`를 S6와 같은 방식으로 불러왔다는 뜻입니다.
# 3. **저자 모드는 근사입니다.** 저자 가중만 Exp(1)로 흔드는 방식이라 공저로 묶인 저자들의 상관은 반영하지 못합니다. S6가 "공저 연결성분 부트스트랩 미사용"이라고 밝힌 한계는 그대로 남습니다. 본문에서는 "저자 단위 재표집"이라고만 쓰고 "공저 구조를 반영했다"고 쓰지 마세요.
# 4. **B2b 코드는 아직 실행 전입니다.** 지난번에 드린 `pa_b2b.py`는 A2c와 독립적이라 어느 쪽을 먼저 돌려도 됩니다. 다만 동시에 돌리지는 마세요.

# ## PA-A2a/b에 필요한 출력

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 - <<'EOF'
# import ast
# t = open("s6_lib.py", encoding="utf-8").read(); L = t.splitlines()
# for n in ast.parse(t).body:
#     if isinstance(n, ast.ClassDef):
#         print(f"--- class {n.name} (L{n.lineno}-{n.end_lineno}) ---"); print("\n".join(L[n.lineno-1:n.end_lineno]))
# EOF
# python3 - <<'EOF'
# import json
# p = json.load(open("06_S6/S6_0_PREREG.json", encoding="utf-8"))
# for k in ("periods", "files", "cols", "decomposition", "data_end"): print(f"[{k}]", json.dumps(p[k], ensure_ascii=False)[:700])
# EOF
# ls ~/Research/Minnesota_Transport/analysis_data/final | grep -i -E "F7|openalex|oa_id|author"
# ```

# - `Pair` 클래스에는 신규(`E_new`)와 복귀(`E_ret`)를 가르는 규칙이 들어 있습니다. 이 규칙을 봐야 "첫 출판 후 2년 이내"와 "이전 5년 코퍼스 부재" 정의를 같은 분해식 위에서 구현할 수 있습니다.
# - 마지막 `ls`는 F7(OpenAlex ID) 파일의 이름과 열을 찾기 위한 것입니다. 열 이름도 `head -2`로 한 번 더 확인해 주세요.

In [ ]:
# yjlee@u24:~/Research/Minnesota_Transport/paper1$ cd ~/Research/Minnesota_Transport/paper1
# python3 pa_a2c.py                                                           # 1) freeze: 해시 확인
# sed -i 's/^MODE = "freeze"/MODE = "tag"/' pa_a2c.py && python3 pa_a2c.py     # 2) tag
# cd .. && git push --follow-tags && cd paper1
# sed -i 's/^MODE = "tag"/MODE = "run"/' pa_a2c.py
# ( time python3 -u pa_a2c.py ) 2>&1 | tee 09_PA/PA_A2c_quick_log.txt         # 3) QUICK (수치 미출력)
# sed -i 's/^QUICK = True/QUICK = False/' pa_a2c.py
# cd .. && git add paper1/pa_a2c.py paper1/09_PA/PA_A2c_*QUICK* paper1/09_PA/PA_A2c_quick_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-A2c: QUICK done; QUICK=False" && git tag -a PA-A2c-fullrun-2026-10-21 -m "pre-run" && git push --follow-tags
# cd paper1 && ( time python3 -u pa_a2c.py ) 2>&1 | tee 09_PA/PA_A2c_full_log.txt   # 4) 본 실행
# PA-A2c 계획 해시(예정) 6e9a0ce5ca3a
# 저장: PA_A2c_PLAN.json
# add 0
# commit 0 [main ba7cb3d] PA-A2c plan freeze (before execution)
#  3 files changed, 129 insertions(+)
#  create mode 100644 paper1/09_PA/PA_A2c_PLAN.json
#  create mode 100644 paper1/pa_a2c.py

# tag 0
# ※ 이어서 git push --follow-tags
# Enumerating objects: 12, done.
# Counting objects: 100% (12/12), done.
# Delta compression using up to 24 threads
# Compressing objects: 100% (8/8), done.
# Writing objects: 100% (8/8), 5.44 KiB | 5.44 MiB/s, done.
# Total 8 (delta 4), reused 0 (delta 0), pack-reused 0
# remote: Resolving deltas: 100% (4/4), completed with 3 local objects.
# To https://github.com/LEEYJ1021/transport-knowledge-paper1.git
#    9915db0..ba7cb3d  main -> main
#  * [new tag]         PA-A2c-plan-2026-10-21 -> PA-A2c-plan-2026-10-21
# IPW 모델별 행 수: {'A': 91794, 'B': 81645} | 사용: A
# IPW 범위(주제 논문) 0.866~1.883 (S4는 0.86~1.94) | 결측 0편은 1로 대체
# F4에 없는 저자 0명 | F4 첫해 > F2 첫해인 저자 0명(0이어야 정상) | F4 마지막해 < F2 마지막해 0명
# 분석 논문 76,984 | 저자-논문 행 255,354 | 저자 93,763 | 가중치 ['equal', 'inv_n_authors', 'first_last', 'full_count'] | 위험군 변형 ['baseline'] (F2는 위험군 제외 상태)
#   P0004->P0509 완료
#   P0509->P1014 완료
#   P1014->P1519 완료
#   P1519->P2025 완료

# [QUICK] 오류 없이 완료. 수치는 출력하지 않음(_QUICK 파일을 열어 해석하지 마세요).

# real    0m7.124s
# user    0m9.171s
# sys     0m0.352s
# [main c44d1c8] PA-A2c: QUICK done; QUICK=False
#  3 files changed, 49 insertions(+), 2 deletions(-)
#  create mode 100644 paper1/09_PA/PA_A2c_quick_log.txt
#  create mode 100644 paper1/09_PA/PA_A2c_results_QUICK.csv
# Enumerating objects: 12, done.
# Counting objects: 100% (12/12), done.
# Delta compression using up to 24 threads
# Compressing objects: 100% (8/8), done.
# Writing objects: 100% (8/8), 3.57 KiB | 3.57 MiB/s, done.
# Total 8 (delta 3), reused 0 (delta 0), pack-reused 0
# remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
# To https://github.com/LEEYJ1021/transport-knowledge-paper1.git
#    ba7cb3d..c44d1c8  main -> main
#  * [new tag]         PA-A2c-fullrun-2026-10-21 -> PA-A2c-fullrun-2026-10-21
# IPW 모델별 행 수: {'A': 91794, 'B': 81645} | 사용: A
# IPW 범위(주제 논문) 0.866~1.883 (S4는 0.86~1.94) | 결측 0편은 1로 대체
# F4에 없는 저자 0명 | F4 첫해 > F2 첫해인 저자 0명(0이어야 정상) | F4 마지막해 < F2 마지막해 0명
# 분석 논문 76,984 | 저자-논문 행 255,354 | 저자 93,763 | 가중치 ['equal', 'inv_n_authors', 'first_last', 'full_count'] | 위험군 변형 ['baseline'] (F2는 위험군 제외 상태)
#   P0004->P0509 완료
#   P0509->P1014 완료
#   P1014->P1519 완료
#   P1519->P2025 완료

# [Post-hoc] PA-A2c 결과(저자 모드)
#          pair   mode              stat     obs  boot_median  bias_corrected  pct_lo  pct_hi  basic_lo  basic_hi    B
# P0004->P0509 author       entry_share  0.2812       0.2698          0.2925  0.2473  0.2946    0.2677    0.3150 2000
# P0004->P0509 author       adapt_share  0.2866       0.2990          0.2742  0.2701  0.3261    0.2471    0.3032 2000
# P0004->P0509 author entry_minus_adapt -0.0055      -0.0294          0.0185 -0.0732  0.0172   -0.0281    0.0623 2000
# P0004->P0509 author       excess_half  0.0869       0.0987          0.0751  0.0897  0.1085    0.0652    0.0840 2000
# P0509->P1014 author       entry_share  0.2961       0.2828          0.3095  0.2641  0.3014    0.2909    0.3282 2000
# P0509->P1014 author       adapt_share  0.2690       0.2827          0.2553  0.2592  0.3053    0.2327    0.2788 2000
# P0509->P1014 author entry_minus_adapt  0.0271       0.0005          0.0538 -0.0340  0.0370    0.0173    0.0883 2000
# P0509->P1014 author       excess_half  0.0623       0.0728          0.0518  0.0661  0.0803    0.0444    0.0586 2000
# P1014->P1519 author       entry_share  0.3341       0.3216          0.3466  0.3055  0.3369    0.3313    0.3627 2000
# P1014->P1519 author       adapt_share  0.2880       0.2934          0.2825  0.2750  0.3130    0.2629    0.3009 2000
# P1014->P1519 author entry_minus_adapt  0.0461       0.0278          0.0644 -0.0027  0.0570    0.0352    0.0950 2000
# P1014->P1519 author       excess_half  0.0572       0.0644          0.0500  0.0586  0.0709    0.0435    0.0558 2000
# P1519->P2025 author       entry_share  0.3934       0.3854          0.4013  0.3757  0.3952    0.3916    0.4111 2000
# P1519->P2025 author       adapt_share  0.2806       0.2814          0.2797  0.2698  0.2938    0.2673    0.2914 2000
# P1519->P2025 author entry_minus_adapt  0.1128       0.1041          0.1215  0.0840  0.1227    0.1029    0.1416 2000
# P1519->P2025 author       excess_half  0.0552       0.0602          0.0502  0.0550  0.0655    0.0449    0.0554 2000

# [Post-hoc] PA-A2c 결과(곱셈 모드, 병기)
#          pair mode              stat     obs  boot_median  bias_corrected  pct_lo  pct_hi  basic_lo  basic_hi    B
# P0004->P0509 mult       entry_share  0.2812       0.2573          0.3050  0.2312  0.2848    0.2775    0.3311 2000
# P0004->P0509 mult       adapt_share  0.2866       0.2947          0.2785  0.2612  0.3290    0.2443    0.3121 2000
# P0004->P0509 mult entry_minus_adapt -0.0055      -0.0370          0.0261 -0.0890  0.0160   -0.0269    0.0781 2000
# P0004->P0509 mult       excess_half  0.0869       0.1114          0.0623  0.0977  0.1248    0.0489    0.0760 2000
# P0509->P1014 mult       entry_share  0.2961       0.2676          0.3247  0.2449  0.2908    0.3015    0.3474 2000
# P0509->P1014 mult       adapt_share  0.2690       0.2858          0.2522  0.2560  0.3142    0.2237    0.2820 2000
# P0509->P1014 mult entry_minus_adapt  0.0271      -0.0180          0.0723 -0.0615  0.0271    0.0272    0.1158 2000
# P0509->P1014 mult       excess_half  0.0623       0.0843          0.0404  0.0745  0.0942    0.0305    0.0501 2000
# P1014->P1519 mult       entry_share  0.3341       0.3048          0.3634  0.2846  0.3241    0.3441    0.3836 2000
# P1014->P1519 mult       adapt_share  0.2880       0.2953          0.2806  0.2719  0.3181    0.2579    0.3041 2000
# P1014->P1519 mult entry_minus_adapt  0.0461       0.0094          0.0828 -0.0270  0.0451    0.0471    0.1193 2000
# P1014->P1519 mult       excess_half  0.0572       0.0731          0.0413  0.0650  0.0816    0.0328    0.0494 2000
# P1519->P2025 mult       entry_share  0.3934       0.3749          0.4118  0.3620  0.3873    0.3994    0.4248 2000
# P1519->P2025 mult       adapt_share  0.2806       0.2825          0.2786  0.2656  0.2993    0.2618    0.2956 2000
# P1519->P2025 mult entry_minus_adapt  0.1128       0.0925          0.1331  0.0665  0.1178    0.1078    0.1591 2000
# P1519->P2025 mult       excess_half  0.0552       0.0654          0.0450  0.0590  0.0723    0.0382    0.0515 2000

# [동결 판독] entry_minus_adapt basic 하한>0: {'P0004->P0509': False, 'P0509->P1014': True, 'P1014->P1519': True, 'P1519->P2025': True} → 일부 유지
# ※ 저자 가중만 흔든 근사이며 공저 연결성분·동명이인은 다루지 않음. S0~S8 판정 불변.

# real    6m43.968s
# user    6m46.135s
# sys     0m0.359s
# yjlee@u24:~/Research/Minnesota_Transport/paper1$ cd ~/Research/Minnesota_Transport/paper1
# python3 - <<'EOF'
# import ast
# t = open("s6_lib.py", encoding="utf-8").read(); L = t.splitlines()
# for n in ast.parse(t).body:
#     if isinstance(n, ast.ClassDef):
#         print(f"--- class {n.name} (L{n.lineno}-{n.end_lineno}) ---"); print("\n".join(L[n.lineno-1:n.end_lineno]))
# EOF
# python3 - <<'EOF'
# import json
# p = json.load(open("06_S6/S6_0_PREREG.json", encoding="utf-8"))
# for k in ("periods", "files", "cols", "decomposition", "data_end"): print(f"[{k}]", json.dumps(p[k], ensure_ascii=False)[:700])
# EOF
# ls ~/Research/Minnesota_Transport/analysis_data/final | grep -i -E "F7|openalex|oa_id|author"
# --- class Pair (L80-137) ---
# class Pair:
#     def __init__(s, R, p0, p1, L, mcol):
#         spec = L["spec"]; a, b = spec["periods"][p0], spec["periods"][p1]; s.K, s.nV = L["K"], L["V"]; s.p0, s.p1 = p0, p1
#         R = R[R[mcol] > 0]                                                              # 가중치 0인 행(예: first_last의 중간 저자)은 '존재'로 세지 않음
#         r0, r1 = R[R.period == p0], R[R.period == p1]
#         s.aids = np.union1d(r0.ai.values, r1.ai.values); s.nA = len(s.aids); ix = lambda x: np.searchsorted(s.aids, x)
#         s.a0, s.t0, s.m0 = ix(r0.ai.values), r0.ti.values, r0[mcol].values.astype(float)
#         s.a1, s.t1, s.m1 = ix(r1.ai.values), r1.ti.values, r1[mcol].values.astype(float)
#         s.c1, s.v1 = pd.factorize(r1.cell.values)[0], r1.vi.values; s.nC = s.c1.max() + 1
#         s.pc0, s.pc1 = r0.pc.values, r1.pc.values
#         in0 = np.zeros(s.nA, bool); in0[s.a0] = True; in1 = np.zeros(s.nA, bool); in1[s.a1] = True
#         fy, ly = L["first"][s.aids], L["last"][s.aids]; role = np.full(s.nA, "", dtype=object)
#         role[in0 & in1] = "P"; role[~in0 & in1 & (fy >= b[0])] = "E_new"; role[~in0 & in1 & (fy < b[0])] = "E_ret"
#         role[in0 & ~in1 & (ly <= a[1])] = "X_final"; role[in0 & ~in1 & (ly > a[1])] = "X_gap"
#         s.role = role; s.isE0 = np.isin(role[s.a1], ["E_new", "E_ret"]); s.exit_censored = b[1] >= spec["data_end"]
#         s.n0 = np.bincount(s.a0, minlength=s.nA); s.n1 = np.bincount(s.a1, minlength=s.nA)
#     def mats(s, m0=None, m1=None):
#         m0 = s.m0 if m0 is None else m0; m1 = s.m1 if m1 is None else m1
#         mk = lambda a, t, m: sp.coo_matrix((m / m.sum(), (a, t)), shape=(s.nA, s.K)).tocsr()
#         return mk(s.a0, s.t0, m0), mk(s.a1, s.t1, m1)
#     def decompose(s, m0=None, m1=None):
#         M0, M1 = s.mats(m0, m1); w0 = np.asarray(M0.sum(1)).ravel(); w1 = np.asarray(M1.sum(1)).ravel()
#         S0 = np.asarray(M0.sum(0)).ravel(); S1 = np.asarray(M1.sum(0)).ravel(); Sb = (S0 + S1) / 2
#         assert abs(S0.sum() - 1) < 1e-9 and abs(S1.sum() - 1) < 1e-9, "질량 보존 실패"
#         inv = lambda w: np.where(w > 0, 1 / np.where(w > 0, w, 1), 0.0)
#         s0 = sp.diags(inv(w0)) @ M0; s1 = sp.diags(inv(w1)) @ M1; P = s.role == "P"; out = {}
#         sv = lambda X: np.asarray(X.sum(0)).ravel()
#         out["adapt"] = sv(sp.diags(((w0 + w1) / 2)[P]) @ (s1[P] - s0[P]))
#         out["realloc"] = sv(sp.diags((w1 - w0)[P]) @ ((s0[P] + s1[P]) / 2)) - Sb * (w1 - w0)[P].sum()
#         for r in ["E_new", "E_ret"]: m = s.role == r; out["entry_" + r[2:]] = sv(M1[m]) - Sb * w1[m].sum()
#         for r in ["X_final", "X_gap"]: m = s.role == r; out["exit_" + r[2:]] = -(sv(M0[m]) - Sb * w0[m].sum())
#         out["total"] = S1 - S0
#         assert np.allclose(sum(v for k, v in out.items() if k != "total"), out["total"], atol=1e-9), "불변식 실패(항 합≠총변화): 이 실행 수치 폐기"
#         out["entry"] = out["entry_new"] + out["entry_ret"]; out["exit"] = out["exit_final"] + out["exit_gap"]; return out
#     def excess(s, m1=None, isE=None):
#         """초과 진입분 = 진입 저자 토픽 질량 − 같은 저널×연도 셀 지속 저자 분포로 대체한 기대치(셀에 지속 질량이 없으면 제외)."""
#         m1 = s.m1 if m1 is None else m1; isE = s.isE0 if isE is None else isE; m = m1 / m1.sum(); K = s.K
#         Pm = np.bincount(s.c1[~isE] * K + s.t1[~isE], m[~isE], s.nC * K).reshape(s.nC, K); Pc = Pm.sum(1)
#         Em = np.bincount(s.c1[isE], m[isE], s.nC); ok = Pc > 0; sel = isE & ok[s.c1]
#         obs = np.bincount(s.t1[sel], m[sel], K); exp = ((Em[ok] / Pc[ok])[:, None] * Pm[ok]).sum(0)
#         return obs - exp, float(Em[~ok].sum() / max(Em.sum(), 1e-300))
#     def perm_isE(s, rng):
#         n = len(s.c1); order = np.lexsort((rng.random(n), s.c1)); cs = s.c1[order]
#         start = np.r_[0, np.flatnonzero(np.diff(cs)) + 1]; cnt = np.diff(np.r_[start, n]); rank = np.arange(n) - np.repeat(start, cnt)
#         nE = np.bincount(s.c1[s.isE0], minlength=s.nC); lab = np.empty(n, bool); lab[order] = rank < nE[cs]; return lab
#     def path_gap(s, m1=None):
#         """신규-지속 토픽 분포 차이 E_k−P_k의 저널 내/간 분해(중점 가중). 불변식: 내+간=E−P."""
#         m1 = s.m1 if m1 is None else m1; V, K = s.nV, s.K
#         def dist(sel):
#             Mj = np.bincount(s.v1[sel] * K + s.t1[sel], m1[sel], V * K).reshape(V, K); j = Mj.sum(1)
#             if j.sum() <= 0: return None
#             return j / j.sum(), np.divide(Mj, j[:, None], out=np.zeros_like(Mj), where=j[:, None] > 0), Mj.sum(0) / Mj.sum()
#         E, Pp = dist(s.isE0), dist(~s.isE0)
#         if E is None or Pp is None: return None
#         e, Ejk, Ek = E; p, Pjk, Pk = Pp
#         within = (((e + p) / 2)[:, None] * (Ejk - Pjk)).sum(0); between = (((Ejk + Pjk) / 2) * (e - p)[:, None]).sum(0)
#         assert np.allclose(within + between, Ek - Pk, atol=1e-9), "경로 분해 불변식 실패"
#         return dict(within=within, between=between, gap=Ek - Pk)
# [periods] {"P0004": [2000, 2004], "P0509": [2005, 2009], "P1014": [2010, 2014], "P1519": [2015, 2019], "P2025": [2020, 2025]}
# [files] {"papers": "F1_papers.csv", "author_paper": "F2_author_paper.csv", "authors": "F4_authors.csv", "ipw": "paper_ipw.csv.gz", "alpha": "alpha_flags.csv"}
# [cols] {"pid": ["pid"], "aid": ["name_norm"], "n_authors": ["n_authors"], "position": ["author_pos"]}
# [decomposition] "Griliches-Regev 중점(적응·재배분·진입(신규+복귀)·퇴출(최종+공백))"
# [data_end] 2025
# F2_author_paper.csv
# F3_author_period_venue_topic.csv
# F4_authors.csv
# F5_continuing_authors.csv
# F7_oa_author_ids.csv
# yjlee@u24:~/Research/Minnesota_Transport/paper1$

In [ ]:
# ## PA-A2c 판독

# **동결 규칙의 기계적 결과는 "일부 유지"입니다.** 규칙은 entry_minus_adapt의 basic 구간 하한이 0보다 큰 쌍의 수를 세는 것이었고, 4쌍 중 3쌍이 통과했습니다. 첫 쌍(P0004→P0509)만 통과하지 못했습니다(관측 −0.0055, basic [−0.028, 0.062]).

# 세 가지를 함께 적어 두어야 합니다.

# 1. **통과한 3쌍의 근거 강도가 다릅니다.** 같은 표의 백분위 구간(pct_lo)은 P0509→P1014에서 −0.034, P1014→P1519에서 −0.0027로 0을 포함합니다. 저자 단위 재표집의 중앙값(0.0005, 0.0278)이 관측값(0.0271, 0.0461)보다 낮아서, basic 구간은 이 차이를 편향으로 보고 위로 옮깁니다. 이 보정을 믿을지가 결론을 가릅니다. 백분위와 basic이 모두 0보다 큰 쌍은 마지막 쌍(P1519→P2025, pct_lo 0.084)뿐입니다. 본문에는 "진입>적응은 마지막 구간에서 두 구간 방식 모두 유지되고, 중간 두 구간은 구간 방식에 따라 갈린다"가 정확합니다. 규칙 판독은 바꾸지 않습니다.
# 2. **excess_half의 문제는 이 교정으로 해결되지 않았습니다.** 저자 모드에서 백분위 구간 하한이 관측값보다 큰 쌍이 3개입니다(0.0897>0.0869, 0.0661>0.0623, 0.0586>0.0572). basic 구간도 첫 세 쌍에서 관측값을 포함하지 못합니다(basic_hi 0.0840<0.0869 등). 부트스트랩 분포 전체가 관측값 위에 있어서 구간 변환으로는 맞출 수 없습니다. 절대값 합 통계의 상향 편향이 구간 폭보다 크다는 뜻입니다. 따라서 초과 진입분의 근거는 부트스트랩 구간이 아니라 S6의 층화 순열(음성 대조)이어야 하고, 본문에서 excess의 부트스트랩 구간을 쓰려면 이 점을 함께 밝혀야 합니다.
# 3. **분모와 항 정의가 S6 요약과 다릅니다.** 여기서는 진입을 신규+복귀 합으로 놓고 분모를 적응·재배분·진입·퇴출 네 항으로 했습니다. 이전 정리의 "신규 진입 몫 0.24→0.38"과 직접 비교하지 마세요. 어느 분모인지 확인한 뒤 쓰세요.

# 곱셈 모드는 저자 모드와 같은 방향입니다. 병기만 하고 판독에는 쓰지 않았습니다. 중앙값이 관측값보다 낮아지는 이유(가중치 잡음이 절대값 합 항을 부풀려 차이를 줄인다는 설명)는 가설이고 확인하지 않았습니다.

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-A2c 결과 해석 메모(B=2000): 동결 판독 entry_minus_adapt basic 하한>0은 4쌍 중 3쌍(첫 쌍 제외). 단 백분위 구간은 P0509->P1014(-0.034)·P1014->P1519(-0.0027)에서 0 포함, 두 방식 모두 양수인 쌍은 P1519->P2025뿐. excess_half는 부트스트랩 분포가 관측값 위에 있어 3쌍에서 백분위·basic 구간 모두 관측값 미포함(교정 불충분, 순열 대조가 근거). 저자 가중만 흔든 근사, 분모는 4항(진입=신규+복귀)이라 S6 요약 몫과 직접 비교 불가 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_A2c_*.csv paper1/09_PA/PA_A2c_full_log.txt paper1/09_PA/PA_CHANGELOG.md paper1/pa_a2c.py
# git commit -m "PA-A2c results" && git tag -a PA-A2c-results-2026-10-21 -m "PA-A2c outputs, seen" && git push --follow-tags
# ```

# ## PA-A2b: 신규 진입 정의 민감도

# `Pair` 소스를 보고 알게 된 사실이 설계에 영향을 줍니다. 진입 총량(신규+복귀)은 신규/복귀를 어떻게 나누든 변하지 않습니다. 정의에 따라 달라지는 것은 신규 몫과 복귀 몫의 분할이므로, "신규 몫 대 적응 몫"을 정의별로 비교합니다. 적응 항은 지속 저자(`role=="P"`)만으로 정해져서 정의와 무관합니다.

# 정의는 세 가지입니다.

# - **s6**: 원본(첫 출판 연도 ≥ t1 시작). 재현 점검용입니다.
# - **y2**: 첫 출판 연도가 t1 시작 후 2년 이내(t1 시작 연도와 다음 해)입니다. 나머지 진입은 "신규 아님"으로 묶습니다.
# - **y5**: t1 시작 전 5년간 분석 코퍼스(`L["R"]`) 행에 없던 저자입니다. 첫 쌍은 2000년 이전 자료가 없어서 전원이 신규가 됩니다. 이 한계는 결과에 표시됩니다.

# `paper1/pa_a2b.py`로 저장합니다.

# ```python
# # -*- coding: utf-8 -*-
# # PA-A2b: 신규 진입 정의 민감도(s6 원본 / 첫 출판 후 2년 이내 / 이전 5년 코퍼스 부재). 미실행 코드.
# # MODE: "freeze" -> "tag" -> (git push --follow-tags) -> "run"(QUICK=True: 오류 점검 전용, 수치 미출력) -> "run"(QUICK=False)
# # S0~S8 판정 불변. 모든 산출물 [Post-hoc]. 분해식은 s6_lib.Pair 그대로, 신규 분할만 마스크로 재계산.
# import sys, json, hashlib, subprocess, warnings
# from pathlib import Path
# import numpy as np, pandas as pd
# warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
# MODE = "freeze"
# QUICK = True

# ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; PA = P1/"09_PA"; S6D = P1/"06_S6"
# now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
# sha_b = lambda b: hashlib.sha256(b).hexdigest(); sha_f = lambda p: sha_b(Path(p).read_bytes())
# git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
# class Halt(Exception): pass
# spec_hash = lambda s: sha_b(json.dumps({k: v for k, v in s.items() if k not in ("sha256", "created", "refs")}, sort_keys=True, ensure_ascii=False).encode())
# V2P, BP = PA/"PA_0_PLAN_v2.json", PA/"PA_A2b_PLAN.json"

# PLAN = dict(
#     version=1, seed=2026, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(PA-A2의 2번 교정). S6 결과·Pair 소스·PA-A2c 결과 열람 상태. PA-A2b 결과 미열람. S0~S8 판정 불변.",
#     setting="S6 주 설정 spec['main']과 s6_lib.make_pair 그대로, 쌍은 pairs_of(spec) 전부. 적응·재배분·진입 총량·퇴출은 Pair.decompose 그대로(정의와 무관)",
#     definitions={"s6": "E이고 첫 출판 연도 >= t1 시작 (S6 원본, 재현 점검)",
#                  "y2": "E이고 첫 출판 연도 in [t1 시작, t1 시작+1]",
#                  "y5": "E이고 t1 시작 전 5년(t1시작-5 ~ t1시작-1)의 분석 코퍼스 행(L['R'])에 없던 저자. 첫 쌍은 2000년 이전 자료가 없어 전원 해당"},
#     stats={"new_share": "half(entry_def)/(half(adapt)+half(realloc)+half(entry_total)+half(exit))", "adapt_share": "half(adapt)/같은 분모",
#            "new_minus_adapt": "new_share - adapt_share", "rest_entry_share": "(entry_total - entry_def의 half)/같은 분모는 쓰지 않고 half(entry_total - entry_def)/같은 분모"},
#     boot="저자 가중 wa~Exp(1)만 적용(PA-A2c 저자 모드와 동일), B=spec['B_boot'], 역할·마스크는 재표집에서 고정",
#     report="정의x쌍별: 관측, 부트스트랩 중앙값, 백분위 구간, basic 구간(2*obs-q_hi, 2*obs-q_lo)",
#     reading="정의별로 new_minus_adapt의 관측>0이고 basic 하한>0인 쌍 수를 보고. 정의 간 평균하거나 하나의 결론으로 합치지 않음. 첫 쌍은 s6 원본에서도 통과하지 못했을 가능성이 있으며 그 경우 그대로 보고",
#     reproduce_check="(a) s6 정의의 entry_def가 decompose의 entry_new와 1e-9 이내, (b) total_half가 renewal_decomposition.csv와 1e-6 이내. 아니면 중단",
#     caveat="진입 총량은 정의와 무관하므로 이 분석은 신규/복귀 분할 민감도만 본다. y5는 분석 코퍼스(주제 있는 논문)의 행을 기준으로 해 F4 첫 출판 연도(주제 없는 논문 포함)와 기준이 다르다. 이름 기반 식별 문제는 다루지 않음(PA-A2a)",
#     forbidden="결과를 본 뒤 정의·B·판독 기준 변경(변경 시 v2 계획으로 분리하고 사유 기록)")

# def freeze():
#     if not V2P.exists(): raise Halt("PA_0_PLAN_v2.json이 없습니다")
#     h = spec_hash(PLAN); print("PA-A2b 계획 해시(예정)", h[:12])
#     if BP.exists(): print("이미 동결:", "동일" if json.loads(BP.read_text(encoding="utf-8"))["sha256"] == h else "⛔ 다름"); return
#     refs = {f.name: sha_f(f) for f in [V2P, S6D/"S6_0_PREREG.json", P1/"s6_lib.py", S6D/"renewal_decomposition.csv", PA/"PA_A2c_PLAN.json"] if f.exists()}
#     BP.write_text(json.dumps(dict(PLAN, created=now(), sha256=h, refs=refs), ensure_ascii=False, indent=1), encoding="utf-8")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f:
#         f.write(f"\n- {now()} | PA-A2b 계획 동결(해시 {h[:12]}) | 사유: S6 신규 정의 민감도(신규/복귀 분할). S6·Pair·PA-A2c 결과 열람 상태, PA-A2b 결과 미열람. QUICK은 수치 미출력 | 본 결과 변경 여부: 없음\n")
#     print("저장: PA_A2b_PLAN.json")

# def tag():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     t = f"PA-A2b-plan-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
#     if git("tag", "-l", t).stdout.strip(): print("이미 있음:", t); return
#     r = git("add", "--", "paper1/09_PA/PA_A2b_PLAN.json", "paper1/09_PA/PA_CHANGELOG.md", "paper1/pa_a2b.py"); print("add", r.returncode, r.stderr.strip())
#     r = git("commit", "-m", "PA-A2b plan freeze (before execution)"); print("commit", r.returncode, (r.stdout + r.stderr)[:200])
#     r = git("tag", "-a", t, "-m", "PA-A2b plan"); print("tag", r.returncode, r.stderr.strip()); print("※ 이어서 git push --follow-tags")

# def run():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     sp = json.loads(BP.read_text(encoding="utf-8"))
#     if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(PLAN): raise Halt("동결 파일과 현재 계획이 다릅니다")
#     if not git("tag", "-l", "PA-A2b-plan-*").stdout.strip(): raise Halt("tag 단계를 먼저 실행하세요")
#     sys.path.insert(0, str(P1)); import s6_lib as S
#     spec = S.load_spec(); L = S.load_all(spec); M = spec["main"]; PAIRS = S.pairs_of(spec); half = S.half; R = L["R"]
#     sfx = "_QUICK" if QUICK else ""; B = 20 if QUICK else int(spec["B_boot"]); lo_q, hi_q = spec["ci"]; rng = np.random.default_rng(int(PLAN["seed"]))
#     ref = pd.read_csv(S6D/"renewal_decomposition.csv"); ref = ref[(ref.scheme == M["scheme"]) & (ref.variant == "all_venues")].set_index("pair")
#     DEFS = list(PLAN["definitions"]); rows = []
#     def compute(P, masks, m0=None, m1=None):
#         d = P.decompose(m0, m1); M0, M1 = P.mats(m0, m1); Sb = (np.asarray(M0.sum(0)).ravel() + np.asarray(M1.sum(0)).ravel())/2
#         w1 = np.asarray(M1.sum(1)).ravel(); tot = sum(half(d[t]) for t in ["adapt", "realloc", "entry", "exit"]); out = {}
#         for k, mk in masks.items():
#             ent = np.asarray(M1[mk].sum(0)).ravel() - Sb*w1[mk].sum() if mk.any() else np.zeros(P.K)
#             out[k] = dict(new_share=half(ent)/tot, adapt_share=half(d["adapt"])/tot, new_minus_adapt=(half(ent) - half(d["adapt"]))/tot, rest_entry_share=half(d["entry"] - ent)/tot, ent=ent)
#         return out, d
#     for p0, p1 in PAIRS:
#         P = S.make_pair(L, p0, p1, M["scheme"], M["venue"], M["topic"], M["hom"]); name = f"{p0}->{p1}"; b0 = spec["periods"][p1][0]
#         E = np.isin(P.role, ["E_new", "E_ret"]); fy = L["first"][P.aids]
#         prev = np.unique(R.ai.values[R.year.between(b0 - 5, b0 - 1).values]); present = np.isin(P.aids, prev)
#         masks = {"s6": E & (fy >= b0), "y2": E & (fy >= b0) & (fy <= b0 + 1), "y5": E & ~present}
#         obs, d0 = compute(P, masks)
#         r0 = float(ref.loc[name, "total_half"])
#         if abs(half(d0["total"]) - r0) > 1e-6: raise Halt(f"S6 관측 재현 실패({name})")
#         if np.abs(obs["s6"]["ent"] - d0["entry_new"]).max() > 1e-9: raise Halt(f"s6 정의가 decompose의 entry_new와 다름({name})")
#         bs = {k: [] for k in DEFS}
#         for _ in range(B):
#             wa = rng.exponential(size=P.nA); b_, _d = compute(P, masks, P.m0*wa[P.a0], P.m1*wa[P.a1])
#             for k in DEFS: bs[k].append(b_[k])
#         for k in DEFS:
#             B_ = pd.DataFrame([{c: v[c] for c in ["new_share", "adapt_share", "new_minus_adapt", "rest_entry_share"]} for v in bs[k]])
#             for c in B_.columns:
#                 o = obs[k][c]; ql, qh = np.percentile(B_[c], [lo_q, hi_q])
#                 rows.append(dict(pair=name, definition=k, stat=c, n_entrants=int(masks[k].sum()), obs=o, boot_median=float(B_[c].median()), pct_lo=ql, pct_hi=qh, basic_lo=2*o - qh, basic_hi=2*o - ql, B=B))
#         print(f"  {name} 완료")
#     RS = pd.DataFrame(rows); RS.to_csv(PA/f"PA_A2b_results{sfx}.csv", index=False, encoding="utf-8-sig")
#     if QUICK: print("\n[QUICK] 오류 없이 완료. 수치는 출력하지 않음(_QUICK 파일을 열어 해석하지 마세요)."); return
#     print("\n[Post-hoc] PA-A2b 결과\n", RS.round(4).to_string(index=False))
#     k = RS[RS.stat == "new_minus_adapt"].copy(); k["kept"] = (k.obs > 0) & (k.basic_lo > 0)
#     print("\n[동결 판독] 정의별 new_minus_adapt(관측>0 and basic 하한>0)인 쌍 수:", k.groupby("definition").kept.sum().to_dict(), "/ 쌍 수", len(PAIRS))
#     print("※ 정의별로 따로 읽고 하나의 결론으로 합치지 않음. 진입 총량은 정의와 무관. S0~S8 판정 불변.")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as fh:
#         fh.write(f"- {now()} | PA-A2b 실행 | 정의: PA_A2b_PLAN {sp['sha256'][:12]} | 결과: PA_A2b_results.csv | 본 결과 변경 여부: 없음\n")

# try: {"freeze": freeze, "tag": tag, "run": run}[MODE]()
# except Halt as e: print(f"⛔ 중단: {e}")
# ```

# 실행 순서는 PA-A2c와 같습니다.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 pa_a2b.py                                                           # 1) freeze
# sed -i 's/^MODE = "freeze"/MODE = "tag"/' pa_a2b.py && python3 pa_a2b.py     # 2) tag
# cd .. && git push --follow-tags && cd paper1
# sed -i 's/^MODE = "tag"/MODE = "run"/' pa_a2b.py
# ( time python3 -u pa_a2b.py ) 2>&1 | tee 09_PA/PA_A2b_quick_log.txt         # 3) QUICK (수치 미출력)
# sed -i 's/^QUICK = True/QUICK = False/' pa_a2b.py
# cd .. && git add paper1/pa_a2b.py paper1/09_PA/PA_A2b_*QUICK* paper1/09_PA/PA_A2b_quick_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-A2b: QUICK done; QUICK=False" && git tag -a PA-A2b-fullrun-2026-10-21 -m "pre-run" && git push --follow-tags
# cd paper1 && ( time python3 -u pa_a2b.py ) 2>&1 | tee 09_PA/PA_A2b_full_log.txt   # 4) 본 실행
# ```

# 주의할 점은 다음과 같습니다.

# - **미실행 코드입니다.** `Pair.mats`, `S.half`, 불리언 마스크로 CSR 행 선택하는 부분은 소스로 확인했지만 실행해 보지는 못했습니다. 오류가 나면 오류만 붙여 주세요.
# - **재현 점검이 두 개 있습니다.** s6 정의가 `decompose`의 `entry_new`와 맞지 않으면 중단합니다. 이것이 통과하면 마스크 방식이 S6와 같다는 확인이 됩니다.
# - **y5 정의의 첫 쌍 한계.** 2000년 이전 자료가 없어서 첫 쌍은 구조적으로 전원이 신규입니다. 결과표에서 첫 쌍의 y5는 해석하지 마세요.
# - 실행 시간은 A2c와 비슷하거나 조금 더 걸릴 것으로 예상합니다(마스크가 3개).

# ## PA-A2a(ID 대조)에 필요한 출력

# `F7_oa_author_ids.csv`가 있다는 것까지는 확인됐고 열 구조는 아직 모릅니다. 아래 출력을 붙여 주세요.

# ```bash
# cd ~/Research/Minnesota_Transport/analysis_data/final
# head -3 F7_oa_author_ids.csv; wc -l F7_oa_author_ids.csv
# head -2 F2_author_paper.csv
# python3 - <<'EOF'
# import pandas as pd
# f7 = pd.read_csv("F7_oa_author_ids.csv", low_memory=False); print(f7.dtypes); print(f7.describe(include="all").T.head(15))
# for c in f7.columns:
#     if "year" in c.lower(): print(c, f7[c].min(), f7[c].max())
# EOF
# ```

# F7이 이름→OpenAlex ID의 대응표인지, 논문-저자 단위 표인지, 어느 연도 논문까지 덮는지에 따라 마지막 쌍(2015–19→2020–25)에서 ID로 신규·지속을 다시 가르는 방식이 달라집니다.

In [ ]:
# # 논문 1 사후분석(PA) 진행 정리 (2026-10-21)

# ## 1. 지금까지 한 일

# ### 1-1. 재현성 정리 (PA-0): 완료
# - 해시 검증: S1~S8 manifest·meta의 파일 해시 152건 모두 일치(불일치 0, 파일없음 0).
# - S4 코드 3종(`s4_main.py`, `s4_6_rotation.py`, `s4_6b_floor_check.py`)을 IPython 히스토리에서 복구해 저장했습니다. 최종 실행본임은 증명하지 못한다고 CHANGELOG에 기록했습니다.
# - 잘못된 S4 태그 2개를 삭제하고 사유를 기록했습니다. 실제로는 S4 산출물이 없는 커밋을 가리켰습니다.
# - S2·S3·S4 산출물과 노트북 등 미추적 파일을 커밋했습니다(태그 `S4-results-recommit-2026-10-21`). 커밋일은 생성일이 아니라고 명시했습니다.
# - `MASTER_MANIFEST.csv`, `MASTER_TAGS.csv`를 생성했고, 모든 태그를 GitHub에 푸시했습니다. `git status`도 깨끗합니다.
# - `.gitignore`에 `oa_raw/`, `_backup_before_income_tv/`, `archive/*.tar.gz`를 추가했습니다.

# ### 1-2. PA-A1: N2 검정력 곡선 (완료)
# | 실행 | 결과 |
# |---|---|
# | run 1 | 검정력이 점프 크기와 무관하게 약 0.05였습니다. 주입 함수의 NaN 정규화 오류로 무효 처리했고, 삭제하지 않고 커밋했습니다. |
# | run 2 (동결 사양) | δ=0 기각률은 0.035/0.075/0.065(정상). 동결 판독은 all 강화, within 강화, fixed14 곡선만 보고입니다. |
# | 탐색 확장 | t*=2020에서 all·within은 δ=0.25부터 검정력 1.00, fixed14는 0.41~1.00입니다. t*=2012(fixed14·within)와 2008은 검정력이 거의 0이거나 불안정합니다(사각지대). 연속 꺾은선 BIC는 꺾임 2개가 최적이고 2020은 선택되지 않았습니다. |

# 주의할 점: δ=1의 추가 점프는 관측 초과분의 약 3배입니다. 근거 있는 서술은 "2020 부근에서 초과분 크기의 점프가 있었다면 탐지되었을 가능성이 높다"까지입니다.

# ### 1-3. PA-B2: 2020 변화의 표본 크기 아티팩트 분리 (완료)
# - 동결 판독(cap=3800, B=200)은 세 표본 모두 "유지"(회수율 1.0, J 비율 0.998~1.006)입니다.
# - 다만 판별력이 약한 검정입니다. cap이 2020년 논문(4,794편)을 약 20%만 줄이기 때문입니다. 말할 수 있는 것은 "표본 크기 차이만으로는 사라지지 않는다"까지입니다.
# - 기술 통계: 2017~19 대 2020~22 변화 상위 토픽은 COVID(71)가 1위이고 거의 전부 저널 내 변화입니다. 전기모터 토픽(45·88)은 저널 간 성분이 큽니다. 2019→2020 논문 증가의 30%가 T-ITS 한 곳입니다.

# ### 1-4. PA-A2c: 부트스트랩 보고 교정 (완료, B=2000)
# - 동결 판독: entry_minus_adapt의 basic 구간 하한이 0보다 큰 쌍은 4쌍 중 3쌍입니다(첫 쌍 제외) → "일부 유지".
# - 백분위 구간은 P0509→P1014(−0.034)와 P1014→P1519(−0.0027)에서 0을 포함합니다. 두 방식 모두 양수인 쌍은 마지막 쌍(P1519→P2025, 관측 0.113)뿐입니다.
# - excess_half는 3쌍에서 부트스트랩 구간이 관측값을 포함하지 못해, 이 교정으로 해결되지 않았습니다. 근거는 S6의 층화 순열 대조여야 합니다.
# - 분모가 4항(진입=신규+복귀)이라 S6 요약의 "신규 진입 몫 0.24→0.38"과 직접 비교할 수 없습니다.

# ### 1-5. PA-A2b: 신규 진입 정의 민감도 (실행 직전)
# - 계획 동결 완료(해시 `e6bf19450afb`, 태그 `PA-A2b-plan-2026-10-21`, 푸시됨).
# - QUICK 통과: 재현 점검 2개(s6 정의가 `entry_new`와 일치, `total_half`가 S6 산출물과 일치)가 모두 통과했습니다.
# - **본 실행(`QUICK=False`)은 아직 안 했습니다.**

# ## 2. 현재 논문이 주장할 수 있는 것

# | 쓸 수 있음 | 쓰면 안 됨 |
# |---|---|
# | 주제 구조 변화는 점프보다 연속적 재배치로 서술하는 편이 자료에 부합한다 | 체제 전환이 없음이 확인됐다 |
# | 2020 부근에는 COVID 토픽의 저널 내 증가와 ML·전기 토픽 성장이 겹친 가속이 있으며, 표본 크기 차이만으로는 사라지지 않는다 | 2020이 아티팩트가 아니다(확정) |
# | 진입>적응은 마지막 구간(2015–19→2020–25)에서 두 구간 방식 모두 유지되고, 중간 두 구간은 구간 방식에 따라 갈린다 | 신규 진입 주도가 모든 구간에서 확인됐다 |
# | 초과 진입분의 근거는 층화 순열 대조다 | 초과 진입분의 부트스트랩 구간을 근거로 쓰기 |

# 검정력은 "관측 2020 점프 방향의 점프"에 한정되고, 인과 해석은 하지 않으며, 해석은 "함께 움직인다"까지입니다.

# ## 3. 앞으로 할 일

# ### 지금 바로
# 1. **PA-A2b 본 실행**

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# sed -i 's/^QUICK = True/QUICK = False/' pa_a2b.py && grep -n '^MODE\|^QUICK' pa_a2b.py
# cd .. && git add paper1/pa_a2b.py paper1/09_PA/PA_A2b_*QUICK* paper1/09_PA/PA_A2b_quick_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-A2b: QUICK done; QUICK=False" && git tag -a PA-A2b-fullrun-2026-10-21 -m "pre-run" && git push --follow-tags
# cd paper1 && ( time python3 -u pa_a2b.py ) 2>&1 | tee 09_PA/PA_A2b_full_log.txt
# ```

#    로그를 붙여 주시면 정의(s6, y2, y5)별로 따로 읽겠습니다. 첫 쌍의 y5는 해석하지 않습니다.

# 2. **F7 구조 확인**(PA-A2a용): `head -3 F7_oa_author_ids.csv`, `wc -l`, `head -2 F2_author_paper.csv`를 붙여 주세요. `describe`와 연도 범위는 본 실행이 끝난 뒤에 돌립니다.

# 3. **PA-0 잔여 두 가지:** `oa_raw_20261013.tar.gz`를 외장 디스크로 복사하고 해시(`3bdc9b92…630f2`) 일치를 확인한 뒤 CHANGELOG에 한 줄 남기기, GitHub 저장소 Private 확인.

# ### 핵심 PA의 나머지
# | 항목 | 내용 | 필요한 것 |
# |---|---|---|
# | PA-A2a | ID 기반 대조(마지막 쌍, 2020+ 한정) | F7 구조 |
# | PA-B2b | 2020 점프의 저널 구성·COVID 토픽 분리(상위 증가 저널 제외, 토픽 71 제외) | `pa_b2b.py`는 작성됨, 미실행. `gen_boot`가 부분 배열에서 동작한다는 가정을 QUICK에서 확인해야 합니다. |
# | PA-C1 | 중국 소속 신규 저자 제외 분해(S6×S7 연결) | `Pair` 소스는 확보했으나 저자 국가 정보 경로가 필요합니다. |
# | PA-C2 | S7 국가 몫을 임베딩 보유 논문만으로 재계산 | `country_mass_long` |

# ### 약점 리프레이밍
# - B5: S4 안정성 Wilson 구간, 분위 컷 곡선, 방향 축만 재계산(기존 S4 파일만 사용, 난이도 낮음)
# - B4: F1+F3 이분 등급 재추정
# - B7: 결측 최악·최선 경계, 분류기 대입, 저널×연도 IPW
# - B3: 신생 핵심 집합(n_defs≥3), D1' (F6 생성 코드 위치와 `BASE` 정의 필요)
# - A3, B9, B8: S7 국가 미상 교정·균등 4구간·중국 제외 HHI, 소득군 내 상관, 미달 4개 도메인 부록 정렬
# - B6: ORCID 소표본과 수작업 50쌍(인력 필요, 병렬 진행)

# ### 마무리·투고
# - A4: 동결·열람 이력 부록 표. s5_lib 소스·N2 요약, B2 QUICK 출력, A1 탐색 결과 등 열람한 항목을 빠짐없이 기재합니다.
# - S9·S10: 후보 풀·추출 규칙 동결, 저널 선례 확인, 심사 공격-대응표, 생성형 AI 사용 선언.
# - 모든 PA 표·그림에 `[Post-hoc]` 표지, 기각된 PA 목록 부록, 영문 원고에 post-hoc 문구 포함.
# - 제목에서 "체제 전환"을 빼고 "신규 진입 주도 갱신 + 선두 국가 교체"를 중심에 두는 구성을 권합니다.

# ## 4. 유의사항
# - 규칙은 결과를 본 뒤에 바꾸지 않습니다. 바꾸면 v2 계획으로 분리하고 사유를 남깁니다.
# - QUICK 출력과 `_QUICK.csv`는 해석하지 않습니다. B2 때는 QUICK 출력에 수치가 포함돼 열람 사실을 기록했고, 이후 코드는 수치를 아예 출력하지 않게 했습니다.
# - v2는 "결과 미열람 상태의 사후 명세"이지 "사전등록"이 아닙니다.
# - S0~S8 통과·실패 판정은 PA 결과로 바꾸지 않습니다.

# ## 5. 지금 필요한 입력
# 1. PA-A2b 본 실행 로그
# 2. F7 구조 확인 출력(`head`, `wc`)
# 3. 수작업 인력 여부(B6의 50쌍, B8-2의 25편)
# 4. PA-B2b를 A2a보다 먼저 할지 여부

In [ ]:
# yjlee@u24:~/Research/Minnesota_Transport/paper1$ cd ~/Research/Minnesota_Transport/paper1
# sed -i 's/^QUICK = True/QUICK = False/' pa_a2b.py && grep -n '^MODE\|^QUICK' pa_a2b.py
# cd .. && git add paper1/pa_a2b.py paper1/09_PA/PA_A2b_*QUICK* paper1/09_PA/PA_A2b_quick_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-A2b: QUICK done; QUICK=False" && git tag -a PA-A2b-fullrun-2026-10-21 -m "pre-run" && git push --follow-tags
# cd paper1 && ( time python3 -u pa_a2b.py ) 2>&1 | tee 09_PA/PA_A2b_full_log.txt
# 9:MODE = "run"
# 10:QUICK = False
# [main aa9e4d6] PA-A2b: QUICK done; QUICK=False
#  3 files changed, 65 insertions(+), 2 deletions(-)
#  create mode 100644 paper1/09_PA/PA_A2b_quick_log.txt
#  create mode 100644 paper1/09_PA/PA_A2b_results_QUICK.csv
# Enumerating objects: 12, done.
# Counting objects: 100% (12/12), done.
# Delta compression using up to 24 threads
# Compressing objects: 100% (8/8), done.
# Writing objects: 100% (8/8), 4.12 KiB | 4.12 MiB/s, done.
# Total 8 (delta 3), reused 0 (delta 0), pack-reused 0
# remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
# To https://github.com/LEEYJ1021/transport-knowledge-paper1.git
#    e717345..aa9e4d6  main -> main
#  * [new tag]         PA-A2b-fullrun-2026-10-21 -> PA-A2b-fullrun-2026-10-21
# IPW 모델별 행 수: {'A': 91794, 'B': 81645} | 사용: A
# IPW 범위(주제 논문) 0.866~1.883 (S4는 0.86~1.94) | 결측 0편은 1로 대체
# F4에 없는 저자 0명 | F4 첫해 > F2 첫해인 저자 0명(0이어야 정상) | F4 마지막해 < F2 마지막해 0명
# 분석 논문 76,984 | 저자-논문 행 255,354 | 저자 93,763 | 가중치 ['equal', 'inv_n_authors', 'first_last', 'full_count'] | 위험군 변형 ['baseline'] (F2는 위험군 제외 상태)
#   P0004->P0509 완료
#   P0509->P1014 완료
#   P1014->P1519 완료
#   P1519->P2025 완료

# [Post-hoc] PA-A2b 결과
#          pair definition             stat  n_entrants     obs  boot_median  pct_lo  pct_hi  basic_lo  basic_hi    B
# P0004->P0509         s6        new_share        7253  0.2665       0.2536  0.2324  0.2770    0.2560    0.3006 2000
# P0004->P0509         s6      adapt_share        7253  0.2866       0.2990  0.2701  0.3261    0.2471    0.3032 2000
# P0004->P0509         s6  new_minus_adapt        7253 -0.0201      -0.0458 -0.0870  0.0006   -0.0408    0.0468 2000
# P0004->P0509         s6 rest_entry_share        7253  0.0604       0.0644  0.0531  0.0786    0.0423    0.0678 2000
# P0004->P0509         y2        new_share        3008  0.1324       0.1398  0.1240  0.1582    0.1066    0.1409 2000
# P0004->P0509         y2      adapt_share        3008  0.2866       0.2990  0.2701  0.3261    0.2471    0.3032 2000
# P0004->P0509         y2  new_minus_adapt        3008 -0.1542      -0.1594 -0.1944 -0.1198   -0.1886   -0.1140 2000
# P0004->P0509         y2 rest_entry_share        3008  0.2098       0.2001  0.1803  0.2207    0.1989    0.2393 2000
# P0004->P0509         y5        new_share        7868  0.2812       0.2698  0.2473  0.2946    0.2677    0.3150 2000
# P0004->P0509         y5      adapt_share        7868  0.2866       0.2990  0.2701  0.3261    0.2471    0.3032 2000
# P0004->P0509         y5  new_minus_adapt        7868 -0.0055      -0.0294 -0.0732  0.0172   -0.0281    0.0623 2000
# P0004->P0509         y5 rest_entry_share        7868  0.0000       0.0000  0.0000  0.0000    0.0000    0.0000 2000
# P0509->P1014         s6        new_share       12207  0.2985       0.2816  0.2629  0.3016    0.2954    0.3341 2000
# P0509->P1014         s6      adapt_share       12207  0.2690       0.2827  0.2603  0.3060    0.2320    0.2777 2000
# P0509->P1014         s6  new_minus_adapt       12207  0.0295      -0.0008 -0.0362  0.0334    0.0255    0.0952 2000
# P0509->P1014         s6 rest_entry_share       12207  0.0411       0.0448  0.0375  0.0547    0.0274    0.0446 2000
# P0509->P1014         y2        new_share        4285  0.1191       0.1234  0.1105  0.1373    0.1010    0.1278 2000
# P0509->P1014         y2      adapt_share        4285  0.2690       0.2827  0.2603  0.3060    0.2320    0.2777 2000
# P0509->P1014         y2  new_minus_adapt        4285 -0.1499      -0.1593 -0.1887 -0.1307   -0.1691   -0.1110 2000
# P0509->P1014         y2 rest_entry_share        4285  0.1994       0.1929  0.1777  0.2075    0.1913    0.2211 2000
# P0509->P1014         y5        new_share       13170  0.2961       0.2830  0.2642  0.3014    0.2909    0.3281 2000
# P0509->P1014         y5      adapt_share       13170  0.2690       0.2827  0.2603  0.3060    0.2320    0.2777 2000
# P0509->P1014         y5  new_minus_adapt       13170  0.0271       0.0009 -0.0343  0.0344    0.0199    0.0886 2000
# P0509->P1014         y5 rest_entry_share       13170  0.0000       0.0000  0.0000  0.0000    0.0000    0.0000 2000
# P1014->P1519         s6        new_share       19263  0.3282       0.3154  0.2996  0.3314    0.3249    0.3568 2000
# P1014->P1519         s6      adapt_share       19263  0.2880       0.2934  0.2748  0.3124    0.2635    0.3011 2000
# P1014->P1519         s6  new_minus_adapt       19263  0.0402       0.0218 -0.0076  0.0512    0.0292    0.0880 2000
# P1014->P1519         s6 rest_entry_share       19263  0.0263       0.0303  0.0261  0.0350    0.0177    0.0266 2000
# P1014->P1519         y2        new_share        6712  0.1280       0.1287  0.1175  0.1429    0.1131    0.1385 2000
# P1014->P1519         y2      adapt_share        6712  0.2880       0.2934  0.2748  0.3124    0.2635    0.3011 2000
# P1014->P1519         y2  new_minus_adapt        6712 -0.1600      -0.1646 -0.1893 -0.1378   -0.1821   -0.1306 2000
# P1014->P1519         y2 rest_entry_share        6712  0.2290       0.2207  0.2081  0.2333    0.2247    0.2499 2000
# P1014->P1519         y5        new_share       20492  0.3341       0.3216  0.3057  0.3379    0.3303    0.3625 2000
# P1014->P1519         y5      adapt_share       20492  0.2880       0.2934  0.2748  0.3124    0.2635    0.3011 2000
# P1014->P1519         y5  new_minus_adapt       20492  0.0461       0.0278 -0.0024  0.0582    0.0340    0.0947 2000
# P1014->P1519         y5 rest_entry_share       20492  0.0000       0.0000  0.0000  0.0000    0.0000    0.0000 2000
# P1519->P2025         s6        new_share       45937  0.3864       0.3780  0.3678  0.3874    0.3853    0.4050 2000
# P1519->P2025         s6      adapt_share       45937  0.2806       0.2818  0.2698  0.2946    0.2665    0.2913 2000
# P1519->P2025         s6  new_minus_adapt       45937  0.1058       0.0960  0.0762  0.1153    0.0962    0.1354 2000
# P1519->P2025         s6 rest_entry_share       45937  0.0166       0.0185  0.0164  0.0210    0.0121    0.0168 2000
# P1519->P2025         y2        new_share       13015  0.1236       0.1231  0.1161  0.1302    0.1171    0.1312 2000
# P1519->P2025         y2      adapt_share       13015  0.2806       0.2818  0.2698  0.2946    0.2665    0.2913 2000
# P1519->P2025         y2  new_minus_adapt       13015 -0.1569      -0.1589 -0.1748 -0.1431   -0.1708   -0.1391 2000
# P1519->P2025         y2 rest_entry_share       13015  0.2805       0.2745  0.2654  0.2830    0.2780    0.2956 2000
# P1519->P2025         y5        new_share       48206  0.3934       0.3853  0.3751  0.3950    0.3918    0.4116 2000
# P1519->P2025         y5      adapt_share       48206  0.2806       0.2818  0.2698  0.2946    0.2665    0.2913 2000
# P1519->P2025         y5  new_minus_adapt       48206  0.1128       0.1033  0.0837  0.1229    0.1027    0.1419 2000
# P1519->P2025         y5 rest_entry_share       48206  0.0000       0.0000  0.0000  0.0000    0.0000    0.0000 2000

# [동결 판독] 정의별 new_minus_adapt(관측>0 and basic 하한>0)인 쌍 수: {'s6': 3, 'y2': 0, 'y5': 3} / 쌍 수 4
# ※ 정의별로 따로 읽고 하나의 결론으로 합치지 않음. 진입 총량은 정의와 무관. S0~S8 판정 불변.

# real    3m57.728s
# user    4m0.170s
# sys     0m0.358s
# yjlee@u24:~/Research/Minnesota_Transport/paper1$

In [ ]:
# # PA-A2b 판독

# **동결 판독의 기계적 결과는 s6 = 3쌍, y2 = 0쌍, y5 = 3쌍입니다.** 정의별로 따로 읽는다고 못 박았으므로 하나로 합치지 않습니다.

# ## 1. 재현 점검 (정상)

# - **s6 정의**는 A2c와 같은 구조입니다. 관측 `new_share + rest_entry_share`가 A2c의 `entry_share`와 일치합니다(예: 마지막 쌍 0.3864 + 0.0166 = 0.4030. A2c의 진입 몫은 0.3934이므로 정확히 같지는 않습니다. 아래 "확인 필요" 참고).
# - **y5 정의**의 관측값은 A2c의 저자 모드 `entry_share`와 정확히 같습니다(0.2812, 0.2961, 0.3341, 0.3934). `rest_entry_share`가 전 쌍에서 0이므로, y5는 사실상 "진입 전체를 신규로 본 것"입니다.

# ## 2. 정의별 읽기

# | 정의 | 통과 쌍 | 읽기 |
# |---|---|---|
# | s6 | 3 (첫 쌍 제외) | S6 원본 정의에서 신규>적응은 뒤 세 쌍에서 basic 구간 기준 유지 |
# | y2 | 0 | 신규를 "첫 출판 후 2년 이내"로 좁히면 4쌍 모두 신규 몫이 적응 몫보다 작고, 차이가 −0.15~−0.16으로 일관되며 구간도 0을 크게 벗어남 |
# | y5 | 3 (첫 쌍 제외) | 첫 쌍은 구조적 한계로 해석 금지. 나머지는 s6과 같은 방향 |

# ## 3. 가장 중요한 발견: 결론이 신규의 정의에 의존합니다

# - **y2에서는 진입>적응이 정반대로 나옵니다.** 신규를 "t1 시작 후 2년 안에 첫 출판한 저자"로 좁히면 신규 몫은 약 0.12~0.13으로 모든 쌍에서 거의 일정하고 적응 몫(0.27~0.29)보다 작습니다. 이것은 반증이 아니라 정의 효과입니다. y2는 진입 중 신규로 분류되는 부분을 줄이고, 나머지는 `rest_entry_share`(0.20~0.28)로 넘어갑니다. 즉 **y2는 진입 저자를 줄인 것이 아니라 "신규"라는 이름표를 좁게 붙인 것**입니다. 진입 총량(신규+복귀 상당분)은 정의와 무관하므로, y2의 신규+rest를 합치면 s6과 비슷한 크기가 됩니다(마지막 쌍: 0.1236 + 0.2805 = 0.4041).
# - 따라서 본문에서 말할 수 있는 것은 "**진입 총량이 적응보다 크다**"입니다. "**신규 저자가 적응보다 크다**"는 신규 정의에 따라 뒤집힙니다. 이 둘을 구분해서 쓰세요. 지난번 정리의 "신규 진입 주도"라는 표현은 "진입 주도(신규·복귀 합)"로 정확히 바꾸는 것이 안전합니다.
# - **마지막 쌍은 정의 불문 방향이 같습니다**(진입 총량 기준). s6과 y5에서 new_minus_adapt가 +0.106, +0.113이고 백분위·basic 구간이 모두 0 위입니다. 가장 강한 근거는 이 쌍입니다.

# ## 4. 확인 필요한 점 (오류 가능성)

# s6 정의의 `new_share`(0.3864)와 A2c의 `entry_share`(0.3934)는 s6 정의가 신규만 세므로 작은 것이 정상이고, `rest_entry_share` 0.0166을 더하면 0.4030이 됩니다. 이 값이 A2c의 0.3934와 다릅니다. 가능한 원인은 두 가지입니다.

# - 분모가 다릅니다. A2c는 `entry` 전체를 한 항으로 계산하지만, 여기서 `rest_entry_share = half(entry − ent)`는 항별 절대값 합이라 `half(entry_new) + half(entry_ret)`와 `half(entry)`가 같지 않습니다(절대값 합은 부호가 다른 항을 더하면 상쇄되기 때문). 이 설명이 맞다면 오류가 아니라 **`half`가 비선형이라 몫이 가법적이지 않다**는 뜻입니다.
# - 이는 가설이고 확인하지 않았습니다. 가법성 위반이 확인되면 "new + rest = entry"는 쓰지 마세요.

# ## 5. 기록할 것

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-A2b 결과 해석 메모(B=2000): 동결 판독 new_minus_adapt(관측>0, basic 하한>0) 쌍 수 s6=3, y2=0, y5=3(첫 쌍 제외, 첫 쌍 y5는 2000년 이전 자료 부재로 전원 신규라 해석 금지). y2에서는 4쌍 모두 신규 몫<적응 몫(차이 -0.15~-0.16)이나 이는 신규 분류를 좁힌 정의 효과이며 진입 총량은 정의와 무관. 결론 문구는 '신규 저자 주도'가 아니라 '진입(신규+복귀) 주도'로 제한. 마지막 쌍은 s6·y5 모두 백분위·basic 구간이 0 위. half가 비선형이라 new+rest가 A2c의 entry_share와 정확히 일치하지 않음(미확인 가설) | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_A2b_results.csv paper1/09_PA/PA_A2b_full_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-A2b results" && git tag -a PA-A2b-results-2026-10-21 -m "PA-A2b outputs, seen" && git push --follow-tags
# ```

# ## 6. 본문에 쓸 때

# - 쓸 수 있음: "진입(신규+복귀)이 적응보다 큰 방향은 마지막 구간(2015–19→2020–25)에서 정의와 구간 방식에 관계없이 유지된다. 앞의 구간들은 구간 방식과 신규 정의에 따라 갈린다."
# - 쓰면 안 됨: "신규 저자가 갱신을 주도한다"(y2에서 반대), "모든 구간에서 진입 우세"(첫 쌍 불통과).
# - 이는 사후분석이고, S6의 사전 명시 결과와 판정은 바꾸지 않습니다.

# ## 다음 단계

# PA-A2a(ID 대조)에 필요한 F7 구조 출력이 아직 없습니다. 이제 CPU가 비었으니 아래를 모두 돌려 붙여 주세요.

# ```bash
# cd ~/Research/Minnesota_Transport/analysis_data/final
# head -3 F7_oa_author_ids.csv; wc -l F7_oa_author_ids.csv
# head -2 F2_author_paper.csv
# python3 - <<'EOF'
# import pandas as pd
# f7 = pd.read_csv("F7_oa_author_ids.csv", low_memory=False); print(f7.dtypes); print(f7.describe(include="all").T.head(15))
# for c in f7.columns:
#     if "year" in c.lower(): print(c, f7[c].min(), f7[c].max())
# EOF
# ```

# 원하시면 F7 확인 전에 PA-B2b(`pa_b2b.py`)를 먼저 진행해도 됩니다. 이 경우 QUICK에서 `gen_boot`가 부분 배열에서 동작하는지가 첫 관문입니다.

In [ ]:
# yjlee@u24:~/Research/Minnesota_Transport/paper1$ cd ~/Research/Minnesota_Transport/paper1
# sed -i 's/^QUICK = True/QUICK = False/' pa_b2b.py && grep -n '^MODE\|^QUICK' pa_b2b.py
# cd .. && git add paper1/pa_b2b.py paper1/09_PA/PA_B2b_*QUICK* paper1/09_PA/PA_B2b_quick_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-B2b: QUICK done; QUICK=False" && git tag -a PA-B2b-fullrun-2026-10-21 -m "pre-run" && git push --follow-tags
# cd paper1 && ( time python3 -u pa_b2b.py ) 2>&1 | tee 09_PA/PA_B2b_full_log.txt
# 9:MODE = "run"
# 10:QUICK = False
# [main 65cfc24] PA-B2b: QUICK done; QUICK=False
#  4 files changed, 39 insertions(+), 2 deletions(-)
#  create mode 100644 paper1/09_PA/PA_B2b_quick_log.txt
#  create mode 100644 paper1/09_PA/PA_B2b_results_QUICK.csv
#  create mode 100644 paper1/09_PA/PA_B2b_top_venues_QUICK.csv
# Enumerating objects: 13, done.
# Counting objects: 100% (13/13), done.
# Delta compression using up to 24 threads
# Compressing objects: 100% (8/8), done.
# Writing objects: 100% (9/9), 1.92 KiB | 1.92 MiB/s, done.
# Total 9 (delta 3), reused 0 (delta 0), pack-reused 0
# remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
# To https://github.com/LEEYJ1021/transport-knowledge-paper1.git
#    4d527b5..65cfc24  main -> main
#  * [new tag]         PA-B2b-fullrun-2026-10-21 -> PA-B2b-fullrun-2026-10-21
#   base 완료
#   drop_top1_venue 완료
#   drop_top3_venues 완료
#   drop_top5_venues 완료
#   drop_topic71 완료
#   drop_top5_venues_and_topic71 완료

# 제외 대상 상위 5개 저널: ['T-ITS', 'T-TE', 'AAP', 'TR-D', 'TRIP']

# [Post-hoc] PA-B2b 결과
#                       variant  sample  n_venues   K  obs_R         obs_breaks     J  J_ratio_vs_base J_class  boot_hit_2020pm1  break_kept
#                         base     all        34 149  0.759       [2012, 2020] 0.190            1.000   점프 유지             1.000        True
#                         base fixed14        34 149  0.775 [2008, 2015, 2020] 0.179            1.000   점프 유지             0.995        True
#                         base  within        34 149  0.703 [2009, 2015, 2020] 0.150            1.000   점프 유지             1.000        True
#              drop_top1_venue     all        33 149  0.727       [2012, 2020] 0.175            0.920   점프 유지             1.000        True
#              drop_top1_venue fixed14        33 149  0.664       [2012, 2019] 0.144            0.804   점프 유지             0.925        True
#              drop_top1_venue  within        33 149  0.662       [2015, 2020] 0.143            0.958   점프 유지             1.000        True
#             drop_top3_venues     all        31 149  0.690       [2012, 2020] 0.160            0.841   점프 유지             1.000        True
#             drop_top3_venues fixed14        31 149  0.661       [2012, 2019] 0.149            0.832   점프 유지             0.965        True
#             drop_top3_venues  within        31 149  0.609       [2012, 2019] 0.147            0.983   점프 유지             0.960        True
#             drop_top5_venues     all        29 149  0.680       [2012, 2020] 0.157            0.828   점프 유지             1.000        True
#             drop_top5_venues fixed14        29 149  0.649       [2012, 2020] 0.156            0.871   점프 유지             0.940        True
#             drop_top5_venues  within        29 149  0.597       [2012, 2019] 0.146            0.979   점프 유지             0.960        True
#                 drop_topic71     all        34 148  0.757       [2012, 2020] 0.177            0.931   점프 유지             0.995        True
#                 drop_topic71 fixed14        34 148  0.776 [2008, 2015, 2020] 0.168            0.940   점프 유지             1.000        True
#                 drop_topic71  within        34 148  0.696 [2009, 2015, 2020] 0.130            0.870   점프 유지             1.000        True
# drop_top5_venues_and_topic71     all        29 148  0.676       [2012, 2020] 0.141            0.739   부분 감소             0.995        True
# drop_top5_venues_and_topic71 fixed14        29 148  0.647       [2012, 2019] 0.135            0.758   부분 감소             0.635       False
# drop_top5_venues_and_topic71  within        29 148  0.589       [2012, 2019] 0.124            0.828   점프 유지             0.675       False

# ※ 동결 판독은 변화점 유지와 J 비율을 따로 읽음. 변형 간 R은 직접 비교 불가. S0~S8 판정 불변.

# real    0m15.547s
# user    0m18.112s
# sys     0m0.256s
# yjlee@u24:~/Research/Minnesota_Transport/paper1$

In [ ]:
# # PA-B2b 판독

# **동결 규칙의 기계적 결과는 "대부분 유지, 마지막 변형(상위 5개 저널+토픽 71 동시 제외)에서만 부분 감소 또는 유지 실패"입니다.** 변화점 유지와 J 비율은 따로 읽습니다.

# ## 1. 재현 점검 (정상)
# - `base`의 관측 변화점과 J(0.190/0.179/0.150)가 PA-A1·B2와 일치하고, 중단 없이 S5 R 재현을 통과했습니다.
# - 제외 대상 상위 5개 저널은 T-ITS, T-TE, AAP, TR-D, TRIP입니다. PA-B2의 증가 상위 목록과 일치합니다.

# ## 2. 변형별 읽기

# | 변형 | J 비율 | 변화점 유지(2020±1, 회수율≥0.80) |
# |---|---|---|
# | 상위 1·3·5 저널 제외 | 0.80~0.98 (전부 "점프 유지") | 9칸 모두 유지 |
# | 토픽 71만 제외 | 0.87~0.94 (유지) | 3칸 모두 유지 |
# | 상위 5 저널+토픽 71 | all 0.74, fixed14 0.76 (부분 감소), within 0.83 (유지) | all만 유지, fixed14·within 실패(회수율 0.635, 0.675) |

# ## 3. 해석

# 1. **저널 한 곳(T-ITS)이나 상위 5개 저널만으로는 2020 점프가 설명되지 않습니다.** 5개를 빼도 J가 0.83~0.98 수준으로 남고, 2020±1 변화점도 회수율 0.94 이상입니다. PA-B2가 못 다룬 저널 구성 효과에 대한 답으로는 이 부분이 가장 분명합니다.
# 2. **COVID 토픽 71 하나만 빼도 점프가 거의 남습니다.** J 감소는 최대 13%입니다. 앞서 기술 통계에서 71이 변화량 1위였던 것과 겹쳐 보면, 점프가 71 하나로 만들어진 것은 아닙니다.
# 3. **두 성분을 함께 빼면 약해집니다.** J 비율이 0.74~0.83으로 내려가고, fixed14와 within에서는 변화점이 2020±1에서 안정적으로 회수되지 않습니다(0.635, 0.675). 단 `obs_breaks`를 보면 해당 칸의 변화점은 2019로 옮겨 가 있어서, 사라졌다기보다 한 해 앞으로 이동한 것일 수 있습니다. 규칙의 허용 범위가 ±1이라 2019는 "관측 변화점이 2020±1에 있음"을 만족하는데도 실패가 나온 것은 부트스트랩 회수율이 0.80 미만이기 때문입니다. 이 설명은 표만 보고 추정한 것이고 확인하지 않았습니다.
# 4. 즉 "2020 부근 가속은 저널 구성 하나나 COVID 토픽 하나로 환원되지 않고, 둘을 동시에 빼면 약해지고 위치가 불안정해진다"가 자료와 맞는 서술입니다.

# ## 4. 주의할 점
# - 변형이 늘면서 K와 저널 수가 바뀌어 **벌점이 S5 값으로 고정**돼 있습니다. 제외 후 R은 비교 불가이고, 변화점 이동이 탐지기 설정 효과일 수 있습니다.
# - **토픽 71만 뺀 것은 COVID 관련 토픽 전체를 뺀 것이 아닙니다.** 123(공유 모빌리티) 등 인접 토픽이 남아 있습니다. 계획에서 카드를 읽기 전에는 다른 토픽을 지정하지 않기로 했으므로, 확장은 새 계획이 필요합니다.
# - 부트스트랩 B=200이라 회수율 표준오차가 약 0.03입니다. 0.635와 0.675는 0.80에서 충분히 떨어져 있지만, 0.92~0.94는 규칙 경계에서 멀지 않습니다.
# - 측정 아티팩트(임베딩 보유율)는 이 분석에 없습니다. 2020이 "아티팩트가 아니다"라고 확정할 수 없다는 점은 그대로입니다.
# - 사후분석이고, S5의 체제 서사 철회 판정은 바뀌지 않습니다.

# ## 5. 기록

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B2b 결과 해석 메모(B=200): 상위 1·3·5 저널 제외와 토픽 71 제외에서 J 비율 0.80~0.98, 2020±1 변화점 회수율 >=0.925(전 칸 유지). 상위 5 저널+토픽 71 동시 제외에서 J 비율 0.74(all)·0.76(fixed14, 부분 감소)·0.83(within), fixed14·within은 회수율 0.635/0.675로 변화점 유지 실패(변화점이 2019로 이동). 벌점 S5 값 고정이라 변형 간 R 비교 불가, 토픽 71 외 COVID 인접 토픽은 미제외, 측정 아티팩트 미다룸. 문구는 '저널 구성 하나나 COVID 토픽 하나로 환원되지 않으며 둘을 함께 빼면 약해진다'로 제한 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_B2b_*.csv paper1/09_PA/PA_B2b_full_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-B2b results" && git tag -a PA-B2b-results-2026-10-21 -m "PA-B2b outputs, seen" && git push --follow-tags
# ```

# ## 6. 현재 논문이 쓸 수 있는 2020 서술
# "2020 부근 변화는 연도별 논문 수 차이(PA-B2), 증가 상위 저널 하나 또는 다섯 곳의 제외(PA-B2b), COVID 토픽 하나의 제외 중 어느 하나만으로는 사라지지 않는다. 상위 저널과 COVID 토픽을 동시에 제외하면 점프가 약해지고 변화점 위치가 불안정해진다. 연속 꺾은선 모형은 2020을 꺾임으로 선택하지 않았다(탐색적)." 단일 체제 전환이라거나 아티팩트가 아니라고 쓰면 안 됩니다.

# ## 다음 단계
# PA-A2a(ID 대조)에 필요한 F7 구조 출력이 아직 없습니다. 아래를 돌려 붙여 주세요.

# ```bash
# cd ~/Research/Minnesota_Transport/analysis_data/final
# head -3 F7_oa_author_ids.csv; wc -l F7_oa_author_ids.csv
# head -2 F2_author_paper.csv
# python3 - <<'EOF'
# import pandas as pd
# f7 = pd.read_csv("F7_oa_author_ids.csv", low_memory=False); print(f7.dtypes); print(f7.describe(include="all").T.head(15))
# for c in f7.columns:
#     if "year" in c.lower(): print(c, f7[c].min(), f7[c].max())
# EOF
# ```

In [ ]:
# yjlee@u24:~/Research/Minnesota_Transport/paper1$ cd ~/Research/Minnesota_Transport/paper1
# python3 pa_b5.py                                                          # freeze
# sed -i 's/^MODE = "freeze"/MODE = "tag"/' pa_b5.py && python3 pa_b5.py     # tag
# cd .. && git push --follow-tags && cd paper1
# sed -i 's/^MODE = "tag"/MODE = "run"/' pa_b5.py
# ( time python3 -u pa_b5.py ) 2>&1 | tee 09_PA/PA_B5_full_log.txt
# PA-B5 계획 해시(예정) 85a7d60d3a9a
# 저장: PA_B5_PLAN.json
# add 0
# commit 0 [main c1fdbd7] PA-B5 plan freeze (before execution)
#  3 files changed, 124 insertions(+)
#  create mode 100644 paper1/09_PA/PA_B5_PLAN.json
#  create mode 100644 paper1/pa_b5.py

# tag 0
# ※ 이어서 git push --follow-tags
# Enumerating objects: 12, done.
# Counting objects: 100% (12/12), done.
# Delta compression using up to 24 threads
# Compressing objects: 100% (8/8), done.
# Writing objects: 100% (8/8), 5.07 KiB | 2.54 MiB/s, done.
# Total 8 (delta 4), reused 0 (delta 0), pack-reused 0
# remote: Resolving deltas: 100% (4/4), completed with 3 local objects.
# To https://github.com/LEEYJ1021/transport-knowledge-paper1.git
#    a11a739..c1fdbd7  main -> main
#  * [new tag]         PA-B5-plan-2026-10-21 -> PA-B5-plan-2026-10-21
# [재현 OK] 원본 컷에서 S4 안정성·라벨 일치

# [Post-hoc] PA-B5 컷 민감도
#      q  n_big  n_big_A  stab_full  stab_A  lost_big  sign_flip  kept  sign_agree  lost_but_sign_same
# 0.600     60       49      0.767   0.816        14          0    46       0.967                  12
# 0.667     50       42      0.700   0.690        15          0    35       0.980                  14
# 0.700     45       38      0.667   0.658        15          0    30       0.978                  14
# 0.750     38       33      0.658   0.636        13          0    25       1.000                  13

# [참고] 원본 35/50의 Wilson 95%: [0.562, 0.809] (토픽을 표본으로 볼 경우의 참고치, 결정적 비율임)

# [Post-hoc] 원본 컷 불일치 토픽
#   topic grade   delta  within        lab_delta   lab_within  lost_big  sign_flip
#     14     A -0.0109 -0.0045    decline_highQ small_change      True      False
#     20     A  0.0089  0.0050     growth_highQ small_change      True      False
#     26     A -0.0079 -0.0025 decline_notHighQ small_change      True      False
#     29     A  0.0102  0.0060  growth_notHighQ small_change      True      False
#     60     B -0.0092 -0.0039 decline_notHighQ small_change      True      False
#     68     A  0.0082  0.0044     growth_highQ small_change      True      False
#     84     A -0.0115 -0.0053 decline_notHighQ small_change      True      False
#     92     A -0.0108 -0.0017 decline_notHighQ small_change      True      False
#     93     A -0.0097 -0.0047 decline_notHighQ small_change      True      False
#    101     A -0.0089 -0.0061 decline_notHighQ small_change      True      False
#    102     B -0.0094  0.0012 decline_notHighQ small_change      True      False
#    106     A -0.0098 -0.0013 decline_notHighQ small_change      True      False
#    107     A -0.0132 -0.0055 decline_notHighQ small_change      True      False
#    120     A  0.0093  0.0046     growth_highQ small_change      True      False
#    148     A  0.0093  0.0022     growth_highQ small_change      True      False

# ※ 불일치 원천은 within 기준 상위 1/3 재선정에 따른 탈락. quality_high 공유라 품질 축 효과 아님. S0~S8 판정 불변.

# real    0m0.587s
# user    0m2.313s
# sys     0m0.077s
# yjlee@u24:~/Research/Minnesota_Transport/paper1$

In [ ]:
# # PA-B5 판독

# **재현 점검이 통과했습니다.** 원본 컷에서 안정성 0.700/0.690이 S4_meta와 일치하고, 저장된 `quadrant`·`quadrant_within` 열과 재계산 라벨이 150개 전부 같았습니다. 이는 복구한 `s4_main.py`의 라벨 규칙이 산출물과 맞는다는 추가 증거입니다(최종 실행본임을 증명하지는 못한다는 기록은 그대로 둡니다).

# ## 1. 결과

# | 컷 q | big 수 | 안정성(전체) | 안정성(A) | big 탈락 | 부호 반전 | 부호 일치율 |
# |---|---|---|---|---|---|---|
# | 0.600 | 60 | 0.767 | 0.816 | 14 | 0 | 0.967 |
# | 0.667(원본) | 50 | 0.700 | 0.690 | 15 | 0 | 0.980 |
# | 0.700 | 45 | 0.667 | 0.658 | 15 | 0 | 0.978 |
# | 0.750 | 38 | 0.658 | 0.636 | 13 | 0 | 1.000 |

# ## 2. 읽기

# 1. **부호 반전은 모든 컷에서 0입니다.** 불일치는 전부 "within 기준 상위 1/3 재선정에서 big이 탈락"한 경우입니다. 원본 컷의 15개 토픽은 모두 `lab_within = small_change`이고, 이전에 세운 예상(탈락만 있고 품질 축 효과 없음)과 맞습니다.
# 2. **안정성은 컷에 따라 0.66~0.77 범위입니다.** 0.60에서만 0.767로 올라가고 나머지는 0.66~0.70입니다. 0.70 기준선을 안정적으로 넘는 컷은 0.60뿐이라, "경계 통과"가 아니라 **"0.70 전후에서 컷에 따라 움직인다"**가 정확합니다. 컷이 높아질수록 낮아지는 단조 경향은 있으나 4개 점이라 경향이라고만 쓰세요.
# 3. **탈락한 15개는 대부분 약한 변화가 아니라 "저널 내 성분이 작은" 토픽입니다.** 불일치 토픽의 `within`은 `delta`의 대략 절반 이하이고(예: 92번 −0.0108 대 −0.0017), 부호는 유지됩니다. 즉 총변화는 컸지만 저널 내 성분은 상위 1/3에 못 든 것입니다. 이 해석은 표의 값을 읽은 것이고, 저널 간 성분이 큰 토픽이라는 점은 S4의 `between_share`와 따로 확인하지 않았습니다.
# 4. **예외 하나:** 102번은 `within`이 +0.0012로 `delta`(−0.0094)와 부호가 달라서 탈락이면서 부호도 다릅니다. 부호 반전 열이 0인 것은 "big이면서 반전"만 센 정의 때문이고, 부호 일치율 0.980(=49/50)이 이 1건을 반영합니다. 원본 컷의 `lost_but_sign_same`이 14인 것과 맞습니다.

# ## 3. 쓸 수 있는 문구
# "4분면 라벨은 총변화 기준 big 50개 중 35개가 저널 내 성분 기준에서도 유지됐다(0.700). 나머지 15개는 부호가 바뀐 것이 아니라 저널 내 성분이 상위 1/3에 들지 못해 탈락한 것이고(부호 일치 49/50), 컷을 0.60~0.75로 바꾸면 안정성은 0.66~0.77이다." "품질 축이 불안정하다"는 쓰지 마세요(두 라벨이 같은 `quality_high`를 공유). Wilson 구간 [0.562, 0.809]는 "토픽을 표본으로 볼 경우의 참고치"로만 병기하세요.

# ## 4. 기록

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B5 결과 해석 메모(결정적 계산): 재현 통과(안정성 0.700/0.690, 저장된 라벨 150개 일치). 컷 0.60/0.667/0.70/0.75에서 안정성 0.767/0.700/0.667/0.658, 부호 반전 0, 불일치는 전부 within 기준 big 탈락(원본 15개). 부호 일치 49/50(102번만 부호 다름). quality_high 공유라 품질 축 불안정이라 쓰지 않음. Wilson [0.562,0.809]는 토픽을 표본으로 볼 때의 참고치 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_B5_*.csv paper1/09_PA/PA_B5_full_log.txt paper1/09_PA/PA_CHANGELOG.md paper1/pa_b5.py
# git commit -m "PA-B5 results" && git tag -a PA-B5-results-2026-10-21 -m "PA-B5 outputs, seen" && git push --follow-tags
# ```

# ## 5. 다음 단계
# PA-A2a(ID 대조)는 F7 구조 출력이 계속 필요합니다. 아래를 붙여 주세요.

# ```bash
# cd ~/Research/Minnesota_Transport/analysis_data/final
# head -3 F7_oa_author_ids.csv; wc -l F7_oa_author_ids.csv
# head -2 F2_author_paper.csv
# python3 - <<'EOF'
# import pandas as pd
# f7 = pd.read_csv("F7_oa_author_ids.csv", low_memory=False); print(f7.dtypes); print(f7.describe(include="all").T.head(15))
# for c in f7.columns:
#     if "year" in c.lower(): print(c, f7[c].min(), f7[c].max())
# EOF
# ```

# F7 없이 진행 가능한 다음 후보는 PA-B4(F1+F3 이분 등급 재추정)와 PA-B7(결측 경계)입니다. 원하시면 해당 입력 구조를 먼저 확인하는 출력을 드리겠습니다.

In [ ]:
# PA-B5 결과와 해석 메모가 커밋·푸시됐습니다(`7d97a10`, 태그 `PA-B5-results-2026-10-21`). 지금까지 끝난 PA는 A1, A2b, A2c, B2, B2b, B5입니다.

# ## 다음에 필요한 입력

# 다음은 둘 중 하나를 진행할 수 있습니다. 둘 다 입력 구조를 먼저 확인해야 코드를 쓸 수 있습니다.

# **1. PA-A2a (ID 대조, 핵심 PA의 마지막 조각)**: F7 구조 출력이 필요합니다. 마지막 쌍(2015–19→2020–25)에서 이름 대신 OpenAlex ID로 신규·지속을 다시 가르려면, F7이 이름→ID 대응표인지 논문-저자 단위 표인지, 어느 연도까지 덮는지를 알아야 합니다.

# ```bash
# cd ~/Research/Minnesota_Transport/analysis_data/final
# head -3 F7_oa_author_ids.csv; wc -l F7_oa_author_ids.csv
# head -2 F2_author_paper.csv
# python3 - <<'EOF'
# import pandas as pd
# f7 = pd.read_csv("F7_oa_author_ids.csv", low_memory=False); print(f7.dtypes); print(f7.describe(include="all").T.head(15))
# for c in f7.columns:
#     if "year" in c.lower(): print(c, f7[c].min(), f7[c].max())
# EOF
# ```

# **2. PA-B4 (F1+F3 이분 등급 재추정)**: 약점 리프레이밍이고 S4·S5·S6 핵심 결과를 "F1·F3 모두 통과 토픽(145개)"으로 다시 추정합니다. 등급 정의와 F1·F3 플래그가 어디에 있는지 확인이 필요합니다.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1/03_S3
# head -3 L5_topic_flags.csv; head -3 L7_topic_grade.csv
# grep -n -i "F1\|F3\|grade\|flag" S3_final_summary.md | head -30
# ```

# 핵심 PA 쪽을 닫는 것이 우선이라 A2a를 권합니다. 어느 쪽 출력이든 먼저 오는 것부터 진행하겠습니다.

# ## 병행해서 할 수 있는 일

# - PA-0 잔여: `oa_raw_20261013.tar.gz` 외부 복사와 해시 확인(`3bdc9b92…630f2`), GitHub 저장소 Private 확인
# - 논문 문구 정리: 지금까지의 결과(A1, B2, B2b, A2b, A2c, B5)를 반영한 "쓸 수 있음 / 쓰면 안 됨" 표를 한 장으로 합쳐 드릴 수 있습니다. 원하시면 말씀해 주세요.

In [ ]:
# yjlee@u24:~/Research/Minnesota_Transport$ cd ~/Research/Minnesota_Transport/analysis_data/final
# head -3 F7_oa_author_ids.csv; wc -l F7_oa_author_ids.csv
# head -2 F2_author_paper.csv
# python3 - <<'EOF'
# import pandas as pd
# f7 = pd.read_csv("F7_oa_author_ids.csv", low_memory=False); print(f7.dtypes); print(f7.describe(include="all").T.head(15))
# for c in f7.columns:
#     if "year" in c.lower(): print(c, f7[c].min(), f7[c].max())
# EOF
# ﻿pid,rank,name_norm,oa_id
# 7349,0,ahmad ali,https://openalex.org/A5087172650
# 7349,1,inam ullah,https://openalex.org/A5042119018
# 141837 F7_oa_author_ids.csv
# ﻿pid,name_norm,author_pos,year,venue_short,n_authors,topic150,w
# 7349,ahmad ali,1,2025,T-ITS,6,69,0.16666666666666666
# pid          int64
# rank         int64
# name_norm      str
# oa_id          str
# dtype: object
#               count unique                               top freq          mean  ...     min      25%      50%      75%      max
# pid        141836.0    NaN                               NaN  NaN  29555.349777  ...  7349.0  19337.0  29753.0  39968.0  50547.0
# rank       141836.0    NaN                               NaN  NaN      1.893602  ...     0.0      0.0      2.0      3.0     30.0
# name_norm    141778  63620               mohamed a abdel aty  145           NaN  ...     NaN      NaN      NaN      NaN      NaN
# oa_id        141836  71702  https://openalex.org/A5040847417  145           NaN  ...     NaN      NaN      NaN      NaN      NaN

# [4 rows x 11 columns]
# yjlee@u24:~/Research/Minnesota_Transport/analysis_data/final$ cd ~/Research/Minnesota_Transport/paper1/03_S3
# head -3 L5_topic_flags.csv; head -3 L7_topic_grade.csv
# grep -n -i "F1\|F3\|grade\|flag" S3_final_summary.md | head -30
# ﻿topic,n_papers,top_title,top_title_share,unique_ratio,flag_small_n,flag_repeated_title,graph_nodes,graph_edges,communities,modularity,flag_composite_auto,manual_decision,flag_manual_nonresearch,F3_fail
# 0,411,"feasibility of truck-only toll lane network in atlanta, geor",0.005,0.995,False,False,30,134,3,0.436,True,,False,True
# 1,586,grid integration of electric vehicles within electricity and,0.002,1.0,False,False,30,311,4,0.07,False,,False,False
# ﻿topic,n_eval,share_ge,n_papers,jaccard_median,npmi_mean,manual_decision,modularity,F1,F2,F3,F4,n_fail,grade,held
# 0,5.0,1.0,411,0.667,-0.4155,,0.436,False,True,True,False,2,C,False
# 1,3.0,1.0,586,0.667,-0.0599,,0.07,False,False,False,False,0,A,False
# 4:- 기준별 실패 수: {'F1': 2, 'F2': 37, 'F3': 2, 'F4': 0}. F2는 상대 기준(하위 25%)이라 구조상 일부 실패, F4는 전부 통과(변별력 없음). F2 제외 시 A 145개(0.973). 통과는 약한 증거로 보고.
# yjlee@u24:~/Research/Minnesota_Transport/paper1/03_S3$

In [ ]:
# # F7 구조 확인 결과와 PA-A2a 설계에 미치는 영향

# ## 1. 확인된 것

# - **F7은 논문-저자 단위 표입니다.** 열은 `pid, rank, name_norm, oa_id`이고 141,836행입니다. 한 논문 안에서 `rank`(0부터)가 저자 순서이며, `pid`는 7349~50547 범위입니다.
# - **F7은 일부 논문만 덮습니다.** F2의 `pid`는 7349 근처에서 시작하는데, 분석 논문은 76,984편이고 F7의 `pid` 범위(7349~50547)와 행 수(14만)로 볼 때 2020년 이후 논문 중심으로 보입니다. 연도 열이 없어서 이 추정은 확인하지 못했습니다. 반드시 F2와 `pid`로 조인해 연도별 커버리지를 봐야 합니다.
# - **이름과 ID가 일대일이 아닙니다.** 고유 `name_norm` 63,620개에 고유 `oa_id` 71,702개입니다. 이름 하나가 여러 ID로 갈라지는 경우(동명이인)와, `name_norm`이 결측인 58행이 있습니다. 이것이 PA-A2a가 보려는 이름 기반 식별의 약점입니다.
# - **`rank`와 F2의 `author_pos`는 다를 수 있습니다.** F2는 `author_pos`가 1부터이고 F7의 `rank`는 0부터라 기준이 다릅니다. 조인 키는 `pid + name_norm`이 안전하고, `pid + rank+1 = author_pos`는 대조용으로만 쓰세요(확인 안 됨).

# ## 2. PA-A2a 설계에 필요한 것

# F7에는 연도가 없어서, 이름 기반으로 만든 신규·지속 판별을 ID 기반으로 바꾸려면 다음이 필요합니다.

# 1. F7을 F2와 `pid`·`name_norm`으로 조인해 연도별 F7 커버리지(분석 행 중 ID가 붙은 비율)를 확인합니다.
# 2. 마지막 쌍의 저자 단위를 `name_norm`에서 `oa_id`로 바꾸려면 **이전 기간(2015–19) 논문에도 ID가 있어야** 지속 저자를 ID로 가를 수 있습니다. F7이 2020+ 논문만 덮는다면 ID 기반 지속/신규 판별은 불가능합니다.
# 3. 따라서 PA-A2a의 실현 가능성은 커버리지에 달려 있습니다. 아래 점검 결과를 보고 설계하겠습니다.

# ```bash
# cd ~/Research/Minnesota_Transport/analysis_data/final
# python3 - <<'EOF'
# import pandas as pd
# f7 = pd.read_csv("F7_oa_author_ids.csv", encoding="utf-8-sig", low_memory=False)
# f2 = pd.read_csv("F2_author_paper.csv", encoding="utf-8-sig", low_memory=False)
# print("F2 행", len(f2), "| F2 pid 고유", f2.pid.nunique(), "| F7 pid 고유", f7.pid.nunique())
# pid_year = f2.drop_duplicates("pid").set_index("pid").year
# f7y = f7.drop_duplicates("pid").pid.map(pid_year)
# print("\nF7 pid의 연도 분포(F2 기준):"); print(f7y.describe().round(1).to_string())
# print(f7y.value_counts().sort_index().to_string())
# tot = f2.drop_duplicates("pid").groupby("year").size(); cov = f7y.value_counts().sort_index()
# print("\n연도별 F7 커버 논문 비율:"); print((cov / tot).dropna().round(3).to_string())
# m = f2.merge(f7, on=["pid", "name_norm"], how="left", indicator=True)
# print("\nF2 저자-논문 행 중 ID 부착 비율:", round(float((m._merge == "both").mean()), 3))
# print("연도별:", m.groupby(m.year >= 2020)["_merge"].apply(lambda s: round(float((s == "both").mean()), 3)).to_dict())
# g = f7.dropna(subset=["name_norm"]).groupby("name_norm").oa_id.nunique()
# print("\n이름당 고유 ID 수 분포:", g.value_counts().sort_index().head(6).to_dict(), "| 2개 이상인 이름 수", int((g >= 2).sum()))
# EOF
# ```

# ## 3. PA-B4용 확인 (등급 정의)

# `L7_topic_grade.csv`의 열과 요약 메모로 PA-B4의 정의를 확인했습니다.

# - 등급은 `n_fail`(F1~F4 실패 수)로 정해지고, 기준별 실패 수는 F1 2, F2 37, F3 2, F4 0입니다. F4는 전부 통과라 변별력이 없고, F2는 하위 25%라는 상대 기준이라 구조상 일부가 실패합니다.
# - 요약 메모에 "F2 제외 시 A 145개(0.973)"가 있습니다. 즉 F1+F3만 쓴 이분 등급이 이미 근사적으로 계산돼 있으며, 지침서의 "F1·F3 통과 145개"와 일치합니다. 다만 이것은 S3 요약의 기존 수치라서, PA-B4에서는 F1·F3 열로 재계산해 145개가 재현되는지를 중단 조건으로 둬야 합니다.
# - `F3_fail`은 `L5_topic_flags.csv`에, F1·F3은 `L7_topic_grade.csv`에 모두 있어서 두 파일만으로 이분 등급을 만들 수 있습니다.
# - PA-B4의 실제 작업은 S4·S5·S6 핵심 결과를 이 145개 토픽만으로 재추정하는 것입니다. 이 재추정은 토픽 부분집합으로 S5 탐지기와 S6 분해를 다시 돌리는 것이므로, `s5_lib.sample_P`의 토픽 마스크(`tmask`)와 S6의 `topic="A"` 설정을 활용할 수 있습니다. 다만 S6의 `topic="A"`는 A등급(112개)이며 F1+F3 통과 145개와 다르므로, 새 마스크를 만들어야 합니다.

# ## 4. 다음 한 걸음

# 위 F7 커버리지 점검 출력을 붙여 주세요. 결과에 따라 이렇게 갈립니다.

# - **F7이 2015–19 논문도 충분히 덮으면:** `oa_id`로 저자 축을 바꾼 `Pair`를 만들어 마지막 쌍의 진입·적응 몫을 이름 기반과 비교합니다(차이 ±0.03 이내인지).
# - **F7이 2020+만 덮으면:** 지속 저자 판별이 불가능하므로 PA-A2a는 "이름 기반 저자가 ID 기준으로 얼마나 분할·병합되는가"를 보는 진단으로 한정하고, 이 한계를 명시합니다.

# PA-B4는 위 설계대로 진행할 수 있으니, 원하시면 A2a 출력이 오는 동안 `pa_b4.py` 동결 코드를 먼저 드리겠습니다.

In [ ]:
# yjlee@u24:~/Research/Minnesota_Transport/paper1/03_S3$ cd ~/Research/Minnesota_Transport/analysis_data/final
# python3 - <<'EOF'
# import pandas as pd
# f7 = pd.read_csv("F7_oa_author_ids.csv", encoding="utf-8-sig", low_memory=False)
# f2 = pd.read_csv("F2_author_paper.csv", encoding="utf-8-sig", low_memory=False)
# print("F2 행", len(f2), "| F2 pid 고유", f2.pid.nunique(), "| F7 pid 고유", f7.pid.nunique())
# pid_year = f2.drop_duplicates("pid").set_index("pid").year
# f7y = f7.drop_duplicates("pid").pid.map(pid_year)
# print("\nF7 pid의 연도 분포(F2 기준):"); print(f7y.describe().round(1).to_string())
# print(f7y.value_counts().sort_index().to_string())
# tot = f2.drop_duplicates("pid").groupby("year").size(); cov = f7y.value_counts().sort_index()
# print("\n연도별 F7 커버 논문 비율:"); print((cov / tot).dropna().round(3).to_string())
# m = f2.merge(f7, on=["pid", "name_norm"], how="left", indicator=True)
# print("\nF2 저자-논문 행 중 ID 부착 비율:", round(float((m._merge == "both").mean()), 3))
# print("연도별:", m.groupby(m.year >= 2020)["_merge"].apply(lambda s: round(float((s == "both").mean()), 3)).to_dict())
# g = f7.dropna(subset=["name_norm"]).groupby("name_norm").oa_id.nunique()
# print("\n이름당 고유 ID 수 분포:", g.value_counts().sort_index().head(6).to_dict(), "| 2개 이상인 이름 수", int((g >= 2).sum()))
# EOF
# F2 행 282040 | F2 pid 고유 87525 | F7 pid 고유 35594

# F7 pid의 연도 분포(F2 기준):
# count    30422.0
# mean      2022.6
# std          1.7
# min       2020.0
# 25%       2021.0
# 50%       2023.0
# 75%       2024.0
# max       2025.0
# pid
# 2020.0    4296
# 2021.0    4904
# 2022.0    5040
# 2023.0    5521
# 2024.0    5682
# 2025.0    4979

# 연도별 F7 커버 논문 비율:
# pid
# 2020.0    0.903
# 2021.0    0.910
# 2022.0    0.901
# 2023.0    0.891
# 2024.0    0.799
# 2025.0    0.695

# F2 저자-논문 행 중 ID 부착 비율: 0.384
# 연도별: {False: 0.0, True: 0.809}

# 이름당 고유 ID 수 분포: {1: 59871, 2: 2452, 3: 563, 4: 249, 5: 155, 6: 75} | 2개 이상인 이름 수 3749
# yjlee@u24:~/Research/Minnesota_Transport/analysis_data/final$

In [ ]:
# # F7 커버리지 결과: PA-A2a는 "ID 기반 재판별"이 불가능하고 "진단"으로 한정됩니다

# ## 1. 확인된 것

# - **F7은 2020~2025년 논문만 덮습니다.** F7의 `pid`는 연도 분포가 2020~2025로 끝나고, 2020년 이전은 0건입니다. 2020년 이후 F2 저자-논문 행의 81%에 ID가 붙지만 2020년 이전은 0%입니다.
# - **연도별 커버리지는 하락합니다.** 논문 기준 0.90(2020~22), 0.89(2023), 0.80(2024), 0.70(2025). 2024~25가 낮은 것은 자료 수집 방식 때문일 가능성이 있으나 이유는 확인하지 않았습니다.
# - **이름당 ID 분할:** 이름 63,620개 중 3,749개(5.9%)가 2개 이상의 ID로 갈립니다(2개 2,452, 3개 563, 4개 249 등). 이름 하나가 여러 사람을 섞는 동명이인 위험의 직접 지표입니다. 반대로 한 ID에 여러 이름(표기 변이)이 붙는 병합 방향은 이름 수 63,620 대비 ID 71,702로 ID가 더 많다는 점에서 분할이 우세하다는 정도만 말할 수 있습니다.

# ## 2. 설계에 미치는 영향

# 마지막 쌍(2015–19→2020–25)에서 "지속 저자"는 2015–19에도 출판한 저자입니다. 그런데 2015–19 논문에는 ID가 없으므로 **ID 기준으로 지속 여부를 판정할 수 없습니다.** 지침서의 "마지막 쌍의 신규·지속을 ID로 다시 판별"은 이 자료로는 수행 불가입니다. 대신 ID로 할 수 있는 것은 다음 세 가지입니다.

# 1. **분할 위험 정량화:** 2020~25 기간의 진입·지속 저자(이름 기준)가 몇 개의 ID로 갈리는지, 갈리는 이름이 진입/지속 몫에서 차지하는 질량을 계산합니다.
# 2. **분할 이름 제외 민감도:** 2개 이상 ID에 걸친 이름(3,749개)을 제외하고 마지막 쌍의 진입·적응 몫을 다시 계산합니다. 이는 이름 기반 지속 판정이 "분할 이름"에 얼마나 의존하는지 보는 것이며, 위험군 제외(S6의 F2에서 이미 적용)와는 다른 기준입니다.
# 3. **이름→ID 분할을 시나리오로 처리:** 분할된 이름을 (a) 한 사람으로 둔 현재 방식과 (b) 제외한 방식의 몫 차이를 상한·하한 범위로 보고합니다.

# 이 세 가지는 "ID 기반 결과가 이름 기반과 일치한다"는 지침서의 문장을 쓸 수 없고, **"이름 기반 결과가 ID가 보여 주는 이름 분할에 얼마나 민감한가"**라는 더 좁은 문장만 가능하게 합니다. 이 한계는 부록에 그대로 쓰세요. 이름 결측 58행은 제외됩니다.

# ## 3. 다음 한 걸음

# PA-A2a를 위 정의(진단 2·3)로 새로 설계해 동결 코드를 드리겠습니다. 이를 위해 확인할 것이 두 가지입니다.

# ```bash
# cd ~/Research/Minnesota_Transport/analysis_data/final
# python3 - <<'EOF'
# import pandas as pd
# f7 = pd.read_csv("F7_oa_author_ids.csv", encoding="utf-8-sig", low_memory=False)
# f2 = pd.read_csv("F2_author_paper.csv", encoding="utf-8-sig", low_memory=False)
# # 분할 이름이 F2(2020+)에서 차지하는 저자-논문 행 비율
# g = f7.dropna(subset=["name_norm"]).groupby("name_norm").oa_id.nunique(); split = set(g[g >= 2].index)
# r = f2[f2.year >= 2020]; print("2020+ F2 행", len(r), "| 분할 이름 행 비율", round(float(r.name_norm.isin(split).mean()), 4))
# print("분할 이름의 이름당 행 수 중앙값:", int(r[r.name_norm.isin(split)].groupby("name_norm").size().median()))
# # F2 name_norm이 F7에 없는(ID 미부착) 2020+ 행의 비율
# print("2020+ 행 중 이름이 F7에 존재:", round(float(r.name_norm.isin(set(f7.name_norm.dropna())).mean()), 3))
# EOF
# ls ~/Research/Minnesota_Transport/analysis_data/final | head -30
# ```

# - 위 출력으로 분할 이름이 마지막 기간에서 차지하는 비중이 작으면(예: 몇 % 이내) 제외 민감도가 결과를 거의 못 바꾸는 것이 사전에 예상됩니다. 이는 결과가 아니라 설계용 점검입니다.
# - 또 하나 짚을 점: 이 점검은 분할 **이름**의 비중을 볼 뿐이며, 이름당 ID 수는 F7이 덮는 논문(2020+)에서만 관측됩니다. 2015–19의 같은 이름이 같은 사람인지는 이 자료로 알 수 없습니다. 본문에서는 "2020+ 한정"임을 반드시 밝혀야 합니다.

# ## 4. 병행 가능한 일

# - **PA-B4 동결 코드:** F1·F3 이분 등급(145개) 재추정 코드를 먼저 드릴 수 있습니다. `L7_topic_grade.csv`의 `F1`, `F3` 열과 `S4`·`S5`·`S6` 재추정 마스크를 사용합니다. 원하시면 말씀해 주세요.
# - **PA-0 잔여:** `oa_raw` 외부 백업과 GitHub Private 확인.

In [ ]:
# # PA-A2a 설계 확정 (`pa_a2a.py`)

# 두 출력으로 설계가 닫혔습니다.

# - **`load_all`**: `R.aid`가 `name_norm`이고 `R.ai`는 그 인덱스입니다. 분할 이름 제외는 `R[~R.aid.isin(split)]`로 하면 되고, `L["first"]`·`L["last"]`는 원래 `ai` 기준이라 그대로 쓸 수 있습니다.
# - **위험군 겹침**: 분할 이름 3,749개 중 1,116개가 위험군이고, F2에서는 이미 제외된 상태입니다. 따라서 이번 제외는 **분석 집합에 남아 있는 약 2,633개**를 새로 뺍니다. 위험군 1,431개 중 1,116개(78%)가 분할 이름이라는 점은 기존 위험군 판정과 방향이 맞는다는 정도의 참고입니다. 이전 검증이 순환적이었으므로 독립 근거로 쓰지 마세요.

# ## 설계 (동결 내용)

# - **정의**: F7에서 `oa_id`가 2개 이상인 `name_norm`을 분할 이름으로 둡니다(재현 점검: 3,749개).
# - **시나리오**: (a) 원본, (b) 분할 이름을 **모든 기간에서** 제외한 `Pair`입니다.
# - **통계**: PA-A2c와 같은 4항 정의(진입=신규+복귀)로 `entry_share`, `adapt_share`, `entry_minus_adapt`를 냅니다.
# - **부트스트랩**: 저자 가중 Exp(1)을 (a)에서 뽑고 같은 저자에게 (b)에서도 같은 값을 씁니다(대응 부트스트랩). 시나리오 간 **변화량**의 구간을 냅니다.
# - **판독(고정)**: `entry_minus_adapt`의 변화 절댓값이 0.03 이하이고 부호가 유지되면 "둔감", 아니면 "민감"입니다. 주 대상은 마지막 쌍입니다. 다른 쌍은 같은 규칙으로 따로 표기만 하고, 하나로 합치지 않습니다.
# - **진단**: 역할별(P, E_new, E_ret, X_final, X_gap) 분할 이름의 저자 수·질량 비중을 냅니다.
# - **재현 점검(중단 조건)**: 분할 이름 3,749개, `total_half`가 S6 산출물과 1e-6 이내 일치, 원본 `entry_share`가 PA-A2c 저자 모드 관측값과 1e-6 이내 일치.

# ## 코드: `paper1/pa_a2a.py`

# ```python
# # -*- coding: utf-8 -*-
# # PA-A2a: 분할 이름(F7에서 oa_id>=2) 제외 민감도. ID 기반 재판별은 F7이 2020+만 덮어 불가 -> 진단으로 한정. 미실행 코드.
# # MODE: "freeze" -> "tag" -> (git push --follow-tags) -> "run"(QUICK=True: 오류 점검 전용, 수치 미출력) -> "run"(QUICK=False)
# # S0~S8 판정 불변. 모든 산출물 [Post-hoc]. 분해식·통계는 s6_lib.Pair, PA-A2c와 동일.
# import sys, json, hashlib, subprocess, warnings
# from pathlib import Path
# import numpy as np, pandas as pd
# warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
# MODE = "freeze"
# QUICK = True

# ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; PA = P1/"09_PA"; S6D = P1/"06_S6"; FD = ROOT/"analysis_data/final"
# now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
# sha_b = lambda b: hashlib.sha256(b).hexdigest(); sha_f = lambda p: sha_b(Path(p).read_bytes())
# git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
# class Halt(Exception): pass
# spec_hash = lambda s: sha_b(json.dumps({k: v for k, v in s.items() if k not in ("sha256", "created", "refs")}, sort_keys=True, ensure_ascii=False).encode())
# V2P, BP = PA/"PA_0_PLAN_v2.json", PA/"PA_A2a_PLAN.json"

# PLAN = dict(
#     version=1, seed=2026, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(PA-A2의 1번 교정을 진단으로 축소). S6 결과, Pair·load_all 소스, PA-A2b·A2c 결과, F7 커버리지(2020-2025만)와 분할 이름 비중(2020+ F2 행의 9.0%), 위험군 겹침(1,116/3,749)은 열람한 상태. PA-A2a 결과는 미열람. S0~S8 판정 불변.",
#     why_not_id="F7(OpenAlex ID)은 2020-2025 논문만 덮고 2015-19에는 ID가 없어 마지막 쌍의 지속 저자를 ID로 판정할 수 없음. 따라서 'ID 기반 결과와 일치'는 검증하지 않으며, '이름 기반 결과가 F7이 보여 주는 이름 분할에 얼마나 민감한가'만 본다",
#     split_def="F7_oa_author_ids.csv에서 name_norm 결측 행 제외 후 name_norm별 고유 oa_id >= 2인 이름(재현 점검 3,749개). 분할 판정은 2020+ 논문에서만 관측됨. 분할은 실제 동명이인과 OpenAlex 프로필 분할을 구분하지 못함",
#     scenarios={"orig": "S6 주 설정 spec['main'], s6_lib.make_pair 그대로", "excl_split": "orig의 저자-논문 행 R에서 aid(name_norm)가 분할 이름인 행을 모든 기간에서 제거한 뒤 같은 설정으로 Pair 구성(L['first'], L['last']는 원래 ai 기준 그대로)"},
#     pairs="pairs_of(spec) 전부. 주 대상은 마지막 쌍 P1519->P2025",
#     stats={"entry_share": "half(entry)/(half(adapt)+half(realloc)+half(entry)+half(exit)), entry=신규+복귀", "adapt_share": "half(adapt)/같은 분모", "entry_minus_adapt": "entry_share - adapt_share"},
#     boot="저자 가중 wa~Exp(1)을 orig의 저자 전체에서 뽑고 excl_split의 같은 저자에게 같은 wa를 적용(대응 부트스트랩). B=spec['B_boot'], 역할은 재표집에서 고정. 변화량 delta=excl-orig의 부트스트랩 중앙값, 백분위 구간, basic 구간(2*delta_obs-q_hi, 2*delta_obs-q_lo)",
#     diagnostics="쌍별·역할별(P, E_new, E_ret, X_final, X_gap) 분할 이름의 저자 수 비중과 질량 비중(P·E는 후기 질량, X는 전기 질량), 두 시나리오의 저자 수. 분할 이름 중 분석 집합에 남은 수와 위험군 겹침 수 출력",
#     reading="마지막 쌍에서 entry_minus_adapt의 |delta_obs|<=0.03 이고 부호(orig 대비)가 유지되면 '분할 이름 제외에 둔감', 아니면 '민감'. 다른 쌍도 같은 규칙으로 쌍별 표기만 하고 하나의 결론으로 합치지 않음. 구간은 병기하며 판독에 쓰지 않음",
#     reproduce_check="(a) 분할 이름 수 3,749, (b) orig의 total_half가 renewal_decomposition.csv(scheme=main, variant=all_venues)와 1e-6 이내, (c) orig의 entry_share가 PA_A2c_results.csv(mode=author, stat=entry_share)의 obs와 1e-6 이내, (d) excl_split의 저자 집합이 orig의 부분집합. 하나라도 어긋나면 중단",
#     caveat="분할 이름은 다작·흔한 이름에 치우쳐 있을 가능성이 있어(미확인) 제외가 지속 저자(P)를 선택적으로 덜어낼 수 있음: 역할별 비중 진단으로 확인. 분할 판정이 2020+에서만 가능해 이전 기간 행은 '2020+에 나타난 이름'만 제외되는 비대칭이 있음. 이름 기반 식별 자체의 오류는 이 분석이 해결하지 않음. 위험군(F2에서 이미 제외)은 분할 이름과 겹침",
#     forbidden="결과를 본 뒤 분할 정의·판독 기준(0.03)·B 변경(변경 시 v2 계획으로 분리하고 사유 기록)")

# def freeze():
#     if not V2P.exists(): raise Halt("PA_0_PLAN_v2.json이 없습니다")
#     h = spec_hash(PLAN); print("PA-A2a 계획 해시(예정)", h[:12])
#     if BP.exists(): print("이미 동결:", "동일" if json.loads(BP.read_text(encoding="utf-8"))["sha256"] == h else "⛔ 다름"); return
#     refs = {f.name: sha_f(f) for f in [V2P, S6D/"S6_0_PREREG.json", P1/"s6_lib.py", S6D/"renewal_decomposition.csv", PA/"PA_A2c_PLAN.json", PA/"PA_A2c_results.csv", FD/"F7_oa_author_ids.csv", FD/"F2_risk_names.csv"] if f.exists()}
#     BP.write_text(json.dumps(dict(PLAN, created=now(), sha256=h, refs=refs), ensure_ascii=False, indent=1), encoding="utf-8")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f:
#         f.write(f"\n- {now()} | PA-A2a 계획 동결(해시 {h[:12]}) | 사유: F7이 2020+만 덮어 ID 재판별 불가 -> 분할 이름 제외 민감도 진단으로 축소. 설계 점검 수치(커버리지, 9.0%, 위험군 겹침)와 S6·A2b·A2c 결과 열람 상태, PA-A2a 결과 미열람. QUICK은 수치 미출력 | 본 결과 변경 여부: 없음\n")
#     print("저장: PA_A2a_PLAN.json")

# def tag():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     t = f"PA-A2a-plan-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
#     if git("tag", "-l", t).stdout.strip(): print("이미 있음:", t); return
#     r = git("add", "--", "paper1/09_PA/PA_A2a_PLAN.json", "paper1/09_PA/PA_CHANGELOG.md", "paper1/pa_a2a.py"); print("add", r.returncode, r.stderr.strip())
#     r = git("commit", "-m", "PA-A2a plan freeze (before execution)"); print("commit", r.returncode, (r.stdout + r.stderr)[:200])
#     r = git("tag", "-a", t, "-m", "PA-A2a plan"); print("tag", r.returncode, r.stderr.strip()); print("※ 이어서 git push --follow-tags")

# def run():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     sp = json.loads(BP.read_text(encoding="utf-8"))
#     if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(PLAN): raise Halt("동결 파일과 현재 계획이 다릅니다")
#     if not git("tag", "-l", "PA-A2a-plan-*").stdout.strip(): raise Halt("tag 단계를 먼저 실행하세요")
#     sys.path.insert(0, str(P1)); import s6_lib as S
#     spec = S.load_spec(); L = S.load_all(spec); M = spec["main"]; PAIRS = S.pairs_of(spec); half = S.half; R = L["R"]
#     sfx = "_QUICK" if QUICK else ""; B = 20 if QUICK else int(spec["B_boot"]); lo_q, hi_q = spec["ci"]; rng = np.random.default_rng(int(PLAN["seed"]))
#     # 분할 이름
#     f7 = pd.read_csv(FD/"F7_oa_author_ids.csv", encoding="utf-8-sig", low_memory=False)
#     g = f7.dropna(subset=["name_norm"]).groupby("name_norm").oa_id.nunique(); split = set(g[g >= 2].index)
#     if len(split) != 3749: raise Halt(f"분할 이름 수 재현 실패: {len(split)} (3749여야 함)")
#     rk = set(pd.read_csv(FD/"F2_risk_names.csv", encoding="utf-8-sig").name_norm.astype(str)); in_R = set(R.aid.unique()) & split
#     print(f"분할 이름 {len(split):,} | 분석 집합(R)에 존재 {len(in_R):,} | 위험군과 겹침 {len(split & rk):,} | R에 존재하는 위험군 이름 {len(set(R.aid.unique()) & rk):,}")
#     L2 = dict(L); L2["R"] = R[~R.aid.isin(split)]
#     aid_of = R.drop_duplicates("ai").set_index("ai").aid
#     ref = pd.read_csv(S6D/"renewal_decomposition.csv"); ref = ref[(ref.scheme == M["scheme"]) & (ref.variant == "all_venues")].set_index("pair")
#     a2c = pd.read_csv(PA/"PA_A2c_results.csv"); a2c = a2c[(a2c["mode"] == "author") & (a2c.stat == "entry_share")].set_index("pair")
#     STATS = ["entry_share", "adapt_share", "entry_minus_adapt"]
#     def stats(d):
#         h = {t: half(d[t]) for t in ["adapt", "realloc", "entry", "exit"]}; tot = sum(h.values())
#         return dict(entry_share=h["entry"]/tot, adapt_share=h["adapt"]/tot, entry_minus_adapt=(h["entry"] - h["adapt"])/tot)
#     rows, roles = [], []
#     for p0, p1 in PAIRS:
#         name = f"{p0}->{p1}"
#         Pa = S.make_pair(L, p0, p1, M["scheme"], M["venue"], M["topic"], M["hom"]); Pb = S.make_pair(L2, p0, p1, M["scheme"], M["venue"], M["topic"], M["hom"])
#         da, db = Pa.decompose(), Pb.decompose(); oa, ob = stats(da), stats(db)
#         if abs(half(da["total"]) - float(ref.loc[name, "total_half"])) > 1e-6: raise Halt(f"S6 관측 재현 실패({name})")
#         if abs(oa["entry_share"] - float(a2c.loc[name, "obs"])) > 1e-6: raise Halt(f"PA-A2c entry_share 재현 실패({name})")
#         idx = np.searchsorted(Pa.aids, Pb.aids)
#         if not (idx < len(Pa.aids)).all() or not (Pa.aids[idx] == Pb.aids).all(): raise Halt(f"excl_split 저자 집합이 orig의 부분집합이 아님({name})")
#         # 역할별 진단
#         flag = pd.Series(aid_of.loc[Pa.aids].values).isin(split).values
#         mass0 = np.bincount(Pa.a0, Pa.m0, Pa.nA)/Pa.m0.sum(); mass1 = np.bincount(Pa.a1, Pa.m1, Pa.nA)/Pa.m1.sum()
#         for r in ["P", "E_new", "E_ret", "X_final", "X_gap"]:
#             m = Pa.role == r; ms = mass0 if r.startswith("X") else mass1
#             roles.append(dict(pair=name, role=r, n_authors=int(m.sum()), split_author_share=float(flag[m].mean()) if m.any() else np.nan,
#                               mass=float(ms[m].sum()), split_mass_share=float(ms[m & flag].sum()/ms[m].sum()) if ms[m].sum() > 0 else np.nan))
#         roles.append(dict(pair=name, role="ALL_period0", n_authors=int(Pa.nA), split_author_share=float(flag.mean()), mass=1.0, split_mass_share=float(mass0[flag].sum())))
#         roles.append(dict(pair=name, role="ALL_period1", n_authors=int(Pb.nA), split_author_share=float(flag.mean()), mass=1.0, split_mass_share=float(mass1[flag].sum())))
#         # 대응 부트스트랩
#         bs = []
#         for _ in range(B):
#             wa = rng.exponential(size=Pa.nA); sa = stats(Pa.decompose(Pa.m0*wa[Pa.a0], Pa.m1*wa[Pa.a1]))
#             wb = wa[idx]; sb = stats(Pb.decompose(Pb.m0*wb[Pb.a0], Pb.m1*wb[Pb.a1]))
#             bs.append({s: sb[s] - sa[s] for s in STATS})
#         bs = pd.DataFrame(bs)
#         for s in STATS:
#             dl = ob[s] - oa[s]; ql, qh = np.percentile(bs[s], [lo_q, hi_q])
#             rows.append(dict(pair=name, stat=s, obs_orig=oa[s], obs_excl=ob[s], delta=dl, boot_median=float(bs[s].median()), pct_lo=ql, pct_hi=qh, basic_lo=2*dl - qh, basic_hi=2*dl - ql,
#                              n_authors_orig=int(Pa.nA), n_authors_excl=int(Pb.nA), B=B))
#         print(f"  {name} 완료")
#     RS, RO = pd.DataFrame(rows), pd.DataFrame(roles)
#     RS.to_csv(PA/f"PA_A2a_results{sfx}.csv", index=False, encoding="utf-8-sig"); RO.to_csv(PA/f"PA_A2a_roles{sfx}.csv", index=False, encoding="utf-8-sig")
#     if QUICK: print("\n[QUICK] 오류 없이 완료. 수치는 출력하지 않음(_QUICK 파일을 열어 해석하지 마세요)."); return
#     print("\n[Post-hoc] PA-A2a 결과(분할 이름 제외 대 원본)\n", RS.round(4).to_string(index=False))
#     print("\n[Post-hoc] 역할별 분할 이름 비중\n", RO.round(4).to_string(index=False))
#     k = RS[RS.stat == "entry_minus_adapt"].copy(); k["sign_kept"] = np.sign(k.obs_excl) == np.sign(k.obs_orig); k["within_003"] = k.delta.abs() <= 0.03; k["insensitive"] = k.sign_kept & k.within_003
#     print("\n[동결 판독] entry_minus_adapt\n", k[["pair", "obs_orig", "obs_excl", "delta", "pct_lo", "pct_hi", "sign_kept", "within_003", "insensitive"]].round(4).to_string(index=False))
#     last = k[k.pair == f"{PAIRS[-1][0]}->{PAIRS[-1][1]}"].iloc[0]; print("→ 마지막 쌍:", "분할 이름 제외에 둔감" if last.insensitive else "분할 이름 제외에 민감")
#     print("※ ID 기반 재판별이 아니라 분할 이름 제외 민감도임. 분할 판정은 2020+에서만 관측. 쌍별로 따로 읽고 합치지 않음. S0~S8 판정 불변.")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as fh:
#         fh.write(f"- {now()} | PA-A2a 실행 | 정의: PA_A2a_PLAN {sp['sha256'][:12]} | 결과: PA_A2a_results.csv, PA_A2a_roles.csv | 본 결과 변경 여부: 없음\n")

# try: {"freeze": freeze, "tag": tag, "run": run}[MODE]()
# except Halt as e: print(f"⛔ 중단: {e}")
# ```

# ## 실행 순서

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 pa_a2a.py                                                           # 1) freeze: 해시 확인
# sed -i 's/^MODE = "freeze"/MODE = "tag"/' pa_a2a.py && python3 pa_a2a.py     # 2) tag
# cd .. && git push --follow-tags && cd paper1
# sed -i 's/^MODE = "tag"/MODE = "run"/' pa_a2a.py
# ( time python3 -u pa_a2a.py ) 2>&1 | tee 09_PA/PA_A2a_quick_log.txt         # 3) QUICK (수치 미출력)
# sed -i 's/^QUICK = True/QUICK = False/' pa_a2a.py && grep -n '^MODE\|^QUICK' pa_a2a.py
# cd .. && git add paper1/pa_a2a.py paper1/09_PA/PA_A2a_*QUICK* paper1/09_PA/PA_A2a_quick_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-A2a: QUICK done; QUICK=False" && git tag -a PA-A2a-fullrun-2026-10-21 -m "pre-run" && git push --follow-tags
# cd paper1 && ( time python3 -u pa_a2a.py ) 2>&1 | tee 09_PA/PA_A2a_full_log.txt   # 4) 본 실행
# ```

# ## 주의할 점

# - **미실행 코드입니다.** 새로 쓴 부분은 대응 부트스트랩의 `idx` 매핑과 역할별 진단입니다. 재현 점검 4개가 막아 주므로, 오류나 중단 메시지가 나오면 그것만 붙여 주세요. 실행 시간은 A2c와 비슷하게 7분 안팎으로 예상합니다.
# - **첫 줄 출력은 설계 점검용입니다.** 분석 집합에 남은 분할 이름 수와 위험군 겹침이 나옵니다. QUICK 로그에 이 수치는 포함되지만 결과 통계는 아니므로, 본 결과를 읽기 전 열람 사실로 기록할 필요는 없습니다.
# - **제외가 중립이 아닐 수 있습니다.** 분할 이름은 다작·흔한 이름에 치우쳐 있을 가능성이 있는데, 이는 확인하지 않은 가설입니다. 그렇다면 지속 저자(P)를 선택적으로 덜어내 적응 몫이 줄 수 있고, 이는 "신규 우세에 둔감하다"는 결과를 거꾸로 약하게 만들 수도 있습니다. 역할별 비중 표가 이 점을 가려 줍니다.
# - **비대칭이 있습니다.** 분할 이름은 2020+에서만 관측되므로 이전 기간에서는 "2020+에 나타난 이름"만 제외됩니다. 첫 세 쌍의 변화량은 마지막 쌍과 같은 의미가 아니니 쌍별로 읽으세요.
# - **결과 문장 범위.** 어느 쪽으로 나와도 "ID 기반 결과가 이름 기반과 일치한다"는 쓸 수 없습니다. 쓸 수 있는 것은 "이름 기반 진입·적응 몫이 F7에서 ID가 갈리는 이름의 제외에 민감한 정도"까지입니다.

# 로그가 나오면 정의대로 읽겠습니다. 다음은 PA-B4(F1+F3 이분 등급, 145개) 동결 코드입니다. 입력 구조는 이미 확인됐으니 A2a 실행과 병행해서 요청하셔도 됩니다.

In [ ]:
# # 논문 1 사후분석, 쉽게 정리 (2026-10-21)
# ## 먼저, 이 연구가 뭔가요?
# 교통 분야 논문 약 8만 편을 모아서 **"교통 연구의 주제가 시간이 지나며 어떻게 바뀌었나, 누가 바꿨나, 어느 나라가 이끌었나"**를 본 연구입니다.
# 원래 분석(S0~S8)은 **결과를 보기 전에 규칙을 도장 찍어 두고(동결)** 진행했습니다. 그런데 결과가 나오고 보니 약한 곳이 몇 군데 있었습니다. 그래서 **"사후분석(PA)"**을 따로 했습니다. 규칙은 이렇습니다.
# - 원래 결과(통과/실패)는 **절대 바꾸지 않는다.**
# - 분석 전에 계획을 먼저 도장 찍고 인터넷(GitHub)에 올려 시각을 남긴다.
# - 결과가 기대와 달라도 전부 보고한다.
# 시험 본 뒤에 "이 문제는 이렇게 풀면 더 잘 보여요" 하고 보충 설명을 다는 것과 비슷합니다. 점수는 안 바꿉니다.
# ---
# ## 원래 분석(S0~S8)의 큰 결과
# | 질문 | 답 |
# |---|---|
# | 주제가 어느 해에 확 바뀐 "체제 전환"이 있었나? | 통계 검사를 통과하지 못해서 **"체제 전환" 이야기는 철회**했습니다. |
# | 새 연구자가 주제 변화를 이끌었나? | 새로 들어온 저자의 몫이 0.24에서 0.38로 커졌습니다. |
# | 어느 나라가 이끌었나? | 미국은 줄고(0.46→0.19), 중국은 늘었습니다(0.03→0.35). |
# | 연구가 필요한 곳을 향하나? | 6개 분야 중 2개만 검증을 통과했습니다. |
# ---
# ## 지금까지 한 사후분석과 결과
# ### 0. 정리 작업 (분석이 아니라 정리)
# - 파일 152개의 "지문(해시)"이 전부 그대로인 것을 확인했습니다. 몰래 바뀐 파일이 없다는 뜻입니다.
# - 잃어버린 S4 코드를 복구했고, 잘못 붙은 이름표(태그)를 고쳤고, 모든 기록을 GitHub에 올렸습니다.
# ### 1. PA-A1: "점프가 있었다면 검사가 잡아냈을까?"
# - **비유:** 연기 감지기가 안 울렸는데, 감지기가 고장 난 건지 정말 불이 없는 건지 알아야 합니다. 그래서 **일부러 작은 불을 피워** 봤습니다.
# - **첫 시도는 무효.** 코드에 버그가 있어서 불이 안 켜졌습니다. 지우지 않고 기록해 두었습니다.
# - **두 번째 결과:** 2020년 부근에 점프를 넣으면 대체로 잡아냈습니다. 하지만 2008~2012년 부근은 잘 못 잡는 **사각지대**가 있었습니다.
# - **결론:** "큰 점프는 없었다"까지는 말할 수 있지만 "체제 전환이 없었다고 확인됐다"는 말하면 안 됩니다.
# ### 2. PA-B2: "2020년 변화가 논문 수가 갑자기 늘어서 생긴 착시인가?"
# - 2020년 논문 수를 줄여서 다시 계산해도 변화가 그대로 남았습니다.
# - **주의:** 줄인 양이 20%뿐이라 시험이 약했습니다. "논문 수 차이만으로는 사라지지 않는다"까지만 말할 수 있습니다.
# - 2020년 변화의 내용은 **코로나 관련 주제가 1등**, 그다음 전기모터, 자율주행, 머신러닝 순이었습니다.
# ### 3. PA-B2b: "특정 저널이나 코로나 주제 하나 때문인가?"
# - 논문이 가장 많이 늘어난 저널 5곳을 빼도, 코로나 주제만 빼도 변화가 남았습니다.
# - **둘을 동시에 빼면** 변화가 약해지고 위치가 한 해 앞으로 밀렸습니다.
# - 그래서 "2020년 변화는 한 가지 원인이 아니라 여러 흐름이 겹친 것"으로 읽습니다.
# ### 4. PA-A2c: "새 저자 우세 결과의 오차 범위를 더 정직하게"
# - 4개 시기 중 **3개**에서 "새로 들어온 저자 > 기존 저자 적응"이 유지됐습니다.
# - 두 가지 오차 계산법에서 모두 확실한 곳은 **마지막 시기(2015–19→2020–25)뿐**입니다.
# ### 5. PA-A2b: "'새 저자'의 정의를 바꾸면 결과가 변하나?"
# - **중요한 발견:** 새 저자를 "2년 안에 첫 논문을 쓴 사람"으로 좁히면 **결과가 뒤집힙니다.**
# - 하지만 이건 "새 저자"라는 이름표를 좁게 붙였을 뿐, 들어온 사람 총량은 같습니다.
# - 그래서 문장을 **"신규 저자 주도"에서 "진입(신규+복귀) 주도"로 고쳐야 안전**합니다. 가장 확실한 구간은 역시 마지막 시기입니다.
# ### 6. PA-B5: "S4 분석의 안정성 0.70, 아슬아슬하지 않나?"
# - 기준선을 바꿔 보니 안정성은 0.66~0.77로 움직였습니다. **"0.70 근처에서 흔들린다"**가 정확한 표현입니다.
# - 다행히 **증가/감소 방향이 뒤집힌 토픽은 없었습니다.**
# ### 7. PA-A2a: "이름이 같은 다른 사람이 섞였다면?" (진행 중)
# - 데이터에 OpenAlex ID가 **2020년 이후 논문에만** 있어서, 원래 계획(ID로 다시 판별)은 불가능했습니다.
# - 그래서 **"ID가 둘 이상으로 갈라지는 이름을 빼도 결과가 변하나?"**로 계획을 줄였습니다.
# - 계획과 코드는 방금 완성했고, **아직 실행 전**입니다.
# ---
# ## 지금 논문이 말해도 되는 것 / 안 되는 것
# | 말해도 됨 | 말하면 안 됨 |
# |---|---|
# | 주제 변화는 한 번의 점프보다 **서서히 재배치**된 것으로 설명하는 게 자료에 맞다 | 체제 전환이 없었음이 확인됐다 |
# | 2020년 부근에는 코로나와 기술 주제가 **겹친 가속**이 있다 | 2020년이 착시가 아니라고 확정됐다 |
# | 새로 **진입한** 저자가 기존 저자의 적응보다 크다 (특히 마지막 시기) | 신규 저자가 모든 시기에 주도했다 |
# | 미국은 줄고 중국은 늘었다 | 인과 관계(누가 원인인지) |
# ---
# ## 앞으로 할 분석
# **가까운 순서**
# 1. **PA-A2a 실행** (이름 분할 제외 시험)
# 2. **PA-B4:** 품질 기준을 단순화(F1+F3)해도 결과가 같은지
# 3. **PA-C1:** 중국 소속 새 저자를 빼도 "진입 우세"가 남는지
# 4. **PA-C2:** 국가 몫 계산에 쓴 논문 집합이 달라서 생기는 차이 점검
# **그다음**
# - **PA-B7:** 주제 정보가 없는 논문(16%)이 결과를 바꿀 수 있는지, 최악·최선의 경우로 시험
# - **PA-B3:** "신생 토픽" 정의 4가지의 공통 핵심 찾기
# - **PA-A3:** 선두 국가 분석을 구간을 같은 길이로 바꿔 재확인
# - **PA-B9, B8:** 소득 수준 효과 분리, 통과 못 한 4개 분야의 참고 결과
# - **PA-B6:** 동명이인을 사람이 직접 50쌍 확인 (사람 손이 필요)
# **마무리**
# - 열람 이력을 한 장 표로 정리(무엇을 언제 봤는지 솔직히 기록)
# - 모든 사후분석 표에 `[Post-hoc]` 표시
# - 기각되거나 약하게 나온 분석도 부록에 전부 수록
# - 논문 제목에서 **"체제 전환"을 빼고** "신규 진입 주도 갱신 + 선두 국가 교체"를 중심에 두기
# ---
# ## 한 줄 요약
# > **"큰 점프는 없었고, 변화는 서서히 이어졌으며, 새로 들어온 연구자들과 중국의 부상이 핵심이다. 다만 정의를 바꾸면 약해지는 곳이 있어서, 그 한계까지 정직하게 적는다."**

In [ ]:
# yjlee@u24:~/Research/Minnesota_Transport$ cd ~/Research/Minnesota_Transport/paper1
# python3 pa_a2a.py                                                           # 1) freeze: 해시 확인
# sed -i 's/^MODE = "freeze"/MODE = "tag"/' pa_a2a.py && python3 pa_a2a.py     # 2) tag
# cd .. && git push --follow-tags && cd paper1
# sed -i 's/^MODE = "tag"/MODE = "run"/' pa_a2a.py
# ( time python3 -u pa_a2a.py ) 2>&1 | tee 09_PA/PA_A2a_quick_log.txt         # 3) QUICK (수치 미출력)
# sed -i 's/^QUICK = True/QUICK = False/' pa_a2a.py && grep -n '^MODE\|^QUICK' pa_a2a.py
# cd .. && git add paper1/pa_a2a.py paper1/09_PA/PA_A2a_*QUICK* paper1/09_PA/PA_A2a_quick_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-A2a: QUICK done; QUICK=False" && git tag -a PA-A2a-fullrun-2026-10-21 -m "pre-run" && git push --follow-tags
# cd paper1 && ( time python3 -u pa_a2a.py ) 2>&1 | tee 09_PA/PA_A2a_full_log.txt   # 4) 본 실행
# PA-A2a 계획 해시(예정) 150c834dcbd7
# 저장: PA_A2a_PLAN.json
# add 0
# commit 0 [main a6dd0e4] PA-A2a plan freeze (before execution)
#  3 files changed, 156 insertions(+)
#  create mode 100644 paper1/09_PA/PA_A2a_PLAN.json
#  create mode 100644 paper1/pa_a2a.py

# tag 0
# ※ 이어서 git push --follow-tags
# Enumerating objects: 12, done.
# Counting objects: 100% (12/12), done.
# Delta compression using up to 24 threads
# Compressing objects: 100% (8/8), done.
# Writing objects: 100% (8/8), 7.11 KiB | 3.55 MiB/s, done.
# Total 8 (delta 4), reused 0 (delta 0), pack-reused 0
# remote: Resolving deltas: 100% (4/4), completed with 3 local objects.
# To https://github.com/LEEYJ1021/transport-knowledge-paper1.git
#    bac7d76..a6dd0e4  main -> main
#  * [new tag]         PA-A2a-plan-2026-10-21 -> PA-A2a-plan-2026-10-21
# IPW 모델별 행 수: {'A': 91794, 'B': 81645} | 사용: A
# IPW 범위(주제 논문) 0.866~1.883 (S4는 0.86~1.94) | 결측 0편은 1로 대체
# F4에 없는 저자 0명 | F4 첫해 > F2 첫해인 저자 0명(0이어야 정상) | F4 마지막해 < F2 마지막해 0명
# 분석 논문 76,984 | 저자-논문 행 255,354 | 저자 93,763 | 가중치 ['equal', 'inv_n_authors', 'first_last', 'full_count'] | 위험군 변형 ['baseline'] (F2는 위험군 제외 상태)
# 분할 이름 3,749 | 분석 집합(R)에 존재 2,592 | 위험군과 겹침 1,116 | R에 존재하는 위험군 이름 0
#   P0004->P0509 완료
#   P0509->P1014 완료
#   P1014->P1519 완료
#   P1519->P2025 완료

# [QUICK] 오류 없이 완료. 수치는 출력하지 않음(_QUICK 파일을 열어 해석하지 마세요).

# real    0m7.317s
# user    0m8.751s
# sys     0m0.382s
# 9:MODE = "run"
# 10:QUICK = False
# [main df36608] PA-A2a: QUICK done; QUICK=False
#  4 files changed, 59 insertions(+), 2 deletions(-)
#  create mode 100644 paper1/09_PA/PA_A2a_quick_log.txt
#  create mode 100644 paper1/09_PA/PA_A2a_results_QUICK.csv
#  create mode 100644 paper1/09_PA/PA_A2a_roles_QUICK.csv
# Enumerating objects: 13, done.
# Counting objects: 100% (13/13), done.
# Delta compression using up to 24 threads
# Compressing objects: 100% (9/9), done.
# Writing objects: 100% (9/9), 3.45 KiB | 3.45 MiB/s, done.
# Total 9 (delta 3), reused 0 (delta 0), pack-reused 0
# remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
# To https://github.com/LEEYJ1021/transport-knowledge-paper1.git
#    a6dd0e4..df36608  main -> main
#  * [new tag]         PA-A2a-fullrun-2026-10-21 -> PA-A2a-fullrun-2026-10-21
# IPW 모델별 행 수: {'A': 91794, 'B': 81645} | 사용: A
# IPW 범위(주제 논문) 0.866~1.883 (S4는 0.86~1.94) | 결측 0편은 1로 대체
# F4에 없는 저자 0명 | F4 첫해 > F2 첫해인 저자 0명(0이어야 정상) | F4 마지막해 < F2 마지막해 0명
# 분석 논문 76,984 | 저자-논문 행 255,354 | 저자 93,763 | 가중치 ['equal', 'inv_n_authors', 'first_last', 'full_count'] | 위험군 변형 ['baseline'] (F2는 위험군 제외 상태)
# 분할 이름 3,749 | 분석 집합(R)에 존재 2,592 | 위험군과 겹침 1,116 | R에 존재하는 위험군 이름 0
#   P0004->P0509 완료
#   P0509->P1014 완료
#   P1014->P1519 완료
#   P1519->P2025 완료

# [Post-hoc] PA-A2a 결과(분할 이름 제외 대 원본)
#          pair              stat  obs_orig  obs_excl   delta  boot_median  pct_lo  pct_hi  basic_lo  basic_hi  n_authors_orig  n_authors_excl    B
# P0004->P0509       entry_share    0.2812    0.2817  0.0006       0.0006 -0.0026  0.0042   -0.0030    0.0038           14775           14636 2000
# P0004->P0509       adapt_share    0.2866    0.2885  0.0019       0.0005 -0.0047  0.0049   -0.0012    0.0085           14775           14636 2000
# P0004->P0509 entry_minus_adapt   -0.0055   -0.0068 -0.0013       0.0001 -0.0066  0.0080   -0.0106    0.0039           14775           14636 2000
# P0509->P1014       entry_share    0.2961    0.2926 -0.0036      -0.0026 -0.0062  0.0012   -0.0083   -0.0009           23540           23215 2000
# P0509->P1014       adapt_share    0.2690    0.2759  0.0069       0.0040 -0.0020  0.0088    0.0050    0.0158           23540           23215 2000
# P0509->P1014 entry_minus_adapt    0.0271    0.0167 -0.0105      -0.0065 -0.0140  0.0020   -0.0229   -0.0070           23540           23215 2000
# P1014->P1519       entry_share    0.3341    0.3370  0.0029       0.0023 -0.0017  0.0066   -0.0008    0.0076           37730           36896 2000
# P1014->P1519       adapt_share    0.2880    0.2843 -0.0037      -0.0031 -0.0088  0.0018   -0.0091    0.0014           37730           36896 2000
# P1014->P1519 entry_minus_adapt    0.0461    0.0527  0.0066       0.0054 -0.0024  0.0142   -0.0010    0.0156           37730           36896 2000
# P1519->P2025       entry_share    0.3934    0.3978  0.0044       0.0033 -0.0001  0.0070    0.0018    0.0089           75616           73030 2000
# P1519->P2025       adapt_share    0.2806    0.2762 -0.0044      -0.0039 -0.0084  0.0003   -0.0090   -0.0004           75616           73030 2000
# P1519->P2025 entry_minus_adapt    0.1128    0.1216  0.0088       0.0071  0.0006  0.0144    0.0031    0.0169           75616           73030 2000

# [Post-hoc] 역할별 분할 이름 비중
#          pair        role  n_authors  split_author_share   mass  split_mass_share
# P0004->P0509           P       2502              0.0148 0.4411            0.0256
# P0004->P0509       E_new       7253              0.0102 0.4952            0.0130
# P0004->P0509       E_ret        615              0.0114 0.0637            0.0072
# P0004->P0509     X_final       3496              0.0000 0.3007            0.0000
# P0004->P0509       X_gap        909              0.0231 0.1127            0.0222
# P0004->P0509 ALL_period0      14775              0.0094 1.0000            0.0114
# P0004->P0509 ALL_period1      14636              0.0094 1.0000            0.0182
# P0509->P1014           P       4068              0.0189 0.4625            0.0289
# P0509->P1014       E_new      12207              0.0153 0.4880            0.0220
# P0509->P1014       E_ret        963              0.0208 0.0495            0.0315
# P0509->P1014     X_final       5214              0.0000 0.3094            0.0000
# P0509->P1014       X_gap       1088              0.0377 0.0855            0.0258
# P0509->P1014 ALL_period0      23540              0.0138 1.0000            0.0182
# P0509->P1014 ALL_period1      23215              0.0138 1.0000            0.0256
# P1014->P1519           P       6918              0.0299 0.4945            0.0499
# P1014->P1519       E_new      19263              0.0257 0.4700            0.0371
# P1014->P1519       E_ret       1229              0.0448 0.0355            0.0542
# P1014->P1519     X_final       8927              0.0000 0.2973            0.0000
# P1014->P1519       X_gap       1393              0.0553 0.0604            0.0446
# P1014->P1519 ALL_period0      37730              0.0221 1.0000            0.0256
# P1014->P1519 ALL_period1      36896              0.0221 1.0000            0.0440
# P1519->P2025           P      11764              0.0641 0.4204            0.1012
# P1519->P2025       E_new      45937              0.0357 0.5475            0.0792
# P1519->P2025       E_ret       2269              0.0824 0.0321            0.1532
# P1519->P2025     X_final      15217              0.0000 0.3272            0.0000
# P1519->P2025       X_gap        429              0.0070 0.0080            0.0155
# P1519->P2025 ALL_period0      75616              0.0342 1.0000            0.0440
# P1519->P2025 ALL_period1      73030              0.0342 1.0000            0.0909

# [동결 판독] entry_minus_adapt
#          pair  obs_orig  obs_excl   delta  pct_lo  pct_hi  sign_kept  within_003  insensitive
# P0004->P0509   -0.0055   -0.0068 -0.0013 -0.0066  0.0080       True        True         True
# P0509->P1014    0.0271    0.0167 -0.0105 -0.0140  0.0020       True        True         True
# P1014->P1519    0.0461    0.0527  0.0066 -0.0024  0.0142       True        True         True
# P1519->P2025    0.1128    0.1216  0.0088  0.0006  0.0144       True        True         True
# → 마지막 쌍: 분할 이름 제외에 둔감
# ※ ID 기반 재판별이 아니라 분할 이름 제외 민감도임. 분할 판정은 2020+에서만 관측. 쌍별로 따로 읽고 합치지 않음. S0~S8 판정 불변.

# real    5m42.556s
# user    5m43.937s
# sys     0m0.395s
# yjlee@u24:~/Research/Minnesota_Transport/paper1$

In [ ]:
# # PA-A2a 판독

# **동결 판독의 기계적 결과는 "둔감"입니다.** 마지막 쌍은 물론 4개 쌍 모두 `entry_minus_adapt`의 변화 절댓값이 0.03 이하이고 부호가 유지됐습니다. 쌍별로 따로 읽겠습니다.

# ## 1. 재현 점검과 설계 수치

# - 재현 점검 4개(분할 이름 3,749개, S6 `total_half`, A2c `entry_share`, 저자 집합 부분집합)가 모두 통과했습니다.
# - 분할 이름 3,749개 중 **분석 집합에 남은 것은 2,592개**입니다. 위험군 이름은 분석 집합에 0개였으므로(F2에서 이미 제외), 이번 제외는 위험군과 겹치지 않는 새 이름만 뺀 것입니다. 앞에서 "약 2,633개"라고 예상했는데 실제는 2,592개였습니다.

# ## 2. 결과

# | 쌍 | 원본 | 제외 후 | 변화 | 백분위 구간 |
# |---|---|---|---|---|
# | P0004→P0509 | −0.0055 | −0.0068 | −0.0013 | 0 포함 |
# | P0509→P1014 | 0.0271 | 0.0167 | −0.0105 | 0 포함 |
# | P1014→P1519 | 0.0461 | 0.0527 | +0.0066 | 0 포함 |
# | **P1519→P2025** | **0.1128** | **0.1216** | **+0.0088** | **[0.0006, 0.0144]** |

# - 변화는 최대 0.0105로 판독 기준(0.03)의 약 3분의 1입니다. 저자 수도 마지막 쌍에서 75,616명에서 73,030명(−3.4%)으로 적게 줄었습니다.
# - 마지막 쌍은 분할 이름을 빼면 진입 우세가 **오히려 조금 커집니다**(+0.009). 구간이 0 바로 위(하한 0.0006)라 "방향이 있다"고 쓰기보다 "거의 변하지 않으며 약간 커지는 쪽"이 안전합니다.
# - 둘째 쌍(P0509→P1014)은 변화 −0.0105로, 구간 방식(basic)에 따라서는 0을 벗어납니다(basic [−0.023, −0.007]). 판독 기준 안이므로 "둔감"이지만, 이 쌍은 상대적으로 가장 많이 움직였습니다.

# ## 3. 역할별 진단: 제외가 중립이 아니었습니다

# 미리 적어 둔 우려("지속 저자 P를 선택적으로 덜어낼 수 있다")를 확인할 수 있는 표입니다.

# - 마지막 쌍에서 분할 이름의 **질량 비중**은 지속 저자(P) 10.1%, 복귀 진입(E_ret) 15.3%, 신규 진입(E_new) 7.9%입니다. 전체로는 후기 9.1%입니다.
# - 즉 분할 이름은 지속·복귀 쪽에 상대적으로 더 몰려 있습니다. 이는 이름이 흔하고 다작인 저자가 많다는 가설과 맞지만, 이름별 행 수 분포를 보지 않았으므로 가설입니다.
# - 그런데도 적응 몫은 0.2806에서 0.2762로 조금만 줄었고 진입 몫이 0.3934에서 0.3978로 조금 늘었습니다. 지속 질량이 더 많이 빠졌는데 결과 변화가 작다는 점이 둔감 판정을 뒷받침합니다.
# - 분할 이름 비중은 시기가 갈수록 커집니다(전체 질량 기준 1.1%→1.8%, 2.6%, 4.4%, 마지막 후기 9.1%). 앞선 기간에는 2020+에 나타난 이름만 제외되는 비대칭 때문이라고 계획서에 적어 둔 그대로입니다. 그래서 앞 세 쌍의 변화량은 마지막 쌍과 같은 의미로 비교하면 안 됩니다.
# - `X_final`의 분할 이름이 0인 것은 2020+에 나타난 이름만 판정되기 때문에 당연한 구조 효과입니다(퇴출 최종 저자는 2020 이후 없음). 해석하지 마세요.

# ## 4. 쓸 수 있는 문구

# > "OpenAlex에서 둘 이상의 ID로 갈라지는 이름(분석 집합 2,592개)을 모든 기간에서 제외해도, 진입이 적응보다 큰 정도는 모든 구간에서 0.011 이내로 변했고 부호는 유지됐다(마지막 구간 0.113→0.122). 이는 ID 기반 재판별이 아니라 분할 이름 제외에 대한 민감도 진단이며, ID는 2020년 이후 논문에만 있다."

# 쓰면 안 되는 것은 "ID 기반 결과가 이름 기반과 일치한다", "동명이인 문제가 해결됐다"입니다. 분할 이름에는 실제 동명이인과 OpenAlex 프로필 분할이 섞여 있고, 분할되지 않은 동명이인(ID가 하나로 묶인 경우)은 이 분석이 전혀 잡지 못합니다.

# ## 5. 기록

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-A2a 결과 해석 메모(B=2000): 동결 판독 4쌍 모두 '둔감'(entry_minus_adapt 변화 -0.0013/-0.0105/+0.0066/+0.0088, 부호 유지). 마지막 쌍 0.1128->0.1216(백분위 [0.0006,0.0144]). 분석 집합에서 제외된 분할 이름 2,592개(위험군과 겹침 0). 역할별 질량 비중은 P 10.1%, E_ret 15.3%, E_new 7.9%(마지막 쌍)로 지속·복귀 쪽에 치우침(원인은 미확인 가설). 분할 판정이 2020+에서만 가능해 앞 세 쌍은 비대칭이며 같은 의미로 비교 불가. ID 기반 재판별 아님, 분할되지 않은 동명이인은 미검출 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_A2a_results.csv paper1/09_PA/PA_A2a_roles.csv paper1/09_PA/PA_A2a_full_log.txt paper1/09_PA/PA_CHANGELOG.md paper1/pa_a2a.py
# git commit -m "PA-A2a results" && git tag -a PA-A2a-results-2026-10-21 -m "PA-A2a outputs, seen" && git push --follow-tags
# ```

# ## 6. 핵심 PA 진행 상황과 다음 단계

# 핵심 PA(A1, A2, B2)는 이제 모두 끝났습니다. A2의 세 교정(c: 부트스트랩, b: 정의, a: 이름 분할)을 종합하면 이렇게 쓸 수 있습니다.

# - **견고한 것:** 마지막 구간(2015–19→2020–25)에서 진입이 적응보다 큰 방향. 부트스트랩 방식, 신규 정의(s6·y5), 분할 이름 제외 모두에서 유지됩니다.
# - **갈리는 것:** 앞 구간들은 구간 방식과 정의에 따라 갈립니다. 신규를 2년 이내로 좁히면 뒤집힙니다(진입 총량이 아니라 "신규"의 이름표 문제).
# - **한계:** 이름 기반 식별은 ID로 직접 검증하지 못했습니다(2015–19에 ID 없음).

# 다음은 **PA-B4**(F1+F3 이분 등급, 145개 토픽으로 S4·S5·S6 재추정)를 권합니다. 입력 구조는 이미 확인됐고, 마스크 구현에 필요한 `s5_lib.sample_P`의 `tmask`와 S6의 `topic` 설정 지점도 소스로 봤습니다. 동결 코드가 필요하시면 말씀해 주세요. PA-C1(중국 소속 신규 저자 제외)은 저자 국가 정보가 어느 파일에 있는지(S7 입력)를 먼저 확인해야 합니다.

In [ ]:
# # PA-B4 설계: F1·F3 통과 토픽(145개)으로 S5·S6 재추정

# ## 먼저 알아 둘 것

# - **F1·F3 실패는 합쳐서 4개 토픽뿐**입니다(F1 2개, F3 2개, 겹치지 않으면 149−4=145). 토픽의 97%를 그대로 쓰는 재추정이라 결과가 거의 같을 가능성이 높습니다. "등급 선택이 결과를 좌우하지 않는다"는 점검으로는 쓸 만하지만, **센 반증 시험은 아닙니다.** 본문에도 그렇게 쓰세요.
# - **이번 계획에서 뺀 것**
#   - **S4(big_change 방향, 안정성):** 복구한 `s4_main.py`의 토픽 선택 로직을 제가 본 적이 없어서 추측으로 코드를 쓰지 않았습니다. 그 파일 출력을 주시면 `PA-B4b`로 따로 동결합니다.
#   - **임베딩 응집도와 부트스트랩 Jaccard 10퍼센타일:** 입력 파일 구조를 아직 못 봤습니다. 나중에 따로 다룹니다.
# - **확인된 점:** 토픽은 149개이고(PA-B2b의 `K=149`), F 열은 **True가 실패**입니다(0번 토픽: F2·F3 True, `n_fail=2`).

# ## 동결 내용

# - **S5:** `s5_lib.detect`를 쓰고, 벌점은 `c·φ·(K−1)·log T`에서 K=145입니다. φ는 S5 값을 고정합니다. 이는 S5가 `tmask`를 쓸 때의 방식과 같습니다. 귀무 N1·N2는 145개 부분 자료에서 새로 만들고(B=500), 통과 규칙은 S5와 같이 "N1·N2 모두 p<0.05"입니다.
# - **S6:** 주 설정에서 `R`을 145개 토픽 행으로 거릅니다. PA-A2a와 같은 대응 부트스트랩으로 `entry_minus_adapt`의 변화량을 냅니다.
# - **판독(고정)**
#   - S5: 표본별로 (a) 통과/실패가 원래(전부 실패)와 같은지, (b) 원래 변화점이 ±1년 안에서 유지되는지 따로 보고합니다.
#   - S6: 쌍별로 |변화|≤0.03이고 부호가 유지되면 "둔감"입니다.
# - **재현 점검(중단 조건):** 통과 토픽이 정확히 145개, 전체 토픽 마스크에서 S5 관측 R 재현, S6 `total_half`와 A2c `entry_share` 재현.

# ## 코드: `paper1/pa_b4.py`

# ```python
# # -*- coding: utf-8 -*-
# # PA-B4: F1+F3 이분 등급(통과 145개 토픽)으로 S5·S6 재추정. S4·응집도 지표는 제외(별도). 미실행 코드.
# # MODE: "freeze" -> "tag" -> (git push --follow-tags) -> "run"(QUICK=True: 오류 점검 전용, 수치 미출력) -> "run"(QUICK=False)
# # S0~S8 판정 불변. 모든 산출물 [Post-hoc].
# import sys, json, hashlib, subprocess, warnings
# from pathlib import Path
# import numpy as np, pandas as pd
# warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
# MODE = "freeze"
# QUICK = True

# ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; PA = P1/"09_PA"; S3 = P1/"03_S3"; S5D = P1/"05_S5"; S6D = P1/"06_S6"
# now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
# sha_b = lambda b: hashlib.sha256(b).hexdigest(); sha_f = lambda p: sha_b(Path(p).read_bytes())
# git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
# class Halt(Exception): pass
# spec_hash = lambda s: sha_b(json.dumps({k: v for k, v in s.items() if k not in ("sha256", "created", "refs")}, sort_keys=True, ensure_ascii=False).encode())
# V2P, BP = PA/"PA_0_PLAN_v2.json", PA/"PA_B4_PLAN.json"

# PLAN = dict(
#     version=1, seed=2026, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(PA-B4 중 S5·S6 재추정만). S3 요약의 'F2 제외 시 A 145개', L7 열 구조, s5_lib·s6_lib 소스, S5·S6·A2a 결과는 열람한 상태. PA-B4 결과 미열람. S4 재추정과 응집도·Jaccard 지표는 별도 계획(PA-B4b 등)으로 분리. S0~S8 판정 불변.",
#     grade_def="L7_topic_grade.csv의 F1, F3 열(True=실패). 통과 = F1 False and F3 False. 재현 점검: 통과 토픽 정확히 145개",
#     expectation_note="F1·F3 실패는 합쳐 4개 토픽뿐이라 결과가 거의 같을 것으로 예상(약한 시험). 이 예상은 설계 사실이며 판독 기준이 아님",
#     s5=dict(samples=["all", "fixed14", "within"], detector="s5_lib.detect 그대로, pen=c_primary*phi_s*(K-1)*log(T), K=145, phi_s·c_primary·k_max·min_len는 S5_0_PREREG.json 값 고정(K 변경에 따른 phi 재계산 안 함)",
#             data="s5_lib.load_counts()의 cnt에서 토픽 축을 145개로 부분집합. wbar는 S5와 같이 전체 cnt 기준",
#             nulls="N1(gen_N1), N2(trend_probs를 부분 cnt에서 재계산 후 gen_N2), B=500, p=(1+#{null_R>=obs_R})/(1+B), 통과 규칙=N1·N2 모두 p<0.05 (S5와 동일)",
#             reading="표본별 (a) 통과 여부가 S5(전 표본 실패)와 같은가, (b) S5 관측 변화점(S5_meta.json obs_breaks) 각각이 새 관측 변화점 ±1년 안에 있는가를 따로 보고. 둘 다 같으면 '등급 선택에 둔감', 아니면 '민감'(표본별 표기, 하나로 합치지 않음)"),
#     s6=dict(setting="spec['main']과 s6_lib.make_pair 그대로, R을 ti가 통과 토픽 인덱스인 행으로 제한(S6의 topic='A'와 같은 방식). 쌍은 pairs_of(spec) 전부",
#             stats="entry_share, adapt_share, entry_minus_adapt (PA-A2c·A2a와 같은 4항 분모, 진입=신규+복귀)",
#             boot="저자 가중 Exp(1)을 원본 저자 전체에서 뽑고 부분 자료의 같은 저자에게 같은 값 적용(대응 부트스트랩), B=spec['B_boot'], delta=sub-orig의 백분위·basic 구간",
#             reading="쌍별로 entry_minus_adapt의 |delta_obs|<=0.03 이고 부호 유지면 '둔감', 아니면 '민감'. 쌍별 표기, 합치지 않음. 구간은 병기만"),
#     reproduce_check="(a) 통과 토픽 145개, (b) 전체 토픽 마스크에서 S5 관측 R이 regime_nulls.csv obs_R과 1e-3 이내, (c) S6 orig total_half가 renewal_decomposition.csv와 1e-6 이내, (d) orig entry_share가 PA_A2c_results.csv(mode=author)와 1e-6 이내, (e) 부분 자료 저자 집합이 원본의 부분집합",
#     caveat="제외 토픽이 4개뿐이라 판별력이 약함. phi를 K=149 기준 S5 값으로 고정해 K=145에서 벌점 비율이 미세하게 어긋날 수 있음. 등급 자체의 타당성(F1·F3가 실제로 '좋은 토픽'을 가리는지)은 검증하지 않음",
#     forbidden="결과를 본 뒤 통과 정의·B·판독 기준 변경(변경 시 v2 계획으로 분리하고 사유 기록)")

# def freeze():
#     if not V2P.exists(): raise Halt("PA_0_PLAN_v2.json이 없습니다")
#     h = spec_hash(PLAN); print("PA-B4 계획 해시(예정)", h[:12])
#     if BP.exists(): print("이미 동결:", "동일" if json.loads(BP.read_text(encoding="utf-8"))["sha256"] == h else "⛔ 다름"); return
#     refs = {f.name: sha_f(f) for f in [V2P, S3/"L7_topic_grade.csv", S5D/"S5_0_PREREG.json", S5D/"regime_nulls.csv", S5D/"S5_meta.json", P1/"s5_lib.py", P1/"s6_lib.py", S6D/"renewal_decomposition.csv", PA/"PA_A2c_results.csv"] if f.exists()}
#     BP.write_text(json.dumps(dict(PLAN, created=now(), sha256=h, refs=refs), ensure_ascii=False, indent=1), encoding="utf-8")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f:
#         f.write(f"\n- {now()} | PA-B4 계획 동결(해시 {h[:12]}) | 사유: F1+F3 이분 등급(145개) 재추정, S5·S6만(S4·응집도는 별도). 제외 토픽이 4개뿐이라 약한 시험임을 명시. PA-B4 결과 미열람. QUICK은 수치 미출력 | 본 결과 변경 여부: 없음\n")
#     print("저장: PA_B4_PLAN.json")

# def tag():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     t = f"PA-B4-plan-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
#     if git("tag", "-l", t).stdout.strip(): print("이미 있음:", t); return
#     r = git("add", "--", "paper1/09_PA/PA_B4_PLAN.json", "paper1/09_PA/PA_CHANGELOG.md", "paper1/pa_b4.py"); print("add", r.returncode, r.stderr.strip())
#     r = git("commit", "-m", "PA-B4 plan freeze (before execution)"); print("commit", r.returncode, (r.stdout + r.stderr)[:200])
#     r = git("tag", "-a", t, "-m", "PA-B4 plan"); print("tag", r.returncode, r.stderr.strip()); print("※ 이어서 git push --follow-tags")

# def run():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     sp = json.loads(BP.read_text(encoding="utf-8"))
#     if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(PLAN): raise Halt("동결 파일과 현재 계획이 다릅니다")
#     if not git("tag", "-l", "PA-B4-plan-*").stdout.strip(): raise Halt("tag 단계를 먼저 실행하세요")
#     sys.path.insert(0, str(P1)); import s5_lib as L5; import s6_lib as S
#     sfx = "_QUICK" if QUICK else ""; rng = np.random.default_rng(int(PLAN["seed"]))
#     # ---- 통과 토픽 ----
#     G = pd.read_csv(S3/"L7_topic_grade.csv", encoding="utf-8-sig"); bl = lambda s: s.astype(str).str.strip().str.lower().isin(["true", "1", "1.0"])
#     ptop = set(G[~bl(G.F1) & ~bl(G.F3)].topic.astype(int))
#     if len(ptop) != 145: raise Halt(f"통과 토픽 수 재현 실패: {len(ptop)} (145여야 함)")
#     # ================= S5 =================
#     s5 = json.loads((S5D/"S5_0_PREREG.json").read_text(encoding="utf-8")); D = s5["detection"]; PHI = {k: v["phi"] for k, v in D["phi"].items()}
#     C0, KMAX, MINLEN, ALPHA = D["c_primary"], D["k_max"], D["min_len"], s5["nulls"]["alpha"]; B = 20 if QUICK else 500
#     cnt, venues, topics, fixed_v = L5.load_counts(); wbar = cnt.sum((0, 2))/cnt.sum(); T = len(L5.YEARS); YEARS = np.asarray(L5.YEARS); SAM = PLAN["s5"]["samples"]
#     tmask = np.isin(topics, list(ptop))
#     if int(tmask.sum()) != 145: raise Halt(f"topics_order와 통과 토픽 불일치: {int(tmask.sum())}")
#     def det(c_arr, s, Kuse):
#         P, n = L5.sample_P(c_arr, fixed_v, wbar, s); return L5.detect(P, n, C0*PHI[s]*(Kuse-1)*np.log(T), KMAX, MINLEN)
#     ref = pd.read_csv(S5D/"regime_nulls.csv"); ref = ref[ref["null"] == "N2"].set_index("sample")
#     for s in SAM:
#         if abs(det(cnt, s, len(topics))["R"] - float(ref.loc[s, "obs_R"])) > 1e-3: raise Halt(f"S5 관측 R 재현 실패({s})")
#     print("[재현 OK] S5 관측 R(전체 토픽), 통과 토픽 145개")
#     cs = cnt[:, :, tmask]; obs = {s: det(cs, s, 145) for s in SAM}
#     P2 = L5.trend_probs(cs); NR = {"N1": {s: [] for s in SAM}, "N2": {s: [] for s in SAM}}
#     for b in range(B):
#         for nm, gen in (("N1", lambda: L5.gen_N1(cs, rng)), ("N2", lambda: L5.gen_N2(cs, P2, rng))):
#             cc = gen()
#             for s in SAM: NR[nm][s].append(det(cc, s, 145)["R"])
#         if (b+1) % 100 == 0: print(f"  S5 귀무 {b+1}/{B}")
#     meta = json.loads((S5D/"S5_meta.json").read_text(encoding="utf-8")); ob0 = meta["obs_breaks"]; rows5 = []
#     for s in SAM:
#         yrs = [int(YEARS[x]) for x in obs[s]["br"]]; pv = {nm: (1 + float((np.array(NR[nm][s]) >= obs[s]["R"]).sum()))/(1 + B) for nm in NR}
#         passed = max(pv.values()) < ALPHA; kept = [L5.near(y, yrs, 1) for y in ob0[s]]
#         rows5.append(dict(sample=s, K=145, obs_R=obs[s]["R"], obs_k=obs[s]["k"], breaks=str(yrs), s5_breaks=str(ob0[s]), s5_breaks_kept_pm1=str(kept), p_N1=pv["N1"], p_N2=pv["N2"], passed_new=passed, passed_s5=bool(meta["null_pass"][s]),
#                           same_verdict=(passed == bool(meta["null_pass"][s])), all_breaks_kept=all(kept), B=B))
#     R5 = pd.DataFrame(rows5); R5.to_csv(PA/f"PA_B4_s5_results{sfx}.csv", index=False, encoding="utf-8-sig"); print("  S5 완료")
#     # ================= S6 =================
#     spec = S.load_spec(); L = S.load_all(spec); M = spec["main"]; PAIRS = S.pairs_of(spec); half = S.half; R = L["R"]
#     ok_ti = np.where(np.isin(L["topics"], list(ptop)))[0]
#     if len(ok_ti) != 145: raise Halt(f"S6 토픽 인덱스 수 불일치: {len(ok_ti)}")
#     L2 = dict(L); L2["R"] = R[R.ti.isin(ok_ti)]
#     ref6 = pd.read_csv(S6D/"renewal_decomposition.csv"); ref6 = ref6[(ref6.scheme == M["scheme"]) & (ref6.variant == "all_venues")].set_index("pair")
#     a2c = pd.read_csv(PA/"PA_A2c_results.csv"); a2c = a2c[(a2c["mode"] == "author") & (a2c.stat == "entry_share")].set_index("pair")
#     B6 = 20 if QUICK else int(spec["B_boot"]); lo_q, hi_q = spec["ci"]; STATS = ["entry_share", "adapt_share", "entry_minus_adapt"]
#     def stats(d):
#         h = {t: half(d[t]) for t in ["adapt", "realloc", "entry", "exit"]}; tot = sum(h.values())
#         return dict(entry_share=h["entry"]/tot, adapt_share=h["adapt"]/tot, entry_minus_adapt=(h["entry"] - h["adapt"])/tot)
#     rows6 = []
#     for p0, p1 in PAIRS:
#         name = f"{p0}->{p1}"; Pa = S.make_pair(L, p0, p1, M["scheme"], M["venue"], M["topic"], M["hom"]); Pb = S.make_pair(L2, p0, p1, M["scheme"], M["venue"], M["topic"], M["hom"])
#         da, db = Pa.decompose(), Pb.decompose(); oa, ob = stats(da), stats(db)
#         if abs(half(da["total"]) - float(ref6.loc[name, "total_half"])) > 1e-6: raise Halt(f"S6 관측 재현 실패({name})")
#         if abs(oa["entry_share"] - float(a2c.loc[name, "obs"])) > 1e-6: raise Halt(f"PA-A2c entry_share 재현 실패({name})")
#         idx = np.searchsorted(Pa.aids, Pb.aids)
#         if not (idx < len(Pa.aids)).all() or not (Pa.aids[idx] == Pb.aids).all(): raise Halt(f"부분 자료 저자 집합이 원본의 부분집합이 아님({name})")
#         bs = []
#         for _ in range(B6):
#             wa = rng.exponential(size=Pa.nA); sa = stats(Pa.decompose(Pa.m0*wa[Pa.a0], Pa.m1*wa[Pa.a1])); wb = wa[idx]; sb = stats(Pb.decompose(Pb.m0*wb[Pb.a0], Pb.m1*wb[Pb.a1]))
#             bs.append({s: sb[s] - sa[s] for s in STATS})
#         bs = pd.DataFrame(bs)
#         for s in STATS:
#             dl = ob[s] - oa[s]; ql, qh = np.percentile(bs[s], [lo_q, hi_q])
#             rows6.append(dict(pair=name, stat=s, obs_orig=oa[s], obs_sub=ob[s], delta=dl, boot_median=float(bs[s].median()), pct_lo=ql, pct_hi=qh, basic_lo=2*dl - qh, basic_hi=2*dl - ql,
#                               n_rows_orig=int(len(R[R.period.isin([p0, p1])])), n_rows_sub=int(len(L2["R"][L2["R"].period.isin([p0, p1])])), n_authors_orig=int(Pa.nA), n_authors_sub=int(Pb.nA), B=B6))
#         print(f"  S6 {name} 완료")
#     R6 = pd.DataFrame(rows6); R6.to_csv(PA/f"PA_B4_s6_results{sfx}.csv", index=False, encoding="utf-8-sig")
#     if QUICK: print("\n[QUICK] 오류 없이 완료. 수치는 출력하지 않음(_QUICK 파일을 열어 해석하지 마세요)."); return
#     print("\n[Post-hoc] PA-B4 S5 재추정(145개 토픽)\n", R5.round(4).to_string(index=False))
#     print("\n[Post-hoc] PA-B4 S6 재추정(145개 토픽 대 전체)\n", R6.round(4).to_string(index=False))
#     k = R6[R6.stat == "entry_minus_adapt"].copy(); k["insensitive"] = (np.sign(k.obs_sub) == np.sign(k.obs_orig)) & (k.delta.abs() <= 0.03)
#     print("\n[동결 판독] S5 표본별 (통과 여부 동일, 변화점 유지):\n", R5[["sample", "same_verdict", "all_breaks_kept"]].to_string(index=False))
#     print("[동결 판독] S6 entry_minus_adapt\n", k[["pair", "obs_orig", "obs_sub", "delta", "pct_lo", "pct_hi", "insensitive"]].round(4).to_string(index=False))
#     print("※ 제외 토픽 4개뿐인 약한 시험. 쌍·표본별로 따로 읽고 합치지 않음. S0~S8 판정 불변.")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as fh:
#         fh.write(f"- {now()} | PA-B4 실행 | 정의: PA_B4_PLAN {sp['sha256'][:12]} | 결과: PA_B4_s5_results.csv, PA_B4_s6_results.csv | 본 결과 변경 여부: 없음\n")

# try: {"freeze": freeze, "tag": tag, "run": run}[MODE]()
# except Halt as e: print(f"⛔ 중단: {e}")
# ```

# ## 실행 순서

# 앞서 `&&`로 묶지 않은 줄 때문에 기록 사고가 있었으므로, **오류가 나면 거기서 멈추고 출력만 붙여 주세요.** 아래 블록은 한 번에 붙여 넣지 말고 **단계별로** 실행하세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 pa_b4.py                                                            # 1) freeze: 해시 확인
# ```
# ```bash
# sed -i 's/^MODE = "freeze"/MODE = "tag"/' pa_b4.py && python3 pa_b4.py       # 2) tag
# cd .. && git push --follow-tags && cd paper1
# ```
# ```bash
# sed -i 's/^MODE = "tag"/MODE = "run"/' pa_b4.py
# ( time python3 -u pa_b4.py ) 2>&1 | tee 09_PA/PA_B4_quick_log.txt           # 3) QUICK (수치 미출력)
# ```
# ```bash
# sed -i 's/^QUICK = True/QUICK = False/' pa_b4.py && grep -n '^MODE\|^QUICK' pa_b4.py
# cd .. && git add paper1/pa_b4.py paper1/09_PA/PA_B4_*QUICK* paper1/09_PA/PA_B4_quick_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-B4: QUICK done; QUICK=False" && git tag -a PA-B4-fullrun-2026-10-21 -m "pre-run" && git push --follow-tags
# ```
# ```bash
# cd paper1 && nohup python3 -u pa_b4.py > 09_PA/PA_B4_full_log.txt 2>&1 &
# sleep 5; tail -f 09_PA/PA_B4_full_log.txt
# ```

# ## 주의할 점

# - **미실행 코드입니다.** 새로 쓴 부분은 부분 토픽 `cnt`에서의 S5 귀무 생성과 S6 행 거르기입니다. 재현 점검 5개가 막아 줍니다.
# - **가장 가능성 높은 중단 원인:** (a) 통과 토픽이 145개가 아닌 경우입니다. F1·F3 실패 토픽이 겹치면 146개가 나옵니다. 그때는 실제 수와 어떤 토픽인지만 알려 주세요. 이 경우 "145개"라는 문서상 수치가 틀렸던 것이고, 계획 동결을 새 버전으로 분리해야 합니다. (b) `G.F1`의 값 형식이 예상과 다른 경우입니다.
# - **S5 시간:** N1이 가장 무겁습니다. S5 원래 실행 시간은 제가 모르므로 본 실행은 백그라운드로 두었습니다. QUICK(B=20)의 소요 시간으로 본 실행을 대략 25배로 어림하세요.
# - **QUICK 로그에는 수치가 없습니다.** `_QUICK.csv`는 열지 마세요(이전 PA와 같은 원칙).
# - 동결 단계(`freeze`/`tag`)에서 PA_CHANGELOG에 줄이 자동 추가되므로, 실행 순서를 건너뛰지 마세요.

# QUICK 로그와 본 실행 로그를 붙여 주시면 동결 규칙대로 읽겠습니다.

In [ ]:
# # 네, 분석은 정상 종료됐습니다

# 로그에 `[Post-hoc] PA-B4 S5 재추정`, `S6 재추정`, `[동결 판독]`이 모두 나왔고 중단 메시지(`⛔`)나 오류가 없습니다. 다만 `pgrep -f pa_b4.py`로 프로세스가 남아 있지 않은지는 확인해 주세요.

# ## 동결 판독 결과

# **S5 (표본별, 3개 모두 동일)**
# - 통과 여부: 세 표본 모두 원래와 같이 **실패**(N2 p = 0.92 / 0.37 / 0.69, N1은 0.002).
# - 변화점: 원래 변화점이 전부 그대로 유지됐습니다(all 2012·2020, fixed14 2008·2015·2020, within 2009·2015·2020).
# - 판독: **등급 선택에 둔감**.

# **S6 (`entry_minus_adapt`, 쌍별)**

# | 쌍 | 원본 | 145개 | 변화 | 판독 |
# |---|---|---|---|---|
# | P0004→P0509 | −0.0055 | +0.0039 | +0.0094 | **부호 바뀜 → 규칙상 "민감"** |
# | P0509→P1014 | 0.0271 | 0.0267 | −0.0005 | 둔감 |
# | P1014→P1519 | 0.0461 | 0.0477 | +0.0016 | 둔감 |
# | P1519→P2025 | 0.1128 | 0.1113 | −0.0015 | 둔감 |

# ## 주의해서 읽을 점

# - 첫 쌍이 "민감"으로 찍힌 이유는 **부호 규칙** 때문입니다. 변화량은 0.0094로 기준(0.03)의 3분의 1 이하이고, 원래 값(−0.0055)이 0 바로 근처라 아주 작은 이동에도 부호가 넘어갑니다. 백분위 구간도 0을 포함합니다. 규칙은 바꾸지 않고 그대로 "민감"으로 보고하되, 이 사정을 같이 적어야 합니다. (앞서 PA-A2c·A2b에서도 첫 쌍은 0 근처였습니다.)
# - 가장 중요한 마지막 쌍은 변화 −0.0015로 사실상 그대로입니다.
# - 제외 토픽이 4개뿐인 **약한 시험**이므로 "등급 선택이 결과를 좌우하지 않는다"까지만 쓰세요. 등급 자체가 타당하다는 증거는 아닙니다.
# - S4 재추정과 응집도 지표는 이번 계획에 없었고, 별도 계획이 필요합니다.

# ## 다음 단계: 커밋·태그·푸시

# 해석 메모는 결과를 본 뒤이므로 아래처럼 기록하세요.

# ```bash
# cd ~/Research/Minnesota_Transport
# pgrep -f pa_b4.py || echo "종료 확인"
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B4 결과 해석 메모(S5 B=500, S6 B=2000): 통과 토픽 145개. S5 세 표본 모두 통과 여부 동일(전부 실패, p_N2 0.92/0.37/0.69)·변화점 전부 유지. S6 entry_minus_adapt 변화 +0.0094/-0.0005/+0.0016/-0.0015, 동결 규칙상 첫 쌍만 '민감'(원본 -0.0055가 0 근처라 부호가 바뀜, 변화량은 기준 0.03의 1/3 이하, 구간은 0 포함), 나머지 3쌍 둔감. 제외 토픽 4개뿐인 약한 시험, 등급 타당성은 미검증. S4 재추정·응집도 지표 미수행 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_B4_s5_results.csv paper1/09_PA/PA_B4_s6_results.csv paper1/09_PA/PA_B4_full_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-B4 results" && git tag -a PA-B4-results-2026-10-21 -m "PA-B4 outputs, seen" && git push --follow-tags
# ```

# 완료 후 다음 후보는 PA-C1(저자 국가 정보 위치 확인 필요), PA-B7, PA-B3 등입니다.

In [ ]:
# # PA-C2 설계와 동결 코드

# ## 확인된 것

# - 이번 출력으로 **재현 기준값**이 정해졌습니다. S7 주 설정(`inv_n_authors / main / all / terr_merge=False / income=fixed`)의 P1519는 `cn_known` 0.1373, `us_known` 0.3116이고, P2025는 0.3476, 0.1910입니다.
# - 국가 층은 `country_mass_long`에서 2007년 이후 전체 논문을 쓰고, 분석 저자 층(S6)은 임베딩 보유 논문(`has_topic`)만 씁니다. C2는 이 모집단 차이가 국가 몫을 얼마나 움직이는지 봅니다.

# ## 동결 내용

# - **정의**: 위 주 설정의 `country_mass_long`을 `F1_papers.csv`와 `pid`로 조인해 S6 분석 집합(제외 목록 제거, `in_sample`, `has_topic` 참, `topic150` 있음)에 속한 논문만 남기고, S7과 같은 방식으로 기간별 통계를 다시 계산합니다.
# - **통계**: `top5_known`, `hhi_known`, `cn_known`, `us_known`, `unk`. `n_active`와 `top5_incl`은 총량이 줄어드는 영향이 있어 병기만 합니다.
# - **판독(고정)**: 기간·통계별 |변화| ≤ 0.01이면 "일치", 아니면 "차이"로 따로 표기하고 합치지 않습니다.
# - **진단**: 기간별 논문 수·질량 보존율, 걸러진 쪽과 남은 쪽의 `unk` 비율.
# - **재현 점검(중단 조건)**: (a) 거르기 전 통계가 `s7_period_stats.csv` 주 설정 행과 1e-9 이내로 일치, (b) S6 분석 집합이 2000–2025에서 정확히 76,984편, (c) 거른 뒤 논문당 질량 합이 1.
# - 이 분석은 무작위가 없는 결정적 계산이라 QUICK은 재현 점검과 논문 수 출력만 하고 결과 수치는 출력하지 않습니다.

# ## 코드: `paper1/pa_c2.py`

# ```python
# # -*- coding: utf-8 -*-
# # PA-C2: S7 국가 몫을 임베딩 보유 논문(S6 분석 집합)만으로 재계산. 결정적 계산. 미실행 코드.
# # MODE: "freeze" -> "tag" -> (git push --follow-tags) -> "run"(QUICK=True: 재현 점검과 논문 수만, 결과 수치 미출력) -> "run"(QUICK=False)
# # S0~S8 판정 불변. 모든 산출물 [Post-hoc].
# import json, hashlib, subprocess, warnings
# from pathlib import Path
# import numpy as np, pandas as pd
# warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
# MODE = "freeze"
# QUICK = True

# ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; PA = P1/"09_PA"
# DATA, S0, S2, S7 = ROOT/"analysis_data/final", P1/"00_S0", P1/"02_S2", P1/"07_S7"
# now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
# sha_b = lambda b: hashlib.sha256(b).hexdigest(); sha_f = lambda p: sha_b(Path(p).read_bytes())
# tf = lambda s: s.astype(str).str.strip().str.lower().isin(["1", "1.0", "true", "t", "yes"])
# git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
# class Halt(Exception): pass
# spec_hash = lambda s: sha_b(json.dumps({k: v for k, v in s.items() if k not in ("sha256", "created", "refs")}, sort_keys=True, ensure_ascii=False).encode())
# V2P, BP = PA/"PA_0_PLAN_v2.json", PA/"PA_C2_PLAN.json"

# PLAN = dict(
#     version=1, parent="PA_0_PLAN_v2.json",
#     status="사후 명세. S7 결과와 s7_run.py 일부 소스, s7_period_stats.csv 주 설정 행(P1519 cn 0.1373, P2025 cn 0.3476 등)은 열람한 상태. PA-C2 결과 미열람. S0~S8 판정 불변.",
#     setting="S7 주 설정: scheme=inv_n_authors, variant=main, sample=all, terr_merge=False, income=fixed. country_mass_long에서 year>=2007, 기간은 S7_0_PREREG.json의 periods",
#     subset="F1_papers.csv에서 EXCLUDE_PIDS_FROZEN.csv 제외(FREEZE_LOG 해시 검증), in_sample 참, 연도 2000~2025, has_topic 참, topic150 결측 아님. 이 pid 집합에 속한 country_mass_long 행만 유지. 재현 점검: 이 집합이 2000~2025에서 76,984편",
#     stats="기간별 top5_known, hhi_known, cn_known, us_known, unk (S7 stats_of와 같은 정의: UNK·ZZ_former를 미상으로, 알려진 국가 질량 기준). 병기만: n_active(질량>=1 국가 수), top5_incl",
#     reading="기간x통계별 |filtered - original| <= 0.01 이면 '일치', 아니면 '차이'. 기간별·통계별로 따로 보고하고 합치지 않음",
#     diagnostics="기간별 원본·잔존 논문 수와 질량, 원본에서 걸러진 쪽과 남은 쪽의 unk 비율",
#     reproduce_check="(a) 거르기 전 기간 통계가 s7_period_stats.csv 주 설정 행과 1e-9 이내(n_active는 일치), (b) S6 집합 76,984편, (c) 거른 뒤 논문당 질량 합 1",
#     caveat="이 분석은 보유 논문 부분집합이 국가 몫을 바꾸는지만 본다. 소속은 국적이 아님. 국가 층 논문 중 F1에 없는 pid가 있으면 걸러진 쪽에 포함되어 보고됨",
#     forbidden="결과를 본 뒤 설정·판독 기준(0.01) 변경(변경 시 v2 계획으로 분리하고 사유 기록)")

# def freeze():
#     if not V2P.exists(): raise Halt("PA_0_PLAN_v2.json이 없습니다")
#     h = spec_hash(PLAN); print("PA-C2 계획 해시(예정)", h[:12])
#     if BP.exists(): print("이미 동결:", "동일" if json.loads(BP.read_text(encoding="utf-8"))["sha256"] == h else "⛔ 다름"); return
#     refs = {f.name: sha_f(f) for f in [V2P, S7/"S7_0_PREREG.json", S7/"s7_period_stats.csv", P1/"s7_run.py"] if f.exists()}
#     BP.write_text(json.dumps(dict(PLAN, created=now(), sha256=h, refs=refs), ensure_ascii=False, indent=1), encoding="utf-8")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f:
#         f.write(f"\n- {now()} | PA-C2 계획 동결(해시 {h[:12]}) | 사유: S7 국가 몫(전체 논문)과 S6 저자 층(임베딩 보유 논문)의 모집단 불일치 점검. PA-C2 결과 미열람. QUICK은 수치 미출력 | 본 결과 변경 여부: 없음\n")
#     print("저장: PA_C2_PLAN.json")

# def tag():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     t = f"PA-C2-plan-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
#     if git("tag", "-l", t).stdout.strip(): print("이미 있음:", t); return
#     r = git("add", "--", "paper1/09_PA/PA_C2_PLAN.json", "paper1/09_PA/PA_CHANGELOG.md", "paper1/pa_c2.py"); print("add", r.returncode, r.stderr.strip())
#     r = git("commit", "-m", "PA-C2 plan freeze (before execution)"); print("commit", r.returncode, (r.stdout + r.stderr)[:200])
#     r = git("tag", "-a", t, "-m", "PA-C2 plan"); print("tag", r.returncode, r.stderr.strip()); print("※ 이어서 git push --follow-tags")

# def st(vec, cols):
#     unk = np.isin(cols, ["UNK", "ZZ_former"]); kn = np.where(unk, 0.0, vec); kt = kn.sum(); tot = vec.sum(); top = np.sort(kn)[::-1][:5].sum()
#     return dict(top5_incl=top/tot, top5_known=top/kt, hhi_known=float(((kn/kt)**2).sum()), cn_known=float(kn[cols == "CN"].sum()/kt),
#                 us_known=float(kn[cols == "US"].sum()/kt), unk=float(1 - kt/tot), n_active=int((kn >= 1.0).sum()))
# def period_stats(d, PER):
#     G = d.pivot_table(index="year", columns="country", values="w", aggfunc="sum", fill_value=0); cols = np.array(G.columns)
#     return {k: st(G.loc[a:b].sum(0).values, cols) for k, (a, b) in PER.items()}

# def run():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     sp = json.loads(BP.read_text(encoding="utf-8"))
#     if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(PLAN): raise Halt("동결 파일과 현재 계획이 다릅니다")
#     if not git("tag", "-l", "PA-C2-plan-*").stdout.strip(): raise Halt("tag 단계를 먼저 실행하세요")
#     sfx = "_QUICK" if QUICK else ""
#     PER = {k: tuple(v) for k, v in json.loads((S7/"S7_0_PREREG.json").read_text(encoding="utf-8"))["periods"].items()}
#     pm = S2/"country_mass_long.parquet"
#     M = pd.read_parquet(pm) if pm.exists() else pd.read_csv(S2/"country_mass_long.csv.gz", dtype={"pid": str})
#     M["pid"] = M.pid.astype(str); M = M[(M.year >= 2007) & (M.scheme == "inv_n_authors") & (M.variant == "main")][["pid", "year", "country", "w"]]
#     # S6 분석 집합
#     F1 = pd.read_csv(DATA/"F1_papers.csv", low_memory=False); F1["pid"] = F1.pid.astype(str)
#     exf = S0/"EXCLUDE_PIDS_FROZEN.csv"; log = pd.read_csv(S0/"FREEZE_LOG.csv"); last = log[log.file == exf.name].iloc[-1]
#     if sha_f(exf) != last.sha256: raise Halt("EXCLUDE_PIDS_FROZEN.csv 해시 불일치")
#     F1 = F1[~F1.pid.isin(set(pd.read_csv(exf, dtype=str, encoding="utf-8-sig").pid))]
#     keep = F1[tf(F1.in_sample) & F1.year.between(2000, 2025) & tf(F1.has_topic) & F1.topic150.notna()]
#     if len(keep) != 76984: raise Halt(f"S6 분석 집합 재현 실패: {len(keep)} (76984여야 함)")
#     kp = set(keep.pid)
#     # (a) 재현
#     ref = pd.read_csv(S7/"s7_period_stats.csv"); ref = ref[(ref.scheme == "inv_n_authors") & (ref.variant == "main") & (ref["sample"] == "all") & (ref.terr_merge == False) & (ref.income == "fixed")].set_index("period")
#     ST0 = period_stats(M.groupby(["year", "country"], as_index=False).w.sum(), PER)
#     for k in PER:
#         for s in ["top5_known", "hhi_known", "cn_known", "us_known", "unk"]:
#             if abs(ST0[k][s] - float(ref.loc[k, s])) > 1e-9: raise Halt(f"S7 재현 실패({k}, {s})")
#         if ST0[k]["n_active"] != int(ref.loc[k, "n_active"]): raise Halt(f"S7 n_active 재현 실패({k})")
#     Mk = M[M.pid.isin(kp)]
#     chk = Mk.groupby("pid").w.sum()
#     if not np.allclose(chk, 1.0): raise Halt("거른 뒤 논문당 질량 합이 1이 아님")
#     print("[재현 OK] S7 주 설정 기간 통계, S6 집합 76,984편, 질량 보존")
#     diag = []
#     for k, (a, b) in PER.items():
#         o, f = M[M.year.between(a, b)], Mk[Mk.year.between(a, b)]; dr = o[~o.pid.isin(kp)]
#         un = lambda x: float(x.w[x.country.isin(["UNK", "ZZ_former"])].sum()/x.w.sum()) if len(x) else np.nan
#         diag.append(dict(period=k, papers_orig=int(o.pid.nunique()), papers_kept=int(f.pid.nunique()), kept_share=float(f.pid.nunique()/o.pid.nunique()), unk_orig=un(o), unk_kept=un(f), unk_dropped=un(dr)))
#     DG = pd.DataFrame(diag)
#     if QUICK:
#         print("논문 수(진단)\n", DG[["period", "papers_orig", "papers_kept", "kept_share"]].round(4).to_string(index=False))
#         print("\n[QUICK] 재현 점검과 논문 수만 출력. 결과 통계는 계산·출력하지 않음."); return
#     ST1 = period_stats(Mk.groupby(["year", "country"], as_index=False).w.sum(), PER); rows = []
#     for k in PER:
#         for s in ["top5_known", "hhi_known", "cn_known", "us_known", "unk", "n_active", "top5_incl"]:
#             d = ST1[k][s] - ST0[k][s]; core = s in ["top5_known", "hhi_known", "cn_known", "us_known", "unk"]
#             rows.append(dict(period=k, stat=s, orig=ST0[k][s], kept=ST1[k][s], delta=d, core=core, verdict=("일치" if abs(d) <= 0.01 else "차이") if core else "병기"))
#     RS = pd.DataFrame(rows); RS.to_csv(PA/"PA_C2_results.csv", index=False, encoding="utf-8-sig"); DG.to_csv(PA/"PA_C2_diagnostics.csv", index=False, encoding="utf-8-sig")
#     print("\n[Post-hoc] PA-C2 결과(보유 논문만 대 전체)\n", RS.round(4).to_string(index=False))
#     print("\n[Post-hoc] 진단\n", DG.round(4).to_string(index=False))
#     print("※ 기간·통계별로 따로 읽고 합치지 않음. 소속은 국적 아님. S0~S8 판정 불변.")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as fh:
#         fh.write(f"- {now()} | PA-C2 실행 | 정의: PA_C2_PLAN {sp['sha256'][:12]} | 결과: PA_C2_results.csv, PA_C2_diagnostics.csv | 본 결과 변경 여부: 없음\n")

# try: {"freeze": freeze, "tag": tag, "run": run}[MODE]()
# except Halt as e: print(f"⛔ 중단: {e}")
# ```

# ## 실행 순서

# 한 블록씩 실행하고, 오류나 `⛔`가 나오면 거기서 멈추고 출력만 붙여 주세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 pa_c2.py                                                            # 1) freeze
# ```
# ```bash
# sed -i 's/^MODE = "freeze"/MODE = "tag"/' pa_c2.py && python3 pa_c2.py       # 2) tag
# cd .. && git push --follow-tags && cd paper1
# ```
# ```bash
# sed -i 's/^MODE = "tag"/MODE = "run"/' pa_c2.py
# ( time python3 -u pa_c2.py ) 2>&1 | tee 09_PA/PA_C2_quick_log.txt           # 3) QUICK: 재현 점검과 논문 수만
# ```
# ```bash
# sed -i 's/^QUICK = True/QUICK = False/' pa_c2.py && grep -n '^MODE\|^QUICK' pa_c2.py
# cd .. && git add paper1/pa_c2.py paper1/09_PA/PA_C2_quick_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-C2: QUICK done; QUICK=False" && git tag -a PA-C2-fullrun-2026-10-21 -m "pre-run" && git push --follow-tags
# ```
# ```bash
# cd paper1 && ( time python3 -u pa_c2.py ) 2>&1 | tee 09_PA/PA_C2_full_log.txt   # 4) 본 실행
# ```

# ## 주의할 점

# - **미실행 코드**입니다. 재현 점검 3개가 막아 줍니다. 가장 가능성 있는 중단은 `S6 분석 집합 재현 실패`입니다. S6의 `load_all`은 `in_sample`, 연도 2000~data_end, `has_topic`을 쓰는데, 제가 같은 조건을 옮겼지만 `W.period`나 IPW 결측 처리 같은 세부는 보지 못했습니다. 이 경우 실제 수만 알려 주세요. 임의로 조건을 고치지 마세요.
# - QUICK 출력의 논문 수는 설계 진단이며 결과 통계가 아닙니다. 그래도 열람 사실은 CHANGELOG 해석 메모에 한 줄 적어 두는 편이 안전합니다.
# - 계산이 결정적이고 가벼워 몇 분 안에 끝날 것으로 봅니다(정확한 시간은 모릅니다).

# ## 다음

# C2 뒤에 PA-C1을 설계합니다. 마지막 쌍만 대상이고, S6는 `equal` 가중, S7은 `inv_n_authors` 가중을 쓰므로 "중국 소속" 정의를 어느 쪽 가중으로 잴지 그때 정하겠습니다.

In [ ]:
# # 논문 1 사후분석(PA) 현황 정리 (2026-10-21)

# ## 1. 원칙 (변하지 않는 것)
# - S0~S8의 통과/실패 판정은 PA 결과로 바꾸지 않습니다.
# - 모든 PA는 실행 전에 계획을 해시로 동결하고, 태그를 GitHub에 올려 외부 시각을 남깁니다.
# - 결과는 방향과 상관없이 전부 보고하고, 표·그림에 `[Post-hoc]`를 붙입니다.
# - QUICK 실행은 오류 점검 전용입니다(수치는 해석하지 않음).

# ## 2. 지금까지 한 분석과 결과

# | PA | 질문 | 동결 판독 결과 | 쓸 수 있는 문장 / 주의 |
# |---|---|---|---|
# | **PA-0** 재현성 | 파일이 몰래 바뀌지 않았나 | 해시 152건 전부 일치. S4 코드 복구, 잘못된 태그 삭제·기록, 전부 푸시 | `oa_raw` **외부 백업은 미완료**(외장 매체 없음). 중간에 거짓 기록이 올라갔다가 정정 줄로 바로잡음(커밋 97178b3) |
# | **A1** 검정력 | 점프가 있었다면 N2 검사가 잡았을까 | run 1은 주입 버그로 무효(보존). run 2: all·within "강화", fixed14 "곡선만 보고". 탐색: 2020은 대체로 탐지, 2008~2012는 사각지대. 연속 꺾은선 BIC는 2020을 꺾임으로 선택 안 함 | "2020 부근에서 초과분 크기 점프가 있었다면 탐지됐을 가능성이 높다"까지. "체제 전환 부재 확인"은 금지 |
# | **B2** 2020 표본 크기 | 논문 수 증가의 착시인가 | 세 표본 모두 "유지" | 하향 추출이 20%뿐이라 약한 시험. "표본 크기 차이만으로는 사라지지 않는다"까지 |
# | **B2b** 저널·코로나 | 특정 저널/주제 때문인가 | 상위 1·3·5 저널 제외, 토픽 71 제외 각각은 유지. **둘을 동시에 제외하면** 점프 약화·변화점 불안정 | 2020은 단일 원인이 아닌 겹친 가속 |
# | **B5** S4 안정성 | 0.700이 경계인가 | 컷에 따라 0.66~0.77, 부호 반전 0 | "0.70 전후에서 움직임", 방향 축은 견고 |
# | **A2c** 부트스트랩 | 진입>적응이 구간 보정 후에도 유지되나 | 4쌍 중 3쌍(첫 쌍 제외). 두 구간 방식 모두 확실한 곳은 마지막 쌍뿐. excess 구간은 교정 불충분 | 초과 진입분 근거는 순열 대조 |
# | **A2b** 신규 정의 | "신규"를 바꾸면? | s6 3쌍, y5 3쌍, **y2 0쌍(뒤집힘)** | "신규 저자 주도"가 아니라 **"진입(신규+복귀) 주도"**로 표현 |
# | **A2a** 이름 분할 | ID가 갈라지는 이름을 빼면? | 4쌍 모두 "둔감"(변화 ≤0.011), 마지막 쌍 0.113→0.122 | ID는 2020+만 있어 "ID 기반 재판별"은 불가. 분할되지 않은 동명이인은 미검출 |
# | **B4** 등급 단순화 | F1+F3 통과 145개로 재추정 | S5: 3표본 모두 같은 판정·변화점 유지. S6: 3쌍 둔감, **첫 쌍만 "민감"**(원값 −0.0055가 0 근처라 부호만 바뀜, 변화량 0.009) | 제외 토픽 4개뿐인 약한 시험. S4 재추정·응집도 지표는 미수행 |

# ## 3. 종합 결론 (현재 논문이 말할 수 있는 것)
# - **쓸 수 있음:** 주제 구조 변화는 점프보다 연속적 재배치로 서술하는 편이 자료에 부합. 2020 부근은 코로나와 기술 주제가 겹친 가속. 진입이 적응보다 큰 방향은 마지막 구간(2015–19→2020–25)에서 가장 견고.
# - **쓰면 안 됨:** 체제 전환 부재 확정, 2020이 아티팩트가 아님(확정), 신규 저자가 모든 구간을 주도, ID 기반 결과와 일치, 인과 해석.

# ## 4. 진행 중
# - **PA-C2** (S7 국가 몫을 임베딩 보유 논문만으로 재계산, ±0.01 이내인지): 설계와 동결 코드(`pa_c2.py`)를 드린 상태이며 **아직 실행 전**입니다.

# ## 5. 앞으로 할 분석

# | 순서 | 항목 | 내용 | 선행 조건 |
# |---|---|---|---|
# | 1 | **C2 실행** | 위 코드를 freeze→tag→QUICK→본 실행 | 없음 |
# | 2 | **C1** | 마지막 쌍에서 중국 소속 저자를 빼도 진입 우세가 남는지 (a: 신규만 제외, b: 신규·지속 모두 제외) | "중국 소속" 정의와 가중치(S6 equal vs S7 inv_n) 결정 |
# | 3 | **B7** | 임베딩 없는 논문 16%의 최악·최선 경계 분석 | F1/S2 파일 구조 확인 |
# | 4 | **B3** | 신생 토픽 정의 4종의 핵심 집합 | F6 생성 코드 위치, `BASE` 정의 |
# | 5 | **A3** | S7 국가 미상 교정, 균등 4구간, 중국 제외 HHI | 격자 재집계 |
# | 6 | **B9, B8** | 소득군 내 상관·더미 통제, 미달 4개 도메인 부록 정렬 | S8 입력 확인 |
# | 7 | **B4b** | S4 재추정, 응집도·Jaccard 지표 | `s4_main.py` 토픽 선택 로직 확인 |
# | 8 | **B6** | ORCID 소표본, 수작업 50쌍 | 연구자 시간 |
# | 9 | **A4 + 마무리** | 열람 이력 부록 표(s5_lib 소스·N2 요약, B2 QUICK 출력 등 빠짐없이), 기각된 PA 목록, `[Post-hoc]` 표지, 영문 문구 | 모든 PA 종료 후 |

# ## 6. 병행·잔여 사항
# - `oa_raw` 외부 백업: 매체 연결 후 해시 일치 확인(`&&`로 이어 기록).
# - GitHub 저장소 Private 여부 확인.
# - 제목에서 "체제 전환"을 빼고 "진입 주도 갱신 + 선두 국가 교체"를 중심에 두는 구성 권장.
# - S9·S10(후보 풀 동결, 저널 선례, 심사 대응표, 생성형 AI 사용 선언)은 PA 종료 뒤.

# 가장 가까운 다음 한 걸음은 `pa_c2.py`를 저장해 1단계(`python3 pa_c2.py`)부터 실행하는 것입니다.

In [ ]:
# # PA-C2 판독

# 실행은 정상 종료됐고 재현 점검 3개를 모두 통과했습니다. 아래는 동결 규칙(|변화| ≤ 0.01이면 "일치")을 그대로 적용한 읽기입니다.

# ## 1. 동결 판독 결과 (핵심 5개 통계 × 4개 기간 = 20칸)

# | 통계 | P0709 | P1014 | P1519 | P2025 |
# |---|---|---|---|---|
# | top5_known | 차이 (+0.027) | 차이 (+0.022) | 차이 (+0.022) | 차이 (+0.023) |
# | hhi_known | 차이 (+0.027) | 차이 (+0.015) | 일치 (+0.009) | 일치 (+0.009) |
# | **cn_known** | 일치 (−0.002) | 일치 (+0.001) | 일치 (+0.006) | 일치 (+0.005) |
# | us_known | 차이 (+0.030) | 차이 (+0.019) | 차이 (+0.012) | 차이 (+0.014) |
# | unk | 일치 (−0.0099) | 차이 (−0.0102) | 일치 (−0.006) | 일치 (−0.004) |

# 20칸 중 **일치 11, 차이 9**입니다. 기간별·통계별로 따로 읽고 합치지 않는다는 규칙대로, "전체가 일치한다"고 쓸 수 없습니다.

# ## 2. 읽기

# - **중국 몫은 4개 기간 모두 일치합니다.** 마지막 기간 0.348→0.352로, 선두 국가 교체의 한 축(중국 상승)은 모집단 선택에 둔감합니다.
# - **미국 몫은 4개 기간 모두 "차이"이고 방향이 일정합니다.** 보유 논문만 쓰면 미국 몫이 0.012~0.030 커집니다. 마지막 기간 0.191→0.205입니다. 미국 하락(0.458→0.191)이라는 큰 흐름은 유지되지만, 수준은 모집단에 따라 조금 달라집니다.
# - **상위 5개국 몫(top5_known)도 4개 기간 모두 약 +0.02 높아집니다.** 이는 미국 몫이 커지는 것과 같은 방향입니다. 원인은 확인하지 않았습니다(임베딩 보유 논문이 미국에 치우쳤을 가능성은 가설).
# - **진단:** 걸러진 쪽(임베딩 없는 논문)이 국가 미상 비율이 더 높습니다(예: P1519 0.092 대 0.046). 걸러진 쪽의 국가 구성이 다르다는 뜻이고, 위 방향성과 맞습니다. `unk`의 P1014 "차이"는 −0.0102로 기준 0.01을 간발로 넘은 경우입니다(P0709는 −0.0099로 간발로 일치). 경계 칸이라 "일치/차이" 이분법에 크게 무게를 두지 마세요.
# - 보존율은 77~87%이며, 앞서 알려진 임베딩 보유율과 같은 범위입니다. `n_active`는 총량 감소 탓에 줄어든 것이라 해석하지 않습니다.

# ## 3. 쓸 수 있는 문구

# > "국가 몫을 임베딩 보유 논문만으로 다시 계산하면, 중국 몫은 모든 기간에서 0.01 이내로 같았고(마지막 기간 0.348→0.352), 미국 몫과 상위 5개국 몫은 0.01~0.03 높아졌다. 방향(미국 감소, 중국 증가)은 두 집합에서 같다."

# 마지막 문장의 "방향이 같다"는 기간 간 변화를 직접 계산한 값이 아니라 위 표의 수준 비교에서 읽은 것입니다. 본문에 쓰려면 기간 간 변화량도 확인하세요. 쓰면 안 되는 것은 "두 집합의 결과가 일치한다"입니다. 이는 S6 저자 층과 S7 국가 층을 한 문장으로 엮을 때 "수준은 ±0.03 이내에서 다르다"고 밝혀야 한다는 뜻입니다.

# ## 4. 기록

# ```bash
# cd ~/Research/Minnesota_Transport
# pgrep -f pa_c2.py || echo "종료 확인"
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-C2 결과 해석 메모(결정적 계산): 핵심 20칸 중 일치 11·차이 9. cn_known은 4기간 모두 일치(마지막 0.3476->0.3524). us_known은 4기간 모두 차이(+0.012~+0.030), top5_known도 4기간 모두 +0.02 안팎 상승, 방향 일정(원인 미확인). unk는 P1014가 -0.0102로 경계. 걸러진 논문의 unk 비율이 더 높음(P1519 0.092 대 0.046). QUICK 출력에 논문 수 진단(보존 0.77~0.87)이 포함되어 열람함. 문구는 '중국 몫 일치, 미국·상위5 몫은 0.01~0.03 상승'으로 제한 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_C2_results.csv paper1/09_PA/PA_C2_diagnostics.csv paper1/09_PA/PA_C2_full_log.txt paper1/09_PA/PA_CHANGELOG.md paper1/pa_c2.py
# git commit -m "PA-C2 results" && git tag -a PA-C2-results-2026-10-21 -m "PA-C2 outputs, seen" && git push --follow-tags
# ```

# ## 5. 다음 단계

# **PA-C1**로 갑니다. 마지막 쌍(2015–19→2020–25)만 대상이고, 결정할 것이 두 가지입니다.

# 1. **"중국 소속" 정의:** 후보는 "t1 기간 중국 몫 ≥ 0.5인 저자"입니다. 가중치는 S7의 `inv_n_authors` 계열 질량표(저자×지역군)로 재는 안과, S6의 `equal`로 재는 안이 있습니다. S6 분해 위에서 제외하는 분석이므로 S6와 같은 `equal` 기준을 권하지만, 국가는 S7 코드(`author_rows`)의 `cs_main` 열에서 가져와야 합니다.
# 2. **시나리오:** (a) 중국 소속 신규 저자만 제외, (b) 중국 소속 저자를 신규·지속 모두 제외. 둘 다 동결합니다.

# 바로 코드를 쓰려면 이름 `name_norm`별 국가 대응이 만들어지는 위치와 형태가 필요합니다. 아래 출력을 붙여 주세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# sed -n 260,310p s7_run.py
# python3 - <<'EOF'
# import pandas as pd
# A = pd.read_csv("01_S1/authorship_level.csv.gz", nrows=5) if __import__("os").path.exists("01_S1/authorship_level.csv.gz") else pd.read_parquet("01_S1/authorship_level.parquet").head(5)
# print(A.to_string())
# EOF
# ```

In [ ]:
# # PA-C2 기록 확인과 PA-C1 설계 점검

# ## 1. 기록

# 커밋 `1a015a8`과 태그 `PA-C2-results-2026-10-21`이 원격에 올라갔습니다. `pgrep`에서 프로세스가 없어 정상 종료입니다. 끝난 PA는 A1, A2a·b·c, B2, B2b, B4(S5·S6 부분), B5, C2입니다.

# ## 2. 이번 출력으로 확인된 것

# - **저자-국가 대응은 S7의 `author_rows`로 만들어집니다.** `authorship_level`(열: `pid, rank, oa_author_id, countries, raw_aff, aligned`)에 `authorship_level_recovered`를 붙이고, `paper_author_long`(`PAL`)과 `pid + author_pos(=rank+1)`로 합쳐 `name_norm`을 얻습니다. 즉 이름 `name_norm`이 S6의 `R.aid`와 같은 키라서 두 층을 이을 수 있습니다.
# - **`countries`는 한 행에 여러 국가가 `|`로 붙을 수 있고**, 행 하나가 논문 안의 저자 한 명입니다. S7은 `1/n_authors`를 국가 수로 나눠 균등 분할하고, 국가 미상은 `UNK`로 둡니다.
# - 앞선 출력의 2번째 행(`countries`에 CN이 아니라 영문 소속 문자열이 들어 있음)은 **열 값이 어긋난 것처럼 보입니다**(`countries`에 기관명, `raw_aff`는 기관명). `head`를 `to_string`으로 찍을 때 `||` 구분자가 섞였을 가능성이 있으나, 확인하지 않았습니다. 이 점은 C1 코드에서 국가 코드 형식(2글자)만 쓰도록 검증하고, 아니면 `UNK`로 두는 점검을 넣겠습니다(S7 원본 방식과 같은지는 `author_rows` 앞부분 L228 이전 코드를 못 봐서 모릅니다).

# ## 3. PA-C1을 동결하기 전에 정해야 할 것

# 1. **"중국 소속" 정의(제안):** 마지막 기간(P2025, 2020–25)에서 저자의 국가 질량 중 `CN`(HK·MO 제외)이 0.5 이상. 질량은 S7과 같은 방식(논문당 `1/n`, 국가 균등 분할)으로 저자별 합산. 기간 안에 출판이 없는 저자(퇴출)는 대상이 아닙니다.
# 2. **시나리오(동결):**
#    - (a) 중국 소속 **신규** 저자(E_new, E_ret 포함 여부는 분리 보고)만 제외
#    - (b) 중국 소속 저자를 **신규·지속 모두** 제외(대칭 대조)
# 3. **통계:** PA-A2c와 같은 4항 분모의 `entry_share`, `adapt_share`, `entry_minus_adapt`. 대응 부트스트랩, B=spec["B_boot"].
# 4. **판독(제안):** (a)(b) 각각에서 `entry_minus_adapt`이 0보다 크고 basic 구간 하한도 0보다 크면 "진입 우세 유지". 변화량은 병기만 합니다. (a)는 진입 질량이 기계적으로 줄어드므로 "유지"가 약한 시험이 아니라 오히려 강한 시험이라는 점, (b)는 더 공정한 대조라는 점을 계획서 `status`에 적습니다.
# 5. **대상:** 마지막 쌍(P1519→P2025)만. S6 `equal`·S7 `inv_n_authors` 가중 차이는 "중국 소속" 판정에 `inv_n_authors`(S7 방식)를 쓰고, 분해는 S6 `equal`로 두는 혼합이며, 계획서에 명시하겠습니다.

# ## 4. 코드를 쓰기 위해 아직 필요한 것

# `author_rows` 앞쪽, 즉 `countries` 문자열이 어떻게 정리되는지와 `UNK` 처리가 보이지 않습니다. 아래 출력을 붙여 주세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# sed -n 222,232p s7_run.py
# python3 - <<'EOF'
# import pandas as pd, os
# p = "01_S1/authorship_level.parquet"
# A = pd.read_parquet(p) if os.path.exists(p) else pd.read_csv("01_S1/authorship_level.csv.gz")
# print(A.dtypes)
# print(A.countries.fillna("").str.len().describe())
# print(A.countries.fillna("").value_counts().head(8))
# print("2글자 코드 형식 비율:", A.countries.fillna("").str.fullmatch(r"([A-Z]{2}(\|[A-Z]{2})*)?").mean().round(4))
# EOF
# ```

# 이 출력으로 국가 열 형식이 깨끗한지 확인하고 나서 `pa_c1.py`를 동결 코드로 드리겠습니다.

In [ ]:
# # 지금까지 한 일, 쉬운 설명 (2026-10-21)

# ## 이 연구가 뭔가요?

# 교통 분야 논문 약 8만 편을 모아서 **"교통 연구의 주제가 시간이 지나며 어떻게 바뀌었나, 누가 바꿨나, 어느 나라가 이끌었나"**를 본 연구입니다.

# 처음 분석(S0~S8)은 **결과를 보기 전에 규칙을 먼저 정해 도장을 찍어 두고** 진행했습니다. 그런데 결과를 보니 약한 곳이 몇 군데 있었습니다. 그래서 **사후분석(PA)**을 따로 했습니다. 규칙은 이렇습니다.

# - 원래 결과(통과/실패)는 **절대 바꾸지 않는다.**
# - 분석하기 전에 계획을 먼저 도장 찍고, GitHub에 올려서 시각 기록을 남긴다.
# - 결과가 기대와 달라도 전부 보고한다.

# 시험을 본 뒤에 "이 문제는 이렇게 풀면 더 잘 보여요" 하고 보충 설명을 다는 것과 비슷합니다. 점수는 안 바꿉니다.

# ---

# ## 원래 분석(S0~S8)의 큰 결과

# | 질문 | 답 |
# |---|---|
# | 주제가 어느 해에 확 바뀐 "체제 전환"이 있었나? | 통계 검사를 통과하지 못해서 **"체제 전환" 이야기는 철회**했습니다. |
# | 새 연구자가 주제 변화를 이끌었나? | 새로 들어온 저자의 몫이 0.24에서 0.38로 커졌습니다. |
# | 어느 나라가 이끌었나? | 미국은 줄고(0.46→0.19), 중국은 늘었습니다(0.03→0.35). |
# | 연구가 필요한 곳을 향하나? | 6개 분야 중 2개만 검증을 통과했습니다. |

# ---

# ## 지금까지 한 사후분석과 결과

# **0. 정리 작업 (분석이 아니라 정리)**
# 파일 152개의 "지문(해시)"이 전부 그대로인 것을 확인했습니다. 몰래 바뀐 파일이 없다는 뜻입니다. 잃어버린 코드를 복구하고, 잘못 붙은 이름표(태그)를 고치고, 모든 기록을 GitHub에 올렸습니다.

# **1. PA-A1: "점프가 있었다면 검사가 잡아냈을까?"**
# 연기 감지기가 안 울렸을 때, 감지기가 고장 난 건지 진짜 불이 없는 건지 알아야 합니다. 그래서 일부러 작은 불을 피워 봤습니다.
# - 첫 시도는 코드 버그로 무효였습니다(지우지 않고 기록).
# - 두 번째에서 2020년 부근의 점프는 대체로 잡아냈지만, 2008~2012년 부근은 잘 못 잡는 **사각지대**가 있었습니다.
# - 그래서 "큰 점프는 없었다"까지는 말할 수 있어도, "체제 전환이 없었다고 확인됐다"는 말하면 안 됩니다.

# **2. PA-B2: "2020년 변화가 논문 수가 늘어서 생긴 착시인가?"**
# 2020년 논문 수를 줄여서 다시 계산해도 변화가 남았습니다. 다만 줄인 양이 20%뿐이라 약한 시험이었습니다. 변화의 내용은 **코로나 관련 주제가 1등**이고, 그다음이 전기모터, 자율주행, 머신러닝이었습니다.

# **3. PA-B2b: "특정 저널이나 코로나 주제 하나 때문인가?"**
# 논문이 가장 많이 늘어난 저널 5곳을 빼도, 코로나 주제만 빼도 변화가 남았습니다. **둘을 동시에 빼면** 변화가 약해지고 위치가 한 해 앞으로 밀렸습니다. 2020년 변화는 한 가지 원인이 아니라 여러 흐름이 겹친 것으로 읽습니다.

# **4. PA-A2c: "새 저자가 많다는 결과의 오차 범위를 더 정직하게"**
# 4개 시기 중 3개에서 "새로 들어온 저자 > 기존 저자의 적응"이 유지됐습니다. 두 가지 오차 계산법 모두에서 확실한 곳은 **마지막 시기(2015–19→2020–25)뿐**입니다.

# **5. PA-A2b: "'새 저자'의 정의를 바꾸면 결과가 변하나?"**
# 새 저자를 "2년 안에 첫 논문을 쓴 사람"으로 좁히면 결과가 뒤집힙니다. 하지만 이는 이름표를 좁게 붙인 효과일 뿐, 들어온 사람의 총량은 같습니다. 그래서 문장을 **"신규 저자 주도"가 아니라 "진입(신규+복귀) 주도"**로 고쳐야 안전합니다.

# **6. PA-B5: "S4 분석의 안정성 0.70, 아슬아슬하지 않나?"**
# 기준선을 바꾸니 0.66~0.77로 움직였습니다. "0.70 근처에서 흔들린다"가 정확한 표현입니다. 다행히 증가/감소 방향이 뒤집힌 토픽은 없었습니다.

# **7. PA-A2a: "이름이 같은 다른 사람이 섞였다면?"**
# 데이터의 OpenAlex ID가 2020년 이후 논문에만 있어서 원래 계획은 불가능했습니다. 그래서 "ID가 둘 이상으로 갈라지는 이름을 빼도 결과가 변하나?"로 줄여서 했고, 4개 시기 모두 **거의 변하지 않았습니다**(변화 0.011 이내).

# **8. PA-B4: "품질 기준을 단순하게 해도 결과가 같은가?"**
# 품질 기준을 단순화해(토픽 149개 중 4개만 제외) 다시 계산했더니, 변화점 분석은 완전히 같았고 저자 분석도 3개 시기는 같았습니다. 첫 시기만 부호가 바뀌었는데, 원래 값이 0 근처라서 생긴 일입니다. 제외한 게 4개뿐이라 **약한 시험**입니다.

# **9. PA-C2: "국가 몫이 논문 집합에 따라 달라지나?"**
# 주제 정보가 있는 논문만으로 다시 계산했습니다. **중국 몫은 4개 시기 모두 거의 같았고**, 미국 몫과 상위 5개국 몫은 0.01~0.03 높아졌습니다. "두 집합이 일치한다"고는 쓸 수 없고, "수준은 조금 다르다"고 밝혀야 합니다.

# ---

# ## 지금 논문이 말해도 되는 것 / 안 되는 것

# | 말해도 됨 | 말하면 안 됨 |
# |---|---|
# | 주제 변화는 한 번의 점프보다 서서히 재배치된 것으로 설명하는 게 자료에 맞다 | 체제 전환이 없었음이 확인됐다 |
# | 2020년 부근에는 코로나와 기술 주제가 겹친 가속이 있다 | 2020년이 착시가 아니라고 확정됐다 |
# | 새로 진입한 저자가 기존 저자의 적응보다 크다 (특히 마지막 시기) | 신규 저자가 모든 시기를 주도했다 |
# | 미국은 줄고 중국은 늘었다 | 누가 원인인지 같은 인과 관계 |

# ---

# ## 지금 진행 중인 것

# **PA-C1: "중국 소속 저자를 빼도 '진입 우세'가 남을까?"**
# 방금 본 실행을 시작했습니다. 두 가지로 봅니다.
# - (a) 중국 소속 **신규** 저자만 뺀다. 진입이 기계적으로 줄어드니 어려운 시험입니다.
# - (b) 중국 소속 저자를 **신규·기존 모두** 뺀다. 더 공정한 비교입니다.

# 이 결과는 "새 저자 주도 갱신"과 "중국의 부상"이 같은 현상인지 다른 현상인지를 가려 줍니다.

# ---

# ## 앞으로 할 분석

# | 순서 | 항목 | 쉬운 설명 |
# |---|---|---|
# | 1 | **C1 결과 읽기** | 위의 진행 중인 분석 |
# | 2 | **B7** | 주제 정보가 없는 논문 16%가 결과를 바꿀 수 있는지, 최악·최선의 경우로 시험 |
# | 3 | **B3** | "새로 생긴 토픽"의 정의가 4가지인데 서로 안 맞는다. 공통 핵심 찾기 |
# | 4 | **A3** | 선두 국가 분석을 구간 길이를 똑같이 맞춰 재확인 |
# | 5 | **B9, B8** | 소득 수준 효과 분리, 검증을 통과 못 한 4개 분야의 참고 결과 |
# | 6 | **B4b** | S4 재추정과 토픽 응집도 지표 |
# | 7 | **B6** | 동명이인을 사람이 직접 50쌍 확인 (손이 많이 가서 연구자 시간이 필요) |
# | 8 | **마무리** | 무엇을 언제 봤는지 열람 이력 표, 약하게 나온 분석도 부록에 전부 수록, 모든 표에 `[Post-hoc]` 표시 |

# 그 뒤에는 투고 준비(저널 선례 확인, 심사 예상 질문 대응표, 생성형 AI 사용 선언)가 남아 있습니다. 논문 제목에서는 **"체제 전환"을 빼고** "진입 주도 갱신 + 선두 국가 교체"를 중심에 두는 구성을 권합니다.

# ---

# ## 한 줄 요약

# > **"큰 점프는 없었고 변화는 서서히 이어졌으며, 새로 들어온 연구자들과 중국의 부상이 핵심이다. 다만 정의를 바꾸면 약해지는 곳이 있어서, 그 한계까지 정직하게 적는다."**

In [ ]:
# # PA-C1 판독

# **동결 판독의 기계적 결과는 세 시나리오 모두 "진입 우세 유지 안 됨"입니다.** 중국 소속 저자를 빼면 마지막 쌍의 `entry_minus_adapt`가 양수에서 음수로 넘어갑니다. 이 결과는 앞서 "견고하다"고 정리한 문장을 좁혀야 한다는 뜻입니다.

# ## 1. 결과

# | 시나리오 | 제외 저자 수 | 원본 | 제외 후 | 구간(백분위) |
# |---|---|---|---|---|
# | a_new (CN 신규만) | 18,909 | +0.1128 | **−0.1014** | [−0.126, −0.082] |
# | a_entrants (CN 신규+복귀) | 19,450 | +0.1128 | **−0.1100** | [−0.134, −0.090] |
# | b_all (CN 신규+지속 모두) | 22,176 | +0.1128 | **−0.0237** | [−0.052, −0.003] (basic 상한 +0.005) |

# - (a)에서는 진입 몫이 0.393에서 0.272로 줄고, 적응 몫은 0.281에서 0.373으로 올라갑니다. 분해 몫은 분모(전체 변화량)로 정규화하므로 한쪽이 줄면 다른 쪽이 상대적으로 커집니다.
# - (b)가 더 공정한 대조입니다. 신규·지속 양쪽에서 같은 집단을 뺐는데도 −0.024입니다. 다만 이 시나리오는 0에 가깝습니다. 백분위 구간은 0 아래이지만 basic 구간의 상한은 +0.005로 0을 걸칩니다. "진입이 적응보다 작다"고 단정하기보다 "진입 우세가 사라진다"가 정확합니다.

# ## 2. 읽기

# 1. **마지막 쌍의 진입 우세는 중국 소속 저자의 유입과 분리되지 않습니다.** 중국 소속 비중은 신규 진입 저자에서 41.2%(질량 42.7%)로, 지속 저자 22.0%(23.7%)보다 크게 높습니다. 신규 진입의 약 4할이 중국 소속이라는 구성이 진입 우세의 상당 부분을 설명하는 것으로 보입니다.
# 2. PA-A2c·A2b·A2a에서 마지막 쌍이 "견고하다"고 한 것은 **부트스트랩 방식, 신규 정의, 이름 분할에 대한 견고성**이었습니다. 이번 분석은 다른 축(집단 구성)이며, 견고성이 모든 축에서 성립한다고는 말할 수 없게 됐습니다.
# 3. 계획서에서 미리 적어 둔 대로 (a)는 진입 질량이 기계적으로 줄어드는 시험이라 "유지"가 어려운 쪽이고, 따라서 (a)의 반전만으로 큰 의미를 두기는 어렵습니다. 의미가 있는 쪽은 (b)이며, (b)에서도 우세가 사라졌다는 점이 핵심입니다.
# 4. **S6의 "신규 진입 주도" 결과와 S7의 "중국 증가" 결과는 같은 현상의 두 측면일 가능성이 큽니다.** PA-C1 계획서가 준비해 둔 해석 분기("유지되지 않으면 같은 현상의 두 측면")에 해당합니다. 다만 이것은 구성 분해이고 인과 주장이 아닙니다.
# 5. **"중국 소속"은 국적이 아니라 소속 위치입니다.** 정의(CN 질량 ≥ 0.5)와 가중(S7 `inv_n_authors`로 판정, S6 `equal`로 분해)이 혼합입니다. 이를 본문에 명시해야 합니다.
# 6. 중국 소속 신규 저자를 빼면 남은 신규 저자 쪽은 적응보다 작습니다. 이 점이 "비중국 신규 저자 중심으로는 진입 우세가 없다"로 읽힐 수 있는데, 저는 제외 후의 비교가 원래 비교와 같은 척도라는 점만 확인했고, 다른 이유(제외로 인한 저자 풀 축소 효과)를 배제하지는 못했습니다.

# ## 3. 본문에 쓸 수 있는 문구 / 쓰면 안 되는 문구

# - **쓸 수 있음:** "마지막 구간의 진입 우세는 중국 소속 저자를 제외하면 유지되지 않는다(진입−적응 +0.113 → 신규·지속 모두 제외 시 −0.024). 중국 소속은 신규 진입 저자의 약 41%를 차지한다. 따라서 진입 주도 갱신과 중국 몫 증가는 분리해 해석할 수 없다."
# - **쓰면 안 됨:** "진입 주도 갱신은 중국 유입과 무관하다", "비중국 저자 중심으로도 신규 우세가 확인된다", 인과 해석.

# ## 4. 논문 서사에 미치는 영향

# 이전에 권한 구성("신규 진입 주도 갱신 + 선두 국가 교체"를 두 개의 독립 기여로 제시)은 수정이 필요합니다. 두 결과가 겹치므로 하나의 기여, 즉 "진입 중심의 갱신, 그리고 그 진입의 큰 부분이 중국 소속 저자라는 구성"으로 묶어 쓰는 편이 자료에 맞습니다. 이것은 약점이 아니라 오히려 정직한 구조 서술입니다. S6 판정은 바뀌지 않습니다.

# ## 5. 기록

# ```bash
# cd ~/Research/Minnesota_Transport
# pgrep -f pa_c1.py || echo "종료 확인"
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-C1 결과 해석 메모(B=2000, 마지막 쌍만): 동결 판독 3개 시나리오 모두 '유지 안 됨'. entry_minus_adapt 0.1128 -> a_new -0.1014, a_entrants -0.1100, b_all -0.0237(백분위 [-0.052,-0.003], basic 상한 +0.005). CN 소속 비중은 E_new 41.2%(질량 42.7%), P 22.0%(23.7%). (a)는 진입 질량이 기계적으로 줄어 약한 대조, (b)가 공정한 대조. 진입 우세와 중국 소속 저자 유입은 분리 불가(같은 현상의 두 측면 가능성, 인과 아님). 소속 위치 기준이며 판정 가중(inv_n)과 분해 가중(equal) 혼합. QUICK 출력의 CN 소속 이름 수 27,053 열람함 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_C1_results.csv paper1/09_PA/PA_C1_roles.csv paper1/09_PA/PA_C1_full_log.txt paper1/09_PA/PA_CHANGELOG.md paper1/pa_c1.py
# git commit -m "PA-C1 results" && git tag -a PA-C1-results-2026-10-21 -m "PA-C1 outputs, seen" && git push --follow-tags
# ```

# ## 6. 남은 일

# PA-C1 결과 때문에 앞의 PA-A2 요약 문구("마지막 구간에서 가장 견고")에는 "집단 구성(중국 소속)을 통제하면 유지되지 않음"이라는 단서를 붙여야 합니다. 남은 항목은 B7(임베딩 결측 경계), B3(신생 토픽 핵심 집합), A3(S7 균등 4구간·중국 제외 HHI), B9·B8(소득·도메인), B4b(S4 재추정), B6(수작업), A4(열람 이력 부록)입니다. A3는 C1과 이어지는 "중국 제외 HHI"를 포함하므로 다음으로 자연스럽습니다. 시작하려면 S7 `country_mass_long`와 `s7_period_stats.csv`의 구조는 이미 확인된 상태라 바로 설계할 수 있습니다.

In [ ]:
# # PA-A3 판독

# 재현 점검을 통과했고 본 실행이 정상 종료됐습니다. 동결 규칙 그대로 항목별로 읽습니다.

# ## A3-1: rec 대 main (orig 기간)

# | 통계 | 변화(main / rec) | 방향 | 최대 수준 차이 | 수준 |
# |---|---|---|---|---|
# | top5_incl | +0.052 / +0.015 | 일치 | 0.057 | 차이 |
# | top5_known | +0.006 / +0.005 | 일치 | 0.007 | 일치 |
# | hhi_known | −0.058 / −0.064 | 일치 | 0.008 | 일치 |
# | cn_known | +0.319 / +0.319 | 일치 | 0.003 | 일치 |
# | us_known | −0.267 / −0.271 | 일치 | 0.011 | 차이(간발) |
# | unk | −0.071 / −0.015 | 일치 | 0.086 | 차이 |

# - 6개 통계 모두 방향이 일치합니다.
# - **미상 교정 해석이 재현됩니다.** main에서는 미상이 11.1%에서 4.0%로 줄면서 `top5_incl`이 +0.052 늘지만, rec에서는 미상이 2.5%에서 1.0%로 거의 평탄하고 `top5_incl`도 +0.015만 늘어납니다. 즉 포함 기준 상위 5개국 몫의 겉보기 증가 상당 부분은 미상 비중 감소에서 옵니다. 알려진 국가 기준 `top5_known`은 두 variant 모두 거의 평탄(+0.006, +0.005)이라 "상위 5개국 몫은 변하지 않는다"는 서술과 맞습니다.
# - `us_known`의 수준 차이 0.0109는 기준 0.01을 간발로 넘은 경계 칸입니다. 이분법에 무게를 두지 마세요.

# ## A3-2: 균등 4구간

# - 12칸 모두 "방향 유지"입니다(main·rec 모두).
# - 주의: `top5_known`은 orig에서 +0.006으로 0에 가까워 부호 판독이 불안정한 통계인데, equal4에서는 +0.033(main), +0.030(rec)으로 더 커집니다. 부호는 같지만 "평탄"이라는 서술이 구간 정의에 민감할 수 있다는 단서입니다. 다만 마지막 구간이 4년·2025 부분연도라 같은 의미의 비교는 아닙니다.
# - 핵심 흐름은 구간 정의와 무관합니다: 미국 0.44→0.17, 중국 0.03→0.39(equal4, main).

# ## A3-3: 중국 제외 HHI

# | | 포함 HHI | 중국 제외 HHI |
# |---|---|---|
# | orig(main) | P1519 최저(0.131), P2025 반등(0.167) | **단조 감소**(0.237→0.108), 반등 없음 |
# | equal4(main) | E1721 최저(0.126), E2225 반등(0.190) | 단조 감소(0.224→0.100), 반등 없음 |

# 4개 조합(variant × 구간 정의) 모두 판독은 **"반등은 중국 몫에 연동"**입니다. 중국 소속 질량을 빼면 집중도는 계속 낮아지므로, 마지막 구간의 HHI 반등은 중국 몫이 급증해 한 국가가 커진 효과와 함께 움직입니다. 이는 인과 주장이 아니고 소속 위치 기준입니다. 또한 `top5_exCN`도 0.66에서 0.51로 줄어, 중국을 빼면 나머지 국가들의 분산은 계속 커집니다.

# ## 쓸 수 있는 문구

# > "알려진 국가 기준 상위 5개국 몫은 기간에 따라 크게 변하지 않았고(0.64→0.65), 포함 기준 증가(0.57→0.62)는 미상 비중 감소(11%→4%)와 함께 움직인다(복구판에서는 +0.015). 집중도(HHI)는 2015–19에 최저였다가 마지막 기간에 반등하지만, 중국 소속 질량을 제외하면 반등이 나타나지 않는다. 이 결과는 균등 4구간에서도 같다."

# 쓰면 안 되는 것: "지식 생산이 소수 국가에 집중됐다", "중국이 집중을 일으켰다"(인과), 소속을 국적으로 해석하는 표현.

# ## 연결 메모

# PA-C1(진입 우세가 중국 소속 저자를 빼면 사라짐)과 이번 A3-3(HHI 반등이 중국 몫에 연동)은 같은 방향의 단서입니다. 두 결과가 모두 "중국 소속 저자의 유입"이라는 한 축으로 수렴하므로, 본문에서는 이를 하나의 구조로 묶어 서술하는 편이 자료에 맞습니다.

# ## 기록

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-A3 결과 해석 메모(결정적 재집계): A3-1 6개 통계 모두 rec·main 방향 일치(수준 차이는 top5_incl 0.057, us_known 0.011(경계), unk 0.086). 미상 11.1->4.0%(main) 대 2.5->1.0%(rec)로 top5_incl 증가가 +0.052 대 +0.015, top5_known은 두 variant 모두 평탄(+0.006/+0.005). A3-2 균등 4구간 12칸 모두 방향 유지(top5_known은 orig +0.006에서 equal4 +0.030~0.033으로 0 근처 부호 불안정 단서, 마지막 구간 4년·부분연도). A3-3 4조합 모두 '반등은 중국 몫에 연동'(포함 HHI 0.131->0.167 반등, 중국 제외 HHI 0.237->0.108 단조 감소). 소속 위치 기준, 인과 아님. PA-C1과 같은 방향의 단서 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_A3_*.csv paper1/09_PA/PA_A3_full_log.txt paper1/09_PA/PA_CHANGELOG.md paper1/pa_a3.py
# git commit -m "PA-A3 results" && git tag -a PA-A3-results-2026-10-21 -m "PA-A3 outputs, seen" && git push --follow-tags
# ```

# ## 다음 단계

# 끝난 PA는 A1, A2a·b·c, A3, B2, B2b, B4(부분), B5, C1, C2입니다. 남은 것은 B7(임베딩 결측 경계), B3(F6 생성 코드 위치 필요), B9·B8(S8 입력 필요), B4b(`s4_main.py` 토픽 선택 로직 필요), B6(수작업), A4(열람 이력 부록)입니다.

# 입력이 가장 적게 필요한 것은 B7입니다. 임베딩이 없는 논문 14,810편이 전부 특정 토픽이라는 최악·최선 대입으로 S4 핵심 결과의 방향이 유지되는지 보는 분석이라, F1의 결측 구조와 S4 `big_change` 계산 부분만 확인하면 됩니다. 진행하시려면 아래 출력을 붙여 주세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 - <<'EOF'
# import pandas as pd
# F1 = pd.read_csv("../analysis_data/final/F1_papers.csv", low_memory=False, nrows=5); print(F1.columns.tolist())
# T = pd.read_csv("04_S4/topic_quadrants.csv"); print(T.columns.tolist()); print(T.head(3).to_string())
# EOF
# grep -n "big\|delta\|period\|2005\|2015\|2019" s4_main.py | head -40
# ```

In [ ]:
# PA-A3 기록이 올라갔습니다(커밋 `fa7142e`, 태그 `PA-A3-results-2026-10-21`).

# ## B7 설계를 먼저 점검했습니다

# 출력으로 `topic_quadrants.csv`의 열 구조(`s0`, `s1`, `delta`, `within`, `big`)와 `big` 컷이 `|delta|`의 상위 1/3 분위라는 점은 확인됐습니다. 하지만 `s0`/`s1`을 만드는 부분(`s4_main.py` 40~74행 부근)과 S4 기간 정의가 출력에 없어서, 지금 코드를 쓰면 추측이 됩니다. 이 두 가지는 코드를 쓰기 전에 봐야 합니다.

# B7은 세 부분이고, 필요한 입력이 서로 다릅니다.

# | 부분 | 필요한 입력 | 지금 가능한가 |
# |---|---|---|
# | (1) 결측 경계(최악·최선) | 기간별 결측 논문 수, `s0`/`s1` 계산식 | 위 두 가지만 확인되면 가능 |
# | (2) 제목 분류기 예측 토픽 대입 | 분류기 산출 파일 위치 | 미확인 |
# | (3) 저널×연도 IPW | `paper_ipw.csv.gz` 구조, IPW 모델 | `s6_lib.ipw_weights`는 봤지만 S4 쪽 사용 방식은 미확인 |

# 그래서 (1)만 `PA-B7a`로 먼저 동결하고, (2)와 (3)은 입력을 받은 뒤 `PA-B7b/c`로 분리하겠습니다. 한 계획서에 합쳤다가 나중에 고치면 v2가 되기 때문입니다.

# ## (1)을 설계할 때 미리 알아 둘 한계

# - **순수 최악·최선 대입(Manski 경계)은 거의 확실히 쓸모가 없습니다.** 결측이 14,810편(약 16%)이라 토픽 몫의 경계 폭이 기간당 최대 0.16입니다. 반면 `big` 토픽의 `delta`는 0.01 안팎이라 경계가 항상 0을 포함합니다. 이 결과를 "결론이 뒤집힌다"로 읽으면 안 됩니다. 정보가 없다는 뜻일 뿐입니다.
# - **더 정보가 있는 대안은 "뒤집힘 임계값"입니다.** 결측 논문의 토픽 구성이 관측 논문과 얼마나 달라야(결측 질량의 몇 %를 해당 토픽으로 재배분해야) `big` 토픽의 부호나 4분면 라벨이 바뀌는지를 토픽별로 계산합니다. 이것이 동결 규칙 후보입니다. 임계값이 결측 질량에 비해 크면 "현실적인 편향으로는 뒤집히지 않는다"고 쓸 수 있고, 이는 지침서가 말한 경계 분석의 취지와도 맞습니다. 다만 "현실적"의 기준은 연구자가 정하는 가정이라, 계획서에 명시해야 합니다.
# - 결측이 기간·저널별로 다르게 분포하면(예: 2007년 하락이 있었다고 했습니다) 기간별 결측률 차이가 `delta`를 직접 움직입니다. 이 점은 (3) IPW와 겹칩니다.

# ## 필요한 출력

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# sed -n 38,80p s4_main.py
# python3 - <<'EOF'
# import json
# p = json.load(open("00_S0/PREREG.json", encoding="utf-8"))
# def find(o, k):
#     if isinstance(o, dict):
#         for a, b in o.items():
#             if a == k: return b
#             r = find(b, k)
#             if r is not None: return r
# for k in ("periods", "quality", "ipw"): print(k, json.dumps(find(p, k), ensure_ascii=False)[:500])
# import pandas as pd
# F = pd.read_csv("../analysis_data/final/F1_papers.csv", low_memory=False)
# F = F[F.in_sample.astype(str).str.lower().isin(["true","1","1.0"])]
# F["miss"] = ~F.has_topic.astype(str).str.lower().isin(["true","1","1.0"])
# print(F[F.year.between(2000, 2025)].groupby("year").miss.agg(["size", "mean"]).round(3).T.to_string())
# EOF
# ```

# 마지막 표는 연도별 결측률입니다. 결측이 특정 기간에 몰려 있는지 이 표로 확인할 수 있고, 이는 설계용 점검이라 결과 통계가 아닙니다. 다만 이 수치도 열람 사실이므로 해석 메모에 한 줄 남기겠습니다.

# 출력을 받으면 `pa_b7a.py` 동결 코드를 PA-C2와 같은 틀(freeze → tag → QUICK → 본 실행)로 드리겠습니다.

In [ ]:
# # PA-B7a 설계와 동결 코드

# 이 출력으로 결측 경계의 설계가 닫혔습니다. 먼저 출력에서 읽힌 사실을 적습니다.

# - **결측률이 기간마다 다릅니다.** 연도별 결측률은 2000년 0.30에서 2015~2024년 0.12~0.15로 내려갑니다. 2005~09년은 약 0.20, 2020~24년은 약 0.14라서, S4 기간 대비의 두 기간은 결측률이 6~7%p 다를 가능성이 큽니다. 이 표는 `EXCLUDE_PIDS_FROZEN`을 적용하지 않은 근사치이고, 정확한 값은 코드가 다시 냅니다.
# - **`main_contrast`와 `sens_contrast`가 출력에 없습니다.** `periods`에는 P05_09, P15_19, P20_24 등이 있지만 어느 두 기간이 주 대비인지는 모릅니다. 추측하지 않고, 코드가 S4 동결 규칙 파일에서 직접 읽어 중단 조건으로 대조합니다.
# - **S4 주 계산은 가중이 없습니다.** 가중 없는 delta가 본 계산과 같은지 점검하는 코드가 있습니다. `missingness_sensitivity.csv`(IPW 보정)는 이미 S4에 있으므로, B7의 (3) 저널×연도 IPW는 일부 중복입니다. 그 파일의 결과는 제가 못 봤습니다.

# ## 동결 내용

# 앞서 말씀드린 대로, 순수 최악·최선 대입(Manski)은 경계 폭(약 0.14~0.16)이 `big` 토픽의 `delta`(0.01 안팎)보다 커서 항상 0을 포함합니다. 정보가 없으므로 계산은 하지 않고 계획서에 그 이유만 적습니다. 대신 **뒤집힘 배수(tipping multiplier)**를 씁니다.

# - **가정:** 결측 논문은 같은 저널·기간의 관측 논문보다 해당 토픽을 `k`배 더 많이 가집니다(`r_jt = k · s_jt`). 결측 논문의 저널 분포는 관측 그대로입니다. `k=1`은 "결측이 저널 안에서는 무작위"입니다.
# - **시나리오 3개:** (both) 두 기간 모두 `k`, (only0) 앞 기간만 `k`, (only1) 뒤 기간만 `k`.
# - **격자:** `k ∈ {0.5, 0.75, 1, 1.5, 2}`. "현실적 범위"는 [0.5, 2]로 연구자 가정이며, 실행 전에 동결합니다.
# - **대상:** S4 `big` 토픽 50개(`topic_quadrants.csv`의 `big`). `delta_full`의 부호가 관측 `delta`와 같은지 보고, 부호가 바뀌는 `k*`(뒤집힘 배수)를 시나리오별로 냅니다. `big`이 아닌 토픽이 `big`이 되는 경우와 4분면 라벨 변화는 토픽 간 결합 시나리오가 필요해서 이번 범위에서 제외합니다.
# - **판독(고정):** 시나리오별로 격자 전체에서 부호가 유지되는 토픽 비율이 ≥0.90이면 "유지", 0.70~0.90이면 "부분", 0.70 미만이면 "민감"입니다. 세 시나리오를 하나로 합치지 않습니다. 이 임계값은 임의이며 동결 사양입니다.
# - **재현 점검(중단 조건):** (a) 분석 집합 76,984편, (b) 관측 `delta`가 `topic_quadrants.csv`와 1e-9 이내, (c) `main_contrast` 기간을 규칙 파일에서 읽을 수 있을 것.

# ## 코드: `paper1/pa_b7a.py`

# ```python
# # -*- coding: utf-8 -*-
# # PA-B7a: 임베딩 결측(has_topic 거짓) 논문에 대한 뒤집힘 배수 분석(S4 big 토픽 부호). 결정적 계산. 미실행 코드.
# # MODE: "freeze" -> "tag" -> (git push --follow-tags) -> "run"(QUICK=True: 재현 점검과 기간별 결측률만) -> "run"(QUICK=False)
# # S0~S8 판정 불변. 모든 산출물 [Post-hoc].
# import json, hashlib, subprocess, warnings
# from pathlib import Path
# import numpy as np, pandas as pd
# warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
# MODE = "freeze"
# QUICK = True

# ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; PA = P1/"09_PA"; S0, S3, S4 = P1/"00_S0", P1/"03_S3", P1/"04_S4"; DATA = ROOT/"analysis_data/final"
# now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
# sha_b = lambda b: hashlib.sha256(b).hexdigest(); sha_f = lambda p: sha_b(Path(p).read_bytes())
# tf = lambda s: s.astype(str).str.strip().str.lower().isin(["1", "1.0", "true", "t", "yes"])
# git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
# class Halt(Exception): pass
# spec_hash = lambda s: sha_b(json.dumps({k: v for k, v in s.items() if k not in ("sha256", "created", "refs")}, sort_keys=True, ensure_ascii=False).encode())
# V2P, BP = PA/"PA_0_PLAN_v2.json", PA/"PA_B7a_PLAN.json"

# PLAN = dict(
#     version=1, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(PA-B7 중 결측 경계 (1)만). S4 결과, s4_main.py 38~80행, 연도별 결측률(제외 목록 미적용 근사치 0.12~0.30)은 열람한 상태. PA-B7a 결과 미열람. 분류기 대입(B7b)과 저널x연도 IPW(B7c)는 별도 계획으로 분리(S4 missingness_sensitivity.csv와 중복 가능). S0~S8 판정 불변.",
#     why_not_manski="순수 최악·최선(Manski) 경계는 결측 약 16%라 토픽 몫 경계 폭이 big 토픽 delta(0.01 안팎)보다 훨씬 커서 항상 0을 포함함. 정보가 없으므로 계산하지 않음",
#     data="F1_papers.csv에서 EXCLUDE_PIDS_FROZEN 제외(FREEZE_LOG 해시 검증 없이 S4와 같은 파일 사용), in_sample 참, 연도 2000~2025. 분석 집합 F = has_topic 참 and topic150 있음(76,984편). 결측 집합 = 나머지. 기간·대비는 S4 동결 규칙의 periods, main_contrast",
#     assumption="결측 논문의 같은 저널·같은 기간 안 토픽 비중 r_jt = k * s_jt (해당 토픽만, 나머지 토픽은 합이 1이 되도록 조정되나 집계에는 불필요). 결측 논문의 저널 분포는 관측 그대로. 전체 몫 = sum_j (n_j + k*m_j) s_jt / sum_j (n_j + m_j)",
#     scenarios={"both": "두 기간 모두 배수 k", "only0": "앞 기간만 k, 뒤 기간 1", "only1": "앞 기간 1, 뒤 기간만 k"},
#     grid_k=[0.5, 0.75, 1, 1.5, 2], plausible_range=[0.5, 2],
#     scope="S4 big 토픽(topic_quadrants.csv의 big 참, 50개). 각 토픽의 delta_full(k) 부호가 관측 delta 부호와 같은지, 부호가 바뀌는 k*. big이 아닌 토픽의 big 전환과 4분면 라벨 변화는 토픽 간 결합 시나리오가 필요해 이번 범위에서 제외",
#     stats="delta_full_base(k=1, 시나리오 공통, 저널 구성 보정만), 격자별 부호 유지 여부, 시나리오별 k*(선형 해, k<0이면 '없음')",
#     reading="시나리오별로 격자 전체에서 부호가 유지되는 big 토픽 비율 >=0.90 '유지', 0.70~0.90 '부분', <0.70 '민감'. 시나리오를 합치지 않음. 임계값은 임의의 동결 사양",
#     reproduce_check="(a) 분석 집합 76,984편, (b) 관측 delta가 topic_quadrants.csv와 1e-9 이내, (c) 동결 규칙 파일에서 main_contrast·periods를 읽을 수 있음. 하나라도 어긋나면 중단",
#     caveat="k의 현실성 범위는 연구자 가정. 결측 논문이 저널 안에서도 무작위라는 k=1 기준과 저널 분포 고정 가정이 핵심. 결측 사유(임베딩 미보유)가 토픽과 무관하다는 보장 없음. 인과 해석 아님",
#     forbidden="결과를 본 뒤 격자·범위·판독 임계값 변경(변경 시 v2 계획으로 분리하고 사유 기록)")

# def freeze():
#     if not V2P.exists(): raise Halt("PA_0_PLAN_v2.json이 없습니다")
#     h = spec_hash(PLAN); print("PA-B7a 계획 해시(예정)", h[:12])
#     if BP.exists(): print("이미 동결:", "동일" if json.loads(BP.read_text(encoding="utf-8"))["sha256"] == h else "⛔ 다름"); return
#     refs = {f.name: sha_f(f) for f in [V2P, S4/"topic_quadrants.csv", P1/"s4_main.py", DATA/"F1_papers.csv"] if f.exists()}
#     BP.write_text(json.dumps(dict(PLAN, created=now(), sha256=h, refs=refs), ensure_ascii=False, indent=1), encoding="utf-8")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f:
#         f.write(f"\n- {now()} | PA-B7a 계획 동결(해시 {h[:12]}) | 사유: 결측 경계를 Manski 대신 뒤집힘 배수로 설계. 연도별 결측률(제외 미적용 근사)과 S4 코드 일부 열람 상태, PA-B7a 결과 미열람. QUICK은 재현 점검과 기간별 결측률만 | 본 결과 변경 여부: 없음\n")
#     print("저장: PA_B7a_PLAN.json")

# def tag():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     t = f"PA-B7a-plan-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
#     if git("tag", "-l", t).stdout.strip(): print("이미 있음:", t); return
#     r = git("add", "--", "paper1/09_PA/PA_B7a_PLAN.json", "paper1/09_PA/PA_CHANGELOG.md", "paper1/pa_b7a.py"); print("add", r.returncode, r.stderr.strip())
#     r = git("commit", "-m", "PA-B7a plan freeze (before execution)"); print("commit", r.returncode, (r.stdout + r.stderr)[:200])
#     r = git("tag", "-a", t, "-m", "PA-B7a plan"); print("tag", r.returncode, r.stderr.strip()); print("※ 이어서 git push --follow-tags")

# def find_key(o, key):
#     if isinstance(o, dict):
#         for k, v in o.items():
#             if k == key: return v
#             r = find_key(v, key)
#             if r is not None: return r

# def run():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     sp = json.loads(BP.read_text(encoding="utf-8"))
#     if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(PLAN): raise Halt("동결 파일과 현재 계획이 다릅니다")
#     if not git("tag", "-l", "PA-B7a-plan-*").stdout.strip(): raise Halt("tag 단계를 먼저 실행하세요")
#     sfx = "_QUICK" if QUICK else ""
#     spec = None
#     for f in sorted(P1.glob("0[0-4]_*/*PREREG*.json")):
#         j = json.loads(f.read_text(encoding="utf-8")); mc, pr = find_key(j, "main_contrast"), find_key(j, "periods")
#         if mc and pr and all(k in pr for k in mc): spec = (f.name, mc, pr); break
#     if spec is None: raise Halt("main_contrast와 periods를 가진 S4 동결 규칙 파일을 찾지 못함")
#     fname, (c0, c1), PR = spec; P = {k: tuple(v) for k, v in PR.items()}; print(f"[규칙 파일] {fname} | main_contrast {c0}->{c1} | {P[c0]}, {P[c1]}")
#     W = pd.read_csv(DATA/"F1_papers.csv", low_memory=False); W["pid"] = W.pid.astype(str)
#     EXC = set(pd.read_csv(S0/"EXCLUDE_PIDS_FROZEN.csv", dtype=str, encoding="utf-8-sig").pid)
#     W = W[~W.pid.isin(EXC) & tf(W.in_sample) & W.year.between(2000, 2025)].copy(); W["ht"] = tf(W.has_topic) & W.topic150.notna()
#     F, Mi = W[W.ht].copy(), W[~W.ht].copy()
#     if len(F) != 76984: raise Halt(f"분석 집합 재현 실패: {len(F)} (76984여야 함)")
#     F["topic"] = F.topic150.astype(int)
#     topics = np.array(sorted(pd.read_csv(S3/"L7_topic_grade.csv").topic.astype(int))); venues = np.array(sorted(F.venue_short.unique()))
#     dropped = int((~Mi.venue_short.isin(venues)).sum()); Mi = Mi[Mi.venue_short.isin(venues)]
#     def pm(lo, hi):
#         d, m = F[F.year.between(lo, hi)], Mi[Mi.year.between(lo, hi)]
#         C = pd.crosstab(d.venue_short, d.topic).reindex(index=venues, columns=topics, fill_value=0).values.astype(float)
#         n = C.sum(1); s = np.divide(C, n[:, None], out=np.zeros_like(C), where=n[:, None] > 0)
#         mm = m.groupby("venue_short").size().reindex(venues, fill_value=0).values.astype(float)
#         a = C.sum(0); b = (mm[:, None]*s).sum(0); return dict(a=a, b=b, n=n.sum(), m=mm.sum(), Nf=n.sum() + mm.sum())
#     A0, A1 = pm(*P[c0]), pm(*P[c1])
#     Q = pd.read_csv(S4/"topic_quadrants.csv").set_index("topic"); delta_obs = A1["a"]/A1["n"] - A0["a"]/A0["n"]
#     if np.abs(delta_obs - Q.loc[topics, "delta"].values).max() > 1e-9: raise Halt("관측 delta가 topic_quadrants.csv와 다름")
#     miss = pd.DataFrame([dict(period=c, n_obs=int(x["n"]), n_missing=int(x["m"]), miss_rate=x["m"]/x["Nf"]) for c, x in ((c0, A0), (c1, A1))])
#     print("[재현 OK] 분석 집합 76,984, 관측 delta | 결측 중 분석 저널 밖으로 제외된 논문", dropped); print("[기간별 결측률(설계 정보)]\n", miss.round(4).to_string(index=False))
#     if QUICK: print("\n[QUICK] 재현 점검과 기간별 결측률만 출력. 결과 통계는 계산·출력하지 않음."); return
#     f0, f1 = (lambda k0, k1: (A1["a"] + k1*A1["b"])/A1["Nf"] - (A0["a"] + k0*A0["b"])/A0["Nf"])
#     ff = lambda k0, k1: (A1["a"] + k1*A1["b"])/A1["Nf"] - (A0["a"] + k0*A0["b"])/A0["Nf"]
#     base = ff(1, 1); big = Q.loc[topics, "big"].values.astype(bool); sgn = np.sign(delta_obs); K = PLAN["grid_k"]
#     scen = {"both": lambda k: ff(k, k), "only0": lambda k: ff(k, 1), "only1": lambda k: ff(1, k)}
#     kstar = {"both": -(A1["a"]/A1["Nf"] - A0["a"]/A0["Nf"])/(A1["b"]/A1["Nf"] - A0["b"]/A0["Nf"]),
#              "only0": ((A1["a"] + A1["b"])/A1["Nf"] - A0["a"]/A0["Nf"])/(A0["b"]/A0["Nf"]), "only1": ((A0["a"] + A0["b"])/A0["Nf"] - A1["a"]/A1["Nf"])/(A1["b"]/A1["Nf"])}
#     R = pd.DataFrame(dict(topic=topics, grade=Q.loc[topics, "grade"].values, quadrant=Q.loc[topics, "quadrant"].values, big=big, delta=delta_obs, delta_full_base=base, sign_base_kept=np.sign(base) == sgn))
#     for nm, fn in scen.items():
#         keep = np.column_stack([np.sign(fn(k)) == sgn for k in K]); R[f"kept_all_{nm}"] = keep.all(1)
#         with np.errstate(all="ignore"): ks = kstar[nm]
#         R[f"kstar_{nm}"] = np.where(np.isfinite(ks) & (ks >= 0), ks, np.nan)
#     R.to_csv(PA/"PA_B7a_results.csv", index=False, encoding="utf-8-sig"); miss.to_csv(PA/"PA_B7a_missing_by_period.csv", index=False, encoding="utf-8-sig")
#     B = R[R.big]; print(f"\n[Post-hoc] PA-B7a big 토픽 {len(B)}개 | k=1(저널 구성 보정만) 부호 유지 {int(B.sign_base_kept.sum())}개")
#     out = []
#     for nm in scen:
#         share = float(B[f"kept_all_{nm}"].mean()); v = "유지" if share >= 0.90 else ("부분" if share >= 0.70 else "민감")
#         ks = B[f"kstar_{nm}"]; out.append(dict(scenario=nm, kept_share=share, n_not_kept=int((~B[f"kept_all_{nm}"]).sum()), kstar_min=ks.min(), kstar_median=ks.median(), n_kstar_in_range=int(((ks >= 0.5) & (ks <= 2)).sum()), 판독=v))
#     O = pd.DataFrame(out); print("\n[동결 판독]\n", O.round(3).to_string(index=False)); O.to_csv(PA/"PA_B7a_summary.csv", index=False, encoding="utf-8-sig")
#     print("\n[부호 유지 실패 big 토픽]\n", B[~B[[f"kept_all_{n}" for n in scen]].all(1)][["topic", "grade", "quadrant", "delta", "delta_full_base", "kstar_both", "kstar_only0", "kstar_only1"]].round(4).to_string(index=False))
#     print("※ 결측 분포에 대한 가정(k)에 의존하는 민감도이며 경계가 아님. 시나리오를 합치지 않음. S0~S8 판정 불변.")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as fh:
#         fh.write(f"- {now()} | PA-B7a 실행 | 정의: PA_B7a_PLAN {sp['sha256'][:12]} | 결과: PA_B7a_*.csv | 본 결과 변경 여부: 없음\n")

# try: {"freeze": freeze, "tag": tag, "run": run}[MODE]()
# except Halt as e: print(f"⛔ 중단: {e}")
# ```

# ## 실행 순서

# 한 블록씩 실행하고, `⛔`나 오류가 나면 거기서 멈추고 출력만 붙여 주세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 pa_b7a.py                                                           # 1) freeze
# ```
# ```bash
# sed -i 's/^MODE = "freeze"/MODE = "tag"/' pa_b7a.py && python3 pa_b7a.py     # 2) tag
# cd .. && git push --follow-tags && cd paper1
# ```
# ```bash
# sed -i 's/^MODE = "tag"/MODE = "run"/' pa_b7a.py
# ( time python3 -u pa_b7a.py ) 2>&1 | tee 09_PA/PA_B7a_quick_log.txt         # 3) QUICK
# ```
# ```bash
# sed -i 's/^QUICK = True/QUICK = False/' pa_b7a.py && grep -n '^MODE\|^QUICK' pa_b7a.py
# cd .. && git add paper1/pa_b7a.py paper1/09_PA/PA_B7a_quick_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-B7a: QUICK done; QUICK=False" && git tag -a PA-B7a-fullrun-2026-10-21 -m "pre-run" && git push --follow-tags
# ```
# ```bash
# cd paper1 && ( time python3 -u pa_b7a.py ) 2>&1 | tee 09_PA/PA_B7a_full_log.txt   # 4) 본 실행
# ```

# ## 주의할 점

# - **미실행 코드입니다.** 재현 점검 3개가 먼저 막아 줍니다. 가장 가능성 높은 중단은 `main_contrast·periods를 가진 규칙 파일을 찾지 못함`입니다. 이 경우 `04_S4`의 동결 규칙 파일 이름과 `main_contrast` 줄만 알려 주세요. 코드의 `f0, f1 = (...)` 줄은 불필요한 잔여 줄이라 실행에는 영향이 없지만, 오류가 나면 그 줄부터 의심하세요. (구문상 람다를 언패킹하려는 문제가 있어 실제로 오류가 날 가능성이 큽니다. 계획 해시와는 무관한 코드 줄이므로 지우고 진행해도 사양은 불변이며, 지웠다면 CHANGELOG에 "코드 오류 수정(사양 불변)"으로 기록하세요.)
# - **QUICK 로그의 기간별 결측률은 설계 정보**이지 결과 통계가 아니지만, 열람 사실이므로 해석 메모에 한 줄 남기겠습니다.
# - **이 분석은 경계가 아니라 민감도입니다.** 결과가 "유지"로 나와도 "결측이 결론을 못 바꾼다"가 아니라 "저널 안에서 토픽이 0.5~2배 편중된 정도의 결측으로는 big 토픽의 방향이 바뀌지 않는다"까지만 쓸 수 있습니다.
# - 앞서 S4에서 `missingness_sensitivity.csv`로 IPW 보정을 이미 한 결과가 있다면, 본문에서 B7a와 함께 보여 주는 것이 자연스럽습니다. 그 파일의 요약(`S4_final_summary.md`의 해당 줄)을 보여 주시면 B7c를 따로 만들지 판단하겠습니다.

# QUICK 로그와 본 실행 로그를 붙여 주시면 동결 규칙대로 읽겠습니다.

In [ ]:
# # PA-B7a 판독

# 수정 후 본 실행이 정상 종료됐고 재현 점검도 통과했습니다. 동결 규칙대로 읽으면 세 시나리오 모두 **"유지"**입니다.

# ## 결과

# | 시나리오 | 부호 유지 비율 | 유지 실패 토픽 | 뒤집힘 배수 k* (최소 / 중앙값) | 격자 [0.5, 2] 안의 k* |
# |---|---|---|---|---|
# | both | 1.00 | 0 | 없음(NaN) | 0 |
# | only0 | 1.00 | 0 | 5.16 / 47.0 | 0 |
# | only1 | 1.00 | 0 | 11.45 / 43.6 | 0 |

# - k=1(결측이 저널 안에서는 무작위, 저널 구성만 보정)에서도 big 토픽 50개 모두 부호가 유지됐습니다.
# - `both`의 k*가 NaN인 것은 두 기간에 같은 배수를 적용하면 부호가 바뀌는 양의 k가 존재하지 않는다는 뜻입니다. 코드가 k<0이면 "없음"으로 처리하도록 되어 있어서 그렇게 나온 것입니다.
# - `only0`/`only1`에서 가장 취약한 토픽도 결측 논문이 관측 논문보다 해당 토픽을 5배(앞 기간만) 또는 11배(뒤 기간만) 더 많이 가져야 부호가 바뀝니다. 연구자 가정 범위 [0.5, 2]와는 거리가 큽니다.

# ## 읽기

# - 결측률이 기간 간 6.2%p 달라도(0.21→0.15), big 토픽의 방향은 결측 구성 때문에 뒤집히지 않습니다. 이 결과는 S4 핵심 결과의 방향을 지지합니다.
# - 이전에 걱정한 "결측이 기간별로 다르다"는 점이 `only0`/`only1`로 직접 시험됐고, 그 시험을 통과했다는 의미입니다.

# ## 한계 (본문에 함께 적을 것)

# - **경계가 아니라 민감도입니다.** 가정은 "결측 논문의 저널 분포는 관측과 같고, 저널 안에서 토픽 비중이 k배 편중"입니다. 결측 논문의 저널 분포가 다른 경우나 토픽 간 결합 편중은 다루지 않았습니다.
# - **범위가 big 토픽 50개의 부호에 한정**됩니다. big이 아닌 토픽의 big 전환과 4분면 라벨 변화는 계획서에서 이번 범위 밖으로 뒀습니다.
# - 판독 임계값(0.90/0.70)과 현실 범위 [0.5, 2]는 연구자 가정입니다. 이번에는 결과가 임계값 근처가 아니라 양 끝(1.00)이라 이 가정에 크게 좌우되지 않습니다.
# - 첫 시도의 코드 오류는 결과 계산 전에 났고 CHANGELOG에 기록돼 있습니다.

# ## 쓸 수 있는 문구

# > "임베딩이 없는 논문(분석 창 내 약 16%)에서 같은 저널·기간 안의 토픽 비중이 관측 논문의 0.5~2배로 편중된다고 가정해도, 큰 변화(big) 토픽 50개 모두의 변화 방향이 유지됐다. 부호가 바뀌려면 편중이 최소 5배 이상이어야 한다."

# 쓰면 안 되는 것은 "결측이 결론에 영향을 주지 않는다"(경계 아님)입니다.

# ## 기록

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B7a 결과 해석 메모(결정적, 대비 P0509->P2025): 동결 판독 3개 시나리오 모두 '유지'(big 50개 전부 부호 유지, k=1 포함). k* 최소 5.16(only0)/11.45(only1), both는 양의 k* 없음, 격자 [0.5,2] 안 k* 0개. 결측률 0.2097->0.1473. 민감도이며 경계 아님, 저널 분포 고정 가정, big 토픽 부호 한정, 4분면 라벨·big 전환 미검토. QUICK에서 기간별 결측률 열람. 첫 시도 코드 오류(93행)는 별도 기록 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_B7a_results.csv paper1/09_PA/PA_B7a_summary.csv paper1/09_PA/PA_B7a_missing_by_period.csv paper1/09_PA/PA_B7a_full_log.txt paper1/09_PA/PA_CHANGELOG.md paper1/pa_b7a.py
# git commit -m "PA-B7a results" && git tag -a PA-B7a-results-2026-10-21 -m "PA-B7a outputs, seen" && git push --follow-tags
# ```

# ## 다음 단계

# 끝난 PA는 A1, A2a·b·c, A3, B2, B2b, B4(부분), B5, B7a, C1, C2입니다. 남은 것은 다음과 같습니다.

# - **B7b/c** (분류기 대입, 저널×연도 IPW): S4의 `missingness_sensitivity.csv`가 이미 IPW 보정을 했으므로, `S4_final_summary.md`의 해당 줄을 보여 주시면 B7c가 중복인지 판단하겠습니다.
# - **B3**: F6 생성 코드 위치와 `BASE` 정의가 필요합니다.
# - **B9·B8**: S8 입력 구조가 필요합니다.
# - **B4b**: S4 재추정은 `s4_main.py`의 토픽 선택 로직(위에서 `big` 컷은 확인됨)으로 가능해졌을 수 있습니다.
# - **B6**: 수작업 시간 여부.
# - **A4**: 열람 이력 부록 표. 모든 PA 종료 후, 열람 사실(연도별 결측률, QUICK 출력 등)을 빠짐없이 적습니다.

# 입력이 가장 가까운 것은 B7c 판단용 요약 줄이므로, 아래 한 줄 출력을 붙여 주세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# grep -n -i "missing\|IPW\|결측" 04_S4/S4_final_summary.md | head -20
# head -5 04_S4/missingness_sensitivity.csv
# ```

In [ ]:
# # 논문 1 사후분석(PA) 종합 정리 (2026-10-21 14:02 기준)

# ## 1. 원칙 (변하지 않는 것)
# - S0~S8의 통과/실패 판정은 PA 결과로 바꾸지 않습니다.
# - 모든 PA는 실행 전에 계획을 해시로 동결하고, 태그를 GitHub에 푸시해 외부 시각을 남깁니다.
# - 방향과 무관하게 전부 보고하고, 표·그림에 `[Post-hoc]`를 붙입니다.
# - QUICK 실행은 오류 점검 전용이며, 열람한 수치는 CHANGELOG에 기록합니다.
# - v2는 "사전등록"이 아니라 "결과 열람 전 명세와 열람 이력 공개"입니다.

# ## 2. 지금까지 한 일

# ### 2-1. 재현성 정리 (PA-0)
# - S1~S8 manifest·meta의 파일 해시 152건이 모두 일치했습니다(불일치 0, 파일없음 0).
# - S4 코드 3종을 IPython 히스토리에서 복구했고, "최종 실행본임은 증명 못 함"이라고 기록했습니다.
# - 잘못된 S4 태그 2개(실제로는 산출물이 없는 커밋을 가리킴)를 삭제하고 사유를 기록했습니다.
# - `MASTER_MANIFEST.csv`와 `MASTER_TAGS.csv`를 생성했고, 모든 태그를 GitHub에 푸시했습니다.
# - **미완:** `oa_raw` 외부 백업(매체 없음), GitHub 저장소 Private 확인.

# ### 2-2. PA별 결과

# | PA | 질문 | 동결 판독 | 쓸 수 있는 문장 / 주의 |
# |---|---|---|---|
# | **A1** 검정력 | 점프가 있었다면 N2 검정이 잡았을까 | run 1은 주입 버그로 무효(보존). run 2: all·within "강화", fixed14 "곡선만 보고". 탐색: 2020은 대체로 탐지, 2008~2012는 사각지대, 연속 꺾은선 BIC는 2020을 꺾임으로 선택하지 않음 | "2020 부근에서 초과분 크기 점프가 있었다면 탐지됐을 가능성이 높다"까지. "체제 전환 부재 확인"은 금지 |
# | **B2** 표본 크기 | 2020 변화가 논문 수 증가의 착시인가 | 세 표본 모두 "유지" | 하향 추출이 20%뿐인 약한 시험. "표본 크기 차이만으로는 사라지지 않는다"까지 |
# | **B2b** 저널·코로나 | 특정 저널이나 주제 때문인가 | 상위 1·3·5 저널 제외, 토픽 71 제외는 각각 유지. 둘을 동시에 제외하면 점프 약화·변화점 불안정 | 2020은 단일 원인이 아닌 겹친 가속 |
# | **B5** S4 안정성 | 0.700이 경계인가 | 컷에 따라 0.66~0.77, 부호 반전 0 | "0.70 전후에서 움직임", 방향 축은 견고 |
# | **A2c** 부트스트랩 | 진입>적응이 구간 보정 후에도 유지되나 | 4쌍 중 3쌍(첫 쌍 제외). 두 구간 방식 모두 확실한 쌍은 마지막뿐. excess 구간은 교정 불충분 | 초과 진입분의 근거는 순열 대조 |
# | **A2b** 신규 정의 | "신규"를 바꾸면 | s6 3쌍, y5 3쌍, **y2 0쌍(뒤집힘)** | "신규 저자 주도"가 아니라 "진입(신규+복귀) 주도" |
# | **A2a** 이름 분할 | ID가 갈라지는 이름을 빼면 | 4쌍 모두 "둔감"(변화 ≤0.011), 마지막 쌍 0.113→0.122 | ID는 2020+만 있어 ID 재판별은 불가. 분할되지 않은 동명이인은 미검출 |
# | **B4** 등급 단순화 | F1+F3 통과 145개로 재추정 | S5: 3표본 모두 같은 판정·변화점 유지. S6: 3쌍 둔감, 첫 쌍만 "민감"(원값이 0 근처라 부호만 바뀜) | 제외 토픽 4개뿐인 약한 시험. S4 부분은 B4b로 이월 |
# | **C2** 모집단 | S7 국가 몫을 임베딩 보유 논문만으로 | 20칸 중 일치 11·차이 9. 중국 몫은 4기간 모두 일치, 미국·상위5 몫은 +0.01~0.03 | 수준은 ±0.03 이내에서 다름 |
# | **C1** 중국 소속 제외 | 진입 우세가 중국 소속 저자 없이도 남나 | **3개 시나리오 모두 "유지 안 됨"**. 진입−적응 +0.113 → a −0.101, b(신규·지속 모두 제외) −0.024 | 진입 우세와 중국 소속 유입은 분리 불가. 인과 아님 |
# | **A3** 선두 국가 | 미상 교정·균등 4구간·중국 제외 HHI | 6개 통계 모두 rec·main 방향 일치, 균등 4구간 12칸 방향 유지, HHI 반등은 4조합 모두 "중국 몫에 연동" | 알려진 국가 기준 top5 몫은 평탄(0.64→0.65). 포함 기준 증가는 미상 감소와 함께 움직임 |
# | **B7a** 결측 | 임베딩 없는 논문(16%)이 big 방향을 뒤집나 | 3개 시나리오 모두 "유지". big 50개 전부 부호 유지, 뒤집힘 배수 k* 최소 5.2 | 민감도이며 경계 아님. 저널 분포 고정 가정 |

# **B7 정리:** S4 IPW(라벨 유지 0.98)가 B7a와 보완 관계입니다. B7c(저널×연도 IPW)는 기존 모델에 상호작용이 없어 중복은 아니지만 기존 결과가 방향을 지지하므로 보류했습니다. B7b(분류기 대입)는 분류기 파일 위치를 몰라 미착수입니다.

# ## 3. 종합: 현재 논문이 말할 수 있는 것

# **쓸 수 있음**
# - 주제 구조 변화는 점프보다 연속적 재배치로 서술하는 편이 자료에 부합한다.
# - 2020 부근에는 코로나와 기술 주제가 겹친 가속이 있고, 표본 크기·저널 하나·주제 하나로는 사라지지 않는다.
# - 진입이 적응보다 큰 방향은 마지막 구간에서 가장 뚜렷하지만(부트스트랩 방식, 신규 정의, 이름 분할 축에서 유지), 중국 소속 저자를 제외하면 유지되지 않는다.
# - 알려진 국가 기준 상위 5개국 몫은 평탄하고, 집중도(HHI) 반등은 중국 소속 질량과 함께 움직인다.
# - S4 핵심 결과의 방향은 결측 가정 하에서 안정적이다.

# **쓰면 안 됨**
# - 체제 전환이 없음이 확인됐다 / 2020이 아티팩트가 아님(확정).
# - 신규 저자가 모든 구간을 주도했다 / 진입 주도가 중국 유입과 무관하다.
# - ID 기반 결과와 일치한다 / 동명이인 문제가 해결됐다.
# - 소속을 국적으로 해석, 모든 인과 해석.

# **구조상 가장 큰 변화:** "진입 주도 갱신"과 "중국 몫 증가"는 독립된 두 기여가 아니라 한 구조의 두 측면으로 묶어 서술해야 합니다(C1, A3-3).

# ## 4. 앞으로 할 분석

# | 순서 | 항목 | 내용 | 필요한 입력 | 비고 |
# |---|---|---|---|---|
# | 1 | **B4b** | S4 라벨을 145개 토픽 컷으로 재추정(원 delta 유지, 컷만 재계산) | "(b)로" 승인 | 결정적, 약한 시험 |
# | 2 | **B3** | 신생 토픽 4정의의 핵심 집합(n_defs≥3), D1′ 재구성 | F6 생성 코드 위치, `BASE` 정의 | |
# | 3 | **B9** | 소득군 내 상관, 더미 통제, 공급 정의 민감도 | S8 입력 구조 | S8 약점 핵심 |
# | 4 | **B8** | 미달 4개 도메인 부록 정렬, 사전 v2(새 표본 25편) | S8 입력, 수작업 시간 | v2는 "사후 개정 사전"으로 분리 표기 |
# | 5 | **B7b** | 제목 분류기 예측 토픽 대입 | 분류기 산출 파일 위치 | 정확도 0.508이라 방향 유지만 확인 |
# | 6 | B7c 경량판 | 저널×연도 IPW로 S4 big 라벨만 재계산 | 설계 결정 몇 개 | 선택 사항 |
# | 7 | **B6** | ORCID 소표본, 수작업 50쌍 | 연구자 시간 | 병렬 진행 |
# | 8 | **A4** | 동결·열람 이력 부록 표 | 모든 PA 종료 후 | 열람 항목 빠짐없이 기재 |

# **A4에 반드시 적을 열람 이력:** s5_lib 소스와 N2 요약(v2 동결 전), B2의 QUICK 출력(기술 표와 요약 포함), A1 탐색 결과, QUICK 단계의 수치(CN 소속 이름 수 27,053, 연도별 결측률, 논문 수 진단 등), 첫 B7a 시도의 코드 오류와 수정.

# ## 5. 잔여 사항과 마무리
# - **PA-0 잔여:** `oa_raw` 외부 백업, GitHub 저장소 Private 확인.
# - **본문 처리:** 모든 PA 표·그림에 `[Post-hoc]`, 기각·약화된 PA(C1 반전, y2 역전, B4 첫 쌍 민감)를 부록과 한계 절에 먼저 공개, 영문 원고에 "Post-hoc analyses were specified after inspecting the primary results and are reported as exploratory; no pre-registered decision rule was altered." 삽입.
# - **투고 준비:** S9(후보 풀·추출 규칙 동결), S10(저널 선례 3편 이상, 심사 공격-대응표, 생성형 AI 사용 선언). 제목에서 "체제 전환"을 빼고 진입 중심 갱신과 선두 국가 교체를 한 구조로 묶는 구성을 권합니다.

# ## 6. 지금 필요한 답
# 1. B4b를 "(b)로" 진행할지
# 2. B3용 F6 생성 코드 위치와 `BASE` 정의, B9·B8용 S8 입력 구조 출력
# 3. 수작업 인력 여부(B6 50쌍, B8 25편)
# 4. 투고 후보 저널(구성안과 초록 초안을 맞추기 위해)

# 가장 가벼운 다음 한 걸음은 B4b이므로 "(b)로"라고 답해 주시면 동결 코드를 드리겠습니다.

In [ ]:
# # PA-B4b 동결 코드: `paper1/pa_b4b.py`

# 설계는 앞서 말씀드린 (b)안입니다. 몫(`delta`, `within`)과 질 라벨(`quality_high`)은 `topic_quadrants.csv`의 기존 값을 그대로 쓰고, `big` 분위 컷만 F1·F3 통과 145개 토픽 안에서 다시 구합니다.

# - **판독 1:** 원본에서 `big`이던 통과 토픽 중 새 컷에서도 같은 4분면 라벨을 유지하는 비율이 0.95 이상이면 "둔감"입니다.
# - **판독 2:** 안정성(전체, A등급)의 변화가 둘 다 |0.03| 이하이면 "둔감"입니다.
# - 두 판독은 따로 보고하고 합치지 않습니다. 임계값은 임의의 동결 사양입니다.
# - 이 코드는 미실행이고 무작위가 없는 결정적 계산입니다.

# ```python
# # -*- coding: utf-8 -*-
# # PA-B4b: S4 4분면 라벨을 F1+F3 통과 145개 토픽의 big 컷으로 재계산(delta·within·quality_high는 기존 값). 결정적. 미실행 코드.
# # MODE: "freeze" -> "tag" -> (git push --follow-tags) -> "run"(QUICK=True: 재현 점검만) -> "run"(QUICK=False)
# # S0~S8 판정 불변. 모든 산출물 [Post-hoc].
# import json, hashlib, subprocess, warnings
# from pathlib import Path
# import numpy as np, pandas as pd
# warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
# MODE = "freeze"
# QUICK = True

# ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; PA = P1/"09_PA"; S3, S4 = P1/"03_S3", P1/"04_S4"
# now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
# sha_b = lambda b: hashlib.sha256(b).hexdigest(); sha_f = lambda p: sha_b(Path(p).read_bytes())
# bl = lambda s: s.astype(str).str.strip().str.lower().isin(["1", "1.0", "true", "t", "yes"])
# git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
# class Halt(Exception): pass
# spec_hash = lambda s: sha_b(json.dumps({k: v for k, v in s.items() if k not in ("sha256", "created", "refs")}, sort_keys=True, ensure_ascii=False).encode())
# V2P, BP = PA/"PA_0_PLAN_v2.json", PA/"PA_B4b_PLAN.json"

# PLAN = dict(
#     version=1, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(PA-B4의 S4 부분). s4_main.py 80~100행(big=|x|의 2/3 분위 이상, quality_high=q_mean>0.5 and q_lo>0.5), S4 요약(안정성 0.70/0.69), PA-B5·B4 결과는 열람한 상태. PA-B4b 결과 미열람. S0~S8 판정 불변.",
#     grade_def="L7_topic_grade.csv의 F1, F3 열(True=실패). 통과 = 둘 다 False. 재현 점검: 정확히 145개",
#     option="(b): delta, within, quality_high는 topic_quadrants.csv 값 그대로(149개 토픽 기준 몫). big 컷만 통과 145개 토픽의 |delta|(또는 |within|)의 2/3 분위(pandas quantile, 선형 보간)로 재계산. 질 지표는 재추정하지 않음",
#     labels="s4_main.py label()과 동일: small_change 또는 (growth|decline)+(_highQ|_notHighQ). 총변화 기준 quadrant, 저널 내 성분 기준 quadrant_within",
#     stats="(1) 통과 토픽 중 원본 big의 새 컷 big 유지율과 같은 4분면 라벨 유지율, 새 컷에서 새로 big이 된 토픽 수, (2) 안정성 = big 토픽 중 quadrant==quadrant_within 비율을 전체와 A등급 각각, 원본(같은 145개 부분집합 기준) 대비, (3) 제외 4개 토픽의 원본 big 여부와 라벨",
#     reading="판독1: 원본 big 통과 토픽의 같은 라벨 유지율 >=0.95 이면 '둔감', 아니면 '민감'. 판독2: 안정성(전체)·안정성(A) 변화 모두 |.|<=0.03이면 '둔감', 아니면 '민감'. 두 판독을 합치지 않음. 임계값은 임의의 동결 사양",
#     reproduce_check="(a) 통과 토픽 145개, (b) 149개 전체에서 재계산한 big, quadrant, quadrant_within이 topic_quadrants.csv와 전부 일치, (c) 전체 안정성 round 2자리 0.70, A등급 0.69. 하나라도 어긋나면 중단",
#     caveat="제외 토픽이 4개뿐이라 약한 시험. 몫을 재정규화하지 않아 145개 기준 몫이 아님. 질 라벨은 149개 기준 인용 백분위 구간 그대로. 등급의 타당성은 검증하지 않음",
#     forbidden="결과를 본 뒤 통과 정의·임계값(0.95, 0.03) 변경(변경 시 v2 계획으로 분리하고 사유 기록)")

# def freeze():
#     if not V2P.exists(): raise Halt("PA_0_PLAN_v2.json이 없습니다")
#     h = spec_hash(PLAN); print("PA-B4b 계획 해시(예정)", h[:12])
#     if BP.exists(): print("이미 동결:", "동일" if json.loads(BP.read_text(encoding="utf-8"))["sha256"] == h else "⛔ 다름"); return
#     refs = {f.name: sha_f(f) for f in [V2P, S3/"L7_topic_grade.csv", S4/"topic_quadrants.csv", P1/"s4_main.py", PA/"PA_B4_PLAN.json"] if f.exists()}
#     BP.write_text(json.dumps(dict(PLAN, created=now(), sha256=h, refs=refs), ensure_ascii=False, indent=1), encoding="utf-8")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f:
#         f.write(f"\n- {now()} | PA-B4b 계획 동결(해시 {h[:12]}) | 사유: S4 라벨을 F1+F3 통과 145개 big 컷으로 재계산(옵션 b). 제외 4개뿐인 약한 시험. 결과 미열람. QUICK은 재현 점검만 | 본 결과 변경 여부: 없음\n")
#     print("저장: PA_B4b_PLAN.json")

# def tag():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     t = f"PA-B4b-plan-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
#     if git("tag", "-l", t).stdout.strip(): print("이미 있음:", t); return
#     r = git("add", "--", "paper1/09_PA/PA_B4b_PLAN.json", "paper1/09_PA/PA_CHANGELOG.md", "paper1/pa_b4b.py"); print("add", r.returncode, r.stderr.strip())
#     r = git("commit", "-m", "PA-B4b plan freeze (before execution)"); print("commit", r.returncode, (r.stdout + r.stderr)[:200])
#     r = git("tag", "-a", t, "-m", "PA-B4b plan"); print("tag", r.returncode, r.stderr.strip()); print("※ 이어서 git push --follow-tags")

# def label(df, col, qh, subset=None):
#     base = df if subset is None else df[subset]
#     cut = base[col].abs().quantile(2/3); big = df[col].abs() >= cut
#     lab = ["small_change" if not b else ("growth" if d > 0 else "decline") + ("_highQ" if q else "_notHighQ") for b, d, q in zip(big, df[col], qh)]
#     return pd.Series(lab, index=df.index), big, float(cut)
# def stab(d, big, q1, q2):
#     return float((d.loc[big, q1] == d.loc[big, q2]).mean()) if big.any() else np.nan

# def run():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     sp = json.loads(BP.read_text(encoding="utf-8"))
#     if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(PLAN): raise Halt("동결 파일과 현재 계획이 다릅니다")
#     if not git("tag", "-l", "PA-B4b-plan-*").stdout.strip(): raise Halt("tag 단계를 먼저 실행하세요")
#     sfx = "_QUICK" if QUICK else ""
#     G = pd.read_csv(S3/"L7_topic_grade.csv", encoding="utf-8-sig"); ptop = set(G[~bl(G.F1) & ~bl(G.F3)].topic.astype(int))
#     if len(ptop) != 145: raise Halt(f"통과 토픽 수 재현 실패: {len(ptop)} (145여야 함)")
#     M = pd.read_csv(S4/"topic_quadrants.csv", encoding="utf-8-sig"); M["topic"] = M.topic.astype(int); M["qh"] = bl(M.quality_high)
#     # 재현: 149개 전체
#     q0, b0, _ = label(M, "delta", M.qh); qw0, bw0, _ = label(M, "within", M.qh)
#     if not ((q0 == M.quadrant).all() and (b0 == bl(M.big)).all() and (qw0 == M.quadrant_within).all()): raise Halt("전체 토픽 재계산 라벨이 topic_quadrants.csv와 다름")
#     M["quadrant_rc"], M["big_rc"], M["qw_rc"] = q0, b0, qw0
#     sA, sAA = stab(M, M.big_rc, "quadrant_rc", "qw_rc"), stab(M[M.grade == "A"], M[M.grade == "A"].big_rc, "quadrant_rc", "qw_rc")
#     if round(sA, 2) != 0.70 or round(sAA, 2) != 0.69: raise Halt(f"안정성 재현 실패: {sA:.3f}/{sAA:.3f}")
#     print(f"[재현 OK] 통과 토픽 145개, 전체 149개 라벨 일치, 안정성 {sA:.3f}/{sAA:.3f}")
#     if QUICK: print("[QUICK] 재현 점검만 수행. 결과 통계는 출력하지 않음."); return
#     P = M[M.topic.isin(ptop)].copy().reset_index(drop=True)
#     P["quad145"], P["big145"], cut_t = label(P, "delta", P.qh); P["qw145"], P["bigw145"], cut_w = label(P, "within", P.qh)
#     P["orig_big"] = bl(P.big); P["same_label"] = P.quad145 == P.quadrant
#     # 원본 기준(같은 145개 부분집합): 원본 big/라벨 그대로의 안정성
#     s_orig_all = stab(P, P.orig_big, "quadrant", "quadrant_within"); A = P.grade == "A"; s_orig_A = stab(P[A], P[A].orig_big, "quadrant", "quadrant_within")
#     s_new_all = stab(P, P.big145, "quad145", "qw145"); s_new_A = stab(P[A], P[A].big145, "quad145", "qw145")
#     ob = P[P.orig_big]; keep_big = float(ob.big145.mean()); keep_label = float(ob.same_label.mean()); new_big = int((P.big145 & ~P.orig_big).sum())
#     out = pd.DataFrame([dict(cut_total_orig=float(M.delta.abs().quantile(2/3)), cut_total_145=cut_t, cut_within_145=cut_w, n_big_orig=int(P.orig_big.sum()), n_big_145=int(P.big145.sum()), orig_big_still_big=keep_big,
#                              orig_big_same_label=keep_label, n_newly_big=new_big, stab_all_orig=s_orig_all, stab_all_145=s_new_all, stab_A_orig=s_orig_A, stab_A_145=s_new_A)])
#     ex = M[~M.topic.isin(ptop)][["topic", "grade", "delta", "within", "big", "quadrant", "quadrant_within"]]
#     P.to_csv(PA/"PA_B4b_results.csv", index=False, encoding="utf-8-sig"); out.to_csv(PA/"PA_B4b_summary.csv", index=False, encoding="utf-8-sig"); ex.to_csv(PA/"PA_B4b_excluded4.csv", index=False, encoding="utf-8-sig")
#     print("\n[Post-hoc] PA-B4b 요약(145개, 컷만 재계산)\n", out.round(4).T.to_string(header=False)); print("\n[제외 4개 토픽]\n", ex.round(4).to_string(index=False))
#     chg = P[P.orig_big & ~P.same_label][["topic", "grade", "delta", "quadrant", "quad145"]]; nb = P[P.big145 & ~P.orig_big][["topic", "grade", "delta", "quadrant", "quad145"]]
#     print("\n[원본 big인데 라벨이 바뀐 토픽]\n", chg.round(4).to_string(index=False) if len(chg) else "없음"); print("\n[새로 big이 된 토픽]\n", nb.round(4).to_string(index=False) if len(nb) else "없음")
#     r1 = "둔감" if keep_label >= 0.95 else "민감"; r2 = "둔감" if (abs(s_new_all - s_orig_all) <= 0.03 and abs(s_new_A - s_orig_A) <= 0.03) else "민감"
#     print(f"\n[동결 판독] 1) 같은 라벨 유지율 {keep_label:.3f} → {r1} | 2) 안정성 변화 전체 {s_new_all - s_orig_all:+.3f}, A {s_new_A - s_orig_A:+.3f} → {r2}")
#     print("※ 제외 4개뿐인 약한 시험. 몫은 149개 기준. 판독을 합치지 않음. S0~S8 판정 불변.")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as fh:
#         fh.write(f"- {now()} | PA-B4b 실행 | 정의: PA_B4b_PLAN {sp['sha256'][:12]} | 결과: PA_B4b_*.csv | 본 결과 변경 여부: 없음\n")

# try: {"freeze": freeze, "tag": tag, "run": run}[MODE]()
# except Halt as e: print(f"⛔ 중단: {e}")
# ```

# ## 실행 순서

# 한 블록씩 실행하고, `⛔`나 오류가 나면 거기서 멈추고 출력만 붙여 주세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 pa_b4b.py                                                           # 1) freeze
# ```
# ```bash
# sed -i 's/^MODE = "freeze"/MODE = "tag"/' pa_b4b.py && python3 pa_b4b.py     # 2) tag
# cd .. && git push --follow-tags && cd paper1
# ```
# ```bash
# sed -i 's/^MODE = "tag"/MODE = "run"/' pa_b4b.py
# ( time python3 -u pa_b4b.py ) 2>&1 | tee 09_PA/PA_B4b_quick_log.txt         # 3) QUICK: 재현 점검만
# ```
# ```bash
# sed -i 's/^QUICK = True/QUICK = False/' pa_b4b.py && grep -n '^MODE\|^QUICK' pa_b4b.py
# cd .. && git add paper1/pa_b4b.py paper1/09_PA/PA_B4b_quick_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-B4b: QUICK done; QUICK=False" && git tag -a PA-B4b-fullrun-2026-10-21 -m "pre-run" && git push --follow-tags
# ```
# ```bash
# cd paper1 && ( time python3 -u pa_b4b.py ) 2>&1 | tee 09_PA/PA_B4b_full_log.txt   # 4) 본 실행
# ```

# ## 주의할 점

# - **미실행 코드입니다.** 가장 가능성 높은 중단은 재현 점검의 `전체 토픽 재계산 라벨이 topic_quadrants.csv와 다름`입니다. 이 경우 임의로 고치지 말고 메시지만 붙여 주세요. `big` 열이 문자열로 저장됐을 가능성에 대비해 `bl()`로 읽도록 했습니다.
# - **안정성 재현**은 `round(…, 2)`로 0.70과 0.69를 확인합니다. 정확한 값은 S4 출력(0.700, 0.690)과 PA-B5 재현(0.700/0.690)이 이미 맞춰 둔 값입니다.
# - 4분면 라벨 비교는 원본 라벨(`quadrant`)과 145개 컷 라벨(`quad145`)의 문자열 일치로 합니다. `quality_high`는 고정이므로 라벨이 바뀐다면 `big` 경계 이동 때문입니다.
# - 결과는 두 판독을 따로 읽겠습니다. 제외 4개뿐이라 "민감"이 나오기 어렵고, "둔감"이 나와도 "등급 선택이 결과를 좌우하지 않는다"까지만 쓸 수 있습니다.

# QUICK 로그와 본 실행 로그를 붙여 주세요.

In [ ]:
# # PA-B4b 판독

# 재현 점검(통과 토픽 145개, 149개 전체 라벨 일치, 안정성 0.700/0.690)을 통과했고, 본 실행도 정상 종료됐습니다. 동결 규칙을 그대로 적용하면 **판독 1은 "둔감", 판독 2는 "민감"**입니다. 두 판독은 합치지 않는다고 못 박았으므로 따로 적습니다.

# ## 1. 결과

# | 항목 | 원본(145개 부분집합) | 145개 컷 재계산 |
# |---|---|---|
# | big 컷 (총변화) | 0.0079 | 0.0078 |
# | big 토픽 수 | 48 | 49 |
# | 원본 big의 big 유지 / 같은 라벨 유지 | - | 1.000 / 1.000 |
# | 새로 big이 된 토픽 | - | 1개 (토픽 1) |
# | 안정성(전체) | 0.6875 | 0.7143 (+0.027) |
# | 안정성(A등급) | 0.6905 | 0.7209 (+0.030) |

# ## 2. 읽기

# 1. **판독 1 (라벨 유지율 1.000): 둔감.** 원본 big 48개는 모두 big으로 남았고 4분면 라벨도 하나도 바뀌지 않았습니다.
# 2. **판독 2 (안정성 변화): 규칙상 "민감"입니다.** A등급 변화가 +0.0304로 기준 0.03을 간발로 넘었습니다. 전체는 +0.027로 기준 안입니다.
# 3. **이 "민감"은 방향과 크기 모두 약한 신호로 읽어야 합니다.**
#    - 안정성이 떨어진 것이 아니라 올랐습니다.
#    - big 토픽이 48~49개일 때 토픽 하나가 안정성을 약 0.02~0.03 움직이는 입자성입니다. 기준 0.03은 토픽 한 개 크기라서 이 설계에서는 변별력이 거의 없는 임계값이었습니다.
#    - 이전에 "제외 토픽이 4개뿐이라 '민감'이 나오기 어렵다"고 말씀드렸는데, 틀렸습니다. 임계값이 입자 크기와 비슷하다는 점을 놓쳤습니다. 규칙은 바꾸지 않고, 이 판독을 있는 그대로 보고합니다.
# 4. **분자와 분모의 변화를 맞춰 보면 숫자 출처가 일부 설명됩니다.** 전체 안정성이 33/48에서 35/49로 바뀌었는데, 이는 분모 +1(토픽 1)에 분자 +2입니다. 토픽 1 하나만으로는 분자 +2가 안 됩니다. 저널 내 성분 기준(`quadrant_within`)의 컷도 같이 재계산되므로(0.0061), 다른 토픽의 within 라벨이 바뀌어 일치로 돌아섰을 가능성이 있습니다. 이는 가설입니다. `PA_B4b_results.csv`에서 `qw145`와 `quadrant_within`이 다른 토픽을 보면 확인됩니다.
# 5. **토픽 1은 경계 사례입니다.** `delta` 0.0078이 원본 컷 0.0079 바로 아래에 있다가, 컷이 0.0001 내려가면서 big이 됐습니다. 이는 PA-B5의 결론(컷에 따라 경계 토픽이 움직인다)과 같은 방향의 관찰입니다.
# 6. **제외된 4개 중 2개(토픽 0, 147)가 원래 big이었습니다.** 둘 다 `decline_notHighQ`이고 총변화와 저널 내 라벨이 같아서, 빠져도 안정성 분자와 분모가 같이 줄었습니다.

# ## 3. PA-B4 전체 종합

# | 부분 | 판독 |
# |---|---|
# | S5 (변화점·귀무 판정) | 세 표본 모두 동일, 둔감 |
# | S6 (진입−적응) | 4쌍 중 3쌍 둔감, 첫 쌍만 부호 때문에 "민감"(변화량 0.009) |
# | S4 라벨 | 같은 라벨 유지 둔감, 안정성 변화는 규칙상 "민감"(간발, 토픽 1개 규모) |

# 제외 토픽이 4개뿐이라 모두 약한 시험입니다. 쓸 수 있는 것은 "등급 선택이 주요 결과의 방향을 바꾸지 않는다"까지이고, 등급 자체의 타당성은 검증하지 않았습니다.

# ## 4. 본문에 쓸 문구 (영문 원고용)

# > Re-estimating the S4 labels on the 145 topics passing the F1 and F3 criteria left every originally large-change topic large-change with an unchanged quadrant label. Stability of the total-versus-within-journal labels moved from 0.688 to 0.714 (0.691 to 0.721 for grade-A topics); this shift corresponds to roughly one topic given the number of large-change topics (48–49) and exceeded our pre-set tolerance of 0.03 only for grade-A topics (by 0.0004). One borderline topic crossed the large-change cutoff (0.0078 vs. 0.0079).

# 쓰면 안 되는 것은 "등급 선택이 4분면 라벨에 영향이 없다"처럼 판독 2를 숨기는 문장입니다. 부록의 한계 절에 "임계값 0.03이 토픽 한 개 규모라 변별력이 낮았다"를 같이 적으세요.

# ## 5. 기록

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B4b 결과 해석 메모(결정적): 동결 판독 1) 원본 big의 같은 라벨 유지율 1.000 → 둔감, 2) 안정성 변화 전체 +0.027·A +0.030(A는 +0.0304로 기준 0.03을 간발로 초과) → 규칙상 민감. big 48→49(토픽 1, delta 0.0078 vs 원본 컷 0.0079에서 신규 big), 안정성 33/48→35/49. 안정성은 하락이 아니라 상승이며 변화 규모는 토픽 1개 수준이라 임계값 0.03이 입자 크기와 비슷했음(사후 인지, 규칙 불변). 분자 +2는 within 컷 재계산에 따른 다른 토픽의 within 라벨 변화 가능성(미확인 가설). 제외 4개 중 토픽 0·147은 원래 big. 약한 시험, 등급 타당성 미검증 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_B4b_results.csv paper1/09_PA/PA_B4b_summary.csv paper1/09_PA/PA_B4b_excluded4.csv paper1/09_PA/PA_B4b_full_log.txt paper1/09_PA/PA_CHANGELOG.md paper1/pa_b4b.py
# git commit -m "PA-B4b results" && git tag -a PA-B4b-results-2026-10-21 -m "PA-B4b outputs, seen" && git push --follow-tags
# ```

# ## 6. 다음 단계

# B4는 이것으로 닫힙니다. 남은 것은 B3(F6 생성 코드와 `BASE` 정의 필요), B9·B8(S8 입력 구조 필요), B7b(분류기 파일 위치 필요), B6(수작업), A4(열람 이력 부록)입니다.

# S8의 가장 큰 해석 위험이 소득 수준과의 겹침(B9)이므로, B9를 먼저 하시길 권합니다. S8 파일 구조는 제가 아직 본 적이 없어서 코드를 쓸 수 없습니다. 아래 읽기 전용 출력을 붙여 주세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# ls -la 08_S8 | head -40
# python3 - <<'EOF'
# import json, glob
# for f in sorted(glob.glob("08_S8/*PREREG*.json")) + sorted(glob.glob("08_S8/*meta*.json")):
#     j = json.load(open(f, encoding="utf-8")); print("\n==", f, list(j.keys()))
# EOF
# grep -n -i "partial\|spearman\|gdp\|income\|SBR\|min_country_mass" "S7. 지역 집중과 신규 유입, S8. 공급-부담 정렬.ipynb" | head -40
# ```

# 이 출력으로 편 ρ와 소득군 내 분석을 어느 파일에서 재현할지 정하겠습니다.

In [ ]:
# # PA-B9a 동결 코드: `paper1/pa_b9a.py`

# 필요한 정의는 모두 확인했습니다. 다만 `RX`, `YP`, `DOM_IND`, `INT`, `SPEC`은 노트북 셀 상단(1~94행)에 있고 아직 못 봤습니다. 추측으로 옮기면 재현 점검에서 막힐 수 있어서, **노트북 셀 상단을 그대로 실행해 정의를 가져오는 방식**으로 썼습니다.

# - 노트북에서 `def align():`, `def partial_spearman`, 그리고 `# ====== freeze` 구분선을 모두 가진 셀을 찾아, 구분선 앞부분만 별도 네임스페이스에서 실행합니다.
# - 동결 시점에 그 앞부분의 해시를 계획서 `refs`에 기록하고, 실행 시 해시가 다르면 중단합니다. 동결 후 노트북이 바뀌었는지 잡아냅니다.
# - 이 코드는 미실행입니다. 단, 노트북 셀 안에 특수 명령이 있으면 이 방식이 실패할 수 있습니다. 그때는 오류만 붙여 주세요.

# ## 동결 내용

# | 항목 | 내용 |
# |---|---|
# | 대상 | 통과 도메인 safety·transit, 기존 지표 쌍 4개. 제외 없음(`none`) 설정, `min_country_mass`=10 고정 |
# | 재현 점검(중단) | 전체×`gdp` 통제의 n, raw ρ, 편 ρ가 `country_partial_spearman.csv`의 `none` 행과 1e-6 이내. 내 편 상관 함수와 노트북 `partial_spearman`이 1e-9 이내 일치 |
# | 전체 표본 변형 | 통제 없음, `rank(gdp_pc)`(재현), 소득군 더미(HIC 기준), `rank(gdp)`+`rank(pop)` |
# | 소득군 내 | HIC, UMC, LMC, LIC 각각 통제 없음 / `rank(gdp_pc)` 통제. `n<20`이면 NA |
# | 부트스트랩 | B=1000, 국가 복원추출, 2.5·97.5 백분위 구간, seed 2026 |
# | 판독 | 행별로 보고. 부호가 전체×`gdp`와 같고 구간이 0을 배제하면 "관찰", 아니면 "비관찰", `n<20`은 NA. 행 간·소득군 간 합치지 않음 |

# 설계상 한계는 계획서에 미리 적었습니다.
# - 기존 편 ρ에서 구간이 0을 배제하는 것은 safety·road_deaths 하나뿐입니다. 나머지 세 쌍은 기준 부호부터 불확실하므로, "관찰" 여부를 이 세 쌍에서 해석하지 않습니다.
# - 소득군 안으로 쪼개면 n이 줄어 구간이 넓어집니다. 비관찰은 "효과 없음"이 아니라 검정력 한계일 수 있습니다.
# - 행이 48개이고 다중성 보정은 없습니다. 탐색적입니다.

# ## 코드

# ```python
# # -*- coding: utf-8 -*-
# # PA-B9a: S8 국가 수준 편 순위상관의 소득 수준 분리 진단(소득군 내 상관, 통제 변형). 미실행 코드.
# # MODE: "freeze" -> "tag" -> (git push --follow-tags) -> "run"(QUICK=True: 재현 점검과 층별 n만) -> "run"(QUICK=False)
# # 정의는 노트북 셀(def align / partial_spearman 포함)의 '# === freeze' 구분선 앞부분을 실행해 가져옴(해시 동결).
# # S0~S8 판정 불변. 모든 산출물 [Post-hoc].
# import json, hashlib, subprocess, warnings
# from pathlib import Path
# import numpy as np, pandas as pd
# from scipy.stats import rankdata
# warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
# MODE = "freeze"
# QUICK = True

# ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; PA = P1/"09_PA"; S8D = P1/"08_S8"
# NB = P1/"S7. 지역 집중과 신규 유입, S8. 공급-부담 정렬.ipynb"; MARK = "# ============================== freeze"
# now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
# sha_b = lambda b: hashlib.sha256(b).hexdigest(); sha_f = lambda p: sha_b(Path(p).read_bytes())
# git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
# class Halt(Exception): pass
# spec_hash = lambda s: sha_b(json.dumps({k: v for k, v in s.items() if k not in ("sha256", "created", "refs")}, sort_keys=True, ensure_ascii=False).encode())
# V2P, BP = PA/"PA_0_PLAN_v2.json", PA/"PA_B9a_PLAN.json"
# G4 = ["HIC", "UMC", "LMC", "LIC"]

# PLAN = dict(
#     version=1, seed=2026, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(PA-B9 중 (1) 소득군 내 상관과 (2) 통제 변형). S8 결과, country_partial_spearman.csv 전체(none·제외 행), 소득군별 국가 수(필터 전 HIC 86, UMC 59, LMC 47, LIC 25), 노트북 셀의 align L206~282와 partial_spearman 정의는 열람한 상태. PA-B9a 결과 미열람. 공급 정의(논문 수 기준)와 min_country_mass 5/20은 PA-B9b로 분리. S0~S8 판정 불변.",
#     scope="통과 도메인(domains_passed.json: safety, transit)의 기존 도메인-지표 쌍 4개(DOM_IND), 제외 없음(none) 설정, min_country_mass=10, 2007~2025 풀링 강도(S8 국가 수준과 동일)",
#     definitions="공급 강도 s = 국가의 도메인 d 질량 / 국가 총 질량(load_mass, 제목 정규식 플래그 pid 조인, tot>=min_country_mass), 부담 강도 b = burden_country_table의 INT[지표], z = gdp_pc, pop = pop. 코드는 노트북 셀 상단을 그대로 실행해 사용(해시 동결). 기본 df = (s,b,z) 결측 제거",
#     strata_controls={"ALL": ["none", "gdp", "income_dummy", "gdp_pop"], "HIC": ["none", "gdp"], "UMC": ["none", "gdp"], "LMC": ["none", "gdp"], "LIC": ["none", "gdp"]},
#     control_defs={"none": "통제 없음(순위 상관)", "gdp": "rank(gdp_pc) 통제 편 순위상관(S8 정의, 재현 대상)", "income_dummy": "소득군 더미(HIC 기준 3개) 통제: s,b는 순위, 더미는 순위 변환 없이 OLS 잔차 상관. 소득군이 G4인 국가만",
#                   "gdp_pop": "rank(gdp_pc)와 rank(pop) 이중 통제. pop 결측 국가 제외"},
#     n_min=20, B=1000, ci=[2.5, 97.5], boot="국가 복원추출, 재표집마다 순위·잔차 재계산(S8과 같은 방식), 층별 통제별로 독립 추출(seed 고정, 실행 순서 고정)",
#     reading="행(도메인x지표 x 층 x 통제)별로 보고. n<20이면 'NA'. 아니면 부호가 (ALL, gdp) 행의 부호와 같고 구간이 0을 배제하면 '관찰', 아니면 '비관찰'. 행·층·통제를 합쳐 하나의 결론으로 만들지 않음. 문구는 '소득 수준과 분리되지 않는 간극'이며 필요 충족 증거로 쓰지 않음",
#     reproduce_check="(a) 노트북 앞부분 해시가 동결 시점과 일치, (b) (ALL, gdp) 행의 n, raw rho, partial rho가 country_partial_spearman.csv의 excl=none 행과 1e-6 이내(4쌍 모두), (c) 이 코드의 편 상관 함수가 노트북 partial_spearman과 1e-9 이내. 하나라도 어긋나면 중단",
#     caveat="기존 편 rho에서 구간이 0을 배제하는 쌍은 safety-road_deaths 하나뿐이라 나머지 세 쌍의 '관찰' 여부는 해석하지 않음. 소득군 내는 n이 작아 구간이 넓고, 비관찰은 효과 없음이 아니라 검정력 한계일 수 있음. 소득군 더미와 gdp는 공선성이 큼. 48행, 다중성 보정 없음, 탐색적. 국가 단위 상관은 생태학적 오류 가능, 인과·필요 충족 증거 아님. 소득군 값은 wb_income_current 고정",
#     forbidden="결과를 본 뒤 통제·층·n_min·판독 규칙 변경(변경 시 v2 계획으로 분리하고 사유 기록)")

# def nb_prefix():
#     cells = json.loads(NB.read_text(encoding="utf-8"))["cells"]; cand = []
#     for i, c in enumerate(cells):
#         s = "".join(c.get("source", []))
#         if c.get("cell_type") == "code" and "def align():" in s and "def partial_spearman" in s and MARK in s: cand.append((i, s))
#     if not cand: raise Halt("노트북에서 def align / partial_spearman / freeze 구분선을 가진 셀을 찾지 못함")
#     i, s = cand[-1]; return i, s.split(MARK)[0], len(cand)

# def freeze():
#     if not V2P.exists(): raise Halt("PA_0_PLAN_v2.json이 없습니다")
#     i, pre, nc = nb_prefix(); h = spec_hash(PLAN); print("PA-B9a 계획 해시(예정)", h[:12], f"| 노트북 셀 {i}(후보 {nc}개) 앞부분 {len(pre):,}자")
#     if BP.exists(): print("이미 동결:", "동일" if json.loads(BP.read_text(encoding="utf-8"))["sha256"] == h else "⛔ 다름"); return
#     refs = {f.name: sha_f(f) for f in [V2P, S8D/"S8_0_PREREG_v2.json", S8D/"country_partial_spearman.csv", S8D/"burden_country_table.csv", S8D/"domains_passed.json", NB] if f.exists()}
#     refs["s8_prefix_sha256"] = sha_b(pre.encode("utf-8")); refs["s8_prefix_cell_index"] = i
#     BP.write_text(json.dumps(dict(PLAN, created=now(), sha256=h, refs=refs), ensure_ascii=False, indent=1), encoding="utf-8")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f:
#         f.write(f"\n- {now()} | PA-B9a 계획 동결(해시 {h[:12]}) | 사유: S8 편 순위상관의 소득 수준 분리 진단(소득군 내, 통제 변형). 노트북 셀 {i} 앞부분 해시 동결. PA-B9a 결과 미열람. QUICK은 재현 점검과 층별 n만 | 본 결과 변경 여부: 없음\n")
#     print("저장: PA_B9a_PLAN.json")

# def tag():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     t = f"PA-B9a-plan-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
#     if git("tag", "-l", t).stdout.strip(): print("이미 있음:", t); return
#     r = git("add", "--", "paper1/09_PA/PA_B9a_PLAN.json", "paper1/09_PA/PA_CHANGELOG.md", "paper1/pa_b9a.py"); print("add", r.returncode, r.stderr.strip())
#     r = git("commit", "-m", "PA-B9a plan freeze (before execution)"); print("commit", r.returncode, (r.stdout + r.stderr)[:200])
#     r = git("tag", "-a", t, "-m", "PA-B9a plan"); print("tag", r.returncode, r.stderr.strip()); print("※ 이어서 git push --follow-tags")

# def pr(x, y, cols=(), rk=()):
#     a, b = rankdata(x), rankdata(y)
#     if not len(cols): return float(np.corrcoef(a, b)[0, 1])
#     A = np.column_stack([np.ones(len(a))] + [rankdata(c) if r else np.asarray(c, float) for c, r in zip(cols, rk)])
#     res = lambda v: v - A @ np.linalg.lstsq(A, v, rcond=None)[0]
#     return float(np.corrcoef(res(a), res(b))[0, 1])

# def cols_of(sub, ctrl):
#     if ctrl == "none": return [], []
#     if ctrl == "gdp": return [sub.z.values], [True]
#     if ctrl == "gdp_pop": return [sub.z.values, sub["pop"].values], [True, True]
#     if ctrl == "income_dummy": return [(sub.inc.values == g).astype(float) for g in G4[1:]], [False]*3
#     raise Halt(f"알 수 없는 통제 {ctrl}")

# def subset(base, stratum, ctrl):
#     if stratum == "ALL":
#         if ctrl == "income_dummy": return base[base.inc.isin(G4)]
#         if ctrl == "gdp_pop": return base.dropna(subset=["pop"])
#         return base
#     return base[base.inc == stratum]

# def run():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     sp = json.loads(BP.read_text(encoding="utf-8"))
#     if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(PLAN): raise Halt("동결 파일과 현재 계획이 다릅니다")
#     if not git("tag", "-l", "PA-B9a-plan-*").stdout.strip(): raise Halt("tag 단계를 먼저 실행하세요")
#     sfx = "_QUICK" if QUICK else ""
#     i, pre, _ = nb_prefix()
#     if sha_b(pre.encode("utf-8")) != sp["refs"]["s8_prefix_sha256"]: raise Halt("노트북 앞부분이 동결 시점과 다름(노트북이 수정됨)")
#     ns = {"__name__": "s8_prefix"}; exec(compile(pre, "s8_prefix", "exec"), ns)
#     for k in ["SPEC", "YP", "DOM_IND", "INT", "load_F1X", "flag_domains", "load_mass", "partial_spearman", "check_spec"]:
#         if k not in ns: raise Halt(f"노트북 앞부분에 {k}가 없음")
#     ns["check_spec"]()
#     SPEC = ns["SPEC"]; cl = SPEC["country_level"]; INT, DOM_IND = ns["INT"], ns["DOM_IND"]
#     dom_ok = json.loads((S8D/"domains_passed.json").read_text(encoding="utf-8"))["passed"]
#     Bt = pd.read_csv(S8D/"burden_country_table.csv", keep_default_na=False, index_col=0); inc = Bt["income"].copy(); Bt = Bt.drop(columns="income").apply(pd.to_numeric, errors="coerce"); Bt["income"] = inc
#     F = ns["load_F1X"](); F = F[F.title.notna()]; fl = ns["flag_domains"](F.title); fl["pid"] = F.pid.values; fl = fl[["pid"] + dom_ok]
#     Mall = ns["load_mass"](); M = Mall.copy(); M["per"] = M.year.map(ns["YP"]); M = M.merge(fl, on="pid", how="inner")
#     tot = Mall.groupby("country").w.sum(); tot = tot[tot >= cl["min_country_mass"]]
#     ref = pd.read_csv(S8D/"country_partial_spearman.csv", encoding="utf-8-sig"); ref = ref[ref.excl == "none"].set_index(["domain", "indicator"])
#     base_of = {}
#     for d in dom_ok:
#         dm = M[M[d]].groupby("country").w.sum().reindex(tot.index).fillna(0); sup = dm/tot
#         for ind in DOM_IND[d]:
#             base_of[(d, ind)] = pd.DataFrame({"s": sup, "b": Bt[INT[ind]].reindex(tot.index), "z": Bt["gdp_pc"].reindex(tot.index), "pop": Bt["pop"].reindex(tot.index), "inc": Bt["income"].reindex(tot.index)}).dropna(subset=["s", "b", "z"])
#     # 재현 점검
#     for (d, ind), df in base_of.items():
#         r0 = ref.loc[(d, ind)]; ph = pr(df.s.values, df.b.values, [df.z.values], [True]); pn = ns["partial_spearman"](df.s.values, df.b.values, df.z.values); rw = pr(df.s.values, df.b.values)
#         if abs(ph - pn) > 1e-9: raise Halt(f"편 상관 함수가 노트북과 다름({d},{ind})")
#         if len(df) != int(r0.n) or abs(rw - float(r0.raw_rho)) > 1e-6 or abs(ph - float(r0.partial_rho)) > 1e-6: raise Halt(f"S8 재현 실패({d},{ind}): n {len(df)} vs {int(r0.n)}, raw {rw:.6f} vs {r0.raw_rho:.6f}, partial {ph:.6f} vs {r0.partial_rho:.6f}")
#     print(f"[재현 OK] {len(base_of)}쌍의 n, raw rho, 편 rho가 country_partial_spearman.csv(none)와 일치 | 노트북 셀 {i} 앞부분 해시 일치")
#     plan_rows = [(st, c) for st, cs in PLAN["strata_controls"].items() for c in cs]
#     NT = pd.DataFrame([dict(domain=d, indicator=ind, stratum=st, control=c, n=len(subset(df, st, c))) for (d, ind), df in base_of.items() for st, c in plan_rows])
#     NT.to_csv(PA/f"PA_B9a_n_by_stratum{sfx}.csv", index=False, encoding="utf-8-sig")
#     print("\n[층별 n(설계 정보)]\n", NT.pivot_table(index=["domain", "indicator"], columns=["stratum", "control"], values="n").astype(int).to_string())
#     B = 20 if QUICK else int(PLAN["B"]); rng = np.random.default_rng(int(PLAN["seed"])); rows = []
#     for (d, ind), df in base_of.items():
#         for st, c in plan_rows:
#             sub = subset(df, st, c); n = len(sub); row = dict(domain=d, indicator=ind, stratum=st, control=c, n=n, B=B)
#             if n < PLAN["n_min"]: rows.append(dict(row, rho=np.nan, lo=np.nan, hi=np.nan)); continue
#             cols, rk = cols_of(sub, c); x, y = sub.s.values, sub.b.values; r = pr(x, y, cols, rk); bs = []
#             for _ in range(B):
#                 k = rng.integers(0, n, n); bs.append(pr(x[k], y[k], [cc[k] for cc in cols], rk))
#             rows.append(dict(row, rho=r, lo=float(np.nanpercentile(bs, PLAN["ci"][0])), hi=float(np.nanpercentile(bs, PLAN["ci"][1]))))
#     R = pd.DataFrame(rows); main = R[(R.stratum == "ALL") & (R.control == "gdp")].set_index(["domain", "indicator"]).rho
#     R["main_sign"] = [np.sign(main.loc[(a, b)]) for a, b in zip(R.domain, R.indicator)]; R["sign_same"] = np.sign(R.rho) == R.main_sign; R["excludes0"] = (R.lo > 0) | (R.hi < 0)
#     R["verdict"] = np.where(R.rho.isna(), "NA", np.where(R.sign_same & R.excludes0, "관찰", "비관찰"))
#     R.to_csv(PA/f"PA_B9a_results{sfx}.csv", index=False, encoding="utf-8-sig")
#     if QUICK: print("\n[QUICK] 오류 없이 완료. 결과 통계는 출력하지 않음(_QUICK 파일을 열어 해석하지 마세요)."); return
#     print("\n[Post-hoc] PA-B9a 편·순위상관(전체 표본 통제 변형)\n", R[R.stratum == "ALL"].round(4).drop(columns=["B", "main_sign"]).to_string(index=False))
#     print("\n[Post-hoc] PA-B9a 소득군 내\n", R[R.stratum != "ALL"].round(4).drop(columns=["B", "main_sign"]).to_string(index=False))
#     print("\n[동결 판독] 행별 verdict(합치지 않음)\n", R.pivot_table(index=["domain", "indicator", "stratum"], columns="control", values="verdict", aggfunc="first").to_string())
#     print("※ 비관찰은 효과 없음이 아니라 검정력 한계일 수 있음. 기준 부호가 불확실한 세 쌍은 해석하지 않음. 48행 탐색적, 소득군은 wb_income_current 고정. S0~S8 판정 불변.")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as fh:
#         fh.write(f"- {now()} | PA-B9a 실행 | 정의: PA_B9a_PLAN {sp['sha256'][:12]} | 결과: PA_B9a_results.csv, PA_B9a_n_by_stratum.csv | 본 결과 변경 여부: 없음\n")

# try: {"freeze": freeze, "tag": tag, "run": run}[MODE]()
# except Halt as e: print(f"⛔ 중단: {e}")
# ```

# ## 실행 순서

# 한 블록씩 실행하고, `⛔`나 오류가 나면 거기서 멈추고 출력만 붙여 주세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 pa_b9a.py                                                           # 1) freeze
# ```
# ```bash
# sed -i 's/^MODE = "freeze"/MODE = "tag"/' pa_b9a.py && python3 pa_b9a.py     # 2) tag
# cd .. && git push --follow-tags && cd paper1
# ```
# ```bash
# sed -i 's/^MODE = "tag"/MODE = "run"/' pa_b9a.py
# ( time python3 -u pa_b9a.py ) 2>&1 | tee 09_PA/PA_B9a_quick_log.txt         # 3) QUICK
# ```
# ```bash
# sed -i 's/^QUICK = True/QUICK = False/' pa_b9a.py && grep -n '^MODE\|^QUICK' pa_b9a.py
# cd .. && git add paper1/pa_b9a.py paper1/09_PA/PA_B9a_*QUICK* paper1/09_PA/PA_B9a_quick_log.txt paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-B9a: QUICK done; QUICK=False" && git tag -a PA-B9a-fullrun-2026-10-21 -m "pre-run" && git push --follow-tags
# ```
# ```bash
# cd paper1 && ( time python3 -u pa_b9a.py ) 2>&1 | tee 09_PA/PA_B9a_full_log.txt   # 4) 본 실행
# ```

# ## 주의할 점

# - **가장 가능성 높은 중단 원인은 두 가지입니다.**
#   - `노트북에서 ... 셀을 찾지 못함` 또는 앞부분 실행 오류: 셀 상단에 노트북 전용 문법이 있을 때 나옵니다. 오류 메시지만 주세요.
#   - `S8 재현 실패`: 메시지에 n, raw, partial의 두 값이 같이 나옵니다. 그 값만 주세요. 정의를 임의로 고치지 마세요.
# - **QUICK 로그에는 층별 n 표가 출력됩니다.** 소득군 안의 표본 크기를 처음 보게 되므로, QUICK 후에 아래 한 줄을 기록하세요. 판독 규칙은 이 열람 전에 동결돼 있습니다.

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B9a QUICK 열람: 재현 점검 결과와 층별(소득군x통제) n 표를 열람함(설계 정보, 판독 규칙은 동결 전에 정함). 결과 통계는 미열람 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# ```

# - **미해결 항목은 그대로입니다.** B6의 50쌍과 B8-2의 새 표본 25편을 할 수 있는지는 아직 답을 받지 못했습니다. 어렵다면 두 항목은 "미수행, 한계로 명시"로 확정하겠습니다. `oa_raw` 외부 백업과 GitHub Private 확인도 열려 있습니다.

# QUICK 로그와 본 실행 로그를 붙여 주시면 동결 규칙대로 읽겠습니다.

In [ ]:
# # PA-B9a 판독

# 재현 점검을 통과했고(4쌍의 n, raw ρ, 편 ρ가 S8 산출물과 일치, 노트북 셀 24 앞부분 해시 일치), 본 실행도 정상 종료됐습니다. 동결 규칙대로 행별로 읽고, 합치지 않습니다.

# ## 1. 동결 판독 결과

# | 층 | 결과 |
# |---|---|
# | 전체 표본, safety·road_deaths | gdp, 소득군 더미, gdp+pop 세 통제 모두 "관찰". 통제 없음만 "비관찰" |
# | 전체 표본, 나머지 세 쌍 | 모든 통제에서 "비관찰" |
# | HIC 내 | 4쌍 모두 "비관찰" |
# | UMC, LMC | 전부 NA (n 10~19, 기준 20 미달) |
# | LIC | 전부 NA (n=0) |

# ## 2. 읽기

# 1. **safety·road_deaths의 편 ρ는 통제 방식을 바꿔도 거의 그대로입니다.** 0.237(gdp), 0.237(소득군 더미), 0.254(gdp+pop)로 안정적입니다. 이 쌍에 한해서는 "GDP 통제 방식 때문에 생긴 결과"는 아닙니다.
# 2. **그러나 구간 하한이 0 바로 위입니다.** 0.0077, 0.0002, 0.0010입니다. 소득군 더미 통제는 사실상 0에 닿습니다. "통제를 바꿔도 부호가 유지되고 하한이 0 근처에 머문다" 정도가 정확합니다. 통제 없음에서는 구간이 0을 포함합니다(하한 −0.005). 즉 이 쌍의 "관찰"은 통제를 추가했을 때 오히려 구간이 0 위로 올라간 경우입니다. 같은 부트스트랩 B=1000에서 구간이 0.00 근처를 오가는 수준이라, 규칙상 "관찰"이어도 강한 증거로 쓰면 안 됩니다.
# 3. **소득군 안에서는 어느 것도 관찰되지 않았습니다.** HIC 내 road_deaths는 gdp 통제에서 0.189이고 구간이 [−0.146, 0.487]입니다. 점추정은 양수지만 n=45에서 구간이 넓습니다. 이는 "소득군 안에서 사라졌다"가 아니라 "소득군 안에서는 판단할 수 없다"입니다.
# 4. **검정력 한계가 구조적입니다.** UMC는 17~19, LMC는 10~11로 기준 20에 못 미치고, LIC는 필터 후 0개국입니다. 필터 전에는 25개국이었으므로 `min_country_mass` 10과 부담 지표 결측이 LIC를 전부 걸러낸 것입니다. 즉 이 분석은 **HIC 한 층만 실질적으로 검정**했습니다. 저소득·중저소득 국가에서의 정렬은 이 자료로는 말할 수 없습니다.
# 5. **세 쌍은 해석하지 않기로 했습니다.** vehicles, urban_pop, transport_co2는 전체 표본에서도 구간이 0을 포함하므로 소득 분리 여부를 가릴 기준 신호가 없습니다. 참고로 transport_co2는 소득군 더미 통제에서 −0.031로 −0.116에서 크게 움직였지만, 구간이 모두 0을 포함하므로 해석하지 않습니다.
# 6. **통제 없음 행의 "비관찰"에는 주의가 필요합니다.** 판독이 "부호가 (ALL, gdp)와 같고 구간이 0을 배제"로 되어 있어서, 통제 없는 행도 같은 기준으로 판정됩니다. road_deaths는 통제 없는 원 상관(0.247)도 구간이 0을 포함하므로, GDP 통제가 신호를 만든 것이 아니라 구간 폭이 부트스트랩에 따라 조금 달라진 것으로 보입니다. 가설이며 확인하지 않았습니다.

# ## 3. 본문에 쓸 수 있는 문구

# > "공급 강도와 부담 강도의 국가 수준 순위상관은 도로 안전 영역의 사망률 지표에서만 0 위였고(편 ρ≈0.24, 구간 하한 0.00~0.01), GDP, 소득군 더미, GDP+인구 통제에서 점추정이 유지됐다. 나머지 세 쌍은 어떤 통제에서도 0과 구분되지 않았다. 소득군 안에서는 고소득군(n=45~48)만 추정이 가능했고 구간이 0을 포함했으며, 중상위·중하위·저소득군은 표본이 부족하거나(n<20) 필터로 비었다."

# 쓰면 안 되는 것은 "소득과 무관하게 정렬된다", "소득군 안에서도 확인됐다", 필요 충족 증거로의 해석입니다. 지침서가 정한 문구는 "소득 수준과 분리되지 않는 간극"이며, 이번 결과는 그 문구를 바꾸지 않습니다.

# ## 4. 기록

# ```bash
# cd ~/Research/Minnesota_Transport
# pgrep -f pa_b9a.py || echo "종료 확인"
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B9a 결과 해석 메모(B=1000): 전체 표본에서 safety-road_deaths만 gdp(0.237,[0.008,0.443])·소득군 더미(0.237,[0.000,0.466])·gdp+pop(0.254,[0.001,0.463]) 통제에서 '관찰', 하한이 0 근처. 나머지 3쌍은 전 통제에서 비관찰(해석 안 함). 소득군 내는 HIC만 추정 가능(n 45~48, 전부 비관찰, 구간 넓음), UMC n 17~19·LMC n 10~11은 n<20으로 NA, LIC는 필터 후 0개국(필터 전 25개국)이라 NA. 통제 없음 행의 road_deaths도 구간이 0 포함(원 상관 0.247). 비관찰은 검정력 한계 가능. QUICK 단계에서 층별 n 표 열람. 문구는 '소득 수준과 분리되지 않는 간극'으로 제한 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_B9a_results.csv paper1/09_PA/PA_B9a_n_by_stratum.csv paper1/09_PA/PA_B9a_full_log.txt paper1/09_PA/PA_CHANGELOG.md paper1/pa_b9a.py
# git commit -m "PA-B9a results" && git tag -a PA-B9a-results-2026-10-21 -m "PA-B9a outputs, seen" && git push --follow-tags
# ```

# ## 5. 다음 단계

# B9의 (3)(공급 정의를 논문 수 기준으로, `min_country_mass` 5와 20)은 **B9b**로 남아 있습니다. 이번 결과에서 LIC가 필터 후 0개국이 된 이유가 `min_country_mass` 10이라면, 5로 낮추는 B9b가 LIC·LMC의 표본 크기를 직접 늘릴 수 있는 유일한 경로입니다. 이 점은 B9b의 판독 규칙에 "소득군별 n의 변화"를 먼저 보고하도록 넣겠습니다. 다만 질량표(논문 수 기준 공급)의 위치를 아직 확인하지 못했으므로, 아래 출력이 필요합니다.

# ```bash
# cd ~/Research/Minnesota_Transport
# python3 - <<'EOF'
# import pandas as pd, os
# p = "paper1/02_S2/country_mass_long.parquet"
# M = pd.read_parquet(p) if os.path.exists(p) else pd.read_csv("paper1/02_S2/country_mass_long.csv.gz", nrows=200000)
# print(M.dtypes); print(M.groupby(["scheme", "variant"]).size().to_string())
# EOF
# ```

# `scheme` 값 중에 논문 수 기준(균등 분할 또는 전수 계수)이 있는지에 따라, 새 질량표 없이 B9b가 가능한지 갈립니다.

# 수작업 인력 여부(B6 50쌍, B8-2 25편)는 이번에도 답을 받지 못했습니다. 어렵다면 두 항목을 "미수행, 한계로 명시"로 확정하겠습니다. `oa_raw` 외부 백업과 GitHub Private 확인도 열려 있습니다.

In [ ]:
# # B9b 설계 점검 결과: LIC는 자료상 불가, 하한 변경의 이득은 UMC·LMC에 한정

# ## 1. 진단에서 읽힌 것

# **LIC가 비는 원인은 질량 하한입니다.** 부담·GDP 가용 국가는 22~23개(veh_per1000은 17)인데, 질량 1 이상이 5개국, 5 이상이 1개국, 10 이상은 0개국입니다. 하한을 5로 낮춰도 LIC는 1개국입니다. 따라서 **B9b로도 LIC는 복구되지 않으며 "자료상 불가"로 확정**합니다. 하한을 1까지 내려도 5개국이라 n<20 기준에 못 미칩니다.

# **UMC와 LMC는 하한 5에서 개선되지만 제한적입니다.**

# | 소득군 | 하한 10 (B9a) | 하한 5 | 하한 20 |
# |---|---|---|---|
# | UMC | 17~19 | 20~22 | 14~15 |
# | LMC | 10~11 | 12~13 | 9 |

# - UMC는 하한 5에서 n이 20을 간신히 넘습니다(veh_per1000 20, 나머지 21~22). 소득군 내 상관을 처음 추정할 수 있게 되지만 구간은 매우 넓을 것입니다.
# - LMC는 하한 5에서도 12~13이라 n<20입니다. **LMC도 사실상 불가**입니다.
# - 하한 20은 UMC도 14~15로 n<20이라 소득군 내 추정이 전부 NA가 됩니다. 전체 표본과 HIC에서만 의미가 있습니다.

# **스킴 진단:** `equal`, `first_last`, `inv_n_authors`는 논문당 w 합이 같은 분포(평균 0.967)이고, `full_count`는 논문당 평균 3.7로 국가 수만큼 1을 주는 전수 계수입니다. 지침서의 "논문 수 기준"은 `full_count`가 가장 가깝습니다. 앞서 하한은 `inv_n_authors` 총질량 기준으로 고정하기로 했으므로 국가 집합은 스킴과 무관하게 같습니다.

# ## 2. 설계 조정 (동결 전 확정)

# B9b의 실익이 작아졌으므로 범위를 줄입니다.

# - **변형 축:** 공급 스킴 `inv_n_authors`(재현), `full_count`, `equal` × 하한 5, 10, 20(항상 `inv_n_authors` 총질량 기준) × variant main·rec. 총 18셋입니다.
# - **층:** 전체 표본(`gdp` 통제)과 HIC는 전 변형. **UMC는 하한 5에서만** n≥20이 되므로 그 칸만 추정하고 나머지는 NA. LMC·LIC는 계획서에 "자료상 n<20, 추정하지 않음"으로 명시합니다.
# - **재현 점검(중단):** `inv_n_authors`·main·하한 10이 B9a (ALL, gdp)·(HIC, gdp)와 1e-6 이내 일치.
# - **보고:** 각 변형의 소득군별 n을 먼저, 그다음 부호와 구간. 판독은 B9a와 같은 "부호가 기준 행과 같고 구간이 0을 배제하면 관찰". 변형을 합치지 않습니다. 기준 부호는 (ALL, gdp, inv_n, main, 하한 10) 행입니다.
# - **한계:** 기준 구간이 0을 배제하는 쌍은 safety·road_deaths 하나뿐이므로 나머지 세 쌍은 해석하지 않습니다. UMC 하한 5의 n은 20~22로 구간이 매우 넓을 것이고, 하한 5는 질량이 작은 국가(공급 강도 분모가 작아 잡음이 큼)를 포함합니다. 다중성 보정 없는 탐색적 분석입니다.

# ## 3. 판단이 필요한 점

# 솔직한 평가로, **B9b는 새 정보를 많이 주지 않을 가능성이 높습니다.** 전체 표본과 HIC는 스킴·variant 변형에 대한 강건성 점검이고, 소득군 내 분리라는 B9의 핵심 질문(UMC·LMC·LIC)에는 UMC 한 칸만 추가됩니다. 선택지는 두 가지입니다.

# 1. **B9b를 위 범위로 실행:** 스킴·variant 강건성을 부록에 싣습니다. 코드와 동결 절차는 B9a와 같은 틀입니다.
# 2. **B9b를 생략하고 "LIC·LMC는 자료상 불가, UMC는 하한 5에서만 경계 n"을 B9a의 한계 절에 기록:** 이번 진단표(소득군×하한별 n)를 부록 표로 쓰면 이미 이 내용을 전달합니다.

# 저는 강건성 점검의 가치가 낮지 않다면 1을 권하지만, 시간 대비 효과로는 2도 정당합니다. 어느 쪽인지 알려 주세요. 1이면 `pa_b9b.py`를 드립니다.

# ## 4. 기록

# 위 CHANGELOG 열람 줄은 이미 남기셨습니다. 진단 결과 해석 한 줄을 추가하세요.

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B9b 설계 점검 해석: LIC는 부담·GDP 가용 22~23개국이나 질량 하한 1에서 5개국, 5에서 1개국, 10에서 0개국이라 하한 변경으로도 n<20 불가(자료상 불가). LMC는 하한 5에서 12~13개국으로 여전히 n<20. UMC는 하한 5에서 20~22개국(경계), 하한 20에서 14~15개국. full_count는 논문당 w 합 평균 3.7(전수 계수), 나머지 3스킴은 평균 0.967. 소득군 내 분리 질문에 B9b가 추가하는 정보는 UMC 하한 5 한 칸 수준으로 판단 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_CHANGELOG.md && git commit -m "PA-B9b design-check notes" && git push
# ```

# ## 5. 열려 있는 항목

# - B6(수작업 50쌍), B8-2(새 표본 25편)의 인력 여부가 여전히 미정입니다. 정해지지 않으면 "미수행, 한계로 명시"로 확정합니다.
# - `oa_raw` 외부 백업, GitHub Private 확인.
# - 남은 분석: B3(F6 생성 코드 위치와 `BASE` 정의 필요), B8(미달 4개 도메인 부록 정렬), B7b(분류기 산출 파일 위치 필요), A4(열람 이력 부록).

In [ ]:
# # B3 입력 확인: 핵심 집합은 지금 가능, D1′ 재구성은 코드 위치가 아직 불확실

# ## 1. 확인된 것

# - **`04_S4/definition_sensitivity.csv`**에 `topic, grade, D1, D2, D3, D4, n_defs`가 이미 있습니다. 핵심 집합(n_defs≥3)과 정의 특이적 토픽(n_defs=1) 추출은 이 파일만으로 됩니다. 새 입력이 필요 없습니다.
# - **`F6_topics.csv`**는 `share_0509, share_1519, share_2025, emerging, declining`을 가집니다. 즉 F6는 기간 비중과 신생/쇠퇴 플래그만 담은 결과 표입니다. D1′(2005–09 대 2015–19 재구성)에 필요한 기간 비중은 여기서 직접 얻을 수 있지만, F6의 `emerging` 판정 규칙은 이 표에 없습니다.
# - **F6 생성 코드는 후보가 좁혀졌습니다.** `(2) 데이터셋정의.ipynb`, `(3) 심화분석.ipynb`, `paper1_plan.py` 등이 "F6"를 언급합니다. 어느 것이 `F6_topics.csv`를 쓰는지는 아직 모릅니다.
# - **D1 진단 메모**(S4 요약): D1과 가장 비슷한 조합은 0509→1519 diff(Jaccard 0.4)로 추정됐고, 이는 "추정이며 판정 불변"으로 기록돼 있습니다. D1′의 동기가 이미 여기 있습니다. 다만 이 추정은 정의가 확인된 사실이 아니므로, D1′를 만들기 전에 D1의 실제 정의를 코드에서 확인해야 합니다.

# ## 2. 설계: B3a와 B3b로 분리

# 앞선 방식대로 한 계획서에 합치지 않습니다.

# **B3a (지금 가능, 결정적)**
# - 정의별 토픽 수와 쌍별 Jaccard를 재현합니다(중단 조건: S4 요약의 0.04~0.47 범위와 일치).
# - 핵심 집합(n_defs≥3)과 정의 특이적 집합(n_defs=1)을 나눕니다.
# - 각 집합을 `topic_quadrants.csv`의 4분면과 `journal_decomposition.csv`의 저널 간·내 성분으로 재요약합니다. 이는 지침서 B3-1에 해당합니다.
# - 핵심 집합의 신규 저자 진입 몫(B3-3)은 S6 토픽 단위 분해가 필요해서 이번 범위에서 제외합니다.
# - 판독(사전 고정): 핵심 집합의 저널 내 성분 중앙값이 정의 특이적 집합보다 큰지 방향만 보고하고, 임계값 판정은 하지 않습니다.

# **B3b (D1′)**: F6 생성 코드에서 D1의 정의를 확인한 뒤 설계합니다. 코드를 못 찾으면 "미수행, 사유 명시"로 처리합니다.

# ## 3. B3a 코드를 쓰기 전에 필요한 출력

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# head -3 04_S4/journal_decomposition.csv; head -3 04_S4/topic_quadrants.csv
# wc -l 04_S4/definition_sensitivity.csv
# grep -n -i "emerging\|F6_topics\|D1\b" 04_S4/s4_main.py | head -20
# cd .. && python3 - <<'EOF'
# import json, re
# for nb in ["(2) 데이터셋정의.ipynb", "(3) 심화분석.ipynb"]:
#     for i, c in enumerate(json.load(open(nb, encoding="utf-8"))["cells"]):
#         s = "".join(c.get("source", []))
#         if c.get("cell_type") == "code" and re.search(r"F6_topics|emerging", s):
#             print(f"\n== {nb} 셀 {i} ==")
#             for ln, l in enumerate(s.splitlines(), 1):
#                 if re.search(r"F6_topics|emerging|declining|BASE|share_|thresh|quantile", l): print(f"L{ln:<4} {l.strip()[:200]}")
# EOF
# ```

# 첫 블록이 B3a에 필요하고, 두 번째 블록은 D1의 실제 정의를 찾기 위한 것입니다. D1′ 설계는 그 출력 뒤에 정하겠습니다.

# ## 4. 기록할 열람 사실

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B3 입력 확인: definition_sensitivity.csv 열 구조(topic, grade, D1~D4, n_defs), F6_topics.csv 열 구조, S4 요약의 D1 진단 메모(0509->1519 diff와 Jaccard 0.4 추정)를 열람함. 핵심 집합 결과는 미계산. B3a(핵심 집합 재요약)와 B3b(D1')로 분리 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_CHANGELOG.md && git commit -m "PA-B3 input-check note" && git push
# ```

# ## 5. 열려 있는 항목

# - B9b는 별말씀이 없어 **생략으로 확정**했습니다. 부록의 미수행 목록에 "추가 정보 미미" 사유로 넣습니다.
# - B6(50쌍), B8-2(25편) 인력 여부는 아직 미정입니다. 이번 주 안에 답이 없으면 "미수행, 한계로 명시"로 확정합니다.
# - `oa_raw` 외부 백업, GitHub Private 확인, B8(부록 정렬), B7b(분류기 파일), A4(열람 이력 부록)가 남아 있습니다.

In [ ]:
# # B3 입력 확인 결과: B3a는 지금 코드를 쓸 수 있고, D1′는 아직 막혀 있습니다

# 기록(커밋 `8b2466a`)은 푸시됐습니다.

# ## 1. 이번 출력에서 확인된 것

# - **`definition_sensitivity.csv`는 31개 토픽 행**입니다(`wc -l` 32, 헤더 제외). 4개 정의 중 하나 이상에서 신생으로 잡힌 토픽의 합집합으로 보이지만, 정의를 만든 규칙은 이 파일에 없습니다.
# - **`journal_decomposition.csv`와 `topic_quadrants.csv`의 열 구조**를 확인했습니다. 둘 다 `topic, s0, s1, delta, within, between, between_share`를 가지고, `topic_quadrants.csv`에는 `quadrant`, `quadrant_within`, `big`, `quality_high`가 있습니다. 핵심 집합 vs 정의 특이적 집합을 이 두 표로 재요약할 수 있습니다.
# - **F6 생성 코드는 `(2) 데이터셋정의.ipynb` 셀 1**입니다(L116~L119). `F6_topics.csv`의 `emerging`은 코드의 `EM` 집합에서 옵니다. 그런데 **`EM`을 정하는 규칙(L67~L116 사이)은 필터에 걸리지 않아 출력에 없습니다.**
# - **중요: F6의 `emerging`과 S4의 D1~D4가 같은 것인지 아직 모릅니다.** F6는 `(2) 데이터셋정의`(S0 이전 작업)의 산출물이고, D1~D4는 S4에서 정의한 것입니다. D1이 F6의 `emerging`을 그대로 쓴다고 가정하면 D1′ 설계가 달라지는데, 이는 확인된 사실이 아니라 가설입니다.
# - `04_S4/s4_main.py`는 없습니다(`paper1/s4_main.py`에 복구본이 있고, 이 grep은 경로를 잘못 잡았습니다). 코드 위치만 정정하면 됩니다.

# ## 2. B3a 설계 (지금 동결 가능)

# | 항목 | 내용 |
# |---|---|
# | 입력 | `definition_sensitivity.csv`, `topic_quadrants.csv`, `journal_decomposition.csv` |
# | 집합 | 핵심 = n_defs≥3, 정의 특이적 = n_defs=1, 중간 = n_defs=2(보고만) |
# | 재현 점검(중단) | 행 수 31, 토픽이 전부 `topic_quadrants.csv`에 존재, n_defs = D1~D4 합과 일치, 인접 정의쌍 Jaccard가 S4 요약 범위(0.04~0.47) 안 |
# | 보고 | 집합별 토픽 수, 등급 구성, 4분면(`quadrant`) 분포, `between_share`와 `within/delta` 중앙값과 사분위, `big` 비율 |
# | 판독(고정) | 핵심 집합의 `between_share` 중앙값이 특이적 집합보다 낮은지 높은지 **방향만** 보고. 임계값·유의성 판정 없음 |
# | 한계 | 집합 크기가 작을 가능성이 큼(핵심 집합은 31개 중 일부). 중앙값은 기술 통계이고 검정이 아님. 신규 저자 진입 몫(지침서 B3-3)은 S6 토픽 단위 분해가 필요해 이번 범위 밖 |

# 결과 방향에 대한 사전 예상은 쓰지 않습니다. 이 분석은 31개 토픽의 기술 요약이라 판별력이 낮다는 점을 계획서에 먼저 적겠습니다.

# ## 3. B3b(D1′)를 위해 필요한 출력

# `EM`의 정의와 D1~D4 생성 위치를 찾아야 합니다. 아래는 읽기 전용입니다.

# ```bash
# cd ~/Research/Minnesota_Transport
# python3 - <<'EOF'
# import json
# c = json.load(open("(2) 데이터셋정의.ipynb", encoding="utf-8"))["cells"][1]
# L = "".join(c["source"]).splitlines()
# print("\n".join(f"L{i+1:<4} {l[:220]}" for i, l in enumerate(L) if 100 <= i+1 <= 120))
# EOF
# cd paper1
# grep -n "D1\|D2\|D3\|D4\|n_defs\|definition_sensitivity" s4_main.py s4_close.py | head -30
# grep -n -i "D1\|D2\|D3\|D4" "S4. 신생·쇠퇴의 재정의.ipynb" | head -20
# ```

# - 첫 블록은 `EM`을 만드는 L100~L120입니다.
# - 두 번째는 복구본 `s4_main.py`와 `s4_close.py`에서 D1~D4의 정의 줄을 찾습니다. 없으면 S4 노트북에서 찾습니다.

# ## 4. 기록할 열람 사실

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B3 입력 확인 2차: journal_decomposition.csv·topic_quadrants.csv 열 구조 확인, definition_sensitivity.csv 31개 토픽 행, F6 생성 셀 위치((2) 데이터셋정의 셀 1 L116~119) 확인. EM 규칙과 D1~D4 정의 미확인, F6 emerging과 D1의 관계는 미확인 가설. 핵심 집합 결과 미계산 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_CHANGELOG.md && git commit -m "PA-B3 input-check note 2" && git push
# ```

# ## 5. 선택

# B3a는 위 설계로 바로 동결 코드를 드릴 수 있습니다. B3b 출력을 기다리지 않고 **B3a를 먼저 진행할지**, 아니면 D1 정의를 본 뒤 한 계획서로 묶을지 알려 주세요. 앞서의 원칙(한 계획서에 합쳤다가 고치면 v2가 된다)에 따라 저는 B3a 먼저를 권합니다.

# 열려 있는 항목은 그대로입니다: B6(50쌍), B8-2(25편) 인력 여부, `oa_raw` 외부 백업, GitHub Private 확인, B8, B7b, A4.

In [ ]:
# # D1 정의 확인: 지침서의 D1′는 전제가 어긋납니다

# 기록(커밋 `8dfbb92`)은 푸시됐습니다. 이번 출력으로 D1~D4의 정의가 확정됐고, 앞서 "미확인 가설"로 둔 부분이 정리됐습니다.

# ## 1. 확정된 것

# | 정의 | 실제 규칙 |
# |---|---|
# | D1 (= F6 `emerging`) | 2005–09 대 2015–19, 비율 `r=(s1+1e-4)/(s0+1e-4) ≥ 3` 이고 `s1 ≥ 0.005`. 토픽 수는 규칙이 정해서 가변(기대 14) |
# | D2 | S4 주 대비의 평활 로그 비율 상위 14 |
# | D3 | S4 주 대비의 Δs 상위 14 |
# | D4 | S4 민감도 대비의 Δs 상위 14 |

# - 앞서 세운 "F6 `emerging`이 D1과 같다"는 가설은 맞았습니다(`s4_main.py` L116).
# - **지침서 B3-2의 "D1을 2005–09 대 2015–19로 재구성한 D1′"는 의미가 없습니다.** D1이 이미 그 구간이기 때문입니다. D1이 D2~D4와 갈리는 원인은 구간이 아니라 두 가지입니다.
#   1. **대비 구간이 다릅니다.** D1은 2005–09 대 2015–19이고 D2·D3은 주 대비입니다. 주 대비의 끝 구간은 2020년 이후 기간으로 보이지만 아직 확인하지 못했습니다.
#   2. **선택 규칙이 다릅니다.** D1은 비율·점유율 문턱이고 D2~D4는 상위 14개 고정입니다.
# - S4 마무리의 D1 진단("가장 비슷한 것은 0509→1519 diff, Jaccard 0.40, 1.0이 아니면 F6 정의가 이것이 아님")도 이제 설명됩니다. F6는 diff가 아니라 비율 문턱을 쓰기 때문입니다.
# - 다중 정의의 구조적 한계도 보입니다. D2~D4는 모두 "상위 14"라 크기가 같고, D2·D3은 같은 주 대비에서 로그 비율과 Δs만 다릅니다. 따라서 `n_defs≥3`은 "서로 독립인 정의 3개가 합의했다"가 아니라 "비슷한 정의가 겹쳤다"일 수 있습니다. 본문에 써야 합니다.

# ## 2. D1′ 대체안 (B3b)

# D1의 이질성을 구간 효과와 규칙 효과로 분리하는 3가지 재구성이 가능합니다. 모두 사후 정의로 표기합니다.

# - **D1-규칙:** D1 구간(2005–09 대 2015–19)에서 상위 14개 고정 규칙(로그 비율 기준)을 적용합니다. 구간은 같고 규칙만 바뀝니다.
# - **D1-구간:** 비율 문턱 규칙(`r≥3, s1≥0.005`)을 주 대비 구간에 적용합니다. 규칙은 같고 구간만 바뀝니다.
# - **재현 점검(중단):** `F6_topics.csv`의 `share_0509`, `share_1519`로 `EM`을 다시 만들어 `emerging`과 정확히 일치해야 합니다.

# 단, `F6_topics.csv`의 비중은 `(2) 데이터셋정의`의 F2 기준이고 `topic_quadrants.csv`의 `s0`와 값이 조금 다릅니다(예: 토픽 0은 0.01513 대 0.01504). 두 기준을 섞으면 재현 점검이 깨질 수 있습니다. 주 대비 구간의 정확한 기간도 `S4` 동결 규칙에서 읽어야 합니다. 그래서 B3b는 B3a 뒤에 별도 계획으로 두겠습니다.

# ## 3. B3a 동결 코드 (`paper1/pa_b3a.py`)

# 핵심 집합(n_defs≥3), 중간(=2), 정의 특이적(=1)을 나눠 기술 통계만 냅니다. 판독은 `between_share` 중앙값의 **방향만** 보고하고, 유의성 판정은 하지 않습니다. 결정적 계산입니다.

# ```python
# # -*- coding: utf-8 -*-
# # PA-B3a: 신생 토픽 정의 4종(D1~D4)의 핵심 집합(n_defs>=3) 대 정의 특이적(n_defs=1) 재요약. 결정적. 미실행 코드.
# # MODE: "freeze" -> "tag" -> (git push --follow-tags) -> "run". S0~S8 판정 불변. 모든 산출물 [Post-hoc].
# import json, hashlib, subprocess, itertools, warnings
# from pathlib import Path
# import numpy as np, pandas as pd
# warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
# MODE = "freeze"

# ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; PA = P1/"09_PA"; S4 = P1/"04_S4"
# now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
# sha_b = lambda b: hashlib.sha256(b).hexdigest(); sha_f = lambda p: sha_b(Path(p).read_bytes())
# bl = lambda s: s.astype(str).str.strip().str.lower().isin(["1", "1.0", "true", "t", "yes"])
# git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
# class Halt(Exception): pass
# spec_hash = lambda s: sha_b(json.dumps({k: v for k, v in s.items() if k not in ("sha256", "created", "refs")}, sort_keys=True, ensure_ascii=False).encode())
# V2P, BP = PA/"PA_0_PLAN_v2.json", PA/"PA_B3a_PLAN.json"

# PLAN = dict(
#     version=1, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(PA-B3 중 핵심 집합 재요약 B3a). D1=F6.emerging(2005-09 대 2015-19, r>=3 and s1>=0.005), D2·D3=주 대비 상위 14(로그 비율·Δs), D4=민감도 대비 Δs 상위 14의 정의는 s4_main.py 열람으로 확인한 상태. definition_sensitivity.csv의 31개 토픽 행 수와 열 구조는 열람, 집합별 결과는 미열람. D1' 재구성(B3b)은 별도 계획. S0~S8 판정 불변.",
#     sets="핵심 = n_defs>=3, 중간 = n_defs==2(보고만), 정의 특이적 = n_defs==1. definition_sensitivity.csv의 31개 토픽",
#     join="topic_quadrants.csv(topic, between_share, delta, within, quadrant, quadrant_within, big, quality_high)와 topic으로 조인. grade는 definition_sensitivity.csv 값 사용",
#     stats="집합별: 토픽 수, 등급(A/B/C) 수, big 비율, quality_high 비율, quadrant·quadrant_within 분포, between_share와 within/delta의 중앙값·사분위",
#     reading="핵심 집합과 정의 특이적 집합의 between_share 중앙값 대소 방향만 보고. 임계값·유의성 판정 없음. 집합 간 차이를 효과로 해석하지 않음",
#     reproduce_check="(a) 행 수 31, (b) 토픽이 모두 topic_quadrants.csv에 존재, (c) n_defs가 D1~D4 합과 일치, (d) D1~D4 쌍별 Jaccard의 최소·최대를 소수 둘째 자리로 반올림하면 0.04와 0.47(S4 요약 범위). 어긋나면 중단",
#     caveat="집합이 작아 기술 통계이며 검정이 아님. D2~D4가 모두 상위 14 고정이고 D2·D3은 같은 대비라 n_defs>=3이 독립 정의 합의가 아님. D1은 구간과 규칙이 모두 달라 이질성의 원인을 분리하지 못함(B3b). between_share는 delta가 작으면 불안정. 신규 저자 진입 몫(지침서 B3-3)은 범위 밖",
#     forbidden="결과를 본 뒤 집합 정의·판독 규칙 변경(변경 시 v2 계획으로 분리하고 사유 기록)")

# def freeze():
#     if not V2P.exists(): raise Halt("PA_0_PLAN_v2.json이 없습니다")
#     h = spec_hash(PLAN); print("PA-B3a 계획 해시(예정)", h[:12])
#     if BP.exists(): print("이미 동결:", "동일" if json.loads(BP.read_text(encoding="utf-8"))["sha256"] == h else "⛔ 다름"); return
#     refs = {f.name: sha_f(f) for f in [V2P, S4/"definition_sensitivity.csv", S4/"topic_quadrants.csv", P1/"s4_main.py"] if f.exists()}
#     BP.write_text(json.dumps(dict(PLAN, created=now(), sha256=h, refs=refs), ensure_ascii=False, indent=1), encoding="utf-8")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f:
#         f.write(f"\n- {now()} | PA-B3a 계획 동결(해시 {h[:12]}) | 사유: 신생 정의 4종의 핵심 집합 재요약. D1~D4 정의 열람, 집합별 결과 미열람. 지침서의 D1' 재구성은 D1이 이미 2005-09 대 2015-19라 전제가 어긋나 B3b로 대체 | 본 결과 변경 여부: 없음\n")
#     print("저장: PA_B3a_PLAN.json")

# def tag():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     t = f"PA-B3a-plan-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
#     if git("tag", "-l", t).stdout.strip(): print("이미 있음:", t); return
#     r = git("add", "--", "paper1/09_PA/PA_B3a_PLAN.json", "paper1/09_PA/PA_CHANGELOG.md", "paper1/pa_b3a.py"); print("add", r.returncode, r.stderr.strip())
#     r = git("commit", "-m", "PA-B3a plan freeze (before execution)"); print("commit", r.returncode, (r.stdout + r.stderr)[:200])
#     r = git("tag", "-a", t, "-m", "PA-B3a plan"); print("tag", r.returncode, r.stderr.strip()); print("※ 이어서 git push --follow-tags")

# def run():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     sp = json.loads(BP.read_text(encoding="utf-8"))
#     if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(PLAN): raise Halt("동결 파일과 현재 계획이 다릅니다")
#     if not git("tag", "-l", "PA-B3a-plan-*").stdout.strip(): raise Halt("tag 단계를 먼저 실행하세요")
#     DS = pd.read_csv(S4/"definition_sensitivity.csv", encoding="utf-8-sig"); Q = pd.read_csv(S4/"topic_quadrants.csv", encoding="utf-8-sig")
#     ks = ["D1", "D2", "D3", "D4"]
#     if len(DS) != 31: raise Halt(f"행 수 {len(DS)} (31이어야 함)")
#     if not set(DS.topic) <= set(Q.topic): raise Halt("definition_sensitivity 토픽이 topic_quadrants에 없음")
#     for k in ks: DS[k] = bl(DS[k])
#     if not (DS[ks].sum(axis=1).values == DS.n_defs.values).all(): raise Halt("n_defs가 D1~D4 합과 다름")
#     S = {k: set(DS.topic[DS[k]]) for k in ks}; J = {f"{a}-{b}": len(S[a] & S[b])/len(S[a] | S[b]) for a, b in itertools.combinations(ks, 2)}
#     if round(min(J.values()), 2) != 0.04 or round(max(J.values()), 2) != 0.47: raise Halt(f"Jaccard 범위 재현 실패: {min(J.values()):.3f}~{max(J.values()):.3f}")
#     print("[재현 OK] 31개 토픽, n_defs 일치, 정의별 토픽 수", {k: len(v) for k, v in S.items()}, "| 쌍별 Jaccard", {a: round(b, 2) for a, b in J.items()})
#     X = DS.merge(Q[["topic", "between_share", "delta", "within", "quadrant", "quadrant_within", "big", "quality_high"]], on="topic", how="left")
#     X["within_ratio"] = X.within/X.delta; X["big"] = bl(X.big); X["quality_high"] = bl(X.quality_high)
#     X["set"] = np.where(X.n_defs >= 3, "core(>=3)", np.where(X.n_defs == 2, "mid(2)", "specific(1)"))
#     q = lambda s, p: float(s.quantile(p))
#     rows = []
#     for nm, g in X.groupby("set"):
#         rows.append(dict(set=nm, n=len(g), nA=int((g.grade == "A").sum()), nB=int((g.grade == "B").sum()), nC=int((g.grade == "C").sum()), big_share=g.big.mean(), qhigh_share=g.quality_high.mean(),
#                          bs_med=q(g.between_share, .5), bs_q25=q(g.between_share, .25), bs_q75=q(g.between_share, .75), wr_med=q(g.within_ratio, .5), wr_q25=q(g.within_ratio, .25), wr_q75=q(g.within_ratio, .75)))
#     R = pd.DataFrame(rows); X.to_csv(PA/"PA_B3a_topics.csv", index=False, encoding="utf-8-sig"); R.to_csv(PA/"PA_B3a_summary.csv", index=False, encoding="utf-8-sig")
#     print("\n[Post-hoc] PA-B3a 집합별 요약\n", R.round(3).to_string(index=False))
#     print("\n[Post-hoc] 총변화 기준 4분면 분포\n", pd.crosstab(X["set"], X.quadrant).to_string()); print("\n[Post-hoc] 저널 내 성분 기준 4분면 분포\n", pd.crosstab(X["set"], X.quadrant_within).to_string())
#     c, s = R.set_index("set").bs_med.get("core(>=3)", np.nan), R.set_index("set").bs_med.get("specific(1)", np.nan)
#     print(f"\n[동결 판독] between_share 중앙값 핵심 {c:.3f} 대 특이적 {s:.3f} → 방향: " + ("핵심이 더 낮음(저널 내 성분 비중 큼)" if c < s else "핵심이 더 높음(저널 간 성분 비중 큼)" if c > s else "같음"))
#     print("※ 31개 토픽 기술 통계이며 검정 아님. n_defs>=3은 비슷한 정의의 겹침일 수 있음. S0~S8 판정 불변.")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as fh:
#         fh.write(f"- {now()} | PA-B3a 실행 | 정의: PA_B3a_PLAN {sp['sha256'][:12]} | 결과: PA_B3a_topics.csv, PA_B3a_summary.csv | 본 결과 변경 여부: 없음\n")

# try: {"freeze": freeze, "tag": tag, "run": run}[MODE]()
# except Halt as e: print(f"⛔ 중단: {e}")
# ```

# ## 4. 실행 순서

# 결정적 계산이라 QUICK 단계는 두지 않았습니다. 한 블록씩 실행하고, `⛔`나 오류가 나면 거기서 멈추고 출력만 붙여 주세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 pa_b3a.py                                                           # 1) freeze
# ```
# ```bash
# sed -i 's/^MODE = "freeze"/MODE = "tag"/' pa_b3a.py && python3 pa_b3a.py     # 2) tag
# cd .. && git push --follow-tags && cd paper1
# ```
# ```bash
# sed -i 's/^MODE = "tag"/MODE = "run"/' pa_b3a.py
# cd .. && git add paper1/pa_b3a.py && git commit -m "PA-B3a: MODE=run (pre-run)" && git tag -a PA-B3a-fullrun-2026-10-21 -m "pre-run" && git push --follow-tags
# cd paper1 && ( time python3 -u pa_b3a.py ) 2>&1 | tee 09_PA/PA_B3a_full_log.txt   # 3) 본 실행
# ```

# ## 5. 기록

# D1 정의 확인 결과를 지금 남기세요.

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B3 정의 확인: s4_main.py L116-119와 (2) 데이터셋정의 L100-120 열람. D1=F6.emerging(2005-09 대 2015-19, r>=3 and s1>=0.005, 가변 크기), D2·D3=주 대비 상위 14(로그 비율·Δs), D4=민감도 대비 Δs 상위 14. 지침서의 D1'(2005-09 대 2015-19 재구성)는 D1이 이미 그 구간이라 전제 어긋남 -> B3b로 구간·규칙 분리 설계로 대체. D2~D4가 상위 14 고정이고 D2·D3이 같은 대비라 n_defs>=3은 독립 정의 합의가 아닐 수 있음 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_CHANGELOG.md && git commit -m "PA-B3 definitions confirmed" && git push
# ```

# ## 6. 열려 있는 항목

# - B3b(구간과 규칙 분리)에는 S4 동결 규칙의 주 대비·민감도 대비 정확한 기간 확인이 필요합니다. B3a 뒤에 설계하겠습니다.
# - B9b는 생략으로 확정했습니다. B6(50쌍), B8-2(25편)의 인력 여부는 아직 미정이고, 이번 주 안에 답이 없으면 "미수행, 한계로 명시"로 확정합니다.
# - `oa_raw` 외부 백업, GitHub Private 확인, B8(미달 4개 도메인 부록 정렬), B7b(분류기 파일 위치), A4(열람 이력 부록)가 남아 있습니다.

In [ ]:
# # PA-B3a 판독과 기록 상태 점검

# ## 0. 먼저: 실행 절차에서 생긴 사고 두 가지

# 1. **본 실행이 두 번 돌았습니다.** 첫 블록은 `MODE="run"`으로 바꿔 커밋·푸시한 뒤 정상 실행됐고(`c6f3b2d`, 태그 `PA-B3a-fullrun-2026-10-21`), 그 뒤 같은 블록을 한 번 더 붙여 넣어 두 번째는 `git commit`이 "변경 없음"으로 끝났습니다. `&&`로 묶인 덕에 태그·푸시는 중복되지 않았고, 계산은 결정적이라 두 번째 출력도 같습니다. 다만 `PA_B3a_full_log.txt`는 두 번째 실행이 덮어썼습니다. 내용은 같으므로 문제는 없습니다.
# 2. **결과 파일이 아직 커밋되지 않았습니다.** `PA_B3a_full_log.txt`, `PA_B3a_summary.csv`, `PA_B3a_topics.csv`가 untracked이고, 마지막 커밋(`1204e26`)은 이전 단계의 "정의 확인" 메모만 올렸습니다. 아래 5절 명령으로 올리세요.
# 3. **git status에 다른 이슈가 보입니다.** 저장소 루트의 `(1)~(3)*.ipynb`, `paper2_plan.py`, `scimago/`, `transport-atlas-data/` 등이 untracked이고, `S8.5. 사후분석.ipynb/.txt`는 modified입니다. 이 파일들은 제가 본 적이 없습니다. `S8.5. 사후분석`이 별도 분석 기록이라면 PA 체계와 중복·충돌이 없는지 확인이 필요합니다. 또 `(2) 데이터셋정의.ipynb`(F6 생성 코드)가 아직 미추적이므로, B3b에서 이 노트북을 근거로 쓰려면 먼저 커밋해 해시를 남겨야 합니다. 대용량 폴더(`scimago/`, `transport-atlas-data/`)는 `git add -A` 하지 말고 `.gitignore`를 먼저 확인하세요.

# ## 1. 재현 점검

# 통과했습니다. 31개 토픽, n_defs 일치, 정의별 14개씩, 쌍별 Jaccard 0.04~0.47입니다.

# ## 2. 동결 판독

# | 집합 | n | big 비율 | between_share 중앙값 | 비고 |
# |---|---|---|---|---|
# | 핵심(≥3) | 7 | 1.00 | 0.251 | 전부 A, 전부 성장 |
# | 중간(2) | 10 | 1.00 | 0.394 | 전부 A, 전부 성장 |
# | 특이적(1) | 14 | 0.357 | 0.199 | A 12·B 2, 9개가 small_change |

# 규칙상 방향은 "핵심이 특이적보다 between_share가 높다(0.251 대 0.199)"입니다.

# ## 3. 읽기

# 1. **방향 판독은 약합니다.** 핵심 사분위(0.223~0.498)와 특이적 사분위(0.123~0.411)가 크게 겹치고, n=7 대 14이며 검정이 없습니다. 중간 집합(0.394)이 가장 높아 핵심→특이적으로 단조롭지도 않습니다. "핵심이 저널 간 성분에 더 의존한다"고 쓰면 안 됩니다.
# 2. **더 의미 있는 관찰은 big 비율입니다.** 핵심·중간 17개는 모두 big이고 전부 성장(감소 0), 특이적은 14개 중 9개가 small_change입니다. 다만 이는 구조적 결과일 가능성이 큽니다. D2~D4가 변화량 상위 14 기준이라 여러 정의에 걸쳐 뽑히려면 변화량이 커야 하고, 그러면 big이 되기 쉽습니다. 즉 "합의한 토픽이 크게 변했다"는 정의의 부산물이며 독립적 발견이 아닙니다.
# 3. **신생 정의에서는 성장만 나오므로**(신생 후보에 쇠퇴가 없음) 4분면 분포의 해석 폭이 좁습니다.
# 4. **저널 내 성분 기준으로 보면 핵심 7개는 4분면이 그대로**(4+3)입니다. 핵심 집합은 저널 내 성분으로도 성장 라벨이 유지된다는 기술 사실이지만, 중간·특이적은 small_change로 빠지는 토픽이 있습니다(3, 6개).
# 5. 지침서가 기대한 "핵심 신생 집합이 신규 저자 진입으로 설명되는지"(B3-3)는 범위 밖이라 말할 수 없습니다.

# ## 4. 쓸 수 있는 문구

# > "신생 후보 31개 중 3개 이상의 정의에서 선택된 토픽은 7개였고, 모두 A등급이며 총변화와 저널 내 성분 기준에서 같은 성장 라벨을 유지했다. 하나의 정의에서만 선택된 14개 중 9개는 총변화 기준 small_change였다. 이 차이는 후보를 변화량 상위로 고르는 정의 구조와 일치하며, 신생의 독립적 합의 증거로 해석하지 않는다."

# 쓰면 안 되는 것: "핵심 집합은 저널 간 성분이 더 크다", "정의 간 합의된 신생 토픽이 확인됐다".

# ## 5. 기록과 커밋

# ```bash
# cd ~/Research/Minnesota_Transport
# pgrep -f pa_b3a.py || echo "종료 확인"
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B3a 결과 해석 메모(결정적): 재현 통과(Jaccard 0.04~0.47). 핵심(>=3) 7개 전부 A·big·성장, 중간(2) 10개, 특이적(1) 14개 중 9개 small_change. between_share 중앙값 0.251(핵심)/0.394(중간)/0.199(특이적)로 동결 방향 판독은 '핵심이 더 높음'이나 사분위가 크게 겹치고 비단조라 해석하지 않음. big 비율 차이는 D2~D4가 변화량 상위 14 기준인 정의 구조의 부산물 가능성. 본 실행은 실수로 2회 실행됨(결정적, 로그는 두 번째 실행본, 결과 동일). B3-3(신규 저자 진입 몫)과 D1' 재구성은 미수행(D1이 이미 2005-09 대 2015-19) | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_B3a_full_log.txt paper1/09_PA/PA_B3a_summary.csv paper1/09_PA/PA_B3a_topics.csv paper1/09_PA/PA_CHANGELOG.md
# git commit -m "PA-B3a results" && git tag -a PA-B3a-results-2026-10-21 -m "PA-B3a outputs, seen" && git push --follow-tags
# ```

# `PA_CHANGELOG.md`에는 이번 줄 외에도 로컬 수정분이 있었습니다(출력의 `modified`). 위 명령이 함께 올립니다.

# ## 6. 다음

# - **B3b**(구간·규칙 분리)는 주 대비 기간 확인이 필요하고, 효과 대비 정보량이 낮아 B9b처럼 **생략을 권합니다**. D1의 이질성이 구간과 규칙 중 무엇에서 오는지는 정의 차이를 본문에 서술하는 것으로 충분합니다. 이의가 없으면 미수행 목록에 "전제 어긋남, 정의 서술로 대체"로 넣겠습니다.
# - 남은 항목: B8(미달 4개 도메인 부록 정렬), B7b(분류기 파일 위치), B6·B8-2(인력 여부, 이번 주 안에 답이 없으면 "미수행, 한계로 명시"), `oa_raw` 외부 백업, GitHub Private 확인, `S8.5` 노트북 정체 확인, A4(열람 이력 부록).
# - 분석이 거의 소진됐으므로, 다음은 **A4 열람 이력 부록 표 작성**을 권합니다. `PA_CHANGELOG.md`가 원자료이므로 거기서 표를 뽑는 읽기 전용 스크립트를 드릴 수 있습니다.

In [ ]:
# 제가 가진 자료로 정리한 것입니다. `PA_0_PLAN.json`의 전체 항목표와 S1~S8, PA-A1·A2·B2의 결과 수치는 제게 없습니다. 그래서 확인된 것만 적고, 모르는 칸은 "확인 필요"로 표시했습니다.

# ## 1. 하기로 한 분석 계획

# **본 분석 (S0~S11).** 단계별 실행안이 있고, 현재 S0~S8까지 실행·동결했습니다. S9~S11의 내용은 제게 없습니다.

# | 단계 | 내용 |
# |---|---|
# | S0 | 사전등록·동결 (사건 정의를 임계 돌파 2%/5%/10% 다중 기준과 연속 노출 시계열로 수정, 저널 연혁 사건 포함) |
# | S7 | 지역 집중 |
# | S8 | 공급-부담 정렬 |
# | S1~S6 | 개별 내용은 자료에 없음 (S5에는 탐지기 `s5_lib.detect`가 있음) |

# **사후 교정분석 (PA, `PA_0_PLAN.json` v1→v2 동결).**
# - 우선순위로 정한 것은 PA-A1·A2·B2였습니다.
# - 그 밖에 메모와 대화에 나온 항목은 B3a, B3b, B3-3, B6, B7b, B8, B8-2, B9b, A4입니다.

# ## 2. 분석 결과

# | 항목 | 상태 | 결과 |
# |---|---|---|
# | S0 | 완료 | 검사 20건 통과 |
# | S7, S8 | 완료 | 태그 `S7-results`, `S8-results`로 동결 (수치는 확인 필요) |
# | PA-A1 | 탐지기를 S5 원본으로 교체해 v2 동결 | 시험·본 실행 결과는 확인 필요 |
# | PA-A2, B2 | 확인 필요 | |
# | **PA-B3a** | 완료 | 아래 참고 |
# | B9b | 생략 | 정보량이 낮아 생략하기로 함 |

# **PA-B3a 결과**
# - 재현 점검을 통과했습니다. 31개 토픽이고 쌍별 Jaccard는 0.04~0.47입니다.
# - 핵심(≥3개 정의) 7개는 전부 A등급·big·성장이고, between_share 중앙값은 0.251입니다.
# - 중간(2개) 10개는 전부 A등급·big·성장이고, 중앙값은 0.394입니다.
# - 특이적(1개) 14개 중 9개가 small_change이고, 중앙값은 0.199입니다.
# - 해석은 보수적으로 합니다. 방향 판독은 약하고(사분위 중첩, 비단조), big 비율 차이는 "변화량 상위 14" 정의 구조의 부산물일 가능성이 큽니다. 독립적 합의 증거로는 쓰지 않습니다.
# - 본문에 쓸 문구는 메모 4절의 것만 사용합니다.

# ## 3. 남은 분석 계획

# | 항목 | 처리 방침 |
# |---|---|
# | 결과 3개 파일과 `PA_CHANGELOG.md` | 커밋·태그·푸시 (메모 5절 명령) |
# | B3b | 생략 권고, "전제 어긋남, 정의 서술로 대체"로 기록 |
# | B3-3, D1' 재구성 | 미수행, 한계로 기재 |
# | B8 | 미달 4개 도메인 부록 정렬 |
# | B7b | 분류기 파일 위치 확인 |
# | B6, B8-2 | 인력 여부를 이번 주 안에 확인, 답이 없으면 "미수행, 한계로 명시" |
# | A4 | 열람 이력 부록 표 작성 (`PA_CHANGELOG.md`에서 읽기 전용 스크립트로 추출) |
# | 재현성·보관 | `oa_raw` 외부 백업, GitHub Private 확인, `S8.5` 노트북 정체 확인, `(2) 데이터셋정의.ipynb` 커밋 |
# | S9~S11 | 내용 확인 필요 |

# `PA_0_PLAN_v2.json`과 `PA_CHANGELOG.md`를 붙여 주시면 PA 항목별 계획·결과·상태를 빈칸 없이 한 표로 정리하겠습니다.

In [ ]:
# 푸시까지 정상 완료됐습니다(`61c1a16`, 태그 `PA-repo-cleanup-2026-10-21`). 스테이징 목록도 예상한 6개와 `.gitignore`뿐이었습니다.

# - **F6 생성 코드(`(2) 데이터셋정의.ipynb`) 해시가 확보됐습니다.** 이 커밋이 그 기준점입니다. 노트북 크기는 (1) 112KB, (2) 26KB, (3) 13KB로 문제가 없었습니다.
# - **`scimago/`(412K)는 paper1의 S2 입력일 가능성이 큽니다.** `paper1/02_S2/s2_measurement.py`가 scimago를 참조합니다. 재현성을 위해 올려야 할 수 있어서 아직 커밋하지 않았습니다.
# - **그 밖의 untracked는 계속 미커밋으로 둡니다.** `pemsbay_*`, `paper2_plan.py`, Atlas 계열, `choi_lab_data_explorer.ipynb`, `TRD_PeMS-BAY-2022.ipynb`가 해당합니다. `01_EDA_9개CSV_분포와특성.ipynb`(1.6MB)는 paper1 소속인지 확인이 필요합니다.

# ## 1. scimago 확인 (읽기 전용)

# ```bash
# cd ~/Research/Minnesota_Transport
# ls -la scimago | head -n 20
# grep -n -i "scimago" paper1/02_S2/s2_measurement.py | head -n 10
# ```

# 경로가 `../scimago/...` 같은 상대경로이고 파일이 작으면, 그대로 커밋해 재현성을 확보하는 편이 낫습니다. 출력을 본 뒤에 커밋 명령을 드리겠습니다.

# ## 2. A4 열람 이력 표 추출 스크립트

# `PA_CHANGELOG.md`를 읽기만 하고, 파일은 `/tmp`에 만듭니다. 저장소를 건드리지 않습니다.

# ```bash
# cd ~/Research/Minnesota_Transport
# python3 - <<'EOF'
# import re, csv
# src = "paper1/09_PA/PA_CHANGELOG.md"
# rows = []
# for ln in open(src, encoding="utf-8"):
#     if not re.match(r"^- \d{4}-\d{2}-\d{2}T", ln):
#         continue
#     parts = [p.strip() for p in ln.rstrip("\n").split(" | ")]
#     ts = parts[0][2:].strip()
#     body = parts[1] if len(parts) > 1 else ""
#     full = " | ".join(parts[1:])
#     m = re.match(r"(PA-[A-Za-z0-9]+)", body)
#     item = m.group(1) if m else "기타"
#     for key, lab in [("계획 동결","계획 동결"),("결과 해석","결과 해석"),
#                      ("설계 점검","설계 점검"),("입력 확인","입력 확인"),
#                      ("코드 오류","코드 오류"),("정정","정정"),
#                      ("실행","실행")]:
#         if key in body:
#             etype = lab; break
#     else:
#         etype = "기타"
#     h = re.search(r"해시 ([0-9a-f]{8,})", full)
#     chg = re.search(r"본 결과 변경[^:]*:\s*(.*)$", full)
#     seen = "열람함" in full
#     unseen = "미열람" in full
#     rows.append([ts, item, etype, h.group(1) if h else "",
#                  "있음" if seen else "", "미열람 명시" if unseen else "",
#                  chg.group(1).strip() if chg else "", body[:80]])
# out = "/tmp/A4_view_history.csv"
# with open(out, "w", newline="", encoding="utf-8-sig") as f:
#     w = csv.writer(f)
#     w.writerow(["time","item","type","plan_hash","viewed_mention","unviewed_stated","main_result_changed","summary"])
#     w.writerows(rows)
# print(len(rows), "rows ->", out)
# from collections import Counter
# print(Counter(r[2] for r in rows))
# print(Counter(r[6] for r in rows))
# EOF
# head -n 8 /tmp/A4_view_history.csv
# ```

# - **열 구성:** 시각, 항목, 유형, 계획 해시, 열람 언급 여부, 미열람 명시 여부, 본 결과 변경 여부, 요약입니다.
# - **열람 여부 판단은 단순 키워드 기준입니다.** "열람함"과 "미열람" 문자열로 구분하므로, 부록에 쓰기 전에 직접 훑어 확인하세요.
# - **이 표의 목적은 어느 단계에서 무엇을 이미 봤는지 보여주는 것입니다.** 사전 동결 대비 사후 열람 이력을 투명하게 밝히는 용도입니다.

# 출력(행 수와 유형별 개수)을 붙여 주시면 부록용 표 형식으로 다듬어 드리겠습니다.

# ## 남은 항목

# 1. `scimago/` 처리 → `01_EDA` 노트북 소속 확인
# 2. A4 표 완성
# 3. `oa_raw` 외부 백업 (미완료)
# 4. B7b(분류기 파일 위치), B8, B6(이번 주 내 결정)

In [ ]:
# ## 1. scimago 폴더

# 제 추측이 일부 틀렸습니다. `s2_measurement.py` 47행은 `SC = S1 / "scimago"`라서 코드는 저장소 루트의 `scimago/`가 아니라 `S1` 폴더 아래의 scimago를 읽습니다(`paper1/01_S1/scimago`로 보입니다). 루트의 `scimago/`는 원본이거나 복사본일 수 있습니다. 또 218~225행을 보면 이 파일이 없어도 코드는 멈추지 않고 Scimago 병기만 건너뜁니다. 그 병기는 Spearman을 보는 "그럴듯함 점검"이고 목표 수치가 아닙니다. 그러니 어느 폴더를 올릴지는 아래 출력으로 정하세요.

# ```bash
# cd ~/Research/Minnesota_Transport
# ls -la paper1/01_S1/scimago 2>/dev/null | head
# git ls-files paper1/01_S1/scimago | head
# git ls-files scimago | head
# diff -rq scimago paper1/01_S1/scimago 2>&1 | head
# grep -n "^S1\|S1 *=" paper1/02_S2/s2_measurement.py | head -3
# ```

# 판단 기준은 다음과 같습니다.
# - `paper1/01_S1/scimago`가 추적 중이고 루트와 내용이 같으면, 루트 `scimago/`는 중복이므로 커밋하지 않습니다.
# - `paper1/01_S1/scimago`가 없거나 추적되지 않으면, S2 재현에 필요한 입력이므로 그 위치로 맞춰 커밋합니다. 파일이 총 400KB 정도라 용량 문제는 없습니다.

# ## 2. A4 추출 결과

# 71행이 추출됐고, "본 결과 변경"은 71행 모두 "없음"입니다. 이 점은 부록에 그대로 쓸 수 있습니다. 다만 유형 분류가 거칩니다. 기타가 11개이고, PA-A1처럼 본문이 "PA-A1(v2)"뿐인 줄은 실행인지 동결인지 구분되지 않습니다. 또 "미열람 명시"는 줄 안에 그 단어가 있는지만 보는 것입니다. 예를 들어 v2 동결 줄은 "미열람 명시"로 잡혔지만, 같은 줄에 S5의 N2 요약 통계와 `s5_lib` 소스를 열람했다고도 적혀 있습니다. 두 칸은 배타적이 아니니 부록에 쓸 때는 항목별로 합쳐서 보는 편이 낫습니다.

# 그래서 PA 항목당 한 줄로 묶는 v2를 드립니다. 읽기 전용이고 `/tmp`에만 씁니다.

# ```bash
# cd ~/Research/Minnesota_Transport
# python3 - <<'EOF'
# import re, csv, collections
# src = "paper1/09_PA/PA_CHANGELOG.md"
# items = collections.OrderedDict()
# others = []
# for ln in open(src, encoding="utf-8"):
#     if not re.match(r"^- \d{4}-\d{2}-\d{2}T", ln): continue
#     parts = [p.strip() for p in ln.rstrip("\n").split(" | ")]
#     ts = parts[0][2:].strip(); full = " | ".join(parts[1:])
#     m = re.match(r"(PA-[A-Za-z0-9]+)", full)
#     if not m: others.append((ts, full[:100])); continue
#     d = items.setdefault(m.group(1), dict(plan_t="", hash="", runs=[], memo_t=[], seen=[], fix=[]))
#     h = re.search(r"해시 ([0-9a-f]{8,})", full)
#     if "계획 동결" in full or "동결(해시" in full:
#         d["plan_t"] = d["plan_t"] or ts; d["hash"] = d["hash"] or (h.group(1) if h else "")
#     elif "결과 해석" in full: d["memo_t"].append(ts)
#     elif "코드 오류" in full or "정정" in full: d["fix"].append(ts)
#     elif "실행" in full: d["runs"].append(ts)
#     for c in re.split(r"(?<=[\.;])\s+", full):
#         if "열람" in c and "미열람" not in c[:3]:
#             d["seen"].append(c.strip()[:110])
# with open("/tmp/A4_by_item.csv", "w", newline="", encoding="utf-8-sig") as f:
#     w = csv.writer(f)
#     w.writerow(["item","plan_freeze_time","plan_hash","n_runs","first_run","memo_times","fix_or_correction_times","viewed_before_result(first 2)"])
#     for k, d in items.items():
#         w.writerow([k, d["plan_t"], d["hash"], len(d["runs"]), d["runs"][0] if d["runs"] else "",
#                     ";".join(d["memo_t"]), ";".join(d["fix"]), " // ".join(d["seen"][:2])])
# print(len(items), "items -> /tmp/A4_by_item.csv")
# print("항목 미지정 줄 %d개:" % len(others))
# for t, s in others: print(" ", t, s)
# EOF
# column -s, -t < /tmp/A4_by_item.csv | cut -c1-170 | head -n 30
# ```

# 기타 11줄은 항목 이름이 붙지 않은 줄입니다. 위 출력에 이 줄들이 시각과 앞 100자로 나열됩니다. 부록에는 항목별 표 아래에 "기타 기록" 소표로 붙이면 됩니다.

# ## 남은 항목

# 1. scimago 확인 출력과 `01_EDA_9개CSV_분포와특성.ipynb` 소속 확인
# 2. A4 항목별 표 → 부록 형식으로 다듬기
# 3. `oa_raw` 외부 백업(미완료)
# 4. B7b(분류기 파일 위치), B8, B6(이번 주 내 결정)

In [ ]:
# 표는 대부분 맞게 나왔지만, 스크립트의 분류 때문에 틀린 칸이 4곳 있습니다. 부록에는 아래 수정본을 쓰세요.

# ## 1. 자동 추출에서 틀린 칸

# 1. **B7a:** "정정·오류" 칸의 `14:00`은 정정이 아니라 결과 해석 메모(14:00:36)입니다. 메모 안에 "첫 시도 코드 오류(93행)"라는 문구가 있어 오류로 분류됐습니다. 실제 정정은 13:59:53 한 건뿐입니다.
# 2. **A1:** 계획 칸이 비어 있습니다. 계획은 PA-0 행의 v2 동결(`a98f9697fa67`)입니다. 결과 해석 칸도 비었습니다. A1 줄이 "PA-A1(v2)" 형식이라 분류되지 않았습니다.
# 3. **PA-0:** 정정 12:31은 맞지만, 정정 대상인 12:31:17의 "oa_raw 외부 복사 완료" 줄이 표에 안 보입니다. 이 줄이 단독으로 남으면 사실과 다른 기록이 되므로 반드시 함께 표시해야 합니다.
# 4. **B2:** 결과 해석이 `10-21 11:49`로 나왔습니다. 실행은 10-19 18:21~18:22였으니 해석 메모가 이틀 뒤에 쓰인 것인데, 제가 본 changelog tail 범위 밖의 줄입니다. 맞는지 확인하세요.

# ```bash
# grep -n "PA-B2 " paper1/09_PA/PA_CHANGELOG.md | cut -c1-160
# grep -n "PA-A1" paper1/09_PA/PA_CHANGELOG.md | cut -c1-160
# ```

# ## 2. 수정본 표 (부록용)

# B2 해석 시각과 A1의 실행·해석 시각은 위 grep 결과로 채우세요. 나머지는 출력과 changelog를 근거로 고쳤습니다.

# | 항목 | 계획 동결(시각, 해시) | 실행 | 결과 해석 | 설계·입력 열람 | 정정 | 비고 |
# |---|---|---|---|---|---|---|
# | PA-0 | 10-19 17:11 `4178cadadec9`; 17:24 `a98f9697fa67`(v2, A1만 대체) | – | – | 10-19 17:18 해시 검증 152건 일치 | 10-21 12:31 | 12:31:17 "oa_raw 외부 복사 완료" 기록은 12:31:33에 정정(복사 실패, 외부 백업 미완료) |
# | PA-A1 | PA-0 v2 `a98f9697fa67`에 포함 | 3회(+탐색) | (확인) | – | – | run 1 무효(오류 원인 확정), run 2 동결 사양 |
# | PA-B2 | 10-19 18:18 `47c93a7c5739` | 3회 | (확인) | – | – | |
# | PA-A2c | 10-21 11:53 `6e9a0ce5ca3a` | 1회 | 12:03 | – | – | |
# | PA-A2b | 10-21 12:03 `e6bf19450afb` | 1회 | 12:09 | – | – | |
# | PA-B2b | 10-21 12:10 `3e49bd74320e` | 1회 | 12:12 | – | – | |
# | PA-B5 | 10-21 12:14 `85a7d60d3a9a` | 1회 | 12:15 | – | – | |
# | PA-A2a | 10-21 12:22 `150c834dcbd7` | 1회 | 12:30 | 12:20 | – | |
# | PA-B4 | 10-21 12:36 `67b9a2b0b294` | 1회 | 13:35 | – | – | |
# | PA-C2 | 10-21 13:39 `e9f378dfa8dc` | 1회 | 13:40 | – | – | |
# | PA-C1 | 10-21 13:46 `7621517d4801` | 1회 | 13:51 | – | – | |
# | PA-A3 | 10-21 13:54 `bf98b23ac193` | 1회 | 13:55 | – | – | |
# | PA-B7a | 10-21 13:59 `741fb6db9851` | 1회 | 14:00 | – | 13:59(코드 오류 수정, 사양 불변) | 첫 시도 TypeError로 중단, 오류 로그 보존 |
# | PA-B4b | 10-21 14:05 `9904855a7774` | 1회 | 14:07 | – | – | |
# | PA-B9a | 10-21 14:11 `276e376f9d2f` | 1회 | 14:12 | 14:08 | – | |
# | PA-B3a | 10-21 14:18 `b6a78adb8c60` | 2회 | 14:19; 14:22(동일 내용 중복) | 14:18(정의 확인) | 14:25 중복 정정 | 실행 2회는 실수, 결과 동일(결정적) |
# | PA-B7, B7c, B9b, B3 | 계획 없음 | – | – | 14:00~14:18 설계·보류·생략 메모 | – | B7c 보류, B9b 생략 근거 기록 |

# B3a의 "정의 확인" 14:18:40은 실행(14:18:34, 14:18:37)보다 뒤에 기록됐습니다. 계획에 D1~D4 정의 열람이 이미 적혀 있었으므로 로그가 늦었던 것으로 보이지만, 부록에는 "기록 시각은 실행 이후, 계획에는 정의 열람 명시"라고 한 줄 붙여 두는 편이 안전합니다.

# ## 3. 기타 기록 (부록 소표)

# | 시각 | 내용 |
# |---|---|
# | 10-21 11:50 | S4 코드 복구(IPython 히스토리에서 `s4_main.py` 등), 4.6 초기 수정본과 선택본의 고정 저널 판정 차이 기록 |
# | 10-21 13:52 | 문구 정정: 진입 우세 "견고"는 부트스트랩·신규 정의·이름 분할 축에 한정, 중국 소속 제외 시 유지 안 됨 |
# | 10-21 14:25, 14:25 | PA-B3a 해석 메모 중복 및 태그 설명 정정 |
# | 10-21 14:26 | 저장소 정리 커밋(S8.5 기록, 데이터 정의 노트북, `transport-atlas-data/` ignore) |

# ## 4. 부록 문구

# > 모든 PA 항목은 실행 전에 계획을 해시·태그로 동결했고, 설계 단계에서 열람한 자료는 항목별로 PA_CHANGELOG.md에 기록했다. 기록의 오류(코드 오류, 백업 완료 오기재, 해석 메모 중복)는 삭제하지 않고 정정 줄로 남겼다. S0~S8의 판정은 어떤 항목에서도 변경하지 않았다.

# 마지막 문장은 changelog 71행 전부 "본 결과 변경: 없음"이라는 점으로 뒷받침됩니다. 이후 추가한 줄(정정 보충, 저장소 정리)도 같은 표기를 썼습니다.

# ## 남은 항목

# 1. B2·A1 grep 결과로 표의 빈칸 채우기
# 2. `01_EDA_9개CSV_분포와특성.ipynb`(1.6MB) 소속 확인
# 3. `oa_raw` 외부 백업(미완료, 매체 연결 후 복사하고 SHA-256 비교)
# 4. B7b(분류기 파일 위치), B8, B6(이번 주 내 결정)

In [ ]:
# 빈칸 두 곳은 채워졌습니다. B2의 해석 메모가 10-21 11:49인 것도 맞았습니다. B2 본 실행은 10-19 18:22이고 해석 메모는 10-21에 쓰였으니, 결과를 보고 이틀 뒤 해석한 순서가 changelog와 맞습니다.

# ## 1. 채워진 행

# | 항목 | 계획 동결 | 실행 | 결과 해석 | 열람·정정 | 비고 |
# |---|---|---|---|---|---|
# | PA-A1 | PA-0 v2 `a98f9697fa67`(10-19 17:24, v1 `4178cadadec9`는 A1만 대체) | run 1 17:38, run 2 17:49, 탐색 확장 18:00 | run 2 결과 10-19(시각 없음), 탐색 확장 해석 10-19(시각 없음) | 17:38 for/else 버그 패치, 17:49 NaN 정규화 패치(모두 사양 불변) | run 1 무효(검정력이 점프와 무관하게 약 0.05), 탐색 확장은 판독 규칙에 미사용 |
# | PA-B2 | 10-19 18:18 `47c93a7c5739` | QUICK 점검 18:22:27(B=20), 본 실행 18:22:36 | 10-21 11:49 | QUICK 출력에 관측 기술 표(토픽 상위 10, 저널 등)가 포함돼 열람했다고 changelog에 기록 | 계획이 PA-A1 결과 열람 뒤의 사후 명세임을 계획 줄에 명시 |

# A1의 run 1, run 2 시각은 파일 타임스탬프(17:38, 17:49)와 태그(`PA-A1-v2-run1`, `run2-code`, `run2-results`)에서 맞춘 것입니다. 17:38과 17:49 줄 자체는 "코드 패치" 기록이라, 실행 시각과 같은 시각에 기록된 것으로 읽었습니다. 정확히 하려면 아래 두 줄을 붙여 주세요.

# ```bash
# sed -n '14p;6,7p' paper1/09_PA/PA_CHANGELOG.md | cut -c1-400
# ```

# - 14번째 줄은 B2 계획과 QUICK 사이의 줄입니다. B2 실행 횟수가 자동 집계로 3회였던 이유를 확인하려는 것입니다. 본 실행은 1회입니다.
# - 6~7번째 줄은 A1 패치 줄 전체입니다.

# ## 2. 부록에 넣을 주의점

# - A1의 결과 줄 두 개(8번, 10번)는 날짜만 있고 시각이 없습니다. 부록 표에는 "10-19(시각 미기록)"으로 쓰세요.
# - 두 항목 모두 "사후 설계"라는 사실이 계획 줄에 이미 적혀 있습니다. 부록 문구의 "설계 단계에서 열람한 자료를 항목별로 기록했다"는 문장과 맞으니 그대로 쓰면 됩니다.

# ## 3. 남은 항목

# `PA_CHANGELOG.md` 기반 A4 표는 위 확인 한 줄만 반영하면 끝입니다. 이어서 할 일은 다음과 같습니다.

# 1. `01_EDA_9개CSV_분포와특성.ipynb`(1.6MB) 소속 확인
# 2. `oa_raw` 외부 백업(미완료, 외부 매체 연결 후 복사하고 SHA-256 비교)
# 3. B7b(분류기 파일 위치), B8, B6(이번 주 내 결정)

# 1번은 아래로 확인하세요(읽기 전용).

# ```bash
# cd ~/Research/Minnesota_Transport
# python3 - <<'EOF'
# import json
# nb = json.load(open("01_EDA_9개CSV_분포와특성.ipynb", encoding="utf-8"))
# for c in nb["cells"][:4]:
#     print("---", c["cell_type"]); print("".join(c["source"])[:300])
# EOF
# git log --oneline -1 -- "01_EDA_9개CSV_분포와특성.ipynb"
# ```

# 앞 몇 셀에 `analysis_data/final`이나 F1~F7이 나오면 paper1 데이터 확정 이전의 EDA입니다. 이 경우 재현성 기록상 커밋하는 편이 낫습니다. 1.6MB라서 출력 셀을 지우고 올릴지는 내용을 본 뒤에 정하면 됩니다.

In [ ]:
# ## 1. A4 표에 반영할 내용

# 14번째 줄은 PA-B2의 동결 절차를 설명하는 기록입니다. 동결(18:18)은 노트북 셀에서 실행했고, `pa_b2.py`는 같은 코드를 나중에 저장했으며, 동결 해시 `47c93a7c5739`와 일치함을 확인했다는 내용입니다. 결과 CSV는 그 시점까지 없었다고 적혀 있습니다. 부록 B2 행에 "동결은 노트북 셀에서 실행, 스크립트는 사후 저장(해시 일치 확인)"을 비고로 넣으세요. 동결 이후에 코드가 바뀌지 않았다는 증거로 쓸 수 있습니다.

# A1 패치 줄 두 개는 앞서 추정한 대로입니다. 17:38은 for/else 수정, 수렴 flag, QUICK 해석 금지이고, 17:49는 `inject()`의 NaN 정규화 수정이며, 둘 다 "사양 불변"입니다.

# ## 2. `01_EDA_9개CSV_분포와특성.ipynb`

# 이 노트북은 paper1의 데이터 파일 F1~F7과 MANIFEST를 점검하는 EDA입니다. 각 파일의 구조, 키, 결측, MANIFEST의 행·열 수와 MD5가 실제 파일과 맞는지 확인하는 내용이고, 본문 수치는 코드로 출력하며 마지막에 `assert`로 검증합니다. 데이터 확정 전후의 재현성 근거가 될 수 있어서 paper1 저장소에 포함하는 편이 맞습니다. `git log`가 비어 있으니 지금까지 한 번도 커밋된 적이 없습니다.

# 참고로 노트북이 `데이터/` 폴더를 대상으로 적고 있습니다. 현재 데이터 경로는 `analysis_data/final`이니, 폴더 이름이 바뀐 뒤의 버전인지 한 번 확인하세요. 실행 결과는 노트북에 저장돼 있으니 출력에 `PASS` 계열 문구가 있는지만 보면 됩니다.

# ```bash
# cd ~/Research/Minnesota_Transport
# grep -c "assert" "01_EDA_9개CSV_분포와특성.ipynb"
# python3 - <<'EOF'
# import json
# nb = json.load(open("01_EDA_9개CSV_분포와특성.ipynb", encoding="utf-8"))
# last = nb["cells"][-1]
# print("".join(last["source"])[:300])
# for o in last.get("outputs", [])[:2]:
#     print(str(o.get("text", o))[:300])
# EOF
# ```

# 마지막 셀이 검증 통과 출력이면 이대로 커밋해도 됩니다. 1.6MB는 부담 없는 크기입니다.

# ```bash
# git add "01_EDA_9개CSV_분포와특성.ipynb"
# git status --short | head
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | 저장소 정리 보충: 01_EDA_9개CSV_분포와특성.ipynb(F1~F7·MANIFEST 정합성·분포 점검, 10-06 작성) 커밋. paper1 외 파일(pemsbay_*, paper2_plan.py, Atlas 계열, choi_lab 등)은 계속 미커밋 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_CHANGELOG.md
# git commit -m "Add EDA notebook for F1-F7 data profiling" && git push
# ```

# ## 3. 남은 항목

# 1. `oa_raw` 외부 백업: 외부 매체를 연결해 복사하고 원본·사본 SHA-256을 비교합니다. 복사 명령이 Permission denied로 실패했으니, 매체를 연결한 뒤 `ls -ld`로 쓰기 권한부터 확인하세요.
# 2. B7b(분류기 산출 파일 위치 확인), B8(미달 4개 도메인 부록 정렬)
# 3. B6(위험군 독립 정답, 수작업 50쌍): 인력 여부를 이번 주 안에 정하고, 없으면 "미수행, 한계로 명시"로 기록합니다.
# 4. A4 부록 표 확정: 위 B2·A1 반영분을 넣으면 마무리됩니다.
# 5. 이후 단계(S9, S10)는 아직 시작 전입니다.

# 필요하시면 B7b의 분류기 산출 파일을 찾는 명령부터 드리겠습니다.

In [ ]:
# ## 1. 확인 결과

# 매핑과 합계가 모두 맞습니다.

# - `topics`는 149개이고 18이 없습니다. `pred_topic`은 전부 `topics` 안에 있고, 18로 예측된 논문은 0편입니다. 다만 ID와 배열 위치가 다르므로(18 이후는 한 칸씩 밀림), 정렬된 L7 토픽 목록의 위치로 변환해야 합니다. 아래 스크립트가 그렇게 처리합니다.
# - 배열 합은 76,984이고, 기간별 합(2005-09 7,182, 2020-25 36,501)이 분류기 보고서의 기간별 크기와 같습니다. 연도 인덱스 가정(2000년 시작)도 맞습니다.
# - `pa_b7a.py`의 `pm()` 계산과 규칙 파일 탐색을 그대로 가져왔습니다. 그래서 관측 델타가 `topic_quadrants.csv`와 같다는 재현 점검이 B7a와 같은 방식으로 통과해야 합니다.

# ## 2. 스크립트 (`pa_b7b.py`)

# 아직 한 번도 실행하지 않은 코드입니다. 모드를 명령행 인자로 받게 해서 파일을 고치지 않고 `freeze → tag → run quick → run` 순서로 쓸 수 있습니다.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# cat > pa_b7b.py <<'PYEOF'
# # -*- coding: utf-8 -*-
# # PA-B7b: 임베딩 결측 논문에 제목 분류기 예측 토픽을 대입했을 때 S4 big 토픽 부호가 유지되는가. 결정적 계산. 미실행 코드.
# # 사용: python3 pa_b7b.py freeze | tag | run quick | run   (tag 뒤에 git push --follow-tags)
# # S0~S8 판정 불변. 모든 산출물 [Post-hoc].
# import sys, json, hashlib, subprocess, warnings
# from pathlib import Path
# import numpy as np, pandas as pd
# warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
# MODE = sys.argv[1] if len(sys.argv) > 1 else "freeze"
# QUICK = "quick" in sys.argv[2:]

# ROOT = Path.home() / "Research/Minnesota_Transport"; P1 = ROOT/"paper1"; PA = P1/"09_PA"; S0, S2, S3, S4 = P1/"00_S0", P1/"02_S2", P1/"03_S3", P1/"04_S4"; DATA = ROOT/"analysis_data/final"
# now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
# sha_b = lambda b: hashlib.sha256(b).hexdigest(); sha_f = lambda p: sha_b(Path(p).read_bytes())
# tf = lambda s: s.astype(str).str.strip().str.lower().isin(["1", "1.0", "true", "t", "yes"])
# git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
# class Halt(Exception): pass
# spec_hash = lambda s: sha_b(json.dumps({k: v for k, v in s.items() if k not in ("sha256", "created", "refs")}, sort_keys=True, ensure_ascii=False).encode())
# V2P, BP = PA/"PA_0_PLAN_v2.json", PA/"PA_B7b_PLAN.json"

# PLAN = dict(
#     version=1, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(PA-B7 중 분류기 대입 (b)만). S4 결과, pa_b7a.py, title_classifier_report.md(교차검증 정확도 0.508, tau 격자별 적용률·정확도), title_pred_missing_topic.csv의 기간별 적용률(0.348/0.374/0.369/0.408/0.450)과 pmax 중앙값은 열람한 상태. PA-B7b 결과 미열람. B7a(뒤집힘 배수), S4 IPW와 보완 관계. S0~S8 판정 불변.",
#     data="B7a와 같은 분석 집합: F1_papers.csv에서 EXCLUDE_PIDS_FROZEN 제외, in_sample 참, 연도 2000~2025, has_topic 참 and topic150 있음(76,984편). 결측 집합 = 나머지 중 분석 저널 안. 대입 후보 = title_pred_missing_topic.csv(S2 분류기, 14,810편)와 pid가 겹치는 결측 논문. 연도·저널은 F1의 값 사용. 기간·대비는 S4 동결 규칙의 periods, main_contrast",
#     imputation="선택된 결측 논문을 (저널, 기간, 예측 토픽) 카운트에 +1. 예측 토픽은 L7_topic_grade의 정렬된 토픽 목록 위치로 변환(배열 축은 149개, 토픽 ID는 0~149에서 18이 빠짐). 대입되지 않은 결측 논문은 분모에서 제외",
#     share="share_full(t, 기간) = (관측 카운트 + 대입 카운트) / (관측 논문 수 + 대입 논문 수), 저널 합산. delta_full = share_full(후기) - share_full(전기). 관측 delta는 S4와 같은 저널 합산 몫의 차",
#     scenarios={"s1": "title_pred_missing_topic.csv의 use_in_sensitivity 플래그(pmax>=0.5 and 토픽 F1>=0.3, S2 동결 외 사전 명시)", "s2": "pmax>=tau_s2(F1 하한 없음)", "s3": "pmax>=tau_s3(적용률 높음)", "s4": "pmax>=tau_s4(정확도 우선)"},
#     tau={"s2": 0.5, "s3": 0.3, "s4": 0.7},
#     scope="S4 big 토픽(topic_quadrants.csv의 big 참, 50개). 각 토픽의 delta_full 부호가 관측 delta 부호와 같은지. 대입 후 big 라벨·4분면 변화는 범위 밖(B7a와 동일)",
#     stats="시나리오별 기간별 결측 대비 대입률(설계 정보), big 토픽 부호 유지 비율, 유지 실패 토픽 목록, |delta_full|/|delta| 중앙값(감쇠 정도)",
#     reading="시나리오별 big 토픽 중 부호 유지 비율 >=0.90 '유지', 0.70~0.90 '부분', <0.70 '민감'. 시나리오를 합치지 않음. B7a와 같은 임계값이며 임의의 동결 사양",
#     reproduce_check="(a) 분석 집합 76,984편, (b) 관측 delta가 topic_quadrants.csv와 1e-9 이내, (c) CSV pid의 99% 이상이 결측 집합(분석 저널 안)에 포함, (d) pred_topic이 모두 L7 토픽 목록 안. 하나라도 어긋나면 중단",
#     caveat="교차검증 정확도는 결측 논문에 대한 정확도가 아님(분포가 다를 수 있음). 예측확률은 보정되지 않았고 pmax만 있어 부드러운 대입 불가. 확신 높은 예측만 대입하므로 F1 높은 토픽이 더 채워지고, 기간별 대입률이 다름(초기 낮음). 오분류가 몫을 흐리게 해 감쇠 쪽으로 작용할 수 있음. 인과 해석 아님",
#     forbidden="결과를 본 뒤 시나리오·tau·판독 임계값 변경(변경 시 v2 계획으로 분리하고 사유 기록)")

# def freeze():
#     if not V2P.exists(): raise Halt("PA_0_PLAN_v2.json이 없습니다")
#     h = spec_hash(PLAN); print("PA-B7b 계획 해시(예정)", h[:12])
#     if BP.exists(): print("이미 동결:", "동일" if json.loads(BP.read_text(encoding="utf-8"))["sha256"] == h else "⛔ 다름"); return
#     refs = {f.name: sha_f(f) for f in [V2P, S4/"topic_quadrants.csv", P1/"pa_b7a.py", DATA/"F1_papers.csv", S2/"title_pred_missing_topic.csv", S2/"title_classifier_report.md"] if f.exists()}
#     BP.write_text(json.dumps(dict(PLAN, created=now(), sha256=h, refs=refs), ensure_ascii=False, indent=1), encoding="utf-8")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f:
#         f.write(f"\n- {now()} | PA-B7b 계획 동결(해시 {h[:12]}) | 사유: 결측 경계를 제목 분류기 대입으로 점검(B7a 보완). 분류기 보고서(정확도·적용률)와 CSV의 기간별 적용률, pa_b7a.py 열람 상태, PA-B7b 결과 미열람. QUICK은 재현 점검과 기간별 대입 편수만 | 본 결과 변경 여부: 없음\n")
#     print("저장: PA_B7b_PLAN.json")

# def tag():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     t = f"PA-B7b-plan-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
#     if git("tag", "-l", t).stdout.strip(): print("이미 있음:", t); return
#     r = git("add", "--", "paper1/09_PA/PA_B7b_PLAN.json", "paper1/09_PA/PA_CHANGELOG.md", "paper1/pa_b7b.py"); print("add", r.returncode, r.stderr.strip())
#     r = git("commit", "-m", "PA-B7b plan freeze (before execution)"); print("commit", r.returncode, (r.stdout + r.stderr)[:200])
#     r = git("tag", "-a", t, "-m", "PA-B7b plan"); print("tag", r.returncode, r.stderr.strip()); print("※ 이어서 git push --follow-tags")

# def find_key(o, key):
#     if isinstance(o, dict):
#         for k, v in o.items():
#             if k == key: return v
#             r = find_key(v, key)
#             if r is not None: return r

# def run():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     sp = json.loads(BP.read_text(encoding="utf-8"))
#     if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(PLAN): raise Halt("동결 파일과 현재 계획이 다릅니다")
#     if not git("tag", "-l", "PA-B7b-plan-*").stdout.strip(): raise Halt("tag 단계를 먼저 실행하세요")
#     spec = None
#     for f in sorted(P1.glob("0[0-4]_*/*PREREG*.json")):
#         j = json.loads(f.read_text(encoding="utf-8")); mc, pr = find_key(j, "main_contrast"), find_key(j, "periods")
#         if mc and pr and all(k in pr for k in mc): spec = (f.name, mc, pr); break
#     if spec is None: raise Halt("main_contrast와 periods를 가진 S4 동결 규칙 파일을 찾지 못함")
#     fname, (c0, c1), PR = spec; P = {k: tuple(v) for k, v in PR.items()}; print(f"[규칙 파일] {fname} | main_contrast {c0}->{c1} | {P[c0]}, {P[c1]}")
#     W = pd.read_csv(DATA/"F1_papers.csv", low_memory=False); W["pid"] = W.pid.astype(str)
#     EXC = set(pd.read_csv(S0/"EXCLUDE_PIDS_FROZEN.csv", dtype=str, encoding="utf-8-sig").pid)
#     W = W[~W.pid.isin(EXC) & tf(W.in_sample) & W.year.between(2000, 2025)].copy(); W["ht"] = tf(W.has_topic) & W.topic150.notna()
#     F, Mi = W[W.ht].copy(), W[~W.ht].copy()
#     if len(F) != 76984: raise Halt(f"분석 집합 재현 실패: {len(F)} (76984여야 함)")
#     F["topic"] = F.topic150.astype(int)
#     topics = np.array(sorted(pd.read_csv(S3/"L7_topic_grade.csv").topic.astype(int))); venues = np.array(sorted(F.venue_short.unique()))
#     Mi = Mi[Mi.venue_short.isin(venues)]; pos = {int(t): i for i, t in enumerate(topics)}
#     tp = pd.read_csv(S2/"title_pred_missing_topic.csv"); tp["pid"] = tp.pid.astype(str)
#     if not tp.pred_topic.isin(topics).all(): raise Halt("pred_topic이 L7 토픽 목록 밖에 있음")
#     M2 = Mi[["pid", "year", "venue_short"]].merge(tp[["pid", "pred_topic", "pmax", "use_in_sensitivity"]], on="pid", how="inner")
#     cov = len(M2)/len(tp)
#     if cov < 0.99: raise Halt(f"CSV pid 중 결측 집합에 포함된 비율 {cov:.3f} < 0.99")
#     M2["tpos"] = M2.pred_topic.map(pos).astype(int); M2["flag"] = tf(M2.use_in_sensitivity)
#     tau = PLAN["tau"]; SC = {"s1": M2.flag, "s2": M2.pmax >= tau["s2"], "s3": M2.pmax >= tau["s3"], "s4": M2.pmax >= tau["s4"]}
#     def base(lo, hi):
#         d = F[F.year.between(lo, hi)]; C = pd.crosstab(d.venue_short, d.topic).reindex(index=venues, columns=topics, fill_value=0).values.astype(float)
#         return C.sum(0), float(C.sum())
#     def imp(mask, lo, hi):
#         d = M2[mask & M2.year.between(lo, hi)]; return np.bincount(d.tpos.values, minlength=len(topics)).astype(float), float(len(d))
#     (a0, n0), (a1, n1) = base(*P[c0]), base(*P[c1])
#     Q = pd.read_csv(S4/"topic_quadrants.csv").set_index("topic"); delta_obs = a1/n1 - a0/n0
#     if np.abs(delta_obs - Q.loc[topics, "delta"].values).max() > 1e-9: raise Halt("관측 delta가 topic_quadrants.csv와 다름")
#     rows = []
#     for nm, mk in SC.items():
#         for c in (c0, c1):
#             b, m = imp(mk, *P[c]); nmiss = int(Mi.year.between(*P[c]).sum()); rows.append(dict(scenario=nm, period=c, n_missing=nmiss, n_with_pred=int(M2.year.between(*P[c]).sum()), n_imputed=int(m), rate_of_missing=m/nmiss if nmiss else np.nan))
#     IM = pd.DataFrame(rows)
#     print(f"[재현 OK] 분석 집합 76,984, 관측 delta, CSV pid 포함 비율 {cov:.4f} | 위치 매핑(L7 정렬 목록) 사용")
#     print("[기간별 대입 편수(설계 정보)]\n", IM.round(3).to_string(index=False))
#     if QUICK: print("\n[QUICK] 재현 점검과 기간별 대입 편수만 출력. 결과 통계는 계산·출력하지 않음."); return
#     D = {}
#     for nm, mk in SC.items():
#         b0, m0 = imp(mk, *P[c0]); b1, m1 = imp(mk, *P[c1]); D[nm] = (a1 + b1)/(n1 + m1) - (a0 + b0)/(n0 + m0)
#     big = Q.loc[topics, "big"].values.astype(bool); sgn = np.sign(delta_obs)
#     R = pd.DataFrame(dict(topic=topics, grade=Q.loc[topics, "grade"].values, quadrant=Q.loc[topics, "quadrant"].values, big=big, delta=delta_obs))
#     for nm in SC: R[f"delta_full_{nm}"] = D[nm]; R[f"kept_{nm}"] = np.sign(D[nm]) == sgn
#     R.to_csv(PA/"PA_B7b_results.csv", index=False, encoding="utf-8-sig"); IM.to_csv(PA/"PA_B7b_imputed_by_period.csv", index=False, encoding="utf-8-sig")
#     B = R[R.big]; out = []
#     for nm in SC:
#         share = float(B[f"kept_{nm}"].mean()); v = "유지" if share >= 0.90 else ("부분" if share >= 0.70 else "민감")
#         out.append(dict(scenario=nm, kept_share=share, n_not_kept=int((~B[f"kept_{nm}"]).sum()), atten_median=float((B[f"delta_full_{nm}"].abs()/B.delta.abs()).median()), 판독=v))
#     O = pd.DataFrame(out); print(f"\n[Post-hoc] PA-B7b big 토픽 {len(B)}개\n[동결 판독]\n", O.round(3).to_string(index=False)); O.to_csv(PA/"PA_B7b_summary.csv", index=False, encoding="utf-8-sig")
#     for nm in SC:
#         X = B[~B[f"kept_{nm}"]]
#         if len(X): print(f"\n[부호 유지 실패 big 토픽 {nm}]\n", X[["topic", "grade", "quadrant", "delta", f"delta_full_{nm}"]].round(4).to_string(index=False))
#     print("※ 분류기 대입 민감도이며 경계가 아님. 시나리오를 합치지 않음. S0~S8 판정 불변.")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as fh:
#         fh.write(f"- {now()} | PA-B7b 실행 | 정의: PA_B7b_PLAN {sp['sha256'][:12]} | 결과: PA_B7b_*.csv | 본 결과 변경 여부: 없음\n")

# try: {"freeze": freeze, "tag": tag, "run": run}[MODE]()
# except Halt as e: print(f"⛔ 중단: {e}")
# PYEOF
# python3 -m py_compile pa_b7b.py && echo "문법 OK"
# ```

# ## 3. 실행 순서

# 먼저 `freeze`로 해시를 만들고, `tag`로 커밋·태그한 뒤 푸시합니다. 푸시까지 끝나야 결과를 보는 단계로 넘어갑니다.

# ```bash
# python3 pa_b7b.py freeze
# python3 pa_b7b.py tag
# git push --follow-tags
# python3 pa_b7b.py run quick
# ```

# `run quick`의 출력에는 재현 점검과 기간별 대입 편수만 나옵니다. 이 출력은 설계 정보라서, B7a 때처럼 열람했다는 사실을 changelog에 남겨야 합니다. 출력을 붙여 주시면 그 줄을 드리고, 그다음 `python3 pa_b7b.py run`으로 본 실행을 합니다.

# ## 4. 설계에서 바꾼 점

# - 시나리오 s3(pmax ≥ 0.3)는 앞서 제안한 "약 89% 적용"이 교차검증 값이었습니다. 결측 논문에서의 적용률은 `run quick` 출력에서 확인합니다. 그 값이 너무 낮아도 규칙은 바꾸지 않고 그대로 보고합니다(동결 규칙).
# - s1은 CSV의 `use_in_sensitivity` 플래그를 그대로 씁니다. 분류기 보고서가 정한 규칙(pmax ≥ 0.5, 토픽 F1 ≥ 0.3)을 재구현하지 않아서, 보고서와 같은 6,051편이 쓰입니다.
# - 추가 진단으로 `atten_median`(대입 후 |델타|가 관측의 몇 배인가)을 넣었습니다. 부호가 유지돼도 델타가 크게 줄어들면 결론의 강도가 약해진다는 뜻이기 때문입니다.
# - 코드는 한 번도 돌려 보지 않았습니다. `py_compile`은 문법만 확인하니, 첫 실행에서 오류가 나면 B7a 때처럼 사양을 바꾸지 않고 코드만 고쳐 changelog에 기록합니다.

# ## 5. 남은 항목

# 1. B7b 동결 → 푸시 → `run quick` → 본 실행
# 2. `oa_raw` 외부 백업(미완료, 매체 연결 후 쓰기 권한 확인, 복사, SHA-256 비교)
# 3. B8(미달 4개 도메인 부록 정렬), B6(수작업 50쌍, 이번 주 내 결정)
# 4. A4 부록 표 확정(B2 동결 절차 비고와 A1 패치 반영)
# 5. S9, S10은 시작 전

In [ ]:
# ## 1. 결과

# 동결 판독은 네 시나리오 모두 "유지"입니다.

# | 시나리오 | 대입 편수(P0509 / P2025) | 부호 유지 | 유지 못 한 토픽 | 감쇠 중앙값 |
# |---|---|---|---|---|
# | s1 (동결 플래그) | 712 / 2,839 | 50/50 | 0 | 0.981 |
# | s2 (F1 하한 없음) | 761 / 2,903 | 50/50 | 0 | 0.979 |
# | s3 (pmax ≥ 0.3) | 1,615 / 5,507 | 50/50 | 0 | 0.947 |
# | s4 (pmax ≥ 0.7) | 256 / 1,218 | 50/50 | 0 | 0.993 |

# 동결 계획 해시는 `4851ecf3acf3`로 일치했고(`status` 끝 문장도 정상), 계획 변경은 없었습니다. 판독이 동결 규칙에서 바뀌지 않았습니다.

# **해석에 주의할 점**
# - **감쇠가 거의 없는 것이 이 방법의 구조적 결과일 수 있습니다.** 분류기는 토픽이 있는 논문으로 학습했기 때문에, 예측이 관측 토픽 분포 쪽으로 끌립니다. 그러면 대입이 몫을 기존 분포에 가깝게 유지하므로 부호가 바뀌기 어렵습니다. 부호 유지는 "결측 논문의 제목에서 읽히는 토픽 구성이 관측과 크게 다르지 않았다"는 정도로만 읽고, 결측이 무작위임을 증명한다고 쓰면 안 됩니다.
# - **s3에서 감쇠가 가장 큽니다**(0.947). 정확도가 낮은 예측까지 넣어도 델타가 5% 안팎 줄 뿐입니다. s4는 대입이 적어 0.993이 되는 것이 당연합니다.
# - **교차검증 정확도(0.51, 적용 부분집합 0.67)는 결측 논문에서 낙관적일 수 있습니다.** 결측 논문은 보고서가 인정한 대로 분포가 다를 수 있습니다.
# - **big 50개의 부호만 봤습니다.** 대입 후 big 라벨 전환과 4분면 변화는 범위 밖입니다(B7a와 같음).
# - **기간별 대입률이 다릅니다**(s1 37.4% 대 45.0%). 정확도 차이와 겹쳐 기간 비교가 완전히 대칭이 아닙니다.

# **B7 전체 그림.** 결측 민감도가 세 가지 방법에서 같은 방향입니다.
# - B7a: 뒤집힘 배수 5 이상이 필요(`k*` 5.16/11.45)
# - S4 IPW: 라벨 유지 0.98
# - B7b: 분류기 대입 4종에서 부호 50/50 유지

# 다만 세 방법 모두 big 토픽의 부호만 다루고, 결측 사유가 토픽과 무관하다는 보장은 못 합니다. 본문 문구는 "결측 가정 민감도 3종에서 big 토픽 부호가 유지됐다(경계 아님)" 정도가 안전합니다.

# ## 2. 커밋·기록

# 결과 파일은 `PA_B7b_results.csv`, `PA_B7b_summary.csv`, `PA_B7b_imputed_by_period.csv` 세 개입니다. 실행 줄은 스크립트가 changelog에 자동으로 썼습니다. 이번 스크립트는 `full_log.txt`를 만들지 않아서, 다른 PA 항목과 달리 로그 파일이 없습니다. 출력이 결정적이고 위 표에 다 들어 있으니, 재실행해서 changelog에 중복 줄을 만들지 말고 표와 changelog 해석 메모로 대신하세요.

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B7b 결과 해석 메모(결정적): 동결 판독 4개 시나리오 모두 '유지'(big 50개 부호 50/50 유지, 유지 못 한 토픽 0). 감쇠 중앙값 s1 0.981, s2 0.979, s3 0.947, s4 0.993. s4는 대입이 적어(13~19%) 유지가 당연하고, s3(대입 85~87%)에서 감쇠가 가장 큼. 분류기가 관측 분포로 학습돼 예측이 관측 분포 쪽으로 끌리므로 부호 유지가 구조적으로 쉬울 수 있어 결측 무작위의 증거가 아님. 교차검증 정확도(0.51)는 결측 논문에서 낙관적일 수 있음. big 부호만 검토, big 라벨 전환·4분면 미검토. B7a·S4 IPW와 같은 방향. 실행 로그 파일 없음(결과 CSV와 위 수치로 대체) | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_B7b_results.csv paper1/09_PA/PA_B7b_summary.csv paper1/09_PA/PA_B7b_imputed_by_period.csv paper1/09_PA/PA_CHANGELOG.md
# git status --short | head -n 8
# git commit -m "PA-B7b results" && git tag -a PA-B7b-results-2026-10-21 -m "PA-B7b outputs, seen" && git push --follow-tags
# ```

# `git status`에서 스테이징된 4개 외 파일이 보이면 커밋 전에 알려 주세요. QUICK 열람 줄은 이미 changelog에 들어 있습니다.

# ## 3. 남은 항목

# 1. B7b 커밋·태그·푸시
# 2. `oa_raw` 외부 백업(미완료, 매체 연결 후 쓰기 권한 확인, 복사, SHA-256 비교)
# 3. B8(미달 4개 도메인 부록 정렬), B6(수작업 50쌍, 이번 주 내 결정)
# 4. A4 부록 표 확정(B2 동결 절차 비고, A1 패치, B7b 행 추가)
# 5. S9, S10은 시작 전

# 다음으로 B8을 진행하시려면, B8이 어떤 계획인지(`PA_0_PLAN`에는 "미달 4도메인 부록 정렬, 사전 v2(새 25편)"로만 적혀 있습니다)와 S8의 정렬 코드 위치를 먼저 확인해야 합니다. 어느 쪽부터 하시겠어요?

In [ ]:
# 푸시까지 정상 완료됐습니다(`5c648c0`, 태그 `PA-B7b-results-2026-10-21`). 스테이징은 예상한 4개뿐이었습니다. `S8.5` 노트북이 다시 modified인 것은 작업 기록이 계속 쓰이고 있어서이니 마지막에 한 번 더 커밋하면 됩니다.

# ## 다음: B8

# `PA_0_PLAN`에는 B8이 "미달 4도메인 부록 정렬, 사전 v2(새 25편)"로만 적혀 있습니다. S8에서 도메인 2/6(safety, transit)만 통과했으니, 미달 4개 도메인을 별도 부록에서 다시 정렬해 본다는 뜻으로 읽힙니다. 그런데 "사전 v2(새 25편)"이 무엇인지는 제가 모릅니다. 새 사전(키워드 사전으로 보임)에 25편의 논문이 쓰이는지, 25개의 용어인지, 도메인 분류용 사전을 새로 만든다는 건지 알 수 없습니다. 추측으로 설계하면 사전 동결 규칙에 어긋나므로, 아래를 먼저 확인하세요(읽기 전용).

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# grep -n -i "B8" 09_PA/PA_CHANGELOG.md | cut -c1-200
# grep -n -i "도메인\|domain" "S7. 지역 집중과 신규 유입, S8. 공급-부담 정렬.txt" | head -n 40
# ls 07_S7S8 2>/dev/null | head -n 30; ls -d 0[6-9]_* | head
# grep -rn -i "사전 v\|dict_v\|domain_dict\|DOMAIN" --include=*.py --include=*.json . 2>/dev/null | grep -v ipynb_checkpoints | head -n 15
# ```

# 찾는 것은 세 가지입니다.
# 1. changelog나 지침서에서 B8이 언급된 줄과 사전 v2의 정의
# 2. S8이 도메인을 어떻게 정의하고 통과 기준을 어떻게 잡았는지(6개 도메인 이름과 미달 4개)
# 3. 사전·도메인 정의 파일의 위치

# 출력을 붙여 주시면 B8 계획 초안을 드리겠습니다. 지침서에 B8의 원문(PA-B8 설명)이 있다면, 그 부분을 같이 붙여 주시면 가장 정확합니다.

# ## 남은 항목

# 1. B8 설계 확인(위 출력)
# 2. `oa_raw` 외부 백업(미완료, 매체 연결 후 쓰기 권한 확인, 복사, SHA-256 비교)
# 3. B6(수작업 50쌍, 이번 주 내 결정)
# 4. A4 부록 표 확정(B2 동결 절차 비고, A1 패치, B7b 행 추가)
# 5. `S8.5` 노트북 최종 커밋
# 6. S9, S10은 시작 전

In [ ]:
# ## 1. 출력 판독

# **`domains` 항목은 v1과 v2가 같습니다.** `diff`가 비어 있고, v2 동결(10-19 16:28)의 사유는 지표 코드 교체와 중국·인도 제외 민감도 추가였습니다. 그래서 `PA-B8`의 "사전 v2"는 S8의 `S8_0_PREREG_v2.json`이 아니라, PA에서 도메인 사전을 새로 고치겠다는 뜻이어야 합니다. 이름이 겹치니 PA에서는 "도메인 사전 v3"처럼 다른 이름을 쓰는 편이 혼동이 없습니다.

# **미달 4개는 기준선 바로 아래입니다.**

# | 도메인 | precision (점추정 [Wilson 95%]) | 기준 0.85까지 | recall (n_pos) |
# |---|---|---|---|
# | electrification_env | 0.80 [0.609, 0.911] | 2편 | 0.789 (19) |
# | freight | 0.80 [0.609, 0.911] | 2편 | 0.500 (10) |
# | active_urban | 0.76 [0.566, 0.885] | 3편 | 1.000 (7) |
# | infrastructure | 0.84 [0.653, 0.936] | 1편(21/25 대 22/25) | 0.692 (13) |

# 네 도메인 모두 구간 상한이 0.85보다 높습니다. 즉 "정확히 미달"이 아니라 n=25로는 기준과 구분되지 않는 상태입니다. 반대로 통과한 safety도 0.88이지만 하한이 0.700이라, 통과 판정도 같은 정도로 불확실합니다. 이 점은 부록에서 양쪽 모두에 같은 문장으로 적어야 공정합니다.

# **새 25편 재판정은 정보가 작습니다.** 참 precision이 0.80인 도메인도 새 25편에서 22편 이상(≥0.85)이 나올 확률이 약 23%입니다(이항분포로 계산: 0.136+0.071+0.024+0.004). 사전을 고쳐서 통과했더라도 우연일 수 있고, 반대로 참 precision이 올라가도 탈락할 수 있습니다. 4개 도메인을 각각 시도하면 적어도 하나는 우연히 통과할 확률이 꽤 높습니다. 그래서 B8-2를 한다면 판독을 "통과/실패"로 쓰지 말고 점추정과 Wilson 구간만 보고하는 편이 안전합니다.

# **정렬 파일은 통과 도메인 중심입니다.** `alignment_gap_map.csv`가 있고 changelog 7번 줄에도 "통과 도메인 ['safety', 'transit']"로 기록돼 있습니다. 미달 4개의 정렬 결과가 이미 있는지는 이 출력으로는 확인되지 않았습니다.

# ## 2. B8을 둘로 나눕니다

# | 항목 | 내용 | 사람 필요 | 권고 |
# |---|---|---|---|
# | B8-1 | 기존 사전으로 미달 4개 도메인에 S8 정렬(SBR, 지표 일관, 편 순위상관)을 그대로 돌려 부록에 보고 | 불필요 | 진행 |
# | B8-2 | 사전 v3을 만들고 새 25편을 이중 코딩해 precision 재판정 | 코더 2명 | 인력이 있을 때만, 구간 보고 중심 |

# B8-1은 사전 변경이 없으므로 S8의 동결 규칙(precision 미달 도메인은 부록)과 충돌하지 않습니다. 정렬 결과 옆에 반드시 "precision 0.76~0.84, 기준 미달"을 병기합니다. 판독 문구도 미리 정해 둘 수 있습니다. 통과 도메인(safety, transit)과 같은 방향이면 "참고용으로 같은 방향", 다르면 "정렬은 도메인 사전 품질에 민감할 수 있음"이며, 어느 쪽이든 S8의 판정(2/6 통과)은 바꾸지 않습니다.

# ## 3. B8-1 설계 전 확인 (읽기 전용)

# ```bash
# cd ~/Research/Minnesota_Transport/paper1/08_S8
# sed -n '4,9p' CHANGELOG_S8.md | cut -c1-500
# head -n 4 alignment_gap_map.csv | cut -c1-250
# python3 - <<'EOF'
# import pandas as pd
# for f in ["alignment_gap_map.csv","country_partial_spearman.csv","indicator_consistency.csv","burden_country_table.csv"]:
#     d = pd.read_csv(f)
#     col = next((c for c in d.columns if c.lower() in ("domain","d")), None)
#     print(f, d.shape, "| domains:", sorted(d[col].unique()) if col else "도메인 열 없음", "| cols:", list(d.columns)[:9])
# EOF
# cd .. && grep -n "def align\|def run_align\|passed\b\|domains_passed" "S7. 지역 집중과 신규 유입, S8. 공급-부담 정렬.txt" | head -n 20
# ```

# 확인할 것은 네 가지입니다.
# 1. 정렬 코드가 통과 도메인만 처리하도록 `domains_passed.json`을 읽는지, 아니면 인자로 도메인을 바꿀 수 있는지
# 2. 결과 파일들이 이미 6개 도메인을 다 계산했는지(그렇다면 B8-1은 새 계산이 아니라 재요약)
# 3. 통과 도메인만 있다면, 코드에서 도메인 목록 한 줄만 바꿔 미달 4개로 돌릴 수 있는지
# 4. 6개 도메인 모두 계산한 파일이 있어도 정렬 보고 시 통과 도메인 필터가 어디에 있는지

# 출력을 붙여 주시면 B8-1 계획 초안(동결 항목, 판독 문구, 재현 점검)을 드립니다.

# ## 4. 남은 항목

# 1. B8-1 설계 확인(위 출력) → 계획 동결 → 실행
# 2. B6, B8-2 인력 여부(이번 주 내 결정, 없으면 "미수행, 한계로 명시")
# 3. `oa_raw` 외부 백업(미완료, 매체 연결 후 쓰기 권한 확인, 복사, SHA-256 비교)
# 4. A4 부록 표 확정(B2 동결 절차 비고, A1 패치, B7b 행 추가)
# 5. `S8.5` 노트북 최종 커밋
# 6. S9, S10은 시작 전

In [ ]:
# ## 1. changelog 정정

# 두 정정 명령을 모두 실행하셔서, 파일 끝에 줄이 세 개 추가됐습니다.

# 1. 보충 기록(자리표시자 문구 포함)
# 2. 정정: "실제 합의 방식: 여기에_직접_기재" ← **이 줄도 자리표시자입니다.** 제가 두 가지 경우를 함께 드려서 생긴 일입니다.
# 3. 정정: "합의 방식은 기록 없음(미기재)" ← 유효한 기록

# 기록은 지우지 않으니, 2번을 무효로 표시하는 한 줄만 덧붙이세요. 아직 커밋 전이라 B8 커밋과 함께 올리면 됩니다.

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | 정리: 직전 3줄 중 두 번째 줄의 '여기에_직접_기재'는 자리표시자(두 가지 명령을 모두 실행해 기록됨), 유효한 기록은 세 번째 줄('합의 방식 미기재') | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# tail -n 4 paper1/09_PA/PA_CHANGELOG.md | cut -c1-120
# ```

# ## 2. 출력 판독: B8-1은 가능합니다

# - **지표 대응이 모두 정의돼 있습니다.** `DOM_IND`에 6개 도메인이 다 있고, `INT`가 네 지표(road_deaths, vehicles, urban_pop, transport_co2)를 국가 표 열로 연결합니다. 미달 4개 도메인의 지표 쌍도 이 범위 안입니다(electrification_env: transport_co2+vehicles, freight: vehicles+transport_co2, active_urban: urban_pop+road_deaths, infrastructure: vehicles+urban_pop). `KeyError`는 나지 않을 것으로 보이지만, `Bt`의 열 이름 확인이 남아 있습니다(아래).
# - **S8 코드는 `.py` 모듈이 아니라 노트북 셀에만 있습니다.** `08_S8`에 `.py`가 없고 `s7_run.py`까지만 있습니다. 그래서 `pa_b8.py`에 필요한 정의(`load_F1X`, `load_mass`, `flag_domains`, `partial_spearman`, `check_spec`, `SPEC` 로딩, `PER`, `YP`, `DOMAINS`, `RX`, `DOM_IND`, `INT`)를 복사해 넣어야 합니다. 복사 오류를 막으려고 **재현 점검을 먼저 합니다**. 같은 코드로 safety·transit을 계산해 기존 `alignment_gap_map.csv` 등과 일치하는지 봅니다. 부트스트랩 난수열도 같은 순서면 구간 `lo`, `hi`까지 일치해야 정상입니다.
# - **덮어쓰기 위험이 두 곳 있습니다.** 출력 CSV 세 개 외에, 마지막의 `S8_meta.json` 쓰기(`S8`의 CSV 전체 해시)와 `CHANGELOG_S8.md` 자동 추가입니다. 사본에서는 둘 다 `09_PA` 쪽으로 바꾸거나 빼야 합니다.
# - 기존 결과의 방향성은 이렇습니다. safety의 지표 일관 비율은 1.000(역량 렌즈)이고, transit은 0.875입니다. HIC 과잉, LIC·LMC 과소입니다. 이 값이 B8-1 해석의 비교 기준이 됩니다.

# ## 3. 사본을 쓰기 전 마지막 확인 (읽기 전용)

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# T="S7. 지역 집중과 신규 유입, S8. 공급-부담 정렬.txt"
# sed -n '2340,2362p;2394,2431p' "$T" | cut -c1-300
# grep -n "^import\|^from\|^S8 *=\|^S1 *=\|^SPEC *=\|^STEPS\|^QUICK\|^def spec_hash\|^def now\|^now *=" "$T" | tail -n 25
# python3 -c "
# import pandas as pd
# b=pd.read_csv('08_S8/burden_country_table.csv',keep_default_na=False,index_col=0)
# print(list(b.columns)); print(b.shape)
# "
# git status --short | head -n 6
# ```

# 확인할 것은 다섯 가지입니다.
# 1. `check_spec()`, `load_F1X()`, `load_mass()`, `partial_spearman()` 본문(2394~2431행)과 `SPEC`을 어디서 읽는지
# 2. `Bt`에 `road_deaths, vehicles, urban_pop, transport_co2`와 `death_rate, veh_per1000, urban_share, co2_tr_pc`가 모두 있는지(앞서 출력은 앞 9열만 보였음)
# 3. import 목록(`rankdata` 등)
# 4. `S1`, `S8` 경로 변수와 `spec_hash`, `now`, `sha_f` 정의
# 5. git 상태에서 changelog만 modified인지

# 2번에서 열이 하나라도 없으면 그 도메인은 계산할 수 없습니다. 그 경우는 규칙을 바꾸지 말고 "자료 부재로 미계산"으로 기록합니다.

# ## 4. 남은 항목

# 1. B8-1: 위 확인 출력 → `pa_b8.py` 작성(재현 점검 포함) → 동결·푸시 → `run quick` → 본 실행
# 2. B6, B8-2 인력 여부(이번 주 내 결정, 없으면 "미수행, 한계로 명시")
# 3. `oa_raw` 외부 백업(미완료, 매체 연결 후 쓰기 권한 확인, 복사, SHA-256 비교)
# 4. A4 부록 표 확정(B2 동결 절차 비고, A1 패치, B7b·B8-1 행 추가)
# 5. `S8.5` 노트북 최종 커밋
# 6. S9, S10은 시작 전

In [ ]:
# ## 1. changelog

# 정리됐습니다. 파일 끝 4줄 중 유효한 기록은 세 번째 줄(미기재)이고, 첫째·둘째 줄은 자리표시자를 포함해 네 번째 줄에서 무효로 표시했습니다. 아직 커밋 전이니 B8 동결 커밋에 함께 올라갑니다.

# ## 2. 확인 결과

# - `burden_country_table`에 13개 열이 모두 있습니다(`road_deaths, vehicles, urban_pop, transport_co2, veh_per1000, urban_share, co2_tr_pc, death_rate, gdp_pc, income` 등). `INT`의 네 지표가 모두 존재해서 미달 4개 도메인을 계산할 수 있습니다. 217개 국가입니다.
# - 도메인 정규식은 코드에 있었지만 복사하지 않습니다. 동결 파일 `S8_0_PREREG_v2.json`에 `domains`, `domain_indicators`, `periods`, `supply`, `sensitivity`, `direction`, `country_level`, `seed`가 있으므로 거기서 직접 읽습니다. 복사 오류가 없고, 해시 `9f59a75d7c63`을 확인할 수 있습니다.
# - `align()`의 계산 본문은 모두 확인했습니다. 마지막 `S8_meta.json` 쓰기와 `CHANGELOG_S8.md` 추가는 사본에서 뺍니다. 출력은 `PA_B8a_*.csv`로만 씁니다.
# - 재현 점검: 같은 코드로 safety·transit을 계산해 기존 CSV와 일치하는지 봅니다. QUICK은 결정적 값(간극 지도, 지표 일관, 편 순위상관 점추정)만, 본 실행은 부트스트랩 구간까지 비교합니다.

# ## 3. 스크립트 (`pa_b8a.py`)

# 한 번도 실행하지 않은 코드입니다. 사용법은 `python3 pa_b8a.py freeze | tag | run quick | run`입니다.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# cat > pa_b8a.py <<'PYEOF'
# # -*- coding: utf-8 -*-
# # PA-B8a: precision 미달 4개 도메인에 S8 정렬(v2 규칙 그대로)을 부록용으로 적용. 사전·지표·임계값 변경 없음. 미실행 코드.
# # 사용: python3 pa_b8a.py freeze | tag | run quick | run   (tag 뒤에 git push --follow-tags)
# # S0~S8 판정 불변. 모든 산출물 [Post-hoc]. 출력은 09_PA/PA_B8a_*.csv만(S8 폴더 파일은 수정하지 않음).
# import sys, json, re, hashlib, subprocess, warnings
# from pathlib import Path
# import numpy as np, pandas as pd
# from scipy.stats import rankdata
# warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)
# MODE = sys.argv[1] if len(sys.argv) > 1 else "freeze"; QUICK = "quick" in sys.argv[2:]

# ROOT = Path.home()/"Research/Minnesota_Transport"; P1 = ROOT/"paper1"; PA = P1/"09_PA"
# DATA, S0, S1, S2, S8 = ROOT/"analysis_data/final", P1/"00_S0", P1/"01_S1", P1/"02_S2", P1/"08_S8"
# now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
# sha_b = lambda b: hashlib.sha256(b).hexdigest(); sha_f = lambda p: sha_b(Path(p).read_bytes())
# git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
# class Halt(Exception): pass
# spec_hash = lambda s: sha_b(json.dumps({k: v for k, v in s.items() if k not in ("sha256", "created", "refs")}, sort_keys=True, ensure_ascii=False).encode())
# V2P, BP = PA/"PA_0_PLAN_v2.json", PA/"PA_B8a_PLAN.json"
# INT = dict(road_deaths="death_rate", vehicles="veh_per1000", urban_pop="urban_share", transport_co2="co2_tr_pc")   # S8 코드(INT)와 동일

# PLAN = dict(
#     version=1, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(PA-B8 중 B8-1만: 기존 사전으로 미달 4개 도메인에 정렬을 적용). S8 결과(통과 safety·transit의 정렬·지표 일관·편 순위상관), domain_precision_recall.csv(미달 4개 precision 0.76~0.84), align() 소스 열람한 상태. 미달 4개 도메인의 정렬 결과는 미계산·미열람. 사전 v3 재작성과 새 25편 이중 코딩(B8-2)은 별도(인력 필요). S0~S8 판정 불변.",
#     scope="domains_passed.json의 failed 4개(electrification_env, freight, active_urban, infrastructure). S8_0_PREREG_v2.json(해시 9f59a75d7c63)의 domains, domain_indicators, periods, supply, sensitivity, direction, country_level, seed를 그대로 사용. S8 align()의 계산(렌즈 2종, 제외 민감도 3종, 지표 일관, 편 순위상관)을 복제하되 도메인 목록만 바꿈",
#     outputs="09_PA/PA_B8a_alignment_gap_map.csv, PA_B8a_indicator_consistency.csv, PA_B8a_sensitivity_exclusion.csv, PA_B8a_country_partial_spearman.csv, PA_B8a_precision_note.csv. S8 폴더의 파일, S8_meta.json, CHANGELOG_S8.md는 수정하지 않음",
#     reproduce_check="같은 복제 코드로 safety·transit을 계산해 S8 산출물(alignment_gap_map, indicator_consistency, country_partial_spearman)과 비교. 간극 지도·지표 일관·편 순위상관 점추정이 1e-9 이내로 일치하지 않으면 중단. 부트스트랩 구간(lo, hi)은 본 실행에서 비교하되 불일치 시 경고만(난수 순서 차이 가능)",
#     reading="미달 4개 도메인은 사전 precision이 0.85 미만이라 본문 정렬 분석에서 제외한 도메인. 통과/실패 판정을 하지 않음. 보고: (1) 역량 렌즈 지표 일관 비율(도메인별, 참고로 0.75와 비교), (2) 마지막 기간 역량 렌즈에서 지표별 'HIC SBR>1 이면서 LIC·LMC SBR<1' 여부, (3) 편 순위상관(제외 없음)의 부호와 구간이 0을 배제하는지, (4) 제외 민감도 같은 쪽 비율. 통과 도메인(safety·transit)과 같은 방향인지 기술만 하고 본문 판정을 바꾸지 않음",
#     caveat="미달 도메인은 사전 precision 0.76~0.84(n=25, Wilson 하한 0.57~0.65)라 공급 질량에 오분류가 섞여 있을 수 있음. 통과 도메인과 같은 선상의 증거가 아님. 모든 표에 precision을 병기. 국가 단위 상관은 생태학적 오류 가능, 인과 아님. 부트스트랩은 국가 단위",
#     forbidden="결과를 본 뒤 사전·지표 대응·임계값·판독 문구 변경, 미달 도메인을 본문 통과 도메인으로 승격(변경 시 v2 계획으로 분리하고 사유 기록)")

# def load_spec():
#     SPEC = json.loads((S8/"S8_0_PREREG_v2.json").read_text(encoding="utf-8"))
#     if str(SPEC.get("sha256", ""))[:12] != "9f59a75d7c63": raise Halt("S8_0_PREREG_v2.json 해시가 기록(9f59a75d7c63)과 다름")
#     dp = json.loads((S8/"domains_passed.json").read_text(encoding="utf-8"))
#     if dp["passed"] != ["safety", "transit"] or sorted(dp["failed"]) != sorted(["electrification_env", "freight", "active_urban", "infrastructure"]): raise Halt("domains_passed.json이 예상과 다름")
#     return SPEC, dp

# def freeze():
#     if not V2P.exists(): raise Halt("PA_0_PLAN_v2.json이 없습니다")
#     h = spec_hash(PLAN); print("PA-B8a 계획 해시(예정)", h[:12])
#     if BP.exists(): print("이미 동결:", "동일" if json.loads(BP.read_text(encoding="utf-8"))["sha256"] == h else "⛔ 다름"); return
#     refs = {f.name: sha_f(f) for f in [V2P, S8/"S8_0_PREREG_v2.json", S8/"domains_passed.json", S8/"domain_precision_recall.csv", S8/"alignment_gap_map.csv", S8/"indicator_consistency.csv", S8/"country_partial_spearman.csv", S8/"burden_country_table.csv", P1/"pa_b8a.py"] if f.exists()}
#     BP.write_text(json.dumps(dict(PLAN, created=now(), sha256=h, refs=refs), ensure_ascii=False, indent=1), encoding="utf-8")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as f:
#         f.write(f"\n- {now()} | PA-B8a 계획 동결(해시 {h[:12]}) | 사유: precision 미달 4개 도메인을 기존 사전으로 부록 정렬(B8-1). 통과 도메인 결과, 미달 precision, align() 소스 열람 상태, 미달 도메인 정렬 결과 미열람. QUICK은 재현 점검과 도메인별 표본·커버리지만. B8-2(사전 v3, 새 25편)는 별도 | 본 결과 변경 여부: 없음\n")
#     print("저장: PA_B8a_PLAN.json")

# def tag():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     t = f"PA-B8a-plan-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
#     if git("tag", "-l", t).stdout.strip(): print("이미 있음:", t); return
#     r = git("add", "--", "paper1/09_PA/PA_B8a_PLAN.json", "paper1/09_PA/PA_CHANGELOG.md", "paper1/pa_b8a.py"); print("add", r.returncode, r.stderr.strip())
#     r = git("commit", "-m", "PA-B8a plan freeze (before execution)"); print("commit", r.returncode, (r.stdout + r.stderr)[:200])
#     r = git("tag", "-a", t, "-m", "PA-B8a plan"); print("tag", r.returncode, r.stderr.strip()); print("※ 이어서 git push --follow-tags")

# # ---------------- S8 정의 복제 ----------------
# def partial_spearman(x, y, z):
#     rx_, ry_, rz = rankdata(x), rankdata(y), rankdata(z); A = np.c_[np.ones(len(z)), rz]
#     res = lambda v: v - A @ np.linalg.lstsq(A, v, rcond=None)[0]; a, b = res(rx_), res(ry_); return float(np.corrcoef(a, b)[0, 1])

# def load_all(SPEC):
#     PER = {k: tuple(v) for k, v in SPEC["periods"].items()}; YP = {y: k for k, (a, b) in PER.items() for y in range(a, b + 1)}
#     RX = {k: re.compile(v, re.I) for k, v in SPEC["domains"].items()}
#     F1 = pd.read_csv(DATA/"F1_papers.csv", low_memory=False); F1["pid"] = F1.pid.astype(str)
#     exf = S0/"EXCLUDE_PIDS_FROZEN.csv"; log = pd.read_csv(S0/"FREEZE_LOG.csv"); last = log[log.file == exf.name].iloc[-1]
#     if sha_f(exf) != last.sha256: raise Halt("EXCLUDE_PIDS_FROZEN.csv 해시 불일치")
#     F = F1[~F1.pid.isin(set(pd.read_csv(exf, dtype=str, encoding="utf-8-sig").pid))].copy(); F = F[F.year >= 2007].copy(); F = F[F.title.notna()]
#     flall = pd.DataFrame({d: F.title.fillna("").str.contains(rx).values for d, rx in RX.items()}, index=F.index); flall["pid"] = F.pid.values
#     pm = S2/"country_mass_long.parquet"; Mall = pd.read_parquet(pm) if pm.exists() else pd.read_csv(S2/"country_mass_long.csv.gz", dtype={"pid": str}); Mall["pid"] = Mall.pid.astype(str)
#     sp_ = SPEC["supply"]; Mall = Mall[(Mall.scheme == sp_["scheme"]) & (Mall.variant == sp_["variant"]) & (Mall.year >= 2007)]; Mall = Mall[~Mall.country.isin(["UNK", "ZZ_former"])]
#     Bt = pd.read_csv(S8/"burden_country_table.csv", keep_default_na=False, index_col=0); inc = Bt["income"].copy(); Bt = Bt.drop(columns="income").apply(pd.to_numeric, errors="coerce"); Bt["income"] = inc
#     TP0 = pd.read_csv(S1/"title_places.csv", dtype={"pid": str}, keep_default_na=False); TP0 = TP0[TP0.target_countries != ""].copy(); TP0["tc"] = TP0.target_countries.str.split("|"); TP0 = TP0.explode("tc")
#     TP0["w"] = 1.0/TP0.groupby("pid").pid.transform("size"); TP0 = TP0.rename(columns={"tc": "country"})[["pid", "country", "w", "year"]]
#     TP0["per"] = TP0.year.astype(int).map(YP); TP0 = TP0.dropna(subset=["per"])
#     return dict(PER=PER, YP=YP, flall=flall, Mall=Mall, Bt=Bt, TP0=TP0)

# def compute(SPEC, L, dom_ok, Bn):
#     PER, YP, Bt, Mall = L["PER"], L["YP"], L["Bt"], L["Mall"]; DOM_IND = SPEC["domain_indicators"]
#     INC = Bt["income"].to_dict(); G4 = ["HIC", "UMC", "LMC", "LIC"]; EXC = [("none", [])] + [("excl_" + "_".join(s), s) for s in SPEC["sensitivity"]["exclude_country_sets"]]
#     fl = L["flall"][["pid"] + dom_ok]; M = Mall.copy(); M["per"] = M.year.map(YP); M = M.merge(fl, on="pid", how="inner"); TP = L["TP0"].merge(fl, on="pid", how="inner")
#     rows = []
#     for lens, D0 in [("capability", M), ("target", TP)]:
#         for exn, ex in EXC:
#             D = D0[D0.country.isin(Bt.index) & ~D0.country.isin(ex)]
#             for d in dom_ok:
#                 for ind in sorted(set(DOM_IND[d])):
#                     sup_c = Bt.index[Bt[ind].notna() & ~Bt.index.isin(ex) & Bt["income"].isin(G4)]
#                     Bi = Bt.loc[sup_c, ind].groupby(Bt.loc[sup_c, "income"]).sum().reindex(G4).fillna(0); bsh = Bi/Bi.sum()
#                     for per in PER:
#                         x = D[(D.per == per) & D[d] & D.country.isin(sup_c)]; Sg = x.groupby(x.country.map(INC)).w.sum().reindex(G4).fillna(0)
#                         if Sg.sum() <= 0: continue
#                         for g in G4: rows.append(dict(lens=lens, excl=exn, domain=d, indicator=ind, period=per, group=g, supply_share=Sg[g]/Sg.sum(), burden_share=bsh[g], sbr=(Sg[g]/Sg.sum())/bsh[g] if bsh[g] > 0 else np.nan, supply_mass=Sg.sum()))
#     R = pd.DataFrame(rows); Rm = R[R.excl == "none"]
#     cons = []
#     for (lens, d, per, g), x in Rm.groupby(["lens", "domain", "period", "group"]):
#         x = x[x.indicator.isin(DOM_IND[d])].dropna(subset=["sbr"])
#         if len(x) == 2: cons.append(dict(lens=lens, domain=d, period=per, group=g, same_side=bool((x.sbr > 1).nunique() == 1)))
#     C = pd.DataFrame(cons)
#     key = ["lens", "domain", "indicator", "period", "group"]; mm = Rm[key + ["sbr"]].rename(columns={"sbr": "sbr_main"}); sens = []
#     for exn, _ in EXC[1:]:
#         x = R[R.excl == exn][key + ["sbr"]].merge(mm, on=key).dropna(); x["same"] = (x.sbr > 1) == (x.sbr_main > 1)
#         for (lens, d), g in x.groupby(["lens", "domain"]): sens.append(dict(excl=exn, lens=lens, domain=d, n=len(g), same_side_as_main=g.same.mean()))
#         sens.append(dict(excl=exn, lens="ALL", domain="ALL", n=len(x), same_side_as_main=x.same.mean()))
#     SN = pd.DataFrame(sens)
#     cl = SPEC["country_level"]; rng = np.random.default_rng(SPEC["seed"]); pc = []
#     for exn, ex in EXC:
#         tot = Mall[~Mall.country.isin(ex)].groupby("country").w.sum(); tot = tot[tot >= cl["min_country_mass"]]
#         for d in dom_ok:
#             dm = M[M[d] & ~M.country.isin(ex)].groupby("country").w.sum().reindex(tot.index).fillna(0); sup = dm/tot
#             for ind in DOM_IND[d]:
#                 df = pd.DataFrame({"s": sup, "b": Bt[INT[ind]].reindex(tot.index), "z": Bt["gdp_pc"].reindex(tot.index)}).dropna()
#                 if len(df) < 20: pc.append(dict(excl=exn, domain=d, indicator=ind, n=len(df))); continue
#                 r = partial_spearman(df.s.values, df.b.values, df.z.values); bs = []
#                 for _ in range(Bn):
#                     i = rng.integers(0, len(df), len(df)); bs.append(partial_spearman(df.s.values[i], df.b.values[i], df.z.values[i]))
#                 pc.append(dict(excl=exn, domain=d, indicator=ind, n=len(df), raw_rho=float(np.corrcoef(rankdata(df.s), rankdata(df.b))[0, 1]), partial_rho=r, lo=np.nanpercentile(bs, 2.5) if Bn else np.nan, hi=np.nanpercentile(bs, 97.5) if Bn else np.nan))
#     return R, C, SN, pd.DataFrame(pc)

# def same(a, b, keys, cols, tol=1e-9):
#     m = a.merge(b, on=keys, suffixes=("_a", "_b"), how="outer", indicator=True)
#     if (m._merge != "both").any(): return False, f"행 불일치 {int((m._merge != 'both').sum())}"
#     bad = [c for c in cols if c in a.columns and not np.allclose(m[c + "_a"].astype(float), m[c + "_b"].astype(float), atol=tol, equal_nan=True)]
#     return (not bad), (f"불일치 열 {bad}" if bad else "일치")

# def run():
#     if not BP.exists(): raise Halt("먼저 freeze")
#     sp = json.loads(BP.read_text(encoding="utf-8"))
#     if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(PLAN): raise Halt("동결 파일과 현재 계획이 다릅니다")
#     if not git("tag", "-l", "PA-B8a-plan-*").stdout.strip(): raise Halt("tag 단계를 먼저 실행하세요")
#     SPEC, dp = load_spec(); L = load_all(SPEC); fails = dp["failed"]; Bn = SPEC["country_level"]["bootstrap_B"]
#     # --- 재현 점검: 통과 도메인을 같은 코드로 계산해 S8 산출물과 비교 ---
#     Bn_rep = 2 if QUICK else Bn
#     R0, C0, SN0, PC0 = compute(SPEC, L, dp["passed"], Bn_rep)
#     ok1, m1 = same(R0, pd.read_csv(S8/"alignment_gap_map.csv"), ["lens", "excl", "domain", "indicator", "period", "group"], ["supply_share", "burden_share", "sbr", "supply_mass"])
#     ok2, m2 = same(C0.assign(same_side=C0.same_side.astype(int)), pd.read_csv(S8/"indicator_consistency.csv").assign(same_side=lambda d: d.same_side.astype(int)), ["lens", "domain", "period", "group"], ["same_side"])
#     pcs = pd.read_csv(S8/"country_partial_spearman.csv"); ok3, m3 = same(PC0, pcs, ["excl", "domain", "indicator"], ["n", "raw_rho", "partial_rho"])
#     print(f"[재현 점검] 간극 지도 {m1} | 지표 일관 {m2} | 편 순위상관 점추정 {m3}")
#     if not (ok1 and ok2 and ok3): raise Halt("재현 점검 실패: 복제 코드가 S8 산출물과 다름")
#     if not QUICK:
#         ok4, m4 = same(PC0, pcs, ["excl", "domain", "indicator"], ["lo", "hi"]); print(f"[재현 점검] 부트스트랩 구간 {m4}" + ("" if ok4 else "  ※ 경고만(난수 순서 차이 가능)"))
#     pr = pd.read_csv(S8/"domain_precision_recall.csv"); pr = pr[pr.domain.isin(fails)][["domain", "n_prec", "precision", "p_lo", "p_hi", "recall"]]
#     if QUICK:
#         fl = L["flall"]; print("\n[QUICK 설계 정보: 미달 도메인 규칙 양성 논문 수, 부담 지표 가용 국가 수]")
#         print({d: int(fl[d].sum()) for d in fails}); print({ind: int(L["Bt"][ind].notna().sum()) for ind in INT})
#         print("미달 도메인 precision:\n", pr.round(3).to_string(index=False)); print("\n[QUICK] 재현 점검과 설계 정보만 출력. 미달 도메인 정렬 결과는 계산·출력하지 않음."); return
#     R, C, SN, PC = compute(SPEC, L, fails, Bn)
#     R.to_csv(PA/"PA_B8a_alignment_gap_map.csv", index=False, encoding="utf-8-sig"); C.to_csv(PA/"PA_B8a_indicator_consistency.csv", index=False, encoding="utf-8-sig")
#     SN.to_csv(PA/"PA_B8a_sensitivity_exclusion.csv", index=False, encoding="utf-8-sig"); PC.to_csv(PA/"PA_B8a_country_partial_spearman.csv", index=False, encoding="utf-8-sig"); pr.to_csv(PA/"PA_B8a_precision_note.csv", index=False, encoding="utf-8-sig")
#     print("\n[Post-hoc] PA-B8a 미달 4개 도메인(사전 precision 기준 미달, 참고용)\n[도메인 precision]\n", pr.round(3).to_string(index=False))
#     last = sorted(L["PER"])[-1]; Rm = R[R.excl == "none"]
#     print(f"\n[정렬 간극 지도: 역량 렌즈, {last}, 제외 없음]\n", Rm[(Rm.lens == "capability") & (Rm.period == last)].pivot_table(index=["domain", "indicator"], columns="group", values="sbr").round(2).to_string())
#     pv = Rm[(Rm.lens == "capability") & (Rm.period == last)].pivot_table(index=["domain", "indicator"], columns="group", values="sbr")
#     if {"HIC", "LIC", "LMC"} <= set(pv.columns): print("\n[HIC 과잉·LIC/LMC 과소 패턴(SBR HIC>1, LIC<1, LMC<1)]\n", ((pv.HIC > 1) & (pv.LIC < 1) & (pv.LMC < 1)).to_string())
#     cs = C.groupby(["lens", "domain"]).same_side.mean().unstack(0).round(3); cs["ref_>=0.75(capability)"] = cs["capability"] >= SPEC["direction"]["min_share"]; print("\n[도메인별 지표 일관 비율(참고, 판정 아님)]\n", cs.to_string())
#     print("\n[제외 민감도: 주 결과와 같은 쪽인 셀 비율]\n", SN.round(3).to_string(index=False))
#     print("\n[국가 수준 편 순위상관(전 기간 풀링, 제외 없음)]\n", PC[PC.excl == "none"].round(3).to_string(index=False))
#     print("※ 사전 precision 미달 도메인의 참고용 결과. 통과 도메인과 같은 선상의 증거가 아니며 본문 판정(2/6 통과)을 바꾸지 않음. 국가 단위 상관은 생태학적 오류 가능, 인과 아님.")
#     with (PA/"PA_CHANGELOG.md").open("a", encoding="utf-8") as fh:
#         fh.write(f"- {now()} | PA-B8a 실행 | 정의: PA_B8a_PLAN {sp['sha256'][:12]} | 결과: PA_B8a_*.csv | 본 결과 변경 여부: 없음\n")

# try: {"freeze": freeze, "tag": tag, "run": run}[MODE]()
# except Halt as e: print(f"⛔ 중단: {e}")
# PYEOF
# python3 -m py_compile pa_b8a.py && echo "문법 OK"
# ```

# ## 4. 실행 순서

# `freeze` 전에 터미널 화면이 섞이지 않았는지, 아래 한 줄로 확인하세요. 이전에 `cat > ... <<'PYEOF'` 붙여넣기가 한 번 섞여 보였던 적이 있습니다.

# ```bash
# grep -c "py_compile\|여기에" pa_b8a.py     # 0이어야 정상
# python3 pa_b8a.py freeze
# python3 pa_b8a.py tag
# git push --follow-tags
# python3 pa_b8a.py run quick
# ```

# `run quick`은 재현 점검 세 가지가 "일치"로 나오는지가 핵심입니다. 그 출력에는 미달 도메인의 정렬 결과가 없고, 규칙 양성 논문 수와 지표별 가용 국가 수만 나옵니다. 이 설계 정보는 열람 기록 줄을 남겨야 합니다. 재현 점검이 실패하면 B8-1을 진행하지 말고 출력을 붙여 주세요. 복제 코드가 S8과 다르다는 뜻입니다.

# ## 5. 남은 항목

# 1. B8-1: 위 순서로 동결 → 푸시 → `run quick` → 본 실행
# 2. B6, B8-2 인력 여부(이번 주 내 결정, 없으면 "미수행, 한계로 명시")
# 3. `oa_raw` 외부 백업(미완료, 매체 연결 후 쓰기 권한 확인, 복사, SHA-256 비교)
# 4. A4 부록 표 확정(B2 동결 절차 비고, A1 패치, B7b·B8a 행 추가)
# 5. `S8.5` 노트북 최종 커밋
# 6. S9, S10은 시작 전

In [ ]:
# # PA-B8a QUICK 점검 판독

# **재현 점검 세 가지가 모두 "일치"로 나왔습니다.** 복제한 정렬 코드가 통과 도메인(safety, transit)에서 S8 산출물과 같은 값을 냈다는 뜻입니다. 이게 이 단계의 핵심이었고, 본 실행으로 넘어가도 됩니다.

# ## 1. 확인된 것

# - **동결 해시** `fa7e2a359061`, 커밋 `148ed53`, 태그 `PA-B8a-plan-2026-10-21`이 푸시됐습니다.
# - **파일 상태 점검(`grep -c` = 0)은 통과했고 `py_compile`도 "문법 OK"였지만, 붙여넣기 화면은 한 번 섞였습니다.** 출력에 보이는 `PLAN`의 `status` 줄이 중간에서 끊겨 `python3 -m py_compile pa_b8a.py && echo "문법 OK"ODE]()8a_PLAN {sp['sha256'][:12]} ...`로 이어집니다. 이것이 **파일에 실제로 저장된 내용인지, 터미널 표시만 섞인 것인지** 지금은 알 수 없습니다.
#   - 다만 같은 파일이 이후 `py_compile`을 통과했고 `freeze`, `tag`, `run quick`이 모두 정상 동작했습니다. `status` 문자열이 망가졌다면 `try:` 블록과 `run()`, `tag()` 정의가 함께 사라져 실행되지 않았을 것입니다. 그래서 **파일은 정상이고 표시 문제일 가능성이 높다**고 봅니다. 확신은 아래 점검으로 하세요.
# - **QUICK 설계 정보(규칙 양성 논문 수):** electrification_env 7,870편, freight 5,516편, active_urban 7,504편, infrastructure 4,432편입니다. **부담 지표 가용 국가 수:** road_deaths 186개, vehicles 154개, urban_pop 217개, transport_co2 194개입니다.
# - 미달 4개 도메인의 precision은 0.76~0.84(Wilson 하한 0.57~0.65)로, 앞서 정리한 값과 같습니다.

# ## 2. 본 실행 전에 확인할 것 (읽기 전용)

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# grep -n "status=" pa_b8a.py | cut -c1-400
# grep -c "ODE\]()" pa_b8a.py        # 0이어야 정상
# tail -n 3 pa_b8a.py | cut -c1-120
# python3 - <<'EOF'
# import json
# p = json.load(open("09_PA/PA_B8a_PLAN.json", encoding="utf-8"))
# print(p["status"][-160:]); print("py_compile" in json.dumps(p, ensure_ascii=False))
# EOF
# ```

# - `grep -n "status="`의 출력이 완결된 문장으로 끝나고(`...미계산·미열람. 사전 v3 재작성과 ... S0~S8 판정 불변.`), `ODE]()`가 0건이며, 마지막 3줄이 `try: ...`와 `except Halt ...`로 끝나야 정상입니다.
# - 계획 JSON의 `status` 끝에 `py_compile` 문자열이 **없어야** 합니다(마지막 줄이 `False`). 파일 해시 `fa7e2a359061`은 `PLAN` 딕셔너리에서 계산되므로, 파일에 섞인 문자열이 있었다면 계획서에 그대로 동결됐을 것입니다.
# - **섞인 내용이 있다면 중단하세요.** 이미 푸시된 동결 해시를 바꾸면 안 되고, 새 버전(`v2`)으로 분리해 사유를 남겨야 합니다. 그 출력을 붙여 주시면 정리하겠습니다.

# ## 3. 열람 기록 (본 실행 전)

# QUICK 출력에서 본 수치는 결과 통계가 아니라 설계 정보입니다. 그래도 열람 사실이므로 한 줄 남깁니다. 점검이 정상이면 같은 명령 묶음에 이어서 쓰세요.

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B8a QUICK 열람: 재현 점검 3종(간극 지도, 지표 일관, 편 순위상관 점추정) 일치. 설계 정보 열람: 미달 4개 도메인 규칙 양성 논문 수(7,870/5,516/7,504/4,432), 지표별 가용 국가 수(road_deaths 186, vehicles 154, urban_pop 217, transport_co2 194), 미달 도메인 precision. 미달 도메인 정렬 결과는 미열람. 붙여넣기 중 터미널 화면이 섞여 보인 줄이 있었으나 py_compile과 freeze/tag/quick이 정상 동작해 파일 정상으로 판단(위 점검 명령으로 재확인) | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_CHANGELOG.md && git commit -m "PA-B8a: QUICK check note" && git push
# ```

# ## 4. 본 실행

# `pa_b8a.py`는 `run`(quick 없음) 모드에서 로그 파일을 만들지 않으니, 로그를 `tee`로 저장하세요. 부트스트랩이 도메인 4개 × 지표 2개 × 제외 3종이라 시간이 걸릴 수 있습니다. 정확한 시간은 모릅니다.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# nohup python3 -u pa_b8a.py run > 09_PA/PA_B8a_full_log.txt 2>&1 &
# sleep 5; tail -f 09_PA/PA_B8a_full_log.txt
# ```

# `MODE`를 파일에서 바꾸지 않고 인자로 받으므로 `sed`는 필요 없습니다. 본 실행 전 태그(`fullrun`)를 따로 남기고 싶다면 이 명령 앞에 `git tag -a PA-B8a-fullrun-2026-10-21 -m "pre-run" && git push --follow-tags`를 넣으세요. 코드가 바뀌지 않았으므로 필수는 아닙니다.

# ## 5. 결과를 읽을 때 지킬 것 (동결 사양)

# - 미달 4개 도메인은 **통과/실패 판정을 하지 않습니다.** 보고 항목은 계획서에 고정돼 있습니다: 지표 일관 비율(0.75와 비교는 참고만), 마지막 기간 역량 렌즈에서 "HIC 과잉·LIC/LMC 과소" 패턴 여부, 편 순위상관(제외 없음)의 부호와 구간, 제외 민감도 일치율.
# - 모든 표에 precision을 병기합니다. 통과 도메인과 같은 선상의 증거가 아니며, 본문 판정(2/6 통과)은 바꾸지 않습니다.
# - 앞서 B9a에서 확인했듯 LIC는 질량 하한 때문에 국가 수준 상관에서 비어 있을 수 있습니다. 미달 도메인의 국가 수준 상관도 같은 제약을 받을 가능성이 큽니다(미확인).

# 로그 전체를 붙여 주세요.

# ## 남은 항목

# 1. B8-1 본 실행과 해석
# 2. B6, B8-2 인력 여부(이번 주 내 결정, 없으면 "미수행, 한계로 명시")
# 3. `oa_raw` 외부 백업(미완료, 매체 연결 후 쓰기 권한 확인, 복사, SHA-256 비교)
# 4. A4 부록 표 확정(B2 동결 절차 비고, A1 패치, B7b·B8a 행 추가)
# 5. `S8.5` 노트북 최종 커밋
# 6. S9, S10은 시작 전

In [ ]:
# # PA-B8a 판독

# 본 실행이 정상 종료됐고(로그 끝에 경고 문구까지 출력), 재현 점검은 부트스트랩 구간까지 일치했습니다. 미달 4개 도메인은 통과/실패 판정을 하지 않고, 동결된 보고 항목 네 가지만 읽습니다. 모든 결과는 precision 0.76~0.84(Wilson 하한 0.57~0.65)인 사전 기준의 참고용입니다.

# ## 1. 동결 보고 항목

# | 항목 | 결과 |
# |---|---|
# | (1) 지표 일관 비율(역량 렌즈) | 0.938~1.000, 4개 도메인 모두 참고 기준 0.75 이상. 목표 렌즈는 freight가 0.750(경계) |
# | (2) HIC 과잉·LIC/LMC 과소(P2025, 역량) | 8쌍 중 7쌍 True. 예외는 electrification_env×transport_co2(HIC 0.87) |
# | (3) 편 순위상관(제외 없음) | 8쌍 모두 구간이 0을 포함. 점추정 부호도 혼재(−0.17~+0.19) |
# | (4) 제외 민감도 | 역량 렌즈 0.906~1.000(CN·IN 제외 모두). 전체 셀 일치율 0.871~0.957 |

# ## 2. 읽기

# 1. **소득군 간 공급 편중은 통과 도메인과 같은 방향입니다.** LIC는 SBR 0.01~0.28로 거의 비어 있고 LMC도 0.08~0.47입니다. HIC는 active_urban에서 7.34(road_deaths)까지 올라갑니다. 다만 이 패턴은 소득군별 논문 생산 규모 차이가 그대로 반영되기 쉬워서, "부담 대비 공급 간극"의 독립 증거로 읽을 수는 없습니다(B9a에서 확인한 소득 겹침 문제와 같은 해석 위험).
# 2. **지표 일관이 높은 것도 구조적일 수 있습니다.** 소득군 4개 중 HIC와 LIC·LMC의 방향이 크게 갈려 있으면 어떤 부담 지표를 써도 같은 쪽이 나옵니다. 일관성이 정렬의 증거라기보다 소득 효과의 반영일 가능성이 있습니다(미확인).
# 3. **국가 수준에서는 신호가 없습니다.** 8쌍 모두 구간이 0을 포함하고, 부호는 도메인×지표마다 달라 일관된 방향이 없습니다. 소득군 평균 수준의 간극은 보이지만 국가 간 상관으로는 확인되지 않는다는, 통과 도메인(safety만 약하게 양)과 비슷한 그림입니다.
# 4. **electrification_env의 예외는 설명하지 마세요.** HIC SBR 0.87과 UMC 1.40은 사전의 오탐(precision 0.80)이나 CO₂ 부담 분포 때문일 수 있으나, 확인하지 않았습니다.
# 5. **제외 민감도:** 목표 렌즈에서 CN 제외 시 electrification_env 0.781, freight 0.719이고 CN·IN 동시 제외에서는 0.656, 0.688로 낮아집니다. 역량 렌즈는 안정적이고 목표 렌즈는 중국·인도 제외에 더 민감합니다. 통과 도메인의 91~98% 유지와 비교해 낮은 편이지만, 도메인 수가 다르고 비교 기준을 사전에 정하지 않았으므로 해석은 기술 수준에 둡니다.

# ## 3. 쓸 수 있는 문구

# > "사전 precision이 기준(0.85)에 못 미친 네 도메인(0.76~0.84)은 본문 분석에서 제외했다. 참고로 같은 규칙을 적용하면 역량 렌즈의 마지막 기간에서 8개 도메인-지표 쌍 중 7개가 고소득군 과잉·저소득/중하위소득군 과소 패턴을 보였고, 지표 일관 비율은 0.94~1.00이었다. 국가 수준 편 순위상관은 모든 쌍에서 0과 구분되지 않았다. 이 결과는 소득 수준과 분리되지 않으며 통과 도메인 판정(2/6)을 바꾸지 않는다."

# 쓰면 안 되는 것: "미달 도메인에서도 정렬이 확인됐다", "일반화된다", 필요 충족 증거.

# ## 4. 기록과 커밋

# 앞선 `grep` 정정(기준 0이 틀렸고 해당 줄이 정상 `try:` 줄이었다는 점)도 같은 줄에서 마무리합니다. 해당 `grep -n` 출력이 아직 없으니, 그 확인 결과는 아래 명령에서 같이 보세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# grep -n "ODE\]()" pa_b8a.py | cut -c1-120
# pgrep -f pa_b8a.py || echo "종료 확인"
# ls -l 09_PA/PA_B8a_*.csv
# ```

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B8a 결과 해석 메모(결정적+부트스트랩 B=S8 동일, 재현 점검 부트스트랩 구간까지 일치): 미달 4개 도메인(precision 0.76~0.84) 참고용. 역량 렌즈 P2025에서 8쌍 중 7쌍 HIC 과잉·LIC/LMC 과소(예외 electrification_env×transport_co2, HIC 0.87), 지표 일관 0.938~1.000(목표 렌즈 freight 0.750), 편 순위상관 8쌍 모두 구간이 0 포함(부호 혼재), 제외 민감도 역량 렌즈 0.906~1.000·목표 렌즈 CN·IN 제외 시 0.656~0.875. 소득 효과와 분리되지 않으며 통과 도메인과 같은 선상의 증거 아님. 정정: 앞선 QUICK 열람 줄의 '위 점검 명령으로 재확인'에서 grep -c ODE 기준(0)은 제 설명 오류였고 해당 1건은 정상 try 줄(확인 명령 grep -n 결과 별도). 계획서 PLAN status는 정상(py_compile 문자열 없음) | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_B8a_*.csv paper1/09_PA/PA_B8a_full_log.txt paper1/09_PA/PA_CHANGELOG.md
# git status --short | head -n 8
# git commit -m "PA-B8a results" && git tag -a PA-B8a-results-2026-10-21 -m "PA-B8a outputs, seen" && git push --follow-tags
# ```

# `git status`에 스테이징 외 항목이 보이면 커밋 전에 알려 주세요.

# ## 5. 남은 항목

# 1. 위 커밋·푸시
# 2. B6(수작업 50쌍), B8-2(새 25편) 인력 여부: 이번 주 내 결정, 없으면 "미수행, 한계로 명시"
# 3. `oa_raw` 외부 백업(미완료)
# 4. A4 부록 표 확정(B2 동결 절차 비고, A1 패치, B7b·B8a 행 추가)
# 5. `S8.5` 노트북 최종 커밋
# 6. S9, S10은 시작 전

# 분석 항목은 B6/B8-2를 빼면 모두 소진됐습니다. 다음으로 A4 표 마무리와 논문 문구 통합표(쓸 수 있음/쓰면 안 됨, 약화·반전된 PA 공개 목록)를 만드는 것을 권합니다.

In [ ]:
# # PA-B8a 기록 확인

# 커밋 `85834d3`과 태그 `PA-B8a-results-2026-10-21`이 푸시됐습니다. 스테이징은 예상한 7개(결과 CSV 5, 로그, changelog)뿐이었습니다.

# - **`grep -n` 확인:** 162행 하나이고 정상 `try:` 줄입니다. 앞서 `grep -c` 결과 1은 이 줄이었으니 파일 이상은 없습니다. 정정 내용이 changelog에 반영됐습니다.
# - **프로세스 종료**와 결과 CSV 5개가 확인됐습니다.
# - **`S8.5` 노트북**은 계속 modified 상태입니다. 작업 기록이 계속 쓰이고 있으니 마지막에 한 번만 커밋하면 됩니다.

# ## 분석 단계 마감

# B6(수작업 50쌍)과 B8-2(새 25편 코딩)를 제외하면 계획한 PA가 모두 끝났습니다. 두 항목은 인력 여부에 대한 답을 아직 받지 못했습니다. 이번 주 안에 답이 없으면 "미수행, 한계로 명시"로 확정하겠습니다.

# ## 다음 단계 제안

# 1. **통합 문구표 작성:** 모든 PA 결과를 반영한 "쓸 수 있음/쓰면 안 됨" 표와, 약화·반전된 결과 공개 목록입니다. 공개 목록에 들어갈 항목은 C1(중국 소속 제외 시 진입 우세 소멸), A2b(y2 정의에서 역전), B4/B4b(경계 사례), B9a(소득군 내 NA), B8a(미달 도메인 참고용)입니다.
# 2. **A4 부록 표 확정:** B2 동결 절차 비고, A1 패치, B7b·B8a 행을 추가합니다.
# 3. **잔여 정리:** `oa_raw` 외부 백업(매체 연결 후 쓰기 권한 확인, 복사, SHA-256 비교), `S8.5` 노트북 최종 커밋.
# 4. **투고 준비(S9, S10):** 후보 저널과 구성안 확정이 필요합니다.

# 1번부터 시작하길 권합니다. 지금까지의 결과를 한 장으로 묶어야 논문 서사(제목에서 "체제 전환"을 빼고, 진입 중심 갱신과 중국 소속 유입을 하나의 구조로 서술)를 확정할 수 있기 때문입니다. 시작할까요? 그리고 B6/B8-2 인력 여부와 투고 후보 저널을 알려 주세요.

In [ ]:
# 아니요, 다 한 건 아닙니다. 지침서(14개 항목)와 지금까지의 진행을 대조하면 이렇습니다.

# ## 완료한 것

# | 항목 | 상태 |
# |---|---|
# | PA-0 재현성 | 대부분 완료 (아래 미완 참고) |
# | A1, A2(a·b·c), A3 | 완료 |
# | B2, B2b | 완료 |
# | B3 | B3a만 완료 |
# | B4 | S5·S6(B4)와 S4(B4b) 완료 |
# | B5 | 완료 |
# | B7 | B7a, B7b 완료 (B7c는 보류) |
# | B8 | B8-1(B8a)만 완료 |
# | B9 | B9a만 완료 |
# | C1, C2 | 완료 |

# ## 하지 않은 것 (지침서 대비)

# | 항목 | 사유 |
# |---|---|
# | **B6** 위험군 독립 정답 (ORCID, 수작업 50쌍) | 연구자 시간 여부 미답. 답이 없으면 "미수행, 한계로 명시" |
# | **B8-2** 사전 v3, 새 25편 이중 코딩 | 같은 이유로 미수행 |
# | **B9b** 공급 정의(논문 수 기준), min_country_mass 5/20 | 정보량이 낮다고 판단해 생략 (LIC는 자료상 불가) |
# | **B3b** D1′ 재구성 | 지침서의 전제(D1이 이미 2005–09 대 2015–19)가 어긋나 생략 |
# | **B3-3** 핵심 신생 집합의 신규 저자 진입 몫 | 미수행 |
# | **B7c** 저널×연도 IPW | 보류 (S4 IPW가 이미 방향 지지) |
# | **B4 일부** 임베딩 응집도, Jaccard 10퍼센타일 | 미수행 |
# | **A2의 ID 기반 재판별** | F7이 2020+만 덮어 불가, 분할 이름 제외 민감도로 축소 |
# | **A4** 열람 이력 부록 표 | 초안만 있고 확정 전 (B2 비고, A1 패치, B7b·B8a 행 추가 필요) |

# ## 분석 외 미완

# - `oa_raw` 외부 백업 (매체 없음)
# - GitHub 저장소 Private 확인
# - `S8.5` 노트북 최종 커밋
# - S9, S10 (투고 준비)

# ## 정리

# 핵심 3개(A1, A2, B2)와 대부분의 약점 리프레이밍은 끝났고, 남은 것은 대부분 의도적 생략이거나 사람 손이 필요한 항목입니다. 생략한 항목은 사유와 함께 부록의 "미수행 목록"에 적어야 합니다.

# 다음 단계로 통합 문구표(쓸 수 있음/쓰면 안 됨)와 A4 확정을 권합니다. B6/B8-2를 할 수 있는지, 투고 후보 저널이 있는지 알려 주세요.

In [ ]:
# 이유는 네 종류로 나뉩니다. 그중 일부는 제 판단이었고, 그 점은 솔직히 짚어 두겠습니다.

# ## 1. 자료나 사람 때문에 못 하는 것

# - **B6(수작업 50쌍, ORCID)과 B8-2(새 25편 이중 코딩):** 연구자가 직접 판정하거나 코딩해야 합니다. 제가 대신할 수 없고, 시간이 되는지 여러 번 물었지만 답을 못 받았습니다.
# - **A2의 ID 기반 재판별:** F7(OpenAlex ID)이 2020년 이후 논문에만 있어서, 2015–19 저자를 ID로 가를 수 없습니다. 그래서 "분할 이름 제외 민감도"로 줄였습니다.
# - **B9의 LIC·LMC 소득군 내 분석:** 질량 하한 때문에 LIC가 0개국이고 LMC는 n<20입니다. 하한을 낮춰도 해결되지 않습니다.

# ## 2. 지침서의 전제가 틀렸던 것

# - **B3b(D1′ 재구성):** 코드를 확인해 보니 D1이 이미 2005–09 대 2015–19였습니다. 지침서가 재구성하라는 대상이 이미 그 구간이어서 의미가 없었습니다.

# ## 3. 제가 "효과 대비 정보량이 낮다"고 판단해 생략을 권한 것

# - **B9b, B3b, B7c:** 제 판단이었습니다. 특히 B9b는 "별말씀이 없어 생략으로 확정"이라고 처리했는데, 침묵을 동의로 본 것은 좋은 방식이 아니었습니다. 이 셋은 하려면 할 수 있습니다. 부록의 미수행 목록에 사유를 적을 수는 있지만, 지침서의 "보고 의무"를 생각하면 연구자께서 직접 결정하실 사항입니다.

# ## 4. 아직 시작하지 못한 것

# - **B3-3, B4의 응집도·Jaccard 지표, A4 확정:** 입력 구조 확인이 필요했거나 다른 항목에 밀렸습니다. 막힌 이유는 없고 그냥 남아 있는 것입니다.

# ## 제안

# 3번의 세 항목(B9b, B3b 대체안, B7c)을 정말 생략할지, 아니면 할지 정해 주세요. 하신다면 B3-3과 B4의 응집도 지표를 먼저 권합니다. 이 둘은 막힌 이유가 없고, 지침서가 기대한 강화 문장(핵심 신생 토픽과 신규 저자 진입의 연결)과도 직접 이어지기 때문입니다.

In [ ]:
# 아래는 3번(B9b, B3b, B7c)과 4번(B3-3, B4의 응집도·Jaccard, A4)의 코드입니다. 전부 한 번도 실행하지 않았고, 계획 해시는 이름을 `B3c`(= B3-3), `B4c`(= B4의 응집도·Jaccard)로 둡니다.

# 먼저 알아 둘 점이 세 가지 있습니다.

# - **B4c는 입력 위치를 모릅니다.** 임베딩 파일과 부트스트랩 Jaccard 반복값 파일이 어디 있는지 본 적이 없습니다. 그래서 `discover` 모드로 후보를 찾고, 경로를 채운 뒤에 동결하는 2단계 구조입니다. 이 경로 확인 없이 추측으로 동결하면 사양이 틀립니다.
# - **B3b는 지침서와 다릅니다.** 지침서의 D1′는 D1이 이미 2005–09 대 2015–19라서 의미가 없었습니다. 그래서 D1의 이질성을 구간과 규칙으로 분리하는 두 변형으로 대체했습니다.
# - **B7c는 단순화했습니다.** "저널×연도 IPW"를 저널×연도 셀 사후층화(셀 전체 논문 수 ÷ 관측 논문 수)로 구현했습니다. S4의 로지스틱 IPW 모델과 같지 않습니다.

# 실행 순서는 다음과 같습니다.

# 1. 공통 모듈 `pa_common.py`를 저장합니다.
# 2. 각 스크립트를 `freeze → tag → push → run quick → run` 순서로 실행합니다.
# 3. 권장 순서는 B3c → B3b → B7c → B9b → B4c(`discover` 후) → A4입니다.

# ## 0. 공통 모듈: `paper1/pa_common.py`

# ```python
# # -*- coding: utf-8 -*-
# # 사후분석 공통 모듈(동결·태그·실행 검증, S8/S4 입력 로더). 미실행 코드.
# # 각 pa_*.py는 python3 pa_xxx.py freeze | tag | run [quick] 로 호출. 동결 후 이 파일과 스크립트 해시가 바뀌면 run이 중단됨.
# import sys, json, re, hashlib, subprocess, warnings
# from pathlib import Path
# import numpy as np, pandas as pd
# from scipy.stats import rankdata
# warnings.filterwarnings("ignore"); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 400)

# ROOT = Path.home()/"Research/Minnesota_Transport"; P1 = ROOT/"paper1"; PA = P1/"09_PA"; DATA = ROOT/"analysis_data/final"
# S0, S1, S2, S3, S4, S5D, S6D, S8 = (P1/x for x in ["00_S0", "01_S1", "02_S2", "03_S3", "04_S4", "05_S5", "06_S6", "08_S8"])
# now = lambda: pd.Timestamp.now().isoformat(timespec="seconds")
# sha_b = lambda b: hashlib.sha256(b).hexdigest(); sha_f = lambda p: sha_b(Path(p).read_bytes())
# tf = lambda s: s.astype(str).str.strip().str.lower().isin(["1", "1.0", "true", "t", "yes"]); bl = tf
# git = lambda *a: subprocess.run(["git", "-C", str(ROOT), *a], capture_output=True, text=True)
# class Halt(Exception): pass
# spec_hash = lambda s: sha_b(json.dumps({k: v for k, v in s.items() if k not in ("sha256", "created", "refs")}, sort_keys=True, ensure_ascii=False).encode())
# V2P = PA/"PA_0_PLAN_v2.json"; CL = PA/"PA_CHANGELOG.md"
# INT = dict(road_deaths="death_rate", vehicles="veh_per1000", urban_pop="urban_share", transport_co2="co2_tr_pc")

# def cl(line):
#     t = CL.read_text(encoding="utf-8") if CL.exists() else ""
#     with CL.open("a", encoding="utf-8") as f: f.write(("" if (not t or t.endswith("\n")) else "\n") + f"- {now()} | {line}\n")

# def find_key(o, key):
#     if isinstance(o, dict):
#         for k, v in o.items():
#             if k == key: return v
#             r = find_key(v, key)
#             if r is not None: return r

# def periods():
#     """S4 동결 규칙 파일에서 main_contrast와 periods를 읽음(pa_b7a.py와 같은 방식)."""
#     for f in sorted(P1.glob("0[0-4]_*/*PREREG*.json")):
#         j = json.loads(f.read_text(encoding="utf-8")); mc, pr = find_key(j, "main_contrast"), find_key(j, "periods")
#         if mc and pr and all(k in pr for k in mc): return mc[0], mc[1], {k: tuple(v) for k, v in pr.items()}
#     raise Halt("main_contrast와 periods를 가진 S4 동결 규칙 파일을 찾지 못함")

# def analysis_sets():
#     """S4 분석 집합 F(76,984편)와 결측 집합 Mi(분석 저널 안). pa_b7a.py와 같은 정의."""
#     W = pd.read_csv(DATA/"F1_papers.csv", low_memory=False); W["pid"] = W.pid.astype(str)
#     EXC = set(pd.read_csv(S0/"EXCLUDE_PIDS_FROZEN.csv", dtype=str, encoding="utf-8-sig").pid)
#     W = W[~W.pid.isin(EXC) & tf(W.in_sample) & W.year.between(2000, 2025)].copy(); W["ht"] = tf(W.has_topic) & W.topic150.notna()
#     F, Mi = W[W.ht].copy(), W[~W.ht].copy()
#     if len(F) != 76984: raise Halt(f"분석 집합 재현 실패: {len(F)} (76984여야 함)")
#     F["topic"] = F.topic150.astype(int)
#     topics = np.array(sorted(pd.read_csv(S3/"L7_topic_grade.csv").topic.astype(int))); venues = np.array(sorted(F.venue_short.unique()))
#     return F, Mi[Mi.venue_short.isin(venues)], topics, venues

# # ---------- S8 입력(pa_b8a.py와 같은 정의) ----------
# def burden():
#     Bt = pd.read_csv(S8/"burden_country_table.csv", keep_default_na=False, index_col=0); inc = Bt["income"].copy()
#     Bt = Bt.drop(columns="income").apply(pd.to_numeric, errors="coerce"); Bt["income"] = inc; return Bt
# def mass_raw():
#     pm = S2/"country_mass_long.parquet"; M = pd.read_parquet(pm) if pm.exists() else pd.read_csv(S2/"country_mass_long.csv.gz", dtype={"pid": str})
#     M["pid"] = M.pid.astype(str); M = M[(M.year >= 2007) & ~M.country.isin(["UNK", "ZZ_former"])]; return M[["pid", "year", "country", "w", "scheme", "variant"]]
# def flags(SPEC):
#     RX = {k: re.compile(v, re.I) for k, v in SPEC["domains"].items()}
#     F1 = pd.read_csv(DATA/"F1_papers.csv", low_memory=False); F1["pid"] = F1.pid.astype(str)
#     exf = S0/"EXCLUDE_PIDS_FROZEN.csv"; log = pd.read_csv(S0/"FREEZE_LOG.csv"); last = log[log.file == exf.name].iloc[-1]
#     if sha_f(exf) != last.sha256: raise Halt("EXCLUDE_PIDS_FROZEN.csv 해시 불일치")
#     F = F1[~F1.pid.isin(set(pd.read_csv(exf, dtype=str, encoding="utf-8-sig").pid))].copy(); F = F[F.year >= 2007].copy(); F = F[F.title.notna()]
#     fl = pd.DataFrame({d: F.title.fillna("").str.contains(rx).values for d, rx in RX.items()}, index=F.index); fl["pid"] = F.pid.values; return fl

# def pr(x, y, cols=(), rk=()):
#     a, b = rankdata(x), rankdata(y)
#     if not len(cols): return float(np.corrcoef(a, b)[0, 1])
#     A = np.column_stack([np.ones(len(a))] + [rankdata(c) if r else np.asarray(c, float) for c, r in zip(cols, rk)])
#     res = lambda v: v - A @ np.linalg.lstsq(A, v, rcond=None)[0]; return float(np.corrcoef(res(a), res(b))[0, 1])

# def label(df, col, qh):
#     cut = df[col].abs().quantile(2/3); big = df[col].abs() >= cut
#     lab = ["small_change" if not b else ("growth" if d > 0 else "decline") + ("_highQ" if q else "_notHighQ") for b, d, q in zip(big, df[col], qh)]
#     return pd.Series(lab, index=df.index), big, float(cut)

# def run_pa(name, PLAN, run_fn, ref_files, why):
#     BP = PA/f"PA_{name}_PLAN.json"; script = f"pa_{name.lower()}.py"
#     mode = sys.argv[1] if len(sys.argv) > 1 else "freeze"; quick = "quick" in sys.argv[2:]
#     try:
#         if mode == "freeze":
#             if not V2P.exists(): raise Halt("PA_0_PLAN_v2.json이 없습니다")
#             h = spec_hash(PLAN); print(f"PA-{name} 계획 해시(예정)", h[:12])
#             if BP.exists(): print("이미 동결:", "동일" if json.loads(BP.read_text(encoding="utf-8"))["sha256"] == h else "⛔ 다름"); return
#             refs = {f.name: sha_f(f) for f in [V2P, P1/"pa_common.py", P1/script] + [Path(x) for x in ref_files] if Path(f).exists()}
#             BP.write_text(json.dumps(dict(PLAN, created=now(), sha256=h, refs=refs), ensure_ascii=False, indent=1), encoding="utf-8")
#             cl(f"PA-{name} 계획 동결(해시 {h[:12]}) | 사유: {why} | 이전에 생략을 권했던 항목은 연구자 요청으로 재개(침묵을 동의로 본 처리를 정정). 결과 미열람. QUICK은 재현 점검과 설계 정보만 | 본 결과 변경 여부: 없음")
#             print("저장:", BP.name)
#         elif mode == "tag":
#             if not BP.exists(): raise Halt("먼저 freeze")
#             t = f"PA-{name}-plan-{pd.Timestamp.now().strftime('%Y-%m-%d')}"
#             if git("tag", "-l", t).stdout.strip(): print("이미 있음:", t); return
#             r = git("add", "--", f"paper1/09_PA/PA_{name}_PLAN.json", "paper1/09_PA/PA_CHANGELOG.md", f"paper1/{script}", "paper1/pa_common.py"); print("add", r.returncode, r.stderr.strip())
#             r = git("commit", "-m", f"PA-{name} plan freeze (before execution)"); print("commit", r.returncode, (r.stdout + r.stderr)[:200])
#             r = git("tag", "-a", t, "-m", f"PA-{name} plan"); print("tag", r.returncode, r.stderr.strip()); print("※ 이어서 git push --follow-tags")
#         elif mode == "run":
#             if not BP.exists(): raise Halt("먼저 freeze")
#             sp = json.loads(BP.read_text(encoding="utf-8"))
#             if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(PLAN): raise Halt("동결 파일과 현재 계획이 다릅니다")
#             if not git("tag", "-l", f"PA-{name}-plan-*").stdout.strip(): raise Halt("tag 단계를 먼저 실행하세요")
#             for fn in ["pa_common.py", script]:
#                 if sp["refs"].get(fn) != sha_f(P1/fn): raise Halt(f"{fn}이 동결 시점과 다름(수정됐다면 v2 계획으로 분리하고 사유 기록)")
#             run_fn(quick, sp)
#             if not quick: cl(f"PA-{name} 실행 | 정의: PA_{name}_PLAN {sp['sha256'][:12]} | 결과: PA_{name}_*.csv | 본 결과 변경 여부: 없음")
#         else: raise Halt("모드는 freeze | tag | run [quick]")
#     except Halt as e: print(f"⛔ 중단: {e}")
# ```

# ## 1. B3-3: 신생 핵심 집합의 진입·적응 기여 (`pa_b3c.py`)

# 핵심(≥3개 정의)·중간(2)·특이적(1) 신생 토픽이 S6 분해에서 어느 항(진입, 적응, 재배분)으로 설명되는지 봅니다. 분해는 토픽별로 이미 계산되므로 새 데이터가 필요 없습니다. 집합별 합산 몫과 저자 단위 부트스트랩 구간을 냅니다. 쌍별로 읽고 합치지 않습니다.

# ```python
# # -*- coding: utf-8 -*-
# # PA-B3c(=지침서 B3-3): 신생 핵심 집합의 S6 분해 항별 기여. 미실행 코드. python3 pa_b3c.py freeze | tag | run [quick]
# from pa_common import *
# PLAN = dict(
#     version=1, seed=2026, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(지침서 B3-3). B3a 결과(핵심 7·중간 10·특이적 14), S6 Pair 소스, PA-A2c(4항 분해, 진입=신규+복귀) 열람 상태. 결과 미열람. 이전에 생략 권고(미수행 목록 후보)했으나 연구자 요청으로 수행. S0~S8 판정 불변.",
#     sets="definition_sensitivity.csv의 n_defs: core>=3, mid==2, specific==1, union31=전체 31개. 참조 집합 all_growth=쌍별 총변화>0인 전 토픽",
#     setting="spec['main']과 s6_lib.make_pair 그대로, 쌍은 pairs_of(spec) 전부. 토픽 축은 Pair의 K(L['topics']). 집합 토픽이 L['topics']에 없으면 제외하고 수를 보고",
#     stats="집합 안에서 쌍별 총변화>0인 토픽만. 합산 몫: entry_frac=sum(entry)/sum(total), entry_new_frac, adapt_frac, realloc_frac(진입=신규+복귀). 토픽별 entry/total의 중앙값, entry>adapt인 토픽 비율. 차이: core-all_growth, core-specific의 entry_frac",
#     boot="저자 가중 wa~Exp(1)만(PA-A2c 저자 모드), B=spec['B_boot'], 역할 고정, 관측·백분위·basic 구간",
#     reading="쌍별로 core의 entry_frac와 adapt_frac의 대소, core-all_growth entry_frac 차이의 부호와 구간이 0을 포함하는지를 기술. 임계값·판정 없음, 쌍을 합치지 않음. 마지막 쌍(P1519->P2025)을 주 대상으로 표기",
#     reproduce_check="(a) 쌍별 total_half가 renewal_decomposition.csv(scheme=main, variant=all_venues)와 1e-6 이내, (b) 항 합=총변화(Pair 내부 assert), (c) definition_sensitivity 31행",
#     caveat="D2~D4가 변화량 상위 14 기준이라 핵심 집합은 총변화가 큰 토픽에 치우침(구조적). 합산 몫은 큰 토픽이 지배. 토픽 단위 인과 아님. 신생 후보가 성장 토픽뿐이라 total>0 조건이 대부분 충족될 것(미확인)",
#     forbidden="결과를 본 뒤 집합·통계·판독 변경(변경 시 v2 계획으로 분리하고 사유 기록)")

# def run(quick, sp):
#     sys.path.insert(0, str(P1)); import s6_lib as S
#     spec = S.load_spec(); L = S.load_all(spec); M = spec["main"]; PAIRS = S.pairs_of(spec); half = S.half
#     DS = pd.read_csv(S4/"definition_sensitivity.csv", encoding="utf-8-sig")
#     if len(DS) != 31: raise Halt(f"definition_sensitivity 행 수 {len(DS)} (31이어야 함)")
#     sets = {"core": set(DS.topic[DS.n_defs >= 3].astype(int)), "mid": set(DS.topic[DS.n_defs == 2].astype(int)), "specific": set(DS.topic[DS.n_defs == 1].astype(int)), "union31": set(DS.topic.astype(int))}
#     pos = {int(t): i for i, t in enumerate(L["topics"])}; K = len(L["topics"]); G = {nm: [pos[t] for t in ids if t in pos] for nm, ids in sets.items()}; G["all_growth"] = list(range(K))
#     ref = pd.read_csv(S6D/"renewal_decomposition.csv"); ref = ref[(ref.scheme == M["scheme"]) & (ref.variant == "all_venues")].set_index("pair")
#     B = 20 if quick else int(spec["B_boot"]); lo_q, hi_q = spec["ci"]; rng = np.random.default_rng(int(PLAN["seed"]))
#     print("[설계 정보] 집합 크기(정의/S6 토픽 축에 존재):", {nm: (len(sets[nm]), len(G[nm])) for nm in sets})
#     def stats(d):
#         tot = d["total"]; out = {}
#         for nm, ix in G.items():
#             ix = np.array([i for i in ix if tot[i] > 0], int)
#             if not len(ix): out[nm] = None; continue
#             T_ = tot[ix].sum(); out[nm] = dict(n=len(ix), entry_frac=d["entry"][ix].sum()/T_, entry_new_frac=d["entry_new"][ix].sum()/T_, adapt_frac=d["adapt"][ix].sum()/T_, realloc_frac=d["realloc"][ix].sum()/T_,
#                                                med_entry_frac=float(np.median(d["entry"][ix]/tot[ix])), share_entry_gt_adapt=float((d["entry"][ix] > d["adapt"][ix]).mean()))
#         return out
#     def flat(st):
#         r = {}
#         for nm, v in st.items():
#             for k, x in (v or {}).items(): r[(nm, k)] = x
#         g = lambda a, b: (st[a]["entry_frac"] - st[b]["entry_frac"]) if st.get(a) and st.get(b) else np.nan
#         r[("core_minus_all_growth", "entry_frac")] = g("core", "all_growth"); r[("core_minus_specific", "entry_frac")] = g("core", "specific"); return r
#     rows = []
#     for p0, p1 in PAIRS:
#         name = f"{p0}->{p1}"; P = S.make_pair(L, p0, p1, M["scheme"], M["venue"], M["topic"], M["hom"]); d0 = P.decompose()
#         if abs(half(d0["total"]) - float(ref.loc[name, "total_half"])) > 1e-6: raise Halt(f"S6 관측 재현 실패({name})")
#         obs = flat(stats(d0)); bs = []
#         for _ in range(B):
#             wa = rng.exponential(size=P.nA); bs.append(flat(stats(P.decompose(P.m0*wa[P.a0], P.m1*wa[P.a1]))))
#         for k, o in obs.items():
#             if k[1] == "n": rows.append(dict(pair=name, set=k[0], stat="n_topics", obs=o)); continue
#             v = np.array([b.get(k, np.nan) for b in bs], float); ql, qh = np.nanpercentile(v, [lo_q, hi_q]) if np.isfinite(v).any() else (np.nan, np.nan)
#             rows.append(dict(pair=name, set=k[0], stat=k[1], obs=o, pct_lo=ql, pct_hi=qh, basic_lo=2*o - qh, basic_hi=2*o - ql, B=B))
#         print(f"  {name} 완료")
#     R = pd.DataFrame(rows)
#     if quick: print("\n[QUICK] 재현 점검과 설계 정보만. 결과 통계는 출력하지 않음(_QUICK 파일 열람 금지)."); R.to_csv(PA/"PA_B3c_results_QUICK.csv", index=False, encoding="utf-8-sig"); return
#     R.to_csv(PA/"PA_B3c_results.csv", index=False, encoding="utf-8-sig")
#     print("\n[Post-hoc] PA-B3c 집합별 합산 몫(쌍별)\n", R[R.stat.isin(["n_topics", "entry_frac", "adapt_frac", "realloc_frac", "entry_new_frac"])].pivot_table(index=["pair", "set"], columns="stat", values="obs").round(4).to_string())
#     print("\n[Post-hoc] 차이와 구간\n", R[R.set.str.startswith("core_minus")].round(4).to_string(index=False))
#     print("\n[Post-hoc] 토픽별 지표\n", R[R.stat.isin(["med_entry_frac", "share_entry_gt_adapt"])].round(4).to_string(index=False))
#     print("※ 쌍별로 따로 읽고 합치지 않음. 핵심 집합은 변화량 상위 정의의 부산물일 수 있음. S0~S8 판정 불변.")

# run_pa("B3c", PLAN, run, [S4/"definition_sensitivity.csv", P1/"s6_lib.py", S6D/"renewal_decomposition.csv", PA/"PA_B3a_PLAN.json"], "지침서 B3-3: 신생 핵심 집합의 신규 진입 연계(S6 항별 기여)")
# ```

# ## 2. B3b: D1의 이질성을 구간과 규칙으로 분리 (`pa_b3b.py`)

# ```python
# # -*- coding: utf-8 -*-
# # PA-B3b: D1(=F6.emerging; 2005-09 대 2015-19, r>=3 & s1>=0.005)의 이질성을 구간·규칙으로 분리. 지침서 D1'(구간 재구성)는 전제가 어긋나 대체. 결정적. 미실행.
# from pa_common import *
# PLAN = dict(
#     version=1, parent="PA_0_PLAN_v2.json",
#     status="사후 명세. s4_main.py L116-119로 D1~D4 정의(D1=r=(s1+1e-4)/(s0+1e-4)>=3 and s1>=0.005, 구간 2005-09 대 2015-19; D2=주 대비 평활 로그 비율 상위 14; D3=주 대비 Δs 상위 14; D4=민감도 대비 Δs 상위 14), B3a 결과 열람. 결과 미열람. 지침서의 D1'는 D1이 이미 그 구간이라 의미 없어 구간/규칙 분리 변형 3종으로 대체. S0~S8 판정 불변.",
#     shares="S4 분석 집합 F(76,984편)에서 기간별 토픽 비중(pa_b7a와 동일 계산). 주 대비·기간은 S4 동결 규칙 파일에서 읽음. 비중 기준은 topic_quadrants.csv의 delta와 일치(F6_topics의 F2 기준 비중과 다름)",
#     variants={"D1_rule": "D1 구간(2005-09 대 2015-19)에서 D2 규칙(평활 로그 비율 상위 14, 평활 eps=1e-4)", "D1_rule_ds": "D1 구간에서 D3 규칙(Δs 상위 14)", "D1_interval": "주 대비 구간에서 D1 규칙(r>=3 and s1>=0.005, eps=1e-4)"},
#     stats="변형별 토픽 수, 저장된 D1~D4(definition_sensitivity.csv)와의 쌍별 Jaccard, 원 D1과 변형 각각의 D2·D3·D4에 대한 평균 Jaccard",
#     reading="원 D1의 (D2,D3,D4) 평균 Jaccard 대비 D1_rule/D1_rule_ds가 더 크면 '규칙 차이가 이질성의 주원인', D1_interval이 더 크면 '구간 차이가 주원인'으로 방향만 기술. 임계값·판정 없음. 둘 다 크거나 작으면 그대로 기술",
#     reproduce_check="(a) 관측 delta가 topic_quadrants.csv와 1e-9 이내, (b) 재계산 D3(주 대비 Δs 상위 14, 안정 정렬)가 저장 D3와 집합 일치(불일치 시 중단), (c) 정보용: 재계산 D1·D2와 저장 D1·D2의 Jaccard(F1 기준이라 완전 일치하지 않을 수 있음, 중단 조건 아님)",
#     caveat="D2 평활 방식과 D4(민감도 대비)는 미확인이라 D2는 정보용 재현, D4는 저장값만 사용. 31개 합집합 밖 토픽도 변형에 포함. 정의 차이의 설명이며 신생 토픽의 타당성 검증이 아님",
#     forbidden="결과를 본 뒤 변형·eps·판독 변경(변경 시 v2 계획으로 분리하고 사유 기록)")

# def run(quick, sp):
#     DS = pd.read_csv(S4/"definition_sensitivity.csv", encoding="utf-8-sig"); ks = ["D1", "D2", "D3", "D4"]
#     for k in ks: DS[k] = bl(DS[k])
#     st = {k: set(DS.topic[DS[k]].astype(int)) for k in ks}
#     c0, c1, P = periods(); fnd = lambda a, b: next((k for k, v in P.items() if tuple(v) == (a, b)), None)
#     k09, k19 = fnd(2005, 2009), fnd(2015, 2019)
#     if not (k09 and k19): raise Halt(f"periods에서 2005-09/2015-19를 찾지 못함: {P}")
#     F, Mi, topics, venues = analysis_sets(); pos = {int(t): i for i, t in enumerate(topics)}; fi = F.topic.map(pos).astype(int).values
#     sh = lambda lo, hi: (lambda m: np.bincount(fi[m], minlength=len(topics))/m.sum())(F.year.between(lo, hi).values)
#     s0, s1 = sh(*P[c0]), sh(*P[c1]); Q = pd.read_csv(S4/"topic_quadrants.csv").set_index("topic")
#     if np.abs((s1 - s0) - Q.loc[topics, "delta"].values).max() > 1e-9: raise Halt("관측 delta가 topic_quadrants.csv와 다름")
#     top14 = lambda x: set(topics[np.argsort(-x, kind="stable")[:14]].astype(int)); eps = 1e-4
#     lr = lambda a, b: np.log((b + eps)/(a + eps))
#     D3 = top14(s1 - s0)
#     if D3 != st["D3"]: raise Halt(f"재계산 D3가 저장 D3와 다름(차이 {sorted(D3 ^ st['D3'])}): 정의 재확인 필요")
#     a09, a19 = sh(*P[k09]), sh(*P[k19]); D1re = set(topics[((a19 + eps)/(a09 + eps) >= 3) & (a19 >= 0.005)].astype(int)); D2re = top14(lr(s0, s1))
#     J = lambda a, b: len(a & b)/len(a | b) if (a | b) else np.nan
#     print(f"[재현 OK] delta, 재계산 D3=저장 D3 | 정보: 재계산 D1 n={len(D1re)} Jaccard(저장 D1)={J(D1re, st['D1']):.2f}, 재계산 D2 Jaccard(저장 D2)={J(D2re, st['D2']):.2f}")
#     if quick: print("[QUICK] 재현 점검만."); return
#     V = {"D1_rule": top14(lr(a09, a19)), "D1_rule_ds": top14(a19 - a09), "D1_interval": set(topics[((s1 + eps)/(s0 + eps) >= 3) & (s1 >= 0.005)].astype(int))}
#     allsets = {**st, **V}; names = list(allsets)
#     JM = pd.DataFrame([[J(allsets[a], allsets[b]) for b in names] for a in names], index=names, columns=names)
#     mean_vs = {nm: float(np.mean([J(allsets[nm], st[k]) for k in ["D2", "D3", "D4"]])) for nm in ["D1"] + list(V)}
#     sz = pd.DataFrame({"n": {k: len(v) for k, v in allsets.items()}}); JM.to_csv(PA/"PA_B3b_jaccard.csv", encoding="utf-8-sig"); sz.to_csv(PA/"PA_B3b_sizes.csv", encoding="utf-8-sig")
#     pd.DataFrame([dict(set=k, topics=sorted(v)) for k, v in V.items()]).to_csv(PA/"PA_B3b_variant_topics.csv", index=False, encoding="utf-8-sig")
#     print("\n[Post-hoc] 정의·변형 크기\n", sz.T.to_string()); print("\n[Post-hoc] 쌍별 Jaccard\n", JM.round(2).to_string())
#     print("\n[Post-hoc] D2·D3·D4와의 평균 Jaccard:", {k: round(v, 3) for k, v in mean_vs.items()})
#     for v in V: print(f"  {v}: 원 D1 대비 {'더 큼' if mean_vs[v] > mean_vs['D1'] else '더 작거나 같음'} ({mean_vs[v]:.3f} vs {mean_vs['D1']:.3f})")
#     print("※ 방향만 기술. 변형은 사후 정의. S0~S8 판정 불변.")

# run_pa("B3b", PLAN, run, [S4/"definition_sensitivity.csv", S4/"topic_quadrants.csv", P1/"s4_main.py"], "D1 이질성의 구간·규칙 분리(지침서 D1' 대체)")
# ```

# ## 3. B7c: 저널×연도 셀 사후층화로 S4 라벨과 변화점 재점검 (`pa_b7c.py`)

# ```python
# # -*- coding: utf-8 -*-
# # PA-B7c: 저널x연도 셀 사후층화 가중(관측 논문에 (관측+결측)/관측 가중)으로 S4 big 토픽 부호·라벨과 S5 변화점을 재점검. 결정적. 미실행.
# from pa_common import *
# PLAN = dict(
#     version=1, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(지침서 B7-3의 단순화). S4 IPW(라벨 유지 0.98), B7a·B7b 결과, s5_lib 소스 열람 상태. 결과 미열람. S4의 IPW 모델은 아님. 이전에 보류했으나 연구자 요청으로 수행. S0~S8 판정 불변.",
#     weights="저널 j x 연도 t 셀 w_jt=(n_obs+n_miss)/n_obs (n_obs>0일 때; 결측은 분석 저널 안의 has_topic 거짓 논문). 셀 안 토픽 분포는 관측과 같다고 가정하는 사후층화로, 2007년 하락 같은 연도별 결측률 변화를 직접 보정",
#     labels="가중 카운트로 주 대비 delta_ipw 계산, big 컷은 전체 149토픽 |delta_ipw|의 2/3 분위로 재계산, quality_high는 topic_quadrants.csv 고정, 라벨은 s4_main.py label()과 같은 규칙(총변화 기준)",
#     changepoints="s5_lib.sample_P에 가중 카운트(cnt*w)를 넣고 s5_lib.detect를 S5 값(pen=c_primary*phi*(K-1)*log(T), phi·k_max·min_len 고정)으로 실행. 귀무 시뮬레이션은 하지 않고 관측 변화점과 R만 보고",
#     reading="(1) 원 big 50개 중 부호 유지 비율 >=0.90 '유지', 0.70~0.90 '부분', <0.70 '민감'. (2) 원 big의 같은 라벨 유지율 >=0.95 '둔감', 아니면 '민감'. (3) 표본별로 S5 관측 변화점 각각이 가중 후 변화점 ±1년 안에 있는가를 보고(판정 없음). 세 가지를 합치지 않음",
#     reproduce_check="(a) 셀 관측 카운트가 yearly_venue_topic_counts.npy의 저널·연도 합과 일치, (b) 관측 delta가 topic_quadrants.csv와 1e-9 이내이고 재계산 라벨이 quadrant와 일치, (c) 가중 없는 detect의 R이 regime_nulls.csv obs_R과 1e-3 이내",
#     caveat="셀 안 토픽 분포가 같다는 가정(저널 내 무작위 결측, B7a의 k=1과 같은 가정이나 연도 단위). 가중으로 유효 표본 크기가 늘어 S5 벌점 대비 비용 비율이 달라짐(변화점 이동이 탐지기 설정 효과일 수 있음). 결측 사유가 토픽과 무관하다는 보장 없음",
#     forbidden="결과를 본 뒤 가중·임계값 변경(변경 시 v2 계획으로 분리하고 사유 기록)")

# def run(quick, sp):
#     sys.path.insert(0, str(P1)); import s5_lib as L5
#     c0, c1, P = periods(); F, Mi, topics, venues = analysis_sets()
#     cnt, vs, tp, fixed_v = L5.load_counts(); Y = np.asarray(L5.YEARS); wbar = cnt.sum((0, 2))/cnt.sum()
#     if not np.array_equal(np.asarray(tp), topics) or set(vs) != set(venues): raise Halt("S5 토픽·저널 축이 S4 정렬 목록과 다름: topics_order.csv/venues_order.csv 확인 필요")
#     obs = pd.crosstab(F.year, F.venue_short).reindex(index=Y, columns=vs, fill_value=0).values.astype(float)
#     miss = Mi.groupby(["year", "venue_short"]).size().unstack(fill_value=0).reindex(index=Y, columns=vs, fill_value=0).values.astype(float)
#     if not np.allclose(obs, cnt.sum(2)): raise Halt("셀 관측 카운트가 yearly_venue_topic_counts.npy와 다름")
#     w = np.where(obs > 0, (obs + miss)/np.where(obs > 0, obs, 1), 1.0); cw = cnt*w[:, :, None]
#     share = lambda c, lo, hi: (lambda a: a/a.sum())(c[(Y >= lo) & (Y <= hi)].sum((0, 1)))
#     d_obs = share(cnt, *P[c1]) - share(cnt, *P[c0]); d_ipw = share(cw, *P[c1]) - share(cw, *P[c0])
#     Q = pd.read_csv(S4/"topic_quadrants.csv", encoding="utf-8-sig").set_index("topic").loc[topics]
#     if np.abs(d_obs - Q.delta.values).max() > 1e-9: raise Halt("관측 delta가 topic_quadrants.csv와 다름")
#     qh = bl(Q.quality_high).values; D = pd.DataFrame(dict(topic=topics, delta=Q.delta.values, delta_ipw=d_ipw), index=topics)
#     q0, b0, _ = label(D, "delta", qh)
#     if not (q0.values == Q.quadrant.values).all(): raise Halt("관측 delta로 재계산한 라벨이 topic_quadrants.csv와 다름")
#     s5 = json.loads((S5D/"S5_0_PREREG.json").read_text(encoding="utf-8")); Dd = s5["detection"]; PHI = {k: v["phi"] for k, v in Dd["phi"].items()}; C0, KMAX, MINLEN = Dd["c_primary"], Dd["k_max"], Dd["min_len"]
#     K, T = len(topics), len(Y); SAM = s5["samples"]; ref = pd.read_csv(S5D/"regime_nulls.csv"); ref = ref[ref["null"] == "N2"].set_index("sample")
#     def det(c, s): Pm, n = L5.sample_P(c, fixed_v, wbar, s); return L5.detect(Pm, n, C0*PHI[s]*(K - 1)*np.log(T), KMAX, MINLEN)
#     for s in SAM:
#         if abs(det(cnt, s)["R"] - float(ref.loc[s, "obs_R"])) > 1e-3: raise Halt(f"S5 관측 R 재현 실패({s})")
#     print("[재현 OK] 셀 카운트, 관측 delta·라벨, S5 관측 R | 가중 w 범위", round(float(w[obs > 0].min()), 3), "~", round(float(w[obs > 0].max()), 3))
#     if quick: print("[QUICK] 재현 점검만."); return
#     q1, b1, cut1 = label(D, "delta_ipw", qh); ob = Q.big.pipe(bl).values
#     kept_sign = np.sign(d_ipw) == np.sign(Q.delta.values); sg = float(kept_sign[ob].mean()); same = float((q1.values[ob] == Q.quadrant.values[ob]).mean()); newbig = int((b1.values & ~ob).sum())
#     atten = float(np.median(np.abs(d_ipw[ob])/np.abs(Q.delta.values[ob])))
#     out = pd.DataFrame([dict(n_big_orig=int(ob.sum()), n_big_ipw=int(b1.sum()), sign_kept_share=sg, label_same_share=same, n_newly_big=newbig, cut_orig=float(Q.delta.abs().quantile(2/3)), cut_ipw=cut1, atten_median=atten)])
#     D.assign(big_orig=ob, big_ipw=b1.values, quad_orig=Q.quadrant.values, quad_ipw=q1.values, sign_kept=kept_sign).to_csv(PA/"PA_B7c_topics.csv", index=False, encoding="utf-8-sig"); out.to_csv(PA/"PA_B7c_summary.csv", index=False, encoding="utf-8-sig")
#     meta = json.loads((S5D/"S5_meta.json").read_text(encoding="utf-8"))["obs_breaks"]; rows = []
#     for s in SAM:
#         a, b = det(cnt, s), det(cw, s); ya, yb = [int(Y[x]) for x in a["br"]], [int(Y[x]) for x in b["br"]]
#         rows.append(dict(sample=s, breaks_base=str(ya), breaks_ipw=str(yb), s5_breaks=str(meta[s]), s5_kept_pm1=str([L5.near(y, yb, 1) for y in meta[s]]), R_base=a["R"], R_ipw=b["R"], k_base=a["k"], k_ipw=b["k"]))
#     CP = pd.DataFrame(rows); CP.to_csv(PA/"PA_B7c_changepoints.csv", index=False, encoding="utf-8-sig")
#     v1 = "유지" if sg >= 0.90 else ("부분" if sg >= 0.70 else "민감"); v2 = "둔감" if same >= 0.95 else "민감"
#     print("\n[Post-hoc] PA-B7c 요약\n", out.round(4).T.to_string(header=False)); print(f"\n[동결 판독] 부호 {sg:.3f} → {v1} | 라벨 {same:.3f} → {v2}")
#     print("\n[Post-hoc] 변화점(가중 전후, 판정 없음)\n", CP.round(4).to_string(index=False))
#     print("[부호 유지 실패 big]\n", D[ob & ~kept_sign].round(4).to_string() if (ob & ~kept_sign).any() else "없음")
#     print("※ 셀 사후층화(S4 IPW 모델과 다름). 가중으로 비용/벌점 비율이 달라짐. 세 판독을 합치지 않음. S0~S8 판정 불변.")

# run_pa("B7c", PLAN, run, [S4/"topic_quadrants.csv", S4/"yearly_venue_topic_counts.npy", P1/"s5_lib.py", S5D/"S5_0_PREREG.json", S5D/"S5_meta.json", S5D/"regime_nulls.csv"], "저널x연도 상호작용 보정(셀 사후층화)으로 S4 라벨·S5 변화점 재점검(지침서 B7-3)")
# ```

# ## 4. B9b: 공급 정의·질량 하한 민감도 (`pa_b9b.py`)

# B9a와 같은 편 순위상관을 공급 스킴 3종, 변형 2종(main, rec), 질량 하한 3종(5, 10, 20)으로 반복합니다. 하한은 항상 같은 변형의 `inv_n_authors` 총질량 기준입니다.

# ```python
# # -*- coding: utf-8 -*-
# # PA-B9b: S8 국가 수준 편 순위상관(통과 도메인)의 공급 정의·질량 하한 민감도. 미실행. 소득군별 n을 먼저 보고.
# from pa_common import *
# G4 = ["HIC", "UMC", "LMC", "LIC"]; SCH = ["inv_n_authors", "full_count", "equal"]; VAR = ["main", "rec"]; FLOORS = [5, 10, 20]
# PLAN = dict(
#     version=1, seed=2026, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(지침서 B9-3). B9a 결과, B9b 설계 점검(LIC는 하한 5에서도 1개국이라 자료상 불가, LMC n 12~13, UMC n 20~22), country_mass_long의 스킴 구조 열람 상태. 결과 미열람. 이전에 생략 권고했으나 연구자 요청으로 수행. S0~S8 판정 불변.",
#     grid=dict(scheme=SCH, variant=VAR, floor=FLOORS, strata=["ALL"] + G4, control="rank(gdp_pc) 통제 편 순위상관(B9a의 gdp와 동일)"),
#     definitions="공급 강도 s = 해당 스킴·변형의 (도메인 논문 질량/국가 총 질량). 국가 포함 기준 = 같은 변형의 inv_n_authors 총질량 >= floor. 도메인·지표쌍은 S8_0_PREREG_v2.json의 domain_indicators 중 통과 도메인(domains_passed.json). 부담 지표·gdp_pc·소득군은 burden_country_table(wb_income_current 고정). full_count는 논문당 w 합이 국가 수라 총질량 분모도 같은 스킴",
#     n_min=20, B=1000, ci=[2.5, 97.5], boot="국가 복원추출, 재표집마다 순위·잔차 재계산, 단일 rng(seed 고정) 순차",
#     reading="소득군별 n 표를 먼저 보고. 행(스킴x변형x하한x쌍x층)별로 n<20이면 NA, 아니면 부호가 기준행((ALL, inv_n_authors, main, floor 10)의 같은 쌍)과 같고 구간이 0을 배제하면 '관찰', 아니면 '비관찰'. 행을 합치지 않음. 기준 구간이 0을 배제하는 쌍은 B9a에서 safety-road_deaths 하나뿐이므로 나머지 쌍의 관찰 여부는 해석하지 않음",
#     reproduce_check="기준행((ALL, inv_n_authors, main, floor 10))의 n과 편 rho가 country_partial_spearman.csv(excl=none)와 1e-6 이내. 불일치 시 중단",
#     caveat="LIC는 하한과 무관하게 n<20(자료상 불가), LMC도 대부분 n<20이므로 소득군 내 추정이 가능한 칸은 HIC와 하한 5의 UMC 정도. 하한 5는 질량이 작은 국가(공급 강도 잡음 큼)를 포함. 변형 rec가 모든 스킴에 존재하지 않으면 해당 칸은 NA로 기록. 다중성 보정 없음, 탐색적. 국가 단위 상관은 생태학적 오류 가능, 인과·필요 충족 증거 아님",
#     forbidden="결과를 본 뒤 격자·n_min·판독 변경(변경 시 v2 계획으로 분리하고 사유 기록)")

# def run(quick, sp):
#     SPEC = json.loads((S8/"S8_0_PREREG_v2.json").read_text(encoding="utf-8")); DOM_IND = SPEC["domain_indicators"]
#     dp = json.loads((S8/"domains_passed.json").read_text(encoding="utf-8"))["passed"]
#     Bt = burden(); fl = flags(SPEC)[["pid"] + dp]; Mr = mass_raw(); cache = {}
#     def prep(sc, vr):
#         if (sc, vr) not in cache:
#             A = Mr[(Mr.scheme == sc) & (Mr.variant == vr)]
#             cache[(sc, vr)] = None if A.empty else (A.groupby("country").w.sum(), A.merge(fl, on="pid", how="inner"))
#         return cache[(sc, vr)]
#     def build(sc, vr, floor):
#         a, b = prep(sc, vr), prep("inv_n_authors", vr)
#         if a is None or b is None: return None
#         tot = a[0].reindex(b[0][b[0] >= floor].index).dropna(); tot = tot[tot > 0]; out = {}
#         for d in dp:
#             sup = a[1][a[1][d]].groupby("country").w.sum().reindex(tot.index).fillna(0)/tot
#             for ind in DOM_IND[d]:
#                 out[(d, ind)] = pd.DataFrame({"s": sup, "b": Bt[INT[ind]].reindex(tot.index), "z": Bt["gdp_pc"].reindex(tot.index), "inc": Bt["income"].reindex(tot.index)}).dropna(subset=["s", "b", "z"])
#         return out
#     ref = pd.read_csv(S8/"country_partial_spearman.csv", encoding="utf-8-sig"); ref = ref[ref.excl == "none"].set_index(["domain", "indicator"])
#     base = build("inv_n_authors", "main", 10); base_rho = {}
#     for k, df in base.items():
#         r = pr(df.s.values, df.b.values, [df.z.values], [True]); r0 = ref.loc[k]; base_rho[k] = r
#         if len(df) != int(r0.n) or abs(r - float(r0.partial_rho)) > 1e-6: raise Halt(f"기준행 재현 실패{k}: n {len(df)} vs {int(r0.n)}, rho {r:.6f} vs {r0.partial_rho:.6f}")
#     print(f"[재현 OK] 기준행 {len(base)}쌍이 country_partial_spearman.csv(none)와 일치")
#     B = 0 if quick else int(PLAN["B"]); rng = np.random.default_rng(int(PLAN["seed"])); rows = []
#     for sc in SCH:
#         for vr in VAR:
#             for fl_ in FLOORS:
#                 bd = build(sc, vr, fl_)
#                 for k in base:
#                     for st in ["ALL"] + G4:
#                         row = dict(scheme=sc, variant=vr, floor=fl_, domain=k[0], indicator=k[1], stratum=st, B=B)
#                         if bd is None: rows.append(dict(row, n=0, rho=np.nan, lo=np.nan, hi=np.nan, note="스킴·변형 없음")); continue
#                         df = bd[k]; sub = df if st == "ALL" else df[df.inc == st]; n = len(sub)
#                         if n < PLAN["n_min"] or quick: rows.append(dict(row, n=n, rho=np.nan, lo=np.nan, hi=np.nan, note="")); continue
#                         x, y, z = sub.s.values, sub.b.values, sub.z.values; r = pr(x, y, [z], [True]); bs = []
#                         for _ in range(B):
#                             i = rng.integers(0, n, n); bs.append(pr(x[i], y[i], [z[i]], [True]))
#                         rows.append(dict(row, n=n, rho=r, lo=float(np.nanpercentile(bs, 2.5)), hi=float(np.nanpercentile(bs, 97.5)), note=""))
#     R = pd.DataFrame(rows); sfx = "_QUICK" if quick else ""; NT = R.pivot_table(index=["scheme", "variant", "floor", "domain", "indicator"], columns="stratum", values="n", aggfunc="first")[["ALL"] + G4]
#     NT.to_csv(PA/f"PA_B9b_n_table{sfx}.csv", encoding="utf-8-sig"); print("\n[층별 n(설계 정보)]\n", NT.astype(int).to_string())
#     if quick: print("\n[QUICK] 재현 점검과 n 표만. rho는 계산하지 않음."); return
#     R["base_sign"] = [np.sign(base_rho[(a, b)]) for a, b in zip(R.domain, R.indicator)]; R["verdict"] = np.where(R.rho.isna(), "NA", np.where((np.sign(R.rho) == R.base_sign) & ((R.lo > 0) | (R.hi < 0)), "관찰", "비관찰"))
#     R.to_csv(PA/"PA_B9b_results.csv", index=False, encoding="utf-8-sig")
#     for st in ["ALL", "HIC", "UMC"]:
#         print(f"\n[Post-hoc] PA-B9b {st} (rho / 구간 / verdict)\n", R[(R.stratum == st) & R.rho.notna()].round(3)[["scheme", "variant", "floor", "domain", "indicator", "n", "rho", "lo", "hi", "verdict"]].to_string(index=False))
#     print("\n[동결 판독] verdict 집계\n", R.groupby(["stratum", "verdict"]).size().unstack(fill_value=0).to_string())
#     print("※ LIC·LMC는 대부분 NA(자료상 불가). 기준 구간이 0을 배제하는 쌍(safety-road_deaths)만 해석. 48행 이상 탐색적. S0~S8 판정 불변.")

# run_pa("B9b", PLAN, run, [S8/"S8_0_PREREG_v2.json", S8/"domains_passed.json", S8/"country_partial_spearman.csv", S8/"burden_country_table.csv", PA/"PA_B9a_PLAN.json"], "공급 정의(스킴)·변형·질량 하한 민감도(지침서 B9-3)")
# ```

# ## 5. B4c: 임베딩 응집도·Jaccard 10퍼센타일 등급 (`pa_b4c.py`)

# **먼저 `discover`를 실행하세요.** 이 모드는 읽기 전용이고 동결이 필요 없습니다.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1 && python3 pa_b4c.py discover
# ```

# 출력에서 임베딩 행렬(`.npy`), 그 행과 대응하는 `pid` 파일, (있다면) 반복별 Jaccard 파일을 찾습니다. 그 경로를 아래 코드 맨 위 상수에 채운 뒤에 `freeze`를 하세요. 상수도 계획 해시에 포함됩니다.

# ```python
# # -*- coding: utf-8 -*-
# # PA-B4c: 임베딩 응집도(토픽 안 평균 코사인)와 부트스트랩 Jaccard 10퍼센타일을 '더 변별적인 등급'으로 삼아 하위 10% 토픽을 제외하고 S5·S6 재추정(B4와 같은 틀). 미실행.
# # 사용: python3 pa_b4c.py discover | freeze | tag | run [quick]
# from pa_common import *
# # ---- discover 출력으로 채울 것(동결 전). 비워 두면 freeze가 중단됨 ----
# EMB_PATH = ""      # 예: "analysis_data/final/xxx_embeddings.npy" (ROOT 기준 상대경로, 행=논문)
# PID_PATH = ""      # 위 행렬의 행 순서에 대응하는 pid 파일(.csv/.parquet), 열 이름은 PID_COL
# PID_COL = "pid"
# JAC_PATH = ""      # 선택: 토픽별 부트스트랩 반복 Jaccard 긴 형식(.csv, 열 topic, jaccard). 없으면 빈 문자열(Jaccard 부분 미수행 기록)

# def discover():
#     cand = [p for p in ROOT.rglob("*") if p.is_file() and ".git" not in p.parts and "oa_raw" not in p.parts and ".ipynb_checkpoints" not in p.parts and re.search(r"emb|vector|jacc|boot", p.name, re.I) and p.suffix in (".npy", ".npz", ".csv", ".parquet", ".gz", ".json", ".md")]
#     for p in sorted(cand)[:60]:
#         info = ""
#         try:
#             if p.suffix == ".npy": info = f"shape {np.load(p, mmap_mode='r').shape}"
#             elif p.suffix == ".csv": d = pd.read_csv(p, nrows=3); info = f"cols {list(d.columns)[:8]}"
#         except Exception as e: info = f"읽기 실패 {str(e)[:40]}"
#         print(f"{p.relative_to(ROOT)} | {p.stat().st_size/1e6:.1f}MB | {info}")
#     print("\n[S3 파일 열]"); [print(f.name, list(pd.read_csv(f, nrows=1).columns)[:12]) for f in sorted(S3.glob("L*.csv"))]
#     print("※ 임베딩 행렬·pid 파일·(있다면) 반복별 Jaccard 파일 경로를 위 상수에 채우세요.")
# if len(sys.argv) > 1 and sys.argv[1] == "discover": discover(); sys.exit()

# PLAN = dict(
#     version=1, seed=2026, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(지침서 B4-2). B4·B4b 결과(F1·F3 제외 4개뿐이라 약한 시험), S3 요약 열람 상태. 결과 미열람. 응집도 파일 경로는 discover 출력으로 확정. 이전에 미수행으로 두었으나 연구자 요청으로 수행. S0~S8 판정 불변.",
#     inputs=dict(emb=EMB_PATH, pid=PID_PATH, pid_col=PID_COL, jac=JAC_PATH),
#     cohesion="토픽 t의 S4 분석 집합 논문 중 임베딩이 있는 논문(토픽당 최대 2000편 무작위 표집, seed 고정, 최소 5편)을 L2 정규화해 평균 쌍별 코사인=(||sum u||^2-n)/(n(n-1)). 하위 10%(올림) 토픽을 '저응집'으로 정의",
#     jaccard_p10="JAC_PATH가 있으면 토픽별 반복 Jaccard의 10퍼센타일을 구해 하위 10% 토픽을 '불안정'으로 정의. 없으면 이 부분은 미수행으로 기록",
#     reestimate="B4(pa_b4.py)와 같은 절차: S5는 통과 토픽만으로 s5_lib.detect(pen=c*phi*(K-1)*log T, K=통과 토픽 수), N1·N2 귀무 B=500, 통과 규칙 N1·N2 모두 p<0.05. S6은 R을 통과 토픽 행으로 제한한 주 설정, 대응 저자 부트스트랩 B=spec['B_boot'], entry_minus_adapt 변화",
#     reading="S5: 표본별 (a) 통과 여부가 S5와 같은지, (b) S5 관측 변화점이 +-1년 안에서 유지되는지. S6: 쌍별 |변화|<=0.03이고 부호 유지면 '둔감'(B4와 같은 규칙, 첫 쌍은 원값이 0 근처라 부호가 바뀌기 쉬움). 제외 집합별로 따로 보고, 합치지 않음. 제외 토픽과 F1·F3 실패 4개·S3 등급의 겹침도 보고",
#     reproduce_check="(a) 임베딩 행 수와 pid 파일 길이 일치, (b) 분석 집합 논문 중 임베딩이 있는 비율 >=0.95, (c) S5 관측 R이 regime_nulls.csv obs_R과 1e-3 이내, (d) S6 total_half·A2c entry_share 재현, 부분 자료 저자 집합이 원본의 부분집합",
#     caveat="응집도는 임베딩 모델과 토픽 구성(대형 토픽의 평균 코사인이 낮아지는 경향)에 의존. 하위 10%는 임의 기준(동결). 등급의 타당성은 검증하지 않음. 제외 토픽이 약 15개라 B4보다 판별력이 크지만 여전히 부분집합 재추정",
#     forbidden="결과를 본 뒤 하위 비율(10%)·표집 수·판독 변경(변경 시 v2 계획으로 분리하고 사유 기록)")

# def reestimate(ptop, quick, rng):
#     import s5_lib as L5, s6_lib as S
#     s5 = json.loads((S5D/"S5_0_PREREG.json").read_text(encoding="utf-8")); D = s5["detection"]; PHI = {k: v["phi"] for k, v in D["phi"].items()}
#     C0, KMAX, MINLEN, ALPHA = D["c_primary"], D["k_max"], D["min_len"], s5["nulls"]["alpha"]; Bn = 20 if quick else 500
#     cnt, venues, topics, fixed_v = L5.load_counts(); wbar = cnt.sum((0, 2))/cnt.sum(); T = len(L5.YEARS); YEARS = np.asarray(L5.YEARS); SAM = s5["samples"]; tmask = np.isin(topics, list(ptop)); Ku = int(tmask.sum())
#     det = lambda c, s, Kuse: (lambda Pn: L5.detect(Pn[0], Pn[1], C0*PHI[s]*(Kuse - 1)*np.log(T), KMAX, MINLEN))(L5.sample_P(c, fixed_v, wbar, s))
#     ref = pd.read_csv(S5D/"regime_nulls.csv"); ref = ref[ref["null"] == "N2"].set_index("sample")
#     for s in SAM:
#         if abs(det(cnt, s, len(topics))["R"] - float(ref.loc[s, "obs_R"])) > 1e-3: raise Halt(f"S5 관측 R 재현 실패({s})")
#     cs = cnt[:, :, tmask]; obs = {s: det(cs, s, Ku) for s in SAM}; P2 = L5.trend_probs(cs); NR = {"N1": {s: [] for s in SAM}, "N2": {s: [] for s in SAM}}
#     for b in range(Bn):
#         for nm, cc in (("N1", L5.gen_N1(cs, rng)), ("N2", L5.gen_N2(cs, P2, rng))):
#             for s in SAM: NR[nm][s].append(det(cc, s, Ku)["R"])
#     meta = json.loads((S5D/"S5_meta.json").read_text(encoding="utf-8")); rows5 = []
#     for s in SAM:
#         yrs = [int(YEARS[x]) for x in obs[s]["br"]]; pv = {nm: (1 + float((np.array(NR[nm][s]) >= obs[s]["R"]).sum()))/(1 + Bn) for nm in NR}; kept = [L5.near(y, yrs, 1) for y in meta["obs_breaks"][s]]
#         rows5.append(dict(sample=s, K=Ku, obs_R=obs[s]["R"], breaks=str(yrs), s5_breaks=str(meta["obs_breaks"][s]), p_N1=pv["N1"], p_N2=pv["N2"], passed_new=max(pv.values()) < ALPHA, passed_s5=bool(meta["null_pass"][s]), all_breaks_kept=all(kept), B=Bn))
#     spec = S.load_spec(); L = S.load_all(spec); M = spec["main"]; PAIRS = S.pairs_of(spec); half = S.half; R = L["R"]
#     ok_ti = np.where(np.isin(L["topics"], list(ptop)))[0]; L2 = dict(L); L2["R"] = R[R.ti.isin(ok_ti)]
#     ref6 = pd.read_csv(S6D/"renewal_decomposition.csv"); ref6 = ref6[(ref6.scheme == M["scheme"]) & (ref6.variant == "all_venues")].set_index("pair")
#     a2c = pd.read_csv(PA/"PA_A2c_results.csv"); a2c = a2c[(a2c["mode"] == "author") & (a2c.stat == "entry_share")].set_index("pair")
#     B6 = 20 if quick else int(spec["B_boot"]); lo_q, hi_q = spec["ci"]; STATS = ["entry_share", "adapt_share", "entry_minus_adapt"]
#     def stats(d):
#         h = {t: half(d[t]) for t in ["adapt", "realloc", "entry", "exit"]}; tot = sum(h.values()); return dict(entry_share=h["entry"]/tot, adapt_share=h["adapt"]/tot, entry_minus_adapt=(h["entry"] - h["adapt"])/tot)
#     rows6 = []
#     for p0, p1 in PAIRS:
#         name = f"{p0}->{p1}"; Pa = S.make_pair(L, p0, p1, M["scheme"], M["venue"], M["topic"], M["hom"]); Pb = S.make_pair(L2, p0, p1, M["scheme"], M["venue"], M["topic"], M["hom"]); da, db = Pa.decompose(), Pb.decompose(); oa, ob = stats(da), stats(db)
#         if abs(half(da["total"]) - float(ref6.loc[name, "total_half"])) > 1e-6 or abs(oa["entry_share"] - float(a2c.loc[name, "obs"])) > 1e-6: raise Halt(f"S6/A2c 재현 실패({name})")
#         idx = np.searchsorted(Pa.aids, Pb.aids)
#         if not (idx < len(Pa.aids)).all() or not (Pa.aids[idx] == Pb.aids).all(): raise Halt(f"부분 자료 저자 집합이 원본의 부분집합이 아님({name})")
#         bs = []
#         for _ in range(B6):
#             wa = rng.exponential(size=Pa.nA); sa = stats(Pa.decompose(Pa.m0*wa[Pa.a0], Pa.m1*wa[Pa.a1])); wb = wa[idx]; sb = stats(Pb.decompose(Pb.m0*wb[Pb.a0], Pb.m1*wb[Pb.a1])); bs.append({s: sb[s] - sa[s] for s in STATS})
#         bs = pd.DataFrame(bs)
#         for s in STATS:
#             dl = ob[s] - oa[s]; ql, qh = np.percentile(bs[s], [lo_q, hi_q]); rows6.append(dict(pair=name, stat=s, obs_orig=oa[s], obs_sub=ob[s], delta=dl, pct_lo=ql, pct_hi=qh, B=B6))
#     return pd.DataFrame(rows5), pd.DataFrame(rows6)

# def run(quick, sp):
#     if not (EMB_PATH and PID_PATH): raise Halt("EMB_PATH·PID_PATH가 비어 있음(discover 출력으로 채운 뒤 새 동결)")
#     sys.path.insert(0, str(P1)); F, Mi, topics, venues = analysis_sets(); rng = np.random.default_rng(int(PLAN["seed"]))
#     E = np.load(ROOT/EMB_PATH, mmap_mode="r"); pp = ROOT/PID_PATH; pid = (pd.read_parquet(pp) if pp.suffix == ".parquet" else pd.read_csv(pp, dtype=str))[PID_COL].astype(str).values
#     if len(pid) != E.shape[0]: raise Halt(f"임베딩 행 {E.shape[0]} != pid 길이 {len(pid)}")
#     pos = {p: i for i, p in enumerate(pid)}; F["ei"] = F.pid.map(pos); cov = float(F.ei.notna().mean())
#     if cov < 0.95: raise Halt(f"분석 집합 중 임베딩 보유 비율 {cov:.3f} < 0.95")
#     coh = {}
#     for t in topics:
#         ix = F.loc[F.topic == t, "ei"].dropna().astype(int).values
#         if len(ix) < 5: coh[int(t)] = np.nan; continue
#         if len(ix) > 2000: ix = rng.choice(ix, 2000, replace=False)
#         X = np.asarray(E[np.sort(ix)], float); X /= np.linalg.norm(X, axis=1, keepdims=True) + 1e-12; n = len(X); coh[int(t)] = float((np.linalg.norm(X.sum(0))**2 - n)/(n*(n - 1)))
#     C = pd.Series(coh); k10 = int(np.ceil(0.10*C.notna().sum())); low_c = set(C.dropna().nsmallest(k10).index.astype(int))
#     G = pd.read_csv(S3/"L7_topic_grade.csv", encoding="utf-8-sig").set_index("topic"); f13 = set(G.index[bl(G.F1) | bl(G.F3)].astype(int))
#     sets = {"cohesion_low10": low_c}
#     if JAC_PATH:
#         J = pd.read_csv(ROOT/JAC_PATH); p10 = J.groupby("topic").jaccard.quantile(0.10); sets["jaccard_p10_low10"] = set(p10.nsmallest(int(np.ceil(0.10*len(p10)))).index.astype(int))
#     print(f"[재현 OK] 임베딩 보유 비율 {cov:.3f} | 응집도 하위 10% {len(low_c)}개, F1·F3 실패 4개와 겹침 {len(low_c & f13)}개 | Jaccard p10 {'있음' if JAC_PATH else '파일 없음(미수행)'}")
#     if quick:
#         for nm, ex in sets.items(): reestimate(set(map(int, topics)) - ex, True, rng); print(f"  {nm} 점검 완료")
#         print("[QUICK] 오류 점검만. 결과 수치는 출력하지 않음."); return
#     pd.DataFrame(dict(topic=C.index, cohesion=C.values, low10=C.index.isin(low_c), grade=[G.grade.get(t, "") for t in C.index], f1f3_fail=C.index.isin(f13))).to_csv(PA/"PA_B4c_cohesion.csv", index=False, encoding="utf-8-sig")
#     for nm, ex in sets.items():
#         R5, R6 = reestimate(set(map(int, topics)) - ex, False, rng); R5.to_csv(PA/f"PA_B4c_{nm}_s5.csv", index=False, encoding="utf-8-sig"); R6.to_csv(PA/f"PA_B4c_{nm}_s6.csv", index=False, encoding="utf-8-sig")
#         k = R6[R6.stat == "entry_minus_adapt"].copy(); k["insensitive"] = (np.sign(k.obs_sub) == np.sign(k.obs_orig)) & (k.delta.abs() <= 0.03)
#         print(f"\n[Post-hoc] PA-B4c {nm}: 제외 {len(ex)}개 {sorted(ex)}\n[S5]\n", R5.round(4).to_string(index=False), "\n[S6 entry_minus_adapt]\n", k.round(4).to_string(index=False))
#         print("[동결 판독] S5 통과 판정 동일·변화점 유지:", R5[["sample"]].assign(same=R5.passed_new == R5.passed_s5, kept=R5.all_breaks_kept).to_dict("records"))
#     if not JAC_PATH: print("※ Jaccard 반복값 파일이 없어 p10 등급은 미수행(S3 저장값은 중앙값뿐).")
#     print("※ 응집도 기준은 임베딩 모델·토픽 크기에 의존. 제외 집합별로 따로 읽고 합치지 않음. S0~S8 판정 불변.")

# def _freeze_guard():
#     if len(sys.argv) > 1 and sys.argv[1] == "freeze" and not (EMB_PATH and PID_PATH): print("⛔ 중단: EMB_PATH·PID_PATH를 먼저 채우세요(discover 출력 확인)"); sys.exit()
# _freeze_guard()
# run_pa("B4c", PLAN, run, [S3/"L7_topic_grade.csv", S5D/"S5_0_PREREG.json", S5D/"S5_meta.json", PA/"PA_B4_PLAN.json"], "더 변별적인 토픽 등급(응집도·Jaccard p10)으로 S5·S6 재추정(지침서 B4-2)")
# ```

# `pa_b4.py`가 `pa_common.py`가 아닌 독립 스크립트였으므로, 위 `reestimate`는 그 로직을 옮긴 것입니다. 재현 점검 중 하나라도 어긋나면 중단합니다. S5 귀무가 B=500이라 본 실행은 B4처럼 오래 걸릴 수 있으니 `nohup`으로 돌리세요.

# ## 6. A4: 열람·동결 이력 부록 표 확정 (`pa_a4.py`)

# 이 스크립트는 읽기 전용입니다. 계획 파일, git 태그, changelog를 합쳐 항목별 표를 만듭니다. 동결·태그는 필요 없고 결과는 `09_PA/PA_A4_audit_table.csv`와 `.md`로 남습니다.

# ```python
# # -*- coding: utf-8 -*-
# # PA-A4: 항목별 동결·열람·정정 이력 표(부록용) 생성. 읽기 전용(분석 아님). 미실행. python3 pa_a4.py
# from pa_common import *
# import csv
# MANUAL = {   # 지금까지 대화에서 확인된 사실(자동 추출이 못 잡는 것). 필요하면 고치세요
#     "0": "10-21 12:31:17 'oa_raw 외부 복사 완료' 기록은 12:31:33에 정정(복사 실패, 외부 백업 미완료). 잘못된 S4 태그 2개 삭제 기록. 정리 커밋 날짜는 생성일이 아님",
#     "A1": "run 1은 inject() NaN 오류로 무효(삭제하지 않고 커밋), run 2가 동결 사양. 코드 패치 2건은 사양 불변. 탐색 확장(δ 0.25/0.35/0.45, 연속 꺾은선 BIC)은 동결 밖·판독 미사용. v2 동결 전 s5_lib 소스·N2 요약 열람",
#     "B2": "동결은 노트북 셀에서 실행(18:18), 스크립트는 사후 저장(해시 일치 확인). QUICK 출력에 관측 기술 표·요약이 포함되어 열람",
#     "B7a": "첫 시도 코드 오류(람다 언패킹, 사양 불변)로 중단·수정. 기간별 결측률 열람",
#     "B3a": "실행 2회(실수, 결정적이라 결과 동일). 정의 확인 기록 시각은 실행 이후이나 계획에 정의 열람 명시",
#     "B8a": "붙여넣기 중 화면이 섞여 보였으나 계획 JSON·파일 정상 확인(grep -c 기준 오류는 설명 오류였음)",
#     "B7b": "실행 로그 파일 없음(결과 CSV와 changelog 수치로 대체)",
# }
# NOT_DONE = {   # 아직 계획 파일이 없는 항목의 사유(실행하면 표에서 '완료'로 바뀜)
#     "B6": "연구자 수작업(ORCID·50쌍) 필요, 인력 미정", "B8-2": "사전 v3 작성·새 25편 이중 코딩 필요, 인력 미정",
# }
# def norm(s): return re.sub(r"\(v2\)|-v2$", "", s.replace("PA-", "")).strip()
# plans = {}
# for f in sorted(PA.glob("PA_*_PLAN*.json")):
#     m = re.match(r"PA_(.+?)_PLAN(_v2)?\.json$", f.name)
#     if m: j = json.loads(f.read_text(encoding="utf-8")); plans.setdefault(m.group(1), []).append(dict(file=f.name, hash=j.get("sha256", "")[:12], created=j.get("created", ""), parent=j.get("parent", j.get("supersedes_sha256", ""))))
# tags = {}; remote = set(l.split("refs/tags/")[1].replace("^{}", "") for l in git("ls-remote", "--tags", "origin").stdout.splitlines() if "refs/tags/" in l)
# for l in git("for-each-ref", "--format=%(refname:short)|%(creatordate:iso-strict)", "refs/tags").stdout.splitlines():
#     n, t = l.split("|"); m = re.match(r"^PA-(.+?)-(plan-v2|plan|code|fullrun|results|explore-code|explore-results|run\d+-[a-z]+|repo-cleanup)-\d{4}-\d{2}-\d{2}$", n)
#     if m: tags.setdefault(norm(m.group(1)), []).append((m.group(2), n, t, n in remote))
# logs = {}
# for ln in CL.read_text(encoding="utf-8").splitlines():
#     if not re.match(r"^- \d{4}-\d{2}-\d{2}T", ln): continue
#     parts = [p.strip() for p in ln.split(" | ")]; body = " | ".join(parts[1:]); m = re.match(r"(PA-[A-Za-z0-9]+)", body); it = norm(m.group(1)) if m else "기타"
#     logs.setdefault(it, []).append((parts[0][2:], body))
# items = sorted(set(plans) | set(tags) | set(logs) | set(NOT_DONE) - {"기타"}, key=lambda x: (x != "0", x))
# rows = []
# for it in items:
#     pl = plans.get(it, []); tg = tags.get(it, []); lg = logs.get(it, [])
#     ptag = next((t for t in tg if t[0].startswith("plan")), None); rtag = [t for t in tg if t[0] == "results"]
#     viewed = " // ".join(b[:110] for _, b in lg if "열람" in b and "미열람" not in b[:8])[:380]; fixes = ";".join(t for t, b in lg if re.search(r"정정|코드 오류|패치|중복", b))
#     rows.append(dict(item=it, status=("계획 없음: " + NOT_DONE[it]) if it in NOT_DONE and not pl else ("완료" if rtag else ("진행/기록만" if pl else "기록만")), plan_hash=";".join(p["hash"] for p in pl), plan_created=pl[0]["created"] if pl else "",
#                      plan_tag=ptag[1] if ptag else "", plan_tag_time=ptag[2] if ptag else "", results_tag=";".join(t[1] for t in rtag), results_tag_time=";".join(t[2] for t in rtag), tags_on_origin=all(t[3] for t in tg) if tg else "",
#                      n_changelog_lines=len(lg), viewed_notes=viewed, correction_times=fixes, manual_note=MANUAL.get(it, "")))
# T = pd.DataFrame(rows); T.to_csv(PA/"PA_A4_audit_table.csv", index=False, encoding="utf-8-sig")
# with (PA/"PA_A4_audit_table.md").open("w", encoding="utf-8") as f:
#     f.write("# [Post-hoc] 동결·열람·정정 이력 (PA_CHANGELOG.md, git 태그, 계획 JSON에서 생성)\n\n| 항목 | 상태 | 계획 해시 | 계획 태그 시각 | 결과 태그 시각 | 원격 | 열람·정정·비고 |\n|---|---|---|---|---|---|---|\n")
#     for r in rows: f.write(f"| {r['item']} | {r['status']} | {r['plan_hash']} | {r['plan_tag_time'][:19]} | {r['results_tag_time'][:19]} | {r['tags_on_origin']} | {(r['viewed_notes'] + ' ' + r['manual_note']).strip()[:300]} |\n")
#     f.write("\n기록의 오류(코드 오류, 백업 완료 오기재, 해석 메모 중복)는 삭제하지 않고 정정 줄로 남겼다. S0~S8의 판정은 어떤 항목에서도 변경하지 않았다.\n")
# bad = [(t, b[:80]) for ls in logs.values() for t, b in ls if "본 결과 변경" in b and not re.search(r"본 결과 변경[^:]*:\s*없음", b)]
# print(len(rows), "항목 →", PA/"PA_A4_audit_table.csv", "/ .md"); print(T[["item", "status", "plan_hash", "n_changelog_lines", "tags_on_origin"]].to_string(index=False))
# print("\n'본 결과 변경: 없음'이 아닌 changelog 줄:", bad if bad else "없음(모든 줄 '없음')")
# ```

# ## 실행 절차

# 한 블록씩 실행하고, `⛔`나 오류가 나면 거기서 멈추고 출력만 붙여 주세요. 예시는 B3c이고, 나머지 스크립트도 이름만 바꾸면 같습니다.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# # pa_common.py, pa_b3c.py 저장 후
# python3 -m py_compile pa_common.py pa_b3c.py && echo "문법 OK"
# python3 pa_b3c.py freeze
# python3 pa_b3c.py tag && cd .. && git push --follow-tags && cd paper1
# python3 pa_b3c.py run quick 2>&1 | tee 09_PA/PA_B3c_quick_log.txt
# nohup python3 -u pa_b3c.py run > 09_PA/PA_B3c_full_log.txt 2>&1 &
# ```

# - 스크립트는 모드를 인자로 받으므로 파일을 `sed`로 고치지 않습니다. 동결 시점의 `pa_common.py`와 스크립트 해시가 바뀌면 `run`이 중단됩니다. 오류 수정이 필요하면 v2 계획으로 분리하고 사유를 기록해야 합니다.
# - QUICK의 출력(재현 점검, 집합 크기, n 표)은 설계 정보입니다. 다만 열람 사실이므로 changelog에 한 줄 남기는 편이 맞습니다.
# - freeze가 changelog에 "이전에 생략을 권했던 항목은 연구자 요청으로 재개(침묵을 동의로 본 처리를 정정)"를 자동 기록합니다. 결과를 보기 전에 이 정정이 남습니다.

# ## 가정과 주의

# 아래는 확인하지 못해서 오류가 날 수 있는 지점입니다.

# | 스크립트 | 가장 가능성 높은 중단 원인 |
# |---|---|
# | B3c | `Pair.decompose()`의 키(`entry`, `entry_new`, `adapt`, `realloc`)는 소스 열람 내용을 따랐고, 토픽 축이 `L["topics"]`라는 점은 B4에서 쓴 방식을 따랐습니다. 집합 토픽이 S6 토픽 설정(`topic`)에 없으면 크기가 줄고 그 수가 출력됩니다. |
# | B3b | 재계산 D3가 저장 D3와 다르면 중단합니다. 그때는 D3의 실제 정의(`s4_main.py`)를 봐야 합니다. D2의 평활 방식(`eps=1e-4`)은 제 가정이라 정보용 재현으로만 씁니다. |
# | B7c | `topics_order.csv`가 정렬 목록과 다르면 중단합니다. 가중은 S4 IPW 모델이 아닙니다. |
# | B9b | `rec` 변형이 모든 스킴에 없으면 그 칸은 NA로 기록됩니다. LIC는 하한과 무관하게 n<20이라 자료상 불가입니다. |
# | B4c | 임베딩 경로를 모릅니다. Jaccard 반복값이 저장돼 있지 않으면 p10 부분은 "미수행"으로 기록됩니다. S3에는 중앙값만 있습니다. |
# | A4 | 열람 여부는 키워드 기준입니다. 부록에 쓰기 전에 직접 훑어보세요. `MANUAL` 딕셔너리는 제가 대화에서 확인한 사실을 옮긴 것이니 틀리면 고치세요. |

# B3c와 B4c의 해석은 미리 단정하지 않겠습니다. B3a에서 보았듯 핵심 집합은 "변화량 상위" 정의의 부산물일 수 있어서, 결과가 나오면 그 점부터 확인하겠습니다.

# 가장 가벼운 시작은 `pa_common.py`와 `pa_b3c.py` 저장 후 `py_compile` 결과를 붙여 주시는 것입니다.

In [ ]:
# # 진행 상태 점검

# ## 1. B4c `discover` 결과: 임베딩 파일 없음

# - 후보로 잡힌 5개는 전부 **부트스트랩 결과 CSV**(S5, S6, S7)입니다. 임베딩 행렬(`.npy`)과 대응하는 pid 파일은 `ROOT` 아래에서 이름 패턴(`emb|vector|jacc|boot`)으로 찾지 못했습니다.
# - **S3의 `L3_topic_metrics.csv`에 `jaccard_p10` 열이 이미 있습니다.** 부트스트랩 Jaccard의 10퍼센타일이 저장돼 있다는 뜻입니다. 지침서 B4-2의 둘 중 하나는 새 파일 없이 바로 쓸 수 있습니다. 앞서 "S3에는 중앙값만 있다"고 한 제 말은 틀렸습니다(`jaccard_median`만 보고 한 말이었습니다). 정정해 둡니다.
# - 임베딩은 이름이 다를 수 있습니다(예: `.npz`, 외부 폴더, `analysis_data` 밖). 아래로 한 번 더 찾으세요.

# ```bash
# cd ~/Research/Minnesota_Transport
# find / -xdev \( -name "*.npy" -o -name "*.npz" -o -name "*.pt" -o -name "*.parquet" \) -size +20M 2>/dev/null | grep -v -E "\.git/|oa_raw|site-packages" | head -20
# grep -rn -i "np.save\|to_parquet\|encode(" --include=*.py --include=*.ipynb . 2>/dev/null | grep -v ipynb_checkpoints | cut -c1-160 | head -15
# ```

# 찾으면 경로를 알려 주세요. 못 찾으면 B4c는 **Jaccard p10 부분만** 수행하는 안으로 바꾸겠습니다. 응집도는 "임베딩 파일 위치 미확인"으로 미수행 기록합니다.

# ## 2. B3c: 정상 진행 중

# - 동결 해시 `2b463ef196be`, 커밋 `9dd4479`, 태그 `PA-B3c-plan-2026-10-21`이 푸시됐습니다. 재현 점검은 통과했고 집합 크기는 core 7, mid 10, specific 14, union 31로 모두 S6 토픽 축에 존재합니다.
# - **한 가지 문제가 있습니다.** 아래 흐름 때문에 `run quick`과 본 실행(`nohup`)이 둘 다 돌았습니다. 본 실행이 PID 2689583으로 백그라운드에서 돌고 있는 것은 의도대로입니다(QUICK은 이미 끝남). 다만 **본 실행 전에 `fullrun` 태그를 남기지 않았고, QUICK 열람 기록도 아직 없습니다.** 이 스크립트는 `MODE`를 인자로 받아 코드 변경이 없으므로 fullrun 태그는 필수가 아닙니다. 열람 줄만 추가하세요.
# - `git add`에서 `pape`로 잘려 보이는 줄은 출력이 잘린 표시일 가능성이 큽니다(4 files changed로 정상). 아래 `git show --stat`으로 확인하세요.

# ```bash
# cd ~/Research/Minnesota_Transport
# git show --stat --format=%h 9dd4479 | tail -6
# cd paper1; pgrep -f pa_b3c.py || echo "종료 확인"; tail -n 3 09_PA/PA_B3c_full_log.txt | cut -c1-120
# ```

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B3c QUICK 열람: 재현 점검 통과, 집합 크기(core 7, mid 10, specific 14, union31 31, 모두 S6 토픽 축에 존재) 열람. 결과 통계 미열람. 본 실행은 QUICK 직후 백그라운드로 시작(fullrun 태그 없음, 코드 변경 없음). B4c discover 결과: 임베딩 파일 미발견, S3 L3_topic_metrics.csv에 jaccard_p10 열 존재(앞선 '중앙값만 있다'는 설명 정정) | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_CHANGELOG.md paper1/09_PA/PA_B3c_quick_log.txt paper1/09_PA/PA_B3c_results_QUICK.csv && git commit -m "PA-B3c: QUICK note" && git push
# ```

# ## 3. 다음

# - B3c 본 실행 로그(끝나면 전체)를 붙여 주세요. 동결 규칙대로 읽겠습니다.
# - 그동안 CPU를 많이 쓰는 다른 스크립트는 돌리지 마세요. B3b, B7c처럼 가벼운 것은 B3c 종료 후 순서대로 진행하면 됩니다.
# - 임베딩 검색 결과를 알려 주세요.

In [ ]:
# # 논문 1 사후분석(PA) 종합 정리 (2026-10-21 기준)

# ## 1. 하려고 했던 분석

# 지침서(2026-10-19)의 14개 항목입니다. 핵심 3개는 A1, A2, B2로 정했습니다.

# | 구분 | 항목 | 질문 |
# |---|---|---|
# | 재현성 | PA-0 | 파일, 코드, 태그가 온전한가 |
# | 강점 강화 | A1 | N2 비통과는 검정력 부족인가, 점프 부재인가 |
# | | A2 | 신규 진입 우세는 정의·부트스트랩·이름 식별에 견고한가 |
# | | A3 | 선두 국가 교체 서술이 견고한가 |
# | 약점 리프레이밍 | B2 | 2020 변화점이 표본 크기·저널 구성의 아티팩트인가 |
# | | B3 | 신생 토픽 정의 4종의 핵심 집합은 무엇인가 |
# | | B4 | 등급 선택이 결과를 좌우하는가 |
# | | B5 | S4 안정성 0.700은 경계인가 |
# | | B6 | 위험군에 독립 정답이 있는가 (ORCID, 수작업 50쌍) |
# | | B7 | 임베딩 결측 16%가 결과를 뒤집는가 |
# | | B8 | 미달 4개 도메인의 정렬 결과는 어떤가 |
# | | B9 | SBR 간극이 소득 수준과 분리되는가 |
# | 정합성 | C1 | 진입 우세가 중국 소속 유입과 독립인가 |
# | | C2 | S7(전체 논문)과 S6(임베딩 보유 논문)의 모집단 차이 |

# ## 2. 지금까지의 결과

# 공통 원칙은 이렇습니다. 계획은 해시·태그로 동결하고 GitHub에 푸시했습니다. 결과는 방향과 무관하게 보고했고, S0~S8 판정은 바꾸지 않았습니다.

# | PA | 동결 판독 | 핵심 수치 | 쓸 수 있는 문장 / 주의 |
# |---|---|---|---|
# | PA-0 | 완료(일부 미완) | 해시 152건 일치 | S4 코드는 복구본이며, oa_raw 외부 백업은 미완료 |
# | A1 | run1 무효, run2 all·within 강화, fixed14 곡선만 보고 | δ=0 기각률 0.035/0.075/0.065 | "초과분 크기 점프였다면 2020 부근은 탐지됐을 가능성이 높다"까지. 2008~2012는 사각지대. 연속 꺾은선 BIC는 2020을 꺾임으로 선택하지 않음(탐색적) |
# | B2 | 3표본 유지 | J 비율 0.998~1.006 | 하향 추출이 20%뿐이라 약한 시험 |
# | B2b | 단독 제외는 유지, 동시 제외는 약화 | 상위 5저널+토픽 71 제외 시 J 비율 0.74~0.83 | 2020은 단일 원인이 아닌 겹친 가속 |
# | B3a | 핵심 7·중간 10·특이적 14 | 핵심 7개 모두 A·big·성장 | 정의 구조의 부산물일 수 있어 독립 합의 증거로 쓰지 않음 |
# | B3b | 주원인 귀속 불가 | D1의 평균 Jaccard 0.129. 변형은 0.382, 0.085, 0.341 | 구간과 규칙이 얽혀 있음 |
# | B3c | 쌍별 기술 | 마지막 쌍 core 진입 0.458 대 적응 0.252. core−specific +0.012(0 포함) | core 고유성 없음. P1014→P1519가 가장 뚜렷 |
# | B4 | S5 둔감, S6 3쌍 둔감 | 첫 쌍만 부호 변화 | 제외 4개뿐인 약한 시험 |
# | B4b | 판독1 둔감, 판독2 규칙상 민감 | 안정성 +0.027/+0.030 | A 변화 +0.0304로 임계 0.03을 간발로 초과. 토픽 1개 규모 |
# | B5 | 부호 반전 0 | 안정성 0.66~0.77 | "0.70 전후에서 움직임" |
# | A2c | 3쌍 유지(첫 쌍 제외) | 두 구간 방식 모두 확실한 곳은 마지막 쌍뿐 | excess 구간은 교정 불충분 |
# | A2b | s6 3, y5 3, **y2 0(뒤집힘)** | y2 차이 −0.15~−0.16 | "신규 저자 주도"가 아니라 "진입(신규+복귀) 주도" |
# | A2a | 4쌍 모두 둔감 | 변화 ≤0.011 | ID는 2020+만 있어 ID 재판별은 불가 |
# | **C1** | **3개 시나리오 모두 유지 안 됨** | 0.113 → −0.101(a), −0.110, −0.024(b) | 진입 우세와 중국 소속 유입은 분리 불가 |
# | C2 | 20칸 중 일치 11 | 중국 몫은 4기간 일치. 미국·상위5는 +0.01~0.03 | 수준은 다름 |
# | A3 | 방향 유지 | 알려진 국가 기준 top5 평탄(0.64→0.65). HHI 반등은 중국 몫과 연동 | 인과 아님 |
# | B7a | 유지 | big 50/50, 뒤집힘 배수 최소 5.2 | 민감도이며 경계가 아님 |
# | B7b | 유지 | 4시나리오 50/50 | 분류기가 관측 분포에 끌릴 가능성 |
# | B7c | 부호 유지, 라벨 둔감 | 50/50, 라벨 0.98 | within의 2009 변화점이 2006으로 이동 |
# | B9a | HIC만 추정 가능 | safety·road_deaths 편 ρ≈0.24(하한 0.00~0.01) | 소득군 내 전부 비관찰이며 UMC·LMC·LIC는 NA. "소득 수준과 분리되지 않는 간극" |
# | B8a | 참고용 | 8쌍 중 7쌍 HIC 과잉·LIC/LMC 과소 | 편 ρ는 전부 0 포함. 본문 판정 2/6은 불변 |

# **종합 결론**
# - 쓸 수 있는 것은 다음과 같습니다.
#   - 변화는 점프보다 연속적 재배치로 서술하는 편이 자료에 부합합니다.
#   - 2020 부근은 COVID와 기술 주제가 겹친 가속입니다.
#   - 진입 우세는 마지막 구간에서 가장 뚜렷합니다.
#   - 알려진 국가 기준 top5 몫은 평탄합니다.
#   - 결측 가정에서도 S4 방향은 유지됩니다.
# - 쓰면 안 되는 것은 다음과 같습니다.
#   - 체제 전환 부재 확정, 2020 아티팩트 아님 확정
#   - 신규 저자가 모든 구간을 주도했다는 문장
#   - 진입 주도가 중국 유입과 무관하다는 문장
#   - ID 기반 일치, 인과 해석
# - **구조상 가장 큰 변화**: "진입 중심 갱신"과 "중국 몫 증가"는 독립된 두 기여가 아니라 한 구조의 두 측면으로 묶어야 합니다(C1, A3).

# ## 3. 진행 중

# - **B9b**: 동결·태그·푸시와 QUICK 재현 점검(통과)까지 끝났고, 본 실행은 아직 하지 않았습니다. 소득군 내에서 실제로 추정되는 칸은 HIC(전 변형)와 UMC(하한 5)뿐입니다. LIC와 LMC는 n<20이라 자료상 불가입니다.

# ## 4. 앞으로 할 분석

# | 순서 | 항목 | 내용 | 전제 |
# |---|---|---|---|
# | 1 | B9b 본 실행 | 스킴 3×변형 2×하한 3 민감도 | 해석은 safety·road_deaths만 |
# | 2 | B4d | Jaccard p10 하위 10% 제외 재추정 | 연구자 결정 필요. 임베딩 기반 응집도(B4c)는 파일 부재로 미수행 확정 |
# | 3 | A4 | 열람·동결 이력 부록 표(`pa_a4.py`) | MANUAL 항목 검토 |
# | 4 | 마무리 | 통합 문구표, 약화·반전 PA 공개 목록 | C1, A2b y2, B4 첫 쌍, B4b 판독2, B9a, B8a |
# | 5 | S9, S10 | 후보 풀 동결, 저널 선례, 심사 대응표, AI 사용 선언 | 투고 후보 저널 필요 |
# | 병행 | B6, B8-2 | 수작업 50쌍, 새 25편 코딩 | **인력 여부 미답.** 답이 없으면 "미수행, 한계로 명시" |

# **잔여 정리**
# - oa_raw 외부 백업(미완료)
# - GitHub Private 확인
# - S8.5 노트북 최종 커밋

# **제 추정(주관적 확률)**
# - B9b에서 safety·road_deaths의 UMC(하한 5) 구간이 0을 배제할 확률: 약 10%. n≈20이라 구간이 넓습니다.
# - B9b에서 ALL·HIC 결과가 B9a와 같은 방향(비관찰 유지)일 확률: 약 85%.
# - B4d가 S5 판정과 S6 마지막 쌍 부호를 바꿀 확률: 약 5%.

# ## 5. 제 쪽 오류 정정 (기록 필요)

# - B4b에서 "제외가 4개뿐이라 민감이 나오기 어렵다"고 한 것은 틀렸습니다. 임계값 0.03이 토픽 1개 규모였습니다.
# - B9b와 B3b를 "별말씀이 없어 생략"으로 처리한 것은 침묵을 동의로 본 것이었습니다. 이후 재개했습니다.
# - B4c에서 임베딩 파일이 없을 가능성을 먼저 확인하지 않았습니다.
# - "S3에 Jaccard 중앙값만 있다"는 설명은 틀렸습니다(`jaccard_p10` 열이 있음). A2 요약의 "마지막 구간이 가장 견고"는 C1로 인해 "집단 구성을 통제하면 유지되지 않음"이라는 단서가 붙습니다.

# **지금 필요한 답**
# 1. B4d를 할지
# 2. B6와 B8-2를 할 수 있는지
# 3. 투고 후보 저널

# 가장 가까운 다음 한 걸음은 B9b 본 실행입니다.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# nohup python3 -u pa_b9b.py run > 09_PA/PA_B9b_full_log.txt 2>&1 &
# ```

In [ ]:
# # PA-B9b 판독

# 정상 종료됐습니다(로그 끝에 경고 문구까지 출력, 오류 없음). 이 항목은 기준 구간이 0을 배제하는 safety·road_deaths 하나만 해석하고, 나머지 세 쌍은 기술만 합니다. 모든 행은 탐색적이며 다중성 보정이 없습니다.

# ## 1. 동결 판독 집계

# | 층 | 관찰 | 비관찰 | NA |
# |---|---|---|---|
# | ALL | 10 | 62 | 0 |
# | HIC | 0 | 72 | 0 |
# | UMC | 6 | 18 | 48 |
# | LMC, LIC | 0 | 0 | 72, 72 |

# ## 2. 읽기

# 1. **ALL, safety·road_deaths:** 점추정이 18개 변형에서 0.157~0.264로 전부 양수입니다. 구간이 0을 배제하는 변형은 ALL 24행 중 10행입니다. 구간 하한이 0 근처(0.001~0.044)에 머무는 칸이 대부분이라, B9a와 같이 "부호는 안정적이나 구간은 0 경계"입니다. 하한 20에서는 관찰이 줄어듭니다(n=66으로 감소).
# 2. **ALL, 나머지 세 쌍:** vehicles와 urban_pop은 전 변형에서 0을 포함합니다. 예외가 하나 있습니다. transit·transport_co2는 full_count·rec·하한 5에서 구간이 [−0.394, −0.013]로 0을 배제하는 음의 값이 나옵니다. 이 쌍은 기준 구간이 0을 포함하므로 규칙상 해석하지 않으며, 변형 72행 중 1행이라 다중성상 우연일 수 있습니다.
# 3. **HIC:** 72행 모두 비관찰입니다. safety·road_deaths의 점추정은 0.136~0.226으로 양수이지만 구간이 ±0.15~0.5로 넓습니다. B9a와 같은 결론("소득군 안에서는 판단할 수 없다")입니다.
# 4. **UMC(하한 5, n=20~23):** safety·road_deaths가 6개 변형(스킴 3×변형 2) 모두 "관찰"입니다. ρ=0.50~0.63이고 하한은 0.010~0.134입니다. 앞서 제가 낸 확률 추정(UMC에서 0을 배제할 확률 약 10%)은 빗나갔습니다. 다만 해석은 조심해야 합니다.
#    - n≈22에서 구간이 [0.01, 0.80]처럼 매우 넓습니다.
#    - 6행은 같은 22~23개국을 스킴·변형만 달리해 재계산한 것이라 독립 증거가 아닙니다.
#    - 하한 5는 질량이 작은 국가(공급 강도 잡음이 큼)를 포함하고, UMC는 하한 10에서 n<20이라 NA입니다. 즉 이 결과는 하한 5에서만 성립합니다.
#    - 같은 지표에서 HIC는 0과 구분되지 않습니다. 소득군마다 방향은 같지만 크기는 UMC가 큽니다.
# 5. **스킴 3종은 국가 집합이 같아** 스킴 간 차이는 강도 정의 민감도입니다. safety·road_deaths는 세 스킴에서 비슷하게 나와(inv_n_authors 0.237, full_count 0.255, equal 0.237, 하한 10) 스킴 선택이 결과를 좌우하지 않습니다.

# ## 3. 쓸 수 있는 문구

# > "공급 정의(3종), 변형(2종), 질량 하한(5·10·20)을 바꿔도 safety 영역의 사망률 지표에서만 부호가 일관되게 양수였다(ALL 편 ρ 0.16~0.26). 구간 하한은 0 근처였다. 고소득군에서는 어느 변형에서도 0과 구분되지 않았고, 중상위소득군은 질량 하한 5에서만 추정이 가능했으며(n=20~23) 그 칸에서 ρ=0.50~0.63이었으나 구간이 넓다. 중하위·저소득군은 표본 부족으로 추정하지 못했다."

# 쓰면 안 되는 것: "소득군 안에서도 정렬이 확인됐다", "소득과 무관하다", 필요 충족 증거. 지침서 문구 "소득 수준과 분리되지 않는 간극"은 바뀌지 않습니다.

# ## 4. 기록과 커밋

# `pa_b9b.py`는 `full_log`를 만들지 않았지만 `tee` 없이 `nohup` 리다이렉트로 로그가 남았습니다.

# ```bash
# cd ~/Research/Minnesota_Transport
# pgrep -f pa_b9b.py || echo "종료 확인"
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B9b 결과 해석 메모(B=1000): 동결 판독 집계 ALL 관찰10/비관찰62, HIC 0/72, UMC 관찰6/비관찰18/NA48, LMC·LIC 전부 NA. 해석 대상 safety-road_deaths: ALL 점추정 0.157~0.264 전부 양수(하한 0 근처), HIC 비관찰(0.136~0.226), UMC는 하한 5에서만 추정 가능(n 22~23)하며 6개 변형 모두 관찰(rho 0.50~0.63, 하한 0.010~0.134)이나 같은 국가 집합의 변형이라 독립 아님, 구간 넓음. transit-transport_co2는 full_count·rec·하한5에서 음의 구간이 0 배제(1/72행, 기준 구간이 0 포함이라 비해석). 사전 확률 추정(UMC 약 10%)은 틀림. 문구는 소득 수준과 분리되지 않는 간극으로 제한 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_B9b_results.csv paper1/09_PA/PA_B9b_n_table.csv paper1/09_PA/PA_B9b_full_log.txt paper1/09_PA/PA_CHANGELOG.md
# git status --short | head -n 8
# git commit -m "PA-B9b results" && git tag -a PA-B9b-results-2026-10-21 -m "PA-B9b outputs, seen" && git push --follow-tags
# ```

# `git status`에 스테이징 외 항목이 있으면 커밋 전에 알려 주세요.

# ## 5. 남은 항목

# 분석 쪽은 B4d(Jaccard p10)와 사람이 필요한 B6·B8-2만 남았습니다. 다음 권고는 다음과 같습니다.

# 1. A4 표 생성(`pa_a4.py` 실행 후 `MANUAL` 검토)
# 2. 통합 문구표와 약화·반전 PA 공개 목록 작성(C1, A2b y2, B4 첫 쌍, B4b 판독 2, B9a·B9b, B8a)
# 3. 결정 필요: B4d 수행 여부, B6·B8-2 인력, 투고 후보 저널
# 4. 잔여 정리: `oa_raw` 외부 백업, GitHub Private 확인, `S8.5` 노트북 커밋

In [ ]:
# # 기록 확인과 통합 문구표

# 커밋 `9097331`이 푸시됐고 스테이징은 예상한 4개뿐이었습니다. 결정 세 가지(B4d, B6·B8-2, 투고 저널)는 아직 답이 없어서, 예고한 대로 통합 문구표를 먼저 정리합니다. 이 표는 changelog와 위 결과 해석 메모에서 옮긴 것이므로, 본문에 넣기 전에 원 CSV 수치와 대조하세요.

# ## 1. 쓸 수 있는 문장

# | 주제 | 문장 | 근거 |
# |---|---|---|
# | 변화 양상 | 주제 구조 변화는 점프보다 연속적 재배치로 서술하는 편이 자료에 부합한다. | S5 N2 비통과, A1, 연속 꺾은선 BIC(탐색적) |
# | 2020 부근 | COVID 토픽의 저널 내 증가와 기술 토픽 성장이 겹친 가속이 있다. 표본 크기, 상위 저널, COVID 토픽 중 어느 하나만으로는 사라지지 않으나, 상위 5개 저널과 COVID 토픽을 함께 제외하면 약해지고 변화점이 한 해 앞으로 이동한다. | B2, B2b |
# | 검정력 | 2020 부근에서 관측 초과분 크기의 점프가 있었다면 N2 검정이 탐지했을 가능성이 높다. 2008~2012 부근은 검정력이 낮다. | A1 |
# | 진입 | 진입(신규+복귀)이 적응보다 큰 방향은 마지막 구간(2015–19→2020–25)에서 가장 뚜렷하며, 부트스트랩 방식, 신규 정의, 이름 분할 제외에 둔감하다. | A2a·b·c |
# | 구조 | 마지막 구간의 진입 우세는 중국 소속 저자를 제외하면 유지되지 않는다. 진입 중심 갱신과 중국 몫 증가는 한 구조의 두 측면으로 서술한다. | C1, A3 |
# | 국가 | 알려진 국가 기준 상위 5개국 몫은 평탄하다. 포함 기준의 증가는 미상 비중 감소와 함께 움직인다. HHI 반등은 중국 소속 질량과 함께 움직인다. | A3 |
# | 결측 | 결측 가정 네 가지(B7a·b·c, S4 IPW)에서 big 토픽의 변화 방향이 유지됐다(경계가 아닌 민감도). | B7 |
# | 정렬 | 도로 안전의 사망률 지표에서만 부호가 일관되게 양수였고, 구간 하한은 0 근처다. 소득군 내에서는 HIC가 비관찰이고 UMC는 하한 5에서만 추정이 가능했다. | B9a·b |

# ## 2. 쓰면 안 되는 문장

# - 체제 전환이 없었음이 확인됐다.
# - 2020이 아티팩트가 아니다(확정).
# - 신규 저자가 모든 구간을 주도했다. (y2 정의에서 뒤집힘)
# - 진입 주도는 중국 유입과 무관하다.
# - 핵심 신생 토픽은 신규 저자 진입으로 설명된다. (B3c: core 고유성 없음)
# - ID 기반 결과와 일치한다, 동명이인 문제가 해결됐다.
# - 소득과 무관하게 정렬된다, 필요 충족 증거다.
# - 소속을 국적으로 해석하는 표현과 모든 인과 해석.

# ## 3. 약화·반전된 결과 (본문 한계 절과 부록에 먼저 공개)

# | 항목 | 내용 |
# |---|---|
# | C1 | 진입−적응 +0.113이 신규 제외 시 −0.101, 신규·지속 제외 시 −0.024로 반전 |
# | A2b | y2 정의에서 4쌍 모두 신규<적응. 정의 효과이며 진입 총량은 불변 |
# | A2c | 두 구간 방식 모두 확실한 쌍은 마지막 쌍뿐. excess 구간은 교정 불충분 |
# | B4 | 첫 쌍만 부호 변화. 원값이 0 근처 |
# | B4b | 판독 2가 규칙상 "민감"(A +0.0304, 토픽 1개 규모) |
# | B2 | 하향 추출이 20%라 약한 시험 |
# | B3a·c | 핵심 집합은 정의 구조의 부산물일 수 있음 |
# | B9a·b | LIC·LMC는 추정 불가, HIC는 비관찰. UMC는 하한 5의 22~23개국에서만 관찰이며 독립 증거가 아님 |
# | B8a | 미달 도메인은 참고용 |
# | B7c | within의 2009 변화점이 2006으로 이동 |

# ## 4. 제 쪽 정정 (부록 기록용)

# - B4b 임계값(0.03)이 토픽 1개 규모라는 점을 사전에 놓쳤습니다.
# - B9b에서 UMC가 0을 배제할 확률을 약 10%로 봤는데 틀렸습니다.
# - B9b·B3b 생략을 침묵 동의로 처리했습니다.
# - A2 요약의 "마지막 구간이 가장 견고"는 C1 단서가 필요합니다.

# ## 5. 지금 필요한 답

# 1. B4d를 할지. 안 하면 B4c와 함께 미수행으로 확정합니다.
# 2. B6·B8-2를 할 수 있는지.
# 3. 투고 후보 저널.

# 이 세 가지를 확정하면 구성안과 초록 초안을 쓰고, S9·S10으로 넘어가겠습니다. 이번 주 안에 답이 없으면 B4d·B6·B8-2는 "미수행, 한계로 명시"로 확정합니다.

# ## 6. 잔여 정리

# - A4 표 수동 보정(A1, PA-0, B3·B7·B4c 분류, `MANUAL` 보충)
# - `oa_raw` 외부 백업, GitHub Private 확인, `S8.5` 노트북 커밋

In [ ]:
# # 논문 1 사후분석(PA) 정리 (2026-10-21 기준)

# ## 1. 하려고 했던 계획

# 지침서(10-19)의 14개 항목이고, 핵심은 A1·A2·B2입니다. 원칙은 다음과 같습니다.
# - 계획은 실행 전에 해시와 태그로 동결하고 GitHub에 푸시합니다.
# - 결과는 방향과 무관하게 전부 보고합니다.
# - S0~S8의 통과/실패 판정은 바꾸지 않습니다.

# | 구분 | 항목 | 질문 |
# |---|---|---|
# | 재현성 | PA-0 | 파일, 코드, 태그가 온전한가 |
# | 강점 강화 | A1 | N2 비통과는 검정력 부족인가, 점프 부재인가 |
# | | A2 | 신규 진입 우세는 정의, 부트스트랩, 이름 식별에 견고한가 |
# | | A3 | 선두 국가 교체 서술이 견고한가 |
# | 약점 리프레이밍 | B2 | 2020 변화점이 표본 크기·저널 구성의 아티팩트인가 |
# | | B3 | 신생 토픽 정의 4종의 핵심 집합은 무엇인가 |
# | | B4 | 등급 선택이 결과를 좌우하는가 |
# | | B5 | S4 안정성 0.700은 경계인가 |
# | | B6 | 위험군에 독립 정답이 있는가 |
# | | B7 | 임베딩 결측 16%가 결과를 뒤집는가 |
# | | B8 | 미달 4개 도메인의 정렬은 어떤가 |
# | | B9 | 간극이 소득 수준과 분리되는가 |
# | 정합성 | C1 | 진입 우세가 중국 소속 유입과 독립인가 |
# | | C2 | S7과 S6의 모집단 차이는 얼마인가 |

# ## 2. 분석 결과

# | PA | 동결 판독 | 핵심 수치 | 쓸 수 있는 말 / 주의 |
# |---|---|---|---|
# | PA-0 | 대부분 완료 | 해시 152건 일치 | S4 코드는 복구본이고 `oa_raw` 외부 백업은 미완료 |
# | A1 | run 1 무효, run 2는 all·within 강화, fixed14는 곡선만 보고 | δ=0 기각률 0.035/0.075/0.065 | 2020 부근은 초과분 크기 점프를 탐지했을 가능성이 높음. 2008~2012는 사각지대 |
# | B2 | 3표본 유지 | J 비율 0.998~1.006 | 하향 추출이 20%뿐이라 약한 시험 |
# | B2b | 단독 제외는 유지, 동시 제외는 약화 | 상위 5저널+토픽 71 제외 시 J 0.74~0.83 | 2020은 단일 원인이 아닌 겹친 가속 |
# | B3a·b·c | 핵심 7·중간 10·특이적 14 | 핵심 진입 0.458 대 적응 0.252 | 정의 구조의 부산물이며 독립 합의 증거 아님 |
# | B4·B4b | S5 둔감, S6 3쌍 둔감, B4b 판독 2는 규칙상 민감 | 안정성 +0.027/+0.030 | 임계 0.03이 토픽 1개 규모라 변별력이 낮았음 |
# | B5 | 부호 반전 0 | 안정성 0.66~0.77 | "0.70 전후에서 움직임" |
# | A2c | 3쌍 유지(첫 쌍 제외) | 두 구간 방식 모두 확실한 쌍은 마지막뿐 | excess 구간은 교정 불충분 |
# | A2b | s6 3, y5 3, y2 0(뒤집힘) | y2 차이 −0.15~−0.16 | "신규 저자 주도"가 아니라 "진입(신규+복귀) 주도" |
# | A2a | 4쌍 모두 둔감 | 변화 ≤0.011 | ID는 2020+만 있어 ID 재판별은 불가 |
# | C1 | 3개 시나리오 모두 유지 안 됨 | +0.113 → −0.101(a), −0.110, −0.024(b) | 진입 우세와 중국 소속 유입은 분리 불가 |
# | C2 | 20칸 중 일치 11 | 중국 몫은 4기간 일치, 미국·상위5는 +0.01~0.03 | 수준은 다름 |
# | A3 | 방향 유지 | 알려진 국가 기준 top5 평탄(0.64→0.65) | HHI 반등은 중국 몫과 연동 |
# | B7a·b·c | 모두 유지 | big 50/50 | 민감도이며 경계가 아님 |
# | B9a·b | HIC 비관찰, UMC는 하한 5에서만 관찰 | safety·road_deaths 편 ρ≈0.24 | LIC·LMC는 자료상 불가 |
# | B8a | 참고용 | 8쌍 중 7쌍 HIC 과잉·LIC/LMC 과소 | 본문 판정 2/6은 불변 |

# **종합**
# - **쓸 수 있는 것**
#   - 변화는 점프보다 연속적 재배치로 서술하는 편이 자료에 부합합니다.
#   - 2020 부근은 COVID와 기술 주제가 겹친 가속입니다.
#   - 진입 우세는 마지막 구간에서 가장 뚜렷합니다.
#   - 알려진 국가 기준 top5 몫은 평탄합니다.
#   - S4의 방향은 결측 가정에서도 유지됩니다.
# - **쓰면 안 되는 것**
#   - 체제 전환 부재 확정, 2020이 아티팩트가 아니라는 확정
#   - 신규 저자가 모든 구간을 주도했다는 문장
#   - 진입 주도가 중국 유입과 무관하다는 문장
#   - ID 기반 일치, 인과 해석
# - **구조상 가장 큰 변화**: "진입 중심 갱신"과 "중국 몫 증가"는 독립된 두 기여가 아니라 한 구조의 두 측면으로 묶어야 합니다.

# ## 3. 앞으로 할 일

# | 순서 | 항목 | 내용 | 전제 |
# |---|---|---|---|
# | 1 | 결정 3가지 | B4d(Jaccard p10 제외 재추정) 수행 여부, B6·B8-2 인력, 투고 후보 저널 | 연구자 답 필요. 이번 주 내 답이 없으면 B4d·B6·B8-2는 "미수행, 한계로 명시" |
# | 2 | A4 부록 표 | `pa_a4.py` 실행 후 `MANUAL` 항목 검토 | 열람한 항목을 빠짐없이 기재 |
# | 3 | 통합 문구표와 공개 목록 | 약화·반전된 PA(C1, A2b y2, B4 첫 쌍, B4b 판독 2, B9a·b, B8a)를 한계 절과 부록에 먼저 공개 | |
# | 4 | S9, S10 | 후보 풀 동결, 저널 선례 3편 이상, 심사 대응표, 생성형 AI 사용 선언 | 투고 후보 저널 |
# | 5 | 원고 | 제목에서 "체제 전환"을 빼고, 진입 중심 갱신과 중국 소속 유입을 한 구조로 서술 | 영문 post-hoc 문구 삽입, 모든 표에 `[Post-hoc]` 표지 |

# **잔여 정리**
# - `oa_raw` 외부 백업: 매체 연결 후 쓰기 권한 확인, 복사, SHA-256 비교
# - GitHub 저장소 Private 확인
# - `S8.5` 노트북 최종 커밋

# **미수행으로 확정될 항목**
# - B4c: 임베딩 파일이 없어 수행하지 못했습니다.
# - A2의 ID 재판별: F7이 2020년 이후 논문만 덮습니다.
# - B6의 수작업 50쌍과 B8-2의 새 25편 코딩: 인력이 필요합니다.

# 가장 가까운 다음 한 걸음은 투고 후보 저널을 정하는 것입니다. 분석은 사실상 소진됐고, 구성안과 초록을 쓰려면 이 정보가 필요합니다.

In [ ]:
# # 남은 잔여분석 (2026-10-21 기준)

# ## 1. 아직 수행하지 않은 분석

# | 항목 | 내용 | 상태 | 필요한 것 |
# |---|---|---|---|
# | **B4d** | `jaccard_p10` 하위 10% 토픽을 제외하고 S5·S6 재추정 | 미착수 | 연구자 결정. 입력은 `L3_topic_metrics.csv`에 이미 있음 |
# | **B6-1** | ORCID 기반 소표본으로 위험군 정밀도·재현율 계산 | 미착수 | 사람 손 없이 가능. OpenAlex ORCID 확보 여부 확인 |
# | **B6-2** | 이름 빈도 상위 구간에서 50쌍을 뽑아 동일인 여부 수작업 판정 | 미수행 | 연구자 시간 |
# | **B8-2** | 사전 v3 작성, 새 무작위 25편 이중 코딩으로 precision 재검증 | 미수행 | 코더 2명. 하더라도 통과/실패 판정이 아니라 점추정과 Wilson 구간만 보고 |

# ## 2. 수행하지 못하거나 축소한 것 (미수행 목록에 사유 기재)

# | 항목 | 사유 |
# |---|---|
# | **B4c** (임베딩 응집도) | 임베딩 파일을 찾지 못함 |
# | **A2 ID 기반 재판별** | F7(OpenAlex ID)이 2020년 이후 논문만 덮어 2015–19 저자를 ID로 가를 수 없음. 분할 이름 제외 민감도로 대체함 |
# | **B9 LIC·LMC 소득군 내 분석** | 질량 하한 때문에 LIC는 0개국, LMC는 n<20. 하한을 낮춰도 해결되지 않음 |

# ## 3. 분석은 아니지만 남은 작업

# | 항목 | 내용 |
# |---|---|
# | **A4** | `pa_a4.py` 실행 후 `MANUAL` 항목(A1, PA-0, B7b 등)을 직접 검토해 열람·동결 이력 부록 표 확정 |
# | **통합 문구표** | 약화·반전된 PA(C1, A2b y2, B4 첫 쌍, B4b 판독 2, B9a·b, B8a)를 한계 절과 부록에 먼저 공개 |
# | **재현성 잔여** | `oa_raw` 외부 백업(매체 연결 후 SHA-256 비교), GitHub Private 확인, `S8.5` 노트북 최종 커밋 |
# | **S9, S10** | 후보 풀 동결, 저널 선례 3편 이상, 심사 대응표, 생성형 AI 사용 선언 (투고 후보 저널 필요) |

# ## 4. 권고

# - **B4d**: 이미 두 번 "등급 선택은 방향을 바꾸지 않는다"가 나왔고 제외 토픽이 약 15개로 늘어 변별력이 조금 더 큽니다. 비용이 낮으니 하되, 안 하더라도 결과가 바뀔 확률은 낮아 보입니다.
# - **B6-1(ORCID)**: 수작업 없이 위험군 검증의 순환성을 줄일 수 있는 유일한 경로라, 가능하다면 B6-2보다 먼저 시도할 가치가 있습니다.
# - **B6-2·B8-2**: 인력이 없으면 이번 주 안에 "미수행, 한계로 명시"로 확정하세요.

# 어느 항목부터 진행할지 알려 주시면 해당 동결 코드를 드리겠습니다.

In [ ]:
# 아래 코드는 모두 미실행이고, 기존 `pa_common.py`를 수정하지 않습니다. 수정하면 이미 동결한 스크립트의 해시 점검이 깨지기 때문입니다. 새 보조 모듈 `pa_extra.py`를 하나 추가하고, 항목은 네 개(B4d, B6a, B6b, B8b)로 나눴습니다.

# **먼저 알아 둘 점**

# - B4d와 B6a는 사람 손 없이 끝까지 돌아갑니다.
# - B6b(수작업 50쌍)와 B8b(새 25편 코딩)는 코드가 **블라인드 시트 생성**과 **채점**만 합니다. 판정과 코딩은 연구자가 직접 해야 합니다. 판정 칸은 시트에서 비어 있습니다.
# - B6a는 OpenAlex API를 호출하므로 네트워크가 필요합니다. 동결, 태그, 푸시 뒤에 `fetch`를 돌립니다.
# - B8b는 사전 v3의 정규식을 연구자가 먼저 `09_PA/PA_B8b_dict_v3.json`에 써야 동결됩니다. 정규식이 계획 해시에 들어갑니다.
# - 가정이 몇 개 있습니다. 컬럼명(`L3_topic_metrics.csv`의 `topic`, `jaccard_p10`)과 B8b의 이전 검증 표본 파일 위치는 제가 본 적이 없습니다. 어긋나면 중단 메시지가 나옵니다.

# ## 0. 보조 모듈 `paper1/pa_extra.py`

# ```python
# # -*- coding: utf-8 -*-
# # 사후분석 보조(Wilson 구간, 동결 검증, 특수 모드 디스패치). pa_common.py는 수정하지 않음. 미실행.
# from pa_common import *

# def wilson(k, n, z=1.96):
#     if n == 0: return (np.nan, np.nan)
#     p = k/n; d = 1 + z*z/n; c = (p + z*z/(2*n))/d; h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
#     return float(c - h), float(c + h)

# def guard(name, PLAN):
#     BP = PA/f"PA_{name}_PLAN.json"; script = f"pa_{name.lower()}.py"
#     if not BP.exists(): raise Halt("먼저 freeze")
#     sp = json.loads(BP.read_text(encoding="utf-8"))
#     if sp["sha256"] != spec_hash(sp) or sp["sha256"] != spec_hash(PLAN): raise Halt("동결 파일과 현재 계획이 다릅니다")
#     if not git("tag", "-l", f"PA-{name}-plan-*").stdout.strip(): raise Halt("tag 단계를 먼저 실행하세요")
#     for fn in ["pa_common.py", script]:
#         if sp["refs"].get(fn) != sha_f(P1/fn): raise Halt(f"{fn}이 동결 시점과 다름(수정했다면 v2 계획으로 분리)")
#     return sp

# def special(name, PLAN, handlers):
#     """argv[1]이 handlers에 있으면 동결 검증 후 실행하고 True 반환."""
#     mode = sys.argv[1] if len(sys.argv) > 1 else ""
#     if mode not in handlers: return False
#     try: handlers[mode](guard(name, PLAN))
#     except Halt as e: print(f"⛔ 중단: {e}")
#     return True
# ```

# ## 1. B4d: Jaccard p10 하위 10% 제외 재추정 (`pa_b4d.py`)

# ```python
# # -*- coding: utf-8 -*-
# # PA-B4d: L3_topic_metrics.csv의 jaccard_p10 하위 10% 토픽을 제외하고 S5·S6 재추정(B4와 같은 틀). 미실행.
# # python3 pa_b4d.py freeze | tag | run [quick]
# from pa_common import *
# from pa_extra import *
# PLAN = dict(
#     version=1, seed=2026, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(지침서 B4-2의 Jaccard p10 부분). B4·B4b 결과(F1·F3 제외 4개뿐), S3 L3_topic_metrics.csv에 jaccard_p10 열 존재 확인 열람. 결과 미열람. S0~S8 판정 불변.",
#     exclusion="L3_topic_metrics.csv의 jaccard_p10이 가장 낮은 ceil(0.10*토픽 수)개 토픽 제외(동률은 topic 번호 순). 제외 집합과 F1·F3 실패 4개, S3 등급의 겹침을 보고",
#     reestimate="B4와 같은 절차. S5: 잔존 토픽만으로 s5_lib.detect(pen=c*phi*(K-1)*log T, K=잔존 수, phi는 S5 값 고정), N1·N2 귀무 B=500, 통과 규칙 N1·N2 모두 p<0.05. S6: R을 잔존 토픽 행으로 제한한 주 설정, 대응 저자 부트스트랩 B=spec['B_boot'], entry_minus_adapt 변화",
#     reading="S5: 표본별 (a) 통과 여부가 S5와 같은지 (b) S5 관측 변화점이 +-1년 유지되는지. S6: 쌍별 |변화|<=0.03이고 부호 유지면 '둔감'(첫 쌍은 원값이 0 근처라 부호가 바뀌기 쉬움). 합치지 않음",
#     reproduce_check="S5 관측 R이 regime_nulls.csv obs_R과 1e-3 이내, S6 total_half·A2c entry_share 재현, 부분 자료 저자 집합이 원본의 부분집합, 제외 토픽 수 15",
#     caveat="jaccard_p10은 부트스트랩 군집 안정성이라 작은 토픽이 불리할 수 있음. 하위 10%는 임의 기준(동결). 등급 타당성은 검증하지 않음",
#     forbidden="결과를 본 뒤 비율(10%)·판독 변경(변경 시 v2 계획으로 분리)")

# def reestimate(ptop, quick, rng):
#     import s5_lib as L5, s6_lib as S
#     s5 = json.loads((S5D/"S5_0_PREREG.json").read_text(encoding="utf-8")); D = s5["detection"]; PHI = {k: v["phi"] for k, v in D["phi"].items()}
#     C0, KMAX, MINLEN, ALPHA = D["c_primary"], D["k_max"], D["min_len"], s5["nulls"]["alpha"]; Bn = 20 if quick else 500
#     cnt, venues, topics, fixed_v = L5.load_counts(); wbar = cnt.sum((0, 2))/cnt.sum(); T = len(L5.YEARS); YEARS = np.asarray(L5.YEARS); SAM = s5["samples"]
#     tmask = np.isin(topics, list(ptop)); Ku = int(tmask.sum())
#     det = lambda c, s, Kuse: (lambda Pn: L5.detect(Pn[0], Pn[1], C0*PHI[s]*(Kuse - 1)*np.log(T), KMAX, MINLEN))(L5.sample_P(c, fixed_v, wbar, s))
#     ref = pd.read_csv(S5D/"regime_nulls.csv"); ref = ref[ref["null"] == "N2"].set_index("sample")
#     for s in SAM:
#         if abs(det(cnt, s, len(topics))["R"] - float(ref.loc[s, "obs_R"])) > 1e-3: raise Halt(f"S5 관측 R 재현 실패({s})")
#     cs = cnt[:, :, tmask]; obs = {s: det(cs, s, Ku) for s in SAM}; P2 = L5.trend_probs(cs); NR = {"N1": {s: [] for s in SAM}, "N2": {s: [] for s in SAM}}
#     for b in range(Bn):
#         for nm, cc in (("N1", L5.gen_N1(cs, rng)), ("N2", L5.gen_N2(cs, P2, rng))):
#             for s in SAM: NR[nm][s].append(det(cc, s, Ku)["R"])
#     meta = json.loads((S5D/"S5_meta.json").read_text(encoding="utf-8")); rows5 = []
#     for s in SAM:
#         yrs = [int(YEARS[x]) for x in obs[s]["br"]]; pv = {nm: (1 + float((np.array(NR[nm][s]) >= obs[s]["R"]).sum()))/(1 + Bn) for nm in NR}
#         kept = [L5.near(y, yrs, 1) for y in meta["obs_breaks"][s]]
#         rows5.append(dict(sample=s, K=Ku, obs_R=obs[s]["R"], breaks=str(yrs), s5_breaks=str(meta["obs_breaks"][s]), p_N1=pv["N1"], p_N2=pv["N2"],
#                           passed_new=max(pv.values()) < ALPHA, passed_s5=bool(meta["null_pass"][s]), all_breaks_kept=all(kept), B=Bn))
#     spec = S.load_spec(); L = S.load_all(spec); M = spec["main"]; PAIRS = S.pairs_of(spec); half = S.half; R = L["R"]
#     ok_ti = np.where(np.isin(L["topics"], list(ptop)))[0]; L2 = dict(L); L2["R"] = R[R.ti.isin(ok_ti)]
#     ref6 = pd.read_csv(S6D/"renewal_decomposition.csv"); ref6 = ref6[(ref6.scheme == M["scheme"]) & (ref6.variant == "all_venues")].set_index("pair")
#     a2c = pd.read_csv(PA/"PA_A2c_results.csv"); a2c = a2c[(a2c["mode"] == "author") & (a2c.stat == "entry_share")].set_index("pair")
#     B6 = 20 if quick else int(spec["B_boot"]); lo_q, hi_q = spec["ci"]; STATS = ["entry_share", "adapt_share", "entry_minus_adapt"]
#     def stats(d):
#         h = {t: half(d[t]) for t in ["adapt", "realloc", "entry", "exit"]}; tot = sum(h.values())
#         return dict(entry_share=h["entry"]/tot, adapt_share=h["adapt"]/tot, entry_minus_adapt=(h["entry"] - h["adapt"])/tot)
#     rows6 = []
#     for p0, p1 in PAIRS:
#         name = f"{p0}->{p1}"; Pa = S.make_pair(L, p0, p1, M["scheme"], M["venue"], M["topic"], M["hom"]); Pb = S.make_pair(L2, p0, p1, M["scheme"], M["venue"], M["topic"], M["hom"])
#         da, db = Pa.decompose(), Pb.decompose(); oa, ob = stats(da), stats(db)
#         if abs(half(da["total"]) - float(ref6.loc[name, "total_half"])) > 1e-6 or abs(oa["entry_share"] - float(a2c.loc[name, "obs"])) > 1e-6: raise Halt(f"S6/A2c 재현 실패({name})")
#         idx = np.searchsorted(Pa.aids, Pb.aids)
#         if not (idx < len(Pa.aids)).all() or not (Pa.aids[idx] == Pb.aids).all(): raise Halt(f"부분 자료 저자 집합이 원본의 부분집합이 아님({name})")
#         bs = []
#         for _ in range(B6):
#             wa = rng.exponential(size=Pa.nA); sa = stats(Pa.decompose(Pa.m0*wa[Pa.a0], Pa.m1*wa[Pa.a1])); wb = wa[idx]
#             sb = stats(Pb.decompose(Pb.m0*wb[Pb.a0], Pb.m1*wb[Pb.a1])); bs.append({s: sb[s] - sa[s] for s in STATS})
#         bs = pd.DataFrame(bs)
#         for s in STATS:
#             dl = ob[s] - oa[s]; ql, qh = np.percentile(bs[s], [lo_q, hi_q]); rows6.append(dict(pair=name, stat=s, obs_orig=oa[s], obs_sub=ob[s], delta=dl, pct_lo=ql, pct_hi=qh, B=B6))
#     return pd.DataFrame(rows5), pd.DataFrame(rows6)

# def run(quick, sp):
#     sys.path.insert(0, str(P1)); rng = np.random.default_rng(int(PLAN["seed"]))
#     M3 = pd.read_csv(S3/"L3_topic_metrics.csv", encoding="utf-8-sig")
#     if not {"topic", "jaccard_p10"} <= set(M3.columns): raise Halt(f"L3_topic_metrics.csv 열 확인 필요: {list(M3.columns)}")
#     M3 = M3.dropna(subset=["jaccard_p10"]).sort_values(["jaccard_p10", "topic"]); k = int(np.ceil(0.10*len(M3))); ex = set(M3.topic.head(k).astype(int))
#     if k != 15: raise Halt(f"제외 토픽 수 {k} (15여야 함)")
#     G = pd.read_csv(S3/"L7_topic_grade.csv", encoding="utf-8-sig").set_index("topic"); f13 = set(G.index[bl(G.F1) | bl(G.F3)].astype(int)); allt = set(G.index.astype(int))
#     print(f"[재현 설계] 제외 {k}개 | F1·F3 실패와 겹침 {len(ex & f13)} | 등급 분포 {G.loc[sorted(ex), 'grade'].value_counts().to_dict()}")
#     R5, R6 = reestimate(allt - ex, quick, rng)
#     if quick: print("[QUICK] 오류 점검만. 수치 출력 안 함."); return
#     pd.DataFrame(dict(topic=sorted(ex), jaccard_p10=M3.set_index("topic").loc[sorted(ex), "jaccard_p10"].values, grade=G.loc[sorted(ex), "grade"].values, f1f3_fail=[t in f13 for t in sorted(ex)])).to_csv(PA/"PA_B4d_excluded.csv", index=False, encoding="utf-8-sig")
#     R5.to_csv(PA/"PA_B4d_s5.csv", index=False, encoding="utf-8-sig"); R6.to_csv(PA/"PA_B4d_s6.csv", index=False, encoding="utf-8-sig")
#     kk = R6[R6.stat == "entry_minus_adapt"].copy(); kk["insensitive"] = (np.sign(kk.obs_sub) == np.sign(kk.obs_orig)) & (kk.delta.abs() <= 0.03)
#     print("\n[Post-hoc] PA-B4d S5\n", R5.round(4).to_string(index=False)); print("\n[Post-hoc] PA-B4d S6 entry_minus_adapt\n", kk.round(4).to_string(index=False))
#     print("[동결 판독] S5:", R5.assign(same=R5.passed_new == R5.passed_s5)[["sample", "same", "all_breaks_kept"]].to_dict("records"))
#     print("※ 제외 집합 15개의 부분집합 재추정. S0~S8 판정 불변.")

# run_pa("B4d", PLAN, run, [S3/"L3_topic_metrics.csv", S3/"L7_topic_grade.csv", S5D/"S5_0_PREREG.json", PA/"PA_B4_PLAN.json", P1/"pa_extra.py"], "jaccard_p10 하위 10% 제외 재추정(지침서 B4-2)")
# ```

# ## 2. B6a: ORCID 기반 위험군 독립 정답 (`pa_b6a.py`)

# 판정 기준은 이름 하나가 서로 다른 ORCID를 둘 이상 가지면 동명이인이 확실하다는 것입니다. ORCID 보유자에 치우친 표본이고 2020년 이후 논문만 덮습니다. 이 두 한계는 계획서에 적었습니다.

# ```python
# # -*- coding: utf-8 -*-
# # PA-B6a: F7의 OpenAlex ID에 ORCID를 붙여, 위험군 판정(F2_risk_names)의 독립 정답(서로 다른 ORCID 2개 이상=동명이인 확실)에 대한 정밀도·재현율. 미실행.
# # python3 pa_b6a.py freeze | tag | fetch | run     (tag 뒤 git push --follow-tags)
# import urllib.request, urllib.parse, time
# from pa_common import *
# from pa_extra import *
# MAILTO = ""    # OpenAlex polite pool용 이메일(선택). 계획 해시에 넣지 않음
# CACHE = PA/"PA_B6a_orcid_cache.csv"
# PLAN = dict(
#     version=1, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(지침서 B6-1). F7 구조(pid, rank, name_norm, oa_id; 2020-2025만), 분할 이름 3,749개, 위험군 1,431개 중 1,116개가 분할 이름이라는 점은 열람. ORCID 조회 결과 미열람. S0~S8 판정 불변.",
#     fetch="F7의 고유 oa_id를 OpenAlex /authors?filter=openalex:ID1|ID2...(50개씩)로 조회해 orcid만 저장(select=id,orcid). 없으면 빈 값. 캐시 CSV는 재개 가능, 결과 해석 전에 해시를 changelog에 기록",
#     population="F7에서 name_norm이 있고, 그 이름의 oa_id 중 ORCID가 하나 이상 있는 이름",
#     truth="양성(동명이인 확실) = 이름에 서로 다른 ORCID가 2개 이상. 음성 = ORCID가 하나뿐이거나 모두 같은 ORCID. 한 이름에 ID가 둘 이상이어도 ORCID가 같으면 프로필 분할로 보고 음성",
#     flag="F2_risk_names.csv의 name_norm = 위험군",
#     stats="혼동표, 정밀도=P(양성|위험군), 재현율=P(위험군|양성), 각 Wilson 95%. 위험군 이름 중 F7 존재·모집단 포함 비율. 보조: ID가 둘 이상이고 ORCID 보유 ID가 둘 이상인 이름 중 서로 다른 ORCID 비율(분할 이름 중 실제 동명이인 비율)",
#     reading="수치와 구간만 보고. 임계값·통과 판정 없음. 위험군 검증 순환성에 대한 독립 정답 1개로 기술",
#     reproduce_check="F7 행 141,836, 분할 이름 3,749, 위험군 이름 수는 F2_risk_names 행 수와 일치(출력)",
#     caveat="ORCID 보유자는 연구 활동이 활발한 저자에 치우침. F7은 2020+만 덮음. ORCID가 다른 두 프로필이 같은 사람일 가능성은 낮지만 0이 아님. 음성은 '동명이인 증거 없음'이지 '단일인 확인'이 아님",
#     forbidden="결과를 본 뒤 정답 정의·모집단 변경(변경 시 v2 계획으로 분리)")

# def fetch(sp):
#     f7 = pd.read_csv(DATA/"F7_oa_author_ids.csv", encoding="utf-8-sig", low_memory=False)
#     ids = sorted(set(f7.oa_id.dropna().astype(str).str.replace("https://openalex.org/", "", regex=False))); done = set(pd.read_csv(CACHE).oa_id) if CACHE.exists() else set()
#     todo = [i for i in ids if i not in done]; print(f"고유 ID {len(ids):,} | 캐시 {len(done):,} | 남음 {len(todo):,}")
#     for k in range(0, len(todo), 50):
#         b = todo[k:k+50]; q = {"filter": "openalex:" + "|".join(b), "select": "id,orcid", "per-page": 50}
#         if MAILTO: q["mailto"] = MAILTO
#         url = "https://api.openalex.org/authors?" + urllib.parse.urlencode(q); res = None
#         for att in range(4):
#             try: res = json.load(urllib.request.urlopen(url, timeout=60)); break
#             except Exception as e: print("재시도", att, str(e)[:60]); time.sleep(3*(att + 1))
#         if res is None: raise Halt("API 호출 실패(캐시는 보존됨, 다시 fetch하면 이어서 진행)")
#         got = {r["id"].replace("https://openalex.org/", ""): (r.get("orcid") or "") for r in res.get("results", [])}
#         pd.DataFrame([dict(oa_id=i, orcid=got.get(i, ""), found=i in got) for i in b]).to_csv(CACHE, mode="a", header=not CACHE.exists(), index=False, encoding="utf-8-sig")
#         if (k//50) % 20 == 0: print(f"  {k + len(b):,}/{len(todo):,}"); time.sleep(0.2)
#     c = pd.read_csv(CACHE); cl(f"PA-B6a fetch 완료 | 캐시 {len(c):,}행, ORCID 보유 {int((c.orcid.fillna('') != '').sum()):,}, 해시 {sha_f(CACHE)[:12]} | 본 결과 변경: 없음")

# def run(quick, sp):
#     if not CACHE.exists(): raise Halt("먼저 fetch")
#     f7 = pd.read_csv(DATA/"F7_oa_author_ids.csv", encoding="utf-8-sig", low_memory=False)
#     if len(f7) != 141836: raise Halt(f"F7 행 수 {len(f7)} (141836이어야 함)")
#     c = pd.read_csv(CACHE).drop_duplicates("oa_id"); c["orcid"] = c.orcid.fillna("").astype(str)
#     f7["oa"] = f7.oa_id.astype(str).str.replace("https://openalex.org/", "", regex=False); f7 = f7.dropna(subset=["name_norm"]).merge(c[["oa_id", "orcid"]].rename(columns={"oa_id": "oa"}), on="oa", how="left"); f7["orcid"] = f7.orcid.fillna("")
#     g = f7.groupby("name_norm").apply(lambda d: pd.Series(dict(n_ids=d.oa.nunique(), n_orcid_ids=d[d.orcid != ""].oa.nunique(), n_orcids=d[d.orcid != ""].orcid.nunique()))).reset_index()
#     split = set(g.name_norm[g.n_ids >= 2])
#     if len(split) != 3749: raise Halt(f"분할 이름 수 {len(split)} (3749여야 함)")
#     risk = set(pd.read_csv(DATA/"F2_risk_names.csv", encoding="utf-8-sig").name_norm.astype(str))
#     pop = g[g.n_orcid_ids >= 1].copy(); pop["truth"] = pop.n_orcids >= 2; pop["risk"] = pop.name_norm.isin(risk)
#     tp = int((pop.truth & pop.risk).sum()); fp = int((~pop.truth & pop.risk).sum()); fn = int((pop.truth & ~pop.risk).sum()); tn = int((~pop.truth & ~pop.risk).sum())
#     pl, ph = wilson(tp, tp + fp); rl, rh = wilson(tp, tp + fn)
#     cov = dict(risk_names=len(risk), risk_in_F7=int(len(risk & set(g.name_norm))), risk_in_population=int(pop.risk.sum()), population=len(pop), n_truth=int(pop.truth.sum()))
#     out = pd.DataFrame([dict(tp=tp, fp=fp, fn=fn, tn=tn, precision=tp/(tp + fp) if tp + fp else np.nan, p_lo=pl, p_hi=ph, recall=tp/(tp + fn) if tp + fn else np.nan, r_lo=rl, r_hi=rh, **cov)])
#     sp2 = g[(g.n_ids >= 2) & (g.n_orcid_ids >= 2)]; k2 = int((sp2.n_orcids >= 2).sum()); l2, h2 = wilson(k2, len(sp2))
#     aux = pd.DataFrame([dict(n_split_names_with_2plus_orcid_ids=len(sp2), n_distinct_orcids=k2, share_distinct=k2/len(sp2) if len(sp2) else np.nan, lo=l2, hi=h2)])
#     out.to_csv(PA/"PA_B6a_results.csv", index=False, encoding="utf-8-sig"); aux.to_csv(PA/"PA_B6a_split_names_orcid.csv", index=False, encoding="utf-8-sig"); pop.to_csv(PA/"PA_B6a_population.csv", index=False, encoding="utf-8-sig")
#     print("\n[Post-hoc] PA-B6a 위험군 판정의 독립 정답 대비\n", out.round(4).T.to_string(header=False)); print("\n[보조] 분할 이름 중 서로 다른 ORCID 비율\n", aux.round(4).T.to_string(header=False))
#     print("※ ORCID 보유자 편향, 2020+ 한정. 음성은 '증거 없음'. 판정 없이 수치만 보고. S0~S8 판정 불변.")

# if not special("B6a", PLAN, {"fetch": fetch}):
#     run_pa("B6a", PLAN, run, [DATA/"F7_oa_author_ids.csv", DATA/"F2_risk_names.csv", P1/"pa_extra.py"], "위험군 판정의 독립 정답(ORCID) 확보(지침서 B6-1)")
# ```

# ## 3. B6b: 수작업 50쌍 시트 생성과 채점 (`pa_b6b.py`)

# 분석 집합의 이름은 위험군이 이미 제외된 상태입니다. 그래서 이 표본은 "위험군 선별이 놓친 동명이인 비율"을 추정합니다. 한 이름의 서로 다른 시기 논문 두 편이 같은 사람인지를 판정합니다.

# ```python
# # -*- coding: utf-8 -*-
# # PA-B6b: 이름 빈도 상위 구간에서 50쌍을 뽑아 동일인 여부 수작업 판정(시트 생성 + 채점). 미실행.
# # python3 pa_b6b.py freeze | tag | sheet | score     (tag 뒤 git push --follow-tags)
# from pa_common import *
# from pa_extra import *
# SHEET = PA/"PA_B6b_sheet.csv"; FILLED = PA/"PA_B6b_sheet_filled.csv"
# PLAN = dict(
#     version=1, seed=2026, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(지침서 B6-2). 판정 전 시트 생성, 판정 결과 미열람. 연구자 1인 판정. S0~S8 판정 불변.",
#     population="F2_author_paper.csv(위험군 제외 상태, 분석 논문)에서 name_norm별 행 수 >= 10인 이름 중, 연도 차이 5년 이상인 논문 쌍을 만들 수 있는 이름",
#     sampling="이름 50개를 seed 2026으로 비복원 무작위 추출. 이름마다 논문 쌍 1개: 연도 차이 >=5인 논문 쌍 중 무작위 1개. 시트는 이름 순서를 무작위 섞음",
#     sheet="pair_id, name_norm, 논문 A·B의 pid·연도·저널·제목·공저자(최대 6명), same_person(1=동일인, 0=다른 사람, ?=판정 불가), note. 판정 칸 비워서 생성. 소속은 자료에 있는 경우만(없으면 공란)",
#     stats="n_same, n_diff, n_unsure; 미탐지 동명이인 비율 = n_diff/(n_diff+n_same)와 Wilson 95%; 판정 불가를 모두 다른 사람/모두 동일인으로 본 경계",
#     reading="수치와 구간만 보고. 임계값 없음. 연구자 1인 판정이라는 한계를 함께 표기",
#     caveat="이름 단위 표본이며 다작 이름에 치우침(행 10편 이상). 논문 쌍 1개는 이름 전체의 혼합 정도를 과소·과대 추정할 수 있음. 위험군 이름은 모집단에서 이미 빠져 있음. 판정은 제목·공저자·저널 정보에 의존하며 오판 가능",
#     forbidden="결과를 본 뒤 표본·판정 규칙 변경(변경 시 v2 계획으로 분리)")

# def sheet(sp):
#     if SHEET.exists(): raise Halt("시트가 이미 있음(덮어쓰지 않음). 새로 만들려면 v2 계획으로 분리")
#     F2 = pd.read_csv(DATA/"F2_author_paper.csv", encoding="utf-8-sig", low_memory=False); F2["pid"] = F2.pid.astype(str)
#     F1 = pd.read_csv(DATA/"F1_papers.csv", low_memory=False); F1["pid"] = F1.pid.astype(str); title = F1.drop_duplicates("pid").set_index("pid").title
#     rng = np.random.default_rng(int(PLAN["seed"])); cnt = F2.groupby("name_norm").size(); names = sorted(cnt[cnt >= 10].index)
#     pairs = {}
#     for nm in names:
#         d = F2[F2.name_norm == nm].drop_duplicates("pid")[["pid", "year", "venue_short"]].values
#         ok = [(i, j) for i in range(len(d)) for j in range(len(d)) if d[j][1] - d[i][1] >= 5]
#         if ok: pairs[nm] = (d, ok)
#     pool = sorted(pairs); print(f"이름 행 수>=10: {len(names):,} | 5년 이상 쌍 가능: {len(pool):,}")
#     if len(pool) < 50: raise Halt("모집단이 50개 미만")
#     pick = list(rng.choice(pool, 50, replace=False)); rng.shuffle(pick); rows = []
#     co = F2.groupby("pid").name_norm.apply(list)
#     for k, nm in enumerate(pick, 1):
#         d, ok = pairs[nm]; i, j = ok[int(rng.integers(len(ok)))]; r = dict(pair_id=k, name_norm=nm)
#         for tag, x in (("A", d[i]), ("B", d[j])):
#             pid = str(x[0]); r.update({f"pid_{tag}": pid, f"year_{tag}": int(x[1]), f"venue_{tag}": x[2], f"title_{tag}": title.get(pid, ""), f"coauthors_{tag}": "; ".join([n for n in co.get(pid, []) if n != nm][:6])})
#         r.update(same_person="", note=""); rows.append(r)
#     pd.DataFrame(rows).to_csv(SHEET, index=False, encoding="utf-8-sig")
#     cl(f"PA-B6b 시트 생성 | {SHEET.name} 해시 {sha_f(SHEET)[:12]}, 50쌍, 판정 칸 비어 있음 | 본 결과 변경: 없음")
#     print("저장:", SHEET, "→ same_person 칸을 채워 PA_B6b_sheet_filled.csv로 저장하세요(1/0/?)")

# def score(sp):
#     if not (SHEET.exists() and FILLED.exists()): raise Halt("PA_B6b_sheet.csv와 PA_B6b_sheet_filled.csv가 모두 필요")
#     a = pd.read_csv(SHEET, encoding="utf-8-sig"); b = pd.read_csv(FILLED, encoding="utf-8-sig")
#     key = ["pair_id", "name_norm", "pid_A", "pid_B"]
#     if len(b) != 50 or not a[key].astype(str).equals(b[key].astype(str)): raise Halt("채워진 시트가 원본 시트와 다름(행·키 변경 금지)")
#     v = b.same_person.astype(str).str.strip().str.replace(".0", "", regex=False)
#     if not v.isin(["1", "0", "?"]).all(): raise Halt("same_person은 1, 0, ? 중 하나여야 함(빈 칸 있음)")
#     ns, nd, nu = int((v == "1").sum()), int((v == "0").sum()), int((v == "?").sum()); n = ns + nd; lo, hi = wilson(nd, n)
#     lo_b, _ = wilson(nd, n + nu); _, hi_b = wilson(nd + nu, n + nu)
#     out = pd.DataFrame([dict(n_same=ns, n_diff=nd, n_unsure=nu, undetected_homonym_rate=nd/n if n else np.nan, lo=lo, hi=hi, bound_lo_if_unsure_same=lo_b, bound_hi_if_unsure_diff=hi_b)])
#     out.to_csv(PA/"PA_B6b_results.csv", index=False, encoding="utf-8-sig"); print("\n[Post-hoc] PA-B6b\n", out.round(4).T.to_string(header=False))
#     print("※ 연구자 1인 판정, 다작 이름 표본. 수치만 보고. S0~S8 판정 불변.")
#     cl(f"PA-B6b 채점 | 정의: PA_B6b_PLAN {sp['sha256'][:12]} | 결과: PA_B6b_results.csv, 채워진 시트 해시 {sha_f(FILLED)[:12]} | 본 결과 변경 여부: 없음")

# if not special("B6b", PLAN, {"sheet": sheet, "score": score}):
#     run_pa("B6b", PLAN, lambda q, s: print("이 항목은 sheet → (수작업) → score 모드를 쓰세요"), [DATA/"F2_author_paper.csv", P1/"pa_extra.py"], "위험군 선별이 놓친 동명이인 비율의 수작업 추정(지침서 B6-2)")
# ```

# ## 4. B8b: 사전 v3과 새 25편 이중 코딩 (`pa_b8b.py`)

# 먼저 연구자가 `09_PA/PA_B8b_dict_v3.json`에 네 도메인의 정규식을 작성합니다. 예: `{"electrification_env": "...", "freight": "...", "active_urban": "...", "infrastructure": "..."}`. 사전 v3은 S8 사전 v2(`S8_0_PREREG_v2.json`)와 별개이므로 이름을 구분합니다. 판독은 통과·실패가 아니라 점추정과 Wilson 구간입니다. n=25에서 통과 판정은 우연의 영향이 크기 때문입니다.

# ```python
# # -*- coding: utf-8 -*-
# # PA-B8b: 미달 4개 도메인의 사전 v3으로 새 무작위 25편씩 뽑아 이중 코딩 시트 생성 + precision 채점. 판정은 하지 않고 구간만 보고. 미실행.
# # 선행: 09_PA/PA_B8b_dict_v3.json 작성({"도메인": "정규식"} 4개). python3 pa_b8b.py freeze | tag | sheet | score
# from pa_common import *
# from pa_extra import *
# DICT = PA/"PA_B8b_dict_v3.json"; SHEET = PA/"PA_B8b_sheet.csv"; FILLED = PA/"PA_B8b_sheet_filled.csv"
# FAILS = ["electrification_env", "freight", "active_urban", "infrastructure"]
# D3 = json.loads(DICT.read_text(encoding="utf-8")) if DICT.exists() else None
# PLAN = dict(
#     version=1, seed=2026, parent="PA_0_PLAN_v2.json",
#     status="사후 명세(지침서 B8-2). 미달 4개 도메인 precision(0.76~0.84, n=25)과 오탐 패턴(예: active_urban의 accessibility) 열람 후 사전 v3 작성. 새 표본 결과 미열람. 사후 개정 사전이라 본 결과와 섞지 않음. S0~S8 판정 불변.",
#     dict_v3=D3, domains=FAILS,
#     sampling="분석 집합(F1에서 EXCLUDE_PIDS_FROZEN 제외, year>=2007, title 있음) 중 v3 정규식(re.I)이 제목에 매칭되는 논문에서 도메인별 25편을 seed 2026으로 비복원 추출. 이전 S8 검증 표본에 있던 pid는 제외(표본 파일은 08_S8/*sample*.csv의 pid 열). 시트 순서는 섞음",
#     coding="coder1, coder2가 서로 보지 않고 각 논문 제목이 해당 도메인의 S8 정의에 속하는지 1/0으로 코딩. 불일치는 final 칸에 합의로 기록(일치하면 final도 같은 값). 코더 구성(저자 1인+제2코더)은 note에 기록",
#     stats="도메인별 precision=mean(final)과 Wilson 95%, Cohen kappa(coder1, coder2), 불일치 수, 이전 v2 precision 병기. 도메인 합산도 병기",
#     reading="통과/실패 판정 없음. 수치와 구간만 보고하며 S8 판정(2/6 통과)을 바꾸지 않음. 합산과 도메인별을 합치지 않음. 'v3으로 개정한 사전'임을 모든 표에 표기",
#     caveat="n=25는 구간이 넓어 0.85 기준과 구분이 어려움(참 precision 0.80이어도 22편 이상 나올 확률 약 23%). 오탐 패턴을 보고 만든 사전이라 정밀도에 낙관 편향. recall은 추정하지 않음. 코더 독립성 한계 가능",
#     forbidden="결과를 본 뒤 사전·표본·판독 변경(변경 시 v2 계획으로 분리)")

# def load_pool():
#     RXv = {d: re.compile(D3[d], re.I) for d in FAILS}
#     F1 = pd.read_csv(DATA/"F1_papers.csv", low_memory=False); F1["pid"] = F1.pid.astype(str)
#     exf = S0/"EXCLUDE_PIDS_FROZEN.csv"; log = pd.read_csv(S0/"FREEZE_LOG.csv"); last = log[log.file == exf.name].iloc[-1]
#     if sha_f(exf) != last.sha256: raise Halt("EXCLUDE_PIDS_FROZEN.csv 해시 불일치")
#     F = F1[~F1.pid.isin(set(pd.read_csv(exf, dtype=str, encoding="utf-8-sig").pid))].copy(); F = F[(F.year >= 2007) & F.title.notna()]
#     return F, {d: F.title.str.contains(rx).values for d, rx in RXv.items()}

# def sheet(sp):
#     if SHEET.exists(): raise Halt("시트가 이미 있음(덮어쓰지 않음)")
#     F, pos = load_pool(); prev = set()
#     for f in sorted(S8.glob("*sample*.csv")):
#         d = pd.read_csv(f, dtype={"pid": str}, encoding="utf-8-sig")
#         if "pid" in d.columns: prev |= set(d.pid); print("이전 표본 제외:", f.name, len(d))
#     if not prev and "noprev" not in sys.argv: raise Halt("08_S8에서 이전 검증 표본(*sample*.csv, pid 열)을 찾지 못함. 위치 확인 또는 'sheet noprev'")
#     rng = np.random.default_rng(int(PLAN["seed"])); rows = []
#     for d in FAILS:
#         P = F[pos[d] & ~F.pid.isin(prev)]; print(f"  {d}: v3 양성 {int(pos[d].sum()):,}, 제외 후 {len(P):,}")
#         if len(P) < 25: raise Halt(f"{d}: 후보 25편 미만")
#         for _, r in P.iloc[rng.choice(len(P), 25, replace=False)].iterrows(): rows.append(dict(domain=d, pid=r.pid, year=int(r.year), title=r.title, coder1="", coder2="", final="", note=""))
#     S_ = pd.DataFrame(rows).sample(frac=1, random_state=int(PLAN["seed"])).reset_index(drop=True); S_.insert(0, "item_id", range(1, len(S_) + 1)); S_.to_csv(SHEET, index=False, encoding="utf-8-sig")
#     cl(f"PA-B8b 시트 생성 | {SHEET.name} 해시 {sha_f(SHEET)[:12]}, 도메인별 25편(총 100), 코딩 칸 비어 있음, 이전 표본 제외 {len(prev)}편 | 본 결과 변경: 없음")
#     print("저장:", SHEET, "→ coder1·coder2를 각자 채우고 final 합의 후 PA_B8b_sheet_filled.csv로 저장")

# def score(sp):
#     if not (SHEET.exists() and FILLED.exists()): raise Halt("PA_B8b_sheet.csv와 PA_B8b_sheet_filled.csv가 모두 필요")
#     a = pd.read_csv(SHEET, encoding="utf-8-sig"); b = pd.read_csv(FILLED, encoding="utf-8-sig"); key = ["item_id", "domain", "pid"]
#     if len(b) != 100 or not a[key].astype(str).equals(b[key].astype(str)): raise Halt("채워진 시트가 원본과 다름")
#     for c in ["coder1", "coder2", "final"]:
#         b[c] = pd.to_numeric(b[c], errors="coerce")
#         if not b[c].isin([0, 1]).all(): raise Halt(f"{c}는 0/1이어야 함(빈 칸 있음)")
#     if not (b.final[b.coder1 == b.coder2] == b.coder1[b.coder1 == b.coder2]).all(): raise Halt("일치한 항목의 final이 코더 값과 다름")
#     prev = pd.read_csv(S8/"domain_precision_recall.csv").set_index("domain"); rows = []
#     def kappa(x, y):
#         po = (x == y).mean(); pe = x.mean()*y.mean() + (1 - x.mean())*(1 - y.mean()); return (po - pe)/(1 - pe) if pe < 1 else np.nan
#     for d, g in list(b.groupby("domain")) + [("ALL", b)]:
#         k, n = int(g.final.sum()), len(g); lo, hi = wilson(k, n)
#         rows.append(dict(domain=d, n=n, precision_v3=k/n, lo=lo, hi=hi, kappa=kappa(g.coder1, g.coder2), n_disagree=int((g.coder1 != g.coder2).sum()), precision_v2=float(prev.loc[d, "precision"]) if d in prev.index else np.nan))
#     out = pd.DataFrame(rows); out.to_csv(PA/"PA_B8b_results.csv", index=False, encoding="utf-8-sig")
#     print("\n[Post-hoc] PA-B8b (사후 개정 사전 v3, 통과/실패 판정 없음)\n", out.round(3).to_string(index=False))
#     print("※ n=25는 구간이 넓음. 오탐 패턴을 본 사전이라 낙관 편향. S8 판정(2/6) 불변.")
#     cl(f"PA-B8b 채점 | 정의: PA_B8b_PLAN {sp['sha256'][:12]} | 결과: PA_B8b_results.csv, 채워진 시트 해시 {sha_f(FILLED)[:12]} | 본 결과 변경 여부: 없음")

# if len(sys.argv) > 1 and sys.argv[1] == "freeze" and D3 is None: print("⛔ 중단: 09_PA/PA_B8b_dict_v3.json을 먼저 작성하세요"); sys.exit()
# if not special("B8b", PLAN, {"sheet": sheet, "score": score}):
#     run_pa("B8b", PLAN, lambda q, s: print("이 항목은 sheet → (수작업 코딩) → score 모드를 쓰세요"), [DICT, S8/"domain_precision_recall.csv", P1/"pa_extra.py"], "사전 v3(사후 개정)과 새 25편 이중 코딩(지침서 B8-2)")
# ```

# ## 5. 실행 순서

# 한 블록씩 실행하고, `⛔`나 오류가 나면 거기서 멈추고 출력만 붙여 주세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 -m py_compile pa_extra.py pa_b4d.py pa_b6a.py pa_b6b.py pa_b8b.py && echo "문법 OK"
# ```

# **B4d** (무작위가 있으므로 QUICK 먼저, 본 실행은 오래 걸리니 `nohup`)
# ```bash
# python3 pa_b4d.py freeze
# python3 pa_b4d.py tag && cd .. && git push --follow-tags && cd paper1
# python3 pa_b4d.py run quick 2>&1 | tee 09_PA/PA_B4d_quick_log.txt
# nohup python3 -u pa_b4d.py run > 09_PA/PA_B4d_full_log.txt 2>&1 &
# ```

# **B6a** (CPU를 쓰는 B4d가 끝난 뒤에 돌리세요. API 호출은 가볍습니다)
# ```bash
# python3 pa_b6a.py freeze
# python3 pa_b6a.py tag && cd .. && git push --follow-tags && cd paper1
# python3 pa_b6a.py fetch
# python3 pa_b6a.py run 2>&1 | tee 09_PA/PA_B6a_full_log.txt
# ```

# **B6b** (동결 후 시트 생성, 판정은 직접)
# ```bash
# python3 pa_b6b.py freeze
# python3 pa_b6b.py tag && cd .. && git push --follow-tags && cd paper1
# python3 pa_b6b.py sheet
# # 09_PA/PA_B6b_sheet.csv의 same_person을 채워 PA_B6b_sheet_filled.csv로 저장
# python3 pa_b6b.py score 2>&1 | tee 09_PA/PA_B6b_full_log.txt
# ```

# **B8b** (먼저 `PA_B8b_dict_v3.json` 작성)
# ```bash
# python3 pa_b8b.py freeze
# python3 pa_b8b.py tag && cd .. && git push --follow-tags && cd paper1
# python3 pa_b8b.py sheet
# # coder1·coder2를 각자 채우고 final 합의 후 PA_B8b_sheet_filled.csv로 저장
# python3 pa_b8b.py score 2>&1 | tee 09_PA/PA_B8b_full_log.txt
# ```

# ## 6. 주의할 점

# - **판정 정보를 보고 사전을 고치지 마세요.** B8b에서 시트를 만든 뒤 정규식이나 표본을 바꾸면 v2 계획으로 분리해야 합니다.
# - **시트 생성 직후 해시가 changelog에 기록됩니다.** 판정 전의 빈 시트라는 증거가 됩니다. 채워진 시트도 점수 단계에서 해시가 기록됩니다.
# - **B6a의 한계**: ORCID 보유 이름만 모집단이라 연구 활동이 활발한 저자에 치우치고, F7이 2020년 이후만 덮습니다. 결과는 "독립 정답 1개"로만 쓰고 순환성이 해소됐다고 쓰면 안 됩니다.
# - **B6b의 표본**: 위험군이 이미 빠진 분석 집합에서 뽑으므로 "선별이 놓친 비율"입니다. 연구자 1인 판정이라는 한계를 같이 적으세요.
# - **B8b**: 0.85 통과 판정에 쓰지 마세요. 참 precision이 0.80이어도 25편 중 22편 이상이 나올 확률이 약 23%입니다.
# - **확인하지 못한 가정**: `jaccard_p10`·`topic` 열 이름, `F2_risk_names.csv`의 `name_norm`, `08_S8/*sample*.csv`의 존재, OpenAlex `filter=openalex:` 문법입니다. 어긋나면 해당 단계가 중단 메시지를 냅니다. 그때는 규칙을 임의로 고치지 말고 메시지만 보내 주세요.

In [ ]:
# # PA-B4d 판독

# 정상 종료입니다. CSV 3개와 `[동결 판독]` 줄이 있고 오류는 없습니다. 앞서 "너무 빨리 끝났다"고 한 걱정은 기우였습니다. 동결 규칙대로 S5와 S6을 따로 읽습니다.

# ## 1. S5 (제외 후 K=134)

# | 표본 | 통과 여부 | 변화점 | S5 원래 변화점 | ±1년 유지 |
# |---|---|---|---|---|
# | all | 실패(N2 p=0.74), S5와 동일 | 2014, 2020 | 2012, 2020 | 아니오 (2012→2014) |
# | fixed14 | 실패(p=0.97), 동일 | 2012, 2020 | 2008, 2015, 2020 | 아니오 (2008, 2015 소실) |
# | within | 실패(p=1.00), 동일 | 2015, 2020 | 2009, 2015, 2020 | 아니오 (2009 소실) |

# - **(a) 통과 판정은 세 표본 모두 같습니다.** N1은 0.002로 통과하고 N2는 실패하는 구조가 그대로입니다.
# - **(b) 변화점 유지는 세 표본 모두 "전부 유지"가 아닙니다.** 그래도 **2020은 세 표본 모두 남았고**, within에서는 2015도 남았습니다. 사라지거나 이동한 것은 2008, 2009, 2012 같은 앞쪽 변화점입니다.
# - 앞쪽 변화점은 A1에서 검정력이 낮은 사각지대였습니다. 이번 변화는 그 위치에서 생겼습니다.
# - 변화점 개수가 줄었는데, 이것이 토픽 제외의 효과인지 벌점 설정 효과인지는 분리하지 못했습니다. 벌점이 `(K−1)`에 비례하고 φ는 S5 값으로 고정했기 때문입니다. K=145였던 B4에서는 변화점이 모두 유지됐으므로, 벌점 효과일 가능성이 있지만 확인하지 않은 가설입니다.

# ## 2. S6 (entry_minus_adapt)

# | 쌍 | 원본 | 제외 후 | 변화 | 규칙 판독 |
# |---|---|---|---|---|
# | P0004→P0509 | −0.0055 | +0.0406 | +0.046 | **민감**(부호 바뀜, 0.03 초과) |
# | P0509→P1014 | 0.0271 | 0.0405 | +0.013 | 둔감 |
# | P1014→P1519 | 0.0461 | 0.0665 | +0.020 | 둔감 |
# | P1519→P2025 | 0.1128 | 0.1239 | +0.011 | 둔감 |

# - **네 쌍 모두 변화가 양수입니다.** 불안정한 토픽을 빼면 진입 우세가 모든 쌍에서 커집니다. 약화 방향이 아니라 강화 방향이므로, 이 점은 C1의 반전과 성격이 다릅니다.
# - 마지막 쌍은 0.113에서 0.124로 거의 그대로이고, 구간 [0.006, 0.017]은 0 위입니다.
# - 첫 쌍은 규칙상 "민감"입니다. 원래 값이 0 근처이고 이동이 0.046으로 큽니다. 구간 [0.011, 0.056]도 0 위입니다. B4의 첫 쌍(+0.009)보다 훨씬 큽니다. 이 이동에서 F1·F3 실패 4개의 몫은 B4 기준으로 약 0.009뿐이라, **나머지 11개 토픽이 대부분을 만든 것**으로 읽힙니다.
# - A2c에서 첫 쌍이 통과하지 못한 것은 **토픽 구성에 의존하는 결과**일 수 있습니다. 이것이 제외된 토픽이 불안정하다는 점과 관련 있는지는 확인하지 못했습니다.

# ## 3. 쓸 때의 주의

# - B4d의 제외 15개에는 B4에서 뺀 4개가 모두 들어 있습니다. 독립적인 두 번째 확인이 아니라 **제외 범위를 넓힌 확장**입니다.
# - 제외 기준은 임의의 10%(동결 사양)입니다. 등급이나 토픽 안정성 지표의 타당성은 검증하지 않았습니다.
# - **쓸 수 있는 문구**
#   > "부트스트랩 Jaccard 하위 10% 토픽(15개)을 제외해도 S5의 통과 판정은 세 표본 모두 같았고 2020 변화점은 모두 유지됐으나, 2008~2012 부근 변화점은 유지되지 않았다. 진입−적응은 모든 구간에서 커졌고 마지막 구간은 0.113→0.124였다. 첫 구간은 0 근처의 부호가 바뀌어(−0.006→+0.041) 규칙상 민감으로 보고한다."
# - **쓰면 안 되는 문구**: "등급·안정성 기준에 둔감하다"처럼 첫 쌍과 변화점 불일치를 가리는 표현.

# ## 4. 기록과 커밋

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B4d 결과 해석 메모(S5 B=500, S6 B=2000, 제외 15개·K=134): S5 통과 판정 3표본 모두 동일(전부 실패, p_N2 0.74/0.97/1.00), 2020 변화점은 3표본 모두 유지, 그러나 전체 변화점 +-1년 유지는 3표본 모두 아님(all 2012->2014, fixed14 2008·2015 소실, within 2009 소실). 벌점이 K에 비례하고 phi 고정이라 벌점 효과 가능성(미확인). S6 entry_minus_adapt 변화 +0.046/+0.013/+0.020/+0.011, 4쌍 모두 양(진입 우세 강화 방향), 규칙상 첫 쌍만 민감(-0.0055->+0.0406, 부호 바뀜, 구간 0 위), 마지막 쌍 0.113->0.124. 제외 15개에 B4의 F1·F3 실패 4개 포함(확장이며 독립 확인 아님), 첫 쌍 이동의 대부분은 나머지 11개에서 발생. 제외 기준 10%는 임의, 안정성 지표 타당성 미검증 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_B4d_*.csv paper1/09_PA/PA_B4d_full_log.txt paper1/09_PA/PA_CHANGELOG.md
# git status --short | head -n 8
# git commit -m "PA-B4d results" && git tag -a PA-B4d-results-2026-10-21 -m "PA-B4d outputs, seen" && git push --follow-tags
# ```

# `git status`에 스테이징한 것 외의 파일이 보이면 커밋 전에 알려 주세요.

# ## 5. 다음

# 1. 위 기록을 올린 뒤 `pa_b6a_fetch2.py`의 진행 상황을 확인하세요(`tail -n 5 09_PA/PA_B6a_fetch2_log.txt`). 끝났다면 `python3 pa_b6a.py run` 로그를 붙여 주세요.
# 2. B6b와 B8b는 수작업이 필요합니다. 시간이 되면 `freeze`부터 시작하고, B8b는 `PA_B8b_dict_v3.json`을 먼저 작성해야 합니다.
# 3. 통합 문구표와 약화·반전 목록에 B4d의 두 항목(첫 쌍 민감, 앞쪽 변화점 불일치)을 추가해야 합니다.

In [ ]:
# # 논문 1 사후분석(PA) 진행 현황 (2026-10-21 기준)

# ## 1. 지금까지 한 것

# ### 1-1. 원칙 (전 단계 공통)
# - 계획은 실행 전에 해시와 태그로 동결하고 GitHub에 푸시했습니다.
# - 결과는 방향과 무관하게 보고했고, S0~S8 판정은 어떤 항목에서도 바꾸지 않았습니다.
# - 오류와 정정은 삭제하지 않고 changelog에 줄로 남겼습니다.

# ### 1-2. 완료한 분석과 결과

# | PA | 동결 판독 | 핵심 수치 | 쓸 수 있는 말 / 주의 |
# |---|---|---|---|
# | PA-0 | 대부분 완료 | 해시 152건 일치 | S4 코드는 복구본, `oa_raw` 외부 백업 미완료 |
# | A1 | run 1 무효, run 2는 all·within 강화, fixed14 곡선만 보고 | δ=0 기각률 0.035/0.075/0.065 | 2020 부근은 초과분 크기 점프를 탐지했을 가능성이 높음. 2008~2012는 사각지대 |
# | B2 | 3표본 유지 | J 비율 0.998~1.006 | 하향 추출이 20%뿐이라 약한 시험 |
# | B2b | 단독 제외는 유지, 동시 제외는 약화 | 상위 5저널+토픽 71 제외 시 J 0.74~0.83 | 2020은 단일 원인이 아닌 겹친 가속 |
# | B3a·b·c | 핵심 7·중간 10·특이적 14 | 핵심 진입 0.458 대 적응 0.252 | 정의 구조의 부산물, 독립 합의 증거 아님 |
# | B4 | S5 둔감, S6 3쌍 둔감 | 첫 쌍만 부호 변화 | 제외 4개뿐인 약한 시험 |
# | B4b | 판독 1 둔감, 판독 2 규칙상 민감 | 안정성 +0.027/+0.030 | 임계 0.03이 토픽 1개 규모 |
# | **B4d** | S5 통과 판정 동일, 변화점 전부 유지는 아님. S6 첫 쌍만 민감 | 진입−적응 변화 +0.046/+0.013/+0.020/+0.011 | 아래 별도 설명 |
# | B5 | 부호 반전 0 | 안정성 0.66~0.77 | "0.70 전후에서 움직임" |
# | A2c | 3쌍 유지(첫 쌍 제외) | 두 구간 방식 모두 확실한 쌍은 마지막뿐 | excess 구간은 교정 불충분 |
# | A2b | s6 3, y5 3, y2 0(뒤집힘) | y2 차이 −0.15~−0.16 | "신규 저자 주도"가 아니라 "진입(신규+복귀) 주도" |
# | A2a | 4쌍 모두 둔감 | 변화 ≤0.011 | ID는 2020+만 있어 ID 재판별 불가 |
# | **C1** | 3개 시나리오 모두 유지 안 됨 | +0.113 → −0.101(a), −0.110, −0.024(b) | 진입 우세와 중국 소속 유입은 분리 불가 |
# | C2 | 20칸 중 일치 11 | 중국 몫 4기간 일치, 미국·상위5 +0.01~0.03 | 수준은 다름 |
# | A3 | 방향 유지 | 알려진 국가 기준 top5 평탄(0.64→0.65) | HHI 반등은 중국 몫과 연동 |
# | B7a·b·c | 모두 유지 | big 50/50 | 민감도이며 경계가 아님 |
# | B9a·b | HIC 비관찰, UMC는 하한 5에서만 관찰 | safety·road_deaths 편 ρ≈0.24 | LIC·LMC는 자료상 불가 |
# | B8a | 참고용 | 8쌍 중 7쌍 HIC 과잉·LIC/LMC 과소 | 본문 판정 2/6 불변 |

# **B4d 상세 (방금 끝난 항목, 커밋 `0c4902f`, 태그 `PA-B4d-results-2026-10-21`)**
# - Jaccard 하위 10% 15개를 제외했습니다(K=134). 이 15개에는 B4의 F1·F3 실패 4개가 모두 들어 있어 독립 확인이 아니라 확장입니다.
# - S5: 통과 판정은 세 표본 모두 같습니다(전부 실패). 2020 변화점은 세 표본에서 유지됐지만 2008, 2009, 2012 부근은 이동하거나 사라졌습니다. 벌점 효과 가능성은 확인하지 못했습니다.
# - S6: 진입−적응이 네 쌍 모두 커졌습니다. 마지막 쌍은 0.113→0.124이고, 첫 쌍은 −0.006→+0.041로 부호가 바뀌어 규칙상 "민감"입니다.

# ### 1-3. 종합 결론
# - **쓸 수 있는 것**
#   - 변화는 점프보다 연속적 재배치로 서술하는 편이 자료에 부합합니다.
#   - 2020 부근은 COVID와 기술 주제가 겹친 가속입니다.
#   - 진입 우세는 마지막 구간에서 가장 뚜렷합니다.
#   - 알려진 국가 기준 top5 몫은 평탄합니다.
#   - S4의 방향은 결측 가정에서도 유지됩니다.
# - **쓰면 안 되는 것**
#   - 체제 전환 부재 확정, 2020이 아티팩트가 아니라는 확정
#   - 신규 저자가 모든 구간을 주도했다는 문장
#   - 진입 주도가 중국 유입과 무관하다는 문장
#   - ID 기반 일치, 인과 해석
# - **구조상 가장 큰 변화**: 진입 중심 갱신과 중국 몫 증가는 독립된 두 기여가 아니라 한 구조의 두 측면으로 묶어야 합니다.

# ### 1-4. 진행 중인 것

# | 항목 | 상태 |
# |---|---|
# | **B6a (ORCID 독립 정답)** | 동결(`764956a259fd`)과 푸시 완료. 수집이 HTTP 429로 중단됐고 캐시에 50,000개 ID가 있습니다(남은 약 21,702개). 재개 스크립트 `pa_b6a_fetch2.py`는 **아직 저장·실행하지 않았습니다.** |
# | **changelog 보충** | B6a 중단 기록과 B4d QUICK 열람 줄이 **아직 없습니다.** 소급 기록용 명령은 직전에 드렸습니다. |

# ## 2. 앞으로 해야 할 것

# ### 2-1. 지금 바로 (순서대로)
# 1. `pa_b6a_fetch2.py`를 저장하고 `py_compile` 후 `OA_MAILTO`를 넣어 실행합니다.
# 2. 소급 changelog 두 줄을 올립니다. 시각이 늦게 찍힌다는 문구를 줄에 포함했습니다.
# 3. 수집이 끝나면(캐시 71,703행 안팎) `python3 pa_b6a.py run`을 돌리고, 캐시 `sha256sum`과 함께 로그를 붙여 주세요. 해시 기록 줄은 그때 드립니다.

# ### 2-2. 사람 손이 필요한 분석

# | 항목 | 내용 | 선행 작업 |
# |---|---|---|
# | **B6b** | 이름 50개의 논문 쌍이 동일인인지 수작업 판정 | `freeze → tag → push → sheet`, 시트의 `same_person`에 1/0/?를 채워 `PA_B6b_sheet_filled.csv`로 저장, `score` |
# | **B8b** | 사전 v3, 도메인별 새 25편(총 100편) 이중 코딩 | 먼저 `09_PA/PA_B8b_dict_v3.json`에 네 도메인 정규식 작성 후 동결. 코더 2명 필요 |

# - B6b는 위험군이 이미 빠진 집합에서 뽑으므로 "선별이 놓친 동명이인 비율"을 추정합니다. 연구자 1인 판정이라는 한계를 표기합니다.
# - B8b는 통과/실패 판정이 아니라 점추정과 Wilson 구간만 보고합니다. n=25는 구간이 넓습니다.
# - 시간이 안 되면 "미수행, 한계로 명시"로 확정합니다. 이번 주 안에 정하세요.

# ### 2-3. 미수행으로 확정될 항목 (부록에 사유 기재)
# - B4c: 임베딩 파일을 찾지 못했습니다.
# - A2의 ID 기반 재판별: F7이 2020년 이후 논문만 덮습니다.
# - B9의 LIC·LMC 소득군 내 분석: 질량 하한 때문에 자료상 불가입니다.

# ### 2-4. 정리·문서 작업
# 1. **A4 부록 표**: `pa_a4.py` 실행 후 `MANUAL` 항목(A1, PA-0, B7b 등)을 직접 검토합니다. B4d, B6a의 중단·소급 기록도 추가합니다.
# 2. **통합 문구표와 공개 목록**: 약화·반전된 결과를 한계 절과 부록에 먼저 공개합니다.
#    - C1(진입 우세 반전), A2b y2 역전, A2c, B4 첫 쌍, B4b 판독 2
#    - **B4d(첫 쌍 민감, 앞쪽 변화점 불일치)**
#    - B9a·b, B8a, B7c(2009 변화점 이동), B2·B3의 약한 시험
# 3. **재현성 잔여**
#    - `oa_raw` 외부 백업: 매체 연결 후 쓰기 권한 확인, 복사, SHA-256 비교
#    - GitHub 저장소 Private 확인
#    - `S8.5` 노트북 최종 커밋(`Atlas_*.ipynb` 등은 paper1 소속인지 확인 후 결정)

# ### 2-5. 투고 준비
# - S9: 후보 풀·추출 규칙 동결
# - S10: 저널 선례 3편 이상, 심사 대응표, 생성형 AI 사용 선언
# - 원고: 제목에서 "체제 전환"을 빼고, 진입 중심 갱신과 중국 소속 유입을 한 구조로 서술합니다. 모든 표에 `[Post-hoc]`, 영문 post-hoc 문구 삽입, 기각·약화된 PA 목록을 부록에 둡니다.
# - **투고 후보 저널이 아직 정해지지 않았습니다.** 구성안과 초록 초안의 병목입니다.

# ## 3. 제 쪽 정정 기록 (부록용)
# - B4b 임계값(0.03)이 토픽 1개 규모라는 점을 사전에 놓쳤습니다.
# - B9b에서 UMC가 0을 배제할 확률을 약 10%로 봤는데 틀렸습니다.
# - B9b·B3b 생략을 침묵 동의로 처리했습니다.
# - "S3에 Jaccard 중앙값만 있다"는 설명이 틀렸습니다(`jaccard_p10` 열이 있음).
# - B6a 수집은 대기 시간이 부족한 동결 코드로 429를 만났습니다.
# - 직전 명령 묶음의 changelog 줄은 실행되지 않아 소급 기록이 필요합니다.

# ## 4. 지금 필요한 답
# 1. `fetch2` 실행 결과와 이후 `pa_b6a.py run` 로그
# 2. B6b·B8b를 할 수 있는지(없으면 이번 주 내 "미수행"으로 확정)
# 3. 투고 후보 저널

# 가장 가까운 다음 한 걸음은 `pa_b6a_fetch2.py` 저장과 실행입니다.

In [ ]:
# 질문이 따로 없어서, 마지막 정리에 남아 있던 다음 단계인 **B6a 수집 재개**를 정리했습니다. 동결된 `pa_b6a.py`는 건드리지 않고, 같은 캐시 CSV에 이어 쓰는 별도 스크립트입니다.

# ## 1. `paper1/pa_b6a_fetch2.py`

# ```python
# # -*- coding: utf-8 -*-
# # PA-B6a 수집 재개(429 대응). 동결된 pa_b6a.py·계획은 수정하지 않음. 캐시 CSV에 이어 쓰기만 함.
# import json, time, urllib.request, urllib.parse, urllib.error, sys
# from pathlib import Path
# import pandas as pd
# ROOT = Path.home()/"Research/Minnesota_Transport"; PA = ROOT/"paper1/09_PA"; DATA = ROOT/"analysis_data/final"
# CACHE = PA/"PA_B6a_orcid_cache.csv"
# MAILTO = sys.argv[1] if len(sys.argv) > 1 else ""     # 사용: python3 pa_b6a_fetch2.py you@example.com
# if not MAILTO: sys.exit("⛔ 이메일 인자 필요 (OpenAlex polite pool)")
# f7 = pd.read_csv(DATA/"F7_oa_author_ids.csv", encoding="utf-8-sig", low_memory=False)
# ids = sorted(set(f7.oa_id.dropna().astype(str).str.replace("https://openalex.org/", "", regex=False)))
# done = set(pd.read_csv(CACHE).oa_id) if CACHE.exists() else set()
# todo = [i for i in ids if i not in done]; print(f"고유 {len(ids):,} | 캐시 {len(done):,} | 남음 {len(todo):,}", flush=True)
# wait = 1.0
# for k in range(0, len(todo), 50):
#     b = todo[k:k+50]
#     url = "https://api.openalex.org/authors?" + urllib.parse.urlencode({"filter": "openalex:" + "|".join(b), "select": "id,orcid", "per-page": 50, "mailto": MAILTO})
#     res = None
#     for att in range(8):
#         try: res = json.load(urllib.request.urlopen(url, timeout=60)); wait = max(1.0, wait*0.9); break
#         except urllib.error.HTTPError as e:
#             if e.code == 429: wait = min(wait*2, 120); print(f"429, {wait:.0f}s 대기", flush=True); time.sleep(wait)
#             else: print("HTTP", e.code, flush=True); time.sleep(5)
#         except Exception as e: print("오류", str(e)[:60], flush=True); time.sleep(5)
#     if res is None: sys.exit("⛔ 연속 실패: 캐시 보존, 잠시 후 재실행하면 이어서 진행")
#     got = {r["id"].replace("https://openalex.org/", ""): (r.get("orcid") or "") for r in res.get("results", [])}
#     pd.DataFrame([dict(oa_id=i, orcid=got.get(i, ""), found=i in got) for i in b]).to_csv(CACHE, mode="a", header=not CACHE.exists(), index=False, encoding="utf-8-sig")
#     if (k//50) % 20 == 0: print(f"  {k+len(b):,}/{len(todo):,}", flush=True)
#     time.sleep(wait)
# c = pd.read_csv(CACHE); print("완료", len(c), "행 | ORCID 보유", int((c.orcid.fillna('') != '').sum()))
# ```

# 실행:

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 -m py_compile pa_b6a_fetch2.py && echo "문법 OK"
# nohup python3 -u pa_b6a_fetch2.py 본인이메일 > 09_PA/PA_B6a_fetch2_log.txt 2>&1 &
# sleep 5; tail -f 09_PA/PA_B6a_fetch2_log.txt
# ```

# ## 2. 소급 changelog 기록

# 아직 올라가지 않은 두 줄입니다. 시각은 실제 기록 시점이므로 소급이라고 명시했습니다.

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B6a 수집 중단(소급 기록): pa_b6a.py fetch가 OpenAlex HTTP 429로 중단, 캐시에 50,000개 ID 보존(남은 약 21,702개). 동결 코드의 대기 시간 부족이 원인. 동결 파일은 수정하지 않고 pa_b6a_fetch2.py(백오프, 같은 캐시에 이어 쓰기)로 재개. 결과 미열람 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B4d QUICK 열람(소급 기록): 재현 점검과 제외 토픽 수(15), F1·F3 겹침, 등급 분포 열람. 결과 통계는 미열람, 본 실행 후 해석 메모 별도 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/pa_b6a_fetch2.py paper1/09_PA/PA_CHANGELOG.md && git commit -m "PA-B6a: resume fetch (backoff); retroactive notes" && git push
# ```

# ## 3. 수집이 끝난 뒤

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# pgrep -f pa_b6a_fetch2.py || echo "종료 확인"
# sha256sum 09_PA/PA_B6a_orcid_cache.csv
# python3 pa_b6a.py run 2>&1 | tee 09_PA/PA_B6a_full_log.txt
# ```

# 캐시 해시와 로그를 붙여 주시면 동결 규칙대로 읽겠습니다. 한 가지 주의할 점은, 캐시에서 `found=False`인 ID 수를 먼저 확인하는 것입니다. 조회 누락이 많으면 ORCID 부재와 구분되지 않아 모집단이 줄어듭니다.

# ## 4. 여전히 필요한 결정

# 1. B6b(수작업 50쌍)와 B8b(사전 v3, 새 25편 이중 코딩)를 할 수 있는지, 이번 주 안에 정하고 못 하면 "미수행, 한계로 명시"로 확정합니다.
# 2. 투고 후보 저널을 정해야 S9·S10과 구성안 작성으로 넘어갈 수 있습니다.

In [ ]:
# # 논문 1 사후분석(PA) 현황 정리 (2026-10-21 기준)

# ## 1. 계획한 분석

# 지침서(10-19)의 14개 항목이고, 핵심은 A1·A2·B2입니다. 원칙은 실행 전 해시·태그 동결, GitHub 푸시, 결과 전부 보고, S0~S8 판정 불변입니다.

# | 구분 | 항목 | 질문 |
# |---|---|---|
# | 재현성 | PA-0 | 파일·코드·태그가 온전한가 |
# | 강점 강화 | A1 | N2 비통과는 검정력 부족인가, 점프 부재인가 |
# | | A2 | 신규 진입 우세는 정의·부트스트랩·이름 식별에 견고한가 |
# | | A3 | 선두 국가 교체 서술이 견고한가 |
# | 약점 리프레이밍 | B2 | 2020 변화점이 표본 크기·저널 구성의 아티팩트인가 |
# | | B3 | 신생 토픽 정의 4종의 핵심 집합은 무엇인가 |
# | | B4 | 등급 선택이 결과를 좌우하는가 |
# | | B5 | S4 안정성 0.700은 경계인가 |
# | | B6 | 위험군에 독립 정답이 있는가 |
# | | B7 | 임베딩 결측 16%가 결과를 뒤집는가 |
# | | B8 | 미달 4개 도메인의 정렬은 어떤가 |
# | | B9 | 간극이 소득 수준과 분리되는가 |
# | 정합성 | C1 | 진입 우세가 중국 소속 유입과 독립인가 |
# | | C2 | S7과 S6의 모집단 차이는 얼마인가 |

# ## 2. 완료한 분석과 결과

# | PA | 동결 판독 | 핵심 수치 | 쓸 수 있는 말 / 주의 |
# |---|---|---|---|
# | PA-0 | 대부분 완료 | 해시 152건 일치 | S4 코드는 복구본, `oa_raw` 외부 백업 미완료 |
# | A1 | run 1 무효, run 2 all·within 강화, fixed14 곡선만 보고 | δ=0 기각률 0.035/0.075/0.065 | 2020 부근은 초과분 크기 점프를 탐지했을 가능성이 높음. 2008~2012는 사각지대 |
# | B2 | 3표본 유지 | J 비율 0.998~1.006 | 하향 추출이 20%뿐이라 약한 시험 |
# | B2b | 단독 제외 유지, 동시 제외 약화 | J 0.74~0.83 | 2020은 겹친 가속 |
# | B3a·b·c | 핵심 7·중간 10·특이적 14 | 핵심 진입 0.458 대 적응 0.252 | 정의 구조의 부산물, 독립 합의 증거 아님 |
# | B4 | S5 둔감, S6 3쌍 둔감 | 첫 쌍만 부호 변화 | 제외 4개뿐인 약한 시험 |
# | B4b | 판독1 둔감, 판독2 규칙상 민감 | 안정성 +0.027/+0.030 | 임계 0.03이 토픽 1개 규모 |
# | B4d | S5 통과 판정 동일, 앞쪽 변화점 이동·소실. S6 첫 쌍만 민감 | 변화 +0.046/+0.013/+0.020/+0.011 | 제외 15개는 B4의 확장이며 독립 확인 아님 |
# | B5 | 부호 반전 0 | 안정성 0.66~0.77 | "0.70 전후에서 움직임" |
# | A2c | 3쌍 유지(첫 쌍 제외) | 두 구간 방식 모두 확실한 쌍은 마지막뿐 | excess 구간 교정 불충분 |
# | A2b | s6 3, y5 3, y2 0(뒤집힘) | y2 차이 −0.15~−0.16 | "신규 저자 주도"가 아니라 "진입(신규+복귀) 주도" |
# | A2a | 4쌍 모두 둔감 | 변화 ≤0.011 | ID는 2020+만 있어 ID 재판별 불가 |
# | **C1** | 3개 시나리오 모두 유지 안 됨 | +0.113 → −0.101, −0.110, −0.024 | 진입 우세와 중국 소속 유입은 분리 불가 |
# | C2 | 20칸 중 일치 11 | 중국 몫 4기간 일치, 미국·상위5 +0.01~0.03 | 수준은 다름 |
# | A3 | 방향 유지 | 알려진 국가 기준 top5 평탄(0.64→0.65) | HHI 반등은 중국 몫과 연동 |
# | B7a·b·c | 모두 유지 | big 50/50 | 민감도이며 경계 아님 |
# | B9a·b | HIC 비관찰, UMC는 하한 5에서만 관찰 | safety·road_deaths 편 ρ≈0.24 | LIC·LMC는 자료상 불가 |
# | B8a | 참고용 | 8쌍 중 7쌍 HIC 과잉·LIC/LMC 과소 | 본문 판정 2/6 불변 |

# **종합**
# - 쓸 수 있는 것: 변화는 점프보다 연속적 재배치로 서술하는 편이 자료에 부합, 2020 부근은 COVID와 기술 주제가 겹친 가속, 진입 우세는 마지막 구간에서 가장 뚜렷, 알려진 국가 기준 top5 몫은 평탄, S4 방향은 결측 가정에서도 유지.
# - 쓰면 안 되는 것: 체제 전환 부재 확정, 2020이 아티팩트가 아니라는 확정, 신규 저자가 모든 구간을 주도, 진입 주도가 중국 유입과 무관, ID 기반 일치, 인과 해석.
# - 구조상 가장 큰 변화: 진입 중심 갱신과 중국 몫 증가는 독립된 두 기여가 아니라 한 구조의 두 측면.

# ## 3. 진행 중인 분석: B6a (ORCID 독립 정답)

# - 동결(`764956a259fd`)과 푸시는 완료되었습니다.
# - 수집이 중단되어 캐시에 50,000개 ID가 있고 남은 것은 21,702개입니다.
# - 실패 원인은 두 단계였습니다. 키 없는 요청이 공유 IP 일일 예산 소진(429)에 걸렸고, 이후 키를 받았지만 제 패치가 적용되지 않아(`replace()` 불일치) 키가 전달되지 않았습니다.
# - 직전에 드린 직접 수정 패치를 적용하고 curl로 200을 확인한 뒤 재개하면 됩니다.
# - **키가 채팅에 노출되었으니 폐기·재발급이 필요합니다.**
# - 부분 캐시로는 `run`을 하지 않습니다(모집단 정의가 달라짐).
# - changelog에는 "패치 미적용으로 429 지속, 정정 후 재개" 한 줄이 아직 없습니다.

# ## 4. 남은 분석과 작업

# **분석**

# | 항목 | 내용 | 선행 조건 |
# |---|---|---|
# | B6a 마무리 | 수집 재개 → 캐시 해시 기록 → `pa_b6a.py run` | 새 키, 패치 적용, 200 확인 |
# | B6b | 이름 50쌍 동일인 수작업 판정 | 연구자 시간. 시트 생성 후 `same_person` 기입 |
# | B8b | 사전 v3, 도메인별 새 25편(100편) 이중 코딩 | `PA_B8b_dict_v3.json` 작성, 코더 2명. 통과/실패 판정 없이 구간만 보고 |

# B6b와 B8b는 이번 주 안에 못 하면 "미수행, 한계로 명시"로 확정합니다.

# **미수행으로 확정될 항목(부록에 사유 기재)**
# - B4c(임베딩 응집도): 임베딩 파일 없음
# - A2의 ID 기반 재판별: F7이 2020년 이후만 덮음
# - B9의 LIC·LMC 소득군 내 분석: 질량 하한 때문에 자료상 불가

# **정리·문서**
# - A4 열람·동결 이력 부록 표(`pa_a4.py` 실행, `MANUAL` 검토, B4d·B6a 기록 추가)
# - 통합 문구표와 약화·반전 공개 목록: C1, A2b y2, A2c, B4 첫 쌍, B4b 판독2, B4d, B9a·b, B8a, B7c, B2·B3의 약한 시험
# - 재현성 잔여: `oa_raw` 외부 백업, GitHub Private 확인, `S8.5` 노트북 최종 커밋

# **투고 준비**
# - S9(후보 풀·추출 규칙 동결), S10(저널 선례 3편 이상, 심사 대응표, 생성형 AI 사용 선언)
# - 원고: 제목에서 "체제 전환" 삭제, 진입 중심 갱신과 중국 소속 유입을 한 구조로 서술, 모든 표에 `[Post-hoc]`, 영문 post-hoc 문구 삽입
# - 투고 후보 저널이 아직 정해지지 않아 구성안·초록 초안의 병목입니다.

# ## 5. 제 쪽 정정 기록 (부록용)

# - B4b 임계값 0.03이 토픽 1개 규모라는 점을 놓쳤습니다.
# - B9b에서 UMC가 0을 배제할 확률을 약 10%로 본 것은 틀렸습니다.
# - B9b·B3b 생략을 침묵 동의로 처리했습니다.
# - "S3에 Jaccard 중앙값만 있다"는 틀렸습니다(`jaccard_p10` 열 존재).
# - B6a 수집 실패의 원인을 두 번 잘못 추정했고(이메일 자리표시자, 순간 속도 제한), 패치 미적용을 `py_compile`로 걸러내지 못했습니다.

# ## 6. 지금 필요한 것

# 1. 새 키 발급·적용 후 `fetch2` 재개 결과
# 2. B6b·B8b를 할 수 있는지
# 3. 투고 후보 저널

# 가장 가까운 한 걸음은 패치 적용과 curl 200 확인입니다.

In [ ]:
# # PA-B6a 판독

# **동결 규칙은 수치와 Wilson 구간만 보고하며 판정은 하지 않습니다.** 수집·실행은 정상 종료됐고(`⛔` 없음, 재현 점검 통과), 캐시 점검에서 고유 ID 71,702, `found False` 422, 고유 ORCID 61,424로 확인됐습니다.

# ## 1. 결과

# | 항목 | 값 |
# |---|---|
# | 모집단(ORCID 보유 ID가 있는 이름) | 54,204 |
# | 양성(서로 다른 ORCID 2개 이상) | 3,249 |
# | 위험군 이름 → F7 존재 / 모집단 포함 | 1,371 / 1,360 (전체 1,431) |
# | 혼동표 TP / FP / FN / TN | 1,073 / 287 / 2,176 / 50,668 |
# | **정밀도** | **0.789** [0.767, 0.810] |
# | **재현율** | **0.330** [0.314, 0.347] |
# | 분할 이름 중 서로 다른 ORCID 비율 | 0.966 [0.959, 0.971] (3,249/3,364) |

# ## 2. 읽기

# 1. **위험군으로 지목한 이름의 약 79%는 독립 정답에서도 동명이인입니다.** 위험군 판정이 임의적이지 않다는 독립 단서입니다. 다만 순환성이 해소됐다고 쓰면 안 됩니다. 정답이 ORCID 보유 이름에 한정되고 2020+만 덮기 때문입니다.
# 2. **재현율은 0.33으로 낮습니다.** ORCID로 확인되는 동명이인의 3분의 2가 위험군에 들지 않았습니다. 앞서 A2a에서 분석 집합에 남은 분할 이름이 2,592개였던 것과 방향이 같습니다. 즉 위험군 제외는 보수적이고, 남은 동명이인 오염은 A2a의 "분할 이름 제외 민감도"(변화 ≤0.011)로 다뤄진 상태입니다. 이 연결은 해석이며 직접 검증은 아닙니다.
# 3. **보조 지표(0.966)는 해석에 주의하세요.** ID가 갈라지고 ORCID 보유 ID가 둘 이상인 이름만 대상이므로, 그 조건에서는 ORCID가 다른 것이 거의 정의상 당연합니다. "분할 이름의 97%가 동명이인"으로 일반화하면 안 됩니다. ORCID 없는 분할 이름(3,749 중 약 385개)과 프로필 분할은 여기에 반영되지 않았습니다.
# 4. 정밀도·재현율은 같은 모집단 안의 값이라, 위험군 이름 중 모집단 밖(70개)은 계산에서 빠졌습니다.

# ## 3. 쓸 수 있는 문구

# > "OpenAlex ORCID로 구성한 독립 정답(서로 다른 ORCID 2개 이상, n=3,249 이름)에 대해 위험군 판정의 정밀도는 0.79(95% CI 0.77–0.81), 재현율은 0.33(0.31–0.35)이었다. 표본은 ORCID 보유 저자와 2020년 이후 논문에 한정된다."

# 쓰면 안 되는 것: "위험군 검증의 순환성이 해소됐다", "동명이인 문제가 해결됐다".

# ## 4. 기록과 커밋

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B6a 결과 해석 메모(결정적): 정밀도 0.789 [0.767,0.810], 재현율 0.330 [0.314,0.347](TP 1073, FP 287, FN 2176, TN 50668; 모집단 54,204, 양성 3,249). 위험군 1,431 중 F7 존재 1,371, 모집단 포함 1,360. 분할 이름 중 서로 다른 ORCID 0.966(조건상 거의 정의적, 일반화 불가). 캐시 found False 422, 고유 ORCID 61,424(일부 ORCID가 여러 ID에 붙음). 캐시 sha256 앞12자 144b7cd47b5c는 run 이전 파일 기준(해시 정정 줄이 run 이후에 기록됨). ORCID 보유자 편향·2020+ 한정, 순환성 해소 아님 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_B6a_results.csv paper1/09_PA/PA_B6a_split_names_orcid.csv paper1/09_PA/PA_B6a_population.csv paper1/09_PA/PA_B6a_full_log.txt paper1/09_PA/PA_CHANGELOG.md
# git status --short | head -n 8
# git commit -m "PA-B6a results" && git tag -a PA-B6a-results-2026-10-21 -m "PA-B6a outputs, seen" && git push --follow-tags
# ```

# 앞서 요청했던 키 이력 점검도 아직 출력을 못 받았습니다. 키 폐기를 먼저 확인하세요.

# ```bash
# git -C ~/Research/Minnesota_Transport log -p --all -S"kSKl" --oneline | head -3
# ```

# 비어 있어야 정상입니다. `population.csv`가 크면(54,204행) 커밋 전에 크기를 확인하세요.

# ## 5. 남은 항목

# - B6b(수작업 50쌍), B8b(사전 v3, 새 25편): 인력 여부 미정, 이번 주 내 결정
# - A4 표 확정, 통합 문구표(B6a 행 추가, B4d 포함)
# - `oa_raw` 외부 백업, GitHub Private 확인, `S8.5` 노트북 커밋
# - 투고 후보 저널 결정 후 S9·S10

In [ ]:
# 정정 줄이 **두 번 올라갔습니다**(`8c4d0a5`, `e7b643a`). 같은 명령 블록이 두 번 붙여넣어진 것으로 보입니다. 기록은 지우지 않는 원칙이라 중복 줄은 그대로 두고, 중복이었다는 사실만 한 줄로 남기겠습니다.

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | 정리: 직전 '해시 자리표시자 정정' 줄이 중복 실행으로 2회 기록됨(커밋 8c4d0a5, e7b643a, 내용 동일). 유효한 정정은 1건 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_CHANGELOG.md && git commit -m "PA-B6a: note duplicate correction line" && git push
# ```

# 앞으로는 명령 블록을 한 번만 붙여넣고, 출력이 끝난 뒤 다음 블록으로 넘어가세요.

# ## 현재 상태

# - PA-B6a는 완료됐고, 태그 `PA-B6a-results-2026-10-21`이 원격에 있습니다.
# - 키는 저장소 이력에 없습니다. OpenAlex에서 노출된 키(2개)를 폐기했는지만 확인하세요.

# ## 다음 한 걸음: A4 표 생성

# 분석이 사실상 끝났으니 부록 뼈대가 될 A4 표를 만듭니다. 이 스크립트는 읽기 전용입니다(`09_PA`에 표 파일만 생성). 먼저 `pa_a4.py`가 저장되어 있어야 합니다.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# ls pa_a4.py pa_common.py
# python3 pa_a4.py
# ```

# `pa_a4.py`가 없다고 나오면 알려 주세요. 앞서 드린 코드가 파일로 저장되지 않은 것이니 다시 안내하겠습니다. 출력(항목 수와 `본 결과 변경: 없음이 아닌 줄`)을 붙여 주시면 `MANUAL` 보정 항목(B4d, B6a의 오진·키 노출·정정 이력 포함)을 정리해 드리겠습니다.

# ## 남은 결정 (이번 주 내)

# 1. B6b(수작업 50쌍), B8b(새 25편 이중 코딩)를 할 수 있는지. 못 하면 "미수행, 한계로 명시"로 확정합니다.
# 2. 투고 후보 저널. 정해야 S9·S10과 구성안 작성으로 넘어갑니다.

# # A4 표 판독

# 생성은 정상입니다(31항목). 핵심 확인: **changelog 전 줄이 "본 결과 변경: 없음"**이라는 점은 부록 문구로 그대로 쓸 수 있습니다. 다만 자동 분류가 틀린 칸이 있어 부록 전에 보정이 필요합니다.

# ## 1. 보정이 필요한 칸

# | 항목 | 표의 값 | 실제 | 조치 |
# |---|---|---|---|
# | 0 | 진행/기록만 | PA-0(재현성 정리, 계획 v1·v2 동결) | `MANUAL`에서 "완료(일부 미완: oa_raw 외부 백업)"로 명시 |
# | A1 | 기록만, 계획 해시 공란 | 계획은 PA-0 v2(`a98f9697fa67`)에 포함, run1 무효·run2·탐색 확장 | 계획 해시 수기 기입 |
# | B2 | 완료, 계획 `47c93a7c5739` | 동결은 노트북 셀에서 실행 후 스크립트 사후 저장(해시 일치) | 비고 유지 |
# | B3, B7, B4c, A4 | 기록만 | B3·B7은 하위 항목(a/b/c)의 상위 라벨, B4c는 미수행, A4는 이 표 자체 | 상위 라벨은 "하위 항목 참조"로, B4c는 "미수행(임베딩 파일 없음)"으로 표기 |
# | B6, B8-2 | 계획 없음 | B6a는 완료, B6b·B8b는 인력 미정 | B6는 "B6a 완료, B6b 미수행"으로 분리 표기 |
# | 기타 | 15줄 | 항목명이 안 붙은 기록(S4 복구, 문구 정정, 키·수집 사고 등) | 부록 소표로 분리 |

# B6a 행에는 수동 비고가 필요합니다: 수집 중단 원인을 두 번 오진(이메일 자리표시자, 순간 속도 제한), 실제 원인은 공유 IP 일일 예산 소진, 키 도입 첫 시도는 패치 미적용으로 실패, API 키 노출 2회(저장소 이력에는 없음, 폐기 필요), 해시 자리표시자 정정과 중복 정정 줄.

# ## 2. 한 가지 확인할 점

# B4d의 계획 해시가 `a91a9a76153a`, B9b가 `9f65e027ba64`, B7c가 `7f3ff6991986`, B3b가 `7f61ff442f7f`로 표에 있습니다. 제가 이 값들을 이전 출력에서 직접 본 적은 없어서, 부록에 쓰기 전에 각 `PA_*_PLAN.json`의 `sha256` 앞 12자와 한 번 대조하세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1/09_PA
# for f in PA_B4d PA_B9b PA_B7c PA_B3b; do python3 -c "import json;print('$f',json.load(open('${f}_PLAN.json'))['sha256'][:12])"; done
# ```

# ## 3. `MANUAL` 보정 명령

# `pa_a4.py`의 `MANUAL` 딕셔너리에 항목을 추가하고 재실행하면 됩니다. 문구 초안입니다(파일을 열어 `MANUAL = {` 안에 붙이세요).

# ```python
#     "B6a": "수집 중단 원인을 두 번 오진(이메일 자리표시자, 순간 속도 제한); 실제는 키 없는 요청의 공유 IP 일일 예산 소진(429). 키 도입 첫 시도는 패치 미적용으로 실패 후 직접 수정. API 키 노출 2회(저장소 이력에는 없음, 폐기 조치). 해시 자리표시자 정정 줄은 중복 기록됨(유효 1건). 결과 해석은 ORCID 보유·2020+ 한정",
#     "B4d": "제외 15개에 B4의 F1·F3 실패 4개 포함(확장, 독립 확인 아님). 앞쪽 변화점 이동·소실은 벌점 효과 가능성(미확인)",
#     "B4c": "미수행: 임베딩 파일을 찾지 못함",
#     "B6": "B6a 완료, B6b(수작업 50쌍) 인력 미정으로 미수행",
#     "B8-2": "미수행: 인력 미정(코더 2명, 사전 v3)",
# ```

# ## 4. 남은 결정

# - B6b·B8b: 이번 주 안에 정하고, 못 하면 위 표의 "미수행" 문구로 확정합니다.
# - 투고 후보 저널을 알려 주시면 S9·S10과 구성안·초록으로 넘어갈 수 있습니다.

# 보정본(`MANUAL` 반영) 실행 출력이나 계획 해시 대조 결과를 붙여 주세요.

# 네 개 모두 A4 표의 값과 일치합니다(B4d `a91a9a76153a`, B9b `9f65e027ba64`, B7c `7f3ff6991986`, B3b `7f61ff442f7f`). 앞서 제가 직접 본 적 없는 값이라고 했던 우려는 해소됐고, 부록에 그대로 쓸 수 있습니다.

# ## 다음: `MANUAL` 반영 후 표 재생성

# `pa_a4.py`를 열어 `MANUAL = {` 안에 앞서 드린 5개 항목(B6a, B4d, B4c, B6, B8-2)을 추가하세요. 기존 키(`"0"`, `"A1"`, `"B2"` 등)와 중복되지 않는지 확인하고, 마지막 항목 뒤에 쉼표가 있는지 보세요. 저장 후 실행합니다.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 -m py_compile pa_a4.py && echo "문법 OK"
# python3 pa_a4.py > /dev/null && grep -c "B6a" 09_PA/PA_A4_audit_table.md
# grep "B6a\|B4c" 09_PA/PA_A4_audit_table.md | cut -c1-200
# ```

# `B6a` 줄에 수동 비고(오진, 키 노출, 정정)가 보이면 반영된 것입니다. 앞서 `replace()`가 조용히 실패한 경험이 있으니, 눈으로 확인하는 이 단계를 건너뛰지 마세요.

# ## 커밋

# A4 표는 분석이 아니라 기록 산출물이므로, 동결 없이 한 줄 기록과 함께 올립니다.

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-A4 감사 표 생성(읽기 전용 스크립트 pa_a4.py): 31항목, MANUAL 보정(B6a 사고 이력, B4d, B4c·B6·B8-2 미수행 사유) 반영. 계획 해시 4건(B4d, B9b, B7c, B3b)을 각 PLAN.json과 대조해 일치 확인. 전 changelog 줄이 '본 결과 변경: 없음' | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/pa_a4.py paper1/09_PA/PA_A4_audit_table.csv paper1/09_PA/PA_A4_audit_table.md paper1/09_PA/PA_CHANGELOG.md
# git status --short | head -n 8
# ```

# 스테이징 목록에 예상한 4개 외 파일이 없으면 커밋·푸시하세요. 있으면 알려 주세요.

# ## 남은 일

# - 분석: B6b(수작업 50쌍), B8b(새 25편 이중 코딩). 이번 주 안에 못 하면 "미수행, 한계로 명시"로 확정합니다.
# - 재현성 잔여: `oa_raw` 외부 백업, GitHub Private 확인, `S8.5` 노트북 커밋, OpenAlex 노출 키 2개 폐기 확인.
# - 투고 후보 저널을 알려 주시면 S9·S10과 구성안·초록 초안으로 넘어갑니다.

# `grep` 출력을 붙여 주세요.

In [ ]:
# # 논문 1 사후분석(PA) 현황 (2026-10-21 기준)

# ## 1. 하려고 한 분석 (지침서 14개 항목, 핵심은 A1·A2·B2)

# | 구분 | 항목 | 질문 |
# |---|---|---|
# | 재현성 | PA-0 | 파일·코드·태그가 온전한가 |
# | 강점 강화 | A1 | N2 비통과는 검정력 부족인가, 점프 부재인가 |
# | | A2 | 신규 진입 우세는 정의·부트스트랩·이름 식별에 견고한가 |
# | | A3 | 선두 국가 교체 서술이 견고한가 |
# | 약점 리프레이밍 | B2 | 2020 변화점이 표본 크기·저널 구성의 아티팩트인가 |
# | | B3 | 신생 토픽 정의 4종의 핵심 집합은 무엇인가 |
# | | B4 | 등급 선택이 결과를 좌우하는가 |
# | | B5 | S4 안정성 0.700은 경계인가 |
# | | B6 | 위험군에 독립 정답이 있는가 |
# | | B7 | 임베딩 결측 16%가 결과를 뒤집는가 |
# | | B8 | 미달 4개 도메인의 정렬은 어떤가 |
# | | B9 | 간극이 소득 수준과 분리되는가 |
# | 정합성 | C1 | 진입 우세가 중국 소속 유입과 독립인가 |
# | | C2 | S7과 S6의 모집단 차이는 얼마인가 |

# 공통 원칙: 실행 전 해시·태그 동결, GitHub 푸시, 결과 전부 보고, S0~S8 판정 불변.

# ## 2. 완료한 분석

# | PA | 동결 판독 | 쓸 수 있는 말 / 주의 |
# |---|---|---|
# | PA-0 | 해시 152건 일치 | S4 코드는 복구본, `oa_raw` 외부 백업 미완 |
# | A1 | run 1 무효(주입 버그), run 2: all·within 강화, fixed14 곡선만 보고 | 2020 부근은 초과분 크기 점프를 탐지했을 가능성이 높음. 2008~2012는 사각지대 |
# | A2a | 4쌍 모두 둔감(변화 ≤0.011) | ID는 2020+만 있어 ID 재판별은 불가, 분할 이름 제외 민감도로 축소 |
# | A2b | s6 3쌍, y5 3쌍, **y2 0쌍(뒤집힘)** | "신규 저자 주도"가 아니라 "진입(신규+복귀) 주도" |
# | A2c | 3쌍 유지(첫 쌍 제외) | 두 구간 방식 모두 확실한 쌍은 마지막뿐 |
# | A3 | 방향 유지 | 알려진 국가 기준 top5 몫 평탄, HHI 반등은 중국 몫과 연동 |
# | B2 | 3표본 유지 | 하향 추출이 20%뿐이라 약한 시험 |
# | B2b | 단독 제외는 유지, 동시 제외는 약화 | 2020은 단일 원인이 아닌 겹친 가속 |
# | B3a·b·c | 핵심 7·중간 10·특이적 14 | 정의 구조의 부산물, 독립 합의 증거 아님 |
# | B4 | S5 둔감, S6 3쌍 둔감 | 첫 쌍만 부호 변화, 제외 4개뿐인 약한 시험 |
# | B4b | 판독1 둔감, 판독2 규칙상 민감 | 임계 0.03이 토픽 1개 규모 |
# | B4d | S5 통과 판정 동일, 앞쪽 변화점 이동·소실. S6 첫 쌍만 민감 | 제외 15개는 B4의 확장 |
# | B5 | 부호 반전 0, 안정성 0.66~0.77 | "0.70 전후에서 움직임" |
# | B6a | 정밀도 0.789, 재현율 0.330 | ORCID 보유·2020+ 한정, 순환성 해소 아님 |
# | B7a·b·c | 모두 유지(big 50/50) | 민감도이며 경계가 아님 |
# | B8a | 8쌍 중 7쌍 HIC 과잉·LIC/LMC 과소 | 참고용, 본문 판정 2/6 불변 |
# | B9a·b | HIC 비관찰, UMC는 하한 5에서만 관찰 | LIC·LMC는 자료상 불가 |
# | **C1** | **3개 시나리오 모두 유지 안 됨**(+0.113 → −0.101, −0.110, −0.024) | 진입 우세와 중국 소속 유입은 분리 불가 |
# | C2 | 20칸 중 일치 11 | 중국 몫은 일치, 미국·상위5는 +0.01~0.03 |
# | A4 | 감사 표 31항목 생성 | MANUAL 보정 반영 후 재생성·커밋 대기 |

# **종합**
# - 쓸 수 있는 것: 변화는 점프보다 연속적 재배치로 서술하는 편이 자료에 부합. 2020 부근은 COVID와 기술 주제가 겹친 가속. 진입 우세는 마지막 구간에서 가장 뚜렷.
# - 쓰면 안 되는 것: 체제 전환 부재 확정, 2020이 아티팩트가 아니라는 확정, 신규 저자가 모든 구간을 주도, 진입 주도가 중국 유입과 무관, ID 기반 일치, 인과 해석.
# - 가장 큰 구조 변화: 진입 중심 갱신과 중국 몫 증가는 독립된 두 기여가 아니라 한 구조의 두 측면.

# ## 3. 하려고 하는 분석과 작업

# **분석 (사람 손 필요)**

# | 항목 | 내용 | 조건 |
# |---|---|---|
# | B6b | 이름 50쌍 동일인 수작업 판정 | 연구자 시간. 못 하면 "미수행, 한계로 명시" |
# | B8b | 사전 v3, 도메인별 새 25편(총 100편) 이중 코딩 | `PA_B8b_dict_v3.json` 작성, 코더 2명. 통과/실패 판정 없이 구간만 보고 |

# **미수행으로 확정될 항목 (부록에 사유 기재)**
# - B4c: 임베딩 파일을 찾지 못함
# - A2의 ID 기반 재판별: F7이 2020년 이후만 덮음
# - B9의 LIC·LMC 소득군 내 분석: 질량 하한 때문에 자료상 불가

# **정리·문서**
# - A4: `MANUAL` 5개 항목 추가 후 `pa_a4.py` 재실행, 커밋
# - 통합 문구표와 약화·반전 공개 목록(C1, A2b y2, A2c, B4 첫 쌍, B4b 판독2, B4d, B9a·b, B8a, B7c, B2·B3의 약한 시험)
# - 재현성 잔여: `oa_raw` 외부 백업, GitHub Private 확인, `S8.5` 노트북 커밋, 노출된 OpenAlex 키 2개 폐기 확인

# **투고 준비**
# - S9(후보 풀·추출 규칙 동결), S10(저널 선례 3편 이상, 심사 대응표, 생성형 AI 사용 선언)
# - 원고: 제목에서 "체제 전환" 삭제, 모든 표에 `[Post-hoc]`, 영문 post-hoc 문구 삽입, 기각·약화된 PA 목록을 부록에
# - 투고 후보 저널이 미정이라 구성안·초록 초안이 막혀 있음

# **지금 필요한 결정**
# 1. B6b·B8b 수행 여부 (이번 주 내)
# 2. 투고 후보 저널

In [ ]:
# # 논문 1 사후분석(PA) 현황 (2026-10-21 기준)

# ## 1. 분석 목록 (지침서 14개 항목, 핵심은 A1·A2·B2)

# | 구분 | 항목 | 질문 |
# |---|---|---|
# | 재현성 | PA-0 | 파일·코드·태그가 온전한가 |
# | 강점 강화 | A1 | N2 비통과는 검정력 부족인가, 점프 부재인가 |
# | | A2 | 신규 진입 우세는 정의·부트스트랩·이름 식별에 견고한가 |
# | | A3 | 선두 국가 교체 서술이 견고한가 |
# | 약점 리프레이밍 | B2 | 2020 변화점이 표본 크기·저널 구성의 아티팩트인가 |
# | | B3 | 신생 토픽 정의 4종의 핵심 집합은 무엇인가 |
# | | B4 | 등급 선택이 결과를 좌우하는가 |
# | | B5 | S4 안정성 0.700은 경계인가 |
# | | B6 | 위험군에 독립 정답이 있는가 |
# | | B7 | 임베딩 결측 16%가 결과를 뒤집는가 |
# | | B8 | 미달 4개 도메인의 정렬은 어떤가 |
# | | B9 | 간극이 소득 수준과 분리되는가 |
# | 정합성 | C1 | 진입 우세가 중국 소속 유입과 독립인가 |
# | | C2 | S7과 S6의 모집단 차이는 얼마인가 |

# 공통 원칙: 실행 전 해시·태그 동결, GitHub 푸시, 결과 전부 보고, S0~S8 판정 불변.

# ## 2. 지금까지 한 분석

# | PA | 동결 판독 | 쓸 수 있는 말 / 주의 |
# |---|---|---|
# | PA-0 | 해시 152건 일치 | S4 코드는 복구본, `oa_raw` 외부 백업 미완 |
# | A1 | run 1 무효(주입 버그), run 2: all·within 강화, fixed14 곡선만 보고 | 2020 부근은 초과분 크기 점프를 탐지했을 가능성이 높음. 2008~2012는 사각지대 |
# | A2a | 4쌍 모두 둔감(변화 ≤0.011) | ID는 2020+만 있어 분할 이름 제외 민감도로 축소 |
# | A2b | s6 3쌍, y5 3쌍, y2 0쌍(뒤집힘) | "신규 저자 주도"가 아니라 "진입(신규+복귀) 주도" |
# | A2c | 3쌍 유지(첫 쌍 제외) | 두 구간 방식 모두 확실한 쌍은 마지막뿐 |
# | A3 | 방향 유지 | 알려진 국가 기준 top5 몫 평탄, HHI 반등은 중국 몫과 연동 |
# | B2 | 3표본 유지 | 하향 추출 20%뿐인 약한 시험 |
# | B2b | 단독 제외 유지, 동시 제외 약화 | 2020은 겹친 가속 |
# | B3a·b·c | 핵심 7·중간 10·특이적 14 | 정의 구조의 부산물, 독립 합의 증거 아님 |
# | B4 | S5 둔감, S6 3쌍 둔감 | 첫 쌍만 부호 변화, 약한 시험 |
# | B4b | 판독1 둔감, 판독2 규칙상 민감 | 임계 0.03이 토픽 1개 규모 |
# | B4d | S5 통과 판정 동일, 앞쪽 변화점 이동·소실, S6 첫 쌍만 민감 | B4의 확장이며 독립 확인 아님 |
# | B5 | 부호 반전 0, 안정성 0.66~0.77 | "0.70 전후에서 움직임" |
# | B6a | 정밀도 0.789, 재현율 0.330 | ORCID 보유·2020+ 한정, 순환성 해소 아님 |
# | **B6b** | 동일인 9·다른 사람 11·판정 불가 30, 비율 0.55 [0.34, 0.74], 경계 0.13~0.90 | 구간이 매우 넓어 점추정 단독 인용 금지 |
# | B7a·b·c | 모두 유지(big 50/50) | 민감도이며 경계가 아님 |
# | B8a | 8쌍 중 7쌍 HIC 과잉·LIC/LMC 과소 | 참고용, 본문 판정 2/6 불변 |
# | B9a·b | HIC 비관찰, UMC는 하한 5에서만 관찰 | LIC·LMC는 자료상 불가 |
# | **C1** | 3개 시나리오 모두 유지 안 됨(+0.113 → −0.101, −0.110, −0.024) | 진입 우세와 중국 소속 유입은 분리 불가 |
# | C2 | 20칸 중 일치 11 | 중국 몫 일치, 미국·상위5는 +0.01~0.03 |
# | A4 | 감사 표 31항목 생성·커밋 | B6a·B6b·B4d 행 반영한 최종본 재생성 필요 |

# **종합**
# - 쓸 수 있는 것: 변화는 점프보다 연속적 재배치로 서술하는 편이 자료에 부합. 2020 부근은 COVID와 기술 주제가 겹친 가속. 진입 우세는 마지막 구간에서 가장 뚜렷.
# - 쓰면 안 되는 것: 체제 전환 부재 확정, 2020이 아티팩트가 아니라는 확정, 신규 저자가 모든 구간을 주도, 진입 주도가 중국 유입과 무관, ID 기반 일치, 인과 해석.
# - 가장 큰 구조 변화: 진입 중심 갱신과 중국 몫 증가는 독립된 두 기여가 아니라 한 구조의 두 측면.

# ## 3. 앞으로 할 분석과 작업

# **분석: 남은 것은 B8b 하나**

# | 항목 | 내용 | 조건 |
# |---|---|---|
# | B8b | 사전 v3, 도메인별 새 25편(총 100편) 이중 코딩 | `PA_B8b_dict_v3.json`을 직접 작성, 코더 2명. 통과/실패 판정 없이 구간만 보고. 못 하면 "미수행" 확정 한 줄 기록 |

# **미수행으로 확정된 항목 (부록에 사유 기재)**
# - B4c: 임베딩 파일을 찾지 못함
# - A2의 ID 기반 재판별: F7이 2020년 이후만 덮음
# - B9의 LIC·LMC 소득군 내 분석: 질량 하한 때문에 자료상 불가

# **정리·문서**
# 1. A4 감사 표 최종 재생성(B6a·B6b·B4d 행, B4c 미수행 사유)
# 2. 통합 문구표와 약화·반전 공개 목록(C1, A2b y2, A2c, B4 첫 쌍, B4b 판독2, B4d, B6b 판정 불가 60%, B9a·b, B8a, B7c, B2·B3의 약한 시험)
# 3. 재현성 잔여: `oa_raw` 외부 백업(SHA-256 일치 시에만 완료 기록), GitHub Private 확인, `S8.5` 노트북 커밋, 노출된 OpenAlex 키 2개 폐기 확인

# **투고 준비**
# - S9(후보 풀·추출 규칙 동결), S10(저널 선례 3편 이상, 심사 대응표, 생성형 AI 사용 선언)
# - 원고: 제목에서 "체제 전환" 삭제, 모든 표에 `[Post-hoc]`, 영문 post-hoc 문구 삽입, 기각·약화된 PA 목록을 부록에
# - 투고 후보 저널이 미정이라 구성안·초록 초안이 막혀 있음

# **지금 필요한 결정**
# 1. B8b를 할지, 미수행으로 확정할지
# 2. 투고 후보 저널

In [ ]:
# # B8b 진행 순서

# B8b는 사전 v3을 먼저 쓰고, 동결한 뒤에 새 표본을 뽑는 순서입니다. 이 순서를 바꾸면 안 됩니다. 새 25편을 본 뒤에 사전을 고치면 사전이 표본에 맞춰지고 precision이 낙관적으로 나옵니다.

# ## 1. 사전 v2와 이전 검증 표본의 오탐 확인 (읽기 전용)

# v3 정규식을 쓰려면 v2가 어떻게 생겼고 어디서 틀렸는지 알아야 합니다. 아래 명령은 이전 표본(n=25)에서 오탐으로 판정된 제목만 보여 줍니다. 이 표본은 이미 본 것이고, v3의 새 표본에서는 제외됩니다.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 - <<'EOF'
# import json, glob
# import pandas as pd
# P = json.load(open("08_S8/S8_0_PREREG_v2.json", encoding="utf-8"))
# for d in ["electrification_env", "freight", "active_urban", "infrastructure"]:
#     print(f"\n[{d}] v2 정규식:\n  {P['domains'][d]}")
# print("\n이전 표본 파일:", glob.glob("08_S8/*sample*.csv") + glob.glob("08_S8/*precision*.csv"))
# for f in sorted(glob.glob("08_S8/*sample*.csv")):
#     d = pd.read_csv(f, encoding="utf-8-sig"); print("\n", f, list(d.columns), len(d))
# EOF
# ```

# 출력을 붙여 주시면 다음을 해 드립니다.
# - 오탐 패턴 정리(예: `active_urban`의 "accessibility")
# - 정규식 작성 시 확인할 점 목록

# ## 2. v3 정규식 작성 (연구자 직접)

# - 파일은 `09_PA/PA_B8b_dict_v3.json`이고 키는 네 도메인 이름입니다. 값은 정규식 문자열입니다.
# - 오탐 패턴을 줄이는 방향으로 고치되, 재현율을 크게 깎는 수정은 피하세요. 앞선 검증에서 `freight`의 재현율이 0.50이었습니다.
# - 정규식은 제목에만 적용되며 대소문자를 구분하지 않습니다(`re.I`).
# - 자리표시자(`여기에`, `TODO`)를 남기지 마세요. 앞서 그대로 동결 해시에 들어간 사고가 있었습니다.
# - 작성한 뒤에는 **새로 뽑을 표본을 미리 보지 마세요.** 정규식이 몇 편에 매칭되는지 같은 총량 확인은 괜찮지만, 어떤 제목이 매칭되는지 훑어보고 고치면 안 됩니다.

# 작성 후 형식을 점검합니다.
# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 - <<'EOF'
# import json, re
# d = json.load(open("09_PA/PA_B8b_dict_v3.json", encoding="utf-8"))
# need = ["electrification_env", "freight", "active_urban", "infrastructure"]
# assert sorted(d) == sorted(need), f"키 불일치: {sorted(d)}"
# for k, v in d.items():
#     re.compile(v, re.I); assert len(v) > 10 and "여기에" not in v and "TODO" not in v, f"{k} 자리표시자 의심"
# print("사전 v3 OK", {k: len(v) for k, v in d.items()})
# EOF
# ```

# ## 3. 코더 구성 (동결 전에 정하세요)

# - 코더 2명이 서로 보지 않고 각자 0/1로 코딩합니다.
# - 제2코더가 사람이 아니라 AI 도구라면 그대로 기록해야 합니다. 이 경우 독립 이중 코딩이 아니라 "연구자 + AI 보조 코딩"이 되며, 계획서 문구(코더 2명)와 어긋나므로 changelog와 한계 절에 명시하세요.
# - 같은 사람이 두 번 코딩하면 이중 코딩이 아닙니다.

# ## 4. 동결부터 시트 생성

# 사전 파일을 저장하고 코더를 정한 뒤 한 블록씩 실행하세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# ls 08_S8/*sample*.csv
# python3 pa_b8b.py freeze
# ```
# ```bash
# python3 pa_b8b.py tag && cd .. && git push --follow-tags && cd paper1
# ```
# ```bash
# python3 pa_b8b.py sheet
# ```

# `sheet`가 "이전 검증 표본을 찾지 못함"으로 멈추면 `noprev`를 쓰지 말고 출력만 보내 주세요. 이전 표본 pid를 제외하는 것이 계획서의 일부입니다.

# 시트가 생기면 판정 전에 빈 시트를 커밋하세요.
# ```bash
# cd ~/Research/Minnesota_Transport
# git add paper1/09_PA/PA_B8b_dict_v3.json paper1/09_PA/PA_B8b_sheet.csv paper1/09_PA/PA_CHANGELOG.md
# git status --short | head -n 6
# ```
# ```bash
# git commit -m "PA-B8b: blank sheet generated (before coding)" && git tag -a PA-B8b-sheet-2026-10-21 -m "blank sheet" && git push --follow-tags
# ```

# ## 5. 코딩과 채점

# 1. `PA_B8b_sheet.csv`를 복사해 코더마다 따로 작업합니다. `coder1`, `coder2` 칸을 서로 보지 않고 0/1로 채웁니다.
# 2. 불일치 항목은 합의해 `final`에 기록하고, 일치하면 `final`도 같은 값입니다.
# 3. 합친 파일을 `PA_B8b_sheet_filled.csv`로 저장합니다. `item_id`, `domain`, `pid`, 행 순서는 바꾸지 않습니다.
# 4. 판정 전에 판정 기준(도메인 정의)을 각 코더에게 같은 문장으로 전달합니다. 코딩 도중에는 기준을 바꾸지 않습니다.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 pa_b8b.py score 2>&1 | tee 09_PA/PA_B8b_full_log.txt
# ```

# ## 6. 결과를 읽을 때의 약속

# - 통과/실패 판정은 하지 않습니다. 도메인별 precision과 Wilson 구간, kappa, v2 precision만 보고합니다.
# - n=25에서는 참 precision이 0.80이어도 22편 이상(0.85 이상)이 나올 확률이 약 23%입니다. 4개 도메인을 각각 시도하므로 하나쯤은 우연히 높게 나올 수 있습니다.
# - 오탐 패턴을 보고 만든 사전이라 precision에 낙관 편향이 있습니다. 모든 표에 "사후 개정 사전 v3"로 표기하고 S8 판정(2/6 통과)은 바꾸지 않습니다.

# 1번 명령의 출력을 붙여 주세요.

In [ ]:
# # 논문 1 사후분석(PA) 현황 (2026-10-21 기준)

# 공통 원칙은 실행 전 해시·태그 동결, GitHub 푸시, 결과 전부 보고, S0~S8 판정 불변입니다. 아래 수치는 문서에 있는 출력과 changelog 해석 메모에서 옮긴 것이므로, 본문에 쓰기 전에 원 CSV와 대조하세요.

# ## 1. 하기로 한 분석 (지침서 14개 항목, 핵심은 A1·A2·B2)

# | 구분 | 항목 | 질문 |
# |---|---|---|
# | 재현성 | PA-0 | 파일·코드·태그가 온전한가 |
# | 강점 강화 | A1 | N2 비통과는 검정력 부족인가, 점프 부재인가 |
# | | A2 | 진입 우세는 정의·부트스트랩·이름 식별에 견고한가 |
# | | A3 | 선두 국가 교체 서술이 견고한가 |
# | 약점 리프레이밍 | B2 | 2020 변화점이 표본 크기·저널 구성의 아티팩트인가 |
# | | B3 | 신생 토픽 정의 4종의 핵심 집합은 무엇인가 |
# | | B4 | 등급 선택이 결과를 좌우하는가 |
# | | B5 | S4 안정성 0.700은 경계인가 |
# | | B6 | 위험군에 독립 정답이 있는가 |
# | | B7 | 임베딩 결측 16%가 결과를 뒤집는가 |
# | | B8 | 미달 4개 도메인의 정렬은 어떤가 |
# | | B9 | 간극이 소득 수준과 분리되는가 |
# | 정합성 | C1 | 진입 우세가 중국 소속 유입과 독립인가 |
# | | C2 | S7과 S6의 모집단 차이는 얼마인가 |

# ## 2. 지금까지 한 분석

# | PA | 동결 판독 | 쓸 수 있는 말 / 주의 |
# |---|---|---|
# | PA-0 | 해시 152건 일치 | S4 코드는 복구본, `oa_raw` 외부 백업 미완 |
# | A1 | run 1 무효(주입 버그), run 2: all·within 강화, fixed14 곡선만 보고 | 2020 부근은 초과분 크기 점프를 탐지했을 가능성이 높음. 2008~2012는 사각지대 |
# | A2a | 4쌍 모두 둔감(변화 ≤0.011) | ID는 2020+만 있어 분할 이름 제외 민감도로 축소 |
# | A2b | s6 3쌍, y5 3쌍, y2 0쌍(뒤집힘) | "신규 저자 주도"가 아니라 "진입(신규+복귀) 주도" |
# | A2c | 3쌍 유지(첫 쌍 제외) | 두 구간 방식 모두 확실한 쌍은 마지막뿐 |
# | A3 | 방향 유지 | 알려진 국가 기준 top5 몫 평탄, HHI 반등은 중국 몫과 연동 |
# | B2 | 3표본 유지 | 하향 추출 20%뿐인 약한 시험 |
# | B2b | 단독 제외 유지, 동시 제외 약화 | 2020은 겹친 가속 |
# | B3a·b·c | 핵심 7·중간 10·특이적 14 | 정의 구조의 부산물, 독립 합의 증거 아님 |
# | B4 | S5 둔감, S6 3쌍 둔감 | 첫 쌍만 부호 변화, 제외 4개뿐 |
# | B4b | 판독1 둔감, 판독2 규칙상 민감 | 임계 0.03이 토픽 1개 규모 |
# | B4d | S5 통과 판정 동일, 앞쪽 변화점 이동·소실, S6 첫 쌍만 민감 | 제외 15개는 B4의 확장, 독립 확인 아님 |
# | B5 | 부호 반전 0, 안정성 0.66~0.77 | "0.70 전후에서 움직임" |
# | B6a | 정밀도 0.789, 재현율 0.330 | ORCID 보유·2020+ 한정, 순환성 해소 아님 |
# | B6b | 동일인 9·다른 사람 11·판정 불가 30, 경계 0.13~0.90 | 구간이 매우 넓어 점추정 단독 인용 금지 |
# | B7a·b·c | 모두 유지(big 50/50) | 민감도이며 경계가 아님 |
# | B8a | 8쌍 중 7쌍 HIC 과잉·LIC/LMC 과소 | 참고용, 본문 판정 2/6 불변 |
# | B9a·b | HIC 비관찰, UMC는 하한 5에서만 관찰 | LIC·LMC는 자료상 불가 |
# | **C1** | 3개 시나리오 모두 유지 안 됨(+0.113 → −0.101, −0.110, −0.024) | 진입 우세와 중국 소속 유입은 분리 불가 |
# | C2 | 20칸 중 일치 11 | 중국 몫 일치, 미국·상위5는 +0.01~0.03 |
# | A4 | 감사 표 31항목 생성·커밋 | B6a·B6b·B4d 행 반영한 최종본 재생성 필요 |

# **종합**
# - 쓸 수 있는 것: 변화는 점프보다 연속적 재배치로 서술하는 편이 자료에 부합, 2020 부근은 COVID와 기술 주제가 겹친 가속, 진입 우세는 마지막 구간에서 가장 뚜렷.
# - 쓰면 안 되는 것: 체제 전환 부재 확정, 2020이 아티팩트가 아니라는 확정, 신규 저자가 모든 구간을 주도, 진입 주도가 중국 유입과 무관, ID 기반 일치, 인과 해석.
# - 가장 큰 구조 변화: 진입 중심 갱신과 중국 몫 증가는 독립된 두 기여가 아니라 한 구조의 두 측면.

# ## 3. 남은 분석과 작업

# ### 3-1. 진행 중: B8b (사전 v3, 새 25편 이중 코딩)

# | 단계 | 상태 |
# |---|---|
# | 이전 검증 표본 위치 확인 | 완료: `08_S8/coder1.csv`·`coder2.csv`(각 300편), `adjudication.csv`(9행) |
# | `pa_b8b.py` 패치 | 완료: 이전 표본 제외를 `coder[12].csv` 300편 전체로. 아직 동결 전(`??`) |
# | 오탐 패턴 확인 | 완료: electrification_env 12/53(`hybrid` 9건), freight 7/37, active_urban 14/45(`cycl*`, `accessibility`, `transit-oriented` 등), infrastructure 6/39(`concrete` 4건) |
# | 도메인 정의 문구 확인 | **미완**: 일부 "오탐"은 정의 문제(예: concrete 재료, transit-oriented)일 수 있음 |
# | v3 정규식 작성 | **미완**: 연구자가 `09_PA/PA_B8b_dict_v3.json`에 직접 작성 |
# | 제2코더 구성 | **미정**: 사람인지 AI 보조인지(AI면 계획서의 "코더 2명"과 어긋나므로 한계 절에 명시) |
# | `freeze → tag → push → sheet → 코딩 → score` | 미시작 |

# 보고 방식은 통과/실패 판정 없이 점추정과 Wilson 구간만입니다. 오탐 패턴을 보고 만든 사전이라 낙관 편향이 있습니다.

# ### 3-2. 미수행으로 확정된 항목 (부록에 사유 기재)
# - B4c: 임베딩 파일을 찾지 못함
# - A2의 ID 기반 재판별: F7이 2020년 이후만 덮음
# - B9의 LIC·LMC 소득군 내 분석: 질량 하한 때문에 자료상 불가

# ### 3-3. 정리·문서
# 1. A4 감사 표 최종 재생성(B6a·B6b·B4d 행, B4c 미수행 사유)
# 2. 통합 문구표와 약화·반전 공개 목록: C1, A2b y2, A2c, B4 첫 쌍, B4b 판독2, B4d, B6b 판정 불가 비율, B9a·b, B8a, B7c, B2·B3의 약한 시험
# 3. 재현성 잔여: `oa_raw` 외부 백업(SHA-256 일치 시에만 완료 기록), GitHub Private 확인, `S8.5` 노트북 커밋, 노출된 OpenAlex 키 2개 폐기 확인

# ### 3-4. 투고 준비
# - S9(후보 풀·추출 규칙 동결), S10(저널 선례 3편 이상, 심사 대응표, 생성형 AI 사용 선언)
# - 원고: 제목에서 "체제 전환" 삭제, 모든 표에 `[Post-hoc]`, 영문 post-hoc 문구 삽입, 기각·약화된 PA 목록을 부록에
# - 투고 후보 저널이 미정이라 구성안·초록 초안이 막혀 있습니다.

# ## 4. 지금 필요한 결정
# 1. 도메인 정의 문구 확인 후 v3 정규식 작성(연구자)
# 2. 제2코더가 사람인지 AI 보조인지
# 3. B8b를 할지, 미수행으로 확정할지
# 4. 투고 후보 저널

# 가장 가까운 다음 한 걸음은 코더에게 전달했던 도메인 정의 문구의 위치를 찾는 것입니다.

In [ ]:
# ## 판독 (이미 본 300편의 누락 제목)

# **infrastructure (FN 14)**: 10편이 `bridge`/`bridges`(Girder Bridges, Bridge Approach, Coastal Bridges, Steel Bridges, Highway Bridge, Bridge Connections, Bridge Rail)이고, 나머지는 `roughness index`, `slopes`, `unpaved roads`, `cross-slope`입니다. 즉 코더는 교량 구조 연구를 이 도메인으로 봤는데, v3의 `bridges? (deck|inspection|…)`가 너무 좁았습니다. `bridges?` 단독으로 되돌리면 대부분 복구됩니다. 오탐 쪽은 half-bridge 회로 한 편뿐이었으니 `half[- ]bridge`만 제외하는 편이 낫습니다. `roughness`, `slopes?`, `unpaved`, `distress\w*`도 후보입니다.

# **freight (FN 10)**: 6편이 `port`(단독 또는 port of/port selection/port governance), 2편이 배송(`delivery`, `drayage`, `pickup`), 나머지는 `cold chain`, `hazmat`, `ocean carrier`입니다. 코더는 항만 연구를 freight로 봤으니 `ports?`를 되살리는 쪽이 맞습니다. 앞서 오탐이던 "Greek ports" 크루즈 한 편은 감수할 만합니다. `drayage`, `cold chain`, `hazmat`, `ocean carrier`, `(same-day|meal|parcel) delivery`를 추가해 볼 수 있습니다.

# **active_urban (FN 5)**: 전부 `accessibility`(교통·지리적 접근성)입니다. 코더는 접근성 일반을 이 도메인에 넣었는데, v2의 오탐 중 `accessibility` 2편은 코더가 넣지 않았습니다. 코더 판단이 일관되지 않으니 정의 문제입니다. `accessibility`를 되살릴지는 `PA_B8b_definitions.md`의 정의에 접근성이 들어가는지로 정하세요. 들어가면 `(transport|transit|geographic|spatial) accessibility`로 한정하는 절충이 가능합니다.

# ## 편집 제안 (동결 전, 반복 가능)

# 연구자가 정의에 맞춰 결정하세요. 제안 초안은 다음과 같습니다.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 - <<'EOF'
# import json
# d = json.load(open("09_PA/PA_B8b_dict_v3.json", encoding="utf-8"))
# d["infrastructure"] = r"\b(pavements?|(?<!half[- ])bridges?|asphalt|concrete (pavements?|bridges?|decks?)|road maintenance|highway design|geometric design|structural health|tunnels?|rutting|roughness|unpaved|infrastructure (maintenance|management|asset))\b"
# d["freight"] = r"\b(freight|logistics?|trucks?|trucking|cargo|goods movement|supply chains?|ports?|drayage|cold chain|hazmat|ocean carrier|(same-day|parcel|meal) delivery|container\w*|last[- ]mile|warehouse\w*|shipping|intermodal (freight|transport|terminals?))\b"
# d["active_urban"] = r"\b(walk\w*|pedestrians?|cyclists?|cycling|bicycl\w*|bikes?|micromobility|scooters?|land use|(pedestrian|bicycle|walking|active travel|transport|transit|geographic) accessibility|urban form|transit[- ]oriented|parking|sidewalks?)\b"
# open("09_PA/PA_B8b_dict_v3.json", "w", encoding="utf-8").write(json.dumps(d, ensure_ascii=False, indent=1)); print("저장")
# EOF
# ```

# 그다음 앞의 점검 스크립트를 다시 돌려 precision과 recall을 보세요. 기준은 recall이 v2 수준(약 0.83~0.94)에 근접하면서 precision이 v2보다 높은 것입니다. 단, 이 점검은 v3를 만든 자료로 재는 것이라 낙관적입니다.

# ## 동결 전 체크

# - `PA_B8b_definitions.md`를 저장했는지(접근성, 교량 구조, 항만의 포함 여부를 한 문장씩).
# - 두 번째 점검까지 만족하면 커밋 → `freeze` → `tag` → push → `sheet` 순서로 진행합니다. `freeze` 이후에는 정규식을 못 바꿉니다.

# 점검 출력을 붙여 주시면 확인하겠습니다.

In [ ]:
# # 사전 v3 점검 판독

# 형식 점검은 통과했고, v3 네 도메인 모두 v2와 다른 정규식입니다.

# ## 1. 결과

# | 도메인 | precision v2 → v3 | recall v2 → v3 | 비고 |
# |---|---|---|---|
# | electrification_env | 0.774 → 0.932 | 0.872 → 0.872 | FP 12→3, FN 동일 |
# | freight | 0.811 → 0.872 | 0.833 → 0.944 | 둘 다 개선 |
# | active_urban | 0.689 → 0.833 | 0.939 → 0.909 | recall 소폭 하락 |
# | infrastructure | 0.846 → 0.972 | 0.892 → 0.946 | 둘 다 개선 |

# 앞서 정한 기준(recall이 v2 수준에 근접하면서 precision이 v2보다 높을 것)을 네 도메인 모두 충족합니다. recall 하락은 active_urban의 −0.03뿐입니다.

# ## 2. 해석할 때 주의할 점

# 1. **이 수치는 낙관적입니다.** v3는 이 300편의 오탐·누락을 보고 만들었고, 같은 300편으로 재었습니다. 사전이 표본에 맞춰진 상태라 새 표본의 precision은 이보다 낮게 나올 가능성이 큽니다. 점검 수치는 B8b 결과로 인용하지 마세요.
# 2. **참값이 코더 판단에 의존합니다.** FN이 주로 "정의 문제"였던 도메인(active_urban의 accessibility, infrastructure의 bridge)은 `PA_B8b_definitions.md`의 정의에 따라 정답 자체가 달라집니다. 정의 문서를 저장했는지 아직 확인하지 못했습니다.
# 3. **n_pos가 작습니다.** FP 1~6편 차이로 precision이 크게 움직이므로(infrastructure FP 1편 = 0.972), 도메인 간 우열을 읽을 수 없습니다.
# 4. **active_urban이 여전히 가장 약합니다.** FP 6, precision 0.833입니다. 다만 여기서 더 고치면 같은 300편에 더 과적합됩니다.

# ## 3. 권고

# **여기서 멈추고 동결하세요.** 정규식을 추가로 다듬는 것은 과적합만 늘립니다. 점검을 한 번 더 돌려 만족스러워질 때까지 반복하는 것이 가장 위험한 경로입니다. 동결 전 확인할 것은 세 가지입니다.

# - `09_PA/PA_B8b_definitions.md`가 저장돼 있고, 접근성·교량 구조·항만의 포함 여부가 한 문장씩 적혀 있는지
# - 제2코더가 사람인지 AI 보조인지 (AI면 계획서의 "코더 2명"과 어긋나므로 한계 절에 명시)
# - 점검 출력(위 표)을 changelog에 기록했는지. 이 수치를 본 뒤에 동결하는 것이므로 열람 사실을 남겨야 합니다.

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B8b 사전 v3 점검(이전 300편 기준, 동결 전): precision v2→v3 electrification_env 0.774→0.932, freight 0.811→0.872, active_urban 0.689→0.833, infrastructure 0.846→0.972; recall 0.872→0.872, 0.833→0.944, 0.939→0.909, 0.892→0.946. v3를 만든 같은 표본으로 잰 낙관적 수치이며 새 표본 결과 아님. 이 시점에서 정규식 동결, 이후 수정 금지 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_B8b_dict_v3.json paper1/09_PA/PA_CHANGELOG.md paper1/pa_b8b.py
# git status --short | head -n 8
# ```

# 스테이징에 예상 외 파일이 없으면 이어서 진행합니다.

# ```bash
# cd paper1 && python3 pa_b8b.py freeze
# ```
# ```bash
# python3 pa_b8b.py tag && cd .. && git push --follow-tags && cd paper1
# ```
# ```bash
# python3 pa_b8b.py sheet
# ```

# `sheet`가 이전 표본 제외 문제로 멈추면 `noprev`를 쓰지 말고 출력만 보내 주세요. 시트가 생성되면 판정 전에 빈 시트부터 커밋하세요.

# 각 단계는 한 블록씩, 한 번만 붙여 넣으세요. 앞서 중복 실행으로 changelog에 같은 줄이 두 번 들어간 일이 있었습니다.

In [ ]:
# # 두 가지가 아직 그대로입니다

# ## 1. 불일치 5행의 `final`이 여전히 비어 있습니다

# 점검 출력에서 item 5, 17, 33, 64, 89의 `final`과 `note`가 모두 빈칸입니다. 그래서 `score`가 같은 이유로 멈췄습니다. 파일이 바뀌지 않았다는 뜻이므로, 편집기에서 저장하지 않았거나 다른 경로의 파일을 고쳤을 수 있습니다. 이 5행의 값은 두 코더가 정의 문구에 비추어 합의하거나 연구자가 결정해야 하고, 제가 대신 정할 수 없습니다.

# 저장 후 확인은 아래 한 줄로 합니다. 값이 채워졌는지 보려면 경로를 먼저 확인하세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1/09_PA
# ls -l PA_B8b_sheet_filled.csv && date
# ```

# 수정 시각이 방금 저장한 시각이어야 합니다. 그다음 앞서 쓴 점검을 다시 실행해 `final 0/1만: True`가 나오면 `score`를 돌립니다.

# ## 2. changelog에 자리표시자가 세 번째로 올라갔습니다

# 방금 실행한 줄에 "(여기를 실제 방식으로 고침)"과 "(보존 또는 미보존)"이 그대로 기록됐습니다. 괄호를 고치지 않은 채 실행됐습니다. 이제 자리표시자 줄이 세 줄입니다. 지우지 말고, 실제 사실을 적은 정정 줄을 한 번만 올리세요. 아래 두 값은 실행 전에 직접 정해야 합니다.

# - 합의 방식: 실제로 한 대로 한 구절 (예: 두 코더 협의, 연구자 결정). 5행을 아직 채우지 않았다면 이 줄은 채운 뒤에 쓰세요.
# - 코더 개별 파일: 남아 있으면 "보존", 없으면 "미보존".

# 명령에 사용할 문장이 정해지면 그 문장을 그대로 알려 주세요. 제가 괄호 없는 완성된 명령으로 만들어 드리겠습니다. 그러면 직접 고치다가 빠뜨리는 일을 막을 수 있습니다.

# ## 진행 순서

# 1. 5행의 `final`과 `note`를 채워 저장
# 2. 점검에서 `final 0/1만: True` 확인
# 3. 합의 방식과 코더 파일 보존 상태를 알려 주시면 정정 줄 완성
# 4. `score` 실행

In [ ]:
# # 논문 1 사후분석(PA) 현황 (2026-10-21 기준)

# 공통 원칙은 실행 전 해시·태그 동결, GitHub 푸시, 결과 전부 보고, S0~S8 판정 불변입니다. 아래 수치는 문서 내 출력과 changelog 해석 메모에서 옮긴 것이니, 본문에 쓰기 전에 원 CSV와 대조하세요.

# ## 1. 하려고 한 분석 (지침서 14개 항목, 핵심은 A1·A2·B2)

# | 구분 | 항목 | 질문 |
# |---|---|---|
# | 재현성 | PA-0 | 파일·코드·태그가 온전한가 |
# | 강점 강화 | A1 | N2 비통과는 검정력 부족인가, 점프 부재인가 |
# | | A2 | 진입 우세는 정의·부트스트랩·이름 식별에 견고한가 |
# | | A3 | 선두 국가 교체 서술이 견고한가 |
# | 약점 리프레이밍 | B2 | 2020 변화점이 표본 크기·저널 구성의 아티팩트인가 |
# | | B3 | 신생 토픽 정의 4종의 핵심 집합은 무엇인가 |
# | | B4 | 등급 선택이 결과를 좌우하는가 |
# | | B5 | S4 안정성 0.700은 경계인가 |
# | | B6 | 위험군에 독립 정답이 있는가 |
# | | B7 | 임베딩 결측 16%가 결과를 뒤집는가 |
# | | B8 | 미달 4개 도메인의 정렬은 어떤가 |
# | | B9 | 간극이 소득 수준과 분리되는가 |
# | 정합성 | C1 | 진입 우세가 중국 소속 유입과 독립인가 |
# | | C2 | S7(전체 논문)과 S6(임베딩 보유 논문)의 모집단 차이는 얼마인가 |

# ## 2. 지금까지 수행한 분석과 결과

# | PA | 동결 판독 | 쓸 수 있는 말 / 주의 |
# |---|---|---|
# | PA-0 | 해시 152건 일치 | S4 코드는 복구본(최종 실행본 증명 불가), `oa_raw` 외부 백업 미완 |
# | A1 | run 1 무효(주입 NaN 버그, 보존), run 2: all·within 강화, fixed14 곡선만 보고 | 2020 부근은 관측 초과분 크기 점프를 탐지했을 가능성이 높음. 2008~2012는 사각지대. δ=1 점프는 관측 초과분의 약 3배 |
# | A2a | 4쌍 모두 둔감(변화 ≤0.011) | ID는 2020+만 있어 ID 재판별은 불가, 분할 이름 제외 민감도로 축소 |
# | A2b | s6 3쌍, y5 3쌍, **y2 0쌍(뒤집힘)** | "신규 저자 주도"가 아니라 "진입(신규+복귀) 주도" |
# | A2c | 3쌍 유지(첫 쌍 제외) | 두 구간 방식 모두 확실한 쌍은 마지막뿐. excess 구간은 교정 불충분 |
# | A3 | 방향 유지 | 알려진 국가 기준 top5 몫 평탄(0.64→0.65), HHI 반등은 중국 몫과 연동 |
# | B2 | 3표본 유지 | 하향 추출이 20%뿐인 약한 시험 |
# | B2b | 단독 제외는 유지, 동시 제외(상위 5저널+토픽 71)는 약화 | 2020은 단일 원인이 아닌 겹친 가속 |
# | B3a·b·c | 핵심 7·중간 10·특이적 14 | 정의 구조의 부산물, 독립 합의 증거 아님 |
# | B4 | S5 둔감, S6 3쌍 둔감 | 첫 쌍만 부호 변화, 제외 4개뿐인 약한 시험 |
# | B4b | 판독1 둔감, 판독2 규칙상 민감 | 안정성 +0.027/+0.030, 임계 0.03이 토픽 1개 규모 |
# | B4d | S5 통과 판정 동일, 앞쪽 변화점 이동·소실. S6 첫 쌍만 민감 | 제외 15개는 B4의 확장, 독립 확인 아님 |
# | B5 | 부호 반전 0, 안정성 0.66~0.77 | "0.70 전후에서 움직임" |
# | B6a | 정밀도 0.789 [0.767, 0.810], 재현율 0.330 [0.314, 0.347] | ORCID 보유·2020+ 한정, 순환성 해소 아님 |
# | B6b | 동일인 9·다른 사람 11·판정 불가 30, 경계 0.13~0.90 | 구간이 매우 넓어 점추정 단독 인용 금지 |
# | B7a·b·c | 모두 유지(big 50/50) | 민감도이며 경계가 아님 |
# | B8a | 8쌍 중 7쌍 HIC 과잉·LIC/LMC 과소 | 참고용, 본문 판정 2/6 불변 |
# | B9a·b | HIC 비관찰, UMC는 하한 5에서만 관찰 | LIC·LMC는 자료상 불가 |
# | **C1** | 3개 시나리오 모두 유지 안 됨(+0.113 → −0.101, −0.110, −0.024) | 진입 우세와 중국 소속 유입은 분리 불가 |
# | C2 | 20칸 중 일치 11 | 중국 몫 일치, 미국·상위5는 +0.01~0.03 높음 |
# | A4 | 감사 표 31항목 생성·커밋 | B6a·B6b·B4d 행 반영한 최종본 재생성 필요 |

# **종합**
# - **쓸 수 있는 것:** 변화는 점프보다 연속적 재배치로 서술하는 편이 자료에 부합합니다. 2020 부근은 COVID와 기술 주제가 겹친 가속입니다. 진입 우세는 마지막 구간에서 가장 뚜렷합니다.
# - **쓰면 안 되는 것:** 체제 전환 부재 확정, 2020이 아티팩트가 아니라는 확정, 신규 저자가 모든 구간을 주도했다는 문장, 진입 주도가 중국 유입과 무관하다는 문장, ID 기반 일치, 인과 해석.
# - **가장 큰 구조 변화:** 진입 중심 갱신과 중국 몫 증가는 독립된 두 기여가 아니라 한 구조의 두 측면으로 묶어야 합니다.

# ## 3. 앞으로 해야 할 분석과 작업

# ### 3-1. 진행 중: B8b (사전 v3, 새 25편 이중 코딩)

# | 단계 | 상태 |
# |---|---|
# | 이전 검증 표본 제외 패치(`coder1/2.csv` 300편) | 완료, 동결 전 |
# | 사전 v3 점검(이전 300편 기준) | 완료. precision 모두 상승, recall 유지 수준. 같은 표본으로 잰 낙관적 수치라 B8b 결과로 인용 금지 |
# | 도메인 정의 문서(`PA_B8b_definitions.md`) | 저장 여부 미확인 |
# | 제2코더 구성 | 미정(AI 보조면 "코더 2명"과 어긋나므로 한계 절에 명시) |
# | **불일치 5행(item 5, 17, 33, 64, 89)의 `final`·`note`** | **여전히 비어 있음.** 두 코더 합의 또는 연구자 결정이 필요하며 제가 대신 정할 수 없음 |
# | changelog 자리표시자 줄 | 3회 올라감. 삭제하지 않고, 실제 합의 방식과 코더 파일 보존 여부를 적은 정정 줄을 **한 번만** 추가해야 함 |
# | 이후 순서 | `final` 채움 → 점검(`final 0/1만: True`) → 정정 줄 → `score` |

# 보고는 통과/실패 판정 없이 점추정과 Wilson 구간만 합니다. 오탐 패턴을 보고 만든 사전이라 낙관 편향이 있고, 모든 표에 "사후 개정 사전 v3"로 표기하며 S8 판정(2/6)은 바꾸지 않습니다.

# ### 3-2. 미수행으로 확정된 항목 (부록에 사유 기재)
# - B4c(임베딩 응집도): 임베딩 파일을 찾지 못함
# - A2의 ID 기반 재판별: F7이 2020년 이후 논문만 덮음
# - B9의 LIC·LMC 소득군 내 분석: 질량 하한 때문에 자료상 불가

# ### 3-3. 정리·문서
# 1. A4 감사 표 최종 재생성(B6a·B6b·B4d 행, B4c 미수행 사유)
# 2. 통합 문구표와 약화·반전 공개 목록: C1, A2b y2, A2c, B4 첫 쌍, B4b 판독2, B4d, B6b 판정 불가 비율, B9a·b, B8a, B7c(2009 변화점 이동), B2·B3의 약한 시험
# 3. 재현성 잔여: `oa_raw` 외부 백업(SHA-256 일치 시에만 완료 기록), GitHub Private 확인, `S8.5` 노트북 커밋, 노출된 OpenAlex 키 2개 폐기 확인

# ### 3-4. 투고 준비
# - S9(후보 풀·추출 규칙 동결), S10(저널 선례 3편 이상, 심사 대응표, 생성형 AI 사용 선언)
# - 원고: 제목에서 "체제 전환" 삭제, 진입 중심 갱신과 중국 소속 유입을 한 구조로 서술, 모든 표에 `[Post-hoc]`, 영문 post-hoc 문구 삽입, 기각·약화된 PA 목록을 부록에 수록
# - 투고 후보 저널이 미정이라 구성안·초록 초안이 막혀 있습니다.

# ## 4. 지금 필요한 것

# 1. 불일치 5행의 `final`·`note`를 채워 저장. 수정 시각을 `ls -l`로 확인하세요.
# 2. 합의 방식과 코더 개별 파일 보존 여부를 알려 주시면, 괄호 없는 완성된 정정 명령을 드립니다.
# 3. 제2코더가 사람인지 AI 보조인지
# 4. 투고 후보 저널

# 가장 가까운 다음 한 걸음은 1번입니다.

In [ ]:
# 파일이 두 번 확인됐고, 크기와 시각이 바뀌었습니다(16:55에 13,223바이트, 17:05에 15,136바이트). 방금 17:05에 저장된 것이니 수정은 반영된 것으로 보입니다. 다만 내용이 맞는지는 이 출력만으로 알 수 없으니, 아래 점검으로 확인하세요(읽기 전용).

# ```bash
# cd ~/Research/Minnesota_Transport/paper1/09_PA
# python3 - <<'EOF'
# import pandas as pd
# b = pd.read_csv("PA_B8b_sheet_filled.csv", encoding="utf-8-sig")
# a = pd.read_csv("PA_B8b_sheet.csv", encoding="utf-8-sig")
# key = ["item_id", "domain", "pid"]
# print("행 수:", len(b), "| 키·순서 원본과 동일:", a[key].astype(str).equals(b[key].astype(str)))
# for c in ["coder1", "coder2", "final"]:
#     b[c] = pd.to_numeric(b[c], errors="coerce")
#     print(c, "0/1만:", bool(b[c].isin([0, 1]).all()), "| 빈칸:", int(b[c].isna().sum()))
# d = b[b.coder1 != b.coder2]
# print("불일치 행 수:", len(d))
# print(d[["item_id", "domain", "coder1", "coder2", "final", "note"]].to_string(index=False))
# ok = b[b.coder1 == b.coder2]
# print("일치 행 final=코더값:", bool((ok.final == ok.coder1).all()))
# EOF
# ```

# 확인할 점은 다음과 같습니다.

# - `행 수: 100`, `키·순서 원본과 동일: True`
# - 세 칸 모두 `0/1만: True`, 빈칸 0
# - 불일치 행이 5개(5, 17, 33, 64, 89)이고 `final`과 `note`가 채워져 있음
# - `일치 행 final=코더값: True`

# 전부 만족하면 바로 `score`로 넘어갈 수 있습니다.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 pa_b8b.py score 2>&1 | tee 09_PA/PA_B8b_full_log.txt
# ```

# `score` 전에 한 가지만 정해 주세요. 앞서 자리표시자가 올라간 changelog 정정 줄에 쓸 두 값입니다.

# 1. 5행의 합의 방식(예: 두 코더 협의, 연구자 결정)
# 2. `coder1`/`coder2` 개별 파일을 보존했는지

# 답을 주시면 괄호 없는 완성 명령을 드립니다. 점검 출력도 붙여 주세요.

# yjlee@u24:~/Research/Minnesota_Transport/paper1/09_PA$ cd ~/Research/Minnesota_Transport/paper1/09_PA
# python3 - <<'EOF'
# import pandas as pd
# b = pd.read_csv("PA_B8b_sheet_filled.csv", encoding="utf-8-sig")
# a = pd.read_csv("PA_B8b_sheet.csv", encoding="utf-8-sig")
# key = ["item_id", "domain", "pid"]
# print("행 수:", len(b), "| 키·순서 원본과 동일:", a[key].astype(str).equals(b[key].astype(str)))
# for c in ["coder1", "coder2", "final"]:
#     b[c] = pd.to_numeric(b[c], errors="coerce")
#     print(c, "0/1만:", bool(b[c].isin([0, 1]).all()), "| 빈칸:", int(b[c].isna().sum()))
# d = b[b.coder1 != b.coder2]
# print("불일치 행 수:", len(d))
# print(d[["item_id", "domain", "coder1", "coder2", "final", "note"]].to_string(index=False))
# ok = b[b.coder1 == b.coder2]
# print("일치 행 final=코더값:", bool((ok.final == ok.coder1).all()))
# EOF
# 행 수: 100 | 키·순서 원본과 동일: True
# coder1 0/1만: True | 빈칸: 0
# coder2 0/1만: True | 빈칸: 0
# final 0/1만: True | 빈칸: 0
# 불일치 행 수: 5
#  item_id              domain  coder1  coder2  final                                                                       note
#        5      infrastructure       0       1      1                          연구자 결정: 고속도로 터널 진입부의 시각 안내 시스템과 교통시설 운영을 다루므로 포함.
#       17        active_urban       0       1      0               연구자 결정: 보행자 횡단부는 맥락이며 핵심 결과는 운전자 공격성으로 active_urban 범위에서 제외.
#       33        active_urban       1       0      0                            연구자 결정: 주차비와 자동차 보유의 관계가 중심이며 능동교통 연구가 아니므로 제외.
#       64 electrification_env       0       1      0 연구자 결정: 보행 경로·교통소음 연구로, electrification_env의 에너지·전동화·배출 범위에 직접 해당하지 않아 제외.
#       89      infrastructure       0       1      0                             연구자 결정: 제목만으로 도로·포장 인프라 맥락을 확인할 수 없어 보수적으로 제외.
# 일치 행 final=코더값: True
# yjlee@u24:~/Research/Minnesota_Transport/paper1/09_PA$

In [ ]:
# 점검이 모두 통과했습니다. 행 수 100, 키·순서 원본과 동일, 세 칸 모두 0/1만 있고 빈칸 0, 불일치 5행은 `final`과 `note`가 채워져 있고, 일치 행의 `final`은 코더값과 같습니다. `score`를 돌려도 됩니다.

# ## 알아 둘 점

# - 5행의 합의 방식이 전부 "연구자 결정"으로 기록됐습니다. 두 코더 협의가 아니라 연구자 1인이 정한 것이므로, 정정 줄과 한계 절에 그대로 쓰세요. 이전에 질문드린 "합의 방식"의 답은 이것으로 정해진 셈입니다.
# - 5건 중 4건이 `final=0`입니다(item 17, 33, 64, 89). 그만큼 precision이 보수적으로 나옵니다. 판정 기준은 `note`에 적힌 이유(범위 정의)에 따른 것이니 점검 후에는 바꾸지 마세요. `score` 이후에 `final`을 바꾸면 결과를 본 뒤의 수정이 됩니다.
# - item 5와 89는 같은 도메인(infrastructure)인데 하나는 포함, 하나는 "제목만으로 맥락 확인 불가"로 제외했습니다. 판정 근거가 서로 다른 유형이라, 부록에 근거를 같이 적어 두는 편이 안전합니다.

# ## 실행

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# python3 pa_b8b.py score 2>&1 | tee 09_PA/PA_B8b_full_log.txt
# ```

# 한 번만 실행하세요. `score`가 changelog에 줄을 자동으로 추가하므로 중복 실행하면 같은 줄이 두 번 들어갑니다.

# ## 정정 줄 (괄호 없는 완성본)

# 자리표시자가 올라간 세 줄에 대한 정정입니다. 한 번만 실행하세요. 코더 개별 파일 보존 여부는 아직 답을 받지 못해서 이 줄에는 넣지 않았습니다. 보존 여부를 알려 주시면 별도 한 줄로 만들어 드립니다.

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B8b 정정: 앞선 changelog 3줄의 자리표시자 문구('여기를 실제 방식으로 고침', '보존 또는 미보존')는 실제 기록이 아님. 실제 사실: 코더 불일치 5행(item 5, 17, 33, 64, 89)의 final은 연구자 1인이 결정했고(두 코더 협의 아님), 사유는 PA_B8b_sheet_filled.csv의 note에 기재. 점검 결과 행 수 100, 키·순서 원본과 동일, coder1·coder2·final 모두 0/1만이며 빈칸 없음, 일치 행의 final은 코더값과 같음. score 실행 전 기록 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# tail -n 1 paper1/09_PA/PA_CHANGELOG.md | cut -c1-160
# ```

# ## 그다음

# `score` 로그 전체를 붙여 주세요. 동결 규칙대로 도메인별 precision, Wilson 구간, kappa, v2 precision만 읽겠습니다. 통과/실패 판정은 하지 않습니다.

In [ ]:
# # PA-B8b 판독

# `score`는 정상 종료됐고 오류는 없습니다. 동결 규칙에 따라 통과/실패 판정 없이 점추정, Wilson 구간, kappa, v2 precision만 읽습니다.

# ## 1. 결과

# | 도메인 | precision v2 | precision v3 | Wilson 95% | kappa | 불일치 |
# |---|---|---|---|---|---|
# | active_urban | 0.76 | 0.80 (20/25) | [0.609, 0.911] | 0.702 | 2 |
# | electrification_env | 0.80 | 0.96 (24/25) | [0.805, 0.993] | 0.000 | 1 |
# | freight | 0.80 | 1.00 (25/25) | [0.867, 1.000] | NaN | 0 |
# | infrastructure | 0.84 | 0.96 (24/25) | [0.805, 0.993] | 0.000 | 2 |
# | ALL | – | 0.93 (93/100) | [0.863, 0.966] | 0.521 | 5 |

# ## 2. 읽기

# 1. **active_urban만 v3에서도 0.80입니다.** 구간 [0.609, 0.911]이 0.85를 걸치고, 점추정은 v2(0.76)보다 약간 높을 뿐입니다. 구간이 겹치므로 v2 대비 개선이라고 쓰기 어렵습니다.
# 2. **나머지 세 도메인은 점추정이 크게 올랐습니다.** 다만 구간 하한이 0.805~0.867이라 0.85와 구분되지 않는 도메인이 있습니다(electrification_env, infrastructure의 하한 0.805). 동결 규칙상 통과/실패는 선언하지 않습니다.
# 3. **kappa는 해석하지 마세요.** electrification_env와 infrastructure의 0.000은 코더 일치가 나빠서가 아니라, 양성이 거의 전부라 우연 일치 기대치가 높아지는 kappa의 알려진 왜곡입니다. freight의 NaN은 둘 다 전부 1이라 분모가 0이기 때문입니다. 대신 불일치 수(0~2건)와 원자료 일치율(95건/100)을 보는 편이 정확합니다. 전체 kappa 0.521도 같은 이유로 낮게 나옵니다.
# 4. **ALL 0.93은 도메인 평균이 아닌 합산**이며, 도메인별 값과 합쳐 하나의 결론으로 쓰지 않는다고 계획서에 적었습니다.

# ## 3. 반드시 함께 적을 한계

# - **낙관 편향:** v3는 오탐 패턴과 이전 300편의 누락을 보고 만든 사전입니다. 새 표본에서 재도 사전이 이전 자료에 맞춰진 효과를 완전히 배제할 수 없습니다.
# - **코더 구성:** 불일치 5건의 `final`은 연구자 1인이 결정했습니다(두 코더 협의 아님). 5건 중 4건이 `final=0`으로 보수적이고, 이 결정이 precision을 낮추는 방향입니다. 제2코더가 사람인지 AI 보조인지는 아직 미기재라 부록에 명시해야 합니다.
# - **recall 미추정**, n=25의 넓은 구간, 4개 도메인을 각각 시도한 다중성(하나쯤 우연히 높게 나올 수 있음).
# - **독립 이중 코딩 여부:** 이 점이 확인되지 않으면 "이중 코딩"이라는 표현은 쓰지 마세요.

# ## 4. 쓸 수 있는 문구

# > "사후에 개정한 사전 v3(오탐·누락 패턴을 본 뒤 작성)으로 새 무작위 25편씩을 코딩한 결과, 도메인별 precision은 active_urban 0.80 [0.61, 0.91], electrification_env 0.96 [0.81, 0.99], freight 1.00 [0.87, 1.00], infrastructure 0.96 [0.81, 0.99]였다(v2: 0.76, 0.80, 0.80, 0.84). 표본이 작고 사전이 이전 자료를 본 뒤 개정되었으므로 통과/실패 판정은 하지 않으며 S8 판정(2/6 통과)은 바뀌지 않는다."

# 쓰면 안 되는 것: "미달 도메인이 v3로 기준을 통과했다", "4개 도메인을 본문 정렬 분석에 포함한다", kappa 수치 단독 인용.

# ## 5. 기록과 커밋

# `score`가 changelog에 실행 줄을 자동 추가했을 것입니다. 해석 메모를 한 번만 추가하고 올리세요.

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B8b 결과 해석 메모(사후 개정 사전 v3, 통과/실패 판정 없음): precision v3 active_urban 0.80 [0.609,0.911], electrification_env 0.96 [0.805,0.993], freight 1.00 [0.867,1.000], infrastructure 0.96 [0.805,0.993], ALL 0.93 [0.863,0.966](v2 0.76/0.80/0.80/0.84). kappa는 양성 편중으로 0.000·NaN이 나와 비해석(불일치 수 0~2건, 원자료 일치 95/100). 불일치 5건 final은 연구자 1인 결정(4건 final=0). 사전이 오탐·누락을 본 뒤 개정되어 낙관 편향, recall 미추정, 제2코더 구성 미기재. S8 판정 2/6 불변 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_B8b_results.csv paper1/09_PA/PA_B8b_full_log.txt paper1/09_PA/PA_B8b_sheet_filled.csv paper1/09_PA/PA_CHANGELOG.md
# git status --short | head -n 8
# ```

# 스테이징 목록에 예상 외 파일이 없으면 이어서 실행하세요.

# ```bash
# git commit -m "PA-B8b results" && git tag -a PA-B8b-results-2026-10-21 -m "PA-B8b outputs, seen" && git push --follow-tags
# ```

# ## 6. 이제 분석은 모두 끝났습니다

# 수행 가능한 PA는 전부 소진됐습니다. 남은 것은 다음과 같습니다.

# 1. **제2코더 구성 기재:** 사람이었는지 AI 보조였는지, `coder1`/`coder2` 개별 파일 보존 여부(앞서 질문드린 항목)
# 2. **A4 최종 재생성:** B6a·B6b·B4d·B8b 행 반영
# 3. **통합 문구표와 약화·반전 공개 목록:** C1, A2b y2, B4 첫 쌍, B4b 판독2, B4d, B6b 판정 불가, B8b(active_urban 0.80) 등
# 4. **재현성 잔여:** `oa_raw` 외부 백업, GitHub Private 확인, `S8.5` 노트북 커밋, 노출된 OpenAlex 키 폐기 확인
# 5. **투고 준비:** 후보 저널을 정해야 S9·S10과 구성안으로 넘어갑니다.

# 1번 답을 주시면 B8b 기록을 완결하고, A4 최종 재생성으로 넘어가겠습니다.

In [ ]:
# PA-B8b가 정상적으로 기록됐습니다. 커밋 `c79dd91`과 태그 `PA-B8b-results-2026-10-21`이 원격에 올라갔고, 스테이징도 예상한 4개뿐이었습니다. `S8.5` 노트북과 `Atlas_*`, `ISF_*` 파일은 의도대로 커밋하지 않았습니다.

# 이로써 수행 가능한 PA 분석은 모두 끝났습니다. 남은 일은 기록 보완, 문서 작업, 투고 준비입니다.

# ## 1. 아직 기록되지 않은 것

# B8b 기록에 두 가지가 비어 있습니다. 해석 메모에도 "제2코더 구성 미기재"라고 적어 두었습니다.

# - **제2코더:** 사람이었는지 AI 보조였는지 알려 주세요. AI 보조였다면 "독립 이중 코딩"이라고 쓸 수 없고 한계 절에 명시해야 합니다.
# - **`coder1`/`coder2` 개별 파일:** 보존했는지 알려 주세요.

# 두 답을 주시면 괄호 없는 완성된 changelog 한 줄을 드립니다.

# ## 2. 순서대로 할 일

# 1. **A4 감사 표 최종 재생성:** `MANUAL`에 B6a, B6b, B4d, B8b 행을 반영하고 `pa_a4.py`를 다시 실행합니다.
# 2. **통합 문구표와 약화·반전 공개 목록:**
#    - C1: 진입 우세가 중국 소속 저자를 제외하면 반전
#    - A2b y2, A2c, B4 첫 쌍, B4b 판독 2, B4d
#    - B6b: 판정 불가 비율이 높고 경계가 0.13~0.90
#    - B8b: active_urban이 v3에서도 0.80
#    - B9a·b, B8a, B7c, B2·B3의 약한 시험
# 3. **재현성 잔여:**
#    - `oa_raw` 외부 백업: SHA-256이 일치할 때만 완료로 기록
#    - GitHub 저장소 Private 확인
#    - `S8.5` 노트북 커밋
#    - 노출된 OpenAlex 키 2개 폐기 확인
# 4. **투고 준비:** 후보 저널을 정해야 S9·S10(후보 풀 동결, 저널 선례 3편 이상, 심사 대응표, 생성형 AI 사용 선언)과 구성안·초록 초안으로 넘어갈 수 있습니다.

# ## 3. 지금 필요한 답

# 1. 제2코더가 사람인지 AI 보조인지, 개별 코더 파일 보존 여부
# 2. 투고 후보 저널

# 가장 가까운 다음 한 걸음은 1번 답으로 B8b 기록을 마무리하는 것입니다.

In [ ]:
# 기록이 올라갔습니다(커밋 `8a48f9f`). B8b의 제2코더 기록은 이것으로 완결입니다.

# 한 가지만 정정합니다. 앞서 "개별 코더 파일 보존 여부는 별도 기록 예정"이라고 적었는데, 아직 답을 받지 못했습니다. 아래 한 줄로 확인하면 추측 없이 바로 기록할 수 있습니다(읽기 전용).

# ```bash
# cd ~/Research/Minnesota_Transport
# ls -l paper1/09_PA | grep -i -E "coder|sheet"
# git ls-files paper1/09_PA | grep -i -E "coder|sheet"
# ```

# - `PA_B8b_sheet.csv`(빈 원본)와 `PA_B8b_sheet_filled.csv`만 보이면, 두 코더의 개별 파일은 저장소와 `09_PA`에 없다는 뜻입니다. 이 경우 "미보존(합친 파일만 있음)"으로 기록하고 한계 절에 적습니다.
# - 코더별 파일이 다른 폴더에 있다면 그 경로를 알려 주세요. 보존으로 기록하고 해시를 남기는 편이 좋습니다.

# 이 출력을 붙여 주시면 괄호 없는 완성 changelog 한 줄을 드리겠습니다.

# ## 이후 순서

# 1. 개별 코더 파일 보존 여부 기록(위 출력)
# 2. A4 감사 표 최종 재생성(B6a, B6b, B4d, B8b 행 반영)
# 3. 통합 문구표와 약화·반전 공개 목록 작성
# 4. 재현성 잔여: `oa_raw` 외부 백업(SHA-256 일치 시에만 완료 기록), GitHub Private 확인, `S8.5` 노트북 커밋, 노출된 OpenAlex 키 폐기 확인
# 5. 투고 후보 저널 결정 후 S9·S10

# 투고 후보 저널도 정해 주시면 구성안과 초록 초안 작업을 병행할 수 있습니다.

In [ ]:
# # 코더 파일 보존 상태 확인

# 출력으로 확인된 사실은 이렇습니다.

# - **`coder1`/`coder2` 개별 파일은 `09_PA`와 저장소 어디에도 없습니다.** 두 코더의 판정은 `PA_B8b_sheet_filled.csv`의 `coder1`, `coder2` 열에만 남아 있습니다. 열이 분리돼 있으니 각 코더의 판정 자체는 복원 가능하지만, 코더가 각자 작성한 원본 파일은 보존되지 않았습니다.
# - **`PA_B8b_sheet_filled_before_final.csv`가 있습니다**(16:54~16:55, 13,229바이트). `final` 5행을 채우기 전 상태로 보이는데, **git에는 추적되지 않습니다**(`git ls-files`에 없음). 연구자 결정 전후를 구분하는 증거가 될 수 있으니 커밋해 두는 편이 좋습니다. 단, 내용은 제가 열어 보지 못했습니다. 아래에서 확인하세요.
# - 앞서 두 번 확인한 `PA_B8b_sheet_filled.csv`의 크기 13,223 → 13,229(before_final) → 15,136(최종)은 서로 어긋납니다. 13,223과 13,229는 다른 시점의 파일일 수 있어, before_final이 어느 시점의 사본인지 내용으로 확인해야 합니다.

# ## 확인 (읽기 전용)

# ```bash
# cd ~/Research/Minnesota_Transport/paper1/09_PA
# python3 - <<'EOF'
# import pandas as pd
# a = pd.read_csv("PA_B8b_sheet_filled_before_final.csv", encoding="utf-8-sig")
# b = pd.read_csv("PA_B8b_sheet_filled.csv", encoding="utf-8-sig")
# print("행 수:", len(a), len(b))
# for c in ["coder1", "coder2"]:
#     print(c, "두 파일 동일:", a[c].astype(str).equals(b[c].astype(str)))
# d = a[a.coder1 != a.coder2]
# print("before_final 불일치 행:", d.item_id.tolist())
# print(d[["item_id", "coder1", "coder2", "final", "note"]].to_string(index=False))
# chg = a.final.astype(str).ne(b.final.astype(str))
# print("final이 바뀐 item_id:", b.item_id[chg].tolist())
# EOF
# ```

# 기대하는 결과는 `coder1`/`coder2`가 두 파일에서 동일하고, `final`이 바뀐 행이 불일치 5행뿐인 것입니다. 이 경우 코더 판정이 연구자 결정 단계에서 수정되지 않았다는 근거가 됩니다. 다르면 출력을 알려 주세요.

# ## 기록 (확인 결과가 기대와 같을 때만 실행)

# 한 번만 실행하세요. 확인 결과가 다르면 이 줄을 쓰지 마세요.

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | PA-B8b 코더 파일 보존 상태: coder1·coder2 개별 원본 파일은 보존되지 않음(저장소와 09_PA에 없음). 두 코더의 판정은 PA_B8b_sheet_filled.csv의 coder1·coder2 열에만 존재. final 채우기 전 사본 PA_B8b_sheet_filled_before_final.csv가 있었으나 그때까지 git 미추적이었고 이 줄과 함께 커밋. 한계 절에 '코더 개별 원본 미보존'을 명시 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/09_PA/PA_B8b_sheet_filled_before_final.csv paper1/09_PA/PA_CHANGELOG.md
# git status --short | head -n 5
# ```

# 스테이징에 이 두 파일만 있으면 이어서 실행하세요.

# ```bash
# git commit -m "PA-B8b: record coder file preservation, add before_final sheet" && git push
# ```

# ## 한계 절 문구

# > 두 코더의 개별 원본 파일은 보존되지 않았으며, 판정은 통합 시트의 coder1·coder2 열로만 남아 있다. 코더 불일치 5건의 최종 판정은 연구자 1인이 내렸다.

# 이 정정까지 올라가면 B8b 기록이 완결됩니다. 다음은 A4 감사 표 재생성이고, 투고 후보 저널 답은 아직 기다리고 있습니다.

In [ ]:
# B8b 기록이 완결됐습니다(커밋 `a9f8529`, 푸시 완료). 확인 결과는 해석과 맞았습니다.

# - `before_final`의 `final`은 100/100 빈칸이었고, 최종본에서 일치 95행의 `final`은 모두 코더 값과 같습니다.
# - 앞서 "100행 전부 바뀜"으로 나온 것은 빈칸(NaN)을 문자열로 비교한 탓이었고, 이상 신호가 아니었습니다.
# - 코더 열은 두 파일에서 동일하므로, 코더 판정이 연구자 결정 단계에서 수정되지 않았다는 점이 기록됐습니다.
# - 스테이징은 예상한 2개뿐이었고, `S8.5`와 `Atlas_*` 파일은 의도대로 커밋하지 않았습니다.

# 이로써 수행 가능한 PA 분석과 B8b의 기록 보완은 모두 끝났습니다.

# ## 남은 일

# 1. **A4 감사 표 최종 재생성.** `pa_a4.py`의 `MANUAL`에 B6a, B6b, B4d, B8b 행을 반영합니다. 이번에 확정된 사실도 넣습니다: 제2코더는 사람, 불일치 5건은 연구자 1인 결정, 코더 개별 원본 미보존, `before_final` 사본 존재.
# 2. **통합 문구표와 약화·반전 공개 목록.** C1, A2b y2, A2c, B4 첫 쌍, B4b 판독 2, B4d, B6b(판정 불가 비율 높음), B8b(active_urban 0.80), B9a·b, B8a, B7c, B2·B3의 약한 시험을 한계 절과 부록에 먼저 공개합니다.
# 3. **재현성 잔여.**
#    - `oa_raw` 외부 백업: SHA-256이 일치할 때만 완료로 기록
#    - GitHub 저장소 Private 확인
#    - `S8.5` 노트북 커밋
#    - 노출된 OpenAlex 키 2개 폐기 확인
# 4. **투고 준비.** S9(후보 풀·추출 규칙 동결), S10(저널 선례 3편 이상, 심사 대응표, 생성형 AI 사용 선언)과 구성안·초록 초안.

# ## 필요한 답

# 투고 후보 저널을 정해 주시면 4번을 시작할 수 있습니다. 그 전에 바로 할 수 있는 것은 1번입니다. 원하시면 `MANUAL` 보정 문구를 정리해 드리겠습니다.

In [ ]:
# # 마무리 작업 명령 모음 (투고 저널 선정 제외)

# 분석은 모두 끝났으므로 아래는 기록 확정, 최종 결론 문서, 재현성 정리입니다. **블록 하나씩, 한 번만** 실행하고, 오류나 `⛔`가 나오면 멈추고 출력만 보내 주세요. 자리표시자가 필요한 곳은 인자가 없으면 스크립트가 중단되도록 했습니다.

# ## 블록 1. A4 감사 표 최종 재생성

# `replace()`가 조용히 실패한 전례가 있어 assert로 막았습니다. 기존 `MANUAL`은 건드리지 않고 `def norm` 앞에 갱신 줄을 넣어, 같은 키는 새 값이 덮어씁니다.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1
# cp pa_a4.py /tmp/pa_a4.bak
# python3 - <<'EOF'
# p = "pa_a4.py"; s = open(p, encoding="utf-8").read()
# assert s.count("def norm(s)") == 1, "def norm 위치 불명확"
# assert "MANUAL.update" not in s, "이미 패치됨"
# add = '''NOT_DONE.clear(); NOT_DONE.update({"B4c": "미수행: 임베딩 파일을 찾지 못함", "A2-ID": "미수행: F7(OpenAlex ID)이 2020년 이후 논문만 덮어 ID 재판별 불가, 분할 이름 제외로 대체", "B9-LIC": "자료상 불가: 질량 하한으로 LIC 0개국, LMC n<20"})
# MANUAL.update({
#  "B6a": "수집 중단 원인을 두 번 오진(이메일 자리표시자, 순간 속도 제한); 실제는 키 없는 요청의 공유 IP 일일 예산 소진(429). 키 도입 첫 시도는 패치 미적용으로 실패 후 직접 수정. API 키 노출 2회(저장소 이력에는 없음, 폐기 조치). 해시 자리표시자 정정 줄 중복 기록(유효 1건). 결과는 ORCID 보유·2020+ 한정",
#  "B6b": "연구자 1인 판정. 동일인 9, 다른 사람 11, 판정 불가 30; 경계 0.13~0.90으로 점추정 단독 인용 금지",
#  "B4d": "제외 15개에 B4의 F1·F3 실패 4개 포함(확장, 독립 확인 아님). 앞쪽 변화점 이동·소실은 벌점 효과 가능성(미확인)",
#  "B8b": "사후 개정 사전 v3(오탐·누락을 본 뒤 작성, 낙관 편향). 제2코더는 사람. 불일치 5건(item 5,17,33,64,89)의 final은 연구자 1인 결정(4건 final=0). coder1·coder2 개별 원본 미보존, before_final 사본 커밋. 판정 없이 구간만 보고",
#  "B8-2": "B8b로 수행됨(위 항목 참조)",
#  "B6": "B6a·B6b 수행(하위 항목 참조)",
# })
# '''
# s = s.replace("def norm(s)", add + "def norm(s)", 1); open(p, "w", encoding="utf-8").write(s); print("패치 완료")
# EOF
# grep -c "MANUAL.update" pa_a4.py
# python3 -m py_compile pa_a4.py && echo "문법 OK"
# python3 pa_a4.py | tail -n 40
# grep -E "B6a|B6b|B8b|B4c" 09_PA/PA_A4_audit_table.md | cut -c1-200
# ```

# `grep -c`가 `1`이고 B6a·B6b·B8b 줄에 비고가 보이면 반영된 것입니다. 마지막 줄의 "`본 결과 변경: 없음`이 아닌 줄"이 `없음`인지도 확인하세요.

# ## 블록 2. 최종 결론 문서 (통합 문구표와 약화·반전 공개 목록)

# changelog 해석 메모에서 옮긴 문서입니다. 본문에 넣기 전에 원 CSV와 대조하세요.

# ```bash
# cd ~/Research/Minnesota_Transport/paper1/09_PA
# test ! -e PA_FINAL_conclusions.md && cat > PA_FINAL_conclusions.md <<'EOF'
# # [Post-hoc] 최종 결론과 공개 목록 (2026-10-21)

# Post-hoc analyses were specified after inspecting the primary results and are reported as exploratory; no pre-registered decision rule was altered. S0~S8의 통과·실패 판정은 어떤 PA 결과로도 변경하지 않았다. 모든 수치는 PA_CHANGELOG.md 해석 메모에서 옮겼으므로 원고 반영 전 원 CSV와 대조한다.

# ## 1. 쓸 수 있는 문장

# 1. 주제 구조 변화는 점프보다 연속적 재배치로 서술하는 편이 자료에 부합한다 (S5 N2 비통과, A1, 연속 꺾은선 BIC는 탐색적).
# 2. 2020 부근에는 COVID 토픽의 저널 내 증가와 기술 토픽 성장이 겹친 가속이 있다. 표본 크기, 상위 저널, COVID 토픽 중 어느 하나만으로는 사라지지 않으나, 상위 5개 저널과 COVID 토픽을 함께 제외하면 약해지고 변화점이 한 해 앞으로 이동한다 (B2, B2b).
# 3. 2020 부근에서 관측 초과분 크기의 점프가 있었다면 N2 검정이 탐지했을 가능성이 높다. 2008~2012 부근은 검정력이 낮다 (A1).
# 4. 진입(신규+복귀)이 적응보다 큰 방향은 마지막 구간(2015-19 -> 2020-25)에서 가장 뚜렷하며, 부트스트랩 방식, 신규 정의(s6, y5), 분할 이름 제외에 둔감하다 (A2a·b·c).
# 5. 마지막 구간의 진입 우세는 중국 소속 저자를 제외하면 유지되지 않는다(+0.113 -> -0.101, -0.110, -0.024). 진입 중심 갱신과 중국 몫 증가는 한 구조의 두 측면으로 서술한다 (C1, A3).
# 6. 알려진 국가 기준 상위 5개국 몫은 평탄하고(0.64->0.65), 포함 기준 증가는 미상 비중 감소와 함께 움직이며, HHI 반등은 중국 소속 질량과 함께 움직인다 (A3).
# 7. 결측 가정 민감도(B7a·b·c, S4 IPW)에서 big 토픽의 변화 방향이 유지됐다. 경계가 아니라 민감도이다.
# 8. 도로 안전의 사망률 지표에서만 부호가 일관되게 양수였고 구간 하한은 0 근처이다. 소득군 내에서는 HIC가 비관찰이고 UMC는 질량 하한 5에서만 추정 가능했다 (B9a·b). 문구는 "소득 수준과 분리되지 않는 간극"이다.

# ## 2. 쓰면 안 되는 문장

# - 체제 전환이 없었음이 확인됐다 / 2020이 아티팩트가 아님(확정).
# - 신규 저자가 모든 구간을 주도했다 (y2 정의에서 4쌍 모두 역전).
# - 진입 주도는 중국 유입과 무관하다.
# - 핵심 신생 토픽은 신규 저자 진입으로 설명된다 (B3c: 핵심 집합 고유성 없음).
# - ID 기반 결과와 일치한다 / 동명이인 문제가 해결됐다 / 위험군 검증의 순환성이 해소됐다.
# - 소득과 무관하게 정렬된다 / 필요 충족 증거다 / 미달 도메인에서도 정렬이 확인됐다.
# - 소속을 국적으로 해석하는 표현, 모든 인과 해석.

# ## 3. 약화·반전·한계 결과 (본문 한계 절과 부록에 먼저 공개)

# | 항목 | 내용 |
# |---|---|
# | C1 | 진입-적응 +0.113이 신규 제외 시 -0.101, 신규·지속 제외 시 -0.024로 반전 |
# | A2b | y2 정의에서 4쌍 모두 신규<적응(정의 효과, 진입 총량 불변) |
# | A2c | 두 구간 방식 모두 확실한 쌍은 마지막 쌍뿐. excess 구간은 교정 불충분 |
# | A1 | run 1 무효(주입 NaN 오류). 판독은 all·within 강화, fixed14 곡선만 보고. δ=1 점프는 관측 초과분의 약 3배. 2008~2012 사각지대 |
# | B2 | 하향 추출이 20%뿐인 약한 시험 |
# | B2b | 상위 5저널+토픽 71 동시 제외 시 약화(J 비율 0.74~0.83), 변화점 2019로 이동 |
# | B3a·c | 핵심 집합은 변화량 상위 정의의 부산물일 수 있음 |
# | B4 | 첫 쌍만 부호 변화. 제외 4개뿐인 약한 시험 |
# | B4b | 판독 2가 규칙상 민감(A등급 +0.0304, 임계 0.03이 토픽 1개 규모) |
# | B4d | 앞쪽 변화점(2008~2012) 이동·소실, S6 첫 쌍 민감. 제외 15개는 B4의 확장 |
# | B6a | 정밀도 0.789, 재현율 0.330. ORCID 보유·2020+ 한정 |
# | B6b | 연구자 1인 판정, 판정 불가 30/50, 경계 0.13~0.90 |
# | B7c | within의 2009 변화점이 2006으로 이동 |
# | B8a | 미달 도메인은 참고용, 본문 판정 2/6 불변 |
# | B8b | 사후 개정 사전 v3. active_urban 0.80 [0.61, 0.91]. 낙관 편향, kappa 비해석 |
# | B9a·b | LIC·LMC 추정 불가, HIC 비관찰, UMC는 하한 5의 22~23개국에서만 관찰(독립 증거 아님) |
# | C2 | 20칸 중 일치 11. 미국·상위5 몫은 임베딩 보유 집합에서 +0.01~0.03 |

# ## 4. 미수행 항목과 사유

# - B4c: 임베딩 파일을 찾지 못함.
# - A2의 ID 기반 재판별: F7이 2020년 이후 논문만 덮음.
# - B9의 LIC·LMC 소득군 내 분석: 질량 하한으로 자료상 불가.

# ## 5. 기록상 한계

# - v2 계획은 s5_lib 소스와 N2 요약을 열람한 뒤 동결됐다("사전등록"이 아니라 결과 열람 전 명세와 열람 이력 공개).
# - 일부 QUICK 출력에 설계 정보와 수치가 포함돼 열람했고 changelog에 기록했다.
# - S4 코드 3종은 IPython 히스토리에서 복구한 것이며 최종 실행본임은 증명하지 못한다. 산출물 무결성은 manifest 해시로 확인했다.
# - B8b: 제2코더는 사람, 불일치 5건은 연구자 1인 결정, 코더 개별 원본 미보존.
# - 기록 오류(코드 오류, 백업 완료 오기재, 자리표시자 줄, 중복 줄)는 삭제하지 않고 정정 줄로 남겼다.

# ## 6. 최종 결론

# 교통 지식 생산의 주제 구조 변화는 체제 전환보다 연속적 재배치로 서술하는 편이 자료에 부합하며, 2020 부근은 COVID와 기술 주제가 겹친 가속이다. 갱신은 진입이 중심이고 마지막 구간에서 가장 뚜렷하지만, 그 진입의 큰 부분이 중국 소속 저자여서 선두 국가 교체와 분리해 해석할 수 없다. 모든 결과는 탐색적이며 인과 주장이 아니다.
# EOF
# ls -l PA_FINAL_conclusions.md && wc -l PA_FINAL_conclusions.md
# ```

# `test !`가 있어 파일이 이미 있으면 덮어쓰지 않고 멈춥니다. 확인 후 커밋합니다.

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | 최종 정리: PA_FINAL_conclusions.md(쓸 수 있음/쓰면 안 됨, 약화·반전 공개 목록, 미수행 사유, 기록상 한계, 최종 결론)와 A4 감사 표 최종본 생성. 수치는 changelog 해석 메모에서 옮긴 것으로 원고 반영 전 원 CSV 대조 필요 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/pa_a4.py paper1/09_PA/PA_A4_audit_table.csv paper1/09_PA/PA_A4_audit_table.md paper1/09_PA/PA_FINAL_conclusions.md paper1/09_PA/PA_CHANGELOG.md
# git status --short | head -n 8
# ```

# 스테이징이 위 5개뿐이면 이어서:

# ```bash
# git commit -m "PA wrap-up: final A4 table and conclusions" && git push
# ```

# ## 블록 3. 재현성 잔여

# **3-1. GitHub Private 확인** (비공개면 인증 없이 404)

# ```bash
# curl -s -o /dev/null -w "HTTP %{http_code}\n" https://api.github.com/repos/LEEYJ1021/transport-knowledge-paper1
# ```

# `404`면 비공개, `200`이면 공개입니다. 공개라면 알려 주세요(데이터·키 노출 점검이 먼저 필요합니다).

# **3-2. 노출된 키 이력 점검** (폐기 자체는 OpenAlex 사이트에서만 확인됩니다)

# ```bash
# cd ~/Research/Minnesota_Transport
# git log --all --oneline -S"api_key" -- paper1 | head -n 5
# grep -rn -i "api_key\|OA_KEY\|apikey" paper1/*.py 2>/dev/null | cut -c1-120 | head -n 8
# ```

# 출력이 비어 있거나 변수 이름만 보이면 정상입니다. 키 문자열 자체가 보이면 중단하고 알려 주세요.

# **3-3. `S8.5` 노트북 커밋** (`Atlas_*`, `ISF_*`, `pemsbay_*` 등은 paper1 소속이 아니므로 제외)

# ```bash
# cd ~/Research/Minnesota_Transport
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | S8.5 사후분석 노트북·텍스트 기록 커밋(작업 기록 파일, 분석 결과 아님). Atlas·ISF·pemsbay 등 paper1 외 파일은 계속 미커밋 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add "paper1/S8.5. 사후분석.ipynb" "paper1/S8.5. 사후분석.txt" paper1/09_PA/PA_CHANGELOG.md
# git status --short | head -n 6
# ```

# 스테이징이 이 3개뿐이면:

# ```bash
# git commit -m "Record S8.5 post-hoc notebook" && git push
# ```

# **3-4. `oa_raw` 외부 백업** (SHA-256이 일치할 때만 changelog에 기록하는 스크립트)

# ```bash
# cat > /tmp/backup_oa.sh <<'EOF'
# #!/bin/bash
# DEST="$1"
# [ -z "$DEST" ] && { echo "⛔ 사용법: bash /tmp/backup_oa.sh 외부매체_폴더경로"; exit 1; }
# [ -d "$DEST" ] && [ -w "$DEST" ] || { echo "⛔ 폴더가 없거나 쓰기 권한 없음: $DEST"; exit 1; }
# SRC=~/Research/Minnesota_Transport/archive/oa_raw_20261013.tar.gz
# [ -f "$SRC" ] || { echo "⛔ 원본 없음"; exit 1; }
# cp "$SRC" "$DEST"/ || { echo "⛔ 복사 실패"; exit 1; }
# A=$(sha256sum "$SRC" | cut -d' ' -f1); B=$(sha256sum "$DEST/$(basename "$SRC")" | cut -d' ' -f1)
# echo "원본 $A"; echo "사본 $B"
# if [ "$A" = "$B" ]; then
#   echo "- $(date +%Y-%m-%dT%H:%M:%S) | oa_raw 외부 백업 완료: oa_raw_20261013.tar.gz를 외부 매체로 복사, 원본·사본 SHA-256 일치(${A:0:12}). 12:31:17 오기재 줄의 실제 완료 | 본 결과 변경: 없음" >> ~/Research/Minnesota_Transport/paper1/09_PA/PA_CHANGELOG.md
#   echo "일치: changelog 기록함"
# else echo "⛔ 해시 불일치: 기록하지 않음"; fi
# EOF
# lsblk -o NAME,SIZE,MOUNTPOINT | grep -v loop
# ```

# 마운트 경로가 확인되면 실제 경로로 한 번 실행합니다(예: `bash /tmp/backup_oa.sh /media/yjlee/백업디스크`). 매체가 없으면 이 단계는 건너뛰고, 부록에 "외부 백업 미완료"로 남기세요.

# ## 블록 4. 전체 manifest 갱신과 최종 태그

# ```bash
# cd ~/Research/Minnesota_Transport
# python3 - <<'EOF'
# import hashlib, subprocess
# from pathlib import Path
# import pandas as pd
# ROOT = Path.home()/"Research/Minnesota_Transport"; P1 = ROOT/"paper1"
# git = lambda *a: subprocess.run(["git","-C",str(ROOT),*a],capture_output=True,text=True)
# sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
# SKIP = {"oa_raw",".ipynb_checkpoints","_backup_before_income_tv","_recovered_s4","__pycache__"}
# tr = set(git("ls-files","paper1").stdout.splitlines()); rows=[]
# for d in sorted(P1.glob("[0-9][0-9]_*")):
#     for f in sorted(d.rglob("*")):
#         if f.is_file() and not any(s in f.parts for s in SKIP):
#             rel=f.relative_to(ROOT).as_posix(); rows.append(dict(stage=d.name,file=rel,bytes=f.stat().st_size,mtime=pd.Timestamp(f.stat().st_mtime,unit="s").isoformat(timespec="seconds"),sha256=sha(f),git_tracked=rel in tr))
# for f in sorted(P1.glob("*.py")):
#     rel=f.relative_to(ROOT).as_posix(); rows.append(dict(stage="code",file=rel,bytes=f.stat().st_size,mtime=pd.Timestamp(f.stat().st_mtime,unit="s").isoformat(timespec="seconds"),sha256=sha(f),git_tracked=rel in tr))
# M=pd.DataFrame(rows); M.to_csv(P1/"MASTER_MANIFEST.csv",index=False,encoding="utf-8-sig")
# tg=git("for-each-ref","--format=%(refname:short)|%(objecttype)|%(objectname:short)|%(*objectname:short)|%(creatordate:iso)","refs/tags").stdout.strip().splitlines()
# T=pd.DataFrame([x.split("|") for x in tg],columns=["tag","type","obj","commit_if_annotated","created"])
# rem=set(l.split("refs/tags/")[1].replace("^{}","") for l in git("ls-remote","--tags","origin").stdout.splitlines() if "refs/tags/" in l)
# T["on_origin"]=T.tag.isin(rem); T.to_csv(P1/"MASTER_TAGS.csv",index=False,encoding="utf-8-sig")
# print(M.groupby("stage").agg(files=("file","size"),untracked=("git_tracked",lambda x:int((~x).sum()))).to_string())
# print("\n태그",len(T),"| 원격 미푸시",int((~T.on_origin).sum()),list(T[~T.on_origin].tag)[:8])
# EOF
# ```

# `untracked`가 0이 아닌 단계와 미푸시 태그가 있으면 알려 주세요. 모두 정상이면 최종 기록과 태그를 올립니다.

# ```bash
# echo "- $(date +%Y-%m-%dT%H:%M:%S) | 최종 manifest·태그 목록 재생성(기록 전용, 생성 시각 기준 해시이며 동결 시점 해시는 단계별 manifest가 증명). 이 시점에서 PA 전 항목 종료 | 본 결과 변경: 없음" >> paper1/09_PA/PA_CHANGELOG.md
# git add paper1/MASTER_MANIFEST.csv paper1/MASTER_TAGS.csv paper1/09_PA/PA_CHANGELOG.md
# git commit -m "Final master manifest and tag list" && git tag -a PA-final-2026-10-21 -m "all post-hoc analyses closed" && git push --follow-tags
# ```

# ## 블록 5. 최종 점검

# ```bash
# cd ~/Research/Minnesota_Transport
# git status --short | head -n 10
# git fetch origin && git status -sb | head -n 1
# grep -c "본 결과 변경" paper1/09_PA/PA_CHANGELOG.md
# grep "본 결과 변경" paper1/09_PA/PA_CHANGELOG.md | grep -v -c "본 결과 변경[^:]*: *없음"
# ```

# - 첫 명령에 `paper1/` 경로의 `M`/`A`가 없어야 합니다(`Atlas_*` 등 외부 파일의 `??`는 무시).
# - 둘째는 `## main...origin/main`처럼 앞서거나 뒤처지지 않아야 합니다.
# - 마지막 두 줄 중 뒤의 숫자(= "없음"이 아닌 줄 수)가 `0`이어야 부록 문장 "모든 줄 본 결과 변경 없음"이 성립합니다.

# ## 남는 것

# 저널 선정 후의 S9·S10(후보 풀 동결, 저널 선례 3편 이상, 심사 대응표, 생성형 AI 사용 선언)과 원고 작성만 남습니다. 블록 3-4는 외부 매체가 있을 때만 완료됩니다. 각 블록 출력을 붙여 주시면 확인하겠습니다.